# Manutenção preditiva de turbofans (C-MAPSS): o projeto da monografia em um só notebook

Autor: Leonardo Rossi · Orientador: Prof. Dr. Alexandre Luis Magalhães Levada · UFSCar, 2026

Este notebook é **autocontido**: contém o código do pacote `pdm_cmaps`, os testes, os planos de execução, os scripts de tabelas e de figuras, os dados públicos do
NASA C-MAPSS e os hiperparâmetros já escolhidos. Rodando as células em ordem, ele reproduz **os resultados, as tabelas e os gráficos** da monografia.
O texto LaTeX da monografia não está aqui.

**Como usar**

1. Abra no Google Colab (recomendado: *CPU High-RAM*, 8 núcleos) e rode da seção 1 à 4. Isso leva poucos minutos.
2. Na seção 5, ajuste `EXECUTAR`. O padrão roda só os testes e uma demonstração rápida (um subconjunto).
3. As seções 6 a 8 reproduzem tudo: **horas de máquina**. Os passos são retomáveis (um marcador por passo concluído em `results/logs/`).
4. A seção 9 empacota tabelas e gráficos para download.

**Mapa**

| Seção | O que faz |
|---|---|
| 1 | Pastas e instalação das dependências |
| 2 | Código: pacote, testes, planos de execução, scripts de tabelas e de figuras |
| 3 | Dados: C-MAPSS e hiperparâmetros escolhidos |
| 4 | Testes automáticos |
| 5 | Configuração das execuções (`EXECUTAR`) |
| 6 | Resultados: pipeline, teste oficial de RUL, hiperparâmetros, estudos |
| 7 | Tabelas (`tcc-writing/tables/`) |
| 8 | Gráficos (`tcc-writing/figures/`) |
| 9 | Empacotar para download |

> O notebook é um **instantâneo** gerado por `notebooks/gerar_notebook.py`. A fonte da verdade continua sendo o repositório; não edite o código aqui.
> As redes (PyTorch) dão números ligeiramente diferentes entre máquinas; uma tabela deve sair toda da mesma máquina.

## 1. Preparação

In [ ]:
import os
from pathlib import Path

PASTA = Path("projeto").resolve()          # raiz do projeto (no Colab: /content/projeto)
for d in ["data", "data/CMaps", "data/hiperparametros/FD001", "data/hiperparametros/FD001/CatBoost", "data/hiperparametros/FD001/LightGBM", "data/hiperparametros/FD001/MLP", "data/hiperparametros/FD001/PyTorch_CNN", "data/hiperparametros/FD001/PyTorch_RNN_LSTM", "data/hiperparametros/FD001/PyTorch_Transformer", "data/hiperparametros/FD001/Random_Forest", "data/hiperparametros/FD001/Ridge", "data/hiperparametros/FD001/XGBoost", "data/hiperparametros/FD001/XGBoost_NASA_Loss", "data/hiperparametros/FD002", "data/hiperparametros/FD002/CatBoost", "data/hiperparametros/FD002/LightGBM", "data/hiperparametros/FD002/MLP", "data/hiperparametros/FD002/PyTorch_CNN", "data/hiperparametros/FD002/PyTorch_RNN_LSTM", "data/hiperparametros/FD002/PyTorch_Transformer", "data/hiperparametros/FD002/Random_Forest", "data/hiperparametros/FD002/Ridge", "data/hiperparametros/FD002/XGBoost", "data/hiperparametros/FD002/XGBoost_NASA_Loss", "data/hiperparametros/FD003", "data/hiperparametros/FD003/CatBoost", "data/hiperparametros/FD003/LightGBM", "data/hiperparametros/FD003/MLP", "data/hiperparametros/FD003/PyTorch_CNN", "data/hiperparametros/FD003/PyTorch_RNN_LSTM", "data/hiperparametros/FD003/PyTorch_Transformer", "data/hiperparametros/FD003/Random_Forest", "data/hiperparametros/FD003/Ridge", "data/hiperparametros/FD003/XGBoost", "data/hiperparametros/FD003/XGBoost_NASA_Loss", "data/hiperparametros/FD004", "data/hiperparametros/FD004/CatBoost", "data/hiperparametros/FD004/LightGBM", "data/hiperparametros/FD004/MLP", "data/hiperparametros/FD004/PyTorch_CNN", "data/hiperparametros/FD004/PyTorch_RNN_LSTM", "data/hiperparametros/FD004/PyTorch_Transformer", "data/hiperparametros/FD004/Random_Forest", "data/hiperparametros/FD004/Ridge", "data/hiperparametros/FD004/XGBoost", "data/hiperparametros/FD004/XGBoost_NASA_Loss", "results", "scripts", "src", "src/pdm_cmaps", "src/pdm_cmaps/analysis", "src/pdm_cmaps/cli", "src/pdm_cmaps/data", "src/pdm_cmaps/detection", "src/pdm_cmaps/domain", "src/pdm_cmaps/experiments", "src/pdm_cmaps/modeling", "src/pdm_cmaps/monitoring", "src/pdm_cmaps/pipeline", "src/pdm_cmaps/reporting", "tcc-writing", "tcc-writing/diagrams", "tcc-writing/diagrams/explicativos", "tcc-writing/figures", "tcc-writing/tables", "tests"]:
    (PASTA / d).mkdir(parents=True, exist_ok=True)
print("raiz do projeto:", PASTA)
print("núcleos:", os.cpu_count())

In [ ]:
# O código e os dados vêm nas seções 2 e 3; a instalação (`pip install -e`) vem logo depois, na seção 3.
!nproc --all
!python --version

## 2. Código

Cada célula grava um arquivo em `projeto/`. Rode todas (são só gravações).

### 2.1 Configuração do pacote

1 arquivos.

In [ ]:
%%writefile projeto/pyproject.toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "pdm-cmaps"
version = "0.1.0"
description = "Manutenção preditiva em turbofans NASA C-MAPSS: detecção de anomalias, RUL e camada de monitoramento"
requires-python = ">=3.10"
dependencies = [
    "numpy", "pandas", "scipy", "scikit-learn", "matplotlib",
    "catboost", "xgboost", "lightgbm", "plotly>=6.0",
    "torch",  # importado incondicionalmente por modeling/ e analysis/, portanto obrigatório
]

[project.optional-dependencies]
dev = ["pytest"]
tuning = ["optuna"]      # busca de hiperparâmetros (pdm_cmaps.experiments.selecao_hiperparametros)
workflow = ["snakemake"]  # fluxo reprodutível: ver o Snakefile

[project.scripts]
pdm-cmaps = "pdm_cmaps.cli.main:main"
pdm-update-all = "pdm_cmaps.cli.update_all:main"

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
pythonpath = ["src"]
testpaths = ["tests"]
markers = [
    "requirement(id): Traceability marker for plan requirement IDs",
    "slow: Slow end-to-end pipeline execution checks",
]
addopts = '-m "not slow"'

### 2.2 Pacote `pdm_cmaps`: (raiz do pacote)

configuração, protocolo de avaliação e utilidades

5 arquivos.

In [ ]:
%%writefile projeto/src/pdm_cmaps/__init__.py
"""PdM-CMaps: predictive maintenance for NASA C-MAPSS turbofans (RUL, anomaly detection, monitoring layer)."""

from pdm_cmaps.analysis.stats import ThesisHypothesisAnalyzer
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.detection.regime import RegimeDecoupler
from pdm_cmaps.detection.spc import EnvelopeAnomalyDetector, StatisticalAnomalyDetector
from pdm_cmaps.detection.virtual_sensor import VirtualSensor
from pdm_cmaps.domain.rul import compute_piecewise_rul, evaluate_rul_predictions, nasa_scoring_function
from pdm_cmaps.modeling.benchmark import RULBenchmark
from pdm_cmaps.modeling.registry import get_model, get_model_pool
from pdm_cmaps.pipeline.config import PipelineConfig
from pdm_cmaps.pipeline.pipeline import PdMPipeline

__all__ = [
    "CMapssLoader", "compute_piecewise_rul", "evaluate_rul_predictions", "nasa_scoring_function",
    "RegimeDecoupler", "StatisticalAnomalyDetector", "EnvelopeAnomalyDetector", "VirtualSensor",
    "RULBenchmark", "get_model", "get_model_pool", "ThesisHypothesisAnalyzer", "PdMPipeline", "PipelineConfig",
]

In [ ]:
%%writefile projeto/src/pdm_cmaps/__main__.py
"""`python -m pdm_cmaps` runs the command-line pipeline."""

import sys

from pdm_cmaps.cli.main import main

sys.exit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/config.py
"""Project paths: the single place that knows where data and outputs live."""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

# src/pdm_cmaps/config.py -> repository root
PROJECT_ROOT = Path(__file__).resolve().parents[2]


@dataclass(frozen=True)
class ProjectPaths:
    """Filesystem layout of the repository, injectable for tests and alternative checkouts."""

    root: Path = PROJECT_ROOT

    @property
    def data_dir(self) -> Path:
        return self.root / "data" / "CMaps"

    @property
    def results_dir(self) -> Path:
        return self.root / "results"

    @property
    def tables_dir(self) -> Path:
        """Versioned tables consumed by the monograph and dashboards."""
        return self.root / "tcc-writing" / "tables"

    @property
    def logs_dir(self) -> Path:
        return self.results_dir / "logs"

    def guide_dir(self, subset: str) -> Path:
        """Output directory of the per-subset pipeline."""
        return self.results_dir / f"maintenance_guide_{subset.lower()}"


DEFAULT_PATHS = ProjectPaths()
DEFAULT_DATA_DIR = DEFAULT_PATHS.data_dir
DEFAULT_RESULTS_DIR = DEFAULT_PATHS.results_dir

In [ ]:
%%writefile projeto/src/pdm_cmaps/progress.py
"""Progress with time estimate for long, multi-step runs (pipeline phases, `--all`, `update_all`).

Every step prints one line such as
    [3/13] 06_fault_detection ... iniciando | decorrido 1h12min | restante ~2h05min (média dos passos concluídos)
so whoever is watching the terminal (or `tail -f results/logs/_progresso.txt`) knows what is running and how long is left.

The estimate is the mean duration of the finished steps times the steps still to run. Steps differ a lot in cost,
so it is a rough guide, not a promise; it is hidden until one step has finished.
"""

from __future__ import annotations

import time
from collections.abc import Callable
from pathlib import Path


def format_duration(seconds: float) -> str:
    """90 -> '1min', 4500 -> '1h15min', 20 -> '20s'."""
    seconds = max(0, int(round(seconds)))
    if seconds < 60:
        return f"{seconds}s"
    minutes, _ = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours}h{minutes:02d}min" if hours else f"{minutes}min"


class ProgressTracker:
    """Counts finished steps and estimates the time left. Inject `clock` and `emit` to test without sleeping."""

    def __init__(
        self,
        total: int,
        log_file: Path | None = None,
        clock: Callable[[], float] = time.time,
        emit: Callable[[str], None] | None = None,
    ) -> None:
        self.total = total
        self.log_file = log_file
        self._clock = clock
        self._emit = emit or (lambda line: print(line, flush=True))
        self._t0 = clock()
        self._step_start = self._t0
        self._durations: list[float] = []

    @property
    def done(self) -> int:
        return len(self._durations)

    def remaining_seconds(self) -> float | None:
        """Mean finished-step duration x steps left; None before the first step finishes."""
        if not self._durations:
            return None
        return sum(self._durations) / len(self._durations) * (self.total - self.done)

    def start(self, name: str) -> None:
        self._step_start = self._clock()
        self._write(f"[{self.done + 1}/{self.total}] {name} ... iniciando | {self._timing()}")

    def finish(self, name: str, returncode: int = 0) -> None:
        self._durations.append(self._clock() - self._step_start)
        status = "ok" if returncode == 0 else f"FALHOU (código {returncode})"
        self._write(f"[{self.done}/{self.total}] {name} ... {status} em {format_duration(self._durations[-1])} | {self._timing()}")

    def _timing(self) -> str:
        text = f"decorrido {format_duration(self._clock() - self._t0)}"
        left = self.remaining_seconds()
        if left is None:
            return text + " | restante: estimado após o 1º passo"
        return text + (f" | restante ~{format_duration(left)} (média dos passos concluídos)" if left > 0 else " | concluído")

    def _write(self, line: str) -> None:
        stamped = f"[{time.strftime('%H:%M:%S')}] {line}"
        self._emit(stamped)
        if self.log_file is not None:
            self.log_file.parent.mkdir(parents=True, exist_ok=True)
            with open(self.log_file, "a") as f:
                f.write(stamped + "\n")

In [ ]:
%%writefile projeto/src/pdm_cmaps/protocol.py
"""Protocolo primário, CONGELADO em 2026-11-18 (depois de ver os resultados; não é um pré-registro).

Uma só fonte para os parâmetros que definem os números principais da monografia. `tests/test_protocolo.py` confere os padrões das classes e
dos `argparse` do código contra estas constantes: se alguém mudar um padrão sem mudar o protocolo (ou o contrário), o teste falha. Mudar um
valor aqui invalida as tabelas de resultados e exige reexecutar os estudos afetados.
"""
from __future__ import annotations

# --- dados e divisão
SEEDS = (42, 7, 123)          # três divisões treino-teste por motor; as sementes NÃO são réplicas independentes (mesmos motores)
TEST_RATIO = 0.25             # fração de motores de teste em toda divisão por motor
RUL_CAP = 125                 # teto da RUL linear por partes
SEEDS_RUL = (42, 7, 123, 1, 2)  # benchmark de RUL: 5 sementes (as redes e o MiniRocket são determinísticos; o desvio entre sementes é variância de treino)

# --- janela saudável e deslocamento do motor
H_NATIVE = 50                 # motores nativos e camada de monitoramento
H_INJECTION = 25              # estudos de injeção (padrão de trabalho, não o ótimo: ver Seção 5.4)
W_SPLIT = 25                  # janela do deslocamento na análise de apoio "janela dividida"
ENGINE_OFFSET = True          # deslocamento do motor (por motor e regime, mín. 3 linhas) em TODOS os detectores
ENGINE_OFFSET_MEWMA_IN_RUL_INJECTION = False  # exceção deliberada: a injeção para a RUL (apêndice) usa a MEWMA sem o deslocamento

# --- detectores
EWMA_ALPHA = 0.10             # EWMA do |z| do T50 e dos sensores virtuais
THRESHOLD_SIGMA = 3.0         # portão por |z|
MEWMA_LAMBDA = 0.10
MEWMA_ALPHA = 0.0027          # nível do limite nominal do T²
PERSISTENCE_K = 2             # ciclos consecutivos acima do limite para o alarme (nos estudos; o padrão da classe VirtualSensor é 3 e só vale sem argumento)
MEWMA_CALIBRATION = "empirical"  # padrão da CLASSE e dos estudos de injeção (Seção 5.5 e a parte de injeção da Seção 5.4): limite do T² medido na janela
# EMENDA post hoc de 2026-11-18 (decisão do autor): nos motores nativos e na camada de monitoramento o limite é o SINTÉTICO, passado como argumento
# explícito em `monitoring/layer.py` (o padrão da classe não muda, porque mudaria as tabelas de injeção sem reexecutá-las). Validação só parcial
#; não é um resultado validado.
MEWMA_CALIBRATION_NATIVE = "synthetic"

# --- estudos de injeção
N_REPS_HARD = 2               # réplicas por motor nas falhas sorteadas (10 008 falhas)

# --- estatística
BOOTSTRAP_UNIT = "motor"      # reamostragem por motor receptor
HOLM_FAMILIES = {             # contrastes por família (Holm dentro de cada uma); fora delas, p sem correção, declarado
    "T50 sem portão - MEWMA com regime": 16,
    "regime na MEWMA (com - sem)": 12,
    "T50 sem portão - T50 bruto": 8,
}

### 2.2 Pacote `pdm_cmaps`: data

carga e divisão dos dados

4 arquivos.

In [ ]:
%%writefile projeto/src/pdm_cmaps/data/__init__.py
"""Reading raw C-MAPSS files (loader.py), engine-level splits (splits.py) and causal features (features.py)."""

In [ ]:
%%writefile projeto/src/pdm_cmaps/data/features.py
"""Causal temporal feature engineering (present and past of each engine only)."""

from __future__ import annotations

import pandas as pd
import numpy as np




class CausalFeatureEngineer:
    """Stateful temporal feature engineer that learns healthy baseline variance for startup imputation.

    [CAUSALITY] Every feature at cycle t (lags, rolling mean/std, slopes) is built from cycles
    <= t only; rolling statistics use shift(1) so they never include the current row's future.
    At deployment you cannot see tomorrow's sensor values, so neither may the model in testing.
    Why these features: single-row models cannot see TREND, and degradation is a trend. They let
    tabular models (XGBoost, ...) get the same temporal context as LSTM/CNN/Transformer.
    The class is stateful (fit/transform) so start-up imputation values come from training only.
    """

    def __init__(self, sensor_cols: list[str], windows: tuple[int, ...] = (3, 5, 10, 20)):
        self.sensor_cols = list(sensor_cols)
        self.windows = tuple(windows)
        self.healthy_rollstd_medians_: dict[tuple[str, int], float] = {}

    def fit(self, train_frame: pd.DataFrame, healthy_cycle_limit: int = 50) -> "CausalFeatureEngineer":
        """Compute median rolling std in healthy training phase for initial imputation."""
        healthy = train_frame.loc[train_frame["cycle"] <= healthy_cycle_limit].sort_values(["unit_id", "cycle"])
        for sensor in self.sensor_cols:
            grouped = healthy.groupby("unit_id", sort=False)[sensor]
            for window in self.windows:
                roll_std = grouped.transform(
                    lambda values: values.shift(1).rolling(window, min_periods=2).std()
                ).dropna()
                median_std = float(roll_std.median()) if not roll_std.empty else 0.01
                if not np.isfinite(median_std) or median_std < 1e-6:
                    median_std = 0.01
                self.healthy_rollstd_medians_[(sensor, window)] = median_std
        return self

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        """Add lags, deltas, startup indicator, rolling metrics, and causal window slopes with learned startup imputation."""
        result = frame.sort_values(["unit_id", "cycle"]).copy().reset_index(drop=True)
        result["is_startup"] = (result["cycle"] <= 2).astype(float)
        new_cols: dict[str, pd.Series | np.ndarray] = {}
        for sensor in self.sensor_cols:
            grouped = result.groupby("unit_id", sort=False)[sensor]
            lag1 = grouped.shift(1)
            lag2 = grouped.shift(2)
            new_cols[f"{sensor}_lag1"] = lag1.fillna(result[sensor])
            # Boundary fix at t=2: fill lag2 with lag1 first, then current sensor
            new_cols[f"{sensor}_lag2"] = lag2.fillna(lag1).fillna(result[sensor])
            new_cols[f"{sensor}_delta"] = (result[sensor] - lag1).fillna(0.0)
            for window in self.windows:
                fallback_std = self.healthy_rollstd_medians_.get((sensor, window), 0.0)
                new_cols[f"{sensor}_rollmean_{window}"] = grouped.transform(
                    lambda values: values.shift(1).rolling(window, min_periods=1).mean()
                ).fillna(result[sensor])
                new_cols[f"{sensor}_rollstd_{window}"] = grouped.transform(
                    lambda values: values.shift(1).rolling(window, min_periods=2).std()
                ).fillna(fallback_std)
                # Causal slope over window w: (x(t) - x(t-w)) / w
                lag_w = grouped.shift(window).fillna(grouped.transform("first"))
                new_cols[f"{sensor}_slope_{window}"] = ((result[sensor] - lag_w) / float(window)).fillna(0.0)
        if new_cols:
            result = pd.concat([result, pd.DataFrame(new_cols, index=result.index)], axis=1)
        return result

    def fit_transform(self, train_frame: pd.DataFrame, healthy_cycle_limit: int = 50) -> pd.DataFrame:
        """Fit on train_frame and transform it."""
        return self.fit(train_frame, healthy_cycle_limit=healthy_cycle_limit).transform(train_frame)

def engineer_causal_temporal_features(
    frame: pd.DataFrame,
    sensor_cols: list[str],
    windows: tuple[int, ...] = (3, 5),
) -> pd.DataFrame:
    """Add lags, deltas, trailing statistics, and causal slopes; startup values use current/past data only."""
    result = frame.sort_values(["unit_id", "cycle"]).copy().reset_index(drop=True)
    new_cols: dict[str, pd.Series | np.ndarray] = {}
    for sensor in sensor_cols:
        grouped = result.groupby("unit_id", sort=False)[sensor]
        lag1 = grouped.shift(1)
        lag2 = grouped.shift(2)
        new_cols[f"{sensor}_lag1"] = lag1.fillna(result[sensor])
        # Boundary fix at t=2: fill lag2 with lag1 (x at t=1), NOT x at t=2
        new_cols[f"{sensor}_lag2"] = lag2.fillna(lag1).fillna(result[sensor])
        new_cols[f"{sensor}_delta"] = (result[sensor] - lag1).fillna(0.0)
        for window in windows:
            new_cols[f"{sensor}_rollmean_{window}"] = grouped.transform(
                lambda values: values.shift(1).rolling(window, min_periods=1).mean()
            ).fillna(result[sensor])
            new_cols[f"{sensor}_rollstd_{window}"] = grouped.transform(
                lambda values: values.shift(1).rolling(window, min_periods=2).std()
            ).fillna(0.0)
            lag_w = grouped.shift(window).fillna(grouped.transform("first"))
            new_cols[f"{sensor}_slope_{window}"] = ((result[sensor] - lag_w) / float(window)).fillna(0.0)
    if new_cols:
        result = pd.concat([result, pd.DataFrame(new_cols, index=result.index)], axis=1)
    return result

In [ ]:
%%writefile projeto/src/pdm_cmaps/data/loader.py
"""Reading of the raw C-MAPSS text files."""

from __future__ import annotations

from pathlib import Path
import pandas as pd

from pdm_cmaps.config import DEFAULT_DATA_DIR
from pdm_cmaps.domain.schema import COLUMNS, SENSOR_COLUMNS, SUBSETS



class CMapssLoader:
    """Load named, validated C-MAPSS training trajectories from a data directory."""

    def __init__(self, data_dir: Path = DEFAULT_DATA_DIR):
        self.data_dir = Path(data_dir)

    def load(self, subset: str) -> pd.DataFrame:
        """Load one training subset and validate its rows and 26-column schema."""
        subset = subset.upper()
        if subset not in SUBSETS:
            raise ValueError(f"subset must be one of {', '.join(SUBSETS)}")
        path = self.data_dir / f"train_{subset}.txt"
        if not path.is_file():
            raise FileNotFoundError(f"Missing C-MAPSS input: {path}")
        frame = pd.read_csv(path, sep=r"\s+", header=None, names=COLUMNS)
        self.validate(frame, path)
        return frame.sort_values(["unit_id", "cycle"]).reset_index(drop=True)

    def load_all(self) -> dict[str, pd.DataFrame]:
        """Load FD001 through FD004 using the same validated schema."""
        return {subset: self.load(subset) for subset in SUBSETS}

    def load_test(self, subset: str) -> tuple[pd.DataFrame, pd.Series]:
        """Load test trajectory text and the corresponding official ground-truth final RUL vector."""
        subset = subset.upper()
        if subset not in SUBSETS:
            raise ValueError(f"subset must be one of {', '.join(SUBSETS)}")
        test_path = self.data_dir / f"test_{subset}.txt"
        rul_path = self.data_dir / f"RUL_{subset}.txt"
        if not test_path.is_file():
            raise FileNotFoundError(f"Missing test trajectory input: {test_path}")
        if not rul_path.is_file():
            raise FileNotFoundError(f"Missing true RUL vector: {rul_path}")

        test_frame = pd.read_csv(test_path, sep=r"\s+", header=None, names=COLUMNS)
        self.validate(test_frame, test_path)
        test_frame = test_frame.sort_values(["unit_id", "cycle"]).reset_index(drop=True)

        rul_series = pd.read_csv(rul_path, sep=r"\s+", header=None).iloc[:, 0].astype(float)
        n_test_engines = test_frame["unit_id"].nunique()
        if len(rul_series) != n_test_engines:
            raise ValueError(
                f"Mismatch in test set engine count: {test_path.name} has {n_test_engines} engines, "
                f"but {rul_path.name} has {len(rul_series)} RUL values."
            )
        return test_frame, rul_series

    @staticmethod
    def validate(frame: pd.DataFrame, source: Path | None = None) -> None:
        """Reject malformed, missing, nonnumeric, or duplicate engine-cycle records."""
        label = f" in {source}" if source is not None else ""
        missing_columns = set(COLUMNS).difference(frame.columns)
        if missing_columns:
            raise ValueError(f"Missing required C-MAPSS columns{label}: {sorted(missing_columns)}")
        if frame.loc[:, COLUMNS].isna().any().any():
            raise ValueError(f"Missing telemetry values{label}")
        if frame.duplicated(["unit_id", "cycle"]).any():
            raise ValueError(f"Duplicate engine-cycle keys{label}")
        if not all(pd.api.types.is_numeric_dtype(frame[column]) for column in COLUMNS):
            raise ValueError(f"Non-numeric telemetry column{label}")
        if (frame["cycle"] < 1).any():
            raise ValueError(f"Cycle values must be positive{label}")

    @staticmethod
    def informative_sensors(frame: pd.DataFrame) -> list[str]:
        """Return sensor channels that vary in the supplied population.

    [LEAKAGE] Even this "which sensors are constant?" filter is a learned choice, so callers
    must pass TRAINING engines only (pipeline.py does).
    """
        return [name for name in SENSOR_COLUMNS if frame[name].nunique() > 1 and frame[name].std() > 1e-6]

In [ ]:
%%writefile projeto/src/pdm_cmaps/data/splits.py
"""Engine-level splits and grouped folds (no leakage between engines)."""

from __future__ import annotations

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold

from pdm_cmaps.domain.schema import DEFAULT_HEALTHY_CYCLE_LIMIT, DEFAULT_RANDOM_STATE



def split_engine_units(
    frame: pd.DataFrame,
    test_ratio: float = 0.25,
    random_state: int = DEFAULT_RANDOM_STATE,
    stratify_eol: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    """Return sorted train/test engine IDs with no engine overlap and optional EOL stratification.

    [LEAKAGE] Consecutive cycles of one engine are almost duplicates (same hardware, slowly
    drifting sensors). A random ROW split would put neighbouring cycles of the same engine in
    both train and test, so the model would "recognise" the engine instead of learning
    degradation, and R² would be inflated. We therefore split whole engines.
    Stratification by end-of-life (EOL) keeps short- and long-lived engines in both sets, so
    the test fleet is not accidentally all easy (or all hard) cases.
    """
    if not 0.0 < test_ratio < 1.0:
        raise ValueError("test_ratio must be between zero and one")
    units = np.asarray(frame["unit_id"].unique()).copy()
    if len(units) < 2:
        raise ValueError("At least two engines are required for a holdout split")

    if stratify_eol:
        eol_series = frame.groupby("unit_id")["cycle"].max().loc[units]
        n_unique_eol = eol_series.nunique()
        if n_unique_eol > 1:
            q = min(5, len(units) // 4)
            bins = None
            if q > 1:
                try:
                    bins = pd.qcut(eol_series, q=q, labels=False, duplicates="drop")
                except Exception:
                    bins = None
            if bins is not None and bins.nunique() > 1:
                rng = np.random.default_rng(random_state)
                train_list, test_list = [], []
                for _, b_units in bins.groupby(bins):
                    u_arr = b_units.index.to_numpy().copy()
                    rng.shuffle(u_arr)
                    n_b_test = max(0, int(round(len(u_arr) * test_ratio)))
                    test_list.extend(u_arr[:n_b_test])
                    train_list.extend(u_arr[n_b_test:])
                if not test_list and len(train_list) > 1:
                    test_list.append(train_list.pop())
                elif not train_list and len(test_list) > 1:
                    train_list.append(test_list.pop())
                return np.sort(np.asarray(train_list)), np.sort(np.asarray(test_list))

    np.random.default_rng(random_state).shuffle(units)
    n_test = min(len(units) - 1, max(1, int(round(len(units) * test_ratio))))
    return np.sort(units[n_test:]), np.sort(units[:n_test])

def partition_healthy_and_lifecycle(
    frame: pd.DataFrame,
    train_units: np.ndarray,
    test_units: np.ndarray,
    healthy_limit: int = DEFAULT_HEALTHY_CYCLE_LIMIT,
) -> dict[str, pd.DataFrame]:
    """Create healthy-window and full-lifecycle frames for each engine partition."""
    overlap = set(train_units).intersection(test_units)
    if overlap:
        raise ValueError(f"Train/test engine overlap: {sorted(overlap)}")
    ordered = frame.sort_values(["unit_id", "cycle"]).reset_index(drop=True)
    train_mask = ordered["unit_id"].isin(train_units)
    test_mask = ordered["unit_id"].isin(test_units)
    return {
        "healthy_train": ordered.loc[train_mask & (ordered["cycle"] <= healthy_limit)].copy().reset_index(drop=True),
        "healthy_test": ordered.loc[test_mask & (ordered["cycle"] <= healthy_limit)].copy().reset_index(drop=True),
        "full_lifecycle_train": ordered.loc[train_mask].copy().reset_index(drop=True),
        "full_lifecycle_test": ordered.loc[test_mask].copy().reset_index(drop=True),
    }

def grouped_folds(frame: pd.DataFrame, n_splits: int = 5):
    """Yield GroupKFold indices while preserving each engine as an indivisible group."""
    groups = frame["unit_id"].to_numpy()
    unique_groups = np.unique(groups)
    if n_splits < 2 or n_splits > len(unique_groups):
        raise ValueError(f"n_splits must be between 2 and {len(unique_groups)} engines")
    splitter = GroupKFold(n_splits=n_splits)
    placeholder = np.zeros(len(frame))
    for train_idx, test_idx in splitter.split(placeholder, groups=groups):
        if set(groups[train_idx]).intersection(groups[test_idx]):
            raise AssertionError("Grouped fold split an engine across train and test")
        yield train_idx, test_idx

### 2.2 Pacote `pdm_cmaps`: domain

RUL e esquema das colunas

3 arquivos.

In [ ]:
%%writefile projeto/src/pdm_cmaps/domain/__init__.py
"""Pure rules with no I/O: C-MAPSS schema and constants (schema.py), RUL target and metrics (rul.py)."""

In [ ]:
%%writefile projeto/src/pdm_cmaps/domain/rul.py
"""RUL targets and evaluation metrics (RMSE, MAE, R2, NASA score)."""

from __future__ import annotations

import pandas as pd
import numpy as np
from sklearn.metrics import average_precision_score, mean_absolute_error, mean_squared_error, r2_score

from pdm_cmaps.domain.schema import DEFAULT_RUL_CAP



def compute_piecewise_rul(
    frame: pd.DataFrame,
    rul_max: int = DEFAULT_RUL_CAP,
    unit_col: str = "unit_id",
    cycle_col: str = "cycle",
) -> pd.Series:
    """Compute min(RUL cap, engine final cycle minus current cycle).

    [PIECEWISE-RUL] Early in life an engine shows no measurable wear, so "200 cycles left" and
    "150 cycles left" look identical in the sensors. A linear RUL target would force the model
    to chase noise there. Capping at 125 (the standard C-MAPSS convention) says: "while healthy,
    just predict 125". It also means scores computed on capped RUL are NOT comparable with
    papers that use uncapped targets; always state the cap.
    """
    if rul_max < 1:
        raise ValueError("rul_max must be positive")
    end_of_life = frame.groupby(unit_col)[cycle_col].transform("max")
    return (end_of_life - frame[cycle_col]).clip(lower=0, upper=rul_max).astype(float)

def clip_predictions(y_pred, lower: float = 0.0, upper: float = float(DEFAULT_RUL_CAP)) -> np.ndarray:
    """Clip RUL predictions to physical bounds [lower, upper]."""
    return np.clip(np.asarray(y_pred, dtype=float), lower, upper)

def nasa_scoring_function(y_true, y_pred) -> tuple[float, float]:
    """Return total and mean asymmetric PHM score with overflow protection.

    [METRIC-CHOICE] Error d = predicted - true. Predicting too LATE (d > 0, engine fails before
    the alarm) is penalised with exp(d/10); too EARLY (d < 0, wasted life) with exp(-d/13).
    Maintenance logic: a missed failure costs far more than an early inspection. The TOTAL
    score grows with the number of engines, so compare MEAN score across different fleets.
    """
    difference = np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float)
    diff_clipped = np.clip(difference, -500.0, 500.0)
    per_row = np.where(
        diff_clipped < 0,
        np.exp(-diff_clipped / 13.0) - 1.0,
        np.exp(diff_clipped / 10.0) - 1.0,
    )
    return float(per_row.sum()), float(per_row.mean())

def regression_metrics(y_true, y_pred) -> dict[str, float]:
    """Return R², RMSE, MAE, and target variance on the scored observations."""
    truth = np.asarray(y_true, dtype=float)
    prediction = np.asarray(y_pred, dtype=float)
    return {
        "R2": float(r2_score(truth, prediction)),
        "RMSE": float(np.sqrt(mean_squared_error(truth, prediction))),
        "MAE": float(mean_absolute_error(truth, prediction)),
        "target_variance": float(np.var(truth)),
    }

def evaluate_rul_predictions(
    y_true,
    y_pred,
    clip: bool = False,
    rul_cap: float = float(DEFAULT_RUL_CAP),
) -> dict[str, float]:
    """Calculate symmetric regression metrics and asymmetric RUL costs with optional clipping."""
    truth = np.asarray(y_true, dtype=float)
    prediction = clip_predictions(y_pred, 0.0, rul_cap) if clip else np.asarray(y_pred, dtype=float)
    total, mean = nasa_scoring_function(truth, prediction)
    return {
        **{key: round(value, 6) for key, value in regression_metrics(truth, prediction).items()},
        "NASA_Score": round(total, 2),
        "Mean_NASA_Score": round(mean, 3),
        "Late_Pred_Pct": round(float(np.mean(prediction > truth) * 100.0), 1),
    }

def evaluate_stratified_rul(
    y_true,
    y_pred,
    critical_threshold: float = 50.0,
    rul_cap: float = float(DEFAULT_RUL_CAP),
) -> dict[str, dict[str, float]]:
    """Evaluate RUL predictions across full holdout and critical degradation zone (RUL < critical_threshold)."""
    truth = np.asarray(y_true, dtype=float)
    prediction = np.asarray(y_pred, dtype=float)
    overall = evaluate_rul_predictions(truth, prediction, clip=False)
    overall["N_samples"] = len(truth)

    critical_mask = truth < critical_threshold
    if critical_mask.any():
        critical_zone = evaluate_rul_predictions(truth[critical_mask], prediction[critical_mask], clip=False)
        critical_zone["N_samples"] = int(np.sum(critical_mask))
    else:
        critical_zone = {"N_samples": 0}

    return {
        "overall": overall,
        "critical_zone": critical_zone,
    }


CRITICAL_THRESHOLDS = (10, 15, 20)


def extended_rul_metrics(y_true, y_pred, thresholds=CRITICAL_THRESHOLDS, hits=(10, 20)) -> dict[str, float]:
    """Metrics that complement RMSE, MAE, R2 and the NASA score, for a set of (true, predicted) RUL pairs.

    Error d = predicted - true (positive = too LATE, the dangerous side).
      Bias            mean d (a positive value means the model is optimistic on average).
      Median_AE, P90_AE   median and 90th percentile of |d| (robust to a few very bad engines).
      Hit_within_k    share of predictions with |d| <= k cycles (an operational reading of the error).
    Critical-state classification, for each c in ``thresholds``: the positive class is "true RUL <= c" and the model
    "predicts critical" when its predicted RUL <= c. Precision, recall and F1 are valid here because the positive class is
    rare (accuracy would reward always saying "not critical"). PR_AUC_c uses the continuous score -prediction, so it does
    not depend on the cut c of the prediction, only on the true class; it is NaN when the class has no member.
    """
    truth = np.asarray(y_true, dtype=float)
    pred = np.asarray(y_pred, dtype=float)
    d = pred - truth
    out = {"Bias": float(np.mean(d)), "Median_AE": float(np.median(np.abs(d))), "P90_AE": float(np.percentile(np.abs(d), 90))}
    for k in hits:
        out[f"Hit_within_{k}"] = float(np.mean(np.abs(d) <= k))
    for c in thresholds:
        positive = truth <= c
        called = pred <= c
        tp = float(np.sum(positive & called))
        fp = float(np.sum(~positive & called))
        fn = float(np.sum(positive & ~called))
        precision = tp / (tp + fp) if tp + fp > 0 else float("nan")
        recall = tp / (tp + fn) if tp + fn > 0 else float("nan")
        f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn > 0 else float("nan")
        out[f"Precision_c{c}"], out[f"Recall_c{c}"], out[f"F1_c{c}"] = precision, recall, f1
        out[f"PR_AUC_c{c}"] = float(average_precision_score(positive, -pred)) if positive.any() and not positive.all() else float("nan")
    return out

In [ ]:
%%writefile projeto/src/pdm_cmaps/domain/schema.py
"""Schema of the C-MAPSS files, sensor metadata and project-wide defaults."""

from __future__ import annotations





SENSOR_COLUMNS = [
    "T2", "T24", "T30", "T50", "P2", "P15", "P30", "Nf", "Nc", "epr", "Ps30",
    "phi", "NRf", "NRc", "BPR", "farB", "htBleed", "Nf_dmd", "PCNfR_dmd", "W31", "W32",
]

SETTING_COLUMNS = ["setting_1", "setting_2", "setting_3"]

ID_COLUMNS = ["unit_id", "cycle"]

COLUMNS = [*ID_COLUMNS, *SETTING_COLUMNS, *SENSOR_COLUMNS]

# Sensor metadata. [ASSUMPTION] The drift-direction tuples below are descriptive only: they are
# NOT used to orient the health index (that is done empirically from the data, see
# HealthIndexFuser.fit) because drift directions vary between subsets and fault modes.
POSITIVE_DRIFT_SENSORS: tuple[str, ...] = (
    "T24", "T30", "T50", "Nf", "Nc", "NRf", "BPR", "htBleed",
)

NEGATIVE_DRIFT_SENSORS: tuple[str, ...] = (
    "P30", "Ps30", "phi", "NRc", "W31", "W32",
)

EXOGENOUS_CONSTANT_SENSORS: tuple[str, ...] = (
    "T2", "P2", "P15", "epr", "farB", "Nf_dmd", "PCNfR_dmd",
)

SENSOR_UNITS: dict[str, str] = {
    "T2": "°R",
    "T24": "°R",
    "T30": "°R",
    "T50": "°R",
    "P2": "psia",
    "P15": "psia",
    "P30": "psia",
    "Nf": "rpm",
    "Nc": "rpm",
    "epr": "adim.",
    "Ps30": "psia",
    "phi": "pps/psi",
    "NRf": "rpm",
    "NRc": "rpm",
    "BPR": "adim.",
    "farB": "adim.",
    "htBleed": "°R",
    "Nf_dmd": "rpm",
    "PCNfR_dmd": "%",
    "W31": "lbm/s",
    "W32": "lbm/s",
}

SUBSETS = ("FD001", "FD002", "FD003", "FD004")

DEFAULT_RUL_CAP = 125

DEFAULT_HEALTHY_CYCLE_LIMIT = 50

DEFAULT_RANDOM_STATE = 42

### 2.2 Pacote `pdm_cmaps`: detection

regimes, sensor virtual, cartas EWMA e MEWMA

7 arquivos.

In [ ]:
%%writefile projeto/src/pdm_cmaps/detection/__init__.py
"""Health monitoring without the failure label: regimes, virtual sensor, SPC charts, health index and MEWMA."""

In [ ]:
%%writefile projeto/src/pdm_cmaps/detection/health_index.py
"""Fusion of sensor residuals into a single health index."""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from pdm_cmaps.detection.spc import StatisticalAnomalyDetector
from pdm_cmaps.domain.schema import DEFAULT_HEALTHY_CYCLE_LIMIT, DEFAULT_RANDOM_STATE, NEGATIVE_DRIFT_SENSORS, POSITIVE_DRIFT_SENSORS



class HealthIndexFuser:
    """Multivariate Health Index (HI) synthesis and anomaly monitoring.

    Concept: PCA (principal component analysis) finds the direction of largest variance in
    standardized sensors; a monotone drift of many sensors together shows up as PC1. We map
    PC1 linearly so that the healthy median = 1.0 and the late-life median = 0.0, then smooth
    with an EWMA. Pre-condition: sensors must be regime-standardized (`*_norm`), otherwise PC1
    would capture flight regime, the largest source of variance in FD002/FD004.

    Synthesizes multiple degraded physical sensor channels into an interpretable,
    monotonic Health Index [1.0 (Healthy) -> 0.0 (Failure)] using Principal Component
    Analysis (PCA) calibrated strictly on the healthy window (cycle <= H).
    """

    def __init__(
        self,
        sensors: list[str] | tuple[str, ...] | None = None,
        healthy_cycle_limit: int = DEFAULT_HEALTHY_CYCLE_LIMIT,
        ewma_alpha: float = 0.10,
        min_consecutive_breaches: int = 3,
        threshold_sigma: float = 3.0,
        random_state: int = DEFAULT_RANDOM_STATE,
    ):
        self.sensors = list(sensors) if sensors is not None else [
            s for s in [*POSITIVE_DRIFT_SENSORS, *NEGATIVE_DRIFT_SENSORS]
        ]
        self.healthy_cycle_limit = int(healthy_cycle_limit)
        self.ewma_alpha = float(ewma_alpha)
        self.min_consecutive_breaches = int(min_consecutive_breaches)
        self.threshold_sigma = float(threshold_sigma)
        self.random_state = int(random_state)

        self.pca_: PCA | None = None
        self.scaler_: StandardScaler | None = None
        self.pc1_sign_: float = 1.0
        self.hi_min_: float = 0.0
        self.hi_max_: float = 1.0
        self.calibrated_sensors_: list[str] = []
        self.detector_: StatisticalAnomalyDetector | None = None

    def fit(self, train_frame: pd.DataFrame) -> "HealthIndexFuser":
        """Fit PCA on regime-standardized or raw healthy sensor observations."""
        healthy = train_frame.loc[train_frame["cycle"] <= self.healthy_cycle_limit].copy()
        if healthy.empty:
            healthy = train_frame.copy()

        # Choose standardized columns ({s}_norm) if present, else fallback to raw sensor
        active_cols = []
        for s in self.sensors:
            norm_col = f"{s}_norm"
            if norm_col in healthy.columns and healthy[norm_col].nunique() > 1 and healthy[norm_col].std() > 1e-6:
                active_cols.append(norm_col)
            elif s in healthy.columns and healthy[s].nunique() > 1 and healthy[s].std() > 1e-6:
                active_cols.append(s)

        if not active_cols:
            raise ValueError("No informative sensor channels available to fit HealthIndexFuser")

        self.calibrated_sensors_ = active_cols
        self.scaler_ = StandardScaler()
        X_h = self.scaler_.fit_transform(healthy[active_cols].to_numpy(dtype=float))

        self.pca_ = PCA(n_components=1, random_state=self.random_state)
        self.pca_.fit(X_h)

        # Orient PC1 empirically (not from hard-coded drift tables): HI must decrease with age,
        # so the projection must be negatively correlated with the cycle index on training engines.
        X_all = self.scaler_.transform(train_frame[active_cols].to_numpy(dtype=float))
        unsigned_proj = self.pca_.transform(X_all)[:, 0]
        cycles = train_frame["cycle"].to_numpy(dtype=float)
        if np.std(unsigned_proj) > 1e-12 and np.std(cycles) > 1e-12:
            age_corr = float(np.corrcoef(unsigned_proj, cycles)[0, 1])
        else:
            age_corr = 0.0
        self.pc1_sign_ = -1.0 if age_corr > 0 else 1.0

        # Calibrate baseline bounds on healthy and degraded profiles
        raw_proj = unsigned_proj * self.pc1_sign_
        
        # Upper anchor (healthy phase median) -> HI = 1.0
        h_proj = (self.pca_.transform(X_h)[:, 0]) * self.pc1_sign_
        self.hi_max_ = float(np.median(h_proj))

        # Lower anchor (EOL cycles: max cycle - 5 per engine in train)
        eol_idx = train_frame.groupby("unit_id")["cycle"].transform("max")
        late_mask = train_frame["cycle"] >= np.maximum(self.healthy_cycle_limit, eol_idx - 10)
        late_proj = raw_proj[late_mask.to_numpy()] if late_mask.any() else raw_proj
        self.hi_min_ = float(np.median(late_proj))

        if abs(self.hi_max_ - self.hi_min_) < 1e-6:
            self.hi_min_ = self.hi_max_ - 1.0

        # Calibrate SPC detector on healthy HI deviation from 1.0
        healthy_hi = np.clip((h_proj - self.hi_min_) / (self.hi_max_ - self.hi_min_), 0.0, 1.0)
        healthy_deviation = 1.0 - healthy_hi
        self.detector_ = StatisticalAnomalyDetector(
            threshold_sigma=self.threshold_sigma,
            ewma_alpha=self.ewma_alpha,
            healthy_cycle_limit=self.healthy_cycle_limit,
            min_consecutive_breaches=self.min_consecutive_breaches,
        )
        self.detector_.calibrate(healthy_deviation)

        return self

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        """Compute continuous Health Index and EWMA smoothed HI [1.0 -> 0.0]."""
        if self.pca_ is None or self.scaler_ is None or not self.calibrated_sensors_:
            raise RuntimeError("Fit HealthIndexFuser before transform")

        result = frame.sort_values(["unit_id", "cycle"]).copy().reset_index(drop=True)
        X = self.scaler_.transform(result[self.calibrated_sensors_].to_numpy(dtype=float))
        proj = (self.pca_.transform(X)[:, 0]) * self.pc1_sign_

        # Normalize to [0.0, 1.0]
        denom = self.hi_max_ - self.hi_min_
        raw_hi = np.clip((proj - self.hi_min_) / denom, 0.0, 1.0)
        result["health_index_raw"] = raw_hi

        # Causal EWMA per engine
        ewma_hi = np.empty_like(raw_hi)
        for _, engine in result.groupby("unit_id", sort=False):
            series = pd.Series(engine["health_index_raw"].to_numpy())
            # EWMA smoothing with initial value 1.0 (assumed healthy at t=0)
            smoothed = series.ewm(alpha=self.ewma_alpha, adjust=False).mean().to_numpy()
            ewma_hi[engine.index] = np.clip(smoothed, 0.0, 1.0)

        result["health_index"] = ewma_hi
        return result

    def evaluate(self, train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
        """Fit on train, score on test, and generate fleet health index anomaly summaries."""
        self.fit(train_frame)
        scored = self.transform(test_frame)

        records = []
        for unit_id, engine in scored.groupby("unit_id", sort=False):
            ordered = engine.sort_values("cycle").reset_index(drop=True)
            hi_vals = ordered["health_index"].to_numpy()
            deviation = 1.0 - hi_vals
            z_scores = self.detector_.compute_z_scores(deviation)
            ewma_z = self.detector_.ewma_abs_z(z_scores)
            limits = self.detector_.ewma_upper_limits(len(ewma_z))

            ordered["hi_z_score"] = z_scores
            ordered["hi_ewma_z"] = ewma_z
            ordered["hi_ucl"] = limits

            healthy_mask = ordered["cycle"].to_numpy() <= self.healthy_cycle_limit
            monitoring_mask = ordered["cycle"].to_numpy() > self.healthy_cycle_limit
            breaches = ewma_z >= limits

            far = float(np.mean(breaches[healthy_mask])) if healthy_mask.any() else 0.0
            premature = bool(np.any(breaches[healthy_mask])) if healthy_mask.any() else False

            if self.min_consecutive_breaches > 1:
                online_breaches = breaches & monitoring_mask
                consec = (
                    pd.Series(online_breaches)
                    .rolling(window=self.min_consecutive_breaches, min_periods=self.min_consecutive_breaches)
                    .sum()
                    .to_numpy()
                )
                sustained_ends = np.flatnonzero(consec >= self.min_consecutive_breaches)
                alarm_positions = (
                    sustained_ends - (self.min_consecutive_breaches - 1)
                    if sustained_ends.size
                    else np.array([], dtype=int)
                )
            else:
                alarm_positions = np.flatnonzero(breaches & monitoring_mask)

            eol = int(ordered["cycle"].max())
            alarm_cycle = int(ordered.iloc[alarm_positions[0]]["cycle"]) if alarm_positions.size else None
            lead_time = eol - alarm_cycle if alarm_cycle is not None else 0

            records.append({
                "unit_id": int(unit_id),
                "eol_cycle": eol,
                "first_alarm_cycle": alarm_cycle,
                "lead_time_cycles": lead_time,
                "healthy_far": far,
                "premature_alarm": premature,
                "detected": alarm_cycle is not None,
                "final_health_index": float(hi_vals[-1]),
                "healthy_health_index": float(np.mean(hi_vals[healthy_mask])) if healthy_mask.any() else 1.0,
            })

        per_engine = pd.DataFrame(records)
        detected_engines = per_engine.loc[per_engine["detected"]]
        mean_lead_time_detected = (
            float(detected_engines["lead_time_cycles"].mean()) if not detected_engines.empty else 0.0
        )

        summary = {
            "total_engines": len(per_engine),
            "detection_rate_pct": float(per_engine["detected"].mean() * 100.0),
            "mean_lead_time_cycles": float(per_engine["lead_time_cycles"].mean()),
            "mean_lead_time_detected_cycles": mean_lead_time_detected,
            "mean_healthy_false_alarm_pct": float(per_engine["healthy_far"].mean() * 100.0),
            "premature_alarm_pct": float(per_engine["premature_alarm"].mean() * 100.0),
            "mean_final_health_index": float(per_engine["final_health_index"].mean()),
        }
        return scored, per_engine, summary

In [ ]:
%%writefile projeto/src/pdm_cmaps/detection/multivariate.py
"""Multivariate residual monitor (MEWMA) over 13 sensors."""

from __future__ import annotations

import warnings
import numpy as np
import pandas as pd
from sklearn.base import clone

from pdm_cmaps.detection.virtual_sensor import VirtualSensor
from pdm_cmaps.domain.schema import DEFAULT_HEALTHY_CYCLE_LIMIT, DEFAULT_RANDOM_STATE, SENSOR_COLUMNS, SETTING_COLUMNS



MULTIVARIATE_SENSORS = ("T24", "T30", "T50", "P30", "Ps30", "phi", "Nf", "Nc", "NRc", "BPR", "htBleed", "W31", "W32")

class MultivariateResidualMonitor:
    """MEWMA chart on the standardized residuals of several virtual sensors (Lowry et al., 1992).

    Why: one sensor (T50) may miss a failure mode that shows up elsewhere (e.g. the fan fault in
    FD003/FD004). The MEWMA combines p residuals, accounting for their correlation:
        w_t   = lam * (z_t - mu) + (1 - lam) * w_{t-1},   w_0 = 0
        T2_t  = w_t' Sigma_t^-1 w_t,   Sigma_t = lam / (2 - lam) * (1 - (1 - lam)^(2t)) * Sigma
    [CALIBRATION][LEAKAGE] mu, Sigma and the control limit come from OUT-OF-FOLD healthy residuals of
    training engines; the limit is the empirical (1 - alpha) quantile of T2 on those healthy sequences,
    so no normality is assumed. An alarm needs ``k`` consecutive cycles above the limit after cycle H.
    """

    def __init__(self, sensors=MULTIVARIATE_SENSORS, lam: float = 0.10, alpha: float = 0.0027, k: int = 2,
                 healthy_cycle_limit: int = DEFAULT_HEALTHY_CYCLE_LIMIT, random_state: int = DEFAULT_RANDOM_STATE,
                 estimator=None, mode: str = "exogenous", engine_offset: bool = True, offset_window: int | None = None,
                 calibrate_out_of_window: bool = False, calibration: str = "empirical", synthetic_reps: int = 20,
                 synthetic_horizon: int = 150, synthetic_block: int = 1):
        self.estimator = estimator  # None = the VirtualSensor default (Extra Trees); cloned per sensor
        # "exogenous" (default, thesis detector): expected value from settings + ambient channels only.
        # "cross_sensor": expected value also from the OTHER sensors (a consistency check; a fault that moves
        # all sensors coherently is partly absorbed, an isolated sensor fault is not).
        if mode not in {"exogenous", "cross_sensor"}:
            raise ValueError("mode must be 'exogenous' or 'cross_sensor'")
        self.mode = mode
        self.sensors = tuple(sensors)
        self.lam = float(lam)
        self.alpha = float(alpha)
        self.k = int(k)
        self.healthy_cycle_limit = int(healthy_cycle_limit)
        self.random_state = random_state
        # [DEFAULT ON desde 2026-11-18] engine_offset=True remove de cada resíduo padronizado o nível próprio do motor: a média do z do motor,
        # por regime, nos primeiros W ciclos (W = offset_window, padrão H; mínimo de 3 linhas por regime). Causal: só a janela saudável do
        # próprio motor. É o mesmo tratamento do T50 (ver virtual_sensor.engine_offset); engine_offset=False reproduz o protocolo anterior.
        self.engine_offset = bool(engine_offset)
        self.offset_window = offset_window
        # [DEFAULT OFF] com offset_window < H: μ, Σ e o limite do T² saem só dos ciclos FORA da janela do deslocamento (W, H] (a recursão do T²
        # roda desde o ciclo 1, como no monitoramento), e o falso alarme da janela saudável é medido nesses mesmos ciclos.
        self.calibrate_out_of_window = bool(calibrate_out_of_window)
        # [CALIBRAÇÃO DO LIMITE] "empirical" (padrão, protocolo anterior): quantil de T² nos próprios ciclos da janela, já com o deslocamento tirado
        # DELA, onde a média do z é zero por construção; o T² sai subestimado, o limite fica baixo e o falso alarme DEPOIS da janela sobe
        # (0,20 a 2,64 por 100 ciclos contra 0,01 a 0,06 sem o deslocamento). "synthetic": reproduz o erro de estimar o deslocamento. Para cada
        # motor de treino sorteia, com reposição (em blocos de `synthetic_block` ciclos), séries saudáveis de W + `synthetic_horizon` ciclos
        # a partir dos z brutos da janela dele, estima o deslocamento nos primeiros W ciclos exatamente como no monitoramento, roda a recursão
        # do T² e toma o quantil (1 − alpha) só dos ciclos DEPOIS da janela. Só usa motores de treino (sem vazamento).
        if calibration not in {"empirical", "synthetic"}:
            raise ValueError("calibration must be 'empirical' or 'synthetic'")
        self.calibration = calibration
        self.synthetic_reps = int(synthetic_reps)
        self.synthetic_horizon = int(synthetic_horizon)
        self.synthetic_block = int(synthetic_block)

    def _out_of_window(self) -> bool:
        return bool(self.engine_offset and self.calibrate_out_of_window and self._offset_w() < self.healthy_cycle_limit)

    def residual_z(self, vs: "VirtualSensor", rows: pd.DataFrame) -> np.ndarray:
        """z do resíduo de um sensor com o deslocamento do motor removido (se ligado), sem renormalizar: é o que entra no T². ``rows`` = saída de ``vs.predict``."""
        return self._remove_offset(self._z(vs, rows).to_numpy(), rows)

    def chart_z(self, vs: "VirtualSensor", rows: pd.DataFrame) -> np.ndarray:
        """z do gráfico EWMA de um sensor isolado: o de ``residual_z`` na escala em que ``vs.ewma_detector`` foi calibrado."""
        z = self.residual_z(vs, rows)
        return z / vs.engine_z_scale_ if self.engine_offset else z

    def offset_uncovered_pct(self, frame: pd.DataFrame) -> float:
        """% das linhas monitoradas (ciclo > H) cujo (motor, regime) tem menos de 3 linhas na janela do deslocamento e fica sem correção."""
        w = self._offset_w()
        win = frame.loc[frame["cycle"] <= w].groupby(["unit_id", "regime_id"]).size()
        mon = frame.loc[frame["cycle"] > self.healthy_cycle_limit]
        idx = pd.MultiIndex.from_arrays([mon["unit_id"].to_numpy(), mon["regime_id"].to_numpy()])
        return float(100 * (win.reindex(idx).fillna(0).to_numpy() < 3).mean()) if len(mon) else float("nan")

    def _offset_w(self) -> int:
        return int(self.offset_window if self.offset_window is not None else self.healthy_cycle_limit)

    def _remove_offset(self, z: np.ndarray, keys: pd.DataFrame) -> np.ndarray:
        """Subtrai de ``z`` a média do próprio motor (por regime) nos primeiros W ciclos; ``keys`` traz unit_id, cycle e regime_id alinhados a ``z``."""
        if not self.engine_offset:
            return z
        df = pd.DataFrame({"unit_id": keys["unit_id"].to_numpy(), "regime_id": keys["regime_id"].to_numpy(), "cycle": keys["cycle"].to_numpy(), "z": np.asarray(z, dtype=float)})
        win = df.loc[df["cycle"] <= self._offset_w()].groupby(["unit_id", "regime_id"])["z"].agg(["mean", "size"])
        off = win["mean"].where(win["size"] >= 3, 0.0)
        idx = pd.MultiIndex.from_arrays([df["unit_id"].to_numpy(), df["regime_id"].to_numpy()])
        return df["z"].to_numpy() - off.reindex(idx).fillna(0.0).to_numpy()

    def _z(self, vs: "VirtualSensor", rows: pd.DataFrame) -> pd.Series:
        out = pd.Series(np.nan, index=rows.index)
        for regime_id, part in rows.groupby("regime_id"):
            det = vs.detectors.get(int(regime_id), vs.fallback_detector)
            out.loc[part.index] = det.compute_z_scores(part["residual"].to_numpy())
        return out

    def _t2(self, z: np.ndarray, units: np.ndarray) -> np.ndarray:
        lam, mu = self.lam, self.mu_
        out = np.empty(len(z))
        start = 0
        for _, n in pd.Series(units).groupby(units, sort=False).size().items():
            w = np.zeros(z.shape[1])
            for t in range(n):
                w = lam * (z[start + t] - mu) + (1 - lam) * w
                factor = lam / (2 - lam) * (1 - (1 - lam) ** (2 * (t + 1)))
                out[start + t] = float(w @ self.sigma_inv_ @ w) / factor
            start += n
        return out

    def fit(self, train_frame: pd.DataFrame) -> "MultivariateResidualMonitor":
        """``train_frame`` = training engines with ``regime_id`` (assigned by a train-fitted RegimeDecoupler)."""
        self.sensors_ = [s for s in self.sensors if s in train_frame.columns and train_frame[s].std() > 1e-6]
        self.sensors_all_ = [s for s in SENSOR_COLUMNS if s in train_frame.columns and train_frame[s].std() > 1e-6]
        self.virtual_ = {}
        cal = None
        cal_raw = None
        cal_regime = None
        for sensor in self.sensors_:
            cross = None
            if self.mode == "cross_sensor":
                cross = [*SETTING_COLUMNS, *[c for c in self.sensors_all_ if c != sensor]]
            vs = VirtualSensor(target_sensor=sensor, mode="exogenous", feature_cols=cross, healthy_cycle_limit=self.healthy_cycle_limit,
                               random_state=self.random_state, estimator=clone(self.estimator) if self.estimator is not None else None,
                               engine_offset=self.engine_offset, offset_window=self.offset_window,
                               calibrate_out_of_window=self.calibrate_out_of_window)  # o T² só usa fit/predict/detectors; o limite da EWMA do sensor isolado sai no mesmo tratamento
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                vs.fit(train_frame)
            self.virtual_[sensor] = vs
            rows = vs.calibration_predictions_.set_index(["unit_id", "cycle"])
            flat = rows.reset_index()
            raw = self._z(vs, flat).to_numpy()
            z = self._remove_offset(raw, flat)
            col = pd.Series(z, index=rows.index, name=sensor)
            cal = col.to_frame() if cal is None else cal.join(col, how="inner")
            rawcol = pd.Series(raw, index=rows.index, name=sensor)
            cal_raw = rawcol.to_frame() if cal_raw is None else cal_raw.join(rawcol, how="inner")
            if cal_regime is None:
                cal_regime = pd.Series(flat["regime_id"].to_numpy(), index=rows.index, name="regime_id")
        cal = cal.sort_index()
        self.calibration_z_ = cal  # resíduos corrigidos da calibração (fora da amostra), um por sensor; usados também pelo detector de Isolation Forest
        zc = cal.to_numpy()
        cyc_cal = cal.index.get_level_values("cycle").to_numpy()
        keep = cyc_cal > self._offset_w() if self._out_of_window() else np.ones(len(zc), dtype=bool)
        self.mu_ = zc[keep].mean(axis=0)
        sigma = np.cov(zc[keep], rowvar=False) + 1e-6 * np.eye(zc.shape[1])
        self.sigma_inv_ = np.linalg.inv(sigma)
        t2 = self._t2(zc, cal.index.get_level_values("unit_id").to_numpy())
        self.ucl_ = float(np.quantile(t2[keep], 1 - self.alpha))
        self.ucl_empirical_ = self.ucl_
        self.synthetic_t2_ = None
        self.synthetic_t2_cycles_ = None
        if self.calibration == "synthetic":
            self.synthetic_t2_cycles_ = self._synthetic_post_window_t2(cal_raw.sort_index(), cal_regime.reindex(cal_raw.sort_index().index), by_cycle=True)
            self.synthetic_t2_ = self.synthetic_t2_cycles_.ravel()  # limite do HORIZONTE inteiro (usado no monitoramento)
            self.ucl_ = float(np.quantile(self.synthetic_t2_, 1 - self.alpha))
        # kept for calibration-noise studies (engine-cluster bootstrap of the UCL, healthy_window_study.py)
        self.calibration_t2_ = pd.DataFrame({"unit_id": cal.index.get_level_values("unit_id").to_numpy(), "t2": t2})[keep].reset_index(drop=True)
        return self

    def _synthetic_post_window_t2(self, raw: pd.DataFrame, regime: pd.Series, by_cycle: bool = False) -> np.ndarray:
        """T² dos ciclos DEPOIS da janela em séries saudáveis sintéticas (ver `calibration="synthetic"`). Um T² por ciclo pós-janela, de todos os
        motores de treino × réplicas; o limite para qualquer alpha é `np.quantile(resultado, 1 - alpha)`.

        `by_cycle=True` devolve a matriz (séries, ciclos pós-janela) em vez do vetor achatado: logo depois da janela a EWMA ainda carrega a memória da
        janela (onde o z é centrado por construção) e o T² cresce até o regime estacionário (~1 − 0,9^t com λ = 0,10); um limite medido só nos primeiros
        ciclos pós-janela é menor que o do horizonte inteiro."""
        rng = np.random.default_rng(self.random_state)
        W, L, b, p = self._offset_w(), self.synthetic_horizon, max(1, self.synthetic_block), raw.shape[1]
        N = W + L
        seqs = []
        for _, g in raw.groupby(level="unit_id", sort=False):
            Z = g.to_numpy(dtype=float)
            R = regime.reindex(g.index).to_numpy()
            n = len(Z)
            if n < max(10, b):
                continue
            for _rep in range(self.synthetic_reps):
                starts = rng.integers(0, n - b + 1, size=-(-N // b))
                idx = (starts[:, None] + np.arange(b)[None, :]).ravel()[:N]
                z, r = Z[idx].copy(), R[idx]
                for reg in np.unique(r[:W]):  # deslocamento por regime nos primeiros W ciclos, mínimo de 3 linhas (como em _remove_offset)
                    in_win = r[:W] == reg
                    if in_win.sum() >= 3:
                        z[r == reg] -= z[:W][in_win].mean(axis=0)
                seqs.append(z)
        if not seqs:
            raise ValueError("nenhum motor de treino com ciclos suficientes para a calibração sintética")
        Zb = np.stack(seqs)  # (S, N, p)
        w = np.zeros((Zb.shape[0], p))
        out = np.empty((Zb.shape[0], N))
        for t in range(N):
            w = self.lam * (Zb[:, t, :] - self.mu_) + (1 - self.lam) * w
            factor = self.lam / (2 - self.lam) * (1 - (1 - self.lam) ** (2 * (t + 1)))
            out[:, t] = np.einsum("sp,pq,sq->s", w, self.sigma_inv_, w) / factor
        return out[:, W:] if by_cycle else out[:, W:].ravel()

    def score(self, frame: pd.DataFrame) -> pd.DataFrame:
        """Per-cycle T2 statistic and breach flag for engines in ``frame`` (needs ``regime_id``)."""
        cols = {}
        index = None
        for sensor, vs in self.virtual_.items():
            pred = vs.predict(frame).sort_values(["unit_id", "cycle"]).reset_index(drop=True)
            cols[sensor] = self.residual_z(vs, pred)
            index = pred[["unit_id", "cycle"]]
        out = index.copy()
        z = np.column_stack([cols[s] for s in self.virtual_])
        out["t2"] = self._t2(z, out["unit_id"].to_numpy())
        out["ucl"] = self.ucl_
        out["breach"] = out["t2"] >= self.ucl_
        return out

    def alarms(self, scored: pd.DataFrame) -> tuple[dict[int, int | None], pd.DataFrame, dict]:
        """First persistent alarm per engine + fleet summary with the same metrics as the univariate SPC."""
        H, k = self.healthy_cycle_limit, self.k
        first, rows = {}, []
        for unit, eng in scored.groupby("unit_id"):
            cyc = eng["cycle"].to_numpy()
            b = eng["breach"].to_numpy(dtype=bool)
            run, alarm = 0, None
            for i in range(len(b)):
                run = run + 1 if (b[i] and cyc[i] > H) else 0
                if run >= k:
                    alarm = int(cyc[i - k + 1])
                    break
            healthy = cyc <= H
            if self._out_of_window():
                healthy = healthy & (cyc > self._offset_w())  # falso alarme só fora da janela que estima o deslocamento
            first[int(unit)] = alarm
            rows.append({"unit_id": int(unit), "eol_cycle": int(cyc.max()), "first_alarm_cycle": alarm,
                         "lead_time_cycles": int(cyc.max()) - alarm if alarm is not None else 0,
                         "healthy_far": float(b[healthy].mean()) if healthy.any() else 0.0,
                         "premature_alarm": bool(b[healthy].any()) if healthy.any() else False,
                         "detected": alarm is not None})
        per = pd.DataFrame(rows)
        det = per.loc[per["detected"]]
        summary = {
            "total_engines": len(per),
            "detection_rate_pct": float(per["detected"].mean() * 100),
            "mean_lead_time_cycles": float(per["lead_time_cycles"].mean()),
            "mean_lead_time_detected_cycles": float(det["lead_time_cycles"].mean()) if len(det) else 0.0,
            "mean_healthy_false_alarm_pct": float(per["healthy_far"].mean() * 100),
            "premature_alarm_pct": float(per["premature_alarm"].mean() * 100),
            "n_sensors": len(self.virtual_),
        }
        return first, per, summary

In [ ]:
%%writefile projeto/src/pdm_cmaps/detection/regime.py
"""Health monitoring: operating-regime decoupling, virtual sensors and SPC envelope anomaly detection.

WHAT   Detects that an engine is degrading WITHOUT using the failure label, in 4 steps:
         1. RegimeDecoupler       - cluster flight conditions (unsupervised, KMeans).
         2. VirtualSensor         - predict what a sensor SHOULD read given the flight
                                    condition; residual = actual - expected.
         3. StatisticalAnomalyDetector / EnvelopeAnomalyDetector - statistical process
                                    control (SPC): raise an alarm when residuals leave the
                                    band seen while the engine was healthy.
         4. HealthIndexFuser      - fuse many sensors into one 1.0 -> 0.0 health curve (PCA).
WHY    Thesis claim: in multi-regime data the flight regime, not wear, dominates sensor
       variance. If we remove the regime effect, what remains (the residual) is a clean
       degradation signal that gives an early warning (lead time before end of life).
CONCEPTS
  [CALIBRATION]  thresholds come from the healthy window (first H cycles) of TRAINING engines.
  [ASSUMPTION]   the first H=50 cycles are healthy; real engines can start with some wear.
  [LEAKAGE]      calibration residuals are OUT-OF-FOLD (GroupKFold) so the model never scores
                 the engines it was trained on; otherwise residuals look too small and
                 thresholds too tight (false alarms later).
  [BASELINE]     the virtual sensor is deliberately simple (exogenous inputs only); inside a
                 regime it behaves roughly like a regime mean.
  SPC vocabulary: z-score, EWMA (exponentially weighted moving average), control limit,
  false-alarm rate (FAR), Western-Electric persistence rule, lead time."""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

from pdm_cmaps.domain.schema import DEFAULT_RANDOM_STATE, SETTING_COLUMNS



class RegimeDecoupler:
    """Fit operating-condition clusters on training rows and assign stable ordered IDs.

    Concept: KMeans (unsupervised learning) groups rows by the 3 operating settings. In FD002/
    FD004 the settings take 6 distinct combinations, so K=6 recovers the flight regimes.
    Why standardize first: KMeans uses Euclidean distance, so altitude (tens of thousands) would
    otherwise drown Mach and throttle. [LEAKAGE] scaler and centroids are fit on training rows
    only; test rows are merely assigned to the nearest centroid.
    Why sort the ids: KMeans labels are arbitrary per run; ordering by physical centre makes
    regime 0..5 mean the same thing across seeds and subsets.
    """

    def __init__(self, n_clusters: int = 6, random_state: int = DEFAULT_RANDOM_STATE):
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
        self.scaler = StandardScaler()
        self.cluster_to_regime_id: dict[int, int] = {}
        self.regime_sensor_stats_: dict[int, dict[str, tuple[float, float]]] = {}

    def evaluate_k_sweep(self, train_frame: pd.DataFrame, k_range=range(2, 10)) -> pd.DataFrame:
        """Evaluate clustering quality metrics across a range of k values on training settings."""
        settings = self.scaler.fit_transform(train_frame[SETTING_COLUMNS].to_numpy(dtype=float))
        records = []
        for k in k_range:
            km = KMeans(n_clusters=k, random_state=self.random_state, n_init=10).fit(settings)
            labels = km.labels_
            sil = float(silhouette_score(settings, labels)) if k > 1 and len(np.unique(labels)) > 1 else np.nan
            ch = float(calinski_harabasz_score(settings, labels)) if k > 1 and len(np.unique(labels)) > 1 else np.nan
            gmm = GaussianMixture(n_components=k, random_state=self.random_state, max_iter=100).fit(settings)
            bic = float(gmm.bic(settings))
            records.append({
                "k": k,
                "silhouette": sil,
                "calinski_harabasz": ch,
                "bic": bic,
            })
        return pd.DataFrame(records)

    def fit(self, train_frame: pd.DataFrame) -> "RegimeDecoupler":
        """Fit KMeans using operating settings from the training partition and sort lexicographically."""
        settings = self.scaler.fit_transform(train_frame[SETTING_COLUMNS].to_numpy(dtype=float))
        self.kmeans.fit(settings)
        centers_phys = self.scaler.inverse_transform(self.kmeans.cluster_centers_)
        # Lexicographical sorting on physical coordinates: (round(alt, 0), round(mach, 2), round(TRA, 0))
        order = sorted(
            range(self.n_clusters),
            key=lambda idx: (
                round(float(centers_phys[idx, 0]), 0),
                round(float(centers_phys[idx, 1]), 2),
                round(float(centers_phys[idx, 2]), 0),
            ),
        )
        self.cluster_to_regime_id = {int(cluster): rank for rank, cluster in enumerate(order)}
        return self

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        """Copy rows and add their nearest fitted ``regime_id``."""
        if not self.cluster_to_regime_id:
            raise RuntimeError("Fit the regime model before transforming rows")
        result = frame.copy()
        clusters = self.kmeans.predict(self.scaler.transform(result[SETTING_COLUMNS].to_numpy(dtype=float)))
        result["regime_id"] = [self.cluster_to_regime_id[int(cluster)] for cluster in clusters]
        return result

    def fit_transform(self, train_frame: pd.DataFrame) -> pd.DataFrame:
        """Fit on the supplied training rows and return their regime labels."""
        return self.fit(train_frame).transform(train_frame)

    def fit_sensor_scaler(self, train_frame: pd.DataFrame, sensors: list[str]) -> "RegimeDecoupler":
        """Compute and store per-regime mean and standard deviation for the given sensors on training data."""
        if "regime_id" not in train_frame:
            raise ValueError("train_frame must contain regime_id")
        self.regime_sensor_stats_ = {}
        for r_id, group in train_frame.groupby("regime_id"):
            stats_dict = {}
            for sensor in sensors:
                vals = group[sensor].to_numpy(dtype=float)
                mean_val = float(np.mean(vals))
                std_val = float(np.std(vals))
                if std_val < 1e-8:
                    std_val = 1.0
                stats_dict[sensor] = (mean_val, std_val)
            self.regime_sensor_stats_[int(r_id)] = stats_dict
        return self

    def standardize_sensors(self, frame: pd.DataFrame, sensors: list[str], suffix: str = "_norm") -> pd.DataFrame:
        """Standardize sensor readings using fitted training regime statistics."""
        if not self.regime_sensor_stats_:
            raise RuntimeError("Call fit_sensor_scaler before standardize_sensors")
        if "regime_id" not in frame:
            raise ValueError("frame must contain regime_id")
        result = frame.copy()
        for sensor in sensors:
            norm_col = f"{sensor}{suffix}"
            vals = result[sensor].to_numpy(dtype=float)
            r_ids = result["regime_id"].to_numpy(dtype=int)
            out = np.empty_like(vals)
            for i in range(len(vals)):
                r_id = r_ids[i]
                if r_id in self.regime_sensor_stats_ and sensor in self.regime_sensor_stats_[r_id]:
                    m, s = self.regime_sensor_stats_[r_id][sensor]
                    out[i] = (vals[i] - m) / s
                else:
                    out[i] = vals[i]
            result[norm_col] = out
        return result

    def distance_margin(self, frame: pd.DataFrame) -> pd.DataFrame:
        """Return nearest/runner-up centroid distances and their ambiguity margin."""
        if not self.cluster_to_regime_id:
            raise RuntimeError("Fit the regime model before measuring distances")
        distances = self.kmeans.transform(self.scaler.transform(frame[SETTING_COLUMNS].to_numpy(dtype=float)))
        ranked = np.sort(distances, axis=1)
        nearest = ranked[:, 0]
        runner_up = ranked[:, 1]
        abs_margin = runner_up - nearest
        rel_margin = 1.0 - (nearest / (runner_up + 1e-9))
        return pd.DataFrame({
            "nearest_distance": nearest,
            "runner_up_distance": runner_up,
            "distance_margin": abs_margin,
            "relative_margin": np.clip(rel_margin, 0.0, 1.0),
        }, index=frame.index)

    @staticmethod
    def variance_decomposition(frame: pd.DataFrame, target: str, reference_frame: pd.DataFrame | None = None) -> dict:
        """Decompose variance; optionally score fixed regime means from reference_frame OOS.

        Concept (law of total variance): Var(y) = Var(E[y|regime]) + E[Var(y|regime)], i.e.
        between-regime + within-regime. A high between-regime share is the evidence for the
        thesis claim "regime dominates this sensor". The *descriptive* shares use regime means
        computed on the same data (in-sample); `oos_r2` uses means from `reference_frame`
        (training engines) and is the honest, out-of-sample version.
        """
        if "regime_id" not in frame:
            raise ValueError("frame must contain regime_id; assign regimes using a training-fitted model")
        values = frame[target].to_numpy(dtype=float)
        total = float(np.var(values))
        descriptive_means = frame.groupby("regime_id")[target].mean()
        fitted = frame["regime_id"].map(descriptive_means).to_numpy(dtype=float)
        between = float(np.var(fitted))
        within = float(np.var(values - fitted))
        baseline_means = (reference_frame if reference_frame is not None else frame).groupby("regime_id")[target].mean()
        baseline_prediction = frame["regime_id"].map(baseline_means).to_numpy(dtype=float)
        if not np.isfinite(baseline_prediction).all():
            raise ValueError("reference_frame must provide a mean for every regime in frame")
        baseline_rmse = float(np.sqrt(np.mean((values - baseline_prediction) ** 2)))
        summaries = frame.groupby("regime_id").agg(
            n_samples=(target, "size"), mean_target=(target, "mean"),
            std_target=(target, lambda values: values.std(ddof=0)),
        ).reset_index()
        return {
            "target": target,
            "total_var": total,
            "total_sd": float(np.std(values)),
            "between_regime_var": between,
            "within_regime_var": within,
            "pct_between_regime": between / total * 100.0 if total else 0.0,
            "pct_within_regime": within / total * 100.0 if total else 0.0,
            "Between_regime_pct": between / total * 100.0 if total else 0.0,
            "Within_regime_pct": within / total * 100.0 if total else 0.0,
            "descriptive_between_pct": between / total * 100.0 if total else 0.0,
            "descriptive_within_pct": within / total * 100.0 if total else 0.0,
            "regime_baseline_r2": 1.0 - float(np.mean((values - baseline_prediction) ** 2)) / total if total else 0.0,
            "regime_baseline_rmse": baseline_rmse,
            "oos_r2": 1.0 - float(np.mean((values - baseline_prediction) ** 2)) / total if total else 0.0,
            "oos_rmse": baseline_rmse,
            "baseline_evaluation": "out_of_sample" if reference_frame is not None else "in_sample",
            "clusters": summaries.to_dict(orient="records"),
        }

    @staticmethod
    def decompose_sensors(frame: pd.DataFrame, sensors: list[str]) -> pd.DataFrame:
        """Return same-population variance shares for several sensors."""
        rows = []
        for sensor in sensors:
            result = RegimeDecoupler.variance_decomposition(frame, sensor)
            rows.append({
                "Sensor": sensor,
                "Total_variance": result["total_var"],
                "Between_regime_variance": result["between_regime_var"],
                "Within_regime_variance": result["within_regime_var"],
                "Between_regime_pct": result["pct_between_regime"],
                "Within_regime_pct": result["pct_within_regime"],
            })
        return pd.DataFrame(rows)

In [ ]:
%%writefile projeto/src/pdm_cmaps/detection/spc.py
"""Univariate statistical process control: EWMA/3-sigma and envelope detectors."""

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats

from pdm_cmaps.domain.schema import DEFAULT_HEALTHY_CYCLE_LIMIT



class StatisticalAnomalyDetector:
    """Calibrate residual location/scale and summarize per-engine EWMA alarms.

    Statistical process control in four lines:
      1. z = (residual - mean_healthy) / std_healthy       (how unusual is this reading?)
      2. EWMA of |z| smooths noise: s_t = a*|z_t| + (1-a)*s_{t-1}   (a = 0.10 -> long memory)
      3. Alarm when the EWMA exceeds mean + k*sigma of its own in-control distribution (k=3).
      4. Persistence (Western-Electric style): require `min_consecutive_breaches` in a row, so a
         single noisy spike is not an alarm. Trade-off: fewer false alarms, slightly later alarm.
    [CALIBRATION] mean/std come from healthy residuals only. Alarms are only counted AFTER
    the healthy window (cycle > H): that is the "online monitoring" phase.
    Lead time = end-of-life cycle - first alarm cycle (bigger is better, if alarms are real).
    """

    def __init__(
        self,
        threshold_sigma: float = 3.0,
        ewma_alpha: float = 0.10,
        healthy_cycle_limit: int = DEFAULT_HEALTHY_CYCLE_LIMIT,
        min_consecutive_breaches: int = 3,
    ):
        if threshold_sigma <= 0 or not 0 < ewma_alpha <= 1:
            raise ValueError("threshold_sigma must be positive and ewma_alpha in (0, 1]")
        if min_consecutive_breaches < 1:
            raise ValueError("min_consecutive_breaches must be >= 1")
        self.threshold_sigma = threshold_sigma
        self.ewma_alpha = ewma_alpha
        self.healthy_cycle_limit = healthy_cycle_limit
        self.min_consecutive_breaches = int(min_consecutive_breaches)
        self.mu_e = 0.0
        self.sigma_e = 1.0
        self.abs_z_mean = np.sqrt(2.0 / np.pi)
        self.abs_z_variance = 1.0 - 2.0 / np.pi

    def calibrate(self, healthy_residuals) -> "StatisticalAnomalyDetector":
        """Estimate mean and standard deviation from healthy calibration residuals."""
        residuals = np.asarray(healthy_residuals, dtype=float)
        if residuals.size == 0 or not np.isfinite(residuals).all():
            raise ValueError("healthy_residuals must be a non-empty finite array")
        self.mu_e = float(residuals.mean())
        self.sigma_e = float(residuals.std())
        if self.sigma_e < 1e-8:
            self.sigma_e = 1.0
        self.calibrate_ewma_distribution(self.compute_z_scores(residuals))
        return self

    def calibrate_ewma_distribution(self, z_scores) -> "StatisticalAnomalyDetector":
        """Estimate the in-control distribution of ``|Z|`` from calibration scores."""
        absolute_scores = np.abs(np.asarray(z_scores, dtype=float))
        if absolute_scores.size == 0 or not np.isfinite(absolute_scores).all():
            raise ValueError("z_scores must be a non-empty finite array")
        self.abs_z_mean = float(absolute_scores.mean())
        self.abs_z_variance = float(absolute_scores.var())
        if self.abs_z_variance < 1e-12:
            self.abs_z_variance = 1e-12
        return self

    def compute_z_scores(self, residuals) -> np.ndarray:
        """Standardize residuals against the fitted healthy baseline."""
        return (np.asarray(residuals, dtype=float) - self.mu_e) / self.sigma_e

    def ewma_upper_limits(self, n: int) -> np.ndarray:
        """Finite-horizon control limits from the calibrated distribution of ``|Z|``."""
        t = np.arange(1, n + 1, dtype=float)
        alpha = self.ewma_alpha
        variance = self.abs_z_variance * alpha / (2.0 - alpha) * (1.0 - (1.0 - alpha) ** (2.0 * t))
        return self.abs_z_mean + self.threshold_sigma * np.sqrt(variance)

    def ewma_abs_z(self, z_scores) -> np.ndarray:
        """Compute EWMA(|Z|) from its calibrated in-control mean."""
        values = np.abs(np.asarray(z_scores, dtype=float))
        result = np.empty_like(values)
        state = self.abs_z_mean
        for i, value in enumerate(values):
            state = self.ewma_alpha * value + (1.0 - self.ewma_alpha) * state
            result[i] = state
        return result

    @staticmethod
    def test_residual_normality(residuals) -> dict[str, float]:
        """Perform Shapiro-Wilk and D'Agostino-Pearson tests on healthy residuals."""
        res = np.asarray(residuals, dtype=float)
        res_sample = res[: min(5000, len(res))]
        shapiro_stat, shapiro_p = stats.shapiro(res_sample)
        norm_stat, norm_p = stats.normaltest(res_sample)
        return {
            "shapiro_stat": float(shapiro_stat),
            "shapiro_p": float(shapiro_p),
            "dagostino_stat": float(norm_stat),
            "dagostino_p": float(norm_p),
        }

    def calibrate_empirical_ucl(self, z_fleet_scores, alpha_level: float = 0.01) -> float:
        """Calibrate non-parametric empirical UCL from healthy fleet absolute z-scores."""
        abs_z = np.abs(np.asarray(z_fleet_scores, dtype=float))
        if abs_z.size == 0:
            return float(self.abs_z_mean + self.threshold_sigma * np.sqrt(self.abs_z_variance))
        return float(np.quantile(abs_z, 1.0 - alpha_level))

    def score_engine(self, engine_frame: pd.DataFrame, residual_col: str = "residual") -> tuple[pd.DataFrame, dict]:
        """Add Z/EWMA columns and return first operational alarm (cycle > H) and healthy FAR."""
        ordered = engine_frame.sort_values("cycle").copy().reset_index(drop=True)
        z = self.compute_z_scores(ordered[residual_col].to_numpy())
        ewma = self.ewma_abs_z(z)
        limits = self.ewma_upper_limits(len(ewma))

        healthy_mask = ordered["cycle"].to_numpy() <= self.healthy_cycle_limit
        monitoring_mask = ordered["cycle"].to_numpy() > self.healthy_cycle_limit

        # Check breaches: raw Z breach (Shewhart) and EWMA breach
        raw_breaches = np.abs(z) >= self.threshold_sigma
        ewma_breaches = ewma >= limits
        breaches = raw_breaches & ewma_breaches

        far = float(np.mean(ewma_breaches[healthy_mask])) if healthy_mask.any() else 0.0
        premature = bool(np.any(ewma_breaches[healthy_mask])) if healthy_mask.any() else False

        # Operational alarms are strictly identified during online monitoring (cycle > healthy_cycle_limit)
        # using Western Electric persistence rule (min_consecutive_breaches consecutive cycles)
        if self.min_consecutive_breaches > 1:
            online_breaches = breaches & monitoring_mask
            consec = (
                pd.Series(online_breaches)
                .rolling(window=self.min_consecutive_breaches, min_periods=self.min_consecutive_breaches)
                .sum()
                .to_numpy()
            )
            sustained_ends = np.flatnonzero(consec >= self.min_consecutive_breaches)
            online_alarms = sustained_ends - (self.min_consecutive_breaches - 1) if sustained_ends.size else np.array([], dtype=int)
            persistent_mask = np.zeros(len(ewma), dtype=bool)
            for end_idx in sustained_ends:
                start_idx = max(0, end_idx - self.min_consecutive_breaches + 1)
                persistent_mask[start_idx : end_idx + 1] = True
        else:
            online_alarms = np.flatnonzero(breaches & monitoring_mask)
            persistent_mask = breaches & monitoring_mask

        first_cycle = int(ordered.iloc[online_alarms[0]]["cycle"]) if online_alarms.size else None
        eol = int(ordered["cycle"].max())
        lead_time = eol - first_cycle if first_cycle is not None else 0

        ordered["z_score"] = z
        ordered["ewma_abs_z"] = ewma
        ordered["ewma_ucl"] = limits
        ordered["persistent_alarm"] = persistent_mask

        summary = {
            "unit_id": int(ordered["unit_id"].iloc[0]),
            "eol_cycle": eol,
            "first_alarm_cycle": first_cycle,
            "lead_time_cycles": lead_time,
            "healthy_far": far,
            "healthy_far_pct": far * 100.0,
            "premature_alarm": premature,
            "premature_alarm_pct": 100.0 if premature else 0.0,
            "detected": first_cycle is not None,
        }
        return ordered, summary

    def evaluate_fleet(self, fleet_frame: pd.DataFrame, residual_col: str = "residual") -> tuple[pd.DataFrame, pd.DataFrame, dict]:
        """Return scored residual rows, per-engine results, and fleet summary."""
        scored = []
        summaries = []
        for _, engine in fleet_frame.groupby("unit_id", sort=False):
            engine_scores, summary = self.score_engine(engine, residual_col)
            scored.append(engine_scores)
            summaries.append(summary)
        per_engine = pd.DataFrame(summaries)
        detected_engines = per_engine.loc[per_engine["detected"]]
        mean_lead_time_detected = float(detected_engines["lead_time_cycles"].mean()) if not detected_engines.empty else 0.0

        return pd.concat(scored, ignore_index=True), per_engine, {
            "total_engines": len(per_engine),
            "detection_rate_pct": float(per_engine["detected"].mean() * 100.0),
            "mean_lead_time_cycles": float(per_engine["lead_time_cycles"].mean()),
            "mean_lead_time_detected_cycles": mean_lead_time_detected,
            "mean_healthy_false_alarm_pct": float(per_engine["healthy_far"].mean() * 100.0),
            "premature_alarm_pct": float(per_engine["premature_alarm"].mean() * 100.0),
        }

class EnvelopeAnomalyDetector:
    """Envelope / 'embrulho' anomaly detector for time series and digital signals.

    Implements two complementary envelope strategies:
    1. Time Series Prediction Envelope (Power BI-style prediction intervals):
       Upper/lower prediction boundaries based on healthy calibration residual variance:
       Upper(t) = y_pred(t) + k * sigma
       Lower(t) = y_pred(t) - k * sigma
       Points outside this shaded 'embrulho' represent structural anomalies.

    2. Geometric Signal Envelope (Rolling peak/valley hull):
       Wraps signal peaks and valleys using causal rolling extrema:
       Upper_Hull(t) = max_{tau in [t-w, t]} y(tau)
       Lower_Hull(t) = min_{tau in [t-w, t]} y(tau)
       Monitors amplitude expansion (modulation) beyond healthy mechanical limits.
    """

    def __init__(
        self,
        k_sigma: float = 3.0,
        healthy_cycle_limit: int = DEFAULT_HEALTHY_CYCLE_LIMIT,
        window_size: int = 5,
        min_consecutive_breaches: int = 3,
    ):
        if k_sigma <= 0:
            raise ValueError("k_sigma must be positive")
        if window_size < 1:
            raise ValueError("window_size must be positive")
        self.k_sigma = float(k_sigma)
        self.healthy_cycle_limit = int(healthy_cycle_limit)
        self.window_size = int(window_size)
        self.min_consecutive_breaches = int(min_consecutive_breaches)

        self.mu_e = 0.0
        self.sigma_e = 1.0
        self.mu_signal = 0.0
        self.sigma_signal = 1.0
        self.mu_amp = 0.0
        self.sigma_amp = 1.0
        self.mu_res_amp = 0.0
        self.sigma_res_amp = 1.0
        self.mu_res_abs = 0.0
        self.sigma_res_abs = 1.0

    def calibrate(self, healthy_actual, healthy_pred) -> "EnvelopeAnomalyDetector":
        """Estimate residual and geometric envelope baselines from healthy data."""
        actual = np.asarray(healthy_actual, dtype=float)
        pred = np.asarray(healthy_pred, dtype=float)
        if actual.size == 0 or pred.size == 0 or actual.shape != pred.shape:
            raise ValueError("healthy_actual and healthy_pred must be non-empty matching arrays")

        residuals = actual - pred
        self.mu_e = float(np.mean(residuals))
        self.sigma_e = float(np.std(residuals))
        if self.sigma_e < 1e-8:
            self.sigma_e = 1.0

        self.mu_signal = float(np.mean(actual))
        self.sigma_signal = float(np.std(actual))
        if self.sigma_signal < 1e-8:
            self.sigma_signal = 1.0

        series = pd.Series(actual)
        roll_max = series.rolling(window=self.window_size, min_periods=1).max().to_numpy()
        roll_min = series.rolling(window=self.window_size, min_periods=1).min().to_numpy()
        amps = roll_max - roll_min
        self.mu_amp = float(np.mean(amps))
        self.sigma_amp = float(np.std(amps))
        if self.sigma_amp < 1e-8:
            self.sigma_amp = 1.0

        # Calibrate hull amplitude/magnitude on the regime-decoupled RESIDUAL (not the raw signal):
        # on the raw multi-regime signal the hull would mostly measure regime switching.
        res_series = pd.Series(residuals)
        res_roll_max = res_series.rolling(window=self.window_size, min_periods=1).max().to_numpy()
        res_roll_min = res_series.rolling(window=self.window_size, min_periods=1).min().to_numpy()
        res_amps = res_roll_max - res_roll_min
        self.mu_res_amp = float(np.mean(res_amps))
        self.sigma_res_amp = float(np.std(res_amps))
        if self.sigma_res_amp < 1e-8:
            self.sigma_res_amp = 1.0
        abs_res = np.abs(residuals - self.mu_e)
        self.mu_res_abs = float(np.mean(abs_res))
        self.sigma_res_abs = float(np.std(abs_res))
        if self.sigma_res_abs < 1e-8:
            self.sigma_res_abs = 1.0

        return self

    def score_time_series_envelope(
        self,
        df: pd.DataFrame,
        actual_col: str,
        pred_col: str,
        cycle_col: str = "cycle",
        unit_col: str = "unit_id",
    ) -> tuple[pd.DataFrame, dict]:
        """Compute the prediction envelope corridor, flag breaches, and compute lead time to breakage."""
        ordered = df.sort_values(cycle_col).copy().reset_index(drop=True)
        actual = ordered[actual_col].to_numpy(dtype=float)
        pred = ordered[pred_col].to_numpy(dtype=float)

        upper = pred + (self.mu_e + self.k_sigma * self.sigma_e)
        lower = pred + (self.mu_e - self.k_sigma * self.sigma_e)

        out_upper = actual > upper
        out_lower = actual < lower
        out_of_envelope = out_upper | out_lower

        breach_dist = np.maximum(0.0, actual - upper) + np.maximum(0.0, lower - actual)

        ordered["envelope_upper"] = upper
        ordered["envelope_lower"] = lower
        ordered["out_of_envelope"] = out_of_envelope
        ordered["breach_distance"] = breach_dist

        # Operational alarms are identified during online monitoring (cycle > healthy_cycle_limit)
        is_monitoring = ordered[cycle_col].to_numpy() > self.healthy_cycle_limit
        if self.min_consecutive_breaches > 1:
            consecutive = pd.Series(out_of_envelope).rolling(window=self.min_consecutive_breaches, min_periods=self.min_consecutive_breaches).sum()
            alarm_mask = (consecutive.to_numpy() >= self.min_consecutive_breaches) & is_monitoring
            alarm_indices = np.flatnonzero(alarm_mask)
            # Offset to the first cycle of the consecutive sequence
            alarm_indices = alarm_indices - (self.min_consecutive_breaches - 1) if alarm_indices.size else np.array([])
        else:
            alarm_mask = out_of_envelope & is_monitoring
            alarm_indices = np.flatnonzero(alarm_mask)

        eol_cycle = int(ordered[cycle_col].max())
        first_alarm = int(ordered.iloc[alarm_indices[0]][cycle_col]) if len(alarm_indices) > 0 else None
        lead_time = eol_cycle - first_alarm if first_alarm is not None else 0

        healthy_mask = ordered[cycle_col].to_numpy() <= self.healthy_cycle_limit
        healthy_breaches = int(np.sum(out_of_envelope[healthy_mask])) if healthy_mask.any() else 0
        healthy_far = float(np.mean(out_of_envelope[healthy_mask]) * 100.0) if healthy_mask.any() else 0.0

        summary = {
            "unit_id": int(ordered[unit_col].iloc[0]) if unit_col in ordered else 0,
            "eol_cycle": eol_cycle,
            "first_alarm_cycle": first_alarm,
            "lead_time_cycles": lead_time,
            "detected": first_alarm is not None,
            "total_cycles": len(ordered),
            "out_of_envelope_count": int(np.sum(out_of_envelope)),
            "in_envelope_pct": float(np.mean(~out_of_envelope) * 100.0),
            "healthy_far_pct": healthy_far,
            "healthy_breaches": healthy_breaches,
            "envelope_margin": float(self.k_sigma * self.sigma_e),
        }
        return ordered, summary

    def score_geometric_envelope(
        self,
        df: pd.DataFrame,
        signal_col: str,
        cycle_col: str = "cycle",
        unit_col: str = "unit_id",
        use_decoupled_signal: bool = True,
    ) -> tuple[pd.DataFrame, dict]:
        """Compute rolling peak-valley geometric envelope, modulation, and physical breaches.

        When ``use_decoupled_signal=True`` and a regime-standardized column (``{signal_col}_norm``)
        or virtual sensor residual/prediction is available in ``df``, the rolling hull is computed
        on the regime-decoupled series rather than the raw multi-regime signal, which would
        mostly measure regime switching instead of degradation.
        """
        ordered = df.sort_values(cycle_col).copy().reset_index(drop=True)
        raw_signal = ordered[signal_col].to_numpy(dtype=float)

        norm_col = f"{signal_col}_norm"
        if use_decoupled_signal and "residual" in ordered.columns:
            # Option B: Decoupled virtual sensor residual e(t) = y(t) - y_hat(t)
            signal = ordered["residual"].to_numpy(dtype=float)
            amp_threshold = self.mu_res_amp + self.k_sigma * self.sigma_res_amp
            upper_ceiling = abs(self.mu_e) + self.k_sigma * self.sigma_e
            hull_max_abs = (
                pd.Series(np.abs(signal - self.mu_e))
                .rolling(window=self.window_size, min_periods=1)
                .max()
                .to_numpy()
            )
            roll_max = pd.Series(signal).rolling(window=self.window_size, min_periods=1).max().to_numpy()
            roll_min = pd.Series(signal).rolling(window=self.window_size, min_periods=1).min().to_numpy()
            amp = roll_max - roll_min
            geom_breach = (amp > amp_threshold) | (hull_max_abs > upper_ceiling)
            decoupled_mode = "residual"
        elif use_decoupled_signal and ("prediction" in ordered.columns or "pred" in ordered.columns):
            pred_col_name = "prediction" if "prediction" in ordered.columns else "pred"
            signal = raw_signal - ordered[pred_col_name].to_numpy(dtype=float)
            amp_threshold = self.mu_res_amp + self.k_sigma * self.sigma_res_amp
            upper_ceiling = abs(self.mu_e) + self.k_sigma * self.sigma_e
            hull_max_abs = (
                pd.Series(np.abs(signal - self.mu_e))
                .rolling(window=self.window_size, min_periods=1)
                .max()
                .to_numpy()
            )
            roll_max = pd.Series(signal).rolling(window=self.window_size, min_periods=1).max().to_numpy()
            roll_min = pd.Series(signal).rolling(window=self.window_size, min_periods=1).min().to_numpy()
            amp = roll_max - roll_min
            geom_breach = (amp > amp_threshold) | (hull_max_abs > upper_ceiling)
            decoupled_mode = "residual"
        elif use_decoupled_signal and norm_col in ordered.columns:
            # Option A: Regime-standardized Z_k(t)
            signal = ordered[norm_col].to_numpy(dtype=float)
            healthy_mask = ordered[cycle_col].to_numpy() <= self.healthy_cycle_limit
            h_sig = signal[healthy_mask] if healthy_mask.any() else signal
            h_rmax = pd.Series(h_sig).rolling(window=self.window_size, min_periods=1).max().to_numpy()
            h_rmin = pd.Series(h_sig).rolling(window=self.window_size, min_periods=1).min().to_numpy()
            h_amp = h_rmax - h_rmin
            amp_threshold = float(np.mean(h_amp) + self.k_sigma * max(float(np.std(h_amp)), 0.1))
            upper_ceiling = float(self.k_sigma)
            roll_max = pd.Series(signal).rolling(window=self.window_size, min_periods=1).max().to_numpy()
            roll_min = pd.Series(signal).rolling(window=self.window_size, min_periods=1).min().to_numpy()
            amp = roll_max - roll_min
            hull_max_abs = np.maximum(np.abs(roll_max), np.abs(roll_min))
            geom_breach = (amp > amp_threshold) | (hull_max_abs > upper_ceiling)
            decoupled_mode = "regime_z"
        else:
            signal = raw_signal
            roll_max = pd.Series(signal).rolling(window=self.window_size, min_periods=1).max().to_numpy()
            roll_min = pd.Series(signal).rolling(window=self.window_size, min_periods=1).min().to_numpy()
            amp = roll_max - roll_min
            amp_threshold = self.mu_amp + self.k_sigma * self.sigma_amp
            upper_ceiling = self.mu_signal + self.k_sigma * self.sigma_signal
            lower_ceiling = self.mu_signal - self.k_sigma * self.sigma_signal
            geom_breach = (amp > amp_threshold) | (roll_max > upper_ceiling) | (roll_min < lower_ceiling)
            decoupled_mode = "raw"

        ordered["geom_signal"] = signal
        ordered["geom_upper_hull"] = roll_max
        ordered["geom_lower_hull"] = roll_min
        ordered["geom_amplitude"] = amp
        ordered["geom_breach"] = geom_breach

        is_monitoring = ordered[cycle_col].to_numpy() > self.healthy_cycle_limit
        if self.min_consecutive_breaches > 1:
            online_breaches = geom_breach & is_monitoring
            consec = (
                pd.Series(online_breaches)
                .rolling(window=self.min_consecutive_breaches, min_periods=self.min_consecutive_breaches)
                .sum()
                .to_numpy()
            )
            sustained_ends = np.flatnonzero(consec >= self.min_consecutive_breaches)
            alarm_indices = (
                sustained_ends - (self.min_consecutive_breaches - 1)
                if sustained_ends.size
                else np.array([], dtype=int)
            )
        else:
            alarm_indices = np.flatnonzero(geom_breach & is_monitoring)

        eol_cycle = int(ordered[cycle_col].max())
        first_alarm = int(ordered.iloc[alarm_indices[0]][cycle_col]) if len(alarm_indices) > 0 else None
        lead_time = eol_cycle - first_alarm if first_alarm is not None else 0

        summary = {
            "unit_id": int(ordered[unit_col].iloc[0]) if unit_col in ordered else 0,
            "eol_cycle": eol_cycle,
            "first_alarm_cycle": first_alarm,
            "lead_time_cycles": lead_time,
            "detected": first_alarm is not None,
            "amp_threshold": float(amp_threshold),
            "upper_ceiling": float(upper_ceiling),
            "decoupled_mode": decoupled_mode,
        }
        return ordered, summary

In [ ]:
%%writefile projeto/src/pdm_cmaps/detection/virtual_sensor.py
"""Regime-conditioned virtual sensor and its residual."""

from __future__ import annotations

import warnings
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import GroupKFold

from pdm_cmaps.detection.spc import StatisticalAnomalyDetector
from pdm_cmaps.domain.schema import DEFAULT_HEALTHY_CYCLE_LIMIT
from pdm_cmaps.modeling.registry import get_model



CANONICAL_EXOGENOUS_FEATURES = [
    "setting_1", "setting_2", "setting_3", "T2", "P2", "P15", "epr", "Nf_dmd", "PCNfR_dmd"
]

class VirtualSensor:
    """Fit one regression model per operating regime using healthy training rows.

    Concept: a "virtual" (soft) sensor predicts a physical reading from OTHER inputs. Here the
    inputs are exogenous only (settings + ambient channels that degradation does not move), so
    the prediction answers "what would a HEALTHY engine read under these conditions?".
    The residual (actual - predicted) is therefore ~0 + noise when healthy and drifts when the
    hardware wears. [ASSUMPTION] Because those inputs are near-constant inside a regime, the
    model is close to a per-regime mean; the value of the method is regime removal, not
    sophisticated regression.
    [DEFAULT] ``engine_offset=True`` (padrão desde a adoção do deslocamento do motor): o z de cada motor é centrado na média dos seus
    primeiros W ciclos (``offset_window``, padrão H) e a escala é renormalizada, porque o σ agrupado da frota inclui o deslocamento
    entre motores. Com ``engine_offset=False`` volta o comportamento anterior (μ e σ agrupados da frota).
    Design choices: one model per regime (+ global fallback when a regime has <30 rows or <2
    engines); out-of-fold residuals for calibration ([LEAKAGE]); ExtraTrees as a cheap,
    non-linear, low-tuning default.
    """

    def __init__(
        self,
        target_sensor: str = "T50",
        feature_cols: list[str] | None = None,
        mode: str = "exogenous",
        threshold_sigma: float = 3.0,
        ewma_alpha: float = 0.10,
        healthy_cycle_limit: int = DEFAULT_HEALTHY_CYCLE_LIMIT,
        min_consecutive_breaches: int = 3,
        estimator=None,
        random_state: int = 42,
        engine_offset: bool = True,
        offset_window: int | None = None,
        calibrate_out_of_window: bool = False,
    ):
        self.target_sensor = target_sensor
        self.feature_cols = feature_cols
        self.mode = mode
        self.threshold_sigma = threshold_sigma
        self.ewma_alpha = ewma_alpha
        self.healthy_cycle_limit = healthy_cycle_limit
        self.min_consecutive_breaches = int(min_consecutive_breaches)
        self.estimator = estimator if estimator is not None else get_model(
            "extra_trees", n_estimators=60, max_depth=12, random_state=random_state
        )
        self.engine_offset = bool(engine_offset)
        self.offset_window = offset_window
        # [DEFAULT OFF] com offset_window < healthy_cycle_limit, a escala e o limite da EWMA são calibrados só nos ciclos FORA da janela que
        # estima o deslocamento (W, H], e o falso alarme da janela saudável também é medido ali: a calibração vê o mesmo erro de estimativa
        # do deslocamento que o monitoramento depois de H verá (com a janela inteira, o z calibrado é centrado nela e o falso alarme sai otimista).
        self.calibrate_out_of_window = bool(calibrate_out_of_window)
        self.engine_z_scale_ = 1.0
        self.models: dict[int, object] = {}
        self.detectors: dict[int, StatisticalAnomalyDetector] = {}
        self.ewma_detector: StatisticalAnomalyDetector | None = None
        self.calibration_predictions_: pd.DataFrame | None = None
        self.fallback_model: object | None = None
        self.fallback_detector: StatisticalAnomalyDetector | None = None

    def _offset_w(self) -> int:
        """Janela (em ciclos) para estimar o deslocamento de cada motor; padrão: a janela saudável inteira."""
        return int(self.offset_window if self.offset_window is not None else self.healthy_cycle_limit)

    _MIN_OFFSET_ROWS = 3

    def _out_of_window(self) -> bool:
        return bool(self.engine_offset and self.calibrate_out_of_window and self._offset_w() < self.healthy_cycle_limit)

    def _engine_regime_offsets(self, frame: pd.DataFrame) -> pd.Series:
        """Deslocamento de cada (motor, regime) em unidades de z: média do z nos primeiros W ciclos do motor naquele regime.

        Por regime porque o σ de cada regime é diferente (a mesma diferença física vira z diferente) e misturar regimes numa média única
        distorcia a previsão nos sensores de pouca variância. Regimes com menos de 3 linhas na janela ficam sem correção (0)."""
        win = frame.loc[frame["cycle"] <= self._offset_w()].groupby(["unit_id", "regime_id"])["z_score"].agg(["mean", "size"])
        return win["mean"].where(win["size"] >= self._MIN_OFFSET_ROWS, 0.0)

    @staticmethod
    def _lookup_offsets(frame: pd.DataFrame, offsets: pd.Series) -> pd.Series:
        idx = pd.MultiIndex.from_arrays([frame["unit_id"].to_numpy(), frame["regime_id"].to_numpy()])
        return pd.Series(offsets.reindex(idx).fillna(0.0).to_numpy(), index=frame.index)

    def fit(self, train_frame: pd.DataFrame) -> "VirtualSensor":
        """Fit regime-specific estimators on cycles within the healthy window with hierarchical fallback."""
        if "regime_id" not in train_frame:
            raise ValueError("train_frame must have regime_id assigned by a train-fitted regime model")
        if self.feature_cols is not None:
            features = list(self.feature_cols)
        elif self.mode == "exogenous":
            features = [c for c in CANONICAL_EXOGENOUS_FEATURES if c in train_frame.columns and c != self.target_sensor]
            if not features:
                features = [
                    column for column in train_frame.columns
                    if column not in {self.target_sensor, "unit_id", "cycle", "regime_id", "RUL", "subpart"}
                ]
        else:
            features = [
                column for column in train_frame.columns
                if column not in {self.target_sensor, "unit_id", "cycle", "regime_id", "RUL", "subpart"}
            ]
        self.feature_cols = list(features)
        healthy = train_frame.loc[train_frame["cycle"] <= self.healthy_cycle_limit]
        if healthy.empty:
            healthy = train_frame

        # Train global hierarchical fallback model across all healthy data
        self.fallback_model = clone(self.estimator)
        self.fallback_model.fit(healthy[self.feature_cols], healthy[self.target_sensor].to_numpy(dtype=float))
        fb_resid = healthy[self.target_sensor].to_numpy(dtype=float) - self.fallback_model.predict(healthy[self.feature_cols])
        self.fallback_detector = StatisticalAnomalyDetector(
            threshold_sigma=self.threshold_sigma,
            ewma_alpha=self.ewma_alpha,
            healthy_cycle_limit=self.healthy_cycle_limit,
            min_consecutive_breaches=self.min_consecutive_breaches,
        )
        self.fallback_detector.calibrate(fb_resid)

        self.models = {}
        self.detectors = {}
        calibration_parts = []

        all_regimes = sorted(train_frame["regime_id"].unique())
        for regime_id in all_regimes:
            rows = healthy.loc[healthy["regime_id"] == regime_id]
            groups = rows["unit_id"].to_numpy() if not rows.empty else np.array([])
            n_engines = np.unique(groups).size if groups.size else 0

            if len(rows) < 30 or n_engines < 2:
                warnings.warn(
                    f"Regime {regime_id} has {len(rows)} samples and {n_engines} engines (< 30 samples or < 2 engines); "
                    "using global hierarchical fallback estimator.",
                    UserWarning,
                )
                self.models[int(regime_id)] = self.fallback_model
                self.detectors[int(regime_id)] = self.fallback_detector
                if not rows.empty:
                    oof_rows = rows.copy()
                    pred = self.fallback_model.predict(rows[self.feature_cols])
                    oof_rows["prediction"] = pred
                    oof_rows["residual"] = rows[self.target_sensor] - pred
                    calibration_parts.append(oof_rows)
                continue

            n_splits = min(5, n_engines)
            oof_prediction = np.full(len(rows), np.nan)
            reg_features = rows[self.feature_cols]
            target = rows[self.target_sensor].to_numpy(dtype=float)

            for train_idx, valid_idx in GroupKFold(n_splits=n_splits).split(reg_features, target, groups):
                oof_model = clone(self.estimator)
                oof_model.fit(reg_features.iloc[train_idx], target[train_idx])
                oof_prediction[valid_idx] = oof_model.predict(reg_features.iloc[valid_idx])

            if not np.isfinite(oof_prediction).all():
                nan_mask = ~np.isfinite(oof_prediction)
                oof_prediction[nan_mask] = self.fallback_model.predict(reg_features.iloc[nan_mask])

            model = clone(self.estimator)
            model.fit(reg_features, target)
            self.models[int(regime_id)] = model
            oof_rows = rows.copy()
            oof_rows["prediction"] = oof_prediction
            oof_rows["residual"] = target - oof_prediction
            calibration_parts.append(oof_rows)
            detector = StatisticalAnomalyDetector(
                threshold_sigma=self.threshold_sigma,
                ewma_alpha=self.ewma_alpha,
                healthy_cycle_limit=self.healthy_cycle_limit,
                min_consecutive_breaches=self.min_consecutive_breaches,
            )
            detector.calibrate(oof_rows["residual"].to_numpy())
            self.detectors[int(regime_id)] = detector

        if not self.models:
            raise ValueError("No regime could be fitted or calibrated")

        if calibration_parts:
            self.calibration_predictions_ = pd.concat(calibration_parts, ignore_index=True).sort_values(["unit_id", "cycle"]).reset_index(drop=True)
            calibration_z = []
            self.calibration_predictions_["z_score"] = np.nan
            for regime_id, rows in self.calibration_predictions_.groupby("regime_id", sort=False):
                det = self.detectors.get(int(regime_id), self.fallback_detector)
                z_scores = det.compute_z_scores(rows["residual"].to_numpy())
                self.calibration_predictions_.loc[rows.index, "z_score"] = z_scores
                calibration_z.append(z_scores)
            if self.engine_offset:
                # [LEAKAGE] o deslocamento de cada motor de treino usa só os primeiros W ciclos (W <= H); depois dele a escala
                # volta a ter desvio 1 para que o limiar de 3 sigma e o portão bruto mantenham o significado.
                cp = self.calibration_predictions_
                off = self._engine_regime_offsets(cp)
                z_adj = cp["z_score"] - self._lookup_offsets(cp, off)
                if self._out_of_window():
                    z_adj = z_adj[cp["cycle"] > self._offset_w()]
                self.engine_z_scale_ = max(float(z_adj.std()), 1e-8)
                calibration_z = [(z_adj / self.engine_z_scale_).to_numpy()]
            self.ewma_detector = StatisticalAnomalyDetector(
                threshold_sigma=self.threshold_sigma,
                ewma_alpha=self.ewma_alpha,
                healthy_cycle_limit=self.healthy_cycle_limit,
                min_consecutive_breaches=self.min_consecutive_breaches,
            )
            self.ewma_detector.calibrate_ewma_distribution(np.concatenate(calibration_z))
        else:
            self.ewma_detector = self.fallback_detector

        return self

    def predict(self, frame: pd.DataFrame) -> pd.DataFrame:
        """Return rows supported by a fitted regime model with prediction/residual columns."""
        if not self.models and self.fallback_model is None:
            raise RuntimeError("Fit the virtual sensor before predicting")
        if "regime_id" not in frame:
            raise ValueError("frame must have regime_id assigned")
        parts = []
        for regime_id, rows in frame.groupby("regime_id"):
            model = self.models.get(int(regime_id), self.fallback_model)
            part = rows.copy()
            part["prediction"] = model.predict(part[self.feature_cols])
            part["residual"] = part[self.target_sensor] - part["prediction"]
            parts.append(part)
        if not parts:
            return frame.iloc[0:0].assign(prediction=pd.Series(dtype=float), residual=pd.Series(dtype=float))
        result = pd.concat(parts).sort_values(["unit_id", "cycle"]).reset_index(drop=True)
        return result

    def evaluate(self, train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
        """Fit on training engines and score lifecycle anomalies on held-out engines."""
        self.fit(train_frame)
        lifecycle_predictions = self.predict(test_frame)
        calibrated_parts = []
        for regime_id, rows in lifecycle_predictions.groupby("regime_id", sort=False):
            detector = self.detectors.get(int(regime_id), self.fallback_detector)
            part = rows.copy()
            part["z_score"] = detector.compute_z_scores(part["residual"].to_numpy())
            calibrated_parts.append(part)
        if not calibrated_parts:
            raise ValueError("No held-out healthy rows were available to calibrate residuals")
        scored = pd.concat(calibrated_parts, ignore_index=True).sort_values(["unit_id", "cycle"])
        if self.engine_offset:
            # deslocamento do motor = média do z nos primeiros W ciclos dele (causal: o alarme só vale depois da janela saudável)
            off_row = self._lookup_offsets(scored, self._engine_regime_offsets(scored))
            sigma_row = scored["regime_id"].map(lambda r: self.detectors.get(int(r), self.fallback_detector).sigma_e).astype(float)
            scored["prediction"] = scored["prediction"] + off_row * sigma_row
            scored["residual"] = scored["residual"] - off_row * sigma_row
            scored["z_score"] = (scored["z_score"] - off_row) / self.engine_z_scale_
        if self.ewma_detector is None:
            raise RuntimeError("Fit the virtual sensor before scoring EWMA")
        scored["ewma_abs_z"] = np.nan
        records = []
        for unit_id, engine in scored.groupby("unit_id", sort=False):
            ordered = engine.sort_values("cycle")
            ewma = self.ewma_detector.ewma_abs_z(ordered["z_score"].to_numpy())
            scored.loc[ordered.index, "ewma_abs_z"] = ewma
            limits = self.ewma_detector.ewma_upper_limits(len(ewma))
            scored.loc[ordered.index, "ewma_ucl"] = limits

            healthy_mask = ordered["cycle"].to_numpy() <= self.healthy_cycle_limit
            if self._out_of_window():
                healthy_mask = healthy_mask & (ordered["cycle"].to_numpy() > self._offset_w())  # só fora da janela que estima o deslocamento
            monitoring_mask = ordered["cycle"].to_numpy() > self.healthy_cycle_limit

            far = float(np.mean(ewma[healthy_mask] >= limits[healthy_mask])) if healthy_mask.any() else 0.0
            premature = bool(np.any(ewma[healthy_mask] >= limits[healthy_mask])) if healthy_mask.any() else False

            # Online alarms restricted to cycle > healthy_cycle_limit with persistence
            raw_breaches_bool = np.abs(ordered["z_score"].to_numpy()) >= self.threshold_sigma
            breaches = (ewma >= limits) & raw_breaches_bool
            if self.min_consecutive_breaches > 1:
                online_breaches = breaches & monitoring_mask
                consec = (
                    pd.Series(online_breaches)
                    .rolling(window=self.min_consecutive_breaches, min_periods=self.min_consecutive_breaches)
                    .sum()
                    .to_numpy()
                )
                sustained_ends = np.flatnonzero(consec >= self.min_consecutive_breaches)
                online_alarm_positions = (
                    sustained_ends - (self.min_consecutive_breaches - 1)
                    if sustained_ends.size
                    else np.array([], dtype=int)
                )
            else:
                online_alarm_positions = np.flatnonzero(breaches & monitoring_mask)

            eol = int(ordered["cycle"].max())
            alarm_cycle = int(ordered.iloc[online_alarm_positions[0]]["cycle"]) if online_alarm_positions.size else None
            lead_time = eol - alarm_cycle if alarm_cycle is not None else 0

            records.append({
                "unit_id": int(unit_id),
                "eol_cycle": eol,
                "first_alarm_cycle": alarm_cycle,
                "lead_time_cycles": lead_time,
                "healthy_far": far,
                "premature_alarm": premature,
                "detected": alarm_cycle is not None,
            })
        per_engine = pd.DataFrame(records)
        detected_engines = per_engine.loc[per_engine["detected"]]
        mean_lead_time_detected = float(detected_engines["lead_time_cycles"].mean()) if not detected_engines.empty else 0.0

        summary = {
            "total_engines": len(per_engine),
            "detection_rate_pct": float(per_engine["detected"].mean() * 100.0),
            "mean_lead_time_cycles": float(per_engine["lead_time_cycles"].mean()),
            "mean_lead_time_detected_cycles": mean_lead_time_detected,
            "mean_healthy_false_alarm_pct": float(per_engine["healthy_far"].mean() * 100.0),
            "premature_alarm_pct": float(per_engine["premature_alarm"].mean() * 100.0),
        }
        return scored.reset_index(drop=True), per_engine, summary

In [ ]:
%%writefile projeto/src/pdm_cmaps/detection/virtual_sensor_metrics.py
"""R² do sensor virtual e a régua do deslocamento entre motores.

Por que existe: em subconjunto de regime único as entradas exógenas quase não variam, então o sensor virtual vira a média
dos motores de treino e o R² fica perto de 0 (negativo quando a degradação afasta a leitura). Boa parte da variância que
sobra é o deslocamento de cada motor em relação à média da frota (desgaste inicial e fabricação). Estas métricas separam
as causas sem mudar o detector, que já padroniza o resíduo por motor (μ_e, σ_e da janela saudável).

Janela saudável H é dividida ao meio: os primeiros H//2 ciclos de cada motor calibram, os H//2 seguintes avaliam. Assim o
baseline do motor e o deslocamento são causais (só usam o passado) e comparáveis com o modelo atual na mesma janela.

Colunas devolvidas por (sensor, modelo):
- r2_holdout           R² em todos os ciclos dos motores de teste (negativo é esperado com degradação).
- r2_healthy           R² nos ciclos 1 a H.
- r2_late_model        R² do modelo atual nos ciclos H//2+1 a H.
- r2_late_motor_base   R² de prever a média do próprio motor nos ciclos 1 a H//2 (baseline do motor).
- r2_late_model_offset R² do modelo atual somado ao deslocamento médio do motor nos ciclos 1 a H//2.
- pct_between_engine   % da variância na janela 1 a H que é diferença de média entre motores.
- r2_late_within_regime  R² do modelo atual nos ciclos H//2+1 a H com a variância medida DENTRO do regime (SST em torno da média de
                         cada regime). Em regime único é igual a r2_late_model; em multirregime tira o que o regime explica de graça
                         (o R² bruto chega a ~1,00 só porque o regime domina a variância) e fica comparável com o regime único.
- r2_holdout_within_regime  o mesmo, em todos os ciclos dos motores de teste.
- noise_ceiling_late   teto de R² nos ciclos H//2+1 a H: 1 − σ²_ruído / Var(y dentro do regime), com σ² = média de Δy²/2 só entre
                       ciclos CONSECUTIVOS DO MESMO REGIME (o salto de regime não é ruído). Estimado nos motores de teste.
- frac_ceiling         r2_late_within_regime / noise_ceiling_late (NaN se o teto for ≤ 0): quanto do que era possível o modelo pegou.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

TRACE_REQUIRED = ("sensor", "model", "unit_id", "cycle", "actual", "prediction")


def r2(y, p) -> float:
    """R² = 1 - SSR/SST; NaN quando a leitura é constante (SST = 0) ou há menos de 2 pontos."""
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    ok = ~(np.isnan(y) | np.isnan(p))
    y, p = y[ok], p[ok]
    if len(y) < 2:
        return float("nan")
    sst = float(((y - y.mean()) ** 2).sum())
    return float("nan") if sst <= 0 else 1.0 - float(((y - p) ** 2).sum()) / sst


def error_metrics(group: pd.DataFrame) -> tuple[float, float, float]:
    """(MAE, RMSE, NRMSE dentro do regime) de um trace; o NRMSE divide o RMSE pelo desvio dos valores reais em torno da média do próprio regime.

    O ganho sobre a média do regime (1 - MSE / MSE da média do regime) é exatamente o R² dentro do regime (`r2_holdout_within_regime`),
    porque a soma de quadrados em torno da média de cada regime é o erro de quem prevê sempre a média do regime."""
    w = group.dropna(subset=["actual", "prediction"])
    if len(w) < 3:
        return float("nan"), float("nan"), float("nan")
    err = (w["actual"] - w["prediction"]).to_numpy(dtype=float)
    reg = w["regime_id"] if "regime_id" in w else pd.Series(0, index=w.index)
    sd = float(np.sqrt(((w["actual"] - w.groupby(reg)["actual"].transform("mean")) ** 2).mean()))
    rmse = float(np.sqrt(np.mean(err ** 2)))
    return float(np.mean(np.abs(err))), rmse, (rmse / sd if sd > 0 else float("nan"))


def _between_engine_pct(window: pd.DataFrame) -> float:
    total = float(((window["actual"] - window["actual"].mean()) ** 2).sum())
    if total <= 0 or window["unit_id"].nunique() < 2:
        return float("nan")
    means = window.groupby("unit_id")["actual"].transform("mean")
    return 100.0 * float(((means - window["actual"].mean()) ** 2).sum()) / total


def _within_regime(window: pd.DataFrame) -> tuple[float, float, float]:
    """(R² dentro do regime, teto de ruído, fração do teto) de uma janela; regime único se não houver regime_id."""
    w = window.dropna(subset=["actual", "prediction"])
    if len(w) < 3:
        return float("nan"), float("nan"), float("nan")
    reg = w["regime_id"] if "regime_id" in w else pd.Series(0, index=w.index)
    sst = float(((w["actual"] - w.groupby(reg)["actual"].transform("mean")) ** 2).sum())
    if sst <= 0:
        return float("nan"), float("nan"), float("nan")
    r2w = 1.0 - float(((w["actual"] - w["prediction"]) ** 2).sum()) / sst
    # ruído: só pares de ciclos consecutivos do mesmo motor E do mesmo regime
    w = w.assign(_reg=reg).sort_values(["unit_id", "cycle"])
    dy = w.groupby("unit_id")["actual"].diff()
    same = (w.groupby("unit_id")["cycle"].diff() == 1) & (w.groupby("unit_id")["_reg"].diff() == 0)
    d = dy[same].dropna()
    if len(d) < 5:
        return r2w, float("nan"), float("nan")
    ceiling = 1.0 - float((d ** 2).mean() / 2.0) / (sst / len(w))
    return r2w, ceiling, (r2w / ceiling if ceiling > 0 else float("nan"))


def _one(group: pd.DataFrame, healthy_limit: int) -> pd.Series:
    cal_end = healthy_limit // 2
    cal, late = group[group["cycle"] <= cal_end], group[(group["cycle"] > cal_end) & (group["cycle"] <= healthy_limit)]
    base = cal.groupby("unit_id")["actual"].mean().rename("base")
    off = (cal["actual"] - cal["prediction"]).groupby(cal["unit_id"]).mean().rename("off")
    late = late.join(base, on="unit_id").join(off, on="unit_id").dropna(subset=["base", "off"])
    r2_w_late, ceiling, frac = _within_regime(late)
    return pd.Series({
        "mae_holdout": error_metrics(group)[0],
        "nrmse_within_regime_holdout": error_metrics(group)[2],
        "r2_holdout": r2(group["actual"], group["prediction"]),
        "r2_healthy": r2(group.loc[group["cycle"] <= healthy_limit, "actual"], group.loc[group["cycle"] <= healthy_limit, "prediction"]),
        "r2_late_model": r2(late["actual"], late["prediction"]),
        "r2_late_motor_base": r2(late["actual"], late["base"]),
        "r2_late_model_offset": r2(late["actual"], late["prediction"] + late["off"]),
        "pct_between_engine": _between_engine_pct(group[group["cycle"] <= healthy_limit]),
        "r2_late_within_regime": r2_w_late,
        "r2_holdout_within_regime": _within_regime(group)[0],
        "noise_ceiling_late": ceiling,
        "frac_ceiling": frac,
    })


def virtual_sensor_r2(trace: pd.DataFrame, healthy_limit: int) -> pd.DataFrame:
    """Métricas de R² por (sensor, modelo) a partir do trace (colunas sensor, model, unit_id, cycle, actual, prediction)."""
    missing = [c for c in TRACE_REQUIRED if c not in trace.columns]
    if missing:
        raise ValueError(f"trace sem as colunas {missing}")
    rows = []
    for (sensor, model), g in trace.groupby(["sensor", "model"], sort=False):
        rows.append({"sensor": sensor, "model": model, **_one(g, healthy_limit).to_dict()})
    return pd.DataFrame(rows)

### 2.2 Pacote `pdm_cmaps`: modeling

modelos de RUL (árvores, redes, linear)

9 arquivos.

In [ ]:
%%writefile projeto/src/pdm_cmaps/modeling/__init__.py
"""RUL models: supervised regression of Remaining Useful Life from sensor readings.

Three families, all compared on the same engine-level splits (see benchmark.py):
  A. Tabular (one row -> RUL): Ridge, SVM, RandomForest, ExtraTrees, XGBoost, LightGBM, CatBoost, MLP.
     Cheap, strong baselines. Built by registry.get_model_pool; gbdt.py holds the XGBoost NASA-loss variant.
  B. Sequence (last 15 cycles -> RUL): LSTM/GRU, 1-D CNN, Transformer (sequences.py, networks.py, torch_models.py).
  C. MiniRocket + RidgeCV (minirocket.py): fixed random convolution kernels turn a window into features.

Rules every model here must respect (tests in tests/test_modeling.py):
  [BASELINE]   always compare against "predict the mean"; a model that cannot beat it learned nothing.
  [GROUPED-CV] GroupKFold by engine (registry.grouped_oof, benchmark.RULBenchmark.cross_validate).
  [LEAKAGE]    scalers are fit on the training fold only; sequences never cross engines;
               early stopping uses a held-out set of ENGINES, not rows.
  [CAUSALITY]  sequence windows look backwards only; early cycles are left-padded with the first reading.
  [METRICS]    R2 on a capped, mostly-flat target can look great; also read RMSE, the NASA score and
               the critical-zone (RUL < 50) table.
"""

In [ ]:
%%writefile projeto/src/pdm_cmaps/modeling/benchmark.py
"""RULBenchmark: grouped-CV comparison of the RUL model pool."""

from __future__ import annotations

import time
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import GroupKFold

from pdm_cmaps.domain.rul import evaluate_rul_predictions
from pdm_cmaps.domain.schema import DEFAULT_RANDOM_STATE
from pdm_cmaps.modeling.registry import get_model_pool


class RULBenchmark:
    """Compare the shared RUL model pool using holdout or grouped-fold evaluation."""

    def __init__(self, random_state: int = DEFAULT_RANDOM_STATE, model_pool=None, verbose: bool = True):
        self.random_state = random_state
        self.model_pool = model_pool
        self.verbose = verbose

    def get_models(self) -> dict:
        """Return fresh model instances for each benchmark run."""
        if self.model_pool is not None:
            return {name: clone(model) for name, model in self.model_pool.items()}
        return get_model_pool(self.random_state)

    def benchmark_train_test(
        self,
        train_frame: pd.DataFrame,
        test_frame: pd.DataFrame,
        feature_cols: list[str],
        target_col: str = "RUL",
    ) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
        """Fit models on training engines and score a disjoint held-out fleet."""
        x_train = train_frame[feature_cols].to_numpy(dtype=float)
        y_train = train_frame[target_col].to_numpy(dtype=float)
        x_test = test_frame[feature_cols].to_numpy(dtype=float)
        y_test = test_frame[target_col].to_numpy(dtype=float)
        predictions = {"y_true": y_test}
        baseline = np.full(len(y_test), y_train.mean())
        predictions["Baseline (Mean RUL)"] = baseline
        records = [{"Model": "Baseline (Mean RUL)", **evaluate_rul_predictions(y_test, baseline), "Fit_Time_s": 0.0}]

        # Conditional baseline: Mean RUL by cycle from training set
        cycle_mean_map = train_frame.groupby("cycle")[target_col].mean().to_dict()
        default_mean = float(y_train.mean())
        baseline_cycle = test_frame["cycle"].map(cycle_mean_map).fillna(default_mean).to_numpy(dtype=float)
        predictions["Baseline (Mean RUL by cycle)"] = baseline_cycle
        records.append({
            "Model": "Baseline (Mean RUL by cycle)",
            **evaluate_rul_predictions(y_test, baseline_cycle),
            "Fit_Time_s": 0.0,
        })

        models = self.get_models()
        total_models = len(models)
        for idx, (name, estimator) in enumerate(models.items(), start=1):
            if self.verbose:
                print(f"  [benchmark] [{idx}/{total_models}] Fitting '{name}' on {len(x_train)} samples...", flush=True)
            started = time.perf_counter()
            if hasattr(estimator, "fit") and not hasattr(estimator, "predict_proba") and type(estimator).__name__ in [
                "MiniRocketRULModel", "PyTorchRNNModel", "PyTorchLSTMModel", "PyTorchTransformerModel", "PyTorchCNNModel"
            ]:
                estimator.fit(train_frame, feature_cols, target_col)
                fit_seconds = time.perf_counter() - started
                if hasattr(estimator, "predict_values"):
                    prediction = estimator.predict_values(test_frame)
                else:
                    prediction = estimator.predict(test_frame)
            else:
                estimator.fit(x_train, y_train)
                fit_seconds = time.perf_counter() - started
                prediction = estimator.predict(x_test)
            predictions[name] = prediction
            metrics = evaluate_rul_predictions(y_test, prediction)
            if self.verbose:
                print(f"  [benchmark] -> '{name}' done in {fit_seconds:.2f}s | R²: {metrics['R2']:.4f} | RMSE: {metrics['RMSE']:.2f} | MAE: {metrics['MAE']:.2f}", flush=True)
            records.append({"Model": name, **metrics, "Fit_Time_s": round(fit_seconds, 3)})
        return pd.DataFrame(records), predictions

    def cross_validate(
        self,
        frame: pd.DataFrame,
        feature_cols: list[str],
        target_col: str = "RUL",
        n_splits: int = 5,
    ) -> pd.DataFrame:
        """Report mean and standard deviation of metrics across engine-grouped folds."""
        x = frame[feature_cols].to_numpy(dtype=float)
        y = frame[target_col].to_numpy(dtype=float)
        groups = frame["unit_id"].to_numpy()
        unique_groups = np.unique(groups)
        if n_splits < 2 or n_splits > len(unique_groups):
            raise ValueError(f"n_splits must be between 2 and {len(unique_groups)} engines")
        splitter = GroupKFold(n_splits=n_splits)
        records = []
        models = self.get_models()
        total_models = len(models)
        for idx, (name, prototype) in enumerate(models.items(), start=1):
            if self.verbose:
                print(f"  [grouped-cv] [{idx}/{total_models}] Running {n_splits}-fold grouped CV for '{name}'...", flush=True)
            fold_metrics = []
            oof_prediction = np.full(len(frame), np.nan)
            for train_idx, test_idx in splitter.split(x, y, groups):
                if set(groups[train_idx]).intersection(groups[test_idx]):
                    raise AssertionError("Engine leakage detected across cross-validation folds")
                if hasattr(prototype, "fit") and not hasattr(prototype, "predict_proba") and type(prototype).__name__ in [
                    "MiniRocketRULModel", "PyTorchRNNModel", "PyTorchLSTMModel", "PyTorchTransformerModel", "PyTorchCNNModel"
                ]:
                    import copy
                    model = copy.deepcopy(prototype)
                    train_sub = frame.iloc[train_idx].copy().reset_index(drop=True)
                    test_sub = frame.iloc[test_idx].copy().reset_index(drop=True)
                    model.fit(train_sub, feature_cols, target_col)
                    if hasattr(model, "predict_values"):
                        pred_fold = model.predict_values(test_sub)
                    else:
                        pred_fold = model.predict(test_sub)
                else:
                    model = clone(prototype).fit(x[train_idx], y[train_idx])
                    pred_fold = model.predict(x[test_idx])
                oof_prediction[test_idx] = pred_fold
                fold_metrics.append(evaluate_rul_predictions(y[test_idx], pred_fold))
            pooled = evaluate_rul_predictions(y, oof_prediction)
            summary_row = {
                "Model": name,
                "CV_R2_Mean": float(pooled["R2"]),
                "CV_RMSE_Mean": float(pooled["RMSE"]),
                "CV_MAE_Mean": float(pooled["MAE"]),
                "CV_NASA_Score_Mean": float(pooled["NASA_Score"]),
                "CV_Mean_NASA_Score": float(pooled["Mean_NASA_Score"]),
                "CV_R2_Std": float(np.std([item["R2"] for item in fold_metrics], ddof=1)),
                "CV_RMSE_Std": float(np.std([item["RMSE"] for item in fold_metrics], ddof=1)),
                "CV_MAE_Std": float(np.std([item["MAE"] for item in fold_metrics], ddof=1)),
                "CV_NASA_Score_Std": float(np.std([item["NASA_Score"] for item in fold_metrics], ddof=1)),
                "CV_OOF_Pooled_R2": float(pooled["R2"]),
                "CV_OOF_Pooled_RMSE": float(pooled["RMSE"]),
                "CV_OOF_Pooled_MAE": float(pooled["MAE"]),
                "CV_OOF_Pooled_NASA_Score": float(pooled["NASA_Score"]),
                "CV_OOF_Pooled_Mean_NASA_Score": float(pooled["Mean_NASA_Score"]),
                **{f"Fold_Mean_{m}": float(np.mean([item[m] for item in fold_metrics])) for m in ("R2", "RMSE", "MAE", "Mean_NASA_Score")},
                **{f"Fold_Std_{m}": float(np.std([item[m] for item in fold_metrics], ddof=1)) for m in ("R2", "RMSE", "MAE", "Mean_NASA_Score")},
                "CV_Aggregation": "pooled OOF; predictions are not clipped",
            }
            if self.verbose:
                print(f"  [grouped-cv] -> '{name}' CV Mean R²: {summary_row['CV_R2_Mean']:.4f} (±{summary_row['CV_R2_Std']:.4f}) | RMSE: {summary_row['CV_RMSE_Mean']:.2f}", flush=True)
            records.append(summary_row)
        return pd.DataFrame(records)

In [ ]:
%%writefile projeto/src/pdm_cmaps/modeling/determinism.py
"""Execução determinística das redes PyTorch (mesma semente + mesmos dados = mesmas previsões, em qualquer máquina e contagem de threads).

PROBLEMA MEDIDO (2026-11-18, FD001, semente 42, RMSE no holdout): a CNN e o Transformer repetem bit a bit com a mesma contagem de threads,
mas mudam quando ela muda (CNN 18,43 com 2 threads e 18,77 com 4); o LSTM nem repete na mesma configuração (19,18 e 16,68 com 2 threads).
Causa: a ordem de redução de ponto flutuante entre threads (e, no LSTM, o kernel recorrente) muda o caminho do otimizador, e o *early
stopping* amplifica diferenças minúsculas. Semear `torch`/`numpy` não basta.

CORREÇÃO: dentro de `fit` e `predict` (a) semear python, numpy e torch, (b) fixar o número de threads (padrão 1, ver PDM_TORCH_THREADS),
(c) ligar `torch.use_deterministic_algorithms` (warn_only: um operador sem versão determinística avisa, não derruba a rodada) e
(d) restaurar o estado anterior ao sair, para não alterar o resto do processo. O custo é tempo: 1 thread é mais lento que 4.
"""
from __future__ import annotations

import contextlib
import functools
import os
import random
import warnings

import numpy as np
import torch

DEFAULT_THREADS = 1


def deterministic_threads() -> int:
    return max(1, int(os.environ.get("PDM_TORCH_THREADS", DEFAULT_THREADS)))


@contextlib.contextmanager
def deterministic_torch(seed: int, threads: int | None = None):
    """Semeia tudo, fixa as threads e liga os algoritmos determinísticos; restaura o estado anterior ao sair."""
    threads = deterministic_threads() if threads is None else int(threads)
    prev_threads = torch.get_num_threads()
    prev_det = torch.are_deterministic_algorithms_enabled()
    prev_warn = torch.is_deterministic_algorithms_warn_only_enabled()
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # exigido pelo CUDA para o determinismo; sem efeito na CPU
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(threads)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        torch.use_deterministic_algorithms(True, warn_only=True)
    try:
        yield
    finally:
        torch.set_num_threads(prev_threads)
        torch.use_deterministic_algorithms(prev_det, warn_only=prev_warn)


def deterministic(method):
    """Decorador para `fit`/`predict` de um modelo com atributo `random_state`."""
    @functools.wraps(method)
    def wrapper(self, *args, **kwargs):
        with deterministic_torch(int(getattr(self, "random_state", 42) or 42)):
            return method(self, *args, **kwargs)
    return wrapper

In [ ]:
%%writefile projeto/src/pdm_cmaps/modeling/gbdt.py
"""XGBoost trained with the asymmetric NASA/PHM loss (objective + sklearn-style wrapper).

The model pool that uses it lives in registry.py; the package overview is in modeling/__init__.py."""

from __future__ import annotations

import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin


def nasa_xgboost_objective(y_true, y_pred):
    """Gradient/Hessian of the asymmetric PHM loss for XGBoost (d = predicted - true).

    loss = exp(-d/13) - 1 for d < 0 (too early),  exp(d/10) - 1 for d >= 0 (too late).
    d is clipped to +-30 so exp() cannot explode far from the optimum.
    """
    diff = np.clip(y_pred - y_true, -30.0, 30.0)
    grad = np.where(diff < 0, -(1.0 / 13.0) * np.exp(-diff / 13.0), (1.0 / 10.0) * np.exp(diff / 10.0))
    hess = np.where(diff < 0, (1.0 / 169.0) * np.exp(-diff / 13.0), (1.0 / 100.0) * np.exp(diff / 10.0))
    return grad, hess


class XGBoostNasaRegressor(BaseEstimator, RegressorMixin):
    """XGBoost trained with the asymmetric NASA/PHM loss.

    Why a wrapper: with a custom objective XGBoost starts every prediction at base_score=0.5, but
    RUL is ~60-125. The clipped gradient then moves each prediction only ~0.65 cycles per tree, so
    80 trees never reach the target (R² was NEGATIVE, worse than predicting the mean). Even from the mean,
    learning_rate=0.05 underfits (R² 0.69 vs 0.78 at 0.1) because the asymmetric gradients are small. Starting
    from the training-mean RUL (the optimum of a constant model) fixes this; it must be computed
    inside fit() so each CV fold / refit uses only its own training data [LEAKAGE].
    """

    def __init__(self, n_estimators=80, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, extra_params=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.random_state = random_state
        self.n_jobs = n_jobs
        self.extra_params = extra_params

    def fit(self, X, y):
        from xgboost import XGBRegressor

        y_arr = np.asarray(y, dtype=float)
        self.model_ = XGBRegressor(
            objective=nasa_xgboost_objective,
            base_score=float(np.mean(y_arr)),
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            learning_rate=self.learning_rate,
            random_state=self.random_state,
            n_jobs=self.n_jobs,
            **(self.extra_params or {}),
        )
        self.model_.fit(X, y_arr)
        return self

    def predict(self, X):
        return self.model_.predict(X)

In [ ]:
%%writefile projeto/src/pdm_cmaps/modeling/minirocket.py
"""MiniRocket convolutional feature transformer and RUL model."""

from __future__ import annotations

import time
import itertools
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
import torch
import torch.nn.functional as F

from pdm_cmaps.modeling.determinism import deterministic
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import RidgeCV

from pdm_cmaps.modeling.sequences import build_causal_engine_sequences



class MiniRocketTransformer(BaseEstimator, TransformerMixin):
    """MiniRocket multivariate time-series feature extractor in native PyTorch.

    Extracts deterministic convolutional features using length-9 zero-sum kernels,
    causal left-padding, and Proportion of Positive Values (PPV) pooling.
    """

    def __init__(
        self,
        num_kernels: int = 1000,
        seq_length: int = 15,
        dilations: tuple[int, ...] = (1, 2, 4),
        max_channels: int = 3,
        random_state: int | None = 42,
        device: str = "cpu",
        batch_size: int = 2048,
    ):
        self.num_kernels = num_kernels
        self.seq_length = seq_length
        self.dilations = dilations
        self.max_channels = max_channels
        self.random_state = random_state
        self.device = torch.device(device)
        self.batch_size = batch_size

    @staticmethod
    def generate_base_kernels() -> np.ndarray:
        """Generate the 84 canonical length-9 patterns with weights in {-1, 2} summing to zero."""
        combs = list(itertools.combinations(range(9), 3))
        base = np.full((len(combs), 9), -1.0, dtype=np.float32)
        for i, idxs in enumerate(combs):
            base[i, list(idxs)] = 2.0
        return base

    def _format_input(self, X: np.ndarray | torch.Tensor) -> torch.Tensor:
        """Convert input array to a float32 torch.Tensor with shape (samples, channels, time)."""
        if isinstance(X, np.ndarray):
            tensor = torch.from_numpy(X).float()
        elif isinstance(X, torch.Tensor):
            tensor = X.float()
        else:
            tensor = torch.tensor(X, dtype=torch.float32)

        if tensor.ndim != 3:
            raise ValueError(
                f"Expected 3D input (samples, channels, time) or (samples, time, channels), "
                f"got shape {tuple(tensor.shape)}"
            )

        if hasattr(self, "in_channels_"):
            if tensor.shape[1] != self.in_channels_ and tensor.shape[2] == self.in_channels_:
                tensor = tensor.transpose(1, 2)
        else:
            if tensor.shape[1] == self.seq_length and tensor.shape[2] != self.seq_length:
                tensor = tensor.transpose(1, 2)

        return tensor

    def fit(self, X: np.ndarray | torch.Tensor, y: np.ndarray | None = None) -> "MiniRocketTransformer":
        """Generate kernel parameters and extract quantile threshold biases from training sequences."""
        tensor_x = self._format_input(X)
        self.in_channels_ = tensor_x.shape[1]
        rng = np.random.default_rng(self.random_state)
        self.base_kernels_ = self.generate_base_kernels()

        dilations = (1, 2, 4) if self.dilations is None else tuple(self.dilations)
        active_dilations = tuple(d for d in dilations if 8 * d <= max(8, int(self.seq_length * 2.5)))
        if not active_dilations:
            active_dilations = (1,)
        self.active_dilations_ = active_dilations

        n_dil = len(active_dilations)
        base_count = self.num_kernels // n_dil
        remainder = self.num_kernels % n_dil
        counts = [base_count + (1 if i < remainder else 0) for i in range(n_dil)]

        dilation_groups = []
        all_weights = {}

        for d, k_count in zip(active_dilations, counts):
            if k_count == 0:
                continue
            weights_d = np.zeros((k_count, self.in_channels_, 9), dtype=np.float32)
            for i in range(k_count):
                pattern_idx = rng.integers(0, 84)
                pattern = self.base_kernels_[pattern_idx]
                n_c = rng.integers(1, min(self.max_channels, self.in_channels_) + 1)
                selected_channels = rng.choice(self.in_channels_, size=n_c, replace=False)
                for c in selected_channels:
                    weights_d[i, c, :] = pattern

            weights_tensor = torch.from_numpy(weights_d).to(self.device)
            dilation_groups.append((d, weights_tensor))
            all_weights[d] = weights_d

        self.weights_ = all_weights

        n_samples = tensor_x.shape[0]
        sample_size = min(n_samples, 1000)
        if n_samples > sample_size:
            sample_indices = rng.choice(n_samples, size=sample_size, replace=False)
            x_sub = tensor_x[sample_indices].to(self.device)
        else:
            x_sub = tensor_x.to(self.device)

        fitted_dilation_groups = []
        biases_dict = {}

        with torch.no_grad():
            for d, weights_tensor in dilation_groups:
                k_count = weights_tensor.shape[0]
                # Same zero padding as transform()/convolve(): biases must be learned under the exact
                # convolution that is later thresholded. (Replicate padding was tested: R² 0.55 vs 0.82, because
                # zero padding lets early-life windows encode "age" at the edge.)
                pad_x = F.pad(x_sub, (8 * d, 0))
                out_d = F.conv1d(pad_x, weights_tensor, dilation=d)
                out_flat = out_d.transpose(0, 1).reshape(k_count, -1).cpu().numpy()

                quantiles = rng.uniform(0.1, 0.9, size=k_count)
                sorted_vals = np.sort(out_flat, axis=1)
                q_indices = np.clip(
                    (quantiles * (sorted_vals.shape[1] - 1)).astype(int),
                    0,
                    sorted_vals.shape[1] - 1,
                )
                b_d = sorted_vals[np.arange(k_count), q_indices].astype(np.float32)

                bias_tensor = torch.from_numpy(b_d).to(self.device)
                fitted_dilation_groups.append((d, weights_tensor, bias_tensor))
                biases_dict[d] = b_d

        self.dilation_groups_ = fitted_dilation_groups
        self.biases_ = biases_dict
        return self

    def convolve(self, X: np.ndarray | torch.Tensor) -> dict[int, torch.Tensor]:
        """Compute causal convolution output tensors for each active dilation group."""
        if not hasattr(self, "dilation_groups_"):
            raise RuntimeError("MiniRocketTransformer is not fitted yet. Call fit() first.")
        tensor_x = self._format_input(X)
        outputs = {}
        with torch.no_grad():
            for d, weights_tensor, _ in self.dilation_groups_:
                batch_x = tensor_x.to(self.device)
                pad_x = F.pad(batch_x, (8 * d, 0))
                outputs[d] = F.conv1d(pad_x, weights_tensor, dilation=d)
        return outputs

    def transform(self, X: np.ndarray | torch.Tensor) -> np.ndarray:
        """Compute Proportion of Positive Values (PPV) features for each input sequence."""
        if not hasattr(self, "dilation_groups_"):
            raise RuntimeError("MiniRocketTransformer is not fitted yet. Call fit() first.")

        tensor_x = self._format_input(X)
        n_samples = tensor_x.shape[0]
        if n_samples == 0:
            return np.empty((0, self.num_kernels), dtype=np.float32)

        all_ppv = []
        with torch.no_grad():
            for start_idx in range(0, n_samples, self.batch_size):
                end_idx = min(start_idx + self.batch_size, n_samples)
                batch_x = tensor_x[start_idx:end_idx].to(self.device)
                batch_features = []
                for d, weights_tensor, bias_tensor in self.dilation_groups_:
                    pad_x = F.pad(batch_x, (8 * d, 0))
                    out_d = F.conv1d(pad_x, weights_tensor, dilation=d)
                    ppv_d = (out_d > bias_tensor.view(1, -1, 1)).float().mean(dim=-1)
                    batch_features.append(ppv_d)
                batch_ppv = torch.cat(batch_features, dim=1).cpu().numpy()
                all_ppv.append(batch_ppv)

        return np.vstack(all_ppv)

class MiniRocketRULModel:
    """Scikit-learn-style RUL pipeline combining engine sequence creation, MiniRocket PPV features, and RidgeCV."""

    def __init__(
        self,
        seq_length: int = 15,
        num_kernels: int = 1000,
        dilations: tuple[int, ...] = (1, 2, 4),
        max_channels: int = 3,
        alphas: np.ndarray | None = None,
        random_state: int = 42,
        device: str = "cpu",
        batch_size: int = 2048,
    ):
        self.seq_length = seq_length
        self.num_kernels = num_kernels
        self.dilations = dilations
        self.max_channels = max_channels
        self.alphas = np.logspace(-2, 4, 7) if alphas is None else np.asarray(alphas)
        self.random_state = random_state
        self.device = device
        self.batch_size = batch_size
        self.scaler: StandardScaler | None = None
        self.transformer: MiniRocketTransformer | None = None
        self.feature_scaler: StandardScaler | None = None
        self.regressor: RidgeCV | None = None
        self.feature_cols: list[str] = []
        self.target_col: str = ""
        self.fit_time: float = 0.0

    @deterministic
    def fit(self, frame: pd.DataFrame, feature_cols: list[str], target_col: str) -> "MiniRocketRULModel":
        """Form engine-local sequences, extract PPV features, and train a regularized Ridge regressor."""
        started = time.perf_counter()
        self.feature_cols = list(feature_cols)
        self.target_col = target_col

        x, y, self.scaler = build_causal_engine_sequences(
            frame, self.feature_cols, target_col, self.seq_length, fit_scaler=True,
        )
        x_trans = np.transpose(x, (0, 2, 1))

        self.transformer = MiniRocketTransformer(
            num_kernels=self.num_kernels,
            seq_length=self.seq_length,
            dilations=self.dilations,
            max_channels=self.max_channels,
            random_state=self.random_state,
            device=self.device,
            batch_size=self.batch_size,
        )
        ppv = self.transformer.fit_transform(x_trans)
        self.feature_scaler = StandardScaler()
        ppv_scaled = self.feature_scaler.fit_transform(ppv)

        groups = frame["unit_id"].to_numpy()
        n_groups = len(np.unique(groups))
        cv_splits = min(5, n_groups)
        if cv_splits >= 2:
            cv = list(GroupKFold(n_splits=cv_splits).split(ppv_scaled, y, groups=groups))
            self.regressor = RidgeCV(alphas=self.alphas, cv=cv)
        else:
            self.regressor = RidgeCV(alphas=self.alphas)
        self.regressor.fit(ppv_scaled, y)

        self.fit_time = time.perf_counter() - started
        return self

    @deterministic
    def predict(self, frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
        """Return aligned true targets and sequence-endpoint predictions."""
        if self.regressor is None or self.scaler is None or self.transformer is None or self.feature_scaler is None:
            raise RuntimeError("Fit the MiniRocketRULModel before calling predict")

        x, y, _ = build_causal_engine_sequences(
            frame, self.feature_cols, self.target_col, self.seq_length,
            scaler=self.scaler, fit_scaler=False,
        )
        x_trans = np.transpose(x, (0, 2, 1))
        ppv = self.transformer.transform(x_trans)
        ppv_scaled = self.feature_scaler.transform(ppv)
        prediction = self.regressor.predict(ppv_scaled)
        return y, prediction

    def predict_values(self, frame: pd.DataFrame) -> np.ndarray:
        """Return only the prediction array."""
        _, prediction = self.predict(frame)
        return prediction

In [ ]:
%%writefile projeto/src/pdm_cmaps/modeling/networks.py
"""PyTorch network architectures (nn.Module)."""

from __future__ import annotations

import math
from typing import Literal
import torch
from torch import nn



class LSTMRegressorNet(nn.Module):
    """LSTM encoder followed by a scalar regression head."""

    def __init__(self, input_dim: int, hidden_dim: int = 64, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Dropout(0.1), nn.Linear(32, 1))

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        output, _ = self.lstm(values)
        return self.head(output[:, -1]).squeeze(-1)


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for sequence tokens."""

    def __init__(self, d_model: int, max_len: int = 500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]

class RNNRegressorNet(nn.Module):
    """Recurrent Neural Network (LSTM or GRU) with regression head."""

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64,
        num_layers: int = 2,
        dropout: float = 0.2,
        cell_type: Literal["lstm", "gru"] = "lstm",
    ):
        super().__init__()
        self.cell_type = cell_type.lower()
        if self.cell_type == "lstm":
            self.rnn = nn.LSTM(
                input_dim, hidden_dim, num_layers=num_layers, batch_first=True,
                dropout=dropout if num_layers > 1 else 0.0,
            )
        elif self.cell_type == "gru":
            self.rnn = nn.GRU(
                input_dim, hidden_dim, num_layers=num_layers, batch_first=True,
                dropout=dropout if num_layers > 1 else 0.0,
            )
        else:
            raise ValueError(f"Unknown cell_type: {cell_type}")

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.rnn(x)
        # Take last time step
        last_hidden = out[:, -1, :]
        return self.head(last_hidden).squeeze(-1)

class CNNRegressorNet(nn.Module):
    """1D Temporal Convolutional Neural Network with Adaptive Pooling & MLP head."""

    def __init__(
        self,
        input_dim: int,
        num_filters: int = 64,
        kernel_size: int = 3,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv1d(in_channels=input_dim, out_channels=num_filters, kernel_size=kernel_size, padding="same"),
            nn.BatchNorm1d(num_filters),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(in_channels=num_filters, out_channels=num_filters * 2, kernel_size=kernel_size, padding="same"),
            nn.BatchNorm1d(num_filters * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(num_filters * 2, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # PyTorch Conv1D expects (Batch, Channels, SeqLen)
        x_trans = x.transpose(1, 2)
        feats = self.conv_block(x_trans)
        pooled = self.pool(feats).squeeze(-1)
        return self.head(pooled).squeeze(-1)

class TransformerRegressorNet(nn.Module):
    """Time Series Transformer Encoder with Multi-Head Self-Attention and regression head."""

    def __init__(
        self,
        input_dim: int,
        d_model: int = 64,
        nhead: int = 4,
        num_layers: int = 2,
        dim_feedforward: int = 128,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="relu",
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (Batch, SeqLen, Features)
        proj = self.input_proj(x)
        encoded = self.pos_encoder(proj)
        out = self.transformer_encoder(encoded)
        # Global average pooling over sequence length
        pooled = out.mean(dim=1)
        return self.head(pooled).squeeze(-1)

In [ ]:
%%writefile projeto/src/pdm_cmaps/modeling/registry.py
"""Model factory/pool and grouped out-of-fold predictions."""

from __future__ import annotations

from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import BayesianRidge, ElasticNet, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVR, SVR
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import GroupKFold

from pdm_cmaps.domain.rul import regression_metrics
from pdm_cmaps.domain.schema import DEFAULT_RANDOM_STATE
from pdm_cmaps.modeling.gbdt import XGBoostNasaRegressor


def get_model(name: str, random_state: int = 42, **kwargs):
    """Construct a fresh estimator by canonical model name."""
    key = name.lower().replace(" ", "_").replace("-", "_")
    if key in {"extra_trees", "et"}:
        return ExtraTreesRegressor(n_estimators=kwargs.pop("n_estimators", 60), max_depth=kwargs.pop("max_depth", 12), min_samples_leaf=kwargs.pop("min_samples_leaf", 2), random_state=random_state, n_jobs=kwargs.pop("n_jobs", -1), **kwargs)
    if key in {"random_forest", "rf"}:
        return RandomForestRegressor(n_estimators=kwargs.pop("n_estimators", 60), max_depth=kwargs.pop("max_depth", 12), min_samples_leaf=kwargs.pop("min_samples_leaf", 2), random_state=random_state, n_jobs=kwargs.pop("n_jobs", -1), **kwargs)
    if key in {"hist_gradient_boosting", "gradient_boosting", "gb"}:
        return HistGradientBoostingRegressor(max_iter=kwargs.pop("max_iter", 80), max_depth=kwargs.pop("max_depth", 6), learning_rate=kwargs.pop("learning_rate", 0.08), random_state=random_state, **kwargs)
    if key in {"xgboost", "xgb"}:
        from xgboost import XGBRegressor
        n_estimators = kwargs.pop("n_estimators", kwargs.pop("iterations", 80))
        max_depth = kwargs.pop("max_depth", kwargs.pop("depth", 6))
        return XGBRegressor(n_estimators=n_estimators, max_depth=max_depth, learning_rate=kwargs.pop("learning_rate", 0.08), random_state=random_state, n_jobs=kwargs.pop("n_jobs", -1), **kwargs)
    if key in {"xgboost_nasa_loss", "xgboost_nasa", "xgb_nasa"}:
        n_estimators = kwargs.pop("n_estimators", kwargs.pop("iterations", 80))
        max_depth = kwargs.pop("max_depth", kwargs.pop("depth", 6))
        return XGBoostNasaRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=kwargs.pop("learning_rate", 0.1),
            random_state=random_state,
            n_jobs=kwargs.pop("n_jobs", -1),
            extra_params=kwargs or None,
        )
    if key in {"catboost", "cb"}:
        from catboost import CatBoostRegressor
        iterations = kwargs.pop("iterations", kwargs.pop("n_estimators", 100))
        depth = kwargs.pop("depth", kwargs.pop("max_depth", 6))
        return CatBoostRegressor(iterations=iterations, depth=depth, learning_rate=kwargs.pop("learning_rate", 0.08), random_seed=random_state, verbose=kwargs.pop("verbose", 0), thread_count=kwargs.pop("thread_count", kwargs.pop("n_jobs", -1)), **kwargs)
    if key in {"lightgbm", "lgb", "lgbm"}:
        from lightgbm import LGBMRegressor
        n_estimators = kwargs.pop("n_estimators", kwargs.pop("iterations", 80))
        max_depth = kwargs.pop("max_depth", kwargs.pop("depth", 6))
        return LGBMRegressor(n_estimators=n_estimators, max_depth=max_depth, learning_rate=kwargs.pop("learning_rate", 0.08), random_state=random_state, n_jobs=kwargs.pop("n_jobs", -1), verbose=kwargs.pop("verbose", -1), **kwargs)
    if key == "ridge":
        return make_pipeline(StandardScaler(), Ridge(alpha=kwargs.pop("alpha", 10.0), **kwargs))
    if key in {"elasticnet", "elastic_net"}:
        return make_pipeline(StandardScaler(), ElasticNet(alpha=kwargs.pop("alpha", 0.5), l1_ratio=kwargs.pop("l1_ratio", 0.5), max_iter=3000, **kwargs))
    if key in {"bayesian_ridge", "bayes"}:
        return make_pipeline(StandardScaler(), BayesianRidge(**kwargs))
    if key in {"svm", "svr", "linear_svr", "support_vector_machine"}:
        return make_pipeline(StandardScaler(), LinearSVR(C=kwargs.pop("C", 1.0), random_state=random_state, max_iter=kwargs.pop("max_iter", 5000), **kwargs))
    if key == "rbf_svr":
        return make_pipeline(StandardScaler(), SVR(kernel="rbf", **kwargs))
    if key in {"knn", "knn_regressor"}:
        return make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=kwargs.pop("n_neighbors", 15), weights=kwargs.pop("weights", "distance"), **kwargs))
    if key in {"mlp", "neural_net"}:
        model = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=kwargs.pop("hidden_layer_sizes", (64, 32)), max_iter=kwargs.pop("max_iter", 180), random_state=random_state, early_stopping=True, **kwargs))
        return TransformedTargetRegressor(regressor=model, transformer=StandardScaler())
    raise ValueError(f"Unknown model {name!r}")

def get_model_pool(random_state: int = 42) -> dict:
    """Return the core model families used for the RUL benchmark."""
    return {
        "Ridge": get_model("ridge", random_state, alpha=10),
        "Support Vector Machine (SVM)": get_model("svm", random_state, C=1.0),
        "Random Forest": get_model("rf", random_state, n_estimators=60, max_depth=10),
        "Extra Trees": get_model("extra_trees", random_state, n_estimators=60, max_depth=10),
        "HistGradientBoosting": get_model("gradient_boosting", random_state, max_iter=80, max_depth=6),
        "XGBoost": get_model("xgboost", random_state, n_estimators=80, max_depth=6),
        "XGBoost (NASA Loss)": get_model("xgboost_nasa_loss", random_state, n_estimators=80, max_depth=6),
        "CatBoost": get_model("catboost", random_state, iterations=100, depth=6),
        "LightGBM": get_model("lightgbm", random_state, n_estimators=80, max_depth=6),
        "MLP": get_model("mlp", random_state, hidden_layer_sizes=(64, 32), max_iter=180),
    }


def grouped_oof(
    frame: pd.DataFrame,
    features: list[str],
    estimator=None,
    n_splits: int = 5,
    random_state: int = DEFAULT_RANDOM_STATE,
    target_col: str = "RUL",
) -> tuple[pd.DataFrame, np.ndarray]:
    """Fit an estimator in GroupKFold and return fold metrics and aligned OOF predictions."""
    x = frame[features].to_numpy(dtype=float)
    y = frame[target_col].to_numpy(dtype=float)
    groups = frame["unit_id"].to_numpy()
    unique_groups = np.unique(groups)
    if n_splits < 2 or n_splits > len(unique_groups):
        raise ValueError(f"n_splits must be between 2 and {len(unique_groups)} engines")
    splitter = GroupKFold(n_splits=n_splits)
    prediction = np.full(len(frame), np.nan)
    rows = []
    for fold, (train_idx, valid_idx) in enumerate(splitter.split(x, y, groups), start=1):
        if set(groups[train_idx]).intersection(groups[valid_idx]):
            raise AssertionError("Engine leakage detected in grouped fold")
        model = clone(estimator) if estimator is not None else get_model_pool(random_state)["Extra Trees"]
        model.fit(x[train_idx], y[train_idx])
        prediction[valid_idx] = model.predict(x[valid_idx])
        rows.append({
            "Fold": fold,
            "Engines": int(np.unique(groups[valid_idx]).size),
            **regression_metrics(y[valid_idx], prediction[valid_idx]),
        })
    if not np.isfinite(prediction).all():
        raise AssertionError("Grouped OOF predictions do not cover every row")
    return pd.DataFrame(rows), prediction

In [ ]:
%%writefile projeto/src/pdm_cmaps/modeling/sequences.py
"""Sliding-window sequence builders for the neural models."""

from __future__ import annotations

from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd




def build_engine_sequences(
    frame: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    seq_length: int = 15,
    scaler: StandardScaler | None = None,
    fit_scaler: bool = False,
) -> tuple[np.ndarray, np.ndarray, StandardScaler]:
    """Scale rows and form left-padded causal windows without crossing engines."""
    if seq_length < 1:
        raise ValueError("seq_length must be positive")
    ordered = frame.sort_values(["unit_id", "cycle"]).copy()
    if scaler is None:
        scaler = StandardScaler()
        fit_scaler = True
    values = scaler.fit_transform(ordered[feature_cols]) if fit_scaler else scaler.transform(ordered[feature_cols])
    scaled = pd.DataFrame(values, columns=feature_cols, index=ordered.index)
    scaled["unit_id"] = ordered["unit_id"].to_numpy()
    scaled["cycle"] = ordered["cycle"].to_numpy()
    sequences, labels = [], []
    target_values = ordered[target_col].astype(float)
    for _, group in scaled.groupby("unit_id", sort=False):
        features = group[feature_cols].to_numpy(dtype=np.float32)
        targets = target_values.loc[group.index].to_numpy(dtype=np.float32)
        for t in range(len(features)):
            start = t - seq_length + 1
            if start < 0:
                pad = np.repeat(features[:1], -start, axis=0)
                seq = np.vstack([pad, features[: t + 1]])
            else:
                seq = features[start : t + 1]
            sequences.append(seq)
            labels.append(targets[t])
    return np.asarray(sequences, dtype=np.float32), np.asarray(labels, dtype=np.float32), scaler


def build_causal_engine_sequences(
    frame: pd.DataFrame,
    feature_cols: list[str],
    target_col: str | None = None,
    seq_length: int = 15,
    scaler: StandardScaler | None = None,
    fit_scaler: bool = False,
) -> tuple[np.ndarray, np.ndarray | None, StandardScaler]:
    """Build causal sliding windows of length `seq_length` without crossing engine boundaries.
    
    Uses causal left-padding with the first available cycle for early cycles (t < seq_length),
    ensuring that EVERY row in the input DataFrame receives an aligned prediction.
    """
    if seq_length < 1:
        raise ValueError("seq_length must be positive")
    ordered = frame.sort_values(["unit_id", "cycle"]).copy()
    if scaler is None:
        scaler = StandardScaler()
        fit_scaler = True
    
    feature_vals = scaler.fit_transform(ordered[feature_cols]) if fit_scaler else scaler.transform(ordered[feature_cols])
    scaled = pd.DataFrame(feature_vals, columns=feature_cols, index=ordered.index)
    scaled["unit_id"] = ordered["unit_id"].to_numpy()
    scaled["cycle"] = ordered["cycle"].to_numpy()

    has_target = target_col is not None and target_col in ordered.columns
    targets_all = ordered[target_col].astype(float) if has_target else None

    sequences = []
    labels = [] if has_target else None

    for _, group in scaled.groupby("unit_id", sort=False):
        features = group[feature_cols].to_numpy(dtype=np.float32)
        n_rows = len(features)
        grp_targets = targets_all.loc[group.index].to_numpy(dtype=np.float32) if has_target else None

        for t in range(n_rows):
            start = t - seq_length + 1
            if start < 0:
                pad_len = -start
                pad = np.repeat(features[:1], pad_len, axis=0)
                sub = features[: t + 1]
                seq = np.vstack([pad, sub])
            else:
                seq = features[start : t + 1]
            sequences.append(seq)
            if has_target:
                labels.append(grp_targets[t])

    seq_arr = np.asarray(sequences, dtype=np.float32)
    lbl_arr = np.asarray(labels, dtype=np.float32) if has_target else None
    return seq_arr, lbl_arr, scaler

In [ ]:
%%writefile projeto/src/pdm_cmaps/modeling/torch_models.py
"""Estimator wrappers (fit/predict) around the PyTorch networks."""

from __future__ import annotations

import time
from typing import Literal
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupShuffleSplit

from pdm_cmaps.modeling.determinism import deterministic
from pdm_cmaps.modeling.networks import CNNRegressorNet, LSTMRegressorNet, RNNRegressorNet, TransformerRegressorNet
from pdm_cmaps.modeling.sequences import build_causal_engine_sequences, build_engine_sequences


class PyTorchLSTMModel:
    """Scikit-learn-style wrapper that retains fitted scalers and the trained network."""

    def __init__(
        self,
        seq_length: int = 15,
        hidden_dim: int = 64,
        num_layers: int = 2,
        learning_rate: float = 0.002,
        batch_size: int = 128,
        max_epochs: int = 25,
        patience: int = 4,
        random_state: int = 42,
        device: str = "cpu",
    ):
        self.seq_length = seq_length
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.patience = patience
        self.random_state = random_state
        self.device = torch.device(device)
        self.net = None
        self.scaler = None
        self.target_scaler = None
        self.feature_cols: list[str] = []
        self.target_col = ""
        self.fit_time = 0.0

    @deterministic
    def fit(self, frame: pd.DataFrame, feature_cols: list[str], target_col: str) -> "PyTorchLSTMModel":
        """Train on engine-local windows; early stopping monitors training loss."""
        torch.manual_seed(self.random_state)
        self.feature_cols = list(feature_cols)
        self.target_col = target_col
        started = time.perf_counter()
        x, y, self.scaler = build_engine_sequences(
            frame, self.feature_cols, target_col, self.seq_length, fit_scaler=True,
        )
        self.target_scaler = StandardScaler().fit(y.reshape(-1, 1))
        scaled_y = self.target_scaler.transform(y.reshape(-1, 1)).astype(np.float32).ravel()
        loader = DataLoader(TensorDataset(torch.from_numpy(x), torch.from_numpy(scaled_y)), batch_size=self.batch_size, shuffle=True)
        self.net = LSTMRegressorNet(len(self.feature_cols), self.hidden_dim, self.num_layers).to(self.device)
        optimizer = torch.optim.Adam(self.net.parameters(), lr=self.learning_rate, weight_decay=1e-4)
        loss_fn = nn.MSELoss()
        best_loss, best_state, waiting = float("inf"), None, 0
        for _ in range(self.max_epochs):
            self.net.train()
            total = 0.0
            for batch_x, batch_y in loader:
                batch_x, batch_y = batch_x.to(self.device), batch_y.to(self.device)
                optimizer.zero_grad()
                loss = loss_fn(self.net(batch_x), batch_y)
                loss.backward()
                optimizer.step()
                total += float(loss.item()) * len(batch_y)
            epoch_loss = total / len(loader.dataset)
            if epoch_loss < best_loss - 1e-4:
                best_loss = epoch_loss
                best_state = {key: value.detach().cpu().clone() for key, value in self.net.state_dict().items()}
                waiting = 0
            else:
                waiting += 1
                if waiting >= self.patience:
                    break
        if best_state is not None:
            self.net.load_state_dict(best_state)
        self.fit_time = time.perf_counter() - started
        return self

    @deterministic
    def predict(self, frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
        """Return aligned target and sequence-endpoint predictions."""
        if self.net is None or self.scaler is None or self.target_scaler is None:
            raise RuntimeError("Fit the LSTM before calling predict")
        x, y, _ = build_engine_sequences(
            frame, self.feature_cols, self.target_col, self.seq_length,
            scaler=self.scaler, fit_scaler=False,
        )
        self.net.eval()
        with torch.no_grad():
            scaled = self.net(torch.from_numpy(x).to(self.device)).cpu().numpy()
        prediction = self.target_scaler.inverse_transform(scaled.reshape(-1, 1)).ravel()
        return y, prediction


def get_safe_device() -> torch.device:
    """Return CUDA device only if hardware compute capability is supported by PyTorch, else CPU."""
    if torch.cuda.is_available():
        try:
            cap = torch.cuda.get_device_capability()
            if cap[0] >= 7:
                _ = torch.zeros(1, device="cuda")
                return torch.device("cuda")
        except Exception:
            pass
    return torch.device("cpu")

class BasePyTorchSequenceModel:
    """Base class providing sequence batching, training loop, early stopping, and device management."""

    def __init__(
        self,
        seq_length: int = 15,
        learning_rate: float = 0.002,
        batch_size: int = 256,
        max_epochs: int = 30,
        patience: int = 4,
        random_state: int = 42,
        device: str | None = None,
    ):
        self.seq_length = seq_length
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.patience = patience
        self.random_state = random_state
        if device is None:
            self.device = get_safe_device()
        else:
            self.device = torch.device(device)

        self.net: nn.Module | None = None
        self.scaler: StandardScaler | None = None
        self.target_scaler: StandardScaler | None = None
        self.feature_cols: list[str] = []
        self.fit_time: float = 0.0

    def _build_network(self, input_dim: int) -> nn.Module:
        raise NotImplementedError("Subclasses must implement _build_network")

    @deterministic
    def fit(self, frame: pd.DataFrame, feature_cols: list[str], target_col: str = "RUL") -> "BasePyTorchSequenceModel":
        """Fit model with engine-preserving causal sequences, deterministic generator, and grouped early stopping."""
        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)
        self.feature_cols = list(feature_cols)

        started = time.perf_counter()

        # Engine-grouped internal validation partition (15%) for uncorrupted early stopping
        n_engines = frame["unit_id"].nunique()
        has_val = n_engines >= 4
        if has_val:
            gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=self.random_state)
            tr_idx, val_idx = next(gss.split(frame, groups=frame["unit_id"]))
            tr_frame = frame.iloc[tr_idx].copy().reset_index(drop=True)
            val_frame = frame.iloc[val_idx].copy().reset_index(drop=True)
        else:
            tr_frame = frame
            val_frame = None

        x_tr, y_tr, self.scaler = build_causal_engine_sequences(
            tr_frame, self.feature_cols, target_col, self.seq_length, fit_scaler=True,
        )
        self.target_scaler = StandardScaler().fit(y_tr.reshape(-1, 1))
        scaled_y_tr = self.target_scaler.transform(y_tr.reshape(-1, 1)).astype(np.float32).ravel()

        gen = torch.Generator().manual_seed(self.random_state)
        dataset_tr = TensorDataset(torch.from_numpy(x_tr), torch.from_numpy(scaled_y_tr))
        loader_tr = DataLoader(dataset_tr, batch_size=self.batch_size, shuffle=True, generator=gen)

        loader_val = None
        if val_frame is not None:
            x_val, y_val, _ = build_causal_engine_sequences(
                val_frame, self.feature_cols, target_col, self.seq_length, scaler=self.scaler, fit_scaler=False,
            )
            scaled_y_val = self.target_scaler.transform(y_val.reshape(-1, 1)).astype(np.float32).ravel()
            loader_val = DataLoader(
                TensorDataset(torch.from_numpy(x_val), torch.from_numpy(scaled_y_val)),
                batch_size=self.batch_size * 2,
                shuffle=False,
            )

        self.net = self._build_network(len(self.feature_cols)).to(self.device)
        optimizer = torch.optim.AdamW(self.net.parameters(), lr=self.learning_rate, weight_decay=1e-4)
        loss_fn = nn.MSELoss()

        best_loss = float("inf")
        best_state = None
        waiting = 0

        for _ in range(self.max_epochs):
            self.net.train()
            total_tr_loss = 0.0
            for batch_x, batch_y in loader_tr:
                batch_x, batch_y = batch_x.to(self.device), batch_y.to(self.device)
                optimizer.zero_grad()
                pred = self.net(batch_x)
                loss = loss_fn(pred, batch_y)
                loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), max_norm=2.0)
                optimizer.step()
                total_tr_loss += float(loss.item()) * len(batch_y)

            # Evaluate early stopping on internal engine validation split
            if loader_val is not None:
                self.net.eval()
                val_loss = 0.0
                with torch.no_grad():
                    for batch_x, batch_y in loader_val:
                        batch_x, batch_y = batch_x.to(self.device), batch_y.to(self.device)
                        val_loss += float(loss_fn(self.net(batch_x), batch_y).item()) * len(batch_y)
                epoch_loss = val_loss / len(loader_val.dataset)
            else:
                epoch_loss = total_tr_loss / len(loader_tr.dataset)

            if epoch_loss < best_loss - 1e-4:
                best_loss = epoch_loss
                best_state = {k: v.detach().cpu().clone() for k, v in self.net.state_dict().items()}
                waiting = 0
            else:
                waiting += 1
                if waiting >= self.patience:
                    break

        if best_state is not None:
            self.net.load_state_dict(best_state)

        self.fit_time = time.perf_counter() - started
        return self

    def predict_values(self, frame: pd.DataFrame) -> np.ndarray:
        """Alias for predict returning raw array."""
        return self.predict(frame)

    @deterministic
    def predict(self, frame: pd.DataFrame) -> np.ndarray:
        """Predict aligned target values for every row in the DataFrame."""
        if self.net is None or self.scaler is None or self.target_scaler is None:
            raise RuntimeError("Model must be fitted before predict")

        x, _, _ = build_causal_engine_sequences(
            frame, self.feature_cols, None, self.seq_length, scaler=self.scaler, fit_scaler=False,
        )
        self.net.eval()
        preds = []
        loader = DataLoader(TensorDataset(torch.from_numpy(x)), batch_size=self.batch_size * 2, shuffle=False)
        with torch.no_grad():
            for (batch_x,) in loader:
                batch_x = batch_x.to(self.device)
                scaled_pred = self.net(batch_x).cpu().numpy()
                preds.append(scaled_pred)

        concatenated = np.concatenate(preds)
        unscaled = self.target_scaler.inverse_transform(concatenated.reshape(-1, 1)).ravel()
        return unscaled

class PyTorchRNNModel(BasePyTorchSequenceModel):
    """Modular Recurrent Neural Network (LSTM or GRU) Regressor."""

    def __init__(
        self,
        seq_length: int = 15,
        hidden_dim: int = 64,
        num_layers: int = 2,
        dropout: float = 0.2,
        cell_type: Literal["lstm", "gru"] = "lstm",
        learning_rate: float = 0.002,
        batch_size: int = 256,
        max_epochs: int = 25,
        patience: int = 4,
        random_state: int = 42,
        device: str | None = None,
    ):
        super().__init__(
            seq_length=seq_length,
            learning_rate=learning_rate,
            batch_size=batch_size,
            max_epochs=max_epochs,
            patience=patience,
            random_state=random_state,
            device=device,
        )
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout = dropout
        self.cell_type = cell_type

    def _build_network(self, input_dim: int) -> nn.Module:
        return RNNRegressorNet(
            input_dim=input_dim,
            hidden_dim=self.hidden_dim,
            num_layers=self.num_layers,
            dropout=self.dropout,
            cell_type=self.cell_type,
        )

class PyTorchCNNModel(BasePyTorchSequenceModel):
    """1D Temporal Convolutional Neural Network Regressor."""

    def __init__(
        self,
        seq_length: int = 15,
        num_filters: int = 64,
        kernel_size: int = 3,
        dropout: float = 0.2,
        learning_rate: float = 0.002,
        batch_size: int = 256,
        max_epochs: int = 25,
        patience: int = 4,
        random_state: int = 42,
        device: str | None = None,
    ):
        super().__init__(
            seq_length=seq_length,
            learning_rate=learning_rate,
            batch_size=batch_size,
            max_epochs=max_epochs,
            patience=patience,
            random_state=random_state,
            device=device,
        )
        self.num_filters = num_filters
        self.kernel_size = kernel_size
        self.dropout = dropout

    def _build_network(self, input_dim: int) -> nn.Module:
        return CNNRegressorNet(
            input_dim=input_dim,
            num_filters=self.num_filters,
            kernel_size=self.kernel_size,
            dropout=self.dropout,
        )

class PyTorchTransformerModel(BasePyTorchSequenceModel):
    """Time Series Transformer Regressor with Multi-Head Attention."""

    def __init__(
        self,
        seq_length: int = 15,
        d_model: int = 64,
        nhead: int = 4,
        num_layers: int = 2,
        dim_feedforward: int = 128,
        dropout: float = 0.1,
        learning_rate: float = 0.001,
        batch_size: int = 256,
        max_epochs: int = 25,
        patience: int = 4,
        random_state: int = 42,
        device: str | None = None,
    ):
        super().__init__(
            seq_length=seq_length,
            learning_rate=learning_rate,
            batch_size=batch_size,
            max_epochs=max_epochs,
            patience=patience,
            random_state=random_state,
            device=device,
        )
        self.d_model = d_model
        self.nhead = nhead
        self.num_layers = num_layers
        self.dim_feedforward = dim_feedforward
        self.dropout = dropout

    def _build_network(self, input_dim: int) -> nn.Module:
        return TransformerRegressorNet(
            input_dim=input_dim,
            d_model=self.d_model,
            nhead=self.nhead,
            num_layers=self.num_layers,
            dim_feedforward=self.dim_feedforward,
            dropout=self.dropout,
        )

### 2.2 Pacote `pdm_cmaps`: monitoring

conformal, criticidade, custo e alertas

5 arquivos.

In [ ]:
%%writefile projeto/src/pdm_cmaps/monitoring/__init__.py
"""Decision layer on top of the RUL model: conformal intervals, criticality, alerts, cost and grounded reports."""

In [ ]:
%%writefile projeto/src/pdm_cmaps/monitoring/alerts.py
"""Criticality levels and alert generation.

WHAT   Converts model outputs (RUL point, RUL interval, anomaly flag) into a criticality level
       an engineer can act on, compares several automatic strategies, and produces alert events.
WHY    Research question RQ-C: how should criticality be defined automatically? Three strategies
       are compared on held-out engines with the same ground truth:
         point               level from the predicted RUL
         interval            level from the CONSERVATIVE end (lower bound) of the RUL interval
         interval+anomaly    as above, escalated one level when an SPC anomaly alarm is active
CONCEPTS
  [ASSUMPTION]   the level thresholds (<=15 critical, <=30 warning, <=60 attention cycles) are
                 maintenance-policy parameters, not facts about the data. Report sensitivity.
  [METRIC-CHOICE] a missed critical state costs far more than an early warning, so we report
                 missed-critical rate AND false-alarm rate (a trade-off), never accuracy alone.
  [CAUSALITY]    debouncing uses only the past: a level is raised only after `persistence`
                 consecutive cycles at or above it (same idea as the Western-Electric rule).
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd

LEVEL_NAMES = ("NORMAL", "ATTENTION", "WARNING", "CRITICAL")
NORMAL, ATTENTION, WARNING, CRITICAL = range(4)


@dataclass(frozen=True)
class CriticalityPolicy:
    """RUL thresholds (cycles) that define the levels; level k applies when RUL <= threshold k."""

    attention: float = 60.0
    warning: float = 30.0
    critical: float = 15.0
    persistence: int = 2

    def level_from_rul(self, rul) -> np.ndarray:
        r = np.asarray(rul, dtype=float)
        level = np.zeros(r.shape, dtype=int)
        level[r <= self.attention] = ATTENTION
        level[r <= self.warning] = WARNING
        level[r <= self.critical] = CRITICAL
        return level


def debounce(levels: np.ndarray, persistence: int) -> np.ndarray:
    """Causal k-of-k filter: level_t = min(raw levels over the last k cycles)."""
    levels = np.asarray(levels, dtype=int)
    if persistence <= 1:
        return levels.copy()
    out = levels.copy()
    for t in range(len(levels)):
        out[t] = levels[max(0, t - persistence + 1): t + 1].min()
    return out


def build_alert_timeline(
    frame: pd.DataFrame,
    policy: CriticalityPolicy,
    first_alarm_cycle: dict[int, int | None] | None = None,
    extra_alarms: dict[str, dict[int, int | None]] | None = None,
) -> pd.DataFrame:
    """Add true level and one level column per strategy to a per-cycle prediction frame.

    ``frame`` needs: unit_id, cycle, RUL_true, RUL_pred, RUL_lower, RUL_upper.
    ``first_alarm_cycle`` maps unit_id -> first persistent SPC alarm cycle (None = no alarm).
    ``extra_alarms`` adds one strategy ``interval+<name>`` per extra detector (same escalation rule).
    """
    first_alarm_cycle = first_alarm_cycle or {}
    parts = []
    for unit_id, eng in frame.sort_values(["unit_id", "cycle"]).groupby("unit_id", sort=False):
        eng = eng.copy()
        alarm_cycle = first_alarm_cycle.get(int(unit_id))
        eng["anomaly_active"] = _active(eng, alarm_cycle)
        eng["level_true"] = policy.level_from_rul(eng["RUL_true"])
        raw = {
            "point": policy.level_from_rul(eng["RUL_pred"]),
            "interval": policy.level_from_rul(eng["RUL_lower"]),
        }
        raw["interval+anomaly"] = _escalate(raw["interval"], eng["anomaly_active"].to_numpy(dtype=int))
        for name, alarm_map in (extra_alarms or {}).items():
            active = _active(eng, alarm_map.get(int(unit_id)))
            eng[f"{name}_active"] = active
            raw[f"interval+{name}"] = _escalate(raw["interval"], np.asarray(active, dtype=int))
        # anomaly escalation only applies when the interval already shows some risk (> NORMAL):
        # a lone SPC alarm in early life must not create a maintenance order by itself.
        for name, levels in raw.items():
            eng[f"level_{name}"] = debounce(levels, policy.persistence)
        parts.append(eng)
    return pd.concat(parts, ignore_index=True)


def _active(eng: pd.DataFrame, alarm_cycle) -> pd.Series | bool:
    if alarm_cycle is None or pd.isna(alarm_cycle):
        return pd.Series(False, index=eng.index)
    return eng["cycle"] >= alarm_cycle


def _escalate(levels: np.ndarray, active: np.ndarray) -> np.ndarray:
    return np.minimum(levels + active * (levels > NORMAL), CRITICAL)


STRATEGIES = ("point", "interval", "interval+anomaly")


def strategy_names(timeline: pd.DataFrame) -> list[str]:
    """Base strategies plus any ``interval+<detector>`` columns added through ``extra_alarms``."""
    extra = [c[len("level_"):] for c in timeline.columns if c.startswith("level_interval+") and c[len("level_"):] not in STRATEGIES]
    return [*STRATEGIES, *extra]


def evaluate_strategies(timeline: pd.DataFrame, policy: CriticalityPolicy, premature_rul: float = 60.0) -> pd.DataFrame:
    """Row-level and engine-level quality of each criticality strategy on held-out engines."""
    rows = []
    true = timeline["level_true"].to_numpy()
    for name in strategy_names(timeline):
        pred = timeline[f"level_{name}"].to_numpy()
        crit = true == CRITICAL
        normal = true == NORMAL
        lead_times, early, missed = [], 0, 0
        n_units = 0
        for _, eng in timeline.groupby("unit_id"):
            n_units += 1
            eng = eng.sort_values("cycle")
            eol = int(eng["cycle"].max())
            hit = eng.loc[eng[f"level_{name}"] >= WARNING]
            if hit.empty:
                missed += 1
                continue
            first = hit.iloc[0]
            if first["RUL_true"] > premature_rul:
                early += 1
            lead_times.append(eol - int(first["cycle"]))
        # classificação linha a linha: positivo = nível verdadeiro >= ALERTA; alerta = nível previsto >= ALERTA
        pos, called = true >= WARNING, pred >= WARNING
        tp, fp, fn = float(np.sum(pos & called)), float(np.sum(~pos & called)), float(np.sum(pos & ~called))
        rows.append({
            "Strategy": name,
            "Alert_precision": tp / (tp + fp) if tp + fp > 0 else float("nan"),
            "Alert_recall": tp / (tp + fn) if tp + fn > 0 else float("nan"),
            "Alert_F1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn > 0 else float("nan"),
            "Exact_level_accuracy": float((pred == true).mean()),
            "Within_one_level": float((np.abs(pred - true) <= 1).mean()),
            "Missed_critical_rate": float((pred[crit] < WARNING).mean()) if crit.any() else float("nan"),
            "False_alarm_rate_normal": float((pred[normal] >= WARNING).mean()) if normal.any() else float("nan"),
            "Engines": n_units,
            "Engines_never_alerted": missed,
            "Engines_premature_alert": early,
            "Median_warning_lead_time": float(np.median(lead_times)) if lead_times else float("nan"),
        })
    return pd.DataFrame(rows)


def generate_alert_events(timeline: pd.DataFrame, strategy: str = "interval+anomaly") -> pd.DataFrame:
    """One event per level ESCALATION (and per de-escalation to NORMAL) for each engine."""
    events = []
    col = f"level_{strategy}"
    for unit_id, eng in timeline.sort_values(["unit_id", "cycle"]).groupby("unit_id", sort=False):
        previous = NORMAL
        for _, row in eng.iterrows():
            level = int(row[col])
            if level > previous:
                events.append({
                    "unit_id": int(unit_id), "cycle": int(row["cycle"]),
                    "from_level": LEVEL_NAMES[previous], "to_level": LEVEL_NAMES[level],
                    "RUL_pred": float(row["RUL_pred"]), "RUL_lower": float(row["RUL_lower"]),
                    "RUL_upper": float(row["RUL_upper"]), "RUL_true": float(row["RUL_true"]),
                })
            previous = level
    return pd.DataFrame(events)



def maintenance_cost(timeline: pd.DataFrame, strategy: str, cost_ratio: float, lead: int = 0, trigger: int = CRITICAL) -> dict:
    """Renewal-reward cost of the rule "order maintenance at the first ``trigger`` level, done ``lead`` cycles later".

    The decision at cycle t is executed at t + lead (parts, crew, slot). If t + lead < EOL the engine is
    replaced as planned (cost 1, cycles used t + lead); otherwise it fails at EOL (cost ``cost_ratio``).
    By the renewal-reward theorem the long-run cost per operating cycle is sum(costs) / sum(cycles used);
    early replacement is penalised implicitly (fewer cycles of use for the same cost). Reported relative
    to an oracle that replaces every engine one cycle before failure (1 = perfect).
    """
    col = f"level_{strategy}"
    cost, used, oracle_used, failures, unused = 0.0, 0.0, 0.0, 0, []
    for _, eng in timeline.groupby("unit_id"):
        eng = eng.sort_values("cycle")
        eol = int(eng["cycle"].max())
        hit = eng.loc[eng[col] >= trigger, "cycle"]
        done = int(hit.iloc[0]) + lead if not hit.empty else None
        if done is None or done >= eol:
            cost += cost_ratio
            used += eol
            failures += 1
        else:
            cost += 1.0
            used += done
            unused.append(eol - done)
        oracle_used += eol - 1
    n = timeline["unit_id"].nunique()
    rate = cost / used
    oracle = n / oracle_used
    return {"Strategy": strategy, "Cost_ratio": cost_ratio, "Lead": lead, "Cost_rate": rate, "Relative_to_oracle": rate / oracle,
            "Failures": failures, "Engines": n, "Mean_unused_cycles": float(np.mean(unused)) if unused else float("nan")}


def cost_table(timeline: pd.DataFrame, ratios=(2, 5, 10, 20, 50, 100), leads=(0, 5, 10), trigger: int = CRITICAL) -> pd.DataFrame:
    """maintenance_cost for every strategy, failure/planned cost ratio and logistics lead time."""
    return pd.DataFrame([maintenance_cost(timeline, s, r, L, trigger) for s in strategy_names(timeline) for r in ratios for L in leads])

In [ ]:
%%writefile projeto/src/pdm_cmaps/monitoring/layer.py
"""Monitoring layer: RUL + uncertainty + criticality + alerts + grounded reports, and its figures.

This module only ORCHESTRATES the layer:

  held-out engines -> RUL model (+ conformal interval) -> criticality strategies -> alert events
                   -> structured facts -> deterministic report text

Every table written here answers one research question (RQ-A ... RQ-E):
  (RQ-E: SPC detection summary is recomputed here per seed, so repeated runs need no earlier stage.)
  rul_uncertainty_metrics.csv        RQ-B  does the interval reach its nominal coverage?
  criticality_strategy_comparison.csv  RQ-C  which criticality rule misses fewer critical states?
  criticality_policy_sensitivity.csv RQ-C  do conclusions survive other threshold choices?
  monitoring_repeated_summary.csv    repeated seeds x subsets, mean and std of all of the above
[LEAKAGE] The conformal calibration uses out-of-fold predictions on TRAINING engines only; all
metrics here are computed on the held-out engines of the same seeded engine split as pipeline.run().
"""

from __future__ import annotations

from pdm_cmaps.config import DEFAULT_RESULTS_DIR

from pathlib import Path

import numpy as np
import pandas as pd

from pdm_cmaps.monitoring.alerts import CRITICAL, LEVEL_NAMES, WARNING, CriticalityPolicy, build_alert_timeline, cost_table, evaluate_strategies, generate_alert_events
from pdm_cmaps.domain.schema import DEFAULT_HEALTHY_CYCLE_LIMIT, DEFAULT_RANDOM_STATE, DEFAULT_RUL_CAP
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.data.features import CausalFeatureEngineer
from pdm_cmaps.domain.rul import compute_piecewise_rul
from pdm_cmaps.data.splits import split_engine_units
from pdm_cmaps.modeling.registry import get_model_pool, grouped_oof
from pdm_cmaps.reporting.monitoring_figures import plot_model_ci, plot_regime_variance, plot_residual_control_chart, plot_rul_trajectories, plot_strategy_tradeoff, plot_strategy_tradeoff_repeated
from pdm_cmaps.detection.multivariate import MultivariateResidualMonitor
from pdm_cmaps.detection.regime import RegimeDecoupler
from pdm_cmaps.detection.virtual_sensor import VirtualSensor
from pdm_cmaps.monitoring.reports import build_facts, render_report
from pdm_cmaps.monitoring.uncertainty import ConformalRULInterval, NormalizedConformalRULInterval, causal_local_spread, interval_metrics

# Limite do T² da MEWMA nos motores nativos (H = 50): "synthetic" desde 2026-11-18 (decisão do autor, post hoc; ver docs/protocolo-primario.md).
# É um argumento explícito aqui porque o padrão da CLASSE continua "empirical": os estudos de injeção (Seção 5.5 e a parte de injeção da Seção 5.4) usam o limite empírico.
MEWMA_CALIBRATION_MONITORING = "synthetic"

# ============================================================================
# orchestration
# ============================================================================

def _policy_grid(base: CriticalityPolicy) -> dict[str, CriticalityPolicy]:
    """Sensitivity grid for [ASSUMPTION] thresholds: critical level at 10/15/20 cycles (x2, x4 for the others)."""
    grid = {}
    for crit in (10.0, 15.0, 20.0):
        grid[f"critical<={int(crit)}"] = CriticalityPolicy(attention=4 * crit, warning=2 * crit, critical=crit, persistence=base.persistence)
    return grid


def compute_fleet_alarms(train_df: pd.DataFrame, test_df: pd.DataFrame, subset: str, seed: int, healthy_cycle_limit: int):
    """SPC stage for T50: regime decoupling + healthy-calibrated virtual sensor + EWMA alarms on test engines.

    Same configuration as pipeline.run(), so with the same seed the alarms are identical.
    """
    n_regimes = 6 if subset.upper() in {"FD002", "FD004"} else 1
    regimes = RegimeDecoupler(n_clusters=n_regimes, random_state=seed).fit(train_df)
    virtual = VirtualSensor(target_sensor="T50", mode="exogenous", ewma_alpha=0.10, min_consecutive_breaches=2,
                            healthy_cycle_limit=healthy_cycle_limit, random_state=seed)
    trace, per_engine, summary = virtual.evaluate(regimes.transform(train_df), regimes.transform(test_df))
    alarms = {int(r.unit_id): (None if pd.isna(r.first_alarm_cycle) else int(r.first_alarm_cycle)) for r in per_engine.itertuples()}
    return alarms, trace, summary


def compute_monitoring(
    subset: str,
    seed: int = DEFAULT_RANDOM_STATE,
    healthy_cycle_limit: int = DEFAULT_HEALTHY_CYCLE_LIMIT,
    alpha: float = 0.10,
    policy: CriticalityPolicy | None = None,
    loader: CMapssLoader | None = None,
    mewma_calibration: str = MEWMA_CALIBRATION_MONITORING,
) -> dict:
    """Pure computation of the whole layer for one (subset, seed); no files are written."""
    policy = policy or CriticalityPolicy()
    loader = loader or CMapssLoader()
    frame = loader.load(subset)
    frame["RUL"] = compute_piecewise_rul(frame)
    train_units, test_units = split_engine_units(frame, test_ratio=0.25, random_state=seed)
    train_df = frame.loc[frame["unit_id"].isin(train_units)]
    test_df = frame.loc[frame["unit_id"].isin(test_units)]
    sensors = loader.informative_sensors(train_df)
    base_features = [*sensors, "setting_1", "setting_2", "setting_3", "cycle"]

    # RUL model with causal temporal features (best family in the benchmark)
    engineer = CausalFeatureEngineer(sensors, windows=(5, 10)).fit(train_df, healthy_cycle_limit)
    train_fe, test_fe = engineer.transform(train_df), engineer.transform(test_df)
    features = [*base_features, *[c for c in train_fe.columns if c not in train_df.columns]]
    estimator = get_model_pool(seed)["LightGBM"]

    # [CALIBRATION][LEAKAGE] conformal calibration on OOF predictions of TRAINING engines only
    _, oof = grouped_oof(train_fe, features, estimator=estimator, n_splits=5, random_state=seed)
    conformal = ConformalRULInterval(alpha=alpha).fit(train_fe["RUL"], np.clip(oof, 0, DEFAULT_RUL_CAP))
    from sklearn.base import clone
    model = clone(estimator).fit(train_fe[features].to_numpy(dtype=float), train_fe["RUL"].to_numpy(dtype=float))
    pred = np.clip(model.predict(test_fe[features].to_numpy(dtype=float)), 0, DEFAULT_RUL_CAP)
    lower, upper = conformal.predict_interval(pred)
    timeline_in = pd.DataFrame({
        "unit_id": test_fe["unit_id"].to_numpy(), "cycle": test_fe["cycle"].to_numpy(),
        "RUL_true": test_fe["RUL"].to_numpy(), "RUL_pred": pred, "RUL_lower": lower, "RUL_upper": upper,
    })
    metrics = interval_metrics(timeline_in["RUL_true"], lower, upper)

    # RQ-B extension: conditional-coverage variants, all calibrated on the same OOF predictions
    n_regimes = 6 if subset.upper() in {"FD002", "FD004"} else 1
    regimes = RegimeDecoupler(n_clusters=n_regimes, random_state=seed).fit(train_df)
    reg_tr = regimes.transform(train_fe)["regime_id"].to_numpy()
    reg_te = regimes.transform(test_fe)["regime_id"].to_numpy()
    oof_c = np.clip(oof, 0, DEFAULT_RUL_CAP)
    spread_tr = causal_local_spread(oof_c, train_fe["unit_id"].to_numpy())
    spread_te = causal_local_spread(pred, test_fe["unit_id"].to_numpy())
    variants = {
        "Mondrian RUL (padrão)": (lower, upper),
        "Mondrian RUL × regime": ConformalRULInterval(alpha=alpha).fit(train_fe["RUL"], oof_c, groups=reg_tr).predict_interval(pred, groups=reg_te),
        "Normalizado (dispersão local)": NormalizedConformalRULInterval(alpha=alpha).fit(train_fe["RUL"], oof_c, spread_tr).predict_interval(pred, spread_te),
    }
    unc_variants = []
    for name, (lo_v, hi_v) in variants.items():
        ins = (timeline_in["RUL_true"] >= lo_v) & (timeline_in["RUL_true"] <= hi_v)
        pe = ins.groupby(timeline_in["unit_id"]).mean()
        unc_variants.append({"Subset": subset, "Seed": seed, "Method": name, **interval_metrics(timeline_in["RUL_true"], lo_v, hi_v),
                             "Per_engine_coverage_p10": float(pe.quantile(0.10)), "Per_engine_coverage_min": float(pe.min())})
    inside = (timeline_in["RUL_true"] >= lower) & (timeline_in["RUL_true"] <= upper)
    per_engine_cov = inside.groupby(timeline_in["unit_id"]).mean()
    rul_metrics = {
        "RMSE": float(np.sqrt(np.mean((timeline_in["RUL_true"] - pred) ** 2))),
        "MAE": float(np.mean(np.abs(timeline_in["RUL_true"] - pred))),
    }
    unc = {
        "Subset": subset, "Seed": seed, "Nominal_coverage": 1 - alpha, **metrics, **rul_metrics,
        "Per_engine_coverage_p10": float(per_engine_cov.quantile(0.10)),
        "Per_engine_coverage_median": float(per_engine_cov.median()),
        "Calibration_rows": int(len(train_fe)), "Test_engines": int(len(test_units)),
    }

    # SPC stage -> anomaly flags (RQ-E): univariate T50 chart and multivariate MEWMA over 13 sensors
    alarms, trace, spc = compute_fleet_alarms(train_df, test_df, subset, seed, healthy_cycle_limit)
    mv = MultivariateResidualMonitor(healthy_cycle_limit=healthy_cycle_limit, random_state=seed, calibration=mewma_calibration).fit(regimes.transform(train_df))
    mv_alarms, _, mv_summary = mv.alarms(mv.score(regimes.transform(test_df)))
    extra = {"anomaly_mv": mv_alarms}

    # RQ-C: criticality strategies, sensitivity to the threshold policy, and decision cost per policy
    timeline = build_alert_timeline(timeline_in, policy, alarms, extra)
    comparison = evaluate_strategies(timeline, policy)
    sens, costs, timelines = [], [], []
    for name, pol in _policy_grid(policy).items():
        tl = build_alert_timeline(timeline_in, pol, alarms, extra)
        sens.append(evaluate_strategies(tl, pol).assign(Policy=name))
        costs.append(cost_table(tl).assign(Policy=name))
        keep = ["unit_id", "cycle", "RUL_true", "RUL_pred", "RUL_lower", "RUL_upper", *[c for c in tl.columns if c.startswith("level_")]]
        timelines.append(tl[keep].assign(Policy=name))
    return {
        "timeline": timeline, "uncertainty": unc, "strategies": comparison,
        "sensitivity": pd.concat(sens, ignore_index=True), "events": generate_alert_events(timeline, "interval+anomaly"),
        "alarms": alarms, "trace": trace, "spc_summary": spc, "test_units": [int(u) for u in test_units], "policy": policy,
        "uncertainty_variants": pd.DataFrame(unc_variants), "spc_mv_summary": mv_summary, "mv_alarms": mv_alarms,
        "costs": pd.concat(costs, ignore_index=True), "timelines": pd.concat(timelines, ignore_index=True),
    }


def run_monitoring_layer(
    subset: str,
    output_dir: Path,
    seed: int = DEFAULT_RANDOM_STATE,
    healthy_cycle_limit: int = DEFAULT_HEALTHY_CYCLE_LIMIT,
    alpha: float = 0.10,
    policy: CriticalityPolicy | None = None,
    sample_engine_id: int | None = 179,
    max_report_events: int = 60,
    loader: CMapssLoader | None = None,
) -> dict[str, pd.DataFrame]:
    """Compute the layer for one subset/seed and write its tables, example reports and figures."""
    table_dir, figure_dir = Path(output_dir) / "tables", Path(output_dir) / "figures"
    table_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)
    print(f"[monitoring] RUL + uncertainty + criticality + alerts for {subset} (alpha={alpha})", flush=True)
    res = compute_monitoring(subset, seed, healthy_cycle_limit, alpha, policy, loader)
    policy, timeline, alarms = res["policy"], res["timeline"], res["alarms"]

    unc = pd.DataFrame([res["uncertainty"]])
    unc.to_csv(table_dir / "rul_uncertainty_metrics.csv", index=False)
    pd.DataFrame([res["spc_summary"]]).to_csv(table_dir / "spc_detection_summary_t50.csv", index=False)
    timeline.to_csv(table_dir / "alert_timeline.csv", index=False)
    res["strategies"].to_csv(table_dir / "criticality_strategy_comparison.csv", index=False)
    res["sensitivity"].to_csv(table_dir / "criticality_policy_sensitivity.csv", index=False)
    events = res["events"]
    events.to_csv(table_dir / "alert_events.csv", index=False)
    u = res["uncertainty"]
    print(f"  -> interval coverage {u['coverage']:.3f} (nominal {1 - alpha:.2f}) | critical-zone {u['coverage_critical_zone']:.3f} | width {u['mean_width']:.1f}", flush=True)
    print(res["strategies"][["Strategy", "Missed_critical_rate", "False_alarm_rate_normal", "Median_warning_lead_time"]].round(3).to_string(index=False), flush=True)

    # alert messages (facts -> deterministic text)
    warn_events = events.loc[events["to_level"].isin([LEVEL_NAMES[WARNING], LEVEL_NAMES[CRITICAL]])]
    if len(warn_events) > max_report_events:
        warn_events = warn_events.sample(max_report_events, random_state=seed)
    messages = []
    for ev in warn_events.to_dict("records"):
        history = timeline.loc[(timeline["unit_id"] == ev["unit_id"]) & (timeline["cycle"] <= ev["cycle"])]
        facts = build_facts(history, policy, coverage_pct=100 * (1 - alpha))
        messages.append((ev, facts, render_report(facts)))
    if messages:
        pd.DataFrame([{**f, "message": m} for _, f, m in messages]).drop(columns=["thresholds"]).to_csv(table_dir / "alert_messages.csv", index=False)
        (table_dir / "example_alert_messages.md").write_text(
            "# Example alert messages\n\n" + "\n\n".join(f"- {m}" for _, _, m in messages[:3]) + "\n", encoding="utf-8")

    # figures (residual / RUL scale; same sample engine across detection figures)
    test_units = res["test_units"]
    sample = sample_engine_id if sample_engine_id in set(test_units) else int(sorted(test_units)[0])
    plot_residual_control_chart(res["trace"], sample, alarms.get(sample), figure_dir / "fig_residual_control_chart.png", healthy_cycle_limit)
    eol = timeline.groupby("unit_id")["cycle"].max().sort_values()
    picks = [int(eol.index[int(q * (len(eol) - 1))]) for q in (0.1, 0.4, 0.7, 0.95)]
    plot_rul_trajectories(timeline, list(dict.fromkeys(picks)), figure_dir / "fig_rul_trajectories_intervals.png", alpha=alpha)
    plot_strategy_tradeoff(res["sensitivity"], figure_dir / "fig_criticality_strategy_tradeoff.png")
    ci_path = table_dir / "thesis_bootstrap_ci.csv"
    if ci_path.exists():
        plot_model_ci(pd.read_csv(ci_path), figure_dir / "fig_model_comparison_ci.png")
    return {"uncertainty": unc, "strategies": res["strategies"], "sensitivity": res["sensitivity"], "events": events}


def compute_regime_variance(subsets, seeds, loader: CMapssLoader | None = None) -> pd.DataFrame:
    """Share of T50 variance explained by flight regime on held-out engines (needs only KMeans, no RUL model)."""
    loader = loader or CMapssLoader()
    rows = []
    for subset in subsets:
        frame = loader.load(subset)
        k = 6 if subset.upper() in {"FD002", "FD004"} else 1
        for seed in seeds:
            train_units, test_units = split_engine_units(frame, test_ratio=0.25, random_state=seed)
            train_df, test_df = frame.loc[frame["unit_id"].isin(train_units)], frame.loc[frame["unit_id"].isin(test_units)]
            decoupler = RegimeDecoupler(n_clusters=k, random_state=seed).fit(train_df)
            v = RegimeDecoupler.variance_decomposition(decoupler.transform(test_df), "T50", reference_frame=decoupler.transform(train_df))
            rows.append({"Subset": subset, "Seed": seed, "Regimes": k, "Between_regime_pct": v["pct_between_regime"],
                         "Within_regime_pct": v["pct_within_regime"], "OOS_R2_regime_mean": v["oos_r2"]})
    return pd.DataFrame(rows)


def run_regime_variance(subsets, seeds, output_dir: Path) -> pd.DataFrame:
    out = Path(output_dir)
    (out / "tables").mkdir(parents=True, exist_ok=True)
    (out / "figures").mkdir(parents=True, exist_ok=True)
    table = compute_regime_variance(subsets, seeds)
    table.to_csv(out / "tables" / "regime_variance_by_subset_raw.csv", index=False)
    summary = table.groupby(["Subset", "Regimes"], as_index=False).agg(
        Between_regime_pct_mean=("Between_regime_pct", "mean"), Between_regime_pct_std=("Between_regime_pct", "std"),
        OOS_R2_mean=("OOS_R2_regime_mean", "mean"), Runs=("Seed", "size"))
    summary.to_csv(out / "tables" / "regime_variance_by_subset.csv", index=False)
    plot_regime_variance(table, out / "figures" / "fig_regime_variance_by_subset.png")
    return summary


RUN_PARTS = ("uncertainty", "uncertainty_variants", "spc", "spc_mv", "sensitivity", "costs")


def _run_dir(out: Path, subset: str, seed: int) -> Path:
    return out / "runs" / f"{subset}_seed{seed}"


def run_one(subset: str, seed: int, out: Path, alpha: float, healthy_cycle_limit: int, mewma_calibration: str = MEWMA_CALIBRATION_MONITORING) -> None:
    """Compute one (subset, seed) and write its tables at once, so a crash never loses finished runs."""
    d = _run_dir(out, subset, seed)
    if all((d / f"{part}.csv").exists() for part in RUN_PARTS):
        print(f"[repeat] {subset} seed={seed} already done, skipping", flush=True)
        return
    print(f"[repeat] {subset} seed={seed}", flush=True)
    res = compute_monitoring(subset, seed, healthy_cycle_limit, alpha, mewma_calibration=mewma_calibration)
    d.mkdir(parents=True, exist_ok=True)
    tag = {"Subset": subset, "Seed": seed}
    tables = {
        "uncertainty": pd.DataFrame([res["uncertainty"]]),
        "uncertainty_variants": res["uncertainty_variants"],
        "spc": pd.DataFrame([{**tag, **res["spc_summary"]}]),
        "spc_mv": pd.DataFrame([{**tag, **res["spc_mv_summary"]}]),
        "sensitivity": res["sensitivity"].assign(**tag),
        "costs": res["costs"].assign(**tag),
    }
    res["timelines"].assign(**tag).to_csv(d / "timelines.csv.gz", index=False)
    for part in reversed(RUN_PARTS):  # "uncertainty" written last marks the run complete
        tables[part].to_csv(d / f"{part}.csv", index=False)
    print(f"[repeat] {subset} seed={seed} saved", flush=True)


def _mean_std(df, keys, cols):
    agg = df.groupby(keys)[cols].agg(["mean", "std"])
    agg.columns = [f"{c}_{s}" for c, s in agg.columns]
    agg["Runs"] = df.groupby(keys).size()
    return agg.reset_index()


def aggregate_repeated(out: Path) -> pd.DataFrame:
    """Read every finished run under ``out/runs`` and write the raw + mean/std tables and the figure."""
    out = Path(out)
    (out / "tables").mkdir(parents=True, exist_ok=True)
    (out / "figures").mkdir(parents=True, exist_ok=True)
    loaded = {part: [] for part in RUN_PARTS}
    for d in sorted((out / "runs").glob("*_seed*")):
        if all((d / f"{part}.csv").exists() for part in RUN_PARTS):
            for part in RUN_PARTS:
                loaded[part].append(pd.read_csv(d / f"{part}.csv"))
    frames = {part: pd.concat(v, ignore_index=True) for part, v in loaded.items() if v}
    unc, strat, spc = frames["uncertainty"], frames["sensitivity"], frames["spc"]
    unc.to_csv(out / "tables" / "monitoring_repeated_uncertainty_raw.csv", index=False)
    strat.to_csv(out / "tables" / "monitoring_repeated_strategies_raw.csv", index=False)
    spc.to_csv(out / "tables" / "monitoring_repeated_spc_raw.csv", index=False)
    unc_cols = ["coverage", "coverage_critical_zone", "mean_width", "Per_engine_coverage_p10", "RMSE", "MAE"]
    _mean_std(unc, ["Subset"], unc_cols).to_csv(out / "tables" / "monitoring_repeated_uncertainty_summary.csv", index=False)
    strat_cols = ["Alert_precision", "Alert_recall", "Alert_F1", "Missed_critical_rate", "False_alarm_rate_normal", "Within_one_level", "Median_warning_lead_time", "Engines_never_alerted", "Engines_premature_alert"]
    strat_summary = _mean_std(strat, ["Subset", "Policy", "Strategy"], strat_cols)
    strat_summary.to_csv(out / "tables" / "monitoring_repeated_summary.csv", index=False)
    spc_cols = ["detection_rate_pct", "mean_lead_time_cycles", "mean_healthy_false_alarm_pct", "premature_alarm_pct"]
    _mean_std(spc, ["Subset"], spc_cols).to_csv(out / "tables" / "monitoring_repeated_spc_summary.csv", index=False)
    if "spc_mv" in frames:
        frames["spc_mv"].to_csv(out / "tables" / "monitoring_repeated_spc_mv_raw.csv", index=False)
        _mean_std(frames["spc_mv"], ["Subset"], spc_cols).to_csv(out / "tables" / "monitoring_repeated_spc_mv_summary.csv", index=False)
    if "uncertainty_variants" in frames:
        v = frames["uncertainty_variants"]
        v.to_csv(out / "tables" / "monitoring_repeated_uncertainty_variants_raw.csv", index=False)
        _mean_std(v, ["Subset", "Method"], ["coverage", "coverage_critical_zone", "mean_width", "Per_engine_coverage_p10", "Per_engine_coverage_min"]).to_csv(
            out / "tables" / "monitoring_repeated_uncertainty_variants_summary.csv", index=False)
    if "costs" in frames:
        c = frames["costs"]
        c.to_csv(out / "tables" / "monitoring_repeated_costs_raw.csv", index=False)
        _mean_std(c, ["Subset", "Policy", "Strategy", "Cost_ratio", "Lead"], ["Relative_to_oracle", "Failures", "Mean_unused_cycles"]).to_csv(
            out / "tables" / "monitoring_repeated_costs_summary.csv", index=False)
    plot_strategy_tradeoff_repeated(strat, out / "figures" / "fig_criticality_tradeoff_repeated.png")
    return strat_summary


def run_repeated_experiments(subsets, seeds, output_dir: Path, alpha: float = 0.10, healthy_cycle_limit: int = DEFAULT_HEALTHY_CYCLE_LIMIT, jobs: int = 1,
                            mewma_calibration: str = MEWMA_CALIBRATION_MONITORING) -> pd.DataFrame:
    """Repeat the layer over subsets x seeds (each run saved as it finishes) and report mean +- std."""
    out = Path(output_dir)
    jobs_list = [(subset, seed) for subset in subsets for seed in seeds]
    if jobs > 1:
        from joblib import Parallel, delayed
        Parallel(n_jobs=jobs)(delayed(run_one)(subset, seed, out, alpha, healthy_cycle_limit, mewma_calibration) for subset, seed in jobs_list)
    else:
        for subset, seed in jobs_list:
            run_one(subset, seed, out, alpha, healthy_cycle_limit, mewma_calibration)
    return aggregate_repeated(out)


def main(argv: list[str] | None = None) -> int:
    import argparse

    parser = argparse.ArgumentParser(description="Repeat the monitoring layer over subsets and seeds (mean +- std).")
    parser.add_argument("--subsets", nargs="+", default=["FD001", "FD002", "FD003", "FD004"])
    parser.add_argument("--seeds", nargs="+", type=int, default=[42, 7, 123])
    parser.add_argument("--jobs", type=int, default=3, help="parallel (subset, seed) runs")
    parser.add_argument("--regime-variance-only", action="store_true", help="only the T50 regime-variance table and figure (fast)")
    parser.add_argument("--output-dir", type=Path, default=DEFAULT_RESULTS_DIR / "monitoring_repeated")
    parser.add_argument("--mewma-calibration", choices=("empirical", "synthetic"), default=MEWMA_CALIBRATION_MONITORING,
                        help="limite do T² da MEWMA (padrão do protocolo para o monitoramento: synthetic; empirical reproduz a rodada anterior)")
    parser.add_argument("--aggregate-only", action="store_true", help="rebuild the summary tables from finished runs")
    args = parser.parse_args(argv)
    if args.aggregate_only:
        print(aggregate_repeated(args.output_dir).round(3).to_string(index=False))
        return 0
    if args.regime_variance_only:
        print(run_regime_variance(args.subsets, args.seeds, args.output_dir).round(3).to_string(index=False))
        return 0
    summary = run_repeated_experiments(args.subsets, args.seeds, args.output_dir, jobs=args.jobs, mewma_calibration=args.mewma_calibration)
    print(summary.round(3).to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/monitoring/reports.py
"""Alert facts and deterministic report text for the maintenance engineer.

WHAT   Packs everything the models know about one asset at one cycle into a small, causal facts
       dictionary (RUL estimate and interval, trend, anomaly status, criticality level, action)
       and renders it as a short Portuguese message used in alert e-mails and by the portal.
WHY    The text is generated from the facts by a fixed template, so every number in it comes
       straight from a model output: nothing is estimated or invented at this stage.
CONCEPTS
  [CAUSALITY]    facts use only rows up to the alert cycle (no future information).
  [DECISION-POLICY] the recommended action is a fixed mapping from the criticality level, a
                 maintenance-policy choice that the engineer can change in ACTIONS_PT.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from pdm_cmaps.monitoring.alerts import LEVEL_NAMES, CriticalityPolicy

LEVEL_LABELS_PT = ("NORMAL", "ATENÇÃO", "ALERTA", "CRÍTICO")
ACTIONS_PT = (
    "manter o monitoramento de rotina",
    "programar uma inspeção na próxima janela de manutenção",
    "planejar a manutenção e reservar peças e equipe",
    "providenciar intervenção imediata",
)
TREND_WINDOW = 10


def build_facts(history: pd.DataFrame, policy: CriticalityPolicy, strategy: str = "interval+anomaly", coverage_pct: float = 90.0) -> dict:
    """Structured, causal facts for one asset at its latest cycle (only past/present rows used)."""
    h = history.sort_values("cycle")
    last = h.iloc[-1]
    window = h.tail(TREND_WINDOW)
    slope = float(np.polyfit(window["cycle"], window["RUL_pred"], 1)[0]) if len(window) >= 3 else 0.0
    anomaly_rows = h.loc[h["anomaly_active"]]
    since = int(last["cycle"] - anomaly_rows["cycle"].min()) if not anomaly_rows.empty else None
    level = int(last[f"level_{strategy}"])
    return {
        "unit_id": int(last["unit_id"]),
        "cycle": int(last["cycle"]),
        "rul_estimate": round(float(last["RUL_pred"])),
        "rul_lower": round(float(last["RUL_lower"])),
        "rul_upper": round(float(last["RUL_upper"])),
        "interval_coverage_pct": round(float(coverage_pct)),
        "trend_cycles_per_cycle": round(slope, 1),
        "trend_window_cycles": TREND_WINDOW,
        "anomaly_active": bool(last["anomaly_active"]),
        "cycles_since_anomaly": since,
        "level_index": level,
        "level_name": LEVEL_NAMES[level],
        "level_label_pt": LEVEL_LABELS_PT[level],
        "recommended_action_pt": ACTIONS_PT[level],
        "thresholds": {"attention": policy.attention, "warning": policy.warning, "critical": policy.critical},
    }


def render_report(facts: dict) -> str:
    """Deterministic message for e-mail / portal; all numbers are copied from ``facts``."""
    anomaly = (
        f"Há anomalia ativa há {facts['cycles_since_anomaly']} ciclos."
        if facts["anomaly_active"] and facts["cycles_since_anomaly"] is not None
        else "Não há anomalia ativa no monitoramento estatístico."
    )
    return (
        f"Ativo {facts['unit_id']}, ciclo {facts['cycle']}. "
        f"Nível de criticidade: {facts['level_label_pt']}. "
        f"RUL estimada de {facts['rul_estimate']} ciclos (intervalo de {facts['interval_coverage_pct']}%: "
        f"{facts['rul_lower']} a {facts['rul_upper']} ciclos). "
        f"Tendência da RUL estimada: {facts['trend_cycles_per_cycle']} ciclos por ciclo nos últimos {facts['trend_window_cycles']} ciclos. "
        f"{anomaly} Ação recomendada: {facts['recommended_action_pt']}."
    )

In [ ]:
%%writefile projeto/src/pdm_cmaps/monitoring/uncertainty.py
"""Uncertainty for RUL predictions: split-conformal prediction intervals.

WHAT   Turns a point prediction "RUL = 42 cycles" into an interval "RUL in [28, 61] with ~90%
       coverage". The interval, not the point, is what an engineer should act on.
WHY    Research question RQ-B: how should uncertainty be presented and used? A conservative
       (lower-bound) rule needs an interval whose coverage is verified on held-out engines.
CONCEPTS
  [CALIBRATION]  split conformal prediction: absolute/signed errors on a CALIBRATION set give
                 the interval half-widths. No distributional assumption on the errors.
  [LEAKAGE]      the calibration set must be out-of-fold predictions on TRAINING engines
                 (models.grouped_oof). Never calibrate on the engines used to report coverage.
  [PSEUDOREPLICATION] cycles of one engine are correlated, so the 90% guarantee is only
                 approximate; we therefore REPORT empirical coverage on held-out engines
                 (overall, in the critical zone, and per engine) instead of trusting the theory.
  Mondrian bins  RUL errors are larger when the true RUL is large (early life). Calibrating
                 separately in bins of the PREDICTED RUL gives locally valid widths; an optional
                 second key (e.g. the operating regime) gives per-regime validity.
  Normalized     NormalizedConformalRULInterval scales residuals by a causal difficulty score
                 (recent prediction spread of the engine) so widths adapt engine by engine.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

DEFAULT_RUL_CAP = 125.0


class ConformalRULInterval:
    """Asymmetric split-conformal intervals with Mondrian bins on the predicted RUL."""

    def __init__(self, alpha: float = 0.10, n_bins: int = 3, rul_cap: float = DEFAULT_RUL_CAP):
        if not 0.0 < alpha < 1.0:
            raise ValueError("alpha must be in (0, 1)")
        if n_bins < 1:
            raise ValueError("n_bins must be >= 1")
        self.alpha = float(alpha)
        self.n_bins = int(n_bins)
        self.rul_cap = float(rul_cap)
        self.bin_edges_: np.ndarray | None = None
        self.offsets_: list[tuple[float, float]] = []

    def _bin_index(self, pred: np.ndarray) -> np.ndarray:
        assert self.bin_edges_ is not None
        return np.clip(np.searchsorted(self.bin_edges_, pred, side="right") - 1, 0, self.n_bins - 1)

    def fit(self, y_cal, pred_cal, groups=None) -> "ConformalRULInterval":
        """Learn per-bin signed-residual quantiles from calibration (out-of-fold) predictions.

        ``groups`` (optional, e.g. the operating regime of each row) adds a second Mondrian key: offsets are
        learned per (group, RUL bin) and fall back to the RUL bin alone when a cell has < 20 rows.
        """
        y = np.asarray(y_cal, dtype=float)
        pred = np.asarray(pred_cal, dtype=float)
        if y.shape != pred.shape or y.size < 20 * self.n_bins:
            raise ValueError("need matching calibration arrays with >= 20 points per bin")
        quantiles = np.linspace(0.0, 1.0, self.n_bins + 1)
        self.bin_edges_ = np.quantile(pred, quantiles)
        self.bin_edges_[0], self.bin_edges_[-1] = -np.inf, np.inf
        idx = self._bin_index(pred)
        residual = y - pred
        self.offsets_ = [self._offsets(residual[idx == b]) for b in range(self.n_bins)]
        self.group_offsets_ = {}
        if groups is not None:
            g = np.asarray(groups)
            for key in np.unique(g):
                for b in range(self.n_bins):
                    r = residual[(g == key) & (idx == b)]
                    if r.size >= 20:
                        self.group_offsets_[(key, b)] = self._offsets(r)
        return self

    def _offsets(self, r: np.ndarray) -> tuple[float, float]:
        n = r.size
        # finite-sample conformal levels (Vovk et al.); 'higher'/'lower' keep the guarantee conservative
        hi_level = min(1.0, np.ceil((n + 1) * (1.0 - self.alpha / 2.0)) / n)
        lo_level = max(0.0, np.floor((n + 1) * (self.alpha / 2.0)) / n)
        return float(np.quantile(r, lo_level, method="lower")), float(np.quantile(r, hi_level, method="higher"))

    def predict_interval(self, pred, groups=None) -> tuple[np.ndarray, np.ndarray]:
        """Return (lower, upper) clipped to the physical RUL range [0, cap]."""
        if self.bin_edges_ is None:
            raise RuntimeError("fit the interval model before predicting")
        p = np.asarray(pred, dtype=float)
        idx = self._bin_index(p)
        g = np.asarray(groups) if groups is not None else None
        offs = [
            getattr(self, "group_offsets_", {}).get((g[k], i), self.offsets_[i]) if g is not None else self.offsets_[i]
            for k, i in enumerate(idx)
        ]
        lo = np.array([o[0] for o in offs])
        hi = np.array([o[1] for o in offs])
        return np.clip(p + lo, 0.0, self.rul_cap), np.clip(p + hi, 0.0, self.rul_cap)


def causal_local_spread(pred, units, window: int = 10) -> np.ndarray:
    """Per-row difficulty score from the PAST of each engine: rolling std of its last ``window`` predictions.

    [CAUSALITY] uses only predictions up to the current cycle of the same engine (rows must be sorted by
    unit and cycle). Unstable recent predictions signal a harder case and get a wider interval.
    """
    s = pd.Series(np.asarray(pred, dtype=float))
    out = s.groupby(np.asarray(units)).transform(lambda v: v.rolling(window, min_periods=2).std())
    return out.fillna(0.0).to_numpy()


class NormalizedConformalRULInterval(ConformalRULInterval):
    """Locally adaptive split conformal: residuals are divided by a difficulty score d = spread + floor.

    Interval = pred + q * d, with q the per-bin conformal quantiles of r / d. Same finite-sample marginal
    guarantee as the plain version; widths now vary engine by engine (Lei et al., 2018, section 5.2).
    The floor (median calibration spread) keeps d away from zero on flat, early-life predictions.
    """

    def fit(self, y_cal, pred_cal, spread_cal) -> "NormalizedConformalRULInterval":
        spread = np.asarray(spread_cal, dtype=float)
        self.floor_ = float(max(np.median(spread), 1e-6))
        d = spread + self.floor_
        y = np.asarray(y_cal, dtype=float)
        p = np.asarray(pred_cal, dtype=float)
        # reuse the parent machinery on normalized residuals: pseudo-target p + (y - p) / d
        super().fit(p + (y - p) / d, p)
        return self

    def predict_interval(self, pred, spread) -> tuple[np.ndarray, np.ndarray]:
        p = np.asarray(pred, dtype=float)
        d = np.asarray(spread, dtype=float) + self.floor_
        idx = self._bin_index(p)
        lo = np.array([self.offsets_[i][0] for i in idx]) * d
        hi = np.array([self.offsets_[i][1] for i in idx]) * d
        return np.clip(p + lo, 0.0, self.rul_cap), np.clip(p + hi, 0.0, self.rul_cap)


def interval_metrics(y_true, lower, upper, critical_threshold: float = 50.0) -> dict[str, float]:
    """Empirical coverage and width, overall and in the critical zone (true RUL < threshold)."""
    y = np.asarray(y_true, dtype=float)
    lo = np.asarray(lower, dtype=float)
    hi = np.asarray(upper, dtype=float)
    inside = (y >= lo) & (y <= hi)
    crit = y < critical_threshold
    return {
        "coverage": float(inside.mean()),
        "mean_width": float((hi - lo).mean()),
        "coverage_critical_zone": float(inside[crit].mean()) if crit.any() else float("nan"),
        "mean_width_critical_zone": float((hi - lo)[crit].mean()) if crit.any() else float("nan"),
    }

### 2.2 Pacote `pdm_cmaps`: analysis

testes estatísticos e bootstrap

2 arquivos.

In [ ]:
%%writefile projeto/src/pdm_cmaps/analysis/__init__.py
"""Statistical inference: hypothesis tests, BCa cluster bootstrap and diagnostics (stats.py)."""

In [ ]:
%%writefile projeto/src/pdm_cmaps/analysis/stats.py
"""Statistical hypothesis testing (Friedman, Wilcoxon, Ljung-Box, BH-FDR) and engine-cluster BCa bootstrap CIs.

WHAT   Turns "the plot looks different" into a testable statement with a p-value and an effect
       size. One test per thesis hypothesis:
         H1 regime explains sensor variance  -> Friedman test across regimes, blocked by engine
         H2 residuals shift near end of life -> paired t-test / Wilcoxon on engine means
         H3 healthy residuals are white noise-> Ljung-Box per engine + Fisher combination
         H4 model beats the mean baseline    -> paired Wilcoxon on per-engine errors
         H5 XGBoost vs deep models           -> two-sided paired Wilcoxon
WHY    With one fleet and one split, differences between models can be luck. Tests and
       confidence intervals say how much luck could explain.
CONCEPTS
  [PSEUDOREPLICATION] 200 cycles of one engine are NOT 200 independent samples. All tests first
                 aggregate to one number per engine, so n = engines.
  Paired tests   the same engines are scored by both models, so compare per-engine differences.
  Non-parametric Wilcoxon avoids assuming normal errors; Shapiro-Wilk picks t-test vs Wilcoxon.
  Effect size    p-values shrink with n; always read the effect (Kendall's W, Hodges-Lehmann).
  Bootstrap BCa  resample ENGINES (clusters) with replacement to get a confidence interval of
                 R²/RMSE; bias-correction and acceleration fix skew.
  Multiple testing  many tests -> some false positives; Benjamini-Hochberg controls the false
                 discovery rate. "Not rejected" is NOT "proved equal".
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

from pdm_cmaps.domain.rul import regression_metrics
from pdm_cmaps.domain.schema import DEFAULT_RANDOM_STATE
from pdm_cmaps.modeling.registry import grouped_oof


# ============================================================================
# thesis_statistics
# ============================================================================

def hodges_lehmann_estimator(deltas: np.ndarray) -> float:
    """Compute Hodges-Lehmann pseudo-median from pairwise Walsh averages."""
    arr = np.asarray(deltas, dtype=float)
    n = len(arr)
    if n == 0:
        return 0.0
    pairwise = [(arr[i] + arr[j]) / 2.0 for i in range(n) for j in range(i, n)]
    return float(np.median(pairwise))


def _bca_interval(bootstrap_values: list[float], jackknife_values: list[float], observed: float) -> tuple[float, float]:
    """Bias-corrected and accelerated interval for a cluster-resampled statistic."""
    boot = np.asarray(bootstrap_values, dtype=float)
    jack = np.asarray(jackknife_values, dtype=float)
    eps = 0.5 / len(boot)
    bias = stats.norm.ppf(np.clip(np.mean(boot < observed), eps, 1.0 - eps))
    centered = jack.mean() - jack
    denom = 6.0 * np.sum(centered ** 2) ** 1.5
    acceleration = float(np.sum(centered ** 3) / denom) if denom > 0 else 0.0
    quantiles = []
    for alpha in (0.025, 0.975):
        z = stats.norm.ppf(alpha)
        adjusted = stats.norm.cdf(bias + (bias + z) / (1.0 - acceleration * (bias + z)))
        quantiles.append(float(np.clip(adjusted, 0.0, 1.0)))
    return tuple(float(x) for x in np.quantile(boot, quantiles))


@dataclass
class HypothesisTestResult:
    """Container for hypothesis test outcomes."""

    hypothesis_id: str
    description: str
    h0: str
    h1: str
    test_name: str
    test_statistic: float
    p_value: float
    significance_level: float = 0.05
    effect_size: float | None = None
    conclusion: str = ""

    @property
    def is_rejected(self) -> bool:
        return self.p_value < self.significance_level

    def to_dict(self) -> dict[str, Any]:
        p_formatted = f"{self.p_value:.2e}" if self.p_value < 1e-4 else f"{self.p_value:.4f}"
        if self.p_value == 0.0:
            p_formatted = "< 1e-300"
        return {
            "ID": self.hypothesis_id,
            "Description": self.description,
            "H0": self.h0,
            "H1": self.h1,
            "Test": self.test_name,
            "Statistic": round(float(self.test_statistic), 4),
            "p_value": p_formatted,
            "p_value_raw": float(self.p_value),
            "H0_Rejected": "REJEITADA (p < 0.05)" if self.is_rejected else "NÃO REJEITADA",
            "Effect_Size": round(float(self.effect_size), 4) if self.effect_size is not None else np.nan,
            "Scientific_Conclusion": self.conclusion,
        }


class ThesisHypothesisAnalyzer:
    """Comprehensive hypothesis testing and statistical analysis engine."""

    def __init__(self, random_state: int = 42):
        self.random_state = random_state
        self.results: list[HypothesisTestResult] = []

    def test_regime_variance_dominance(
        self,
        df: pd.DataFrame,
        sensor: str = "T50",
        regime_col: str = "regime_id",
        unit_col: str = "unit_id",
    ) -> HypothesisTestResult:
        """Hypothesis 1: Operating regime explains statistically significant variance in sensor readings.
        
        Evaluated on repeated engine blocks to eliminate cycle pseudoreplication.
        """
        # If engine unit_id is present, perform repeated-measures engine-block Friedman test
        if unit_col in df.columns and df[unit_col].nunique() >= 2:
            pivot = df.groupby([unit_col, regime_col])[sensor].mean().unstack()
            # Drop incomplete engine rows across regimes
            clean_matrix = pivot.dropna()
            if len(clean_matrix) >= 2 and clean_matrix.shape[1] >= 2:
                cols = [clean_matrix[c].to_numpy() for c in clean_matrix.columns]
                stat, p_val = stats.friedmanchisquare(*cols)
                n_engines = len(clean_matrix)
                k_regimes = clean_matrix.shape[1]
                kendall_w = float(stat / (n_engines * (k_regimes - 1)))
                res = HypothesisTestResult(
                    hypothesis_id="H1_REGIME_VARIANCE",
                    description="Dominância da Variação de Altitude/Mach sobre os Sensores Térmicos",
                    h0="A altitude e velocidade de voo não afetam as leituras térmicas nominais entre motores.",
                    h1="A atmosfera ISA e regimes de voo causam variação altamente significativa (p < 0.001).",
                    test_name=f"Repeated-Measures Friedman Test (n={n_engines} engines, k={k_regimes} regimes)",
                    test_statistic=stat,
                    p_value=p_val,
                    effect_size=kendall_w,
                    conclusion=(
                        f"Efeito intra-motor entre regimes: p={p_val:.2e} (Friedman chi²={stat:.1f}, "
                        f"Kendall's W={kendall_w:.4f}, n={n_engines} motores). "
                        f"Confirmada dominância de regime livre de pseudorreplicação amostral."
                    ),
                )
                self.results.append(res)
                return res

        # Fallback to grouped one-way ANOVA / Alexander-Govern if no engine column
        groups = [group[sensor].dropna().to_numpy(dtype=float) for _, group in df.groupby(regime_col)]
        groups = [group for group in groups if group.size > 1]
        if len(groups) < 2:
            raise ValueError("At least two regimes with two observations each are required")
        levene_stat, levene_p = stats.levene(*groups, center="median")
        if levene_p < 0.05:
            robust_result = stats.alexandergovern(*groups)
            test_statistic, p_val = float(robust_result.statistic), float(robust_result.pvalue)
            test_name = f"Alexander-Govern test (Levene p={levene_p:.2e})"
            statistic_label = "AG"
        else:
            f_result = stats.f_oneway(*groups)
            test_statistic, p_val = float(f_result.statistic), float(f_result.pvalue)
            test_name = f"One-Way ANOVA (Levene p={levene_p:.2e})"
            statistic_label = "F"

        all_vals = df[sensor].dropna().to_numpy(dtype=float)
        ss_total = np.sum((all_vals - np.mean(all_vals)) ** 2)
        ss_between = sum(len(g) * (np.mean(g) - np.mean(all_vals)) ** 2 for g in groups)
        eta_squared = ss_between / (ss_total + 1e-12)

        res = HypothesisTestResult(
            hypothesis_id="H1_REGIME_VARIANCE",
            description="Dominância da Variação de Altitude/Mach sobre os Sensores Térmicos",
            h0="A altitude e velocidade de voo não afetam as leituras térmicas nominais.",
            h1="A atmosfera ISA e regimes de voo causam variação altamente significativa (p < 0.001).",
            test_name=test_name,
            test_statistic=test_statistic,
            p_value=p_val,
            effect_size=eta_squared,
            conclusion=(
                f"Efeito entre regimes: p={p_val:.2e} ({statistic_label}={test_statistic:.1f}; "
                f"Levene p={levene_p:.2e}; eta²={eta_squared:.4f}). "
                f"A estatística de efeito descreve {eta_squared*100:.1f}% da variância da amostra."
            ),
        )
        self.results.append(res)
        return res

    def test_physical_degradation_shift(
        self,
        residuals_df: pd.DataFrame,
        healthy_cutoff: int = 50,
        degraded_start: int | None = None,
        residual_col: str = "residual",
        cycle_col: str = "cycle",
        unit_col: str = "unit_id",
    ) -> HypothesisTestResult:
        """Hypothesis 2: Virtual sensor residuals shift significantly during degradation.
        
        Uses survivor-bias-free relative window (t > max(H, EOL_i - 50)) to preserve all engines.
        """
        if unit_col not in residuals_df:
            raise ValueError(f"{unit_col} is required for engine-level paired inference")

        ordered = residuals_df.copy()
        eol_map = ordered.groupby(unit_col)[cycle_col].transform("max")
        ordered["_eol"] = eol_map

        # Relative degradation window: t > max(healthy_cutoff, EOL_i - 50)
        if degraded_start is None:
            ordered["_is_degraded"] = ordered[cycle_col] > np.maximum(healthy_cutoff, ordered["_eol"] - 50)
        else:
            ordered["_is_degraded"] = ordered[cycle_col] >= degraded_start

        healthy = ordered.loc[ordered[cycle_col] <= healthy_cutoff].groupby(unit_col)[residual_col].mean()
        degraded = ordered.loc[ordered["_is_degraded"]].groupby(unit_col)[residual_col].mean()

        paired = pd.concat([healthy.rename("healthy"), degraded.rename("degraded")], axis=1).dropna()
        n_engines = len(paired)
        if n_engines < 2:
            raise ValueError("At least two engines need observations in both phases")

        deltas = (paired["degraded"] - paired["healthy"]).to_numpy(dtype=float)
        # Normality check on deltas
        shapiro_stat, shapiro_p = stats.shapiro(deltas)
        hl_delta = hodges_lehmann_estimator(deltas)
        cohens_d = float(np.mean(deltas) / (np.std(deltas, ddof=1) + 1e-12))

        if shapiro_p >= 0.05:
            t_stat, p_val = stats.ttest_1samp(deltas, popmean=0.0)
            test_name = f"Paired t-test on engine means (n={n_engines}, Shapiro p={shapiro_p:.3f})"
            stat_used = float(t_stat)
        else:
            w_stat, p_val = stats.wilcoxon(deltas, alternative="greater")
            test_name = f"Paired Wilcoxon signed-rank on engine means (n={n_engines}, Shapiro p={shapiro_p:.3f})"
            stat_used = float(w_stat)

        res = HypothesisTestResult(
            hypothesis_id="H2_PHYSICAL_DEGRADATION",
            description="Divergência Térmica Real vs Ruído de Medição no Fim de Vida",
            h0="Os resíduos do Sensor Virtual no final de vida possuem a mesma distribuição da fase saudável.",
            h1="O desgaste físico das pás provoca um deslocamento sistemático e significativo (p < 1e-15).",
            test_name=test_name,
            test_statistic=stat_used,
            p_value=float(p_val),
            effect_size=hl_delta,
            conclusion=(
                f"Confirmada com p={p_val:.2e} (n={n_engines} motores, d de Cohen={cohens_d:.2f}, "
                f"Hodges-Lehmann Δ={hl_delta:+.2f} unidades do sensor). "
                f"O desvio médio de {np.mean(deltas):+.2f} unidades do sensor indica deslocamento sistemático do resíduo."
            ),
        )
        self.results.append(res)
        return res

    def test_white_noise_residual_autocorrelation(
        self,
        residuals: np.ndarray,
        max_lags: int = 10,
        phase_label: str = "healthy",
    ) -> HypothesisTestResult:
        """Hypothesis 3: Ljung-Box test for residual serial correlation."""
        n = len(residuals)
        mean_r = np.mean(residuals)
        var_r = np.var(residuals)
        lags_autocorr = []
        for k in range(1, max_lags + 1):
            cov = np.sum((residuals[:-k] - mean_r) * (residuals[k:] - mean_r)) / n
            lags_autocorr.append(cov / (var_r + 1e-12))

        q_stat = n * (n + 2) * sum((rk ** 2) / (n - k) for k, rk in enumerate(lags_autocorr, start=1))
        p_val = stats.chi2.sf(q_stat, df=max_lags)

        hyp_id = f"H3_AUTOCORRELATION_{phase_label.upper()}"
        desc = f"Estrutura de Autocorrelação Residual (Fase {phase_label.capitalize()})"
        h0 = "Os resíduos são ruído branco puramente aleatório (sem autocorrelação serial)."
        h1 = "Os resíduos apresentam memória serial e correlação persistente."

        conclusion = (
            f"Fase {phase_label}: Q={q_stat:.2f}, p={p_val:.2e}. "
            + ("Não se rejeita H0; isso não demonstra ausência de autocorrelação." if p_val > 0.05 else "Evidência de autocorrelação serial.")
        )

        res = HypothesisTestResult(
            hypothesis_id=hyp_id,
            description=desc,
            h0=h0,
            h1=h1,
            test_name=f"Ljung-Box Portmanteau Test (lags={max_lags})",
            test_statistic=q_stat,
            p_value=p_val,
            effect_size=lags_autocorr[0] if lags_autocorr else None,
            conclusion=conclusion,
        )
        self.results.append(res)
        return res

    def test_fleet_residual_autocorrelation(
        self,
        residuals_df: pd.DataFrame,
        healthy_cutoff: int = 50,
        max_lags: int = 10,
        residual_col: str = "residual",
        cycle_col: str = "cycle",
        unit_col: str = "unit_id",
        phase_label: str = "frota_saudavel",
        min_obs: int = 20,
    ) -> HypothesisTestResult:
        """Hypothesis 3 (fleet): per-engine Ljung-Box on the healthy window, combined with Fisher's method.

        Each engine is tested separately (no concatenation across engines, which would create
        artificial lag-k pairs at engine boundaries). Under H0 each per-engine p-value is Uniform(0,1),
        so -2*sum(ln p_i) ~ chi2(2m).
        """
        q_stats, p_vals = [], []
        for _, eng in residuals_df.loc[residuals_df[cycle_col] <= healthy_cutoff].groupby(unit_col):
            r = eng.sort_values(cycle_col)[residual_col].dropna().to_numpy(dtype=float)
            n = len(r)
            if n < max(min_obs, max_lags + 2) or np.var(r) < 1e-12:
                continue
            centered = r - r.mean()
            denom = float(np.sum(centered ** 2))
            rho = [float(np.sum(centered[:-k] * centered[k:]) / denom) for k in range(1, max_lags + 1)]
            q = n * (n + 2) * sum((rk ** 2) / (n - k) for k, rk in enumerate(rho, start=1))
            q_stats.append(q)
            p_vals.append(float(stats.chi2.sf(q, df=max_lags)))
        if len(p_vals) < 2:
            raise ValueError("At least two engines with enough healthy observations are required")
        p_arr = np.clip(np.asarray(p_vals), 1e-300, 1.0)
        fisher = float(-2.0 * np.sum(np.log(p_arr)))
        p_comb = float(stats.chi2.sf(fisher, df=2 * len(p_arr)))
        frac_reject = float(np.mean(p_arr < 0.05))
        res = HypothesisTestResult(
            hypothesis_id=f"H3_AUTOCORRELATION_{phase_label.upper()}",
            description=f"Estrutura de Autocorrelação Residual por Motor (Fase {phase_label})",
            h0="Os resíduos de cada motor são ruído branco (sem autocorrelação serial).",
            h1="Ao menos parte da frota apresenta autocorrelação serial nos resíduos.",
            test_name=f"Per-engine Ljung-Box (lags={max_lags}) + Fisher combination (m={len(p_arr)} engines)",
            test_statistic=fisher,
            p_value=p_comb,
            effect_size=frac_reject,
            conclusion=(
                f"Fase {phase_label}: Q médio={np.mean(q_stats):.2f}, {frac_reject*100:.1f}% dos motores rejeitam H0 a 5% "
                f"(esperado ≈5% sob ruído branco); Fisher p={p_comb:.2e}."
            ),
        )
        self.results.append(res)
        return res

    def test_model_superiority_vs_baseline(
        self,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        baseline_pred: np.ndarray,
        model_name: str,
        unit_ids: np.ndarray | None = None,
    ) -> HypothesisTestResult:
        """Hypothesis 4: ML model provides statistically significantly lower error than naive baseline."""
        err_model = np.abs(y_true - y_pred)
        err_base = np.abs(y_true - baseline_pred)

        if unit_ids is None or len(unit_ids) != len(err_model):
            diff = err_base - err_model
        else:
            diff = pd.DataFrame({"unit_id": unit_ids, "diff": err_base - err_model}).groupby("unit_id")["diff"].mean().to_numpy()

        w_stat, p_val = stats.wilcoxon(diff, alternative="greater")
        mean_reduction = np.mean(err_base) - np.mean(err_model)
        hl_reduction = hodges_lehmann_estimator(diff)

        res = HypothesisTestResult(
            hypothesis_id=f"H4_SUPERIORITY_{model_name.upper().replace(' ', '_')}",
            description=f"Superioridade Estatística de {model_name} sobre a Linha de Base (Média)",
            h0=f"O modelo {model_name} possui a mesma distribuição de erro absoluto da média incondicional.",
            h1=f"O modelo {model_name} tem erro estatisticamente inferior (p < 0.001).",
            test_name="Wilcoxon Paired Signed-Rank Test",
            test_statistic=w_stat,
            p_value=p_val,
            effect_size=hl_reduction,
            conclusion=(
                f"{model_name} superou a linha de base com p={p_val:.2e} (W={w_stat:.1f}, "
                f"Hodges-Lehmann Δ={hl_reduction:.2f} ciclos). "
                f"Redução média no erro absoluto de {mean_reduction:.2f} ciclos."
            ),
        )
        self.results.append(res)
        return res

    def test_gbdt_vs_sequence_model_parity(
        self,
        y_true: np.ndarray,
        pred_gbdt: np.ndarray,
        pred_deep: np.ndarray,
        deep_model_name: str = "Transformer",
        unit_ids: np.ndarray | None = None,
    ) -> HypothesisTestResult:
        """Hypothesis 5: IEEE CIM 2026 RQ2 Dialogue: GBDT vs Deep Sequence Model Parity Test."""
        err_gbdt = np.abs(y_true - pred_gbdt)
        err_deep = np.abs(y_true - pred_deep)

        if unit_ids is None or len(unit_ids) != len(err_gbdt):
            sub_gbdt, sub_deep = err_gbdt, err_deep
        else:
            paired = pd.DataFrame({"unit_id": unit_ids, "gbdt": err_gbdt, "deep": err_deep}).groupby("unit_id")[["gbdt", "deep"]].mean()
            sub_gbdt, sub_deep = paired["gbdt"].to_numpy(), paired["deep"].to_numpy()

        w_stat, p_val = stats.wilcoxon(sub_gbdt, sub_deep)
        diff_mae = np.mean(err_deep) - np.mean(err_gbdt)
        hl_diff = hodges_lehmann_estimator(sub_deep - sub_gbdt)

        if p_val < 0.05:
            direction = "XGBoost superior" if diff_mae > 0 else f"{deep_model_name} superior"
            conclusion = (
                f"Comparação XGBoost vs {deep_model_name}: W={w_stat:.1f}, p={p_val:.2e}. "
                f"{direction} por Hodges-Lehmann Δ={hl_diff:.2f} ciclos (ΔMAE = {diff_mae:.2f})."
            )
        else:
            conclusion = (
                f"Comparação XGBoost vs {deep_model_name}: W={w_stat:.1f}, p={p_val:.2e}. "
                f"Paridade estatística: não se rejeita H0 a 5% (Hodges-Lehmann Δ={hl_diff:.2f} ciclos, ΔMAE = {diff_mae:.2f})."
            )

        res = HypothesisTestResult(
            hypothesis_id=f"H5_IEEE_RQ2_XGB_VS_{deep_model_name.upper()}",
            description=f"Diálogo IEEE CIM 2026: XGBoost vs {deep_model_name}",
            h0=f"XGBoost e {deep_model_name} apresentam acurácia preditiva idêntica.",
            h1="Existe divergência estatisticamente significativa entre as arquiteturas.",
            test_name="Two-Sided Paired Wilcoxon Test",
            test_statistic=w_stat,
            p_value=p_val,
            effect_size=hl_diff,
            conclusion=conclusion,
        )
        self.results.append(res)
        return res

    def compute_bootstrap_confidence_intervals(
        self,
        test_df: pd.DataFrame,
        model_cols: dict[str, str],
        target_col: str = "RUL_actual",
        n_bootstraps: int = 2000,
    ) -> pd.DataFrame:
        """Compute 95% Confidence Intervals via fleet engine cluster bootstrapping."""
        rng = np.random.default_rng(self.random_state)
        engines = test_df["unit_id"].unique()
        n_engines = len(engines)
        available_models = {name: col for name, col in model_cols.items() if col in test_df.columns}
        if not available_models:
            return pd.DataFrame()

        unit_ids = test_df["unit_id"].to_numpy()
        truth = test_df[target_col].to_numpy(dtype=float)
        predictions = {name: test_df[col].to_numpy(dtype=float) for name, col in available_models.items()}
        row_indices = [np.flatnonzero(unit_ids == engine) for engine in engines]

        def metrics(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, float, float]:
            errors = y_true - y_pred
            ss_total = np.sum((y_true - y_true.mean()) ** 2)
            return (
                float(1.0 - np.sum(errors ** 2) / (ss_total + 1e-12)),
                float(np.sqrt(np.mean(errors ** 2))),
                float(np.mean(np.abs(errors))),
            )

        score_store = {
            name: {metric: np.empty(n_bootstraps, dtype=float) for metric in ("r2", "rmse", "mae")}
            for name in available_models
        }
        for iteration in range(n_bootstraps):
            sampled = rng.integers(n_engines, size=n_engines)
            sample_index = np.concatenate([row_indices[index] for index in sampled])
            sample_truth = truth[sample_index]
            for name, prediction in predictions.items():
                r2, rmse, mae = metrics(sample_truth, prediction[sample_index])
                score_store[name]["r2"][iteration] = r2
                score_store[name]["rmse"][iteration] = rmse
                score_store[name]["mae"][iteration] = mae

        rows = []
        for name, prediction in predictions.items():
            observed = metrics(truth, prediction)
            jackknife = []
            for index in row_indices:
                keep = np.ones(len(truth), dtype=bool)
                keep[index] = False
                jackknife.append(metrics(truth[keep], prediction[keep]))
            r2_ci = _bca_interval(score_store[name]["r2"], [row[0] for row in jackknife], observed[0])
            rmse_ci = _bca_interval(score_store[name]["rmse"], [row[1] for row in jackknife], observed[1])
            mae_ci = _bca_interval(score_store[name]["mae"], [row[2] for row in jackknife], observed[2])
            rows.append({
                "Model": name,
                "R2_Observed": round(observed[0], 4),
                "R2_Mean": float(score_store[name]["r2"].mean()),
                "R2_CI_Lower": r2_ci[0],
                "R2_CI_Upper": r2_ci[1],
                "RMSE_Observed": round(observed[1], 2),
                "RMSE_Mean": float(score_store[name]["rmse"].mean()),
                "RMSE_CI_Lower": rmse_ci[0],
                "RMSE_CI_Upper": rmse_ci[1],
                "MAE_Observed": round(observed[2], 2),
                "MAE_Mean": float(score_store[name]["mae"].mean()),
                "MAE_CI_Lower": mae_ci[0],
                "MAE_CI_Upper": mae_ci[1],
                "CI_Method": "engine-cluster BCa",
            })
        return pd.DataFrame(rows)

    def summary_dataframe(self) -> pd.DataFrame:
        """Return all tested hypotheses as a structured DataFrame with FDR correction."""
        frame = pd.DataFrame([res.to_dict() for res in self.results])
        if not frame.empty:
            p = frame["p_value_raw"].to_numpy(dtype=float)
            order = np.argsort(p)
            adjusted = np.empty_like(p)
            running = 1.0
            m = len(p)
            for rank in range(m - 1, -1, -1):
                idx = order[rank]
                running = min(running, p[idx] * m / (rank + 1))
                adjusted[idx] = running
            frame["p_value_BH_FDR"] = adjusted
            frame["BH_FDR_Rejected_0.05"] = adjusted < 0.05
        return frame

    def export_results(self, output_dir: Path) -> tuple[pd.DataFrame, Path]:
        """Export hypothesis summary to CSV in output_dir."""
        output_dir.mkdir(parents=True, exist_ok=True)
        summary = self.summary_dataframe()
        out_path = output_dir / "thesis_hypothesis_tests.csv"
        summary.to_csv(out_path, index=False)
        return summary, out_path


# ============================================================================
# diagnostics
# ============================================================================

# Stateless diagnostics for the cross-subset RUL investigation.

SUBSYSTEM_GROUPS = {
    "Fan_Inlet": ["T2", "P2", "Nf", "NRf"],
    "Compressors_LPC_HPC": ["T24", "T30", "P15", "P30", "Nc", "NRc"],
    "Turbines_Combustor": ["T50", "epr", "Ps30", "BPR", "farB", "htBleed", "W31", "W32"],
    "Controls_Demands": ["Nf_dmd", "PCNfR_dmd", "phi"],
    "Flight_Settings": ["setting_1", "setting_2", "setting_3"],
}


def extra_trees_baseline(random_state: int = DEFAULT_RANDOM_STATE, **kwargs) -> ExtraTreesRegressor:
    """Return the common Extra Trees baseline used across FD001-FD004."""
    params = {"n_estimators": 60, "max_depth": 12, "min_samples_leaf": 2, "max_features": 1.0, "n_jobs": -1}
    params.update(kwargs)
    return ExtraTreesRegressor(random_state=random_state, **params)


def compare_rul_subsets(datasets: dict[str, pd.DataFrame], features: list[str], n_splits: int = 5, random_state: int = DEFAULT_RANDOM_STATE):
    """Compute comparable grouped-fold mean and pooled OOF RUL results."""
    fold_results, predictions = {}, {}
    rows = []
    for subset, frame in datasets.items():
        folds, oof = grouped_oof(
            frame, features,
            estimator=extra_trees_baseline(random_state=random_state),
            n_splits=n_splits, target_col="RUL",
        )
        fold_results[subset], predictions[subset] = folds, oof
        pooled = regression_metrics(frame["RUL"], oof)
        rows.append({
            "Subpart": subset,
            "R2_fold_mean": folds["R2"].mean(),
            "R2_fold_std": folds["R2"].std(ddof=1),
            "R2_pooled_OOF": pooled["R2"],
            "RMSE_fold_mean": folds["RMSE"].mean(),
            "MAE_fold_mean": folds["MAE"].mean(),
            "Rows": len(frame),
            "Engines": frame["unit_id"].nunique(),
        })
    summary = pd.DataFrame(rows).sort_values("R2_pooled_OOF", ascending=False).reset_index(drop=True)
    return summary, fold_results, predictions


def compare_split_strategies(frame: pd.DataFrame, features: list[str], n_repeats: int = 10, random_state: int = DEFAULT_RANDOM_STATE) -> dict:
    """Compare repeated random-row splits with grouped engine holdouts."""
    x, y, groups = frame[features].to_numpy(), frame["RUL"].to_numpy(), frame["unit_id"].to_numpy()
    row_scores, group_scores = [], []
    for rep in range(n_repeats):
        row_train, row_test = train_test_split(np.arange(len(frame)), test_size=0.25, random_state=random_state + rep)
        group_train, group_test = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=random_state + rep).split(x, y, groups))
        row_model = extra_trees_baseline(random_state + rep).fit(x[row_train], y[row_train])
        group_model = extra_trees_baseline(random_state + rep).fit(x[group_train], y[group_train])
        row_scores.append(float(r2_score(y[row_test], row_model.predict(x[row_test]))))
        group_scores.append(float(r2_score(y[group_test], group_model.predict(x[group_test]))))

    row_r2 = float(np.mean(row_scores))
    group_r2 = float(np.mean(group_scores))
    return {
        "Row_random_R2": row_r2,
        "Engine_holdout_R2": group_r2,
        "Row_split_optimism_delta": row_r2 - group_r2,
        "Row_random_R2_std": float(np.std(row_scores, ddof=1)) if n_repeats > 1 else 0.0,
        "Engine_holdout_R2_std": float(np.std(group_scores, ddof=1)) if n_repeats > 1 else 0.0,
        "Engines_shared_in_row_split": len(set(groups[row_train]).intersection(groups[row_test])),
        "Engines_shared_in_group_split": len(set(groups[group_train]).intersection(groups[group_test])),
    }


def future_cycle_score(frame: pd.DataFrame, features: list[str], random_state: int = DEFAULT_RANDOM_STATE) -> dict:
    """Train on each engine's first 75% of cycles and score its later cycles."""
    cutoff = frame.groupby("unit_id")["cycle"].transform(lambda cycles: np.floor(cycles.max() * 0.75))
    train_mask, future_mask = frame["cycle"] <= cutoff, frame["cycle"] > cutoff
    model = extra_trees_baseline(random_state).fit(frame.loc[train_mask, features], frame.loc[train_mask, "RUL"])
    prediction = model.predict(frame.loc[future_mask, features])
    return {
        "Temporal_future_R2": float(r2_score(frame.loc[future_mask, "RUL"], prediction)),
        "Train_rows": int(train_mask.sum()),
        "Future_rows": int(future_mask.sum()),
        "Engines_shared_by_design": int(frame.loc[train_mask, "unit_id"].nunique()),
    }


def capacity_sweep(datasets: dict[str, pd.DataFrame], features: list[str], random_state: int = DEFAULT_RANDOM_STATE) -> pd.DataFrame:
    """Compare Extra Trees depths and Ridge strengths on the same grouped folds."""
    from sklearn.linear_model import Ridge
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    rows = []
    for subset, frame in datasets.items():
        for depth in (4, 8, 16, None):
            estimator = extra_trees_baseline(random_state, n_estimators=40, max_depth=depth)
            folds, _ = grouped_oof(frame, features, estimator=estimator)
            rows.append({"Subpart": subset, "Family": "Extra Trees", "Configuration": f"max_depth={depth or 'unlimited'}", "Mean_fold_R2": folds["R2"].mean(), "Std_fold_R2": folds["R2"].std(ddof=1)})
        for alpha in (0.1, 1.0, 10.0, 100.0):
            estimator = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
            folds, _ = grouped_oof(frame, features, estimator=estimator)
            rows.append({"Subpart": subset, "Family": "Ridge", "Configuration": f"alpha={alpha:g}", "Mean_fold_R2": folds["R2"].mean(), "Std_fold_R2": folds["R2"].std(ddof=1)})
    return pd.DataFrame(rows)


def learning_curves(frame: pd.DataFrame, features: list[str], subset: str, repeats: int = 10, random_state: int = DEFAULT_RANDOM_STATE) -> pd.DataFrame:
    """Measure holdout R² as the number of training engines increases."""
    groups = frame["unit_id"].to_numpy()
    x, y = frame[features].to_numpy(), frame["RUL"].to_numpy()
    rows = []
    for repeat in range(repeats):
        train_idx, valid_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=random_state + repeat).split(x, y, groups))
        train_units = np.unique(groups[train_idx])
        shuffled = np.random.default_rng(random_state + repeat).permutation(train_units)
        for fraction in (0.25, 0.5, 0.75, 1.0):
            count = max(2, int(np.ceil(len(shuffled) * fraction)))
            selected = shuffled[:count]
            partial = train_idx[np.isin(groups[train_idx], selected)]
            model = extra_trees_baseline(random_state, n_estimators=40).fit(x[partial], y[partial])
            rows.append({"Subpart": subset, "Repeat": repeat, "Train_engine_fraction": fraction, "Train_engines": count, "Validation_R2": r2_score(y[valid_idx], model.predict(x[valid_idx]))})
    return pd.DataFrame(rows)


def summarize_learning_curves(repeats: pd.DataFrame) -> pd.DataFrame:
    """Aggregate repeated learning-curve scores into mean and standard deviation."""
    return repeats.groupby(["Subpart", "Train_engine_fraction"], as_index=False).agg(
        Mean_R2=("Validation_R2", "mean"), Std_R2=("Validation_R2", "std"),
    )


def residual_segments(frame: pd.DataFrame, truth, prediction, subset: str) -> pd.DataFrame:
    """Summarize OOF residual direction and RMSE over quantile-defined RUL bands."""
    residuals = pd.DataFrame({
        "unit_id": frame["unit_id"].to_numpy(), "cycle": frame["cycle"].to_numpy(),
        "RUL": frame["RUL"].to_numpy(), "prediction": np.asarray(prediction),
    })
    residuals["residual"] = np.asarray(truth) - np.asarray(prediction)
    residuals["RUL_band"] = pd.qcut(residuals["RUL"], q=5, duplicates="drop")
    result = residuals.groupby("RUL_band", observed=True).agg(
        rows=("residual", "size"), mean_error=("residual", "mean"),
        mae=("residual", lambda values: np.abs(values).mean()),
        rmse=("residual", lambda values: np.sqrt(np.mean(values ** 2))),
    ).reset_index()
    result.insert(0, "Subpart", subset)
    return result


def feature_signal(frame: pd.DataFrame, features: list[str], subset: str, sample_size: int = 5000, random_state: int = DEFAULT_RANDOM_STATE) -> pd.DataFrame:
    """Return sampled univariate absolute correlation and mutual information with RUL."""
    sample = frame.sample(n=min(sample_size, len(frame)), random_state=random_state)
    x = sample[features].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    mutual_info = mutual_info_regression(x, sample["RUL"], random_state=random_state)
    correlation = pd.Series(np.nan, index=features, dtype=float)
    varying_features = [feature for feature in features if x[feature].nunique() > 1]
    if varying_features and sample["RUL"].nunique() > 1:
        correlation.loc[varying_features] = x[varying_features].corrwith(sample["RUL"]).abs()
    return pd.DataFrame({
        "Subpart": subset,
        "Feature": features,
        "Abs_correlation": [float(correlation[feature]) for feature in features],
        "Mutual_information": mutual_info,
    })


def permutation_importances(frame: pd.DataFrame, features: list[str], subset: str, n_repeats: int = 10, random_state: int = DEFAULT_RANDOM_STATE) -> pd.DataFrame:
    """Compute held-out engine permutation importance with subsystem mapping."""
    x, y, groups = frame[features].to_numpy(), frame["RUL"].to_numpy(), frame["unit_id"].to_numpy()
    fit_idx, score_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=random_state).split(x, y, groups))
    model = extra_trees_baseline(random_state, n_estimators=40).fit(x[fit_idx], y[fit_idx])
    result = permutation_importance(model, x[score_idx], y[score_idx], scoring="r2", n_repeats=n_repeats, random_state=random_state, n_jobs=-1)

    # Map each feature to its physical subsystem
    feature_to_subsystem = {}
    for sub_name, sub_feats in SUBSYSTEM_GROUPS.items():
        for f in sub_feats:
            feature_to_subsystem[f] = sub_name

    subsystems = [feature_to_subsystem.get(f, "Other") for f in features]
    return pd.DataFrame({
        "Subpart": subset,
        "Feature": features,
        "Subsystem": subsystems,
        "R2_drop_mean": result.importances_mean,
        "R2_drop_std": result.importances_std,
    })


def distribution_shift(high_frame: pd.DataFrame, low_frame: pd.DataFrame, features: list[str], high_name: str, low_name: str, sample_size: int = 20000, random_state: int = DEFAULT_RANDOM_STATE) -> pd.DataFrame:
    """Compare feature distributions with KS effect sizes and Wasserstein-1 distances."""
    rows = []
    for feature in [*features, "RUL"]:
        high = high_frame[feature].dropna()
        low = low_frame[feature].dropna()
        high = high.sample(n=min(sample_size, len(high)), random_state=random_state)
        low = low.sample(n=min(sample_size, len(low)), random_state=random_state)
        result = ks_2samp(high, low)

        # Standardized Wasserstein-1 distance
        pooled_vals = np.concatenate([high.to_numpy(), low.to_numpy()])
        p_mean = np.mean(pooled_vals)
        p_std = np.std(pooled_vals) if np.std(pooled_vals) > 1e-8 else 1.0
        w_dist = float(wasserstein_distance((high.to_numpy() - p_mean) / p_std, (low.to_numpy() - p_mean) / p_std))

        rows.append({
            "Feature": feature,
            "KS_statistic": float(result.statistic),
            "Wasserstein_distance": w_dist,
            "p_value": float(result.pvalue),
            "High_subset_median": float(high.median()),
            "Low_subset_median": float(low.median()),
            "High_subset": high_name,
            "Low_subset": low_name,
        })
    return pd.DataFrame(rows).sort_values("KS_statistic", ascending=False).reset_index(drop=True)


def repeated_grouped_cv(frame: pd.DataFrame, features: list[str], subset: str, repeats: int = 5, n_splits: int = 5, random_state: int = DEFAULT_RANDOM_STATE) -> pd.DataFrame:
    """Repeat pooled grouped OOF scoring with shuffled engine-to-fold assignment."""
    groups = frame["unit_id"].to_numpy()
    x, y = frame[features].to_numpy(dtype=float), frame["RUL"].to_numpy(dtype=float)
    rows = []
    for repeat in range(repeats):
        units = np.random.default_rng(random_state + repeat).permutation(np.unique(groups))
        fold_for = {unit: i % n_splits for i, unit in enumerate(units)}
        fold_ids = np.array([fold_for[group] for group in groups])
        predictions = np.full(len(frame), np.nan)
        for fold in range(n_splits):
            valid = np.flatnonzero(fold_ids == fold)
            train = np.flatnonzero(fold_ids != fold)
            model = extra_trees_baseline(random_state + repeat * 10 + fold).fit(x[train], y[train])
            predictions[valid] = model.predict(x[valid])
        rows.append({"Subpart": subset, "Repeat": repeat, "R2_pooled_OOF": float(r2_score(y, predictions))})
    return pd.DataFrame(rows)

### 2.2 Pacote `pdm_cmaps`: pipeline

orquestração do pipeline por subconjunto

4 arquivos.

In [ ]:
%%writefile projeto/src/pdm_cmaps/pipeline/__init__.py
"""Per-subset orchestration: PipelineConfig describes a run, PdMPipeline executes it end to end."""

In [ ]:
%%writefile projeto/src/pdm_cmaps/pipeline/config.py
"""Strongly-typed pipeline configuration."""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

from pdm_cmaps.config import DEFAULT_PATHS
from pdm_cmaps.domain.schema import DEFAULT_HEALTHY_CYCLE_LIMIT, DEFAULT_RANDOM_STATE


@dataclass
class PipelineConfig:
    """Strongly-typed pipeline configuration parameters."""

    subset: str = "FD004"
    seed: int = DEFAULT_RANDOM_STATE
    healthy_cycle_limit: int = DEFAULT_HEALTHY_CYCLE_LIMIT
    unit_id: int = 179
    include_minirocket: bool = False
    include_lstm: bool = False
    clean_output: bool = True
    run_all: bool = False
    output_dir: Path | None = None

    @property
    def resolved_output_dir(self) -> Path:
        """Return the effective output directory path."""
        if self.output_dir is not None:
            return Path(self.output_dir)
        return DEFAULT_PATHS.guide_dir(self.subset)

In [ ]:
%%writefile projeto/src/pdm_cmaps/pipeline/extra_models.py
"""Supplementary models (SVM + PyTorch) and a uniform fit/predict helper."""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVR

from pdm_cmaps.modeling.torch_models import PyTorchCNNModel, PyTorchRNNModel, PyTorchTransformerModel


def build_extra_models(seed: int) -> dict[str, tuple[str, object]]:
    """Supplementary models appended to the benchmark (SVM + PyTorch sequence models), fresh and unfitted."""
    return {
        "Support Vector Machine (SVM)": ("sklearn", make_pipeline(StandardScaler(), LinearSVR(C=1.0, random_state=seed, max_iter=5000))),
        "PyTorch CNN": ("deep", PyTorchCNNModel(seq_length=15, random_state=seed)),
        "PyTorch RNN (LSTM)": ("deep", PyTorchRNNModel(seq_length=15, random_state=seed)),
        "PyTorch Transformer": ("deep", PyTorchTransformerModel(seq_length=15, random_state=seed)),
    }

def fit_predict(m_type: str, model, train_frame: pd.DataFrame, features: list[str], eval_frame: pd.DataFrame) -> np.ndarray:
    """Fit ``model`` on ``train_frame`` and return one prediction per row of ``eval_frame`` (row order preserved)."""
    if m_type == "sklearn":
        model.fit(train_frame[features].to_numpy(dtype=float), train_frame["RUL"].to_numpy(dtype=float))
        return np.asarray(model.predict(eval_frame[features].to_numpy(dtype=float)), dtype=float)
    model.fit(train_frame, features, "RUL")
    return np.asarray(model.predict_values(eval_frame), dtype=float)

In [ ]:
%%writefile projeto/src/pdm_cmaps/pipeline/pipeline.py
"""Pipeline stages: virtual sensor / SPC / RUL benchmark, additional models and statistical inference.

This file is the RECIPE that calls everything else.

The 5 phases of PdMPipeline.execute() (stages hand data to each other through tables/*.csv):
  1. run()                 load -> split by engine -> regimes -> virtual sensor + SPC alarms ->
                           sensor sweep -> health index -> RUL benchmark + grouped CV
  2. run_auxiliary_spc_horizon   same as 1 with a longer healthy window (H=100) as a
                           sensitivity check; isolated in a scratch folder.
  3. train_additional_models_and_stats   SVM/CNN/LSTM/Transformer, critical-zone table,
                           OFFICIAL test benchmark, hypothesis tests, bootstrap CIs.
  4. monitoring layer      conformal RUL intervals, criticality levels, alert events, grounded
                           reports + faithfulness audit (monitoring/layer.py, alerts.py, reports.py).
  5. figure generation     thesis figures (FD004 only).

Two evaluation protocols exist on purpose:
  [HOLDOUT]  75/25 engine split of the training file, every cycle scored. Good for comparing
             models with each other; optimistic in absolute terms.
  [OFFICIAL] fit on all training engines, score ONLY the last cycle of each truncated test
             engine vs RUL_FDxxx.txt. Use this one when comparing with the literature."""

from __future__ import annotations

from pdm_cmaps.pipeline.extra_models import build_extra_models, fit_predict

from pdm_cmaps.analysis.stats import ThesisHypothesisAnalyzer
from pdm_cmaps.reporting.plots import figure_1_noise_vs_damage
from pdm_cmaps.reporting.plots import figure_2_physical_cascade
from pdm_cmaps.reporting.plots import figure_3_autocorrelation_evidence
from pdm_cmaps.reporting.plots import figure_4_fleet_persistence_tradeoff
from pdm_cmaps.reporting.plots import plot_actual_vs_predicted
from pdm_cmaps.reporting.plots import plot_geometric_envelope_detection

import argparse
from pathlib import Path
import time
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import shutil

from pdm_cmaps.config import DEFAULT_RESULTS_DIR
from pdm_cmaps.data.features import CausalFeatureEngineer
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.data.splits import partition_healthy_and_lifecycle, split_engine_units
from pdm_cmaps.detection.regime import RegimeDecoupler
from pdm_cmaps.detection.spc import EnvelopeAnomalyDetector
from pdm_cmaps.detection.virtual_sensor import VirtualSensor
from pdm_cmaps.domain.rul import compute_piecewise_rul, evaluate_rul_predictions
from pdm_cmaps.domain.schema import DEFAULT_HEALTHY_CYCLE_LIMIT, DEFAULT_RANDOM_STATE, DEFAULT_RUL_CAP
from pdm_cmaps.modeling.benchmark import RULBenchmark
from pdm_cmaps.modeling.minirocket import MiniRocketRULModel
from pdm_cmaps.modeling.registry import get_model_pool
from pdm_cmaps.modeling.torch_models import PyTorchRNNModel
from pdm_cmaps.pipeline.config import PipelineConfig
from pdm_cmaps.progress import ProgressTracker




PRIMARY_TARGET_SENSORS = ("T50", "P30", "BPR", "Nc")

def run(
    subset: str,
    output_dir: Path,
    seed: int,
    include_lstm: bool = False,
    include_minirocket: bool = False,
    healthy_cycle_limit: int = DEFAULT_HEALTHY_CYCLE_LIMIT,
    target_sensors: tuple[str, ...] = PRIMARY_TARGET_SENSORS,
    sample_engine_id: int | None = 179,
    min_consecutive_breaches: int = 2,
    loader: CMapssLoader | None = None,
    decoupler: RegimeDecoupler | None = None,
    benchmark: RULBenchmark | None = None,
) -> dict[str, pd.DataFrame]:
    """Execute the guide's core analyses and write their tables/figures.

    ``loader``, ``decoupler`` and ``benchmark`` are optional injected components (defaults are built from ``subset``/``seed``).
    """
    table_dir = output_dir / "tables"
    figure_dir = output_dir / "figures"
    table_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)
    print("\n=======================================================================", flush=True)
    print(f"[maintenance] STARTING PdM PIPELINE: Subset={subset} | Seed={seed} | Healthy limit={healthy_cycle_limit} cycles", flush=True)
    print("=======================================================================\n", flush=True)

    # STEP 1 -- [LEAKAGE] split by engine first; everything learned later (regimes, scalers,
    # sensor filter, thresholds, models) is fit on the training engines only.
    print(f"[maintenance] [STEP 1/7] Loading {subset} data and partitioning disjoint engine holdout...", flush=True)
    loader = loader or CMapssLoader()
    frame = loader.load(subset)
    frame["RUL"] = compute_piecewise_rul(frame)
    train_units, test_units = split_engine_units(frame, test_ratio=0.25, random_state=seed)
    # Informative-sensor filter is learned on TRAINING engines only (no test-fleet information).
    sensors = loader.informative_sensors(frame.loc[frame["unit_id"].isin(train_units)])
    parts = partition_healthy_and_lifecycle(frame, train_units, test_units, healthy_cycle_limit)
    print(f"  -> Total rows: {len(frame)} | Engines: {frame['unit_id'].nunique()} (Train: {len(train_units)}, Test: {len(test_units)})", flush=True)

    # STEP 2 -- unsupervised regime discovery. FD002/FD004 were simulated with 6 operating
    # conditions, FD001/FD003 with 1 (so decoupling is a no-op there and H1 is skipped).
    n_regimes = 6 if subset in {"FD002", "FD004"} else 1
    print(f"[maintenance] [STEP 2/7] Decoupling operating flight regimes ({n_regimes} clusters)...", flush=True)
    regimes = (decoupler or RegimeDecoupler(n_clusters=n_regimes, random_state=seed)).fit(parts["full_lifecycle_train"])
    train_full = regimes.transform(parts["full_lifecycle_train"])
    test_full = regimes.transform(parts["full_lifecycle_test"])
    healthy_train = regimes.transform(parts["healthy_train"])
    healthy_test = regimes.transform(parts["healthy_test"])

    valid_targets = [s for s in target_sensors if s in sensors]
    if not valid_targets:
        valid_targets = ["T50"] if "T50" in sensors else [sensors[0]]

    target_unit_id = sample_engine_id if (sample_engine_id is not None and sample_engine_id in test_units) else int(test_units[0])
    print(f"  -> Primary sample engine for trace visualization: Engine #{target_unit_id}", flush=True)


    # STEPS 3-5 -- for each target sensor: (a) how much of its variance is regime? (b) fit the
    # healthy-state virtual sensor and turn residuals into SPC alarms (c) draw envelopes.
    print(f"[maintenance] [STEPS 3-5/7] Profiling Virtual Sensors & Envelopes across targets: {', '.join(valid_targets)}...", flush=True)
    for target_sensor in valid_targets:
        s_lower = target_sensor.lower()
        print(f"\n  [sensor: {target_sensor}] Decoupling variance and fitting Virtual Sensor...", flush=True)

        # 1. Variance Decomposition
        variance = RegimeDecoupler.variance_decomposition(test_full, target_sensor, reference_frame=train_full)
        pd.DataFrame([variance]).drop(columns=["clusters"]).to_csv(table_dir / f"{s_lower}_variance_decomposition.csv", index=False)
        if target_sensor == "T50":
            pd.DataFrame([variance]).drop(columns=["clusters"]).to_csv(table_dir / "t50_variance_decomposition.csv", index=False)

        between_pct = variance.get("Between_regime_pct", variance.get("pct_between_regime", 0.0))
        within_pct = variance.get("Within_regime_pct", variance.get("pct_within_regime", 100.0))
        base_r2 = variance.get("regime_baseline_r2", between_pct / 100.0 if between_pct else 0.0)
        print(f"    -> {target_sensor} variance: Between regimes = {between_pct:.1f}%, Within regime = {within_pct:.1f}% (Baseline R² = {base_r2:.4f})", flush=True)

        # 2. Virtual Sensor fitting & Fleet SPC (Strictly Exogenous features, ewma_alpha=0.10)
        virtual_model = VirtualSensor(
            target_sensor=target_sensor,
            mode="exogenous",
            ewma_alpha=0.10,
            min_consecutive_breaches=min_consecutive_breaches,
            healthy_cycle_limit=healthy_cycle_limit,
            random_state=seed,
        )
        residual_trace, anomalies, anomaly_summary = virtual_model.evaluate(healthy_train, test_full)
        h_suffix = f"_h{healthy_cycle_limit}"
        residual_trace.to_csv(table_dir / f"virtual_sensor_residuals_{s_lower}.csv", index=False)
        residual_trace.to_csv(table_dir / f"virtual_sensor_residuals_{s_lower}{h_suffix}.csv", index=False)
        anomalies.to_csv(table_dir / f"fleet_anomalies_{s_lower}.csv", index=False)
        anomalies.to_csv(table_dir / f"fleet_anomalies_{s_lower}{h_suffix}.csv", index=False)
        pd.DataFrame([anomaly_summary]).to_csv(table_dir / f"fleet_anomaly_summary_{s_lower}.csv", index=False)
        pd.DataFrame([anomaly_summary]).to_csv(table_dir / f"fleet_anomaly_summary_{s_lower}{h_suffix}.csv", index=False)
        if target_sensor == "T50":
            residual_trace.to_csv(table_dir / "virtual_sensor_residuals.csv", index=False)
            anomalies.to_csv(table_dir / "fleet_anomalies.csv", index=False)
            pd.DataFrame([anomaly_summary]).to_csv(table_dir / "fleet_anomaly_summary.csv", index=False)

        print(f"    -> Virtual Sensor {target_sensor}: Fleet detection = {anomaly_summary['detection_rate_pct']:.1f}% | Mean lead time = {anomaly_summary['mean_lead_time_cycles']:.1f} cycles", flush=True)

        # 4. Envelope Anomaly Detection for target_unit_id
        envelope_detector = EnvelopeAnomalyDetector(
            k_sigma=3.0,
            healthy_cycle_limit=healthy_cycle_limit,
            window_size=5,
            min_consecutive_breaches=min_consecutive_breaches,
        )
        healthy_res_rows = virtual_model.calibration_predictions_
        if healthy_res_rows is None:
            raise RuntimeError("Virtual sensor did not produce OOF healthy calibration predictions")
        envelope_detector.calibrate(healthy_res_rows[target_sensor], healthy_res_rows["prediction"])

        sample_trace = residual_trace.loc[residual_trace["unit_id"] == target_unit_id].copy()
        scored_env, env_summary = envelope_detector.score_time_series_envelope(
            sample_trace, actual_col=target_sensor, pred_col="prediction",
        )
        scored_geom, geom_summary = envelope_detector.score_geometric_envelope(
            sample_trace, signal_col=target_sensor, use_decoupled_signal=True,
        )

        scored_env.to_csv(table_dir / f"envelope_scored_trace_{s_lower}.csv", index=False)
        scored_env.to_csv(table_dir / f"envelope_scored_trace_{s_lower}{h_suffix}.csv", index=False)
        pd.DataFrame([env_summary]).to_csv(table_dir / f"envelope_summary_{s_lower}.csv", index=False)
        pd.DataFrame([env_summary]).to_csv(table_dir / f"envelope_summary_{s_lower}{h_suffix}.csv", index=False)
        pd.DataFrame([geom_summary]).to_csv(table_dir / f"geometric_envelope_summary_{s_lower}.csv", index=False)
        pd.DataFrame([geom_summary]).to_csv(table_dir / f"geometric_envelope_summary_{s_lower}{h_suffix}.csv", index=False)
        if target_sensor == "T50":
            scored_env.to_csv(table_dir / "envelope_scored_trace.csv", index=False)
            scored_env.to_csv(table_dir / f"envelope_scored_trace{h_suffix}.csv", index=False)
            pd.DataFrame([env_summary]).to_csv(table_dir / "envelope_summary.csv", index=False)
            pd.DataFrame([env_summary]).to_csv(table_dir / f"envelope_summary{h_suffix}.csv", index=False)
            pd.DataFrame([geom_summary]).to_csv(table_dir / "geometric_envelope_summary.csv", index=False)
            pd.DataFrame([geom_summary]).to_csv(table_dir / f"geometric_envelope_summary{h_suffix}.csv", index=False)

        plot_geometric_envelope_detection(
            scored_geom,
            geom_summary,
            figure_dir / f"{s_lower}_geometric_envelope_detection.png",
            title=f"Envelope Geométrico Digital (Hull): {target_sensor}",
            sensor_name=target_sensor,
            healthy_cycle_limit=healthy_cycle_limit,
        )
        if target_sensor == "T50":
            plot_geometric_envelope_detection(
                scored_geom,
                geom_summary,
                figure_dir / "t50_geometric_envelope_detection.png",
                title="Envelope Geométrico de Sinal Digital (Rolling Peak/Valley Hull)",
                sensor_name="T50",
                healthy_cycle_limit=healthy_cycle_limit,
            )
        print(f"    -> Time-series envelope ({target_sensor}): Alarm at cycle {env_summary['first_alarm_cycle']} | EOL: {env_summary['eol_cycle']} | Lead time: {env_summary['lead_time_cycles']} cycles", flush=True)

        # 5. Held-out healthy regime validation
        healthy_prediction = virtual_model.predict(healthy_test)
        regime_rows = []
        for regime_id, rows in healthy_prediction.groupby("regime_id"):
            truth, prediction = rows[target_sensor].to_numpy(), rows["prediction"].to_numpy()
            regime_rows.append({
                "Regime_ID": int(regime_id), "Test_rows": len(rows),
                "R2": float(r2_score(truth, prediction)),
                "RMSE": float(np.sqrt(mean_squared_error(truth, prediction))),
                "MAE": float(mean_absolute_error(truth, prediction)),
            })
        regime_scores = pd.DataFrame(regime_rows)
        regime_scores.to_csv(table_dir / f"healthy_{s_lower}_by_regime.csv", index=False)
        regime_scores.to_csv(table_dir / f"healthy_{s_lower}_by_regime{h_suffix}.csv", index=False)
        if target_sensor == "T50":
            regime_scores.to_csv(table_dir / "healthy_t50_by_regime.csv", index=False)
            regime_scores.to_csv(table_dir / f"healthy_t50_by_regime{h_suffix}.csv", index=False)

    # STEP 7 -- supervised RUL benchmark. [BASELINE] two naive baselines are included so every
    # model's gain is measured against "no information". Grouped CV (below) gives a spread
    # across engine folds; the holdout gives the per-engine predictions used by stats.py.
    print("[maintenance] [STEP 7/7] Benchmarking RUL models on held-out engines (CatBoost, LightGBM, XGBoost, etc.)...", flush=True)

    rul_features = [*sensors, "setting_1", "setting_2", "setting_3", "cycle"]
    benchmark = benchmark or RULBenchmark(random_state=seed)
    rul_train = parts["full_lifecycle_train"].assign(RUL=compute_piecewise_rul(parts["full_lifecycle_train"]))
    rul_test = parts["full_lifecycle_test"].assign(RUL=compute_piecewise_rul(parts["full_lifecycle_test"]))
    holdout, predictions = benchmark.benchmark_train_test(rul_train, rul_test, rul_features)

    # Representation-matched GBDT/tree baselines: same causal temporal context that sequence models receive.
    # Removes the "single row vs 15-cycle window" confound from the GBDT-vs-deep comparison (RQ2).
    engineer = CausalFeatureEngineer(sensors, windows=(5, 10)).fit(rul_train, healthy_cycle_limit)
    train_fe = engineer.transform(rul_train)
    test_fe = engineer.transform(rul_test)
    if not np.allclose(test_fe["RUL"].to_numpy(), predictions["y_true"]):
        raise AssertionError("Temporal-feature frame is not row-aligned with the shared holdout")
    fe_features = [*rul_features, *[c for c in train_fe.columns if c not in rul_train.columns]]
    temporal_pool = {
        f"{name} (+temporal)": estimator
        for name, estimator in get_model_pool(seed).items()
        if name in {"XGBoost", "LightGBM", "CatBoost", "Extra Trees"}
    }
    fe_holdout, fe_predictions = RULBenchmark(random_state=seed, model_pool=temporal_pool, verbose=False).benchmark_train_test(
        train_fe, test_fe, fe_features,
    )
    fe_holdout = fe_holdout.loc[~fe_holdout["Model"].str.startswith("Baseline")]
    holdout = pd.concat([holdout, fe_holdout], ignore_index=True)
    for name in temporal_pool:
        predictions[name] = fe_predictions[name]
    holdout.to_csv(table_dir / "rul_holdout_benchmark.csv", index=False)
    best_model = str(holdout.loc[holdout["R2"].idxmax(), "Model"])
    plot_actual_vs_predicted(
        predictions["y_true"], predictions[best_model],
        figure_dir / "rul_best_model_actual_vs_predicted.png",
        f"RUL holdout: {best_model}", "RUL (cycles)",
    )
    test_prediction_frame = parts["full_lifecycle_test"][["unit_id", "cycle"]].reset_index(drop=True)
    test_prediction_frame["RUL_actual"] = predictions["y_true"]
    for model_name, values in predictions.items():
        if model_name != "y_true":
            test_prediction_frame[f"RUL_predicted_{model_name}"] = values

    cv_frame = frame.loc[frame["unit_id"].isin(train_units)].copy()
    cv = benchmark.cross_validate(cv_frame, rul_features, n_splits=5)
    table_dir.mkdir(parents=True, exist_ok=True)
    cv.to_csv(table_dir / "rul_grouped_cross_validation.csv", index=False)

    tables = {
        "regime_variance": pd.DataFrame([variance]),
        "virtual_sensor_regimes": regime_scores,
        "anomaly_summary": pd.DataFrame([anomaly_summary]),
        "rul_holdout": holdout,
        "rul_grouped_cv": cv,
    }
    if include_lstm:
        lstm_frame = frame.copy()
        lstm_train = lstm_frame.loc[lstm_frame["unit_id"].isin(train_units)]
        lstm_test = lstm_frame.loc[lstm_frame["unit_id"].isin(test_units)]
        lstm_features = [*sensors, "setting_1", "setting_2", "setting_3", "cycle"]
        # The legacy PyTorchLSTMModel stopped on TRAINING loss and trained only 4 epochs; the shared
        # sequence base class early-stops on a held-out set of engines, so use it instead.
        model = PyTorchRNNModel(seq_length=15, cell_type="lstm", random_state=seed)
        model.fit(lstm_train, lstm_features, "RUL")
        truth = lstm_test["RUL"].to_numpy(dtype=float)
        estimate = model.predict(lstm_test)
        lstm_results = pd.DataFrame([{
            "Model": "PyTorch LSTM",
            **evaluate_rul_predictions(truth, estimate),
            "Fit_Time_s": round(model.fit_time, 3),
        }])
        lstm_results.to_csv(table_dir / "rul_lstm_holdout.csv", index=False)
        tables["rul_lstm"] = lstm_results

    if include_minirocket:
        print("[maintenance] Fitting MiniRocket + RidgeCV and benchmarking RUL on held-out engines", flush=True)

        mr_train = parts["full_lifecycle_train"].assign(RUL=compute_piecewise_rul(parts["full_lifecycle_train"]))
        mr_test = parts["full_lifecycle_test"].assign(RUL=compute_piecewise_rul(parts["full_lifecycle_test"]))
        mr_features = [*sensors, "setting_1", "setting_2", "setting_3", "cycle"]
        mr_model = MiniRocketRULModel(seq_length=15, num_kernels=1000, random_state=seed)
        mr_model.fit(mr_train, mr_features, "RUL")
        mr_truth, mr_estimate = mr_model.predict(mr_test)
        if not np.allclose(mr_truth, predictions["y_true"]):
            raise AssertionError("MiniRocket predictions are not aligned with the shared holdout rows")
        predictions["MiniRocket + RidgeCV"] = mr_estimate
        test_prediction_frame["RUL_predicted_MiniRocket + RidgeCV"] = mr_estimate
        mr_results = pd.DataFrame([{
            "Model": "MiniRocket + RidgeCV",
            **evaluate_rul_predictions(mr_truth, mr_estimate),
            "Fit_Time_s": round(mr_model.fit_time, 3),
        }])
        combined_holdout = pd.concat([holdout, mr_results], ignore_index=True)
        combined_holdout.to_csv(table_dir / "rul_holdout_benchmark.csv", index=False)
        tables["rul_minirocket"] = mr_results
    test_prediction_frame.to_csv(table_dir / "rul_holdout_predictions.csv", index=False)
    return tables

def run_guide_cli() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--subset", choices=("FD001", "FD002", "FD003", "FD004"), default="FD004")
    parser.add_argument("--output-dir", type=Path, default=DEFAULT_RESULTS_DIR / "maintenance_guide")
    parser.add_argument("--seed", type=int, default=DEFAULT_RANDOM_STATE)
    parser.add_argument("--include-lstm", action="store_true", help="Run an additional sequence-model holdout comparison")
    parser.add_argument("--include-minirocket", action="store_true", help="Run an additional MiniRocket sequence-model holdout comparison")
    parser.add_argument("--healthy-cycle-limit", type=int, default=DEFAULT_HEALTHY_CYCLE_LIMIT, help="Healthy cycle limit used for virtual sensor calibration (default: 50)")
    parser.add_argument("--target-sensors", nargs="+", default=list(PRIMARY_TARGET_SENSORS), help="Target sensors to evaluate for virtual sensor and envelope detection (default: T50 P30 BPR Nc)")
    parser.add_argument("--min-consecutive-breaches", type=int, default=2, help="Minimum consecutive breaches required to trigger an envelope alarm (default: 2, Western Electric rule)")
    parser.add_argument("--unit-id", type=int, default=179, help="Engine unit ID to use for sample traces (default: 179)")
    args = parser.parse_args()
    tables = run(
        args.subset,
        args.output_dir,
        args.seed,
        args.include_lstm,
        args.include_minirocket,
        args.healthy_cycle_limit,
        target_sensors=tuple(args.target_sensors),
        sample_engine_id=args.unit_id,
        min_consecutive_breaches=args.min_consecutive_breaches,
    )
    print("\n=======================================================================", flush=True)
    print("[maintenance] PIPELINE EXECUTION COMPLETE! Summary of generated tables:", flush=True)
    print("=======================================================================", flush=True)
    for name, table in tables.items():
        print(f"  - Table: {name:<28} ({len(table)} rows)", flush=True)
    print(f"\n[maintenance] Tables written to: {(args.output_dir / 'tables').resolve()}", flush=True)
    print(f"[maintenance] Figures written to: {(args.output_dir / 'figures').resolve()}\n", flush=True)

# Benchmark names that already denote one of the extra models (legacy aliases included).
_EXTRA_ALIASES: dict[str, tuple[str, ...]] = {
    "Support Vector Machine (SVM)": ("Support Vector Machine (SVM)", "SVM"),
    "PyTorch CNN": ("PyTorch CNN", "1D CNN"),
    "PyTorch RNN (LSTM)": ("PyTorch RNN (LSTM)", "RNN (LSTM)", "PyTorch LSTM"),
    "PyTorch Transformer": ("PyTorch Transformer", "Transformer"),
}

def _official_test_benchmark(
    subset: str,
    full_train: pd.DataFrame,
    sensors: list[str],
    features: list[str],
    model_names: list[str],
    seed: int,
    healthy_cycle_limit: int,
) -> pd.DataFrame | None:
    """Score every named model on the official C-MAPSS test protocol (last cycle per engine vs RUL_FDxxx.txt)."""
    loader = CMapssLoader()
    if not ((loader.data_dir / f"test_{subset}.txt").is_file() and (loader.data_dir / f"RUL_{subset}.txt").is_file()):
        return None
    off_test, true_rul = loader.load_test(subset)
    y_true = np.clip(true_rul.to_numpy(dtype=float), 0.0, float(DEFAULT_RUL_CAP))
    n_units = off_test["unit_id"].nunique()
    unit_keys = off_test["unit_id"].to_numpy()

    def last(values) -> np.ndarray:
        return pd.Series(np.asarray(values, dtype=float)).groupby(unit_keys).last().sort_index().to_numpy()

    pool = get_model_pool(seed)
    extras = build_extra_models(seed)
    y_mean = float(full_train["RUL"].mean())
    cycle_map = full_train.groupby("cycle")["RUL"].mean()
    temporal_suffix = " (+temporal)"
    fe_cache: dict[str, object] = {}
    rows = []
    for name in model_names:
        if name == "Baseline (Mean RUL)":
            pred = np.full(n_units, y_mean)
        elif name == "Baseline (Mean RUL by cycle)":
            last_cycle = off_test.groupby("unit_id")["cycle"].max().sort_index()
            pred = last_cycle.map(cycle_map).fillna(y_mean).to_numpy(dtype=float)
        elif name in extras:
            m_type, model = extras[name]
            pred = last(fit_predict(m_type, model, full_train, features, off_test))
        elif name in pool:
            pred = last(fit_predict("sklearn", pool[name], full_train, features, off_test))
        elif name.endswith(temporal_suffix) and name[: -len(temporal_suffix)] in pool:
            if not fe_cache:
                engineer = CausalFeatureEngineer(sensors, windows=(5, 10)).fit(full_train, healthy_cycle_limit)
                fe_train = engineer.transform(full_train)
                fe_cache.update(
                    train=fe_train,
                    test=engineer.transform(off_test),
                    features=[*features, *[c for c in fe_train.columns if c not in full_train.columns]],
                )
            pred = last(fit_predict(
                "sklearn", pool[name[: -len(temporal_suffix)]], fe_cache["train"], fe_cache["features"], fe_cache["test"],
            ))
        elif name == "MiniRocket + RidgeCV":
            model = MiniRocketRULModel(seq_length=15, num_kernels=1000, random_state=seed)
            pred = last(fit_predict("deep", model, full_train, features, off_test))
        else:
            print(f"[WARN] No official-test recipe for model '{name}'; omitted (no placeholder predictions are fabricated).", flush=True)
            continue
        if len(pred) != len(y_true):
            raise AssertionError(f"{name}: {len(pred)} predictions for {len(y_true)} official test engines")
        rows.append({"Model": name, "N_engines": len(y_true), **evaluate_rul_predictions(y_true, pred)})
    if not rows:
        return None
    return pd.DataFrame(rows).sort_values("R2", ascending=False).reset_index(drop=True)

def train_additional_models_and_stats(
    output_dir: Path | None = None,
    subset: str = "FD004",
    seed: int = DEFAULT_RANDOM_STATE,
    healthy_cycle_limit: int = DEFAULT_HEALTHY_CYCLE_LIMIT,
) -> int:
    results_dir = output_dir if output_dir is not None else DEFAULT_RESULTS_DIR / f"maintenance_guide_{subset.lower()}"
    table_dir = results_dir / "tables"
    figure_dir = results_dir / "figures"
    table_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)

    benchmark_path = table_dir / "rul_holdout_benchmark.csv"
    predictions_path = table_dir / "rul_holdout_predictions.csv"

    if not benchmark_path.exists() or not predictions_path.exists():
        print(f"[ERROR] Baseline tables not found at {table_dir}. Please run pipeline first.")
        return 1

    benchmark_df = pd.read_csv(benchmark_path)
    predictions_df = pd.read_csv(predictions_path)

    existing_models = set(benchmark_df["Model"].unique())
    print(f"[INFO] Existing models in benchmark ({len(existing_models)}):")
    for m in existing_models:
        print(f"  - {m}")

    models_to_train = {}
    
    for extra_name, spec in build_extra_models(seed).items():
        if not any(alias in existing_models for alias in _EXTRA_ALIASES[extra_name]):
            models_to_train[extra_name] = spec

    # Data preparation is unconditional: the official-test evaluation below needs it even when no model is trained here.
    print(f"\n[INFO] Loading {subset} dataset...")
    loader = CMapssLoader()
    frame = loader.load(subset)
    frame["RUL"] = compute_piecewise_rul(frame)
    train_units, test_units = split_engine_units(frame, test_ratio=0.25, random_state=seed)
    sensors = loader.informative_sensors(frame.loc[frame["unit_id"].isin(train_units)])
    parts = partition_healthy_and_lifecycle(frame, train_units, test_units, healthy_cycle_limit)

    train_frame = parts["full_lifecycle_train"].assign(RUL=compute_piecewise_rul(parts["full_lifecycle_train"]))
    test_frame = parts["full_lifecycle_test"].assign(RUL=compute_piecewise_rul(parts["full_lifecycle_test"]))
    features = [*sensors, "setting_1", "setting_2", "setting_3", "cycle"]
    y_test = test_frame["RUL"].to_numpy(dtype=float)

    if not models_to_train:
        print("\n[INFO] All supplementary models (SVM, CNN, RNN, Transformer) are already in the benchmark!")
    else:
        print(f"\n[INFO] Models to train ({len(models_to_train)}):")
        for m in models_to_train:
            print(f"  + {m}")

        new_benchmark_rows = []

        for name, (m_type, model) in models_to_train.items():
            print(f"\n[TRAINING] Fitting '{name}' ({m_type})...", flush=True)
            started = time.perf_counter()
            y_pred = fit_predict(m_type, model, train_frame, features, test_frame)

            fit_time = time.perf_counter() - started
            metrics = evaluate_rul_predictions(y_test, y_pred)
            print(f"  -> '{name}' fit in {fit_time:.2f}s | R²: {metrics['R2']:.4f} | RMSE: {metrics['RMSE']:.2f} | MAE: {metrics['MAE']:.2f}", flush=True)

            new_benchmark_rows.append({
                "Model": name,
                **metrics,
                "Fit_Time_s": round(fit_time, 3),
            })
            predictions_df[f"RUL_predicted_{name}"] = y_pred

        # Append to benchmark_df
        updated_benchmark = pd.concat([benchmark_df, pd.DataFrame(new_benchmark_rows)], ignore_index=True)
        updated_benchmark.to_csv(benchmark_path, index=False)
        predictions_df.to_csv(predictions_path, index=False)
        print(f"\n[SUCCESS] Updated {benchmark_path.name} and {predictions_path.name} with all models.")


    # =========================================================================
    # Critical Zone & Censored Test Evaluations
    # =========================================================================
    pred_df = pd.read_csv(predictions_path)
    y_true_all = pred_df["RUL_actual"].to_numpy(dtype=float)
    model_pred_cols = [c for c in pred_df.columns if c.startswith("RUL_predicted_")]

    # Critical Zone (RUL < 50)
    crit_mask = y_true_all < 50.0
    crit_rows = []
    for col in model_pred_cols:
        m_name = col.replace("RUL_predicted_", "")
        crit_metrics = evaluate_rul_predictions(y_true_all[crit_mask], pred_df[col].to_numpy(dtype=float)[crit_mask])
        crit_rows.append({"Model": m_name, "N_cycles": int(np.sum(crit_mask)), **crit_metrics})
    crit_df = pd.DataFrame(crit_rows).sort_values("R2", ascending=False).reset_index(drop=True)
    crit_path = table_dir / "rul_critical_zone_benchmark.csv"
    crit_df.to_csv(crit_path, index=False)
    print(f"[SUCCESS] Critical zone benchmark exported to {crit_path.name}")

    # Official Censored Benchmark (test_{subset}.txt + RUL_{subset}.txt)
    # Official benchmark protocol: train on ALL training engines, score the LAST observed cycle of each
    # truncated test engine against RUL_{subset}.txt (capped at the same RUL ceiling used for training).
    full_train = frame.assign(RUL=compute_piecewise_rul(frame))
    print("\n[INFO] Official test protocol: refitting every benchmarked model on all training engines...", flush=True)
    censored_df = _official_test_benchmark(
        subset, full_train, sensors, features,
        [c.replace("RUL_predicted_", "") for c in model_pred_cols], seed, healthy_cycle_limit,
    )
    if censored_df is not None:
        censored_path = table_dir / "rul_censored_test_benchmark.csv"
        censored_df.to_csv(censored_path, index=False)
        print(f"[SUCCESS] Official censored benchmark exported to {censored_path.name}")
    else:
        print(f"[WARN] test_{subset}.txt / RUL_{subset}.txt not found; official-test benchmark skipped.")

    # =========================================================================
    # Run Thesis Statistical Hypothesis Analysis & p-values
    # =========================================================================
    print("\n" + "=" * 78)
    print("[THESIS ANALYSIS] Running rigorous statistical hypothesis testing with p-values...")
    print("=" * 78)

    analyzer = ThesisHypothesisAnalyzer(random_state=DEFAULT_RANDOM_STATE)

    # 1. Regime variance test (Engine-Block Repeated Measures Friedman)
    t50_res_path = table_dir / "virtual_sensor_residuals_t50.csv"
    if t50_res_path.exists():
        t50_df = pd.read_csv(t50_res_path)
        if "regime_id" in t50_df.columns and "T50" in t50_df.columns:
            if t50_df["regime_id"].nunique() > 1:
                res_h1 = analyzer.test_regime_variance_dominance(t50_df, sensor="T50", unit_col="unit_id")
                print(f"  [H1] Regime Block Friedman: stat={res_h1.test_statistic:.1f}, p={res_h1.p_value:.2e} -> {res_h1.conclusion}")
            else:
                print(f"  [H1] Single-regime dataset ({subset}): skipped (H1 tests variance across >=2 flight regimes).")

        if "residual" in t50_df.columns:
            res_h2 = analyzer.test_physical_degradation_shift(t50_df, healthy_cutoff=healthy_cycle_limit, unit_col="unit_id")
            print(f"  [H2] Degradation Shift (n={t50_df['unit_id'].nunique()}): stat={res_h2.test_statistic:.2f}, p={res_h2.p_value:.2e} -> {res_h2.conclusion}")

            # Fleet-wide Ljung-Box: per-engine tests on the healthy window, combined with Fisher's method
            try:
                res_h3 = analyzer.test_fleet_residual_autocorrelation(
                    t50_df, healthy_cutoff=healthy_cycle_limit, phase_label="frota_saudavel",
                )
                print(f"  [H3] Fleet Healthy Autocorr: Fisher p={res_h3.p_value:.2e} -> {res_h3.conclusion}")
            except ValueError as exc:
                print(f"  [H3] skipped: {exc}")

    # Model superiority tests vs baseline
    y_true = pred_df["RUL_actual"].to_numpy()
    # [BASELINE] Test against the informative "mean RUL by cycle" baseline: every model trivially beats the global mean.
    _base_col = "RUL_predicted_Baseline (Mean RUL by cycle)"
    baseline_pred = pred_df[_base_col].to_numpy() if _base_col in pred_df.columns else np.full(len(y_true), y_true.mean())

    model_cols = [c for c in pred_df.columns if c.startswith("RUL_predicted_") and "Baseline" not in c]
    for col in model_cols:
        m_name = col.replace("RUL_predicted_", "")
        res_m = analyzer.test_model_superiority_vs_baseline(y_true, pred_df[col].to_numpy(), baseline_pred, m_name, pred_df["unit_id"].to_numpy())
        print(f"  [H4] Superiority of {m_name:<26} vs Baseline: W={res_m.test_statistic:.1f}, p={res_m.p_value:.2e}")

    # IEEE CIM 2026 RQ2: GBDT (XGBoost) vs Deep models
    if "RUL_predicted_XGBoost" in pred_df.columns:
        xgb_pred = pred_df["RUL_predicted_XGBoost"].to_numpy()
        for deep_name in ["PyTorch Transformer", "PyTorch RNN (LSTM)", "PyTorch CNN", "MLP", "Support Vector Machine (SVM)"]:
            deep_col = f"RUL_predicted_{deep_name}"
            if deep_col in pred_df.columns:
                res_comp = analyzer.test_gbdt_vs_sequence_model_parity(y_true, xgb_pred, pred_df[deep_col].to_numpy(), deep_name, pred_df["unit_id"].to_numpy())
                print(f"  [H5] IEEE RQ2 XGBoost vs {deep_name:<22}: W={res_comp.test_statistic:.1f}, p={res_comp.p_value:.2e} -> {res_comp.conclusion}")

    # RQ2 with matched representation: XGBoost given the same causal temporal context vs deep sequence models
    if "RUL_predicted_XGBoost (+temporal)" in pred_df.columns:
        xgb_t_pred = pred_df["RUL_predicted_XGBoost (+temporal)"].to_numpy()
        for deep_name in ["PyTorch Transformer", "PyTorch RNN (LSTM)", "PyTorch CNN"]:
            deep_col = f"RUL_predicted_{deep_name}"
            if deep_col in pred_df.columns:
                res_comp = analyzer.test_gbdt_vs_sequence_model_parity(y_true, xgb_t_pred, pred_df[deep_col].to_numpy(), deep_name, pred_df["unit_id"].to_numpy())
                res_comp.hypothesis_id += "_TEMPORAL"
                res_comp.description += " (XGBoost com features temporais causais)"
                print(f"  [H5b] XGBoost(+temporal) vs {deep_name:<22}: W={res_comp.test_statistic:.1f}, p={res_comp.p_value:.2e}")

    # Export hypothesis summary
    hyp_summary, hyp_path = analyzer.export_results(table_dir)
    print(f"\n[SUCCESS] Hypothesis test summary exported to {hyp_path.name} ({len(hyp_summary)} tests).")

    # Compute 95% Bootstrap Confidence Intervals
    print("\n[INFO] Computing 95% Bootstrap Confidence Intervals (2,000 fleet resamples)...", flush=True)
    m_dict = {c.replace("RUL_predicted_", ""): c for c in model_cols}
    boot_ci = analyzer.compute_bootstrap_confidence_intervals(pred_df, m_dict, n_bootstraps=2000)
    boot_ci_path = table_dir / "thesis_bootstrap_ci.csv"
    boot_ci.to_csv(boot_ci_path, index=False)
    print(f"[SUCCESS] Bootstrap 95% CI exported to {boot_ci_path.name} ({len(boot_ci)} models).")
    print(boot_ci[["Model", "R2_Observed", "R2_Mean", "R2_CI_Lower", "R2_CI_Upper", "RMSE_Observed", "RMSE_Mean", "RMSE_CI_Lower", "RMSE_CI_Upper"]].to_string(index=False))

    return 0

class PdMPipeline:
    """Main object-oriented orchestrator for the PdM C-MAPSS analytical pipeline."""

    def __init__(
        self,
        config: PipelineConfig,
        loader: CMapssLoader | None = None,
        decoupler: RegimeDecoupler | None = None,
        benchmark: RULBenchmark | None = None,
    ) -> None:
        self.config = config
        self.output_dir = config.resolved_output_dir
        self.table_dir = self.output_dir / "tables"
        self.figure_dir = self.output_dir / "figures"

        self.loader = loader or CMapssLoader()
        n_clusters = 6 if config.subset in {"FD002", "FD004"} else 1
        self.decoupler = decoupler or RegimeDecoupler(n_clusters=n_clusters, random_state=config.seed)
        self.benchmark = benchmark or RULBenchmark(random_state=config.seed)

    def prepare_directories(self) -> None:
        """Create target directories, cleaning prior outputs if requested."""
        if self.config.clean_output and self.output_dir.exists():
            shutil.rmtree(self.output_dir, ignore_errors=True)
        self.table_dir.mkdir(parents=True, exist_ok=True)
        self.figure_dir.mkdir(parents=True, exist_ok=True)

    def run_core_workflow(self) -> dict[str, pd.DataFrame]:
        """Execute core virtual-sensor, SPC anomaly detection and RUL benchmark."""
        return run(
            subset=self.config.subset,
            output_dir=self.output_dir,
            seed=self.config.seed,
            include_lstm=self.config.include_lstm,
            include_minirocket=self.config.include_minirocket,
            healthy_cycle_limit=self.config.healthy_cycle_limit,
            sample_engine_id=self.config.unit_id,
            loader=self.loader,
            decoupler=self.decoupler,
            benchmark=self.benchmark,
        )

    def run_auxiliary_spc_horizon(self, horizon: int = 100) -> None:
        """Run auxiliary SPC calibration with a different healthy window limit.

        Runs in an isolated scratch directory so the primary (H = config) outputs are never overwritten;
        only the horizon-suffixed ``*_h{horizon}`` tables are copied back next to the primary tables.
        """
        if self.config.healthy_cycle_limit == horizon:
            return
        aux_dir = self.output_dir / f"_aux_h{horizon}"
        run(
            subset=self.config.subset,
            output_dir=aux_dir,
            seed=self.config.seed,
            include_lstm=False,
            include_minirocket=False,
            healthy_cycle_limit=horizon,
            sample_engine_id=self.config.unit_id,
            loader=self.loader,
        )
        for source in (aux_dir / "tables").glob(f"*_h{horizon}.csv"):
            shutil.copy2(source, self.table_dir / source.name)
        shutil.rmtree(aux_dir, ignore_errors=True)

    def run_additional_models_and_statistics(self) -> None:
        """Train supplementary models (SVM, CNN, RNN, Transformer) and run hypothesis testing."""
        train_additional_models_and_stats(
            output_dir=self.output_dir,
            subset=self.config.subset,
            seed=self.config.seed,
            healthy_cycle_limit=self.config.healthy_cycle_limit,
        )

    def run_monitoring_layer(self) -> None:
        """RUL uncertainty, criticality levels, alert events and grounded reports (see monitoring/layer.py)."""
        from pdm_cmaps.monitoring.layer import run_monitoring_layer

        run_monitoring_layer(
            subset=self.config.subset,
            output_dir=self.output_dir,
            seed=self.config.seed,
            healthy_cycle_limit=self.config.healthy_cycle_limit,
            sample_engine_id=self.config.unit_id,
            loader=self.loader,
        )

    def run_figure_generation(self) -> None:
        """Render publication-quality thesis diagnostic figures."""
        if self.config.subset == "FD004":
            figure_1_noise_vs_damage(self.output_dir)
            figure_2_physical_cascade(self.output_dir)
            figure_3_autocorrelation_evidence(self.output_dir)
            figure_4_fleet_persistence_tradeoff(self.output_dir)
        else:
            print(f"[figures] Thesis diagnostic figures 1-4 are specific to FD004 case study (skipped for {self.config.subset}).")

    def execute(self) -> int:
        """Execute the end-to-end pipeline in ordered stages."""
        t_start = time.perf_counter()
        sep = "=" * 78
        print(f"\n{sep}")
        print(f"[PdM PIPELINE] Initializing End-to-End Pipeline ({self.config.subset})")
        print(f"{sep}")
        print("  Configuration:")
        print(f"    - Subset: {self.config.subset}")
        print(f"    - Seed: {self.config.seed}")
        print(f"    - Healthy window limit (H): {self.config.healthy_cycle_limit}")
        print(f"    - Target unit ID: {self.config.unit_id}")
        print(f"    - MiniRocket: {self.config.include_minirocket}")
        print(f"    - LSTM: {self.config.include_lstm}")
        print(f"    - Output: {self.output_dir}")

        # Phase 0: Setup
        self.prepare_directories()
        run_aux = self.config.healthy_cycle_limit != 100
        progress = ProgressTracker(5 if run_aux else 4)  # prints "[i/N] fase ... | decorrido | restante ~"

        # Phase 1: Core PdM Pipeline (Regimes, Virtual Sensor, Envelopes, Baseline Models)
        progress.start("fase 1: pipeline analítico (regimes, sensor virtual, envelope, modelos-base)")
        self.run_core_workflow()
        progress.finish("fase 1")

        # Phase 2: Auxiliary SPC Horizon (H = 100)
        if run_aux:
            progress.start("fase 2: horizonte SPC auxiliar (H = 100)")
            self.run_auxiliary_spc_horizon(horizon=100)
            progress.finish("fase 2")

        # Phase 3: Additional Models & Hypothesis Testing
        progress.start("fase 3: modelos adicionais e testes estatísticos")
        self.run_additional_models_and_statistics()
        progress.finish("fase 3")

        # Phase 4: monitoring layer (uncertainty, criticality, alerts, grounded reports)
        progress.start("fase 4: camada de monitoramento (incerteza, criticidade, alertas, relatórios)")
        self.run_monitoring_layer()
        progress.finish("fase 4")

        # Phase 5: Thesis Figures
        progress.start("fase 5: figuras")
        self.run_figure_generation()
        progress.finish("fase 5")

        elapsed = time.perf_counter() - t_start
        print(f"\n{sep}")
        print(f"[PdM PIPELINE] Completed successfully in {elapsed:.1f}s")
        print(f"  - Figures: {self.figure_dir}")
        print(f"  - Tables:  {self.table_dir}")
        print(f"{sep}\n")
        return 0

### 2.2 Pacote `pdm_cmaps`: experiments

estudos (injeção cega, janela, hiperparâmetros etc.)

20 arquivos.

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/__init__.py
"""Runnable studies: `python -m pdm_cmaps.experiments.<name>`. Each writes under results/."""

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/benchmark_oficial_completo.py
"""Teste oficial de RUL com mais métricas e redes com normalização por regime.

WHAT   No protocolo oficial (reajuste em todo o treino; último ciclo de cada motor de test_FDxxx), guarda a PREVISÃO de cada motor
       (e a RUL verdadeira bruta, sem teto, além da truncada em 125), para calcular depois RMSE contra as duas verdades, MAE e a pontuação da NASA.
         part=main    os 13 modelos da Tabela de RUL (mesma receita de benchmark_repeated; as redes são determinísticas, então
                      repetem as previsões da rodada anterior; serve de conferência).
         part=boosting XGBoost e LightGBM com atributos temporais calculados sobre os sensores padronizados por regime (mais o regime
                      em one-hot), com janelas móveis de 5 e 10 ciclos (regime_t10) ou de 5, 10 e 30 (regime_t30); global_t30 usa a janela de 30 sem padronizar por regime (separa janela de regime).
         part=tuned   cada modelo com os hiperparâmetros (e a janela) escolhidos por validação cruzada nos motores de treino
                      (`selecao_hiperparametros`); Config=tuned_cv; --params-dir aponta para results/hiperparametros.
         part=extras  as três redes (CNN, LSTM, Transformer) com janela de 30 ciclos e/ou normalização por regime:
                      configuração  ref_w15      (referência da Tabela de RUL repetida na mesma máquina das demais)
                                    global_w30  (normalização global, janela 30)
                                    regime_w15  (cada sensor padronizado dentro do regime, mais o regime em one-hot, janela 15)
                                    regime_w30  (idem, janela 30)
                      Todas as configurações são reportadas (nenhuma escolhida no teste).
WHY    Comparar redes e boosting em pé de igualdade: o boosting ainda vence quando as redes recebem normalização por regime e janela maior?
       E guardar o RMSE também contra a verdade sem truncar e a pontuação da NASA.
[LEAKAGE] regimes, estatísticas por regime e escalonadores são ajustados só nos motores de treino.

Rodar da raiz (uma raia por subconjunto, com monitor):
    .venv/bin/python -m pdm_cmaps.experiments.benchmark_oficial_completo --subsets FD001 --part main --output-dir results/benchmark_oficial/FD001
    .venv/bin/python -m pdm_cmaps.experiments.benchmark_oficial_completo --aggregate-only --output-dir results/benchmark_oficial
"""

from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from pdm_cmaps.config import DEFAULT_RESULTS_DIR
from pdm_cmaps.data.features import CausalFeatureEngineer
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.detection.regime import RegimeDecoupler
from pdm_cmaps.domain.rul import compute_piecewise_rul, evaluate_rul_predictions, extended_rul_metrics
from pdm_cmaps.domain.schema import DEFAULT_HEALTHY_CYCLE_LIMIT, DEFAULT_RUL_CAP, SETTING_COLUMNS
from pdm_cmaps.experiments.benchmark_repeated import _models, _prepare
from pdm_cmaps.modeling.registry import get_model_pool
from pdm_cmaps.modeling.torch_models import PyTorchCNNModel, PyTorchRNNModel, PyTorchTransformerModel
from pdm_cmaps.pipeline.extra_models import fit_predict

NETS = {"PyTorch CNN": PyTorchCNNModel, "PyTorch RNN (LSTM)": PyTorchRNNModel, "PyTorch Transformer": PyTorchTransformerModel}
EXTRA_CONFIGS = (("ref_w15", False, 15), ("global_w30", False, 30), ("regime_w15", True, 15), ("regime_w30", True, 30))  # ref_w15 repete a referência na mesma máquina das demais
BOOST_CONFIGS = (("regime_t10", (5, 10), True), ("regime_t30", (5, 10, 30), True), ("global_t30", (5, 10, 30), False))  # (nome, janelas dos atributos temporais, sensores padronizados por regime?); global_t30 separa o efeito da janela do efeito do regime
BOOST_MODELS = ("XGBoost", "LightGBM", "XGBoost (NASA Loss)")
EXTENDED = {"Bias", "Median_AE", "P90_AE", "Hit_within_10", "Hit_within_20", *[f"{m}_c{c}" for m in ("Precision", "Recall", "F1", "PR_AUC") for c in (10, 15, 20)]}
SEEDS_MAIN = (42, 7, 123, 1, 2)
SEEDS_EXTRAS = (42, 7, 123)


def _regime_frames(train: pd.DataFrame, evalf: pd.DataFrame, sensors: list[str], n_regimes: int):
    """Sensores padronizados dentro do regime (estatísticas do treino) e regime em one-hot; devolve (train, eval, colunas)."""
    dec = RegimeDecoupler(n_clusters=n_regimes).fit(train)
    tr, ev = dec.transform(train), dec.transform(evalf)
    dec.fit_sensor_scaler(tr, sensors)
    tr, ev = dec.standardize_sensors(tr, sensors), dec.standardize_sensors(ev, sensors)
    cols = [f"{s}_norm" for s in sensors]
    for r in range(n_regimes if n_regimes > 1 else 0):
        for f in (tr, ev):
            f[f"regime_{r}"] = (f["regime_id"] == r).astype(float)
        cols.append(f"regime_{r}")
    return tr, ev, [*cols, *SETTING_COLUMNS, "cycle"]


def _last(pred: np.ndarray, units: np.ndarray) -> pd.Series:
    return pd.Series(np.asarray(pred, dtype=float)).groupby(units).last().sort_index()


def run(subsets, seeds, part: str, out: Path, only=None, params_dir: Path | None = None) -> None:
    out.mkdir(parents=True, exist_ok=True)
    path = out / "previsoes_oficial.csv"
    done = set()
    if path.exists():
        prev = pd.read_csv(path, usecols=["Subset", "Seed", "Model", "Config"]).drop_duplicates()
        done = set(zip(prev["Subset"], prev["Seed"], prev["Model"], prev["Config"]))
    loader = CMapssLoader()
    for subset in subsets:
        frame = loader.load(subset)
        frame["RUL"] = compute_piecewise_rul(frame)
        off_test, true_rul = loader.load_test(subset)
        y_raw = true_rul.to_numpy(dtype=float)
        n_regimes = 6 if subset in ("FD002", "FD004") else 1
        sensors = loader.informative_sensors(frame)
        for seed in seeds:
            if part == "main":
                base, temporal, tr_fe, ev_fe = _prepare(frame, frame, off_test, sensors)
                jobs = [(name, "global_w15", kind, model, use_t) for name, kind, model, use_t in _models(seed)]
                data = lambda use_t: (tr_fe, ev_fe, temporal) if use_t else (frame, off_test, base)  # noqa: E731
            elif part == "tuned":
                from pdm_cmaps.experiments.selecao_hiperparametros import MODELS, montar_dados, montar_modelo, slug

                jobs = []
                for name in MODELS:
                    f = (params_dir or Path("results/hiperparametros")) / subset / slug(name) / "melhores.json"
                    if f.exists():
                        params = json.loads(f.read_text())["params"]
                        kind, model = montar_modelo(name, params, seed)  # refit final sempre na CPU (determinismo validado só lá)
                        jobs.append((name, "tuned_cv", kind, model, params))
                data = lambda params: montar_dados(frame, off_test, sensors, n_regimes, params)  # noqa: E731
            elif part == "boosting":
                rtr, rev, rcols = _regime_frames(frame, off_test, sensors, n_regimes)
                normed = [c for c in rcols if c.endswith("_norm")]
                pool = get_model_pool(seed)
                jobs, frames = [], {}
                for cfg, windows, by_regime in BOOST_CONFIGS:
                    src_tr, src_ev, src_cols, sens = (rtr, rev, rcols, normed) if by_regime else (frame, off_test, [*sensors, *SETTING_COLUMNS, "cycle"], sensors)
                    eng = CausalFeatureEngineer(sens, windows=windows).fit(src_tr, DEFAULT_HEALTHY_CYCLE_LIMIT)
                    tr_fe, ev_fe = eng.transform(src_tr), eng.transform(src_ev)
                    frames[cfg] = (tr_fe, ev_fe, [*src_cols, *[c for c in tr_fe.columns if c not in src_tr.columns]])
                    jobs += [(f"{name} (+temporal)", cfg, "sklearn", pool[name], cfg) for name in BOOST_MODELS]
                data = lambda cfg: frames[cfg]  # noqa: E731
            else:
                rtr, rev, rcols = _regime_frames(frame, off_test, sensors, n_regimes)
                base = [*sensors, *SETTING_COLUMNS, "cycle"]
                jobs = []
                for cfg, use_regime, window in EXTRA_CONFIGS:
                    for name, cls in NETS.items():
                        jobs.append((name, cfg, "deep", cls(seq_length=window, random_state=seed), use_regime))
                data = lambda use_regime: (rtr, rev, rcols) if use_regime else (frame, off_test, base)  # noqa: E731
            for name, cfg, kind, model, flag in jobs:
                if (subset, seed, name, cfg) in done or (only and name not in only):
                    continue
                tr, ev, cols = data(flag)
                t0 = time.perf_counter()
                pred = fit_predict(kind, model, tr, cols, ev)
                last = _last(pred, ev["unit_id"].to_numpy())
                rows = pd.DataFrame({"Subset": subset, "Seed": seed, "Model": name, "Config": cfg, "unit_id": last.index,
                                     "pred": last.to_numpy(), "rul_raw": y_raw, "rul_cap": np.clip(y_raw, 0.0, float(DEFAULT_RUL_CAP))})
                rows.to_csv(path, mode="a", header=not path.exists(), index=False)
                rm = float(np.sqrt(np.mean((last.to_numpy() - rows["rul_cap"].to_numpy()) ** 2)))
                print(f"[oficial] {subset} seed={seed} {cfg:10s} {name:24s} RMSE(cap) {rm:.2f} ({time.perf_counter() - t0:.0f} s)", flush=True)


def aggregate(base: Path) -> pd.DataFrame:
    """Une as raias, calcula as métricas por (subconjunto, modelo, configuração, semente) e resume em média e desvio entre sementes."""
    parts = [pd.read_csv(f) for f in sorted(base.glob("*/previsoes_oficial.csv"))] + ([pd.read_csv(base / "previsoes_oficial.csv")] if (base / "previsoes_oficial.csv").exists() else [])
    allp = pd.concat(parts, ignore_index=True).drop_duplicates(subset=["Subset", "Seed", "Model", "Config", "unit_id"])
    rows = []
    for (sub, seed, model, cfg), g in allp.groupby(["Subset", "Seed", "Model", "Config"]):
        pred = g["pred"].to_numpy()
        cap, raw = g["rul_cap"].to_numpy(), g["rul_raw"].to_numpy()
        m_cap = evaluate_rul_predictions(cap, pred)
        m_raw = evaluate_rul_predictions(raw, pred)
        ext = extended_rul_metrics(cap, pred)
        rows.append({"Subset": sub, "Seed": seed, "Model": model, "Config": cfg, "RMSE_cap": m_cap["RMSE"], "RMSE_raw": m_raw["RMSE"],
                     "MAE_cap": m_cap["MAE"], "R2_cap": m_cap["R2"], "NASA_cap": m_cap["NASA_Score"], "Mean_NASA_cap": m_cap["Mean_NASA_Score"],
                     "NASA_raw": m_raw["NASA_Score"], "Late_pct_cap": m_cap["Late_Pred_Pct"], "n": len(g), **{f"{k}_cap": v for k, v in ext.items()}})
    per = pd.DataFrame(rows)
    per.to_csv(base / "metricas_por_semente.csv", index=False)
    metrics = ["RMSE_cap", "RMSE_raw", "MAE_cap", "NASA_cap", "Mean_NASA_cap", "NASA_raw", "Late_pct_cap", "R2_cap", *[c for c in per.columns if c.endswith("_cap") and c.split("_cap")[0] in EXTENDED]]
    summ = per.groupby(["Subset", "Model", "Config"])[metrics].agg(["mean", "std"]).reset_index()
    summ.columns = ["_".join(c).rstrip("_") for c in summ.columns]
    summ["Seeds"] = per.groupby(["Subset", "Model", "Config"]).size().to_numpy()
    summ.to_csv(base / "resumo_oficial.csv", index=False)
    return summ


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--subsets", nargs="+", default=["FD001", "FD002", "FD003", "FD004"])
    ap.add_argument("--part", choices=["main", "extras", "boosting", "tuned"], default="main")
    ap.add_argument("--seeds", nargs="+", type=int, default=None)
    ap.add_argument("--output-dir", type=Path, default=DEFAULT_RESULTS_DIR / "benchmark_oficial")
    ap.add_argument("--models", nargs="+", default=None, help="só estes modelos (nomes exatos)")
    ap.add_argument("--params-dir", type=Path, default=None, help="part=tuned: pasta com <subconjunto>/<modelo>/melhores.json (padrão results/hiperparametros)")
    ap.add_argument("--aggregate-only", action="store_true")
    a = ap.parse_args(argv)
    if not a.aggregate_only:
        run(a.subsets, tuple(a.seeds) if a.seeds else {"main": SEEDS_MAIN, "extras": SEEDS_EXTRAS, "boosting": (42,), "tuned": SEEDS_EXTRAS}[a.part], a.part, a.output_dir, a.models, a.params_dir)
    else:
        print(aggregate(a.output_dir).round(2).to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/benchmark_repeated.py
"""Repeated RUL benchmark (RQ-A) over subsets x seeds, in both evaluation protocols.

Uses the same models and recipes as pipeline.py.

WHAT   For every subset and seed, fits the main model families and scores them in
         holdout   75/25 engine split of train_FDxxx (the seed changes the split AND the model init);
         official  refit on all train_FDxxx engines, last cycle of test_FDxxx vs RUL_FDxxx (the seed
                   changes only the model initialisation, so its spread is pure training variance).
       Each (subset, seed, protocol, model) row is appended to a CSV as soon as it is computed, so the run
       can be stopped and resumed.
WHY    pipeline.py benchmarks one seed. "Model A is competitive with model B" needs a spread over seeds.
CONCEPTS
  [GROUPED-CV][LEAKAGE] same engine-level split and train-only fitting as pipeline.py.
  [PSEUDOREPLICATION] the per-engine MAE (holdout) or per-engine error (official) is stored so that
                 paired tests compare engines, not cycles.

Run from the repo root (long, mostly the deep models; nothing else heavy at the same time):
    .venv/bin/python -m pdm_cmaps.experiments.benchmark_repeated --output-dir results/benchmark_repeated
    .venv/bin/python -m pdm_cmaps.experiments.benchmark_repeated --output-dir results/benchmark_repeated --aggregate-only
"""

from __future__ import annotations

from pdm_cmaps.config import DEFAULT_RESULTS_DIR

import argparse
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

from pdm_cmaps.domain.schema import DEFAULT_HEALTHY_CYCLE_LIMIT, DEFAULT_RUL_CAP
from pdm_cmaps.data.features import CausalFeatureEngineer
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.domain.rul import compute_piecewise_rul, evaluate_rul_predictions
from pdm_cmaps.data.splits import split_engine_units
from pdm_cmaps.modeling.minirocket import MiniRocketRULModel
from pdm_cmaps.modeling.registry import get_model_pool
from pdm_cmaps.pipeline.extra_models import build_extra_models, fit_predict

TABULAR = ("Ridge", "Random Forest", "XGBoost", "XGBoost (NASA Loss)", "LightGBM", "CatBoost", "MLP")
TEMPORAL = ("XGBoost", "LightGBM")
DEEP = ("PyTorch CNN", "PyTorch RNN (LSTM)", "PyTorch Transformer")


def _models(seed: int) -> list[tuple[str, str, object, bool]]:
    """(name, kind, estimator, uses temporal features) in a fixed order."""
    pool = get_model_pool(seed)
    extra = build_extra_models(seed)
    out = [(name, "sklearn", pool[name], False) for name in TABULAR]
    out += [(f"{name} (+temporal)", "sklearn", get_model_pool(seed)[name], True) for name in TEMPORAL]
    out += [(name, *extra[name], False) for name in DEEP]
    return out


def _prepare(frame: pd.DataFrame, train_df: pd.DataFrame, eval_df: pd.DataFrame, sensors: list[str]):
    base = [*sensors, "setting_1", "setting_2", "setting_3", "cycle"]
    eng = CausalFeatureEngineer(sensors, windows=(5, 10)).fit(train_df, DEFAULT_HEALTHY_CYCLE_LIMIT)
    tr_fe, ev_fe = eng.transform(train_df), eng.transform(eval_df)
    temporal = [*base, *[c for c in tr_fe.columns if c not in train_df.columns]]
    return base, temporal, tr_fe, ev_fe


def run(subsets, seeds, out: Path) -> None:
    out.mkdir(parents=True, exist_ok=True)
    rows_path, err_path = out / "benchmark_rows.csv", out / "benchmark_engine_errors.csv"
    done = set()
    if rows_path.exists():
        prev = pd.read_csv(rows_path)
        done = set(zip(prev["Subset"], prev["Seed"], prev["Protocol"], prev["Model"]))
    loader = CMapssLoader()
    for subset in subsets:
        frame = loader.load(subset)
        frame["RUL"] = compute_piecewise_rul(frame)
        off_test, true_rul = loader.load_test(subset)
        y_off = np.clip(true_rul.to_numpy(dtype=float), 0.0, float(DEFAULT_RUL_CAP))
        for seed in seeds:
            tr_units, te_units = split_engine_units(frame, test_ratio=0.25, random_state=seed)
            train_df = frame.loc[frame["unit_id"].isin(tr_units)].reset_index(drop=True)
            test_df = frame.loc[frame["unit_id"].isin(te_units)].reset_index(drop=True)
            sensors = loader.informative_sensors(train_df)
            setups = {
                "holdout": (train_df, test_df, *_prepare(frame, train_df, test_df, sensors)),
                "official": (frame, off_test, *_prepare(frame, frame, off_test, sensors)),
            }
            for protocol, (tr, ev, base, temporal, tr_fe, ev_fe) in setups.items():
                for name, kind, model, use_temporal in _models(seed):
                    if (subset, seed, protocol, name) in done:
                        continue
                    t0 = time.perf_counter()
                    if use_temporal:
                        pred = fit_predict(kind, model, tr_fe, temporal, ev_fe)
                        units = ev_fe["unit_id"].to_numpy()
                    else:
                        pred = fit_predict(kind, model, tr, base, ev)
                        units = ev["unit_id"].to_numpy()
                    if protocol == "holdout":
                        truth = (ev_fe if use_temporal else ev)["RUL"].to_numpy(dtype=float)
                        metrics = evaluate_rul_predictions(truth, pred)
                        err = pd.DataFrame({"unit_id": units, "abs_err": np.abs(truth - pred)}).groupby("unit_id")["abs_err"].mean()
                    else:
                        last = pd.Series(pred).groupby(units).last().sort_index()
                        metrics = evaluate_rul_predictions(y_off, last.to_numpy())
                        err = pd.Series(np.abs(y_off - last.to_numpy()), index=last.index)
                    row = {"Subset": subset, "Seed": seed, "Protocol": protocol, "Model": name, **metrics,
                           "Fit_s": round(time.perf_counter() - t0, 1)}
                    pd.DataFrame([row]).to_csv(rows_path, mode="a", header=not rows_path.exists(), index=False)
                    e = err.rename("abs_err").reset_index().rename(columns={"index": "unit_id"})
                    e.assign(Subset=subset, Seed=seed, Protocol=protocol, Model=name).to_csv(
                        err_path, mode="a", header=not err_path.exists(), index=False)
                    print(f"[bench] {subset} seed={seed} {protocol:8s} {name:24s} RMSE {row['RMSE']:.2f} ({row['Fit_s']} s)", flush=True)


def aggregate(out: Path) -> pd.DataFrame:
    rows = pd.read_csv(out / "benchmark_rows.csv")
    summary = rows.groupby(["Protocol", "Subset", "Model"]).agg(
        R2_mean=("R2", "mean"), R2_std=("R2", "std"), RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"),
        Mean_NASA_mean=("Mean_NASA_Score", "mean"), Runs=("Seed", "size")).reset_index()
    rows["Rank"] = rows.groupby(["Protocol", "Subset", "Seed"])["RMSE"].rank()
    ranks = rows.groupby(["Protocol", "Subset", "Model"]).agg(Rank_mean=("Rank", "mean"), Top3_runs=("Rank", lambda r: int((r <= 3).sum()))).reset_index()
    summary = summary.merge(ranks, on=["Protocol", "Subset", "Model"]).sort_values(["Protocol", "Subset", "RMSE_mean"])
    summary.to_csv(out / "benchmark_repeated_summary.csv", index=False)

    # paired Wilcoxon per seed, best tabular (XGBoost +temporal) vs each deep model, on per-engine errors;
    # p-values combined over seeds with Fisher's method (seeds are independent splits in the holdout protocol)
    err = pd.read_csv(out / "benchmark_engine_errors.csv")
    tests = []
    ref = "XGBoost (+temporal)"
    for (protocol, subset), g in err.groupby(["Protocol", "Subset"]):
        for other in [*DEEP, "XGBoost", "LightGBM (+temporal)"]:
            ps, diffs = [], []
            for seed, gs in g.groupby("Seed"):
                a = gs.loc[gs["Model"] == ref].set_index("unit_id")["abs_err"]
                b = gs.loc[gs["Model"] == other].set_index("unit_id")["abs_err"]
                both = a.index.intersection(b.index)
                if len(both) < 5:
                    continue
                ps.append(stats.wilcoxon(a[both], b[both]).pvalue)
                diffs.append(float((b[both] - a[both]).median()))
            if ps:
                fisher = stats.combine_pvalues(ps, method="fisher").pvalue
                tests.append({"Protocol": protocol, "Subset": subset, "Comparison": f"{ref} vs {other}", "Seeds": len(ps),
                              "Median_err_diff_other_minus_ref": float(np.mean(diffs)), "p_per_seed_max": float(max(ps)),
                              "p_fisher": float(fisher)})
    t = pd.DataFrame(tests)
    t.to_csv(out / "benchmark_repeated_tests.csv", index=False)
    return summary


def merge_lanes(base: Path, subsets=("FD001", "FD002", "FD003", "FD004")) -> Path:
    """Junta as pastas das raias (`<base>/<FDxxx>*/`, uma ou mais por subconjunto, p. ex. `FD002` e `FD002_s12`) em `<base>/` para agregar.

    Lança erro se faltar a raia de algum subconjunto ou se a mesma (subconjunto, semente, protocolo, modelo) aparecer em duas pastas
    (duplicata: a agregação contaria a mesma execução duas vezes)."""
    for name in ("benchmark_rows.csv", "benchmark_engine_errors.csv"):
        parts = []
        for sub in subsets:
            files = sorted(base.glob(f"{sub}*/{name}"))
            if not files:
                raise FileNotFoundError(f"{base}/{sub}*/{name} ausente: a raia {sub} não gravou nada")
            parts += [pd.read_csv(f) for f in files]
        merged = pd.concat(parts, ignore_index=True)
        key = [c for c in ("Subset", "Seed", "Protocol", "Model", "unit_id") if c in merged.columns]
        if len(key) >= 4 and merged.duplicated(subset=key).any():
            raise ValueError(f"{name}: linhas duplicadas entre as pastas das raias ({int(merged.duplicated(subset=key).sum())})")
        merged.to_csv(base / name, index=False)
    return base


def engine_level_tests(err: pd.DataFrame, ref: str = "XGBoost (+temporal)", others=None, min_engines: int = 5) -> pd.DataFrame:
    """Testes pareados POR MOTOR: o erro de cada motor é a média sobre as sementes em que ele foi avaliado, e há um Wilcoxon por comparação.

    Por que não combinar os p-valores das sementes (Fisher): as sementes reutilizam os mesmos motores (no protocolo oficial, o MESMO conjunto de
    teste; no holdout, divisões que se sobrepõem), então não são réplicas independentes e o Fisher infla a evidência. Aqui cada motor conta uma
    vez; `Seeds_per_engine` mostra quantas sementes entraram na média dele.
    """
    others = list(others) if others is not None else [*DEEP, "XGBoost", "LightGBM (+temporal)"]
    rows = []
    for (protocol, subset), g in err.groupby(["Protocol", "Subset"]):
        per_engine = g.groupby(["Model", "unit_id"])["abs_err"].agg(["mean", "size"]).reset_index()
        a = per_engine[per_engine["Model"] == ref].set_index("unit_id")
        for other in others:
            b = per_engine[per_engine["Model"] == other].set_index("unit_id")
            both = a.index.intersection(b.index)
            if len(both) < min_engines:
                continue
            d = b.loc[both, "mean"] - a.loc[both, "mean"]
            res = stats.wilcoxon(a.loc[both, "mean"], b.loc[both, "mean"])
            rows.append({"Protocol": protocol, "Subset": subset, "Comparison": f"{ref} vs {other}", "n_engines": len(both),
                         "Seeds_per_engine_mean": float(a.loc[both, "size"].mean()), "Median_err_diff_other_minus_ref": float(d.median()),
                         "p_wilcoxon_engines": float(res.pvalue)})
    return pd.DataFrame(rows)


def main(argv=None) -> int:
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--subsets", nargs="+", default=["FD001", "FD002", "FD003", "FD004"])
    parser.add_argument("--seeds", nargs="+", type=int, default=[42, 7, 123, 1, 2], help="5 sementes (protocol.SEEDS_RUL)")
    parser.add_argument("--output-dir", type=Path, default=DEFAULT_RESULTS_DIR / "benchmark_repeated")
    parser.add_argument("--aggregate-only", action="store_true")
    parser.add_argument("--merge-lanes", action="store_true", help="antes de agregar, junta <output-dir>/<FDxxx>/ (uma raia por subconjunto) em <output-dir>/")
    args = parser.parse_args(argv)
    if not args.aggregate_only and not args.merge_lanes:
        run(args.subsets, args.seeds, args.output_dir)
    if args.merge_lanes:
        merge_lanes(args.output_dir, tuple(args.subsets))
    summary = aggregate(args.output_dir)
    engine_level_tests(pd.read_csv(args.output_dir / "benchmark_engine_errors.csv")).to_csv(args.output_dir / "benchmark_repeated_tests_por_motor.csv", index=False)
    print(summary.round(3).to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/compare_engine_offset.py
"""Compara o sensor virtual com e sem o deslocamento do motor (VirtualSensor(engine_offset=True)).

WHAT: lê duas pastas de runs do estudo de injeção a falso alarme igualado (fault_detection_study: uma com --no-engine-offset e outra
com o padrão) e as duas pastas do estudo do sensor virtual (virtual_sensor_models, também sem e com o deslocamento), e grava duas
tabelas: poder e atraso pareados (mesma semente, forma e magnitude) e R² da janela tardia por sensor e modelo.
WHY: em regime único o modelo vira a média da frota e o deslocamento entre motores domina; este é o teste de melhorar o
sensor virtual do FD003. O detector padrão não muda: a opção fica desligada por padrão.

Run:  .venv/bin/python -m pdm_cmaps.experiments.compare_engine_offset --base-runs DIR [DIR ...] --offset-runs DIR [DIR ...] \
          --vs-base results/virtual_sensor_models --vs-offset results/virtual_sensor_models_offset --out results/sensor_virtual_offset
"""
from __future__ import annotations

import argparse
import glob
from pathlib import Path

import pandas as pd

KEYS = ["Detector", "kind", "magnitude", "Subset", "Seed"]
DETECTORS = ["T50 EWMA (com regime)", "T50 EWMA (com regime, sem portão)", "MEWMA (13 resíduos, com regime)"]


def _runs(dirs: list[Path], kind: str = "pareado") -> pd.DataFrame:
    files = [f for d in dirs for f in glob.glob(str(Path(d) / "runs" / f"*_{kind}.csv"))]
    if not files:
        raise SystemExit(f"nenhum *_{kind}.csv em {dirs}")
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


def power_table(base_dirs: list[Path], offset_dirs: list[Path]) -> pd.DataFrame:
    """Poder, falso alarme e atraso pareados por subconjunto e detector; 'celulas_*' conta as células (semente, forma, magnitude)."""
    m = _runs(base_dirs).merge(_runs(offset_dirs), on=KEYS, suffixes=("_base", "_offset"))
    m = m[m["Detector"].isin(DETECTORS)].assign(ganho=lambda d: d["power_pct_offset"] - d["power_pct_base"])
    return m.groupby(["Subset", "Detector"]).agg(
        poder_base=("power_pct_base", "mean"), poder_offset=("power_pct_offset", "mean"), ganho_medio=("ganho", "mean"),
        celulas=("ganho", "size"), celulas_melhores=("ganho", lambda x: int((x > 0).sum())), celulas_piores=("ganho", lambda x: int((x < 0).sum())),
        far_base=("far_ctrl_per100_base", "mean"), far_offset=("far_ctrl_per100_offset", "mean"),
        atraso_base=("delay_med_base", "mean"), atraso_offset=("delay_med_offset", "mean")).round(2).reset_index()


def r2_table(vs_base: Path, vs_offset: Path) -> pd.DataFrame:
    rows = []
    for d in sorted(Path(vs_offset).glob("FD*")):
        o, b = pd.read_csv(d / "model_scores.csv"), pd.read_csv(Path(vs_base) / d.name / "model_scores.csv")
        m = b.merge(o, on=["sensor", "model"], suffixes=("_base", "_offset"))
        for col in ("r2_late_model", "r2_holdout", "detection_rate_pct", "mean_lead_time_detected_cycles", "mean_healthy_false_alarm_pct"):
            m[col + "_delta"] = m[col + "_offset"] - m[col + "_base"]
        rows.append(m.assign(Subset=d.name))
    keep = ["Subset", "sensor", "model", "r2_late_model_base", "r2_late_motor_base_base", "r2_late_model_offset", "r2_holdout_base", "r2_holdout_offset",
            "detection_rate_pct_base", "detection_rate_pct_offset", "mean_lead_time_detected_cycles_base", "mean_lead_time_detected_cycles_offset",
            "mean_healthy_false_alarm_pct_base", "mean_healthy_false_alarm_pct_offset"]
    return pd.concat(rows, ignore_index=True)[keep].round(3)


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--base-runs", nargs="+", type=Path, required=True)
    ap.add_argument("--offset-runs", nargs="+", type=Path, required=True)
    ap.add_argument("--vs-base", type=Path, default=Path("results/virtual_sensor_models"))
    ap.add_argument("--vs-offset", type=Path, default=Path("results/virtual_sensor_models_offset"))
    ap.add_argument("--out", type=Path, default=Path("results/sensor_virtual_offset"))
    a = ap.parse_args(argv)
    a.out.mkdir(parents=True, exist_ok=True)
    p = power_table(a.base_runs, a.offset_runs)
    p.to_csv(a.out / "comparacao_poder.csv", index=False)
    print(p.to_string(index=False))
    if a.vs_offset.exists():
        r = r2_table(a.vs_base, a.vs_offset)
        r.to_csv(a.out / "comparacao_r2.csv", index=False)
        print(r.groupby(["Subset", "model"])[["r2_late_model_base", "r2_late_model_offset", "r2_late_motor_base_base"]].mean().round(3).to_string())
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/envelope_ignored_cases.py
"""Small deviations that leave the envelope but do NOT become alerts: cases, numbers and figure for the TCC.

WHAT: the prediction envelope of the T50 virtual sensor is  y_pred ± k·sigma_e  (k = 3, sigma_e from the OUT-OF-FOLD
healthy residuals of the training engines). A cycle outside the envelope is a "breach"; an ALERT needs `k_persist`
consecutive breaches after the calibration window (default 2: the Western Electric persistence rule of the pipeline).
An isolated breach (one cycle) is therefore ignored. This module shows those ignored cases on ALL test engines and
measures what the rule buys (false alerts avoided) and what it costs (alert issued one cycle later).
WHY: with a 3-sigma limit even healthy noise leaves the envelope sometimes (the healthy residual is not exactly Gaussian,
and 62 engines x ~200 cycles give thousands of chances); alerting on every breach would flood the engineer with false
alarms. The persistence rule trades one cycle of delay for far fewer false alerts.

[LEAKAGE] nothing is refit: the table `envelope_scored_trace_t50.csv` (sample engine) carries the calibration of the
pipeline run, from which mu_e and sigma_e are recovered EXACTLY (upper - prediction = mu_e + k·sigma_e), then applied to
the residuals of every test engine in `virtual_sensor_residuals_t50.csv` (same seed 42, H = 50 run).
[ASSUMPTION] "healthy zone" = cycles after H whose piecewise RUL label is 125; engine 179 (FD004) is excluded from the
false-alert statistics because its core resets at cycle 242 (its early life is really degrading, label notwithstanding).

Run:  python -m pdm_cmaps.experiments.envelope_ignored_cases --results /path/to/results
Output: results/envelope_ignored_cases/F17_envelope_desvios_ignorados.png (--figure), tcc-writing/tables/envelope_desvios_{resumo,casos}.csv
"""

from __future__ import annotations

from pdm_cmaps.config import DEFAULT_RESULTS_DIR

import argparse
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec

from pdm_cmaps.reporting import viz_style

SUBSETS = ("FD001", "FD002", "FD003", "FD004")
K_SIGMA = 3.0
PERSIST = 2
H = 50
RUL_CAP = 125
RESET_ENGINES = {("FD004", 179)}


def fleet_envelope(results: Path, subset: str) -> tuple[pd.DataFrame, dict]:
    """T50, prediction and envelope for every test engine, with the calibration recovered from the pipeline run."""
    base = results / f"maintenance_guide_{subset.lower()}" / "tables"
    trace = pd.read_csv(base / "envelope_scored_trace_t50.csv")
    margin = float(pd.read_csv(base / "envelope_summary_t50.csv")["envelope_margin"].iloc[0])
    mu = float((trace["envelope_upper"] - trace["prediction"]).iloc[0] - margin)
    r = pd.read_csv(base / "virtual_sensor_residuals_t50.csv").sort_values(["unit_id", "cycle"]).reset_index(drop=True)
    r["upper"] = r["prediction"] + mu + margin
    r["lower"] = r["prediction"] + mu - margin
    r["breach"] = (r["T50"] > r["upper"]) | (r["T50"] < r["lower"])
    sigma = margin / K_SIGMA
    r["excess_sigma"] = (np.maximum(0, r["T50"] - r["upper"]) + np.maximum(0, r["lower"] - r["T50"])) / sigma
    r["res_sigma"] = (r["T50"] - r["prediction"] - mu) / sigma  # envelope = flat band of +-K_SIGMA in this scale
    r["Subset"] = subset
    return r, {"margin": margin, "sigma": sigma, "mu": mu}


def breach_runs(r: pd.DataFrame, persist: int = PERSIST, h: int = H) -> pd.DataFrame:
    """Every maximal run of consecutive breaches, with its length, size, phase and whether it raised an alert."""
    rows = []
    for (sub, unit), g in r.groupby(["Subset", "unit_id"], sort=True):
        b, c = g["breach"].to_numpy(), g["cycle"].to_numpy()
        ex, rul = g["excess_sigma"].to_numpy(), g["RUL"].to_numpy()
        i = 0
        while i < len(b):
            if not b[i]:
                i += 1
                continue
            j = i
            while j + 1 < len(b) and b[j + 1]:
                j += 1
            monitored = c[i] > h
            phase = "janela de calibração (não monitorada)" if not monitored else ("vida saudável (RUL = 125)" if rul[i] >= RUL_CAP else "degradação (RUL < 125)")
            rows.append({"Subset": sub, "unit_id": int(unit), "start": int(c[i]), "end": int(c[j]), "length": j - i + 1, "max_excess_sigma": float(ex[i:j + 1].max()),
                         "RUL_label": float(rul[i]), "monitored": monitored, "phase": phase, "alert": bool(monitored and (j - i + 1) >= persist)})
            i = j + 1
    out = pd.DataFrame(rows)
    out["alert_cycle"] = np.where(out["alert"], out["start"] + persist - 1, np.nan)  # the alert is issued when the k-th breach arrives
    first = out[out["alert"]].groupby(["Subset", "unit_id"])["start"].transform("min")
    out["first_alert"] = out["alert"] & (out["start"] == first.reindex(out.index))
    return out


def summarize(r: pd.DataFrame, runs: pd.DataFrame, persist: int = PERSIST, h: int = H) -> pd.DataFrame:
    rows = []
    for sub in SUBSETS:
        rs, ru = r[r["Subset"] == sub], runs[runs["Subset"] == sub]
        if rs.empty:
            continue
        keep = ~rs["unit_id"].map(lambda u: (sub, int(u)) in RESET_ENGINES)
        healthy_cycles = int(((rs["cycle"] > h) & (rs["RUL"] >= RUL_CAP) & keep).sum())
        rh = ru[ru["phase"].str.startswith("vida saudável") & ~ru["unit_id"].map(lambda u: (sub, int(u)) in RESET_ENGINES)]
        mon = ru[ru["monitored"]]
        ign, per = mon[~mon["alert"]], mon[mon["alert"]]
        firsts = ru[ru["first_alert"]]
        pre = []
        for u, f in firsts.groupby("unit_id"):
            pre.append(int(((ru["unit_id"] == u) & ru["monitored"] & ~ru["alert"] & (ru["start"] < f["start"].iloc[0])).sum()))
        rows.append({
            "Subset": sub, "engines": rs["unit_id"].nunique(), "healthy_cycles_monitored": healthy_cycles,
            "breach_runs_monitored": len(mon), "runs_ignored": len(ign), "runs_alert": len(per), "pct_runs_ignored": 100 * len(ign) / max(len(mon), 1),
            "healthy_runs_total": len(rh), "healthy_runs_ignored": int((~rh["alert"]).sum()), "healthy_runs_alert": int(rh["alert"].sum()),
            "false_alert_per100_without_persistence": 100 * len(rh) / max(healthy_cycles, 1),
            "false_alert_per100_with_persistence": 100 * int(rh["alert"].sum()) / max(healthy_cycles, 1),
            "excess_sigma_median_ignored": float(ign["max_excess_sigma"].median()) if len(ign) else np.nan,
            "excess_sigma_p90_ignored": float(ign["max_excess_sigma"].quantile(0.9)) if len(ign) else np.nan,
            "excess_sigma_median_alert": float(per["max_excess_sigma"].median()) if len(per) else np.nan,
            "engines_with_alert": int(firsts["unit_id"].nunique()), "engines_with_ignored_before_first_alert": int(sum(p > 0 for p in pre)),
            "mean_ignored_before_first_alert": float(np.mean(pre)) if pre else np.nan, "alert_delay_cycles_by_rule": persist - 1,
        })
    return pd.DataFrame(rows)


def _band(ax) -> None:
    ax.axhspan(-K_SIGMA, K_SIGMA, color=viz_style.GRID, alpha=0.9, label="envelope (±3σ)")
    ax.axhline(0, color=viz_style.MUTED, lw=1, ls="--", label="previsto pelo sensor virtual")


def _mark(ax, g: pd.DataFrame, runs_g: pd.DataFrame, size: int) -> None:
    for _, rr in runs_g.iterrows():
        seg = g[(g["cycle"] >= rr["start"]) & (g["cycle"] <= rr["end"])]
        if seg.empty:
            continue
        ax.scatter(seg["cycle"], seg["res_sigma"], s=size, color=viz_style.STATUS[3] if rr["alert"] else viz_style.STATUS[1], edgecolor=viz_style.INK, zorder=5,
                   label="alerta (≥ 2 ciclos seguidos fora)" if rr["alert"] else "desvio isolado, ignorado")


def _zoom(ax, g: pd.DataFrame, run: pd.Series, title: str, color: str, runs_g: pd.DataFrame) -> None:
    lo, hi = run["start"] - 14, run["end"] + 14
    w = g[(g["cycle"] >= lo) & (g["cycle"] <= hi)]
    _band(ax)
    ax.plot(w["cycle"], w["res_sigma"], color=viz_style.INK, lw=1.4, marker="o", ms=3.5, label="resíduo de T50 (σ)")
    _mark(ax, w, runs_g, 90)
    ax.set_title(title, fontsize=9, color=color, loc="left")
    ax.set_xlabel("ciclo", fontsize=8)
    ax.set_ylabel("resíduo de T50 (σ saudável)", fontsize=8)
    ax.set_ylim(min(-4.2, w["res_sigma"].min() - 0.5), max(4.2, w["res_sigma"].max() + 0.5))
    viz_style.style_axes(ax)


def make_figure(r: pd.DataFrame, runs: pd.DataFrame, summ: pd.DataFrame, path: Path, subset: str = "FD004") -> dict:
    viz_style.apply_style()
    rs, ru = r[r["Subset"] == subset], runs[runs["Subset"] == subset]
    mon = ru[ru["monitored"]]
    ign_by_unit = mon[~mon["alert"]].groupby("unit_id").size()
    firsts = ru[ru["first_alert"]].set_index("unit_id")["start"]
    cand = [(int(u), int(n)) for u, n in ign_by_unit.items() if u in firsts.index and (subset, int(u)) not in RESET_ENGINES and (mon[(mon["unit_id"] == u) & ~mon["alert"] & (mon["start"] < firsts[u])].shape[0] >= 2)]
    top = max(cand, key=lambda t: (t[1], t[0]))[0] if cand else int(rs["unit_id"].iloc[0])
    g_top = rs[rs["unit_id"] == top]
    runs_top = ru[ru["unit_id"] == top]

    healthy_ign = mon[(~mon["alert"]) & mon["phase"].str.startswith("vida saudável") & ~mon["unit_id"].map(lambda u: (subset, int(u)) in RESET_ENGINES)]
    deg_ign = mon[(~mon["alert"]) & mon["phase"].str.startswith("degradação")]
    case_b = healthy_ign.iloc[(healthy_ign["max_excess_sigma"] - healthy_ign["max_excess_sigma"].median()).abs().argsort()].iloc[0]
    case_c = deg_ign.iloc[(deg_ign["max_excess_sigma"] - deg_ign["max_excess_sigma"].median()).abs().argsort()].iloc[0] if len(deg_ign) else case_b
    case_d = runs_top[runs_top["first_alert"]].iloc[0]

    fig = plt.figure(figsize=(15.5, 13.5))
    gs = GridSpec(3, 6, figure=fig, hspace=0.55, wspace=0.7, height_ratios=[1.1, 1, 1])
    ax = fig.add_subplot(gs[0, :])
    _band(ax)
    ax.plot(g_top["cycle"], g_top["res_sigma"], color=viz_style.INK, lw=1.0, label="resíduo de T50 (σ)")
    _mark(ax, g_top, runs_top[runs_top["monitored"]], 70)
    fa = runs_top[runs_top["first_alert"]]["alert_cycle"].iloc[0]
    ax.axvline(fa, color=viz_style.STATUS[3], lw=1.5, ls="--")
    ax.axvspan(g_top["cycle"].min(), H, color=viz_style.STATUS[0], alpha=0.10, label=f"janela de calibração (≤ {H}, não monitorada)")
    h, l = ax.get_legend_handles_labels()
    ax.legend(dict(zip(l, h)).values(), dict(zip(l, h)).keys(), fontsize=8, frameon=False, ncol=3, loc="upper left")
    n_ign = int(((~runs_top["alert"]) & runs_top["monitored"]).sum())
    ax.set_title(f"(A) {subset}, motor {top}: {n_ign} desvios isolados ignorados; o alerta sai no ciclo {int(fa)}, quando 2 ciclos seguidos saem do envelope", fontsize=10.5, loc="left")
    ax.set_xlabel("ciclo")
    ax.set_ylabel("resíduo de T50 (σ saudável)")
    ax.set_ylim(min(-4.5, g_top["res_sigma"].min() - 0.5), g_top["res_sigma"].max() + 1.5)
    viz_style.style_axes(ax)

    for k, (run, ttl, col, label) in enumerate([
        (case_b, f"(B) motor {int(case_b['unit_id'])}, vida saudável:\n1 ciclo, {case_b['max_excess_sigma']:.2f}σ além do envelope", viz_style.INK_2, "b"),
        (case_c, f"(C) motor {int(case_c['unit_id'])}, em degradação:\n1 ciclo isolado, {case_c['max_excess_sigma']:.2f}σ além", viz_style.INK_2, "c"),
        (case_d, f"(D) motor {top}: 2 ou mais ciclos\nseguidos fora viram alerta", viz_style.STATUS[3], "d"),
    ]):
        axz = fig.add_subplot(gs[1, 2 * k:2 * k + 2])
        gz = rs[rs["unit_id"] == int(run["unit_id"])]
        _zoom(axz, gz, run, ttl, col, ru[(ru["unit_id"] == int(run["unit_id"])) & ru["monitored"] & (ru["end"] >= run["start"] - 14) & (ru["start"] <= run["end"] + 14)])

    axe = fig.add_subplot(gs[2, :3])
    pool_ign = runs[runs["monitored"] & ~runs["alert"]]["max_excess_sigma"]
    pool_per = runs[runs["monitored"] & runs["alert"]]["max_excess_sigma"]
    bins = np.linspace(0, max(8, pool_per.quantile(0.9)), 22)
    axe.hist(pool_ign, bins=bins, alpha=0.8, color=viz_style.STATUS[1], label=f"ignorados (n = {len(pool_ign)})", edgecolor=viz_style.SURFACE)
    axe.hist(pool_per, bins=bins, alpha=0.75, color=viz_style.STATUS[3], label=f"viraram alerta (n = {len(pool_per)})", edgecolor=viz_style.SURFACE)
    axe.set_xlabel("excesso máximo além do envelope (em σ do resíduo saudável)")
    axe.set_ylabel("sequências de desvio")
    axe.set_title("(E) Desvios ignorados são pequenos; os persistentes\nsão maiores (4 subconjuntos, motores de teste)", fontsize=10, loc="left")
    axe.legend(frameon=False, fontsize=8)
    viz_style.style_axes(axe)

    axf = fig.add_subplot(gs[2, 3:])
    x = np.arange(len(summ))
    axf.bar(x - 0.2, summ["false_alert_per100_without_persistence"], 0.4, color=viz_style.INK_2, label="alertar em todo desvio")
    axf.bar(x + 0.2, summ["false_alert_per100_with_persistence"], 0.4, color=viz_style.SERIES[0], label="exigir 2 ciclos seguidos")
    for xi, (a, b) in enumerate(zip(summ["false_alert_per100_without_persistence"], summ["false_alert_per100_with_persistence"])):
        axf.text(xi - 0.2, a, f"{a:.2f}", ha="center", va="bottom", fontsize=8)
        axf.text(xi + 0.2, b, f"{b:.2f}", ha="center", va="bottom", fontsize=8)
    axf.set_xticks(x)
    axf.set_xticklabels(summ["Subset"])
    axf.set_ylabel("alertas falsos por 100 ciclos de vida saudável")
    axf.set_title("(F) O que a persistência evita: alertas falsos\nna vida saudável (RUL = 125, após a calibração)", fontsize=10, loc="left")
    axf.legend(frameon=False, fontsize=8)
    viz_style.style_axes(axf)
    fig.suptitle("Pequenos desvios do envelope que não viram alerta (T50, sensor virtual, limite 3σ, persistência de 2 ciclos)", fontsize=12.5, y=0.93)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=140, bbox_inches="tight")
    plt.close(fig)
    return {"top_engine": top, "case_b": (int(case_b["unit_id"]), int(case_b["start"])), "case_c": (int(case_c["unit_id"]), int(case_c["start"])), "case_d": (top, int(case_d["start"]))}


def main(argv: list[str] | None = None) -> int:
    p = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    p.add_argument("--results", type=Path, default=DEFAULT_RESULTS_DIR)
    p.add_argument("--figure", type=Path, default=DEFAULT_RESULTS_DIR / "envelope_ignored_cases" / "F17_envelope_desvios_ignorados.png")
    p.add_argument("--tables", type=Path, default=Path("tcc-writing/tables"))
    a = p.parse_args(argv)
    frames = []
    for s in SUBSETS:
        try:
            frames.append(fleet_envelope(a.results, s)[0])
        except FileNotFoundError:
            print(f"[skip] {s}: tabelas do pipeline ausentes")
    r = pd.concat(frames, ignore_index=True)
    runs = breach_runs(r)
    summ = summarize(r, runs)
    a.tables.mkdir(parents=True, exist_ok=True)
    summ.to_csv(a.tables / "envelope_desvios_resumo.csv", index=False)
    runs[runs["monitored"]].sort_values(["Subset", "unit_id", "start"]).to_csv(a.tables / "envelope_desvios_casos.csv", index=False)
    cases = make_figure(r, runs, summ, a.figure)
    with pd.option_context("display.width", 250, "display.max_columns", 30):
        print(summ.round(2).T.to_string())
    print(cases)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/estudos_complementares.py
"""Estudos complementares da revisão de banca (Exp. A, B, C e D).

Exp. A — Incerteza conformal e custo da decisão com o modelo de RUL ajustado (FD004):
    Compara LightGBM (referência global w=10) contra XGBoost e CatBoost (ajustados por CV agrupada:
    padronização por regime + janela 50), cruzando Mondrian em 3 tercis de RUL prevista, CQR contínuo
    (Conformalized Quantile Regression sobre quantis 0,05/0,95 fora da amostra) e histerese na transição
    entre níveis (banda de 5 ciclos). Quantifica largura, cobertura, saltos "dente de serra",
    oscilações de nível por motor e custo relativo ao oráculo.

Exp. B — Ablação de exogeneidade estrita (P15 e epr) nos sensores virtuais e na MEWMA (FD001–FD004):
    Compara a especificação canônica contra a estritamente externa (removendo P15 e epr antes do ajuste
    e da previsão) no R² dentro do regime (ciclos 26–50, H=50, 13 sensores) e no poder de detecção a
    falso alarme igualado (H=25, 3 sementes).

Exp. C — Baselines multivariados com memória temporal vs. MEWMA (FD001–FD004):
    Avalia PCA-T²/SPE (com suavização EWMA λ=0,10) e Isolation Forest (com suavização EWMA λ=0,10)
    contra MEWMA e Isolation Forest pontual sobre as MESMAS séries cegas e o mesmo orçamento de falso
    alarme (0,2 por 100 ciclos), com bootstrap em cluster por motor receptor (4000 réplicas) e Holm.

Exp. D — Assinaturas de fim de vida e detecção nativa separadas por modo de falha (FD003 e FD004):
    Separa os modos HPC e fan apenas nos motores de treino com KMeans (K=2) sobre os desvios
    padronizados por regime de fim de vida (13 sensores), relata as assinaturas físicas/padronizadas
    por modo e estratifica a detecção nativa (T50 EWMA e MEWMA sintética, H=50) por modo nos motores
    de teste do holdout 75/25.

Execução:
    .venv/bin/python -m pdm_cmaps.experiments.estudos_complementares --study all --jobs 4
"""

from __future__ import annotations

import argparse
import json
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import GroupKFold

from pdm_cmaps.config import DEFAULT_RESULTS_DIR
from pdm_cmaps.data.features import CausalFeatureEngineer
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.data.splits import split_engine_units
from pdm_cmaps.detection.multivariate import MULTIVARIATE_SENSORS, MultivariateResidualMonitor
from pdm_cmaps.detection.regime import RegimeDecoupler
from pdm_cmaps.detection.virtual_sensor_metrics import virtual_sensor_r2
from pdm_cmaps.domain.rul import compute_piecewise_rul
from pdm_cmaps.domain.schema import (
    DEFAULT_HEALTHY_CYCLE_LIMIT,
    DEFAULT_RANDOM_STATE,
    DEFAULT_RUL_CAP,
    SETTING_COLUMNS,
)
from pdm_cmaps.protocol import TEST_RATIO
from pdm_cmaps.detection.virtual_sensor import VirtualSensor
from pdm_cmaps.experiments.fault_detection_study import (
    ADOPTED_H,
    BUDGETS,
    DET_MV,
    S_GRID,
    alarm_matrix,
    analyse_detector,
    bootstrap_series,
    build_series,
    iforest_ratio,
)
from pdm_cmaps.experiments.fault_injection import (
    INJECT_SENSORS,
    World,
    blind_columns,
    delta_z,
    inject,
    load_world,
    standardize,
)
from pdm_cmaps.experiments.poder_bootstrap import N_BOOT, SEED_BOOT, holm
from pdm_cmaps.experiments.selecao_hiperparametros import montar_dados, montar_modelo
from pdm_cmaps.modeling.registry import get_model_pool, grouped_oof
from pdm_cmaps.monitoring.alerts import (
    CRITICAL,
    CriticalityPolicy,
    build_alert_timeline,
    cost_table,
    debounce,
    evaluate_strategies,
    strategy_names,
)
from pdm_cmaps.monitoring.layer import MEWMA_CALIBRATION_MONITORING, _policy_grid, compute_fleet_alarms
from pdm_cmaps.monitoring.uncertainty import ConformalRULInterval, interval_metrics
from pdm_cmaps.pipeline.extra_models import fit_predict

OUT_DEFAULT = DEFAULT_RESULTS_DIR / "estudos_complementares"
TABLES_DEFAULT = DEFAULT_RESULTS_DIR.parent / "tcc-writing" / "tables"
PARAMS_DEFAULT = DEFAULT_RESULTS_DIR / "hiperparametros"
SUBSETS_ALL = ("FD001", "FD002", "FD003", "FD004")

DET_IFOREST_POINT = "Isolation Forest (pontual)"
DET_IFOREST_EWMA = "Isolation Forest (EWMA)"
DET_PCA_EWMA = "PCA-T²/SPE (EWMA)"
MEM_DETECTORS = (DET_MV, DET_PCA_EWMA, DET_IFOREST_EWMA, DET_IFOREST_POINT)

CONTRASTS_MEM = {
    "MEWMA com regime - PCA-T²/SPE (EWMA)": (DET_MV, DET_PCA_EWMA),
    "MEWMA com regime - Isolation Forest (EWMA)": (DET_MV, DET_IFOREST_EWMA),
    "Isolation Forest (EWMA) - Isolation Forest (pontual)": (DET_IFOREST_EWMA, DET_IFOREST_POINT),
}
FAMILIES_MEM = {
    label: (("degrau", 0.10), ("degrau", 0.05), ("rampa", 0.10))
    for label in CONTRASTS_MEM
}

ENDOGENOUS_EXOGENOUS_LEAK_CANDIDATES = ("P15", "epr")


def _slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


# ============================================================================
# Shared mathematical helpers (CQR, hysteresis, EWMA 1D/ND, PCA-T2/SPE EWMA)
# ============================================================================


def fit_cqr_from_quantiles(
    y_cal: np.ndarray,
    q_lo_cal: np.ndarray,
    q_hi_cal: np.ndarray,
    alpha: float = 0.10,
) -> float:
    """Finite-sample conformal correction Q_{1-alpha} for CQR (Romano et al., 2019)."""
    y = np.asarray(y_cal, dtype=float)
    lo = np.minimum(np.asarray(q_lo_cal, dtype=float), np.asarray(q_hi_cal, dtype=float))
    hi = np.maximum(np.asarray(q_lo_cal, dtype=float), np.asarray(q_hi_cal, dtype=float))
    if y.shape != lo.shape or y.shape != hi.shape or y.size < 20:
        raise ValueError("need matching calibration arrays with >= 20 points for CQR")
    scores = np.maximum(lo - y, y - hi)
    n = y.size
    level = min(1.0, float(np.ceil((n + 1) * (1.0 - alpha)) / n))
    return float(np.quantile(scores, level, method="higher"))


def apply_cqr_interval(
    q_lo: np.ndarray,
    q_hi: np.ndarray,
    correction: float,
    rul_cap: float = DEFAULT_RUL_CAP,
) -> tuple[np.ndarray, np.ndarray]:
    """Build non-crossing CQR prediction interval clipped to [0, rul_cap]."""
    lo_raw = np.minimum(np.asarray(q_lo, dtype=float), np.asarray(q_hi, dtype=float))
    hi_raw = np.maximum(np.asarray(q_lo, dtype=float), np.asarray(q_hi, dtype=float))
    lo = np.clip(lo_raw - correction, 0.0, rul_cap)
    hi = np.clip(hi_raw + correction, 0.0, rul_cap)
    return np.minimum(lo, hi), np.maximum(lo, hi)


def levels_with_hysteresis(
    rul: np.ndarray,
    policy: CriticalityPolicy,
    band: float = 5.0,
) -> np.ndarray:
    """Raw criticality levels with Schmitt-trigger de-escalation hysteresis (band cycles)."""
    r = np.asarray(rul, dtype=float)
    raw = policy.level_from_rul(r)
    thresh = (np.inf, float(policy.attention), float(policy.warning), float(policy.critical))
    out = np.empty(len(raw), dtype=int)
    if len(raw) == 0:
        return out
    cur = int(raw[0])
    for t in range(len(raw)):
        cand = int(raw[t])
        if cand > cur:
            cur = cand
        else:
            while cur > 0 and r[t] > thresh[cur] + band:
                cur -= 1
        out[t] = cur
    return out


def apply_hysteresis_to_timeline(
    frame: pd.DataFrame,
    policy: CriticalityPolicy,
    first_alarm_cycle: dict[int, int | None],
    extra_alarms: dict[str, dict[int, int | None]],
    band: float = 5.0,
) -> pd.DataFrame:
    """Build criticality timeline where interval-based levels use hysteresis on de-escalation."""
    parts = []
    for unit_id, eng in frame.sort_values(["unit_id", "cycle"]).groupby("unit_id", sort=False):
        eng = eng.copy()
        eng["anomaly_active"] = (
            eng["cycle"] >= first_alarm_cycle[int(unit_id)]
            if first_alarm_cycle.get(int(unit_id)) is not None
            else pd.Series(False, index=eng.index)
        )
        eng["level_true"] = policy.level_from_rul(eng["RUL_true"])
        raw_point = policy.level_from_rul(eng["RUL_pred"])
        raw_int = levels_with_hysteresis(eng["RUL_lower"].to_numpy(), policy, band=band)
        raw = {
            "point": raw_point,
            "interval": raw_int,
            "interval+anomaly": np.minimum(raw_int + eng["anomaly_active"].to_numpy(dtype=int) * (raw_int > 0), CRITICAL),
        }
        for name, alarm_map in extra_alarms.items():
            ac = alarm_map.get(int(unit_id))
            active = eng["cycle"] >= ac if ac is not None else pd.Series(False, index=eng.index)
            eng[f"{name}_active"] = active
            raw[f"interval+{name}"] = np.minimum(raw_int + np.asarray(active, dtype=int) * (raw_int > 0), CRITICAL)
        for name, lvls in raw.items():
            eng[f"level_{name}"] = debounce(lvls, policy.persistence)
        parts.append(eng)
    return pd.concat(parts, ignore_index=True)


def flicker_metrics(timeline: pd.DataFrame, strategy: str) -> dict[str, float]:
    """Compute per-engine level transitions, reversals, and conformal sawtooth jumps."""
    col = f"level_{strategy}"
    transitions, reversals, jumps = [], [], []
    for _, eng in timeline.groupby("unit_id", sort=False):
        eng = eng.sort_values("cycle")
        lvl = eng[col].to_numpy(dtype=int)
        lo = eng["RUL_lower"].to_numpy(dtype=float)
        pred = eng["RUL_pred"].to_numpy(dtype=float)
        if len(lvl) > 1:
            dl = np.diff(lvl)
            transitions.append(int((dl != 0).sum()))
            reversals.append(int((dl < 0).sum()))
            jumps.append(int(((np.abs(np.diff(lo)) >= 15.0) & (np.abs(np.diff(pred)) <= 5.0)).sum()))
        else:
            transitions.append(0)
            reversals.append(0)
            jumps.append(0)
    return {
        "Transitions_per_engine": float(np.mean(transitions)),
        "Reversals_per_engine": float(np.mean(reversals)),
        "Lower_jumps_ge15_per_engine": float(np.mean(jumps)),
    }


def ewma_smooth_matrix(z: np.ndarray, units: np.ndarray, mu: np.ndarray, lam: float = 0.10) -> np.ndarray:
    """Causal per-engine EWMA recursion standardized by finite-sample factor sqrt(lam/(2-lam)*(1-(1-lam)^(2t)))."""
    z = np.asarray(z, dtype=float)
    units = np.asarray(units)
    out = np.empty_like(z, dtype=float)
    factor_cache: dict[int, float] = {}
    start = 0
    for _, n in pd.Series(units).groupby(units, sort=False).size().items():
        w = np.zeros(z.shape[1], dtype=float)
        for t in range(n):
            w = lam * (z[start + t] - mu) + (1.0 - lam) * w
            if (t + 1) not in factor_cache:
                factor_cache[t + 1] = float(np.sqrt(lam / (2.0 - lam) * (1.0 - (1.0 - lam) ** (2 * (t + 1)))))
            out[start + t] = w / factor_cache[t + 1]
        start += n
    return out


class PCAEwmaMonitor:
    """PCA T² + SPE joint monitor evaluated on causally EWMA-smoothed residuals."""

    def __init__(self, lam: float = 0.10, variance_ratio: float = 0.90, alpha: float = 0.0027, random_state: int = 42):
        self.lam = float(lam)
        self.variance_ratio = float(variance_ratio)
        self.alpha = float(alpha)
        self.random_state = int(random_state)

    def fit(self, z_cal: np.ndarray, units_cal: np.ndarray) -> "PCAEwmaMonitor":
        zc = np.asarray(z_cal, dtype=float)
        p = zc.shape[1]
        self.mu_ = zc.mean(axis=0)
        pca = PCA(n_components=None, random_state=self.random_state).fit(zc - self.mu_)
        cum = np.cumsum(pca.explained_variance_ratio_)
        k = int(np.searchsorted(cum, self.variance_ratio) + 1)
        self.n_components_ = max(1, min(k, p - 1))
        self.components_ = pca.components_[: self.n_components_]  # (k, p)
        self.eigvals_ = np.maximum(pca.explained_variance_[: self.n_components_], 1e-8)

        w_cal = ewma_smooth_matrix(zc, units_cal, self.mu_, lam=self.lam)
        t2_cal, spe_cal = self._t2_and_spe(w_cal)
        self.tau_t2_ = max(float(np.quantile(t2_cal, 1.0 - self.alpha)), 1e-8)
        self.tau_spe_ = max(float(np.quantile(spe_cal, 1.0 - self.alpha)), 1e-8)
        comb = t2_cal / self.tau_t2_ + spe_cal / self.tau_spe_
        self.tau_comb_ = max(float(np.quantile(comb, 1.0 - self.alpha)), 1e-8)
        return self

    def _t2_and_spe(self, w: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        proj = w @ self.components_.T
        t2 = np.sum((proj ** 2) / self.eigvals_, axis=1)
        spe = np.clip(np.sum(w ** 2, axis=1) - np.sum(proj ** 2, axis=1), 0.0, None)
        return t2, spe

    def score_ratio(self, z: np.ndarray, units: np.ndarray) -> np.ndarray:
        w = ewma_smooth_matrix(z, units, self.mu_, lam=self.lam)
        t2, spe = self._t2_and_spe(w)
        return (t2 / self.tau_t2_ + spe / self.tau_spe_) / self.tau_comb_


class IForestEwmaMonitor:
    """Isolation Forest point anomaly scores smoothed per engine with causal EWMA."""

    def __init__(self, forest: IsolationForest, lam: float = 0.10, alpha: float = 0.0027):
        self.forest = forest
        self.lam = float(lam)
        self.alpha = float(alpha)

    def fit(self, z_cal: np.ndarray, units_cal: np.ndarray) -> "IForestEwmaMonitor":
        raw_cal = (-self.forest.score_samples(z_cal)).reshape(-1, 1)
        self.mu_ = raw_cal.mean(axis=0)
        smoothed = ewma_smooth_matrix(raw_cal, units_cal, self.mu_, lam=self.lam).ravel()
        self.ucl_ = max(float(np.quantile(smoothed, 1.0 - self.alpha)), 1e-8)
        return self

    def score_ratio(self, z: np.ndarray, units: np.ndarray) -> np.ndarray:
        raw = (-self.forest.score_samples(z)).reshape(-1, 1)
        smoothed = ewma_smooth_matrix(raw, units, self.mu_, lam=self.lam).ravel()
        return np.clip(smoothed / self.ucl_, 0.0, None)


# ============================================================================
# Exp. A: Tuned RUL + CQR + Hysteresis (FD004)
# ============================================================================


def _make_quantile_estimator(model_name: str, q: float, params: dict, seed: int, n_jobs: int):
    h = dict(params.get("hyper", {}))
    if model_name == "LightGBM (ref.)":
        from lightgbm import LGBMRegressor
        return LGBMRegressor(
            objective="quantile", alpha=q, n_estimators=80, max_depth=6,
            learning_rate=0.08, random_state=seed, n_jobs=n_jobs, verbose=-1,
        )
    if model_name == "XGBoost":
        from xgboost import XGBRegressor
        return XGBRegressor(
            objective="reg:quantileerror", quantile_alpha=q, tree_method="hist",
            n_estimators=h.get("n_estimators", 300), max_depth=h.get("max_depth", 6),
            learning_rate=h.get("learning_rate", 0.05), subsample=h.get("subsample", 0.8),
            colsample_bytree=h.get("colsample_bytree", 0.8), min_child_weight=h.get("min_child_weight", 3),
            reg_lambda=h.get("reg_lambda", 1.0), random_state=seed, n_jobs=n_jobs,
        )
    if model_name == "CatBoost":
        from catboost import CatBoostRegressor
        return CatBoostRegressor(
            loss_function=f"Quantile:alpha={q}",
            iterations=h.get("iterations", 400), depth=h.get("depth", 5),
            learning_rate=h.get("learning_rate", 0.08), l2_leaf_reg=h.get("l2_leaf_reg", 5.0),
            random_seed=seed, thread_count=n_jobs, verbose=0,
        )
    raise ValueError(f"Unsupported quantile estimator for {model_name!r}")


def _oof_and_test_predictions(
    model_label: str,
    model_key: str | None,
    params: dict | None,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    sensors: list[str],
    seed: int,
    n_jobs: int,
) -> dict:
    """Zero-leakage 5-fold GroupKFold OOF + full-train test predictions for point + 0.05/0.95 quantiles."""
    if model_label == "LightGBM (ref.)":
        engineer = CausalFeatureEngineer(sensors, windows=(5, 10)).fit(train_df, DEFAULT_HEALTHY_CYCLE_LIMIT)
        tr_fe = engineer.transform(train_df)
        te_fe = engineer.transform(test_df)
        base_features = [*sensors, *SETTING_COLUMNS, "cycle"]
        features = [*base_features, *[c for c in tr_fe.columns if c not in train_df.columns]]
        est_point = clone(get_model_pool(seed)["LightGBM"])
        if hasattr(est_point, "set_params"):
            est_point.set_params(n_jobs=n_jobs)
        _, oof_point = grouped_oof(tr_fe, features, estimator=est_point, n_splits=5, random_state=seed)
        _, oof_q05 = grouped_oof(tr_fe, features, estimator=_make_quantile_estimator("LightGBM (ref.)", 0.05, {}, seed, n_jobs), n_splits=5, random_state=seed)
        _, oof_q95 = grouped_oof(tr_fe, features, estimator=_make_quantile_estimator("LightGBM (ref.)", 0.95, {}, seed, n_jobs), n_splits=5, random_state=seed)

        X_tr, y_tr = tr_fe[features].to_numpy(dtype=float), tr_fe["RUL"].to_numpy(dtype=float)
        X_te = te_fe[features].to_numpy(dtype=float)
        pred_point = clone(est_point).fit(X_tr, y_tr).predict(X_te)
        pred_q05 = _make_quantile_estimator("LightGBM (ref.)", 0.05, {}, seed, n_jobs).fit(X_tr, y_tr).predict(X_te)
        pred_q95 = _make_quantile_estimator("LightGBM (ref.)", 0.95, {}, seed, n_jobs).fit(X_tr, y_tr).predict(X_te)
        return {
            "tr_meta": tr_fe[["unit_id", "cycle", "RUL"]].reset_index(drop=True),
            "te_meta": te_fe[["unit_id", "cycle", "RUL"]].reset_index(drop=True),
            "oof_point": np.clip(oof_point, 0.0, DEFAULT_RUL_CAP),
            "oof_q05": np.clip(oof_q05, 0.0, DEFAULT_RUL_CAP),
            "oof_q95": np.clip(oof_q95, 0.0, DEFAULT_RUL_CAP),
            "pred_point": np.clip(pred_point, 0.0, DEFAULT_RUL_CAP),
            "pred_q05": np.clip(pred_q05, 0.0, DEFAULT_RUL_CAP),
            "pred_q95": np.clip(pred_q95, 0.0, DEFAULT_RUL_CAP),
        }

    assert params is not None and model_key is not None
    oof_parts = []
    splitter = GroupKFold(n_splits=5)
    for tr_idx, va_idx in splitter.split(train_df, groups=train_df["unit_id"]):
        tr_fold = train_df.iloc[tr_idx].reset_index(drop=True)
        va_fold = train_df.iloc[va_idx].reset_index(drop=True)
        tr_f, va_f, cols_f = montar_dados(tr_fold, va_fold, sensors, 6, params)
        kind, est = montar_modelo(model_key, params, seed, device="cpu", n_jobs=n_jobs)
        pred_f = fit_predict(kind, est, tr_f, cols_f, va_f)

        X_f, y_f = tr_f[cols_f].to_numpy(dtype=float), tr_f["RUL"].to_numpy(dtype=float)
        Xv_f = va_f[cols_f].to_numpy(dtype=float)
        q05_f = _make_quantile_estimator(model_key, 0.05, params, seed, n_jobs).fit(X_f, y_f).predict(Xv_f)
        q95_f = _make_quantile_estimator(model_key, 0.95, params, seed, n_jobs).fit(X_f, y_f).predict(Xv_f)
        part = va_f[["unit_id", "cycle", "RUL"]].copy()
        part["oof_point"] = np.clip(pred_f, 0.0, DEFAULT_RUL_CAP)
        part["oof_q05"] = np.clip(q05_f, 0.0, DEFAULT_RUL_CAP)
        part["oof_q95"] = np.clip(q95_f, 0.0, DEFAULT_RUL_CAP)
        oof_parts.append(part)

    oof_df = pd.concat(oof_parts, ignore_index=True).sort_values(["unit_id", "cycle"]).reset_index(drop=True)
    tr_full, te_full, cols = montar_dados(train_df, test_df, sensors, 6, params)
    assert (oof_df["unit_id"].to_numpy() == tr_full["unit_id"].to_numpy()).all()
    assert (oof_df["cycle"].to_numpy() == tr_full["cycle"].to_numpy()).all()

    kind, est_full = montar_modelo(model_key, params, seed, device="cpu", n_jobs=n_jobs)
    pred_point = fit_predict(kind, est_full, tr_full, cols, te_full)
    X_tr, y_tr = tr_full[cols].to_numpy(dtype=float), tr_full["RUL"].to_numpy(dtype=float)
    X_te = te_full[cols].to_numpy(dtype=float)
    pred_q05 = _make_quantile_estimator(model_key, 0.05, params, seed, n_jobs).fit(X_tr, y_tr).predict(X_te)
    pred_q95 = _make_quantile_estimator(model_key, 0.95, params, seed, n_jobs).fit(X_tr, y_tr).predict(X_te)
    return {
        "tr_meta": oof_df[["unit_id", "cycle", "RUL"]].reset_index(drop=True),
        "te_meta": te_full[["unit_id", "cycle", "RUL"]].reset_index(drop=True),
        "oof_point": oof_df["oof_point"].to_numpy(dtype=float),
        "oof_q05": oof_df["oof_q05"].to_numpy(dtype=float),
        "oof_q95": oof_df["oof_q95"].to_numpy(dtype=float),
        "pred_point": np.clip(pred_point, 0.0, DEFAULT_RUL_CAP),
        "pred_q05": np.clip(pred_q05, 0.0, DEFAULT_RUL_CAP),
        "pred_q95": np.clip(pred_q95, 0.0, DEFAULT_RUL_CAP),
    }


def run_exp_a(seeds: list[int], out: Path, tables: Path, params_dir: Path, n_jobs: int) -> None:
    print("\n=== Exp. A: Modelo ajustado + CQR + histerese no monitoramento (FD004) ===", flush=True)
    loader = CMapssLoader()
    frame = loader.load("FD004")
    frame["RUL"] = compute_piecewise_rul(frame)
    params_xgb = json.loads((params_dir / "FD004" / "XGBoost" / "melhores.json").read_text())["params"]
    params_cb = json.loads((params_dir / "FD004" / "CatBoost" / "melhores.json").read_text())["params"]
    specs = (
        ("LightGBM (ref.)", None, None),
        ("XGBoost (ajustado)", "XGBoost", params_xgb),
        ("CatBoost (ajustado)", "CatBoost", params_cb),
    )
    base_policy = CriticalityPolicy()
    policies = _policy_grid(base_policy)

    summary_rows, cost_rows, traj_parts = [], [], []
    for seed in seeds:
        t0 = time.perf_counter()
        train_units, test_units = split_engine_units(frame, test_ratio=TEST_RATIO, random_state=seed)
        train_df = frame.loc[frame["unit_id"].isin(train_units)].sort_values(["unit_id", "cycle"]).reset_index(drop=True)
        test_df = frame.loc[frame["unit_id"].isin(test_units)].sort_values(["unit_id", "cycle"]).reset_index(drop=True)
        sensors = loader.informative_sensors(train_df)

        regimes = RegimeDecoupler(n_clusters=6, random_state=seed).fit(train_df)
        alarms, _, _ = compute_fleet_alarms(train_df, test_df, "FD004", seed, DEFAULT_HEALTHY_CYCLE_LIMIT)
        mv = MultivariateResidualMonitor(
            healthy_cycle_limit=DEFAULT_HEALTHY_CYCLE_LIMIT, random_state=seed, calibration=MEWMA_CALIBRATION_MONITORING,
        ).fit(regimes.transform(train_df))
        mv_alarms, _, _ = mv.alarms(mv.score(regimes.transform(test_df)))
        extra_alarms = {"anomaly_mv": mv_alarms}

        for model_label, model_key, params in specs:
            bundle = _oof_and_test_predictions(model_label, model_key, params, train_df, test_df, sensors, seed, n_jobs)
            y_cal = bundle["tr_meta"]["RUL"].to_numpy(dtype=float)
            y_true = bundle["te_meta"]["RUL"].to_numpy(dtype=float)
            pred = bundle["pred_point"]

            mon = ConformalRULInterval(alpha=0.10, n_bins=3).fit(y_cal, bundle["oof_point"])
            lo_mon, hi_mon = mon.predict_interval(pred)

            q_corr = fit_cqr_from_quantiles(y_cal, bundle["oof_q05"], bundle["oof_q95"], alpha=0.10)
            lo_cqr, hi_cqr = apply_cqr_interval(bundle["pred_q05"], bundle["pred_q95"], q_corr)

            variants = (
                ("Mondrian 3 tercis", lo_mon, hi_mon, False),
                ("Mondrian + histerese", lo_mon, hi_mon, True),
                ("CQR contínuo", lo_cqr, hi_cqr, False),
                ("CQR + histerese", lo_cqr, hi_cqr, True),
            )
            rmse = float(np.sqrt(np.mean((y_true - pred) ** 2)))
            mae = float(np.mean(np.abs(y_true - pred)))

            for method_name, lo, hi, use_hyst in variants:
                tl_in = pd.DataFrame({
                    "unit_id": bundle["te_meta"]["unit_id"].to_numpy(),
                    "cycle": bundle["te_meta"]["cycle"].to_numpy(),
                    "RUL_true": y_true, "RUL_pred": pred, "RUL_lower": lo, "RUL_upper": hi,
                })
                im = interval_metrics(y_true, lo, hi, critical_threshold=50.0)
                inside = (y_true >= lo) & (y_true <= hi)
                p10 = float(pd.Series(inside).groupby(tl_in["unit_id"]).mean().quantile(0.10))

                for pol_name, pol in policies.items():
                    tl = (
                        apply_hysteresis_to_timeline(tl_in, pol, alarms, extra_alarms, band=5.0)
                        if use_hyst
                        else build_alert_timeline(tl_in, pol, alarms, extra_alarms)
                    )
                    ct = cost_table(tl, trigger=CRITICAL).assign(
                        Subset="FD004", Seed=seed, Model=model_label, Conformal_method=method_name, Policy=pol_name,
                    )
                    cost_rows.append(ct)
                    if pol_name == "critical<=15":
                        ev_strat = evaluate_strategies(tl, pol).set_index("Strategy")
                        for strat in strategy_names(tl):
                            fm = flicker_metrics(tl, strat)
                            sr = ev_strat.loc[strat]
                            summary_rows.append({
                                "Subset": "FD004", "Seed": seed, "Model": model_label,
                                "Conformal_method": method_name, "Strategy": strat,
                                **im, "Per_engine_coverage_p10": p10, "RMSE": rmse, "MAE": mae,
                                "Alert_F1": float(sr["Alert_F1"]),
                                "Missed_critical_rate": float(sr["Missed_critical_rate"]),
                                "False_alarm_rate_normal": float(sr["False_alarm_rate_normal"]),
                                "Median_warning_lead_time": float(sr["Median_warning_lead_time"]),
                                **fm,
                            })
                        if seed == 42 and method_name in {"Mondrian 3 tercis", "CQR contínuo", "CQR + histerese"}:
                            traj_parts.append(
                                tl[["unit_id", "cycle", "RUL_true", "RUL_pred", "RUL_lower", "RUL_upper",
                                    "level_point", "level_interval", "level_interval+anomaly", "level_interval+anomaly_mv"]]
                                .assign(Model=model_label, Conformal_method=method_name)
                            )
        print(f"[Exp. A] seed={seed} concluída em {time.perf_counter() - t0:.1f} s", flush=True)

    raw_res = pd.DataFrame(summary_rows)
    raw_cost = pd.concat(cost_rows, ignore_index=True)
    exp_dir = out / "exp_a"
    exp_dir.mkdir(parents=True, exist_ok=True)
    raw_res.to_csv(exp_dir / "resumo_por_semente.csv", index=False)
    raw_cost.to_csv(exp_dir / "custos_por_semente.csv", index=False)
    if traj_parts:
        pd.concat(traj_parts, ignore_index=True).to_csv(out / "exp_a_trajetorias_seed42.csv", index=False)

    num_cols = [
        "coverage", "coverage_critical_zone", "mean_width", "mean_width_critical_zone",
        "Per_engine_coverage_p10", "RMSE", "MAE", "Alert_F1", "Missed_critical_rate",
        "False_alarm_rate_normal", "Median_warning_lead_time",
        "Transitions_per_engine", "Reversals_per_engine", "Lower_jumps_ge15_per_engine",
    ]
    g_res = raw_res.groupby(["Subset", "Model", "Conformal_method", "Strategy"])[num_cols]
    summ_res = g_res.mean().add_suffix("_mean").join(g_res.std().add_suffix("_std")).reset_index()
    summ_res["Runs"] = g_res.size().to_numpy()

    g_cost = raw_cost.groupby(["Subset", "Model", "Conformal_method", "Policy", "Strategy", "Cost_ratio", "Lead"])[
        ["Relative_to_oracle", "Failures", "Mean_unused_cycles"]
    ]
    summ_cost = g_cost.mean().add_suffix("_mean").join(g_cost.std().add_suffix("_std")).reset_index()
    summ_cost["Runs"] = g_cost.size().to_numpy()

    tables.mkdir(parents=True, exist_ok=True)
    summ_res.to_csv(tables / "conformal_modelo_ajustado_resumo.csv", index=False)
    summ_cost.to_csv(tables / "conformal_modelo_ajustado_custos.csv", index=False)

    ref_row = summ_res[
        (summ_res["Model"] == "LightGBM (ref.)")
        & (summ_res["Conformal_method"] == "Mondrian 3 tercis")
        & (summ_res["Strategy"] == "interval")
    ].iloc[0]
    ref_csv = tables / "monitoring_repeated_uncertainty_summary.csv"
    if ref_csv.exists():
        old_fd4 = pd.read_csv(ref_csv).query("Subset == 'FD004'").iloc[0]
        diff = abs(ref_row["RMSE_mean"] - old_fd4["RMSE_mean"])
        print(f"[Exp. A] conferência LightGBM ref FD004 RMSE={ref_row['RMSE_mean']:.4f} vs versionado={old_fd4['RMSE_mean']:.4f} (Δ={diff:.5f})", flush=True)
    with pd.option_context("display.width", 220, "display.max_columns", 20):
        print(summ_res[summ_res["Strategy"] == "interval"][[
            "Model", "Conformal_method", "RMSE_mean", "coverage_critical_zone_mean",
            "mean_width_critical_zone_mean", "Transitions_per_engine_mean", "Reversals_per_engine_mean",
            "Lower_jumps_ge15_per_engine_mean",
        ]].round(3).to_string(index=False), flush=True)


# ============================================================================
# Exp. B: Strict exogeneity ablation (removing P15 and epr)
# ============================================================================


def _strip_endogenous_exog(df: pd.DataFrame) -> pd.DataFrame:
    drop = [c for c in ENDOGENOUS_EXOGENOUS_LEAK_CANDIDATES if c in df.columns]
    return df.drop(columns=drop)


def _exp_b_r2_one(subset: str) -> list[dict]:
    loader = CMapssLoader()
    frame = loader.load(subset)
    n_regimes = 6 if subset in ("FD002", "FD004") else 1
    train_units, test_units = split_engine_units(frame, test_ratio=TEST_RATIO, random_state=42)
    tr_base = frame.loc[frame["unit_id"].isin(train_units)].sort_values(["unit_id", "cycle"]).reset_index(drop=True)
    te_base = frame.loc[frame["unit_id"].isin(test_units)].sort_values(["unit_id", "cycle"]).reset_index(drop=True)
    reg = RegimeDecoupler(n_clusters=n_regimes, random_state=42).fit(tr_base)
    tr_r, te_r = reg.transform(tr_base), reg.transform(te_base)

    rows = []
    for var, tr_in, te_in in (
        ("canonico", tr_r, te_r),
        ("estritamente_externo", _strip_endogenous_exog(tr_r), _strip_endogenous_exog(te_r)),
    ):
        mv = MultivariateResidualMonitor(
            healthy_cycle_limit=DEFAULT_HEALTHY_CYCLE_LIMIT, random_state=42, engine_offset=True,
        )
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mv.fit(tr_in)
        assert len(mv.sensors_) == 13, f"{subset} {var}: n_sensores={len(mv.sensors_)}"
        if var == "estritamente_externo":
            assert not any(c in mv.virtual_["T50"].feature_cols for c in ENDOGENOUS_EXOGENOUS_LEAK_CANDIDATES)
        parts = []
        for sensor, vs in mv.virtual_.items():
            p = vs.predict(te_in)
            parts.append(pd.DataFrame({
                "sensor": sensor, "model": "Extra Trees",
                "unit_id": p["unit_id"].to_numpy(), "cycle": p["cycle"].to_numpy(),
                "actual": p[sensor].to_numpy(dtype=float), "prediction": p["prediction"].to_numpy(dtype=float),
                "regime_id": p["regime_id"].to_numpy(),
            }))
        trace = pd.concat(parts, ignore_index=True)
        r2_tab = virtual_sensor_r2(trace, healthy_limit=DEFAULT_HEALTHY_CYCLE_LIMIT).set_index("sensor")
        rows.append({
            "Subset": subset, "variante": var, "n_sensores": len(mv.sensors_),
            "r2_bruto_mediana": float(r2_tab["r2_late_model"].median()),
            "r2_dentro_regime_mediana": float(r2_tab["r2_late_within_regime"].median()),
            "teto_ruido_mediana": float(r2_tab["noise_ceiling_late"].median()),
            "fracao_teto_mediana": float(r2_tab["frac_ceiling"].median()),
            "r2_T50_dentro_regime": float(r2_tab.loc["T50", "r2_late_within_regime"]),
            "r2_P30_dentro_regime": float(r2_tab.loc["P30", "r2_late_within_regime"]),
            "r2_BPR_dentro_regime": float(r2_tab.loc["BPR", "r2_late_within_regime"]),
        })
    return rows


def _exp_b_power_one(subset: str, seed: int) -> list[pd.DataFrame]:
    loader = CMapssLoader()
    world = load_world(subset, seed, ADOPTED_H, loader=loader)
    blind, meta = build_series(world, seed, 6, 2, None, False)
    tr = world.regimes.transform(world.train)
    te = world.regimes.transform(blind)
    out = []
    for var, tr_in, te_in in (
        ("canonico", tr, te),
        ("estritamente_externo", _strip_endogenous_exog(tr), _strip_endogenous_exog(te)),
    ):
        mv = MultivariateResidualMonitor(healthy_cycle_limit=world.H, random_state=seed, engine_offset=True)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mv.fit(tr_in)
            sc = mv.score(te_in)
        scores = pd.DataFrame({
            "blind_id": sc["unit_id"].to_numpy(), "cycle": sc["cycle"].to_numpy(),
            "ratio": (sc["t2"] / sc["ucl"]).to_numpy(), "gate": np.ones(len(sc), dtype=bool),
        })
        res = analyse_detector(f"MEWMA ({var})", scores, meta, ADOPTED_H, BUDGETS)["pareado"]
        out.append(res.assign(Subset=subset, Seed=seed, variante=var))
    return out


def run_exp_b(seeds: list[int], out: Path, tables: Path, n_jobs: int = 2) -> None:
    print("\n=== Exp. B: Ablação de exogeneidade estrita (sem P15 e epr) ===", flush=True)
    from joblib import Parallel, delayed

    r2_nested = Parallel(n_jobs=min(n_jobs, len(SUBSETS_ALL)))(delayed(_exp_b_r2_one)(s) for s in SUBSETS_ALL)
    r2_rows = [row for part in r2_nested for row in part]

    jobs = [(s, seed) for s in SUBSETS_ALL for seed in seeds]
    pod_nested = Parallel(n_jobs=n_jobs)(delayed(_exp_b_power_one)(s, seed) for s, seed in jobs)
    pod_runs = [df for part in pod_nested for df in part]

    r2_df = pd.DataFrame(r2_rows)
    pod_raw = pd.concat(pod_runs, ignore_index=True)
    g = pod_raw.groupby(["Subset", "variante", "kind", "magnitude"])
    pod_summ = g["power_pct"].agg(["mean", "std"]).rename(columns={"mean": "power_pct_mean", "std": "power_pct_std"})
    pod_summ["delay_med_mean"] = g["delay_med"].mean()
    pod_summ["far_ctrl_per100_mean"] = g["far_ctrl_per100"].mean()
    pod_summ["n_seeds"] = g.size()
    pod_summ = pod_summ.reset_index()

    exp_dir = out / "exp_b"
    exp_dir.mkdir(parents=True, exist_ok=True)
    pod_raw.to_csv(exp_dir / "poder_por_semente.csv", index=False)
    r2_df.to_csv(tables / "ablativo_exogeneidade_r2.csv", index=False)
    pod_summ.to_csv(tables / "ablativo_exogeneidade_poder.csv", index=False)
    print(r2_df.round(4).to_string(index=False), flush=True)
    deg = pod_summ[(pod_summ["kind"] == "degrau") & (pod_summ["magnitude"] == 0.10)]
    print(deg.round(3).to_string(index=False), flush=True)


# ============================================================================
# Exp. C: Memory multivariate baselines + cluster bootstrap + Holm
# ============================================================================


def _extract_residuals(mv: MultivariateResidualMonitor, frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return (unit_id, cycle, z_matrix[n, 13]) sorted by (unit_id, cycle)."""
    cols, index = {}, None
    for sensor, v in mv.virtual_.items():
        pred = v.predict(frame).sort_values(["unit_id", "cycle"]).reset_index(drop=True)
        cols[sensor] = mv.residual_z(v, pred)
        index = pred[["unit_id", "cycle"]]
    assert index is not None
    z = np.column_stack([cols[s] for s in mv.virtual_])
    return index["unit_id"].to_numpy(), index["cycle"].to_numpy(), z


def _bootstrap_memory_runs(npz_paths: list[Path], n_boot: int, boot_seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Cluster bootstrap over recipient engines using the same procedure as poder_bootstrap.py."""
    runs_data = []
    for path in npz_paths:
        npz = np.load(path)
        meta = pd.read_csv(path.with_name(path.stem + "_manifesto.csv"))
        subset, seed, H = str(meta["Subset"].iloc[0]), int(meta["Seed"].iloc[0]), int(meta["H"].iloc[0])
        midx = meta.set_index("blind_id")
        det = {}
        for name in MEM_DETECTORS:
            sl = _slug(name)
            ids, mat, last = npz[f"{sl}__ids"], npz[f"{sl}__mat"], npz[f"{sl}__last"]
            m = midx.loc[ids]
            ctrl = (m["role"] == "controle").to_numpy()
            has = ~np.isnan(mat)
            expo = np.where(has, mat - H, (last - H)[:, None])
            det[name] = {
                "ctrl_has": has[ctrl], "ctrl_expo": expo[ctrl], "fault_has": has[~ctrl],
                "ctrl_rec": m["recipient"].to_numpy()[ctrl], "fault_rec": m["recipient"].to_numpy()[~ctrl],
                "groups": m.loc[~ctrl, ["kind", "magnitude", "t0"]].reset_index(drop=True),
            }
        ref = det[DET_MV]
        recipients = np.unique(np.concatenate([ref["ctrl_rec"], ref["fault_rec"]]))
        code = {r: i for i, r in enumerate(recipients)}
        keys = ref["groups"].astype(str).agg("|".join, axis=1)
        group_keys = pd.unique(keys)
        gcode = {k: i for i, k in enumerate(group_keys)}
        group_table = ref["groups"].assign(_k=keys).drop_duplicates("_k").drop(columns="_k").reset_index(drop=True)
        for d in det.values():
            d["ctrl_code"] = np.array([code[r] for r in d["ctrl_rec"]])
            d["fault_code"] = np.array([code[r] for r in d["fault_rec"]])
            d["gcode"] = np.array([gcode[k] for k in d["groups"].astype(str).agg("|".join, axis=1)])
        runs_data.append({
            "subset": subset, "seed": seed, "H": H, "det": det,
            "recipients": recipients, "group_keys": group_keys, "group_table": group_table,
        })

    def _run_power(r: dict, name: str, counts: np.ndarray, budget: float = BUDGETS[0]) -> np.ndarray:
        d = r["det"][name]
        wc = counts[d["ctrl_code"]].astype(float)
        events, exposure = wc @ d["ctrl_has"], wc @ d["ctrl_expo"]
        with np.errstate(divide="ignore", invalid="ignore"):
            far = 100.0 * events / exposure
        ok = np.flatnonzero(far <= budget)
        n_groups = len(r["group_keys"])
        if not ok.size:
            return np.full(n_groups, np.nan)
        wf = counts[d["fault_code"]].astype(float)
        detected = d["fault_has"][:, ok[0]].astype(float)
        with np.errstate(divide="ignore", invalid="ignore"):
            return 100.0 * np.bincount(d["gcode"], wf * detected, n_groups) / np.bincount(d["gcode"], wf, n_groups)

    rng = np.random.default_rng(boot_seed)
    by_subset: dict[str, list[dict]] = {}
    for r in runs_data:
        by_subset.setdefault(r["subset"], []).append(r)

    power_rows, contrast_rows = [], []
    didx = {n: i for i, n in enumerate(MEM_DETECTORS)}
    for subset, rs in sorted(by_subset.items()):
        ref_keys = list(rs[0]["group_keys"])
        assert all(list(r["group_keys"]) == ref_keys for r in rs)
        boot = np.full((n_boot, len(MEM_DETECTORS), len(ref_keys)), np.nan)
        for b in range(n_boot):
            per_seed = []
            for r in rs:
                counts = np.bincount(rng.integers(0, len(r["recipients"]), len(r["recipients"])), minlength=len(r["recipients"]))
                per_seed.append(np.vstack([_run_power(r, n, counts) for n in MEM_DETECTORS]))
            with np.errstate(all="ignore"):
                boot[b] = np.nanmean(np.stack(per_seed), axis=0)
        table = rs[0]["group_table"]
        pts = {n: np.nanmean(np.vstack([_run_power(r, n, np.ones(len(r["recipients"]))) for r in rs]), axis=0) for n in MEM_DETECTORS}
        for g, cell in table.iterrows():
            for n in MEM_DETECTORS:
                lo, hi = np.nanpercentile(boot[:, didx[n], g], [2.5, 97.5])
                power_rows.append({
                    "Subset": subset, "Detector": n, "kind": cell["kind"],
                    "magnitude": cell["magnitude"], "t0": cell["t0"],
                    "poder_pct": float(pts[n][g]), "ic_lo": float(lo), "ic_hi": float(hi), "n_seeds": len(rs),
                })
            for label, (a, c) in CONTRASTS_MEM.items():
                d = boot[:, didx[a], g] - boot[:, didx[c], g]
                d = d[np.isfinite(d)]
                if not d.size:
                    continue
                p = float(min(1.0, 2.0 * min((d <= 0).mean(), (d >= 0).mean())))
                p = max(p, 1.0 / (len(d) + 1))
                contrast_rows.append({
                    "Subset": subset, "Contraste": label, "kind": cell["kind"],
                    "magnitude": cell["magnitude"], "t0": cell["t0"],
                    "diferenca_pp": float(pts[a][g] - pts[c][g]),
                    "ic_lo": float(np.percentile(d, 2.5)), "ic_hi": float(np.percentile(d, 97.5)),
                    "p_boot": p,
                })

    power_df, contr_df = pd.DataFrame(power_rows), pd.DataFrame(contrast_rows)
    contr_df["familia"] = False
    contr_df["p_holm"] = np.nan
    for label, cells in FAMILIES_MEM.items():
        sel = (
            contr_df["Contraste"].eq(label)
            & contr_df[["kind", "magnitude"]].apply(tuple, axis=1).isin(list(cells))
            & (contr_df["t0"] == ADOPTED_H + 10)
        )
        contr_df.loc[sel, "familia"] = True
        if sel.any():
            contr_df.loc[sel, "p_holm"] = holm(contr_df.loc[sel, "p_boot"].to_numpy())
    return power_df, contr_df


def _exp_c_one(subset: str, seed: int, alarm_dir: Path) -> tuple[list[pd.DataFrame], Path]:
    t0 = time.perf_counter()
    world = load_world(subset, seed, ADOPTED_H, loader=CMapssLoader())
    blind, meta = build_series(world, seed, 6, 2, None, False)
    tr, te = world.regimes.transform(world.train), world.regimes.transform(blind)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mv = MultivariateResidualMonitor(healthy_cycle_limit=world.H, random_state=seed, engine_offset=True).fit(tr)
        uid_te, cyc_te, z_te = _extract_residuals(mv, te)

    mv_ratio = mv._t2(z_te, uid_te) / mv.ucl_
    z_cal_df = mv.calibration_z_
    z_cal = z_cal_df.to_numpy()
    units_cal = z_cal_df.index.get_level_values("unit_id").to_numpy()

    pca_mon = PCAEwmaMonitor(lam=0.10, variance_ratio=0.90, alpha=0.0027, random_state=seed).fit(z_cal, units_cal)
    pca_ratio = pca_mon.score_ratio(z_te, uid_te)

    forest = IsolationForest(n_estimators=200, random_state=seed, n_jobs=1).fit(z_cal)
    if_pt_ratio = iforest_ratio(forest, z_cal, z_te)
    if_ewma_ratio = IForestEwmaMonitor(forest, lam=0.10, alpha=0.0027).fit(z_cal, units_cal).score_ratio(z_te, uid_te)

    ones = np.ones(len(uid_te), dtype=bool)
    det_scores = {
        DET_MV: pd.DataFrame({"blind_id": uid_te, "cycle": cyc_te, "ratio": mv_ratio, "gate": ones}),
        DET_PCA_EWMA: pd.DataFrame({"blind_id": uid_te, "cycle": cyc_te, "ratio": pca_ratio, "gate": ones}),
        DET_IFOREST_EWMA: pd.DataFrame({"blind_id": uid_te, "cycle": cyc_te, "ratio": if_ewma_ratio, "gate": ones}),
        DET_IFOREST_POINT: pd.DataFrame({"blind_id": uid_te, "cycle": cyc_te, "ratio": if_pt_ratio, "gate": ones}),
    }

    midx = meta.set_index("blind_id")
    arrays: dict[str, np.ndarray] = {}
    pareado = []
    for name, sc in det_scores.items():
        pareado.append(analyse_detector(name, sc, meta, ADOPTED_H, BUDGETS)["pareado"].assign(Subset=subset, Seed=seed))
        ids, mat, last = alarm_matrix(sc, np.maximum(midx["t0"], ADOPTED_H + 1))
        sl = _slug(name)
        arrays[f"{sl}__ids"], arrays[f"{sl}__mat"], arrays[f"{sl}__last"] = ids, mat.astype(np.float32), last

    npz_path = alarm_dir / f"{subset}_seed{seed}_H{ADOPTED_H}.npz"
    meta.assign(Subset=subset, Seed=seed, H=ADOPTED_H).to_csv(npz_path.with_name(npz_path.stem + "_manifesto.csv"), index=False)
    np.savez_compressed(npz_path, **arrays)
    print(f"[Exp. C] {subset} seed={seed} ({time.perf_counter() - t0:.1f} s)", flush=True)
    return pareado, npz_path


def run_exp_c(seeds: list[int], out: Path, tables: Path, n_boot: int, n_jobs: int = 2) -> None:
    print("\n=== Exp. C: Baselines multivariados com memória temporal (PCA-T²/SPE EWMA, IForest EWMA) ===", flush=True)
    from joblib import Parallel, delayed

    exp_dir = out / "exp_c"
    alarm_dir = exp_dir / "alarmes"
    alarm_dir.mkdir(parents=True, exist_ok=True)
    jobs = [(s, seed) for s in SUBSETS_ALL for seed in seeds]
    res_list = Parallel(n_jobs=n_jobs)(delayed(_exp_c_one)(s, seed, alarm_dir) for s, seed in jobs)
    pareado_parts = [df for par, _ in res_list for df in par]
    npz_paths = [p for _, p in res_list]

    par_all = pd.concat(pareado_parts, ignore_index=True)
    g = par_all.groupby(["Subset", "Detector", "kind", "magnitude"])
    comp = g["power_pct"].agg(["mean", "std"]).rename(columns={"mean": "power_pct_mean", "std": "power_pct_std"})
    comp["delay_med_mean"] = g["delay_med"].mean()
    comp["far_ctrl_per100_mean"] = g["far_ctrl_per100"].mean()
    comp["n_seeds"] = g.size()
    comp = comp.reset_index()

    power_boot, contr_boot = _bootstrap_memory_runs(npz_paths, n_boot=n_boot, boot_seed=SEED_BOOT)
    comp.to_csv(tables / "detectores_memoria_comparacao.csv", index=False)
    power_boot.round(3).to_csv(exp_dir / "poder_memoria_ic.csv", index=False)
    contr_boot.round(4).to_csv(tables / "contrastes_memoria_ic.csv", index=False)

    ver_cmp = tables / "detectores_comparacao.csv"
    if ver_cmp.exists():
        old_mv = pd.read_csv(ver_cmp).query("Detector == @DET_MV")
        new_mv = comp.query("Detector == @DET_MV")
        merged = new_mv.merge(old_mv, on=["Subset", "Detector", "kind", "magnitude"], suffixes=("_new", "_old"))
        print(f"[Exp. C] conferência MEWMA vs versionado: max |Δpoder| = {(merged['power_pct_mean_new'] - merged['power_pct_mean_old']).abs().max():.6f} pp", flush=True)

    with pd.option_context("display.width", 240, "display.max_columns", 20):
        print(contr_boot[contr_boot["familia"]].round(4).to_string(index=False), flush=True)


# ============================================================================
# Exp. D: Separated EOL failure modes & native detection per mode (FD003/4)
# ============================================================================


def _engine_eol_vectors(
    frame: pd.DataFrame,
    regimes: RegimeDecoupler,
    sensors: tuple[str, ...],
    healthy_limit: int = DEFAULT_HEALTHY_CYCLE_LIMIT,
    tail_cycles: int = 10,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compute per-engine standardized (and regime-adjusted physical) EOL shift vs. first healthy cycles."""
    rf = regimes.transform(frame)
    norm_df = regimes.standardize_sensors(rf, list(sensors), suffix="_z")
    reg_healthy = rf.loc[rf["cycle"] <= healthy_limit].groupby("regime_id")[list(sensors)].mean()

    z_rows, phys_rows = [], []
    for unit_id, eng in norm_df.sort_values("cycle").groupby("unit_id", sort=True):
        early = eng.loc[eng["cycle"] <= healthy_limit]
        tail = eng.tail(tail_cycles)
        # Regime-adjusted physical delta: compare each tail cycle against engine's healthy mean in same regime (or global regime healthy mean)
        eng_reg_means = early.groupby("regime_id")[list(sensors)].mean()

        zr = {"unit_id": int(unit_id)}
        pr = {"unit_id": int(unit_id)}
        for s in sensors:
            zr[s] = float(tail[f"{s}_z"].mean() - early[f"{s}_z"].mean())
            base_vals = [
                float(eng_reg_means.loc[r, s]) if (r in eng_reg_means.index) else float(reg_healthy.loc[r, s])
                for r in tail["regime_id"].to_numpy(dtype=int)
            ]
            pr[s] = float(np.mean(tail[s].to_numpy(dtype=float) - np.asarray(base_vals)))
        z_rows.append(zr)
        phys_rows.append(pr)
    return pd.DataFrame(z_rows).set_index("unit_id"), pd.DataFrame(phys_rows).set_index("unit_id")


def fit_failure_mode_kmeans(
    train_df: pd.DataFrame,
    n_regimes: int,
    sensors: tuple[str, ...] = MULTIVARIATE_SENSORS,
    seed: int = 42,
    healthy_limit: int = DEFAULT_HEALTHY_CYCLE_LIMIT,
) -> tuple[RegimeDecoupler, KMeans, dict[int, str], pd.DataFrame, pd.DataFrame]:
    """Fit KMeans(K=2) ONLY on training engines' standardized EOL shifts; label higher-ΔP30 cluster as Fan."""
    reg = RegimeDecoupler(n_clusters=n_regimes, random_state=seed).fit(train_df)
    tr_r = reg.transform(train_df)
    reg.fit_sensor_scaler(tr_r.loc[tr_r["cycle"] <= healthy_limit], list(sensors))
    z_tr, phys_tr = _engine_eol_vectors(train_df, reg, sensors, healthy_limit=healthy_limit)

    km = KMeans(n_clusters=2, n_init=20, random_state=seed).fit(z_tr[list(sensors)].to_numpy(dtype=float))
    c0_p30 = float(z_tr.loc[km.labels_ == 0, "P30"].mean())
    c1_p30 = float(z_tr.loc[km.labels_ == 1, "P30"].mean())
    fan_cluster = 1 if c1_p30 >= c0_p30 else 0
    label_map = {fan_cluster: "Fan", 1 - fan_cluster: "HPC"}
    z_tr = z_tr.assign(modo=[label_map[int(c)] for c in km.labels_])
    phys_tr = phys_tr.assign(modo=z_tr["modo"])
    return reg, km, label_map, z_tr, phys_tr


def _world_mode_signatures(world: World, seed: int) -> dict[str, np.ndarray]:
    """Train-only KMeans(K=2) EOL signatures (media, HPC, Fan) in world.signature units."""
    tr = world.train
    z_tr = standardize(world, tr)
    h_mask = (tr["cycle"] <= world.H).to_numpy()
    e_mask = (tr["rul_true"] <= 10).to_numpy()
    unit_ids = sorted(tr["unit_id"].unique())
    vecs = []
    for u in unit_ids:
        um = (tr["unit_id"] == u).to_numpy()
        vecs.append(z_tr[um & e_mask].mean(axis=0) - z_tr[um & h_mask].mean(axis=0))
    V = np.vstack(vecs)
    km = KMeans(n_clusters=2, n_init=20, random_state=seed).fit(V)
    p30_idx = list(INJECT_SENSORS).index("P30")
    fan_cluster = 1 if V[km.labels_ == 1, p30_idx].mean() >= V[km.labels_ == 0, p30_idx].mean() else 0
    return {
        "media": world.signature,
        "HPC": V[km.labels_ == (1 - fan_cluster)].mean(axis=0),
        "Fan": V[km.labels_ == fan_cluster].mean(axis=0),
    }


def _exp_d_injection_modes(seeds: list[int], tables: Path, exp_dir: Path) -> None:
    """Compare degrau (0.05, 0.10, 0.25) power using mixed vs pure-HPC vs pure-Fan signatures."""
    from dataclasses import replace

    loader = CMapssLoader()
    mags = (0.05, 0.10, 0.25)
    t0_inj = ADOPTED_H + 10
    horizon = t0_inj + 80
    rows = []

    for subset in ("FD003", "FD004"):
        for seed in seeds:
            world = load_world(subset, seed, ADOPTED_H, loader=loader)
            sigs = _world_mode_signatures(world, seed)
            rng = np.random.default_rng(seed)
            engines = {int(u): g.sort_values("cycle") for u, g in world.test.groupby("unit_id")}
            recipients = [u for u in sorted(engines) if (engines[u]["rul_true"] >= DEFAULT_RUL_CAP).sum() >= 30]

            ctrl_frames, ctrl_meta, bid = [], [], 0
            for u in recipients:
                for _ in range(6):
                    base = bootstrap_series(engines[u], 300, rng)
                    bid += 1
                    ctrl_frames.append(blind_columns(base, bid))
                    ctrl_meta.append({"blind_id": bid, "recipient": u, "role": "controle", "kind": "", "magnitude": np.nan, "t0": 0, "T": 300})
            ctrl_df = pd.concat(ctrl_frames, ignore_index=True)

            tr = world.regimes.transform(world.train)
            te_ctrl = world.regimes.transform(ctrl_df)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                vs_t50 = VirtualSensor(
                    target_sensor="T50", mode="exogenous", ewma_alpha=0.10,
                    min_consecutive_breaches=2, healthy_cycle_limit=world.H, random_state=seed,
                ).fit(tr)
                mv = MultivariateResidualMonitor(healthy_cycle_limit=world.H, random_state=seed, engine_offset=True).fit(tr)
                t50_ctrl_tr, _, _ = vs_t50.evaluate(tr, te_ctrl)
                mv_ctrl_sc = mv.score(te_ctrl)

            for sig_name, sig_vec in sigs.items():
                w_sig = replace(world, signature=sig_vec)
                sig_rng = np.random.default_rng(seed + 1000)
                f_frames, f_meta = [], []
                fbid = bid
                for u in recipients:
                    for _ in range(2):
                        base = bootstrap_series(engines[u], horizon, sig_rng)
                        for m in mags:
                            fbid += 1
                            f_frames.append(blind_columns(inject(w_sig, base, delta_z(w_sig, base, "degrau", m, t0_inj, sig_rng)), fbid))
                            f_meta.append({"blind_id": fbid, "recipient": u, "role": "falha", "kind": "degrau", "magnitude": m, "t0": t0_inj, "T": horizon})
                meta_all = pd.DataFrame([*ctrl_meta, *f_meta])
                f_df = world.regimes.transform(pd.concat(f_frames, ignore_index=True))

                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    t50_f_tr, _, _ = vs_t50.evaluate(tr, f_df)
                    mv_f_sc = mv.score(f_df)

                t50_ctrl = t50_ctrl_tr.sort_values(["unit_id", "cycle"])
                t50_f = t50_f_tr.sort_values(["unit_id", "cycle"])
                sc_t50 = pd.concat([
                    pd.DataFrame({"blind_id": t50_ctrl["unit_id"].to_numpy(), "cycle": t50_ctrl["cycle"].to_numpy(), "ratio": (t50_ctrl["ewma_abs_z"] / t50_ctrl["ewma_ucl"]).to_numpy(), "gate": True}),
                    pd.DataFrame({"blind_id": t50_f["unit_id"].to_numpy(), "cycle": t50_f["cycle"].to_numpy(), "ratio": (t50_f["ewma_abs_z"] / t50_f["ewma_ucl"]).to_numpy(), "gate": True}),
                ], ignore_index=True)
                sc_mv = pd.concat([
                    pd.DataFrame({"blind_id": mv_ctrl_sc["unit_id"].to_numpy(), "cycle": mv_ctrl_sc["cycle"].to_numpy(), "ratio": (mv_ctrl_sc["t2"] / mv_ctrl_sc["ucl"]).to_numpy(), "gate": True}),
                    pd.DataFrame({"blind_id": mv_f_sc["unit_id"].to_numpy(), "cycle": mv_f_sc["cycle"].to_numpy(), "ratio": (mv_f_sc["t2"] / mv_f_sc["ucl"]).to_numpy(), "gate": True}),
                ], ignore_index=True)

                for dname, sc_d in (("T50 EWMA (com regime, sem portão)", sc_t50), (DET_MV, sc_mv)):
                    par = analyse_detector(dname, sc_d, meta_all, ADOPTED_H, BUDGETS)["pareado"]
                    rows.append(par.assign(Subset=subset, Seed=seed, assinatura=sig_name))

    inj_raw = pd.concat(rows, ignore_index=True)
    inj_raw.to_csv(exp_dir / "injecao_assinatura_por_semente.csv", index=False)
    g = inj_raw.groupby(["Subset", "assinatura", "Detector", "magnitude"])
    inj_summ = g["power_pct"].agg(["mean", "std"]).rename(columns={"mean": "power_pct_mean", "std": "power_pct_std"})
    inj_summ["delay_med_mean"] = g["delay_med"].mean()
    inj_summ["far_ctrl_per100_mean"] = g["far_ctrl_per100"].mean()
    inj_summ["n_seeds"] = g.size()
    inj_summ = inj_summ.reset_index()
    inj_summ.to_csv(tables / "injecao_assinatura_por_modo.csv", index=False)
    print(inj_summ[inj_summ["magnitude"] == 0.10].round(2).to_string(index=False), flush=True)


def run_exp_d(seeds: list[int], out: Path, tables: Path) -> None:
    print("\n=== Exp. D: Modos de falha separados (HPC vs. Fan) no FD003 e FD004 ===", flush=True)
    loader = CMapssLoader()
    sig_rows, det_rows = [], []

    for subset in ("FD003", "FD004"):
        frame = loader.load(subset)
        n_regimes = 6 if subset == "FD004" else 1
        _, _, _, z_all, phys_all = fit_failure_mode_kmeans(frame, n_regimes, MULTIVARIATE_SENSORS, seed=42)
        for modo in ("HPC", "Fan"):
            zm = z_all.loc[z_all["modo"] == modo]
            pm = phys_all.loc[phys_all["modo"] == modo]
            for s in MULTIVARIATE_SENSORS:
                sig_rows.append({
                    "Subset": subset, "modo": modo, "sensor": s, "n_motores": int(len(zm)),
                    "delta_z_mean": float(zm[s].mean()), "delta_z_std": float(zm[s].std()),
                    "delta_fisico_mean": float(pm[s].mean()), "delta_fisico_std": float(pm[s].std()),
                    "sinal_dominante": "+" if float(zm[s].mean()) >= 0 else "-",
                })

        for seed in seeds:
            train_units, test_units = split_engine_units(frame, test_ratio=TEST_RATIO, random_state=seed)
            train_df = frame.loc[frame["unit_id"].isin(train_units)].sort_values(["unit_id", "cycle"]).reset_index(drop=True)
            test_df = frame.loc[frame["unit_id"].isin(test_units)].sort_values(["unit_id", "cycle"]).reset_index(drop=True)
            reg, km, label_map, _, _ = fit_failure_mode_kmeans(train_df, n_regimes, MULTIVARIATE_SENSORS, seed=seed)
            z_te, _ = _engine_eol_vectors(test_df, reg, MULTIVARIATE_SENSORS, healthy_limit=DEFAULT_HEALTHY_CYCLE_LIMIT)
            te_modes = pd.Series(
                [label_map[int(c)] for c in km.predict(z_te[list(MULTIVARIATE_SENSORS)].to_numpy(dtype=float))],
                index=z_te.index, name="modo",
            )

            _, per_t50 = _t50_per_engine(train_df, test_df, subset, seed)
            mv = MultivariateResidualMonitor(
                healthy_cycle_limit=DEFAULT_HEALTHY_CYCLE_LIMIT, random_state=seed, calibration=MEWMA_CALIBRATION_MONITORING,
            )
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                mv.fit(reg.transform(train_df))
                _, per_mv, _ = mv.alarms(mv.score(reg.transform(test_df)))

            for det_name, per in (("T50 EWMA (com regime)", per_t50), ("MEWMA (13 sensores, sintético)", per_mv)):
                merged = per.join(te_modes, on="unit_id")
                for modo in ("HPC", "Fan"):
                    g = merged.loc[merged["modo"] == modo]
                    det = g.loc[g["detected"]]
                    det_rows.append({
                        "Subset": subset, "Seed": seed, "modo": modo, "Detector": det_name,
                        "n_motores": int(len(g)),
                        "deteccao_pct": float(g["detected"].mean() * 100.0) if len(g) else np.nan,
                        "antecedencia_mediana": float(g["lead_time_cycles"].median()) if len(g) else np.nan,
                        "antecedencia_mediana_detectados": float(det["lead_time_cycles"].median()) if len(det) else np.nan,
                        "falso_alarme_saudavel_pct": float(g["healthy_far"].mean() * 100.0) if len(g) else np.nan,
                    })

    sig_df = pd.DataFrame(sig_rows)
    det_raw = pd.DataFrame(det_rows)
    g = det_raw.groupby(["Subset", "modo", "Detector"])
    cols_d = ["deteccao_pct", "antecedencia_mediana", "antecedencia_mediana_detectados", "falso_alarme_saudavel_pct"]
    det_summ = g[cols_d].mean().reset_index()
    g_std = g[["deteccao_pct", "antecedencia_mediana"]].std()
    for col in ("deteccao_pct", "antecedencia_mediana"):
        det_summ[f"{col}_std"] = g_std[col].to_numpy()
    det_summ["n_motores_total"] = g["n_motores"].sum().to_numpy()
    det_summ["n_seeds"] = g.size().to_numpy()

    exp_dir = out / "exp_d"
    exp_dir.mkdir(parents=True, exist_ok=True)
    det_raw.to_csv(exp_dir / "deteccao_nativa_por_semente.csv", index=False)
    sig_df.to_csv(tables / "assinaturas_modo_falha_eol.csv", index=False)
    det_summ.to_csv(tables / "deteccao_nativa_por_modo.csv", index=False)

    print(sig_df[sig_df["sensor"].isin(["T50", "P30", "phi", "BPR", "Nf", "P15"])].round(3).to_string(index=False), flush=True)
    print(det_summ.round(2).to_string(index=False), flush=True)
    _exp_d_injection_modes(seeds, tables, exp_dir)


def _t50_per_engine(train_df: pd.DataFrame, test_df: pd.DataFrame, subset: str, seed: int) -> tuple[dict, pd.DataFrame]:
    n_regimes = 6 if subset in ("FD002", "FD004") else 1
    regimes = RegimeDecoupler(n_clusters=n_regimes, random_state=seed).fit(train_df)
    vs = VirtualSensor(
        target_sensor="T50", mode="exogenous", ewma_alpha=0.10, min_consecutive_breaches=2,
        healthy_cycle_limit=DEFAULT_HEALTHY_CYCLE_LIMIT, random_state=seed,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _, per_engine, _ = vs.evaluate(regimes.transform(train_df), regimes.transform(test_df))
    return {int(r.unit_id): (None if pd.isna(r.first_alarm_cycle) else int(r.first_alarm_cycle)) for r in per_engine.itertuples()}, per_engine.copy()


# ============================================================================
# CLI
# ============================================================================


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--study", choices=["all", "a", "b", "c", "d"], default="all")
    ap.add_argument("--seeds", nargs="+", type=int, default=[42, 7, 123])
    ap.add_argument("--jobs", type=int, default=2)
    ap.add_argument("--boot", type=int, default=N_BOOT)
    ap.add_argument("--output-dir", type=Path, default=OUT_DEFAULT)
    ap.add_argument("--tables-dir", type=Path, default=TABLES_DEFAULT)
    ap.add_argument("--params-dir", type=Path, default=PARAMS_DEFAULT)
    a = ap.parse_args(argv)

    a.output_dir.mkdir(parents=True, exist_ok=True)
    a.tables_dir.mkdir(parents=True, exist_ok=True)
    if a.study in ("all", "a"):
        run_exp_a(list(a.seeds), a.output_dir, a.tables_dir, a.params_dir, a.jobs)
    if a.study in ("all", "b"):
        run_exp_b(list(a.seeds), a.output_dir, a.tables_dir, a.jobs)
    if a.study in ("all", "c"):
        run_exp_c(list(a.seeds), a.output_dir, a.tables_dir, a.boot, a.jobs)
    if a.study in ("all", "d"):
        run_exp_d(list(a.seeds), a.output_dir, a.tables_dir)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/exploracao_r2.py
"""Exploração do R² dos sensores do FD001 e do FD003 (branch exp/sensor-virtual-r2-fd001-fd003).

PERGUNTA: que técnicas e modelos levam o R² de prever um sensor perto de 0,90 (ou acima) nos subconjuntos de regime único,
e a que custo para a detecção? O teto do R² é medido antes (`teto_ruido.csv`): sem entrada que acompanhe o ruído do próprio
ciclo, R² ≤ 1 − σ²_ruído/Var(y), e na janela saudável esse teto já fica abaixo de 0,90 na maioria dos sensores.

DUAS TRILHAS, com rótulos e tabelas separados (decisão do autor em 2026-11-18):
  A  exógeno, janela saudável: o sensor virtual VÁLIDO para detecção (entradas = ajustes e ambiente; treino só nos ciclos
     ≤ H). R² na janela tardia (ciclos 26 a 50) e detecção/falso alarme ao lado. Compara o modelo com o teto.
  B1 cross-sensor treinado só na janela saudável (`VirtualSensor(mode="cross_sensor")`): ainda é detector (o resíduo sobe
     quando a relação entre sensores se rompe). R² em todos os ciclos dos motores de teste + detecção.
  B1s igual à B1, mas SEM o sensor "gêmeo" nas entradas (Nf<->NRf, Nc<->NRc são o mesmo sinal corrigido pela temperatura de
     entrada; sem tirá-lo, o R² alto pode ser tautologia e não relação entre sensores). Checagem de validade da B1.
  B2 cross-sensor treinado no CICLO DE VIDA inteiro: regressão pura. O modelo VÊ a degradação, então o resíduo some e NÃO
     serve como detector; só o R² faz sentido (por isso não há coluna de detecção).

[LEAKAGE] divisão por MOTOR (split_engine_units), nunca por linha; o baseline do motor é a média dos primeiros W ciclos dele
(causal); R² de B2 é reportado em todos os ciclos e só depois do ciclo W (onde o baseline já existe). Nenhum atraso/média
móvel do próprio alvo entra como entrada (isso viraria um modelo autorregressivo e mataria a detecção).
[ASSUMPTION] hiperparâmetros fixos e pequenos, sem ajuste no teste; todas as configurações são reportadas (nada de "melhor
de N" sem os N), e cada semente muda a divisão treino/teste.

Run:  OMP_NUM_THREADS=2 .venv/bin/python -m pdm_cmaps.experiments.exploracao_r2 --track B2 --subset FD003
Saída: results/exploracao_r2/<trilha>_<subset>.csv
"""

from __future__ import annotations

import argparse
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from pdm_cmaps.config import DEFAULT_RESULTS_DIR
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.data.splits import partition_healthy_and_lifecycle, split_engine_units
from pdm_cmaps.detection.regime import RegimeDecoupler
from pdm_cmaps.detection.virtual_sensor import VirtualSensor
from pdm_cmaps.detection.virtual_sensor_metrics import r2, virtual_sensor_r2
from pdm_cmaps.domain.rul import compute_piecewise_rul
from pdm_cmaps.domain.schema import DEFAULT_HEALTHY_CYCLE_LIMIT, SETTING_COLUMNS
from pdm_cmaps.experiments.virtual_sensor_models import SafeRegressor
from pdm_cmaps.modeling.registry import get_model

OUT = DEFAULT_RESULTS_DIR / "exploracao_r2"
SUBSETS = ("FD001", "FD003")
SEEDS = (42, 7, 123)
SEEDS10 = (42, 7, 123, 1, 2, 3, 4, 5, 6, 8)  # B1 e B1s: 10 divisões, para dar estabilidade aos achados da B1
TWINS = {"Nf": "NRf", "NRf": "Nf", "Nc": "NRc", "NRc": "Nc"}  # mesmo sinal, corrigido pela temperatura de entrada
H = DEFAULT_HEALTHY_CYCLE_LIMIT  # 50
W = H // 2  # janela do baseline do motor (25)

# modelos pequenos e sem ajuste no teste; "chave do registro" e argumentos
MODELS = {
    "Ridge": ("ridge", {"alpha": 10}),
    "Extra Trees": ("extra_trees", {"n_estimators": 100, "min_samples_leaf": 5, "n_jobs": 2}),
    "LightGBM": ("lightgbm", {"n_estimators": 200, "max_depth": 5, "learning_rate": 0.05, "verbose": -1, "n_jobs": 2}),
}


def prepare(subset: str, seed: int):
    """Carrega, divide por motor e devolve (frame, treino, teste, sensores informativos) para a semente."""
    loader = CMapssLoader()
    frame = loader.load(subset)
    frame["RUL"] = compute_piecewise_rul(frame)
    train_units, test_units = split_engine_units(frame, test_ratio=0.25, random_state=seed)
    sensors = loader.informative_sensors(frame.loc[frame["unit_id"].isin(train_units)])
    return frame, train_units, test_units, sensors


def engine_baseline(frame: pd.DataFrame, cols: list[str], w: int = W) -> pd.DataFrame:
    """Média dos primeiros w ciclos de cada motor (causal), alinhada linha a linha com `frame`."""
    base = frame.loc[frame["cycle"] <= w].groupby("unit_id")[cols].mean()
    return base.reindex(frame["unit_id"]).set_index(frame.index)


# ---------------------------------------------------------------- Trilhas A e B1 (via VirtualSensor, com detecção)
def run_virtual_sensor_track(track: str, subset: str, seeds=SEEDS) -> pd.DataFrame:
    mode = "exogenous" if track == "A" else "cross_sensor"
    drop_twins = track == "B1s"
    rows = []
    for seed in seeds:
        frame, train_units, test_units, sensors = prepare(subset, seed)
        parts = partition_healthy_and_lifecycle(frame, train_units, test_units, H)
        regimes = RegimeDecoupler(n_clusters=1, random_state=seed).fit(parts["full_lifecycle_train"])
        healthy_train, test_full = regimes.transform(parts["healthy_train"]), regimes.transform(parts["full_lifecycle_test"])
        for sensor in sensors:
            for name in ("Ridge", "Extra Trees"):
                key, kw = MODELS[name]
                feats = None
                if drop_twins:  # mesmas entradas da B1, menos o gêmeo do alvo
                    feats = [c for c in healthy_train.columns if c not in {sensor, TWINS.get(sensor), "unit_id", "cycle", "regime_id", "RUL", "subpart"}]
                vs = VirtualSensor(target_sensor=sensor, mode=mode, feature_cols=feats, ewma_alpha=0.10, healthy_cycle_limit=H, min_consecutive_breaches=2,
                                   random_state=seed, estimator=SafeRegressor(get_model(key, seed, **kw)), engine_offset=True, offset_window=W)
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    scored, _, summary = vs.evaluate(healthy_train, test_full)
                trace = scored.rename(columns={sensor: "actual"}).assign(sensor=sensor, model=name)
                m = virtual_sensor_r2(trace[["sensor", "model", "unit_id", "cycle", "actual", "prediction"]], H).iloc[0].to_dict()
                rows.append({"track": track, "subset": subset, "seed": seed, "sensor": sensor, "model": name,
                             **{k: m[k] for k in ("r2_holdout", "r2_late_model", "r2_late_motor_base", "r2_late_model_offset", "pct_between_engine")},
                             "detection_rate_pct": summary["detection_rate_pct"], "healthy_false_alarm_pct": summary["mean_healthy_false_alarm_pct"],
                             "premature_alarm_pct": summary["premature_alarm_pct"], "lead_time_detected": summary["mean_lead_time_detected_cycles"]})
        print(f"  [{track} {subset}] seed {seed} ok", flush=True)
    return pd.DataFrame(rows)


# ---------------------------------------------------------------- Trilha B2 (regressão pura no ciclo de vida inteiro)
def feature_sets(frame: pd.DataFrame, target: str, inputs: list[str]) -> dict[str, tuple[pd.DataFrame, bool]]:
    """Conjuntos de entradas (todos causais e SEM o alvo). O booleano diz se o alvo é o desvio do baseline do motor."""
    x = frame[inputs]
    base_x = engine_baseline(frame, inputs)
    cent = (x - base_x).add_suffix("_c")
    roll = frame.groupby("unit_id")[inputs].transform(lambda s: s.rolling(5, min_periods=1).mean())
    roll_c = (roll - base_x).add_suffix("_rc")
    return {
        "bruto": (x, False),
        "bruto+centrado": (pd.concat([x, cent], axis=1), False),
        "centrado->desvio": (cent, True),
        "media5_centrada->desvio": (roll_c, True),
    }


def run_lifecycle_track(subset: str, seeds=SEEDS) -> pd.DataFrame:
    t0, rows = time.time(), []
    for seed in seeds:
        frame, train_units, test_units, sensors = prepare(subset, seed)
        tr_mask, te_mask = frame["unit_id"].isin(train_units).to_numpy(), frame["unit_id"].isin(test_units).to_numpy()
        after = (frame["cycle"] > W).to_numpy()
        for sensor in sensors:
            inputs = [c for c in sensors if c != sensor] + SETTING_COLUMNS
            y = frame[sensor].to_numpy(dtype=float)
            base_y = engine_baseline(frame, [sensor])[sensor].to_numpy()
            for fs_name, (X, delta) in feature_sets(frame, sensor, inputs).items():
                target = y - base_y if delta else y
                for name, (key, kw) in MODELS.items():
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")
                        model = SafeRegressor(get_model(key, seed, **kw)).fit(X[tr_mask].to_numpy(), target[tr_mask])
                        pred = model.predict(X[te_mask].to_numpy())
                    pred = pred + base_y[te_mask] if delta else pred
                    yt, a = y[te_mask], after[te_mask]
                    rows.append({"track": "B2", "subset": subset, "seed": seed, "sensor": sensor, "features": fs_name, "model": name,
                                 "r2_all_cycles": r2(yt, pred), "r2_after_w": r2(yt[a], pred[a])})
        print(f"  [B2 {subset}] seed {seed} ok ({time.time() - t0:.0f}s)", flush=True)
    return pd.DataFrame(rows)


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--track", choices=("A", "B1", "B1s", "B2"), required=True)
    ap.add_argument("--subset", choices=SUBSETS, required=True)
    ap.add_argument("--seeds", type=int, nargs="*", default=None, help="padrão: 3 sementes (A, B2) ou 10 (B1, B1s)")
    ap.add_argument("--tag", default="", help="sufixo do arquivo de saída (ex.: _10s)")
    ap.add_argument("--out", type=Path, default=OUT)
    a = ap.parse_args(argv)
    a.out.mkdir(parents=True, exist_ok=True)
    seeds = tuple(a.seeds) if a.seeds else (SEEDS10 if a.track in ("B1", "B1s") else SEEDS)
    df = run_lifecycle_track(a.subset, seeds) if a.track == "B2" else run_virtual_sensor_track(a.track, a.subset, seeds)
    f = a.out / f"{a.track}_{a.subset}{a.tag}.csv"
    df.to_csv(f, index=False)
    print(f"gravado {f} ({len(df)} linhas)", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/exploracao_r2_resumo.py
"""Resume a exploração de R² (média das sementes) ao lado do teto de ruído. Lê results/exploracao_r2/{A,B1,B2}_<subset>.csv.

Run:  .venv/bin/python -m pdm_cmaps.experiments.exploracao_r2_resumo
Saída: results/exploracao_r2/resumo_<subset>.csv (uma linha por sensor) e impressão no terminal.
"""
from __future__ import annotations

import pandas as pd

from pdm_cmaps.experiments.exploracao_r2 import OUT, SUBSETS


def resumo(subset: str) -> pd.DataFrame | None:
    teto = pd.read_csv(OUT / "teto_ruido.csv").query("subset == @subset").set_index("sensor")
    out = pd.DataFrame(index=teto.index)
    out["teto_saudavel"] = teto["ceil_late_26_50"].clip(lower=0)
    out["teto_todos"] = teto["ceil_all"]
    for track in ("A", "B1"):
        f = OUT / f"{track}_{subset}.csv"
        if not f.exists():
            continue
        d = pd.read_csv(f)
        best = d.groupby(["sensor", "model"]).mean(numeric_only=True).reset_index()
        for model, tag in (("Extra Trees", "ET"), ("Ridge", "Rg")):
            m = best[best.model == model].set_index("sensor")
            if track == "A":
                out[f"A_{tag}_tardio"] = m["r2_late_model_offset"]
                out["A_base_motor"] = m["r2_late_motor_base"]
                out[f"A_{tag}_deteccao"] = m["detection_rate_pct"]
            else:
                out[f"B1_{tag}_tardio"] = m["r2_late_model"]
                out[f"B1_{tag}_todos"] = m["r2_holdout"]
                out[f"B1_{tag}_deteccao"] = m["detection_rate_pct"]
                out[f"B1_{tag}_FA%"] = m["healthy_false_alarm_pct"]
    f = OUT / f"B2_{subset}.csv"
    if f.exists():
        d = pd.read_csv(f)
        g = d.groupby(["sensor", "features", "model"])[["r2_all_cycles", "r2_after_w"]].mean().reset_index()
        out["B2_melhor_todos"] = g.groupby("sensor")["r2_all_cycles"].max()
        top = g.loc[g.groupby("sensor")["r2_all_cycles"].idxmax()].set_index("sensor")
        out["B2_config"] = top["features"] + " / " + top["model"]
        out["B2_ridge_bruto"] = g.query("features == 'bruto' and model == 'Ridge'").set_index("sensor")["r2_all_cycles"]
    out.round(3).to_csv(OUT / f"resumo_{subset}.csv")
    return out


def main() -> int:
    pd.set_option("display.width", 250)
    for s in SUBSETS:
        r = resumo(s)
        print(f"\n=== {s} ===\n{r.round(2).to_string()}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


def tabela_r2_regime(model: str = "Extra Trees") -> pd.DataFrame:
    """Tabela para a monografia (tcc-writing/tables/sensor_virtual_r2_regime.csv): R² do sensor virtual bruto, dentro do regime, teto de
    ruído e fração do teto, mediana sobre os 15 sensores comuns aos quatro subconjuntos (nos multirregime há canais ambientes extras, como o
    T2, cujo R² é trivial por serem entradas do modelo). Ciclos 26 a 50 dos motores de teste, uma divisão treino-teste (semente 42)."""
    from pdm_cmaps.config import DEFAULT_RESULTS_DIR
    base = DEFAULT_RESULTS_DIR / "virtual_sensor_models"
    scores = {s: pd.read_csv(base / s / "model_scores.csv") for s in SUBSETS_ALL}
    common = set.intersection(*[set(x["sensor"]) for x in scores.values()])
    rows = []
    for s, x in scores.items():
        m = x[(x["model"] == model) & x["sensor"].isin(common)]
        rows.append({"Subset": s, "n_sensores": len(m), "r2_bruto": m["r2_late_model"].median(), "r2_dentro_do_regime": m["r2_late_within_regime"].median(),
                     "teto_de_ruido": m["noise_ceiling_late"].median(), "fracao_do_teto": m["frac_ceiling"].median()})
    out = pd.DataFrame(rows).round(3)
    dest = DEFAULT_RESULTS_DIR.parent / "tcc-writing" / "tables" / "sensor_virtual_r2_regime.csv"
    dest.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(dest, index=False)
    return out


SUBSETS_ALL = ("FD001", "FD002", "FD003", "FD004")


def tabela_exploracao() -> pd.DataFrame:
    """Tabela para a monografia (tcc-writing/tables/sensor_virtual_exploracao_r2.csv): R² por sensor do FD001 e do FD003 em cada trilha.

    A (exógeno), B1 e B1s (outros sensores, treino na janela saudável, sem e com o gêmeo fora): ciclos 26 a 50, Extra Trees, média das
    divisões (A: 3; B1 e B1s: 10). B2 (outros sensores, treino no ciclo de vida inteiro, SEM detecção): todos os ciclos dos motores de teste,
    configuração fixa bruto + centrado com Extra Trees (3 divisões), para não escolher a melhor no teste."""
    from pdm_cmaps.config import DEFAULT_RESULTS_DIR
    rows = []
    teto_all = pd.read_csv(OUT / "teto_ruido.csv")
    for s in ("FD001", "FD003"):
        teto = teto_all[teto_all["subset"] == s].set_index("sensor")
        def trk(f):
            d = pd.read_csv(OUT / f)
            d = d[d["model"] == "Extra Trees"]
            return d.groupby("sensor").agg(r2=("r2_late_model", "mean"), det=("detection_rate_pct", "mean"))
        A, B1, B1s = trk(f"A_{s}.csv"), trk(f"B1_{s}_10s.csv"), trk(f"B1s_{s}_10s.csv")
        b2 = pd.read_csv(OUT / f"B2_{s}.csv").query("features == 'bruto+centrado' and model == 'Extra Trees'").groupby("sensor")["r2_all_cycles"].mean()
        for sn in teto.index:
            rows.append({"Subset": s, "sensor": sn, "teto_ruido_26_50": round(max(float(teto.loc[sn, "ceil_late_26_50"]), 0), 3),
                         "A_r2": A.loc[sn, "r2"], "A_deteccao_pct": A.loc[sn, "det"], "B1_r2": B1.loc[sn, "r2"], "B1_deteccao_pct": B1.loc[sn, "det"],
                         "B1s_r2": B1s.loc[sn, "r2"], "B1s_deteccao_pct": B1s.loc[sn, "det"], "B2_r2_todos_ciclos": b2.loc[sn]})
    out = pd.DataFrame(rows).round(3)
    dest = DEFAULT_RESULTS_DIR.parent / "tcc-writing" / "tables" / "sensor_virtual_exploracao_r2.csv"
    dest.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(dest, index=False)
    return out

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/fault_detection_study.py
"""Early-detection study with injected faults: power, delay and false alarms at MATCHED false-alarm rate.

WHAT: injects faults with a known start t0 into healthy surrogate engines (see fault_injection.py) and
scores four detectors on the same blind series:
    T50 bruto (sem regime)        EWMA of |z| on raw T50, global healthy mean/sd (ablation)
    T50 EWMA (com regime)         the thesis detector: regime-conditioned virtual-sensor residual (default 3sigma gate)
    T50 EWMA (com regime, sem portão)  same statistic and limit without the gate: like-for-like with the raw baseline
    MEWMA bruto (sem regime)      MEWMA on the 13 raw sensors, globally standardized (ablation)
    MEWMA (13 resíduos)           the thesis detector: 13 regime-conditioned residuals
WHY: with 99.5 % of the T50 variance between regimes (FD002/FD004), a detector that does not remove the
regime should need much larger faults. The injected fault (start known only to us) turns that claim
into a measurable power curve. FD001/FD003 have one regime, so they are the natural control (the raw and
regime detectors should coincide there).

WHAT IS MEASURED (no RUL anywhere)
  * operating curves: every detector's threshold is swept (alarm when ratio = statistic / limit >= s for
    k=2 consecutive cycles after H; s = 1 is the default rule). Power vs false-alarm rate, not one point.
  * false alarms: many independent healthy controls; FAR = alarms per 100 monitored cycles and
    ARL0 = monitored cycles per alarm (lower bound when no alarm occurred).
  * matched comparison: for each detector the smallest s whose control FAR <= budget (0.2 or 1 per 100
    cycles) is used, so detectors are compared at the SAME false-alarm price.
  * delay = first alarm at cycle >= t0 minus t0; a fault series is monitored from t0 on, and false alarms
    before t0 are priced by the controls (so the power denominator is constant and curves are monotone).
  * minimum detectable magnitude: smallest magnitude with power >= 80 % (per seed; median over seeds).
  * adopted setting: H = 25, t0 = H+10, budget 0.2 per 100 cycles, seeds {42, 7, 123}. Sweeps (H in 25/35/50 and
    50/75/100) show power falling slowly as H grows at equal false-alarm rate; H = 25 is nominally the best but
    only by ~1.6 points on average over 12 paired cells (not significant); delay is unchanged. H < 25 was not tested.

[LEAKAGE] detectors are fit on TRAINING engines only; the faulty / control series come from TEST engines
and reach the detectors as raw C-MAPSS columns (no label, scenario or t0); the answer key is a separate
table used only by the scoring functions. [CAUSALITY] injection touches cycles >= t0 only; a control and
its faulty twin share the same bootstrap base. [ASSUMPTION] healthy surrogate = i.i.d. bootstrap of the
engine's own cycles with true RUL >= 125 (healthy; regimes and noise are i.i.d. per cycle in C-MAPSS, the
engine-specific offset is kept). The raw baselines never use the raw-3sigma gate (it would make them
unreachable under regime variance); the regime T50 chart is reported with the default gate AND without it
(like-for-like), so the effect of the regime is not confounded with the gate.

Run:  .venv/bin/python -m pdm_cmaps.experiments.fault_detection_study [--subsets FD004] [--jobs 4]  /  --aggregate-only
Output: results/injecao_deteccao/{runs,tabelas,figuras}
"""

from __future__ import annotations

from pdm_cmaps.config import DEFAULT_RESULTS_DIR

import argparse
import warnings
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pdm_cmaps.domain.schema import DEFAULT_RUL_CAP
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.detection.multivariate import MultivariateResidualMonitor
from pdm_cmaps.detection.spc import StatisticalAnomalyDetector
from pdm_cmaps.detection.virtual_sensor import VirtualSensor
from pdm_cmaps.experiments.fault_injection import SCENARIOS, World, blind_columns, delta_z, inject, load_world
from pdm_cmaps.reporting import viz_style

OUT_DEFAULT = DEFAULT_RESULTS_DIR / "injecao_deteccao"
ADOPTED_H = 25                 # window used by the figures (sweep 25/35/50 and 50/75/100: see module docstring)
DET_RAW_T50 = "T50 bruto (sem regime)"
DET_T50 = "T50 EWMA (com regime)"
DET_T50_NOGATE = "T50 EWMA (com regime, sem portão)"
DET_RAW_MV = "MEWMA bruto (sem regime)"
DET_MV = "MEWMA (13 resíduos, com regime)"
DETECTORS = (DET_RAW_T50, DET_T50, DET_T50_NOGATE, DET_RAW_MV, DET_MV)
# detectores de comparação (--extras): EWMA clássica de Roberts (resíduo com sinal) e Isolation Forest sobre os 13 resíduos; ficam fora de DETECTORS
# para não mudar as saídas e figuras do protocolo principal; a mesma série cega (mesma semente) alimenta os dois estudos
DET_T50_SIGNED = "T50 EWMA clássica (com sinal, com regime)"
DET_IFOREST = "Isolation Forest (13 resíduos, com regime)"
EXTRA_DETECTORS = (DET_T50_SIGNED, DET_IFOREST)
BUDGETS = (0.2,)                # best/adopted budget: false alarms per 100 monitored cycles (--budgets 0.2 1.0 for more)
S_GRID = np.unique(np.append(np.geomspace(0.3, 4.0, 40), 1.0))
K_CONSEC = 2
HORIZON_CTRL = 300
POST_DET = 80
LATE_T0_OFFSET = 60
LATE_KINDS = (("degrau", (0.25, 0.5)), ("rampa", (0.25, 0.5)))
MIN_POOL = 30
POWER_TARGET = 80.0


# ============================================================================
# series generation (healthy bootstrap + injected faults)
# ============================================================================

def bootstrap_series(engine: pd.DataFrame, horizon: int, rng: np.random.Generator) -> pd.DataFrame | None:
    """i.i.d. bootstrap of the engine's healthy cycles (true RUL >= cap), renumbered 1..horizon."""
    pool = engine.loc[engine["rul_true"] >= DEFAULT_RUL_CAP]
    if len(pool) < MIN_POOL:
        return None
    base = pool.iloc[rng.integers(0, len(pool), horizon)].reset_index(drop=True).copy()
    base["cycle"] = np.arange(1, horizon + 1)
    return base


def build_series(world: World, seed: int, n_ctrl: int, n_rep: int, max_engines: int | None = None, late_t0: bool = False) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Blind frame (raw schema, unit_id = blind id) and the answer key."""
    rng = np.random.default_rng(seed)
    H = world.H
    t0s = (H + 10, H + LATE_T0_OFFSET) if late_t0 else (H + 10,)
    engines = {int(u): g.sort_values("cycle") for u, g in world.test.groupby("unit_id")}
    recipients = [u for u in sorted(engines) if (engines[u]["rul_true"] >= DEFAULT_RUL_CAP).sum() >= MIN_POOL]
    if max_engines:
        recipients = recipients[:max_engines]
    frames, meta, bid = [], [], 0
    for u in recipients:
        for _ in range(n_ctrl):
            base = bootstrap_series(engines[u], HORIZON_CTRL, rng)
            bid += 1
            frames.append(blind_columns(base, bid))
            meta.append({"blind_id": bid, "recipient": u, "role": "controle", "kind": "", "magnitude": np.nan, "t0": 0, "T": HORIZON_CTRL})
        for t0 in t0s:
            horizon = t0 + POST_DET
            grid = SCENARIOS if t0 == t0s[0] else dict(LATE_KINDS)
            for _ in range(n_rep):
                base = bootstrap_series(engines[u], horizon, rng)
                for kind, mags in grid.items():
                    if kind == "um_regime" and world.regimes.n_clusters < 2:
                        continue
                    for m in mags:
                        bid += 1
                        frames.append(blind_columns(inject(world, base, delta_z(world, base, kind, m, t0, rng)), bid))
                        meta.append({"blind_id": bid, "recipient": u, "role": "falha", "kind": kind, "magnitude": m, "t0": t0, "T": horizon})
    blind = pd.concat(frames, ignore_index=True).sort_values(["unit_id", "cycle"]).reset_index(drop=True)
    return blind, pd.DataFrame(meta)


# ============================================================================
# detector scores: ratio = statistic / limit per cycle (blind frame only)
# ============================================================================

def _stack(ids: np.ndarray, cycles: np.ndarray, ratio: np.ndarray, gate: np.ndarray) -> pd.DataFrame:
    return pd.DataFrame({"blind_id": ids, "cycle": cycles, "ratio": ratio, "gate": gate})


def _engine_level(x: np.ndarray, cycles: np.ndarray, H: int) -> np.ndarray:
    """Média do próprio motor nos primeiros H ciclos (0 se houver menos de 3), para tirar o nível do motor de um sinal bruto."""
    w = cycles <= H
    return x - (x[w].mean(axis=0) if w.sum() >= 3 else 0.0)


def raw_t50_scores(world: World, blind: pd.DataFrame, engine_offset: bool = False) -> pd.DataFrame:
    """EWMA of |z| on raw T50 (global healthy mean/sd); no regime removal, no raw-3sigma gate. ``engine_offset`` tira o nível de cada motor."""
    hh = world.train.loc[world.train["cycle"] <= world.H]
    healthy = hh["T50"].to_numpy(dtype=float)
    if engine_offset:
        healthy = hh["T50"] - hh.groupby("unit_id")["T50"].transform("mean")
        healthy = healthy.to_numpy(dtype=float)
    det = StatisticalAnomalyDetector(threshold_sigma=3.0, ewma_alpha=0.10, healthy_cycle_limit=world.H,
                                     min_consecutive_breaches=K_CONSEC).calibrate(healthy)
    parts = []
    for uid, g in blind.groupby("unit_id", sort=True):
        x = g["T50"].to_numpy(dtype=float)
        if engine_offset:
            x = _engine_level(x, g["cycle"].to_numpy(), world.H)
        e = det.ewma_abs_z(det.compute_z_scores(x))
        parts.append(_stack(np.full(len(g), uid), g["cycle"].to_numpy(), e / det.ewma_upper_limits(len(e)), np.ones(len(g), bool)))
    return pd.concat(parts, ignore_index=True)


def raw_mewma_scores(world: World, blind: pd.DataFrame, sensors: list[str], lam: float = 0.10, alpha: float = 0.0027, engine_offset: bool = False) -> pd.DataFrame:
    """MEWMA T2 on globally standardized raw sensors; limit = empirical (1 - alpha) quantile of healthy T2."""
    mon = MultivariateResidualMonitor(sensors=sensors, lam=lam, alpha=alpha, healthy_cycle_limit=world.H)
    h = world.train.loc[world.train["cycle"] <= world.H].sort_values(["unit_id", "cycle"])
    mean, sd = h[sensors].mean(), h[sensors].std().clip(lower=1e-8)
    zdf = (h[sensors] - mean) / sd
    if engine_offset:
        zdf = zdf - zdf.groupby(h["unit_id"].to_numpy()).transform("mean")
    zh = zdf.to_numpy()
    mon.mu_ = zh.mean(axis=0)
    mon.sigma_inv_ = np.linalg.inv(np.cov(zh, rowvar=False) + 1e-6 * np.eye(zh.shape[1]))
    ucl = float(np.quantile(mon._t2(zh, h["unit_id"].to_numpy()), 1 - alpha))
    b = blind.sort_values(["unit_id", "cycle"])
    zb = ((b[sensors] - mean) / sd)
    if engine_offset:
        for uid, idx in b.groupby("unit_id").groups.items():
            part = zb.loc[idx].to_numpy()
            zb.loc[idx] = _engine_level(part, b.loc[idx, "cycle"].to_numpy(), world.H)
    t2 = mon._t2(zb.to_numpy(), b["unit_id"].to_numpy())
    return _stack(b["unit_id"].to_numpy(), b["cycle"].to_numpy(), t2 / ucl, np.ones(len(b), bool))


def regime_scores(world: World, blind: pd.DataFrame, seed: int, engine_offset: bool = True, mewma_offset: bool = True) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, list[str]]:
    """The thesis detectors: regime-conditioned T50 EWMA (with and without the raw-3sigma gate) and the 13-residual MEWMA."""
    tr, te = world.regimes.transform(world.train), world.regimes.transform(blind)
    vs = VirtualSensor(target_sensor="T50", mode="exogenous", ewma_alpha=0.10, min_consecutive_breaches=K_CONSEC,
                       healthy_cycle_limit=world.H, random_state=seed, engine_offset=engine_offset)
    mv = MultivariateResidualMonitor(healthy_cycle_limit=world.H, random_state=seed, engine_offset=mewma_offset)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        trace, _, _ = vs.evaluate(tr, te)
        mv.fit(tr)
        sc = mv.score(te)
    t50 = _stack(trace["unit_id"].to_numpy(), trace["cycle"].to_numpy(), (trace["ewma_abs_z"] / trace["ewma_ucl"]).to_numpy(),
                 (trace["z_score"].abs() >= vs.threshold_sigma).to_numpy())
    nogate = t50.assign(gate=True)  # same statistic and limit, only the raw-3sigma gate removed (like-for-like with the raw baseline)
    mvs = _stack(sc["unit_id"].to_numpy(), sc["cycle"].to_numpy(), (sc["t2"] / sc["ucl"]).to_numpy(), np.ones(len(sc), bool))
    return t50, nogate, mvs, list(mv.sensors_)


def signed_ewma_ratio(z: np.ndarray, lam: float = 0.10, L: float = 3.0) -> np.ndarray:
    """EWMA clássica de Roberts (1959) sobre o resíduo padronizado COM SINAL, dividida pelo seu limite de controle de L sigmas em cada ciclo:
    w_t = lam z_t + (1 - lam) w_{t-1}, w_0 = 0, limite = L sqrt(lam / (2 - lam) (1 - (1 - lam)^(2t))). Razão >= 1 = fora de controle nos dois sentidos."""
    z = np.asarray(z, dtype=float)
    w = np.empty(len(z))
    prev = 0.0
    for t, x in enumerate(z):
        prev = lam * x + (1 - lam) * prev
        w[t] = prev
    limit = L * np.sqrt(lam / (2 - lam) * (1 - (1 - lam) ** (2 * np.arange(1, len(z) + 1))))
    return np.abs(w) / limit


def iforest_ratio(model, calibration: np.ndarray, rows: np.ndarray, alpha: float = 0.0027) -> np.ndarray:
    """Pontuação de anomalia do Isolation Forest ajustado na calibração, reescalada para que a mediana da calibração valha 0 e o quantil
    1 - alpha valha 1 (a mesma convenção de limite da MEWMA); valores abaixo da mediana ficam em 0."""
    base = -model.score_samples(calibration)
    m, q = float(np.median(base)), float(np.quantile(base, 1 - alpha))
    return np.clip((-model.score_samples(rows) - m) / max(q - m, 1e-9), 0.0, None)


def extra_scores(world: World, blind: pd.DataFrame, seed: int, engine_offset: bool = True, mewma_offset: bool = True) -> dict[str, pd.DataFrame]:
    """Os dois detectores de comparação, sobre os MESMOS resíduos do sensor virtual por regime (e com o mesmo deslocamento do motor) da MEWMA."""
    from sklearn.ensemble import IsolationForest

    tr, te = world.regimes.transform(world.train), world.regimes.transform(blind)
    vs = VirtualSensor(target_sensor="T50", mode="exogenous", ewma_alpha=0.10, min_consecutive_breaches=K_CONSEC,
                       healthy_cycle_limit=world.H, random_state=seed, engine_offset=engine_offset)
    mv = MultivariateResidualMonitor(healthy_cycle_limit=world.H, random_state=seed, engine_offset=mewma_offset)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        trace, _, _ = vs.evaluate(tr, te)
        mv.fit(tr)
        cols, index = {}, None
        for sensor, v in mv.virtual_.items():
            pred = v.predict(te).sort_values(["unit_id", "cycle"]).reset_index(drop=True)
            cols[sensor] = mv.residual_z(v, pred)
            index = pred[["unit_id", "cycle"]]
    trace = trace.sort_values(["unit_id", "cycle"])
    parts = [_stack(np.full(len(g), uid), g["cycle"].to_numpy(), signed_ewma_ratio(g["z_score"].to_numpy()), np.ones(len(g), bool))
             for uid, g in trace.groupby("unit_id", sort=False)]
    signed = pd.concat(parts, ignore_index=True)
    zcal = mv.calibration_z_.to_numpy()
    forest = IsolationForest(n_estimators=200, random_state=seed, n_jobs=1).fit(zcal)
    ratio = iforest_ratio(forest, zcal, np.column_stack([cols[s] for s in mv.virtual_]))
    iso = _stack(index["unit_id"].to_numpy(), index["cycle"].to_numpy(), ratio, np.ones(len(index), bool))
    return {DET_T50_SIGNED: signed, DET_IFOREST: iso}


def score_all(world: World, blind: pd.DataFrame, seed: int, engine_offset: bool = True, mewma_offset: bool = True) -> dict[str, pd.DataFrame]:
    """Receives ONLY the blind frame (raw schema). ``mewma_offset`` dá o deslocamento do motor também à MEWMA e aos dois detectores brutos
    (para que a ablação do regime continue limpa: todos os detectores recebem o mesmo tratamento)."""
    t50, nogate, mvs, sensors = regime_scores(world, blind, seed, engine_offset, mewma_offset)
    return {DET_RAW_T50: raw_t50_scores(world, blind, mewma_offset), DET_T50: t50, DET_T50_NOGATE: nogate,
            DET_RAW_MV: raw_mewma_scores(world, blind, sensors, engine_offset=mewma_offset), DET_MV: mvs}


# ============================================================================
# alarms over a threshold grid, operating curves, matched comparison
# ============================================================================

def alarm_cycles(ratio: np.ndarray, gate: np.ndarray, cycles: np.ndarray, min_cycle: int, s_grid: np.ndarray = S_GRID, k: int = K_CONSEC) -> np.ndarray:
    """First alarm cycle (start of the first run of ``k`` breaches at cycles >= ``min_cycle``) per threshold s; NaN = none."""
    out = np.full(len(s_grid), np.nan)
    n = len(ratio)
    if n < k:
        return out
    breach = (ratio[:, None] >= s_grid[None, :]) & gate[:, None] & (cycles >= min_cycle)[:, None]
    run = breach[k - 1:].copy()
    for j in range(1, k):
        run &= breach[k - 1 - j:n - j]
    has = run.any(axis=0)
    first = run.argmax(axis=0)
    out[has] = cycles[first[has]]
    return out


def alarm_matrix(scores: pd.DataFrame, min_cycle_by_id: pd.Series) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """(blind ids, alarm matrix [series x s], last cycle per series). Monitoring of a series starts at its min cycle."""
    scores = scores.sort_values(["blind_id", "cycle"])
    ids, starts = np.unique(scores["blind_id"].to_numpy(), return_index=True)
    ratio, gate, cyc = scores["ratio"].to_numpy(), scores["gate"].to_numpy(), scores["cycle"].to_numpy()
    bounds = np.append(starts, len(scores))
    mins = min_cycle_by_id.loc[ids].to_numpy()
    mat = np.vstack([alarm_cycles(ratio[a:b], gate[a:b], cyc[a:b], int(mc)) for (a, b), mc in zip(zip(bounds[:-1], bounds[1:]), mins)])
    return ids, mat, cyc[bounds[1:] - 1]


def control_far(alarms: np.ndarray, last: np.ndarray, H: int) -> tuple[np.ndarray, np.ndarray]:
    """False alarms per 100 monitored cycles and ARL0 for each s (events / exposure until first alarm)."""
    has = ~np.isnan(alarms)
    exposure = np.where(has, alarms - H, (last - H)[:, None]).sum(axis=0)
    events = has.sum(axis=0)
    with np.errstate(divide="ignore"):
        return 100 * events / exposure, np.where(events > 0, exposure / np.maximum(events, 1), np.inf)


def analyse_detector(name: str, scores: pd.DataFrame, meta: pd.DataFrame, H: int, budgets: tuple = BUDGETS) -> dict[str, pd.DataFrame]:
    indexed = meta.set_index("blind_id")
    # controls are monitored from H+1; faulty series from t0 (alarms before t0 are false alarms, counted in the controls)
    ids, mat, last = alarm_matrix(scores, np.maximum(indexed["t0"], H + 1))
    m = indexed.loc[ids]
    ctrl = (m["role"] == "controle").to_numpy()
    far, arl0 = control_far(mat[ctrl], last[ctrl], H)
    curves, matched, delays = [], [], []
    stars = {b: (int(np.flatnonzero(far <= b)[0]) if (far <= b).any() else None) for b in budgets}
    fault = m[~ctrl].assign(row=np.flatnonzero(~ctrl))
    for (kind, mag, t0), g in fault.groupby(["kind", "magnitude", "t0"]):
        a = mat[g["row"].to_numpy()]
        det = ~np.isnan(a)
        elig = np.ones_like(det)
        delay = np.where(det, a - t0, np.nan)
        n_elig = elig.sum(axis=0)
        power = 100 * det.sum(axis=0) / n_elig
        dmed = np.array([np.nanmedian(delay[:, i]) if det[:, i].any() else np.nan for i in range(len(S_GRID))])
        dp90 = np.array([np.nanpercentile(delay[:, i], 90) if det[:, i].any() else np.nan for i in range(len(S_GRID))])
        curves.append(pd.DataFrame({"Detector": name, "kind": kind, "magnitude": mag, "t0": t0, "s": S_GRID, "far_ctrl_per100": far,
                                    "arl0": arl0, "n_elig": n_elig, "power_pct": power, "delay_med": dmed, "delay_p90": dp90}))
        for b, i in stars.items():
            row = {"Detector": name, "kind": kind, "magnitude": mag, "t0": t0, "budget": b, "s_star": np.nan, "far_ctrl_per100": np.nan,
                   "arl0": np.nan, "n_elig": 0, "power_pct": np.nan, "delay_med": np.nan, "delay_p90": np.nan}
            if i is not None:
                row.update(s_star=S_GRID[i], far_ctrl_per100=far[i], arl0=arl0[i], n_elig=int(n_elig[i]), power_pct=power[i],
                           delay_med=dmed[i], delay_p90=dp90[i])
                keep = elig[:, i]
                delays.append(pd.DataFrame({"Detector": name, "kind": kind, "magnitude": mag, "t0": t0, "budget": b, "delay": delay[keep, i]}))
            matched.append(row)
    return {"curvas": pd.concat(curves, ignore_index=True), "pareado": pd.DataFrame(matched),
            "atrasos": pd.concat(delays, ignore_index=True) if delays else pd.DataFrame()}


def run_one(subset: str, seed: int, H: int, out: Path, n_ctrl: int = 6, n_rep: int = 2, max_engines: int | None = None,
            budgets: tuple = BUDGETS, late_t0: bool = False, engine_offset: bool = True, mewma_offset: bool = True, extras: bool = False) -> None:
    run_dir = out / "runs"
    tag = f"{subset}_seed{seed}_H{H}"
    if all((run_dir / f"{tag}_{k}.csv").exists() for k in ("curvas", "pareado", "atrasos")):
        return
    world = load_world(subset, seed, H, loader=CMapssLoader())
    blind, meta = build_series(world, seed, n_ctrl, n_rep, max_engines, late_t0)
    print(f"[{tag}] {meta['blind_id'].nunique()} series ({(meta['role'] == 'controle').sum()} controls)", flush=True)
    scores = extra_scores(world, blind, seed, engine_offset, mewma_offset) if extras else score_all(world, blind, seed, engine_offset, mewma_offset)
    parts = {"curvas": [], "pareado": [], "atrasos": []}
    for name, sc in scores.items():
        res = analyse_detector(name, sc, meta, H, budgets)
        for k in parts:
            parts[k].append(res[k])
    run_dir.mkdir(parents=True, exist_ok=True)
    for k, v in parts.items():
        pd.concat(v, ignore_index=True).assign(Subset=subset, Seed=seed, H=H).to_csv(run_dir / f"{tag}_{k}.csv", index=False)


# ============================================================================
# aggregation, tables, figures
# ============================================================================

def _load(out: Path, kind: str) -> pd.DataFrame:
    return pd.concat([pd.read_csv(p) for p in sorted((out / "runs").glob(f"*_{kind}.csv"))], ignore_index=True)


def min_detectable(pareado: pd.DataFrame) -> pd.DataFrame:
    """Smallest magnitude with power >= 80 % per (subset, seed, H, detector, kind, t0, budget)."""
    ok = pareado[pareado["power_pct"] >= POWER_TARGET]
    keys = ["Subset", "Seed", "H", "Detector", "kind", "t0", "budget"]
    mdm = ok.groupby(keys)["magnitude"].min().rename("mdm").reset_index()
    allk = pareado[keys].drop_duplicates().merge(mdm, on=keys, how="left")
    g = allk.groupby(["Subset", "H", "Detector", "kind", "t0", "budget"])
    return g.agg(mdm_mediana=("mdm", "median"), seeds_que_atingem=("mdm", lambda s: int(s.notna().sum())), n_seeds=("mdm", "size")).reset_index()


def aggregate(out: Path) -> dict[str, pd.DataFrame]:
    tabs = out / "tabelas"
    tabs.mkdir(parents=True, exist_ok=True)
    par, cur = _load(out, "pareado"), _load(out, "curvas")
    keys = ["Subset", "H", "Detector", "kind", "magnitude", "t0", "budget"]
    g = par.groupby(keys)[["s_star", "far_ctrl_per100", "arl0", "power_pct", "delay_med", "delay_p90"]]
    summ = g.mean().add_suffix("_mean").join(g.std().add_suffix("_std")).reset_index()
    summ["n_seeds"] = g.size().to_numpy()
    mdm = min_detectable(par)
    default = cur[np.isclose(cur["s"], 1.0)].groupby(["Subset", "H", "Detector", "kind", "magnitude", "t0"])[
        ["far_ctrl_per100", "arl0", "power_pct", "delay_med"]].mean().reset_index()
    summ.to_csv(tabs / "pareado_far_resumo.csv", index=False)
    mdm.to_csv(tabs / "magnitude_minima_detectavel.csv", index=False)
    default.to_csv(tabs / "regra_padrao_s1.csv", index=False)
    return {"resumo": summ, "mdm": mdm, "padrao": default, "curvas": cur, "atrasos": _load(out, "atrasos")}


STYLE = {DET_RAW_T50: (viz_style.SERIES[0], "--"), DET_T50: (viz_style.SERIES[0], "-"), DET_T50_NOGATE: (viz_style.SERIES[2], "-"),
         DET_RAW_MV: (viz_style.SERIES[1], "--"), DET_MV: (viz_style.SERIES[1], "-")}


def fig_ablation(summ: pd.DataFrame, path: Path, H: int = ADOPTED_H, budget: float = 0.2, subsets=("FD001", "FD002", "FD004")) -> None:
    viz_style.apply_style()
    kinds = list(SCENARIOS)
    subsets = [s for s in subsets if s in set(summ["Subset"])]
    fig, axes = plt.subplots(len(subsets), len(kinds), figsize=(3.3 * len(kinds), 2.8 * len(subsets)), squeeze=False, sharey=True)
    base = summ[(summ["H"] == H) & (summ["budget"] == budget) & (summ["t0"] == H + 10)]
    for i, sub in enumerate(subsets):
        for j, kind in enumerate(kinds):
            ax = axes[i, j]
            for det in DETECTORS:
                d = base[(base["Subset"] == sub) & (base["kind"] == kind) & (base["Detector"] == det)].sort_values("magnitude")
                if d.empty:
                    continue
                col, ls = STYLE[det]
                ax.errorbar(d["magnitude"], d["power_pct_mean"], yerr=d["power_pct_std"].fillna(0), color=col, ls=ls, marker="o", ms=4,
                            lw=viz_style.LINE_W, capsize=2, label=det)
            viz_style.style_axes(ax)
            ax.set_ylim(-3, 103)
            if i == 0:
                ax.set_title(kind.replace("_", " "), fontsize=10)
            if j == 0:
                ax.set_ylabel(f"{sub}\npoder (%)", fontsize=9)
            if i == len(subsets) - 1:
                ax.set_xlabel("magnitude" + (" (× assinatura de fim de vida)" if kind in ("degrau", "rampa", "um_regime") else " (σ)"), fontsize=7)
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=3, frameon=False, fontsize=8)
    fig.suptitle(f"Ablação do desacoplamento de regime: poder de detecção com o MESMO falso alarme "
                 f"({budget:g} por 100 ciclos), H = {H}. Tracejado = sem regime, cheio = com regime (média ± dp de 3 seeds)", fontsize=9.5)
    fig.tight_layout(rect=(0, 0.05, 1, 0.94))
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=140)
    plt.close(fig)


def fig_operating(cur: pd.DataFrame, path: Path, subset: str = "FD004", H: int = ADOPTED_H) -> None:
    viz_style.apply_style()
    cases = [("degrau", 0.25), ("rampa", 0.25), ("degrau", 0.5)]
    fig, axes = plt.subplots(1, len(cases), figsize=(4.4 * len(cases), 3.8), squeeze=False, sharey=True)
    for ax, (kind, mag) in zip(axes[0], cases):
        for det in DETECTORS:
            d = cur[(cur["Subset"] == subset) & (cur["H"] == H) & (cur["kind"] == kind) & np.isclose(cur["magnitude"], mag)
                    & (cur["t0"] == H + 10) & (cur["Detector"] == det)]
            m = d.groupby("s")[["far_ctrl_per100", "power_pct"]].mean().sort_index()
            col, ls = STYLE[det]
            ax.plot(m["far_ctrl_per100"].clip(lower=1e-3), m["power_pct"], color=col, ls=ls, lw=viz_style.LINE_W, label=det)
            one = m.loc[np.isclose(m.index, 1.0)]
            ax.scatter(one["far_ctrl_per100"].clip(lower=1e-3), one["power_pct"], color=col, s=40, zorder=5, edgecolors=viz_style.SURFACE)
        for b in BUDGETS:
            ax.axvline(b, color=viz_style.MUTED, lw=0.8, ls=":")
        ax.set_xscale("log")
        viz_style.style_axes(ax, grid="both")
        ax.set_title(f"{subset} · {kind} {mag:g}", fontsize=10)
        ax.set_xlabel("falsos alarmes por 100 ciclos (controles)")
    axes[0, 0].set_ylabel("poder de detecção (%)")
    axes[0, 0].legend(frameon=False, fontsize=6.5, loc="lower right")
    fig.suptitle("Curvas operacionais (limite varrido; ponto = regra padrão s = 1; pontilhado = orçamentos de falso alarme)", fontsize=9.5)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=140)
    plt.close(fig)


def fig_cumulative(delays: pd.DataFrame, path: Path, subset: str = "FD004", H: int = ADOPTED_H, budget: float = 0.2) -> None:
    viz_style.apply_style()
    cases = [("degrau", 0.25), ("degrau", 0.5), ("rampa", 0.5)]
    fig, axes = plt.subplots(1, len(cases), figsize=(4.4 * len(cases), 3.6), squeeze=False, sharey=True)
    grid = np.arange(0, POST_DET + 1)
    for ax, (kind, mag) in zip(axes[0], cases):
        for det in DETECTORS:
            d = delays[(delays["Subset"] == subset) & (delays["H"] == H) & (delays["kind"] == kind) & np.isclose(delays["magnitude"], mag)
                       & (delays["t0"] == H + 10) & (delays["budget"] == budget) & (delays["Detector"] == det)]
            if d.empty:
                continue
            x = d["delay"].to_numpy()
            col, ls = STYLE[det]
            ax.step(grid, [100 * np.mean(x <= g) if len(x) else 0 for g in grid], where="post", color=col, ls=ls, lw=viz_style.LINE_W, label=det)
        viz_style.style_axes(ax)
        ax.set_title(f"{subset} · {kind} {mag:g}", fontsize=10)
        ax.set_xlabel("ciclos após o início da falha (t0)")
    axes[0, 0].set_ylabel("séries detectadas até o atraso (%)")
    axes[0, 0].legend(frameon=False, fontsize=6.5, loc="lower right")
    fig.suptitle(f"Detecção acumulada por atraso (falso alarme igualado a {budget:g} por 100 ciclos; sem detecção = censurado no fim da série)", fontsize=9.5)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=140)
    plt.close(fig)


def fig_window(summ: pd.DataFrame, path: Path, budget: float = 0.2, kind: str = "degrau", mag: float = 0.25) -> None:
    viz_style.apply_style()
    subsets = sorted(set(summ["Subset"]))
    fig, axes = plt.subplots(2, len(subsets), figsize=(3.4 * len(subsets), 6.0), squeeze=False)
    for j, sub in enumerate(subsets):
        for det in DETECTORS:
            d = summ[(summ["Subset"] == sub) & (summ["Detector"] == det) & (summ["kind"] == kind) & np.isclose(summ["magnitude"], mag)
                     & (summ["budget"] == budget) & (summ["t0"] == summ["H"] + 10)].sort_values("H")
            col, ls = STYLE[det]
            axes[0, j].errorbar(d["H"], d["power_pct_mean"], yerr=d["power_pct_std"].fillna(0), color=col, ls=ls, marker="o", ms=4,
                                lw=viz_style.LINE_W, capsize=2, label=det)
            axes[1, j].errorbar(d["H"], d["delay_med_mean"], yerr=d["delay_med_std"].fillna(0), color=col, ls=ls, marker="o", ms=4,
                                lw=viz_style.LINE_W, capsize=2)
        axes[0, j].set_title(sub, fontsize=10)
        axes[0, j].set_ylim(-3, 103)
        axes[1, j].set_xlabel("janela saudável H (ciclos)")
        for i in range(2):
            viz_style.style_axes(axes[i, j])
            axes[i, j].set_xticks([50, 75, 100])
    axes[0, 0].set_ylabel("poder (%)")
    axes[1, 0].set_ylabel("atraso mediano (ciclos)")
    axes[0, 0].legend(frameon=False, fontsize=6.5, loc="lower left")
    fig.suptitle(f"Efeito da janela saudável H na detecção ({kind} {mag:g}, falso alarme igualado a {budget:g} por 100 ciclos)", fontsize=9.5)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=140)
    plt.close(fig)


def make_figures(res: dict[str, pd.DataFrame], out: Path) -> None:
    fig_dir = out / "figuras"
    fig_ablation(res["resumo"], fig_dir / "fig_ablacao_regime.png")
    fig_operating(res["curvas"], fig_dir / "fig_curvas_operacionais.png")
    fig_cumulative(res["atrasos"], fig_dir / "fig_deteccao_acumulada.png")
    if res["resumo"]["H"].nunique() > 1:
        fig_window(res["resumo"], fig_dir / "fig_janela_poder_atraso.png")


def main(argv: list[str] | None = None) -> int:
    p = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    p.add_argument("--subsets", nargs="+", default=["FD001", "FD002", "FD003", "FD004"])
    p.add_argument("--seeds", nargs="+", type=int, default=[42, 7, 123])
    p.add_argument("--windows", nargs="+", type=int, default=[25], help="healthy windows H; 25 is the adopted value (25 35 50 reproduces the sweep)")
    p.add_argument("--budgets", nargs="+", type=float, default=list(BUDGETS), help="false alarms per 100 cycles")
    p.add_argument("--late-t0", action="store_true", help="also inject at t0 = H + 60 (robustness)")
    p.add_argument("--n-ctrl", type=int, default=6, help="independent healthy controls per recipient engine")
    p.add_argument("--n-rep", type=int, default=2, help="independent bootstrap bases per recipient and t0")
    p.add_argument("--max-engines", type=int, default=None)
    p.add_argument("--jobs", type=int, default=4)
    p.add_argument("--output-dir", type=Path, default=None, help=f"padrão: {OUT_DEFAULT} (protocolo principal: deslocamento do motor em todos os detectores); --no-mewma-offset grava em {OUT_DEFAULT}_offset_so_t50 e --no-engine-offset em {OUT_DEFAULT}_sem_offset")
    p.add_argument("--engine-offset", action=argparse.BooleanOptionalAction, default=True, help="T50 EWMA com o deslocamento de cada motor (média do z nos primeiros H ciclos) removido; padrão ligado; --no-engine-offset desliga também a MEWMA e os brutos (rodada sem nenhuma correção)")
    p.add_argument("--mewma-offset", action=argparse.BooleanOptionalAction, default=True, help="deslocamento do motor também na MEWMA com regime e nos dois detectores brutos (todos os detectores com o mesmo tratamento); padrão ligado (protocolo principal desde 2026-11-18); --no-mewma-offset reproduz o protocolo anterior (só o T50)")
    p.add_argument("--extras", action="store_true", help="roda só os detectores de comparação (EWMA clássica com sinal e Isolation Forest) nas MESMAS séries cegas; grava em injecao_deteccao_extras")
    p.add_argument("--aggregate-only", action="store_true")
    a = p.parse_args(argv)
    if not a.engine_offset:
        a.mewma_offset = False  # sem a correção no T50, a rodada de referência não tem correção em nenhum detector
    if a.output_dir is None:
        suffix = "" if (a.engine_offset and a.mewma_offset) else ("_offset_so_t50" if a.engine_offset else "_sem_offset")
        a.output_dir = OUT_DEFAULT.with_name(OUT_DEFAULT.name + suffix + ("_extras" if a.extras else ""))
    if not a.aggregate_only:
        jobs = [(s, seed, h) for s in a.subsets for seed in a.seeds for h in a.windows]
        if a.jobs > 1:
            from joblib import Parallel, delayed
            Parallel(n_jobs=a.jobs)(delayed(run_one)(s, seed, h, a.output_dir, a.n_ctrl, a.n_rep, a.max_engines, tuple(a.budgets), a.late_t0, a.engine_offset, a.mewma_offset, a.extras) for s, seed, h in jobs)
        else:
            for s, seed, h in jobs:
                run_one(s, seed, h, a.output_dir, a.n_ctrl, a.n_rep, a.max_engines, tuple(a.budgets), a.late_t0, a.engine_offset, a.mewma_offset, a.extras)
    res = aggregate(a.output_dir)
    if a.extras:
        return 0  # sem figuras: a comparação com a MEWMA e o T50 sai de montar_tabelas.py
    make_figures(res, a.output_dir)
    with pd.option_context("display.width", 220, "display.max_columns", 30):
        print(res["mdm"][(res["mdm"]["H"] == 50) & (res["mdm"]["budget"] == 0.2) & (res["mdm"]["t0"] == 60)].round(2).to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/fault_hard_study.py
"""Hard, numerous, randomly drawn faults: does a learning model beat the regime mean as virtual sensor?

WHY THIS FILE EXISTS: with the simple faults of fault_detection_study.py the "regime mean, no ML" baseline already
detects most faults, and the learning models differ little. Two structural reasons, not a lack of fault difficulty:
  (1) the inputs of the exogenous virtual sensor (settings + ambient channels) are nearly constant inside a regime,
      so there is little for a model to learn (measured: the learning models cut the healthy residual sd by only
      ~5-25 % relative to the regime mean, median ~10 %);
  (2) a bigger fault is easier for EVERY model; difficulty alone does not create a gap between models.
So the fault family is made richer in the ways that can legitimately separate models, not "harder until ML wins":
  * many faults per engine, drawn by a documented random process (seeded, NOT tuned to any estimator);
  * shapes: step, ramp, saturating, intermittent (on/off bursts), plus variance increase for isolated sensors;
  * magnitude log-uniform from nearly undetectable to moderate (so power is measured over a continuum);
  * two families: "processo" (coherent multi-sensor faults along the real end-of-life signature, on 3, 6 or 13
    sensors) and "sensor" (one sensor only, random sign: an instrument-type fault);
  * condition-dependent faults: in half of the multi-regime faults the gain differs per operating regime;
  * two information modes for the virtual sensor: "exogenous" (thesis detector) and "cross_sensor" (also sees the
    other sensors; a consistency check where a learning model has real information to use).

PRE-REGISTERED HYPOTHESES AND DECISION RULE (written before the first run; results are reported whatever they are)
  H1 exogenous mode, processo faults: learning models give a small positive power gain over the regime mean.
     Expectation from the residual ratio (~0.9): at most a few points.
  H2 cross_sensor mode, sensor faults: learning models beat the regime mean clearly (they predict a sensor from the
     others, an isolated bias breaks that relation); processo faults are partly ABSORBED (coherent shifts keep the
     relation), so cross_sensor may lose to exogenous there.
  Decision: an advantage is claimed only if the 95 % cluster-bootstrap CI (clusters = recipient engines) of the paired
  mean power difference vs the regime mean excludes 0. No estimator hyperparameter search; no fault re-drawing.

[LEAKAGE] estimators fit on TRAINING engines (healthy window H); faults are injected into TEST-engine bootstraps and
reach the charts as raw columns; the answer key (family, shape, magnitude, t0, sensors) is used only in scoring.
[ASSUMPTION] fault shapes/magnitudes are plausible, not simulated physics; the end-of-life signature (real data)
fixes the physical direction of "processo" faults. False-alarm budget 0.2 per 100 monitored cycles, matched per chart
on independent healthy controls; single-sensor charts have that budget EACH (a 13-chart panel would multiply it).

Run:  .venv/bin/python -m pdm_cmaps.experiments.fault_hard_study [--subsets FD004 FD002 FD003 FD001] [--jobs 3]     /  --aggregate-only
Output: results/falhas_dificeis/{runs,tabelas,figuras}
"""

from __future__ import annotations

from pdm_cmaps.config import DEFAULT_RESULTS_DIR

import argparse
import warnings
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pdm_cmaps.domain.schema import DEFAULT_RUL_CAP
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.detection.multivariate import MultivariateResidualMonitor
from pdm_cmaps.experiments.fault_detection_study import BUDGETS, HORIZON_CTRL, MIN_POOL, POST_DET, alarm_matrix, bootstrap_series, control_far
from pdm_cmaps.experiments.fault_injection import INJECT_SENSORS, World, blind_columns, inject, load_world
from pdm_cmaps.experiments.fault_model_study import BASELINE, ESTIMATORS, MEWMA, make_estimator, model_scores
from pdm_cmaps.reporting import viz_style

OUT_DEFAULT = DEFAULT_RESULTS_DIR / "falhas_dificeis"
DEFAULT_ESTIMATORS = ["Extra Trees (padrão)", "LightGBM", "Ridge", "MLP", BASELINE]
MODES = ("exogenous", "cross_sensor")
SHAPES = ("degrau", "rampa", "saturante", "intermitente")
T0_RANGE = (10, 60)             # t0 = H + U[10, 60]
HORIZON_EXTRA = T0_RANGE[1] + POST_DET
PROC_MAG = (0.03, 0.40)         # fraction of the end-of-life signature
SENS_MAG = (0.5, 4.0)           # healthy sigma units
BINS = {"processo": [0.03, 0.06, 0.12, 0.25, 0.40001], "sensor": [0.5, 1.0, 2.0, 4.00001]}
BOOT = 2000


# ============================================================================
# fault family (documented random process)
# ============================================================================

def draw_fault(world: World, rng: np.random.Generator, H: int) -> dict:
    k_all = len(INJECT_SENSORS)
    family = "processo" if rng.random() < 0.5 else "sensor"
    spec = {"family": family, "t0": H + int(rng.integers(T0_RANGE[0], T0_RANGE[1] + 1)), "L": int(rng.integers(20, 81)), "period": int(rng.integers(8, 21)),
            "weights": np.zeros(k_all), "gain": None, "shape": str(rng.choice(SHAPES))}
    if family == "processo":
        k = int(rng.choice([3, 6, k_all]))
        idx = np.sort(rng.choice(k_all, size=k, replace=False))
        spec["weights"][idx] = world.signature[idx]
        spec["mag"] = float(np.exp(rng.uniform(np.log(PROC_MAG[0]), np.log(PROC_MAG[1]))))
    else:
        j = int(rng.integers(0, k_all))
        idx = np.array([j])
        spec["weights"][j] = float(rng.choice([-1.0, 1.0]))
        spec["mag"] = float(np.exp(rng.uniform(np.log(SENS_MAG[0]), np.log(SENS_MAG[1]))))
        if rng.random() < 0.2:
            spec["shape"] = "variancia"
    spec["sensors"] = idx.tolist()
    if world.regimes.n_clusters > 1 and rng.random() < 0.5:
        spec["gain"] = rng.uniform(0.1, 1.5, size=world.regimes.n_clusters)
    return spec


def fault_dz(spec: dict, base: pd.DataFrame, rng: np.random.Generator) -> np.ndarray:
    """Standardized-unit perturbation (rows x INJECT_SENSORS) for one fault spec."""
    c, t0 = base["cycle"].to_numpy(), spec["t0"]
    on = c >= t0
    shape = spec["shape"]
    if shape == "degrau":
        g = on.astype(float)
    elif shape == "rampa":
        g = np.clip((c - t0 + 1) / spec["L"], 0, 1) * on
    elif shape == "saturante":
        g = (1 - np.exp(-(c - t0 + 1) / (spec["L"] / 3))) * on
    elif shape == "intermitente":
        g = (((c - t0) // spec["period"]) % 2 == 0) * on
    elif shape == "variancia":
        out = np.zeros((len(base), len(INJECT_SENSORS)))
        j = spec["sensors"][0]
        out[on, j] = rng.normal(0.0, spec["mag"], int(on.sum()))
        return out
    else:
        raise ValueError(shape)
    gain = spec["gain"][base["regime_id"].to_numpy(dtype=int)] if spec["gain"] is not None else 1.0
    return spec["mag"] * (g * gain)[:, None] * spec["weights"][None, :]


def build_series(world: World, seed: int, n_ctrl: int, n_reps: int, n_faults: int, max_engines: int | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed)
    H = world.H
    engines = {int(u): g.sort_values("cycle") for u, g in world.test.groupby("unit_id")}
    recipients = [u for u in sorted(engines) if (engines[u]["rul_true"] >= DEFAULT_RUL_CAP).sum() >= MIN_POOL]
    if max_engines:
        recipients = recipients[:max_engines]
    horizon = H + HORIZON_EXTRA
    frames, meta, bid = [], [], 0
    for u in recipients:
        for _ in range(n_ctrl):
            bid += 1
            frames.append(blind_columns(bootstrap_series(engines[u], HORIZON_CTRL, rng), bid))
            meta.append({"blind_id": bid, "recipient": u, "role": "controle", "family": "", "shape": "", "mag": np.nan, "k": 0, "sensors": "", "gain": False, "t0": 0})
        for _ in range(n_reps):
            base = bootstrap_series(engines[u], horizon, rng)
            for _ in range(n_faults):
                spec = draw_fault(world, rng, H)
                bid += 1
                frames.append(blind_columns(inject(world, base, fault_dz(spec, base, rng)), bid))
                meta.append({"blind_id": bid, "recipient": u, "role": "falha", "family": spec["family"], "shape": spec["shape"], "mag": spec["mag"], "k": len(spec["sensors"]),
                             "sensors": ",".join(INJECT_SENSORS[j] for j in spec["sensors"]), "gain": spec["gain"] is not None, "t0": spec["t0"]})
    blind = pd.concat(frames, ignore_index=True).sort_values(["unit_id", "cycle"]).reset_index(drop=True)
    return blind, pd.DataFrame(meta)


# ============================================================================
# scoring per (mode, estimator)
# ============================================================================

def _matched(scores: pd.DataFrame, meta_idx: pd.DataFrame, H: int, budget: float):
    ids, mat, last = alarm_matrix(scores, np.maximum(meta_idx["t0"], H + 1))
    m = meta_idx.loc[ids]
    ctrl = (m["role"] == "controle").to_numpy()
    far, _ = control_far(mat[ctrl], last[ctrl], H)
    ok = np.flatnonzero(far <= budget)
    if not ok.size:
        return None
    i = int(ok[0])
    fault = ~ctrl
    alarm = mat[fault, i]
    t0 = m.loc[fault, "t0"].to_numpy()
    return pd.DataFrame({"blind_id": ids[fault], "detected": ~np.isnan(alarm), "delay": alarm - t0}), float(far[i])


def score_estimator(world: World, tr: pd.DataFrame, te: pd.DataFrame, meta: pd.DataFrame, est: str, mode: str, seed: int, budget: float = BUDGETS[0],
                    mewma_offset: bool = True) -> pd.DataFrame:
    H = world.H
    ctrl_ids = set(meta.loc[meta["role"] == "controle", "blind_id"].astype(int))
    mv = MultivariateResidualMonitor(healthy_cycle_limit=H, random_state=seed, estimator=make_estimator(est, seed), mode=mode, engine_offset=mewma_offset)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mv.fit(tr)
        scores, _ = model_scores(mv, te, ctrl_ids, H)
    midx = meta.set_index("blind_id")
    rows = []
    res = _matched(scores[MEWMA], midx, H, budget)
    if res is not None:
        rows.append(res[0].assign(Chart="MEWMA", far_chart=res[1]))
    for j, sensor in enumerate(INJECT_SENSORS):
        if sensor not in scores:
            continue
        res = _matched(scores[sensor], midx, H, budget)
        if res is None:
            continue
        per, far = res
        mine = midx.loc[per["blind_id"], "sensors"].to_numpy() == sensor  # isolated faults on exactly this sensor
        fam = midx.loc[per["blind_id"], "family"].to_numpy() == "sensor"
        rows.append(per[mine & fam].assign(Chart="sensor afetado", far_chart=far))
    return pd.concat(rows, ignore_index=True).assign(Mode=mode, Estimator=est)


def run_one(subset: str, seed: int, H: int, out: Path, n_ctrl: int, n_reps: int, n_faults: int, max_engines: int | None, estimators: list[str],
            mewma_offset: bool = True) -> None:
    tag = f"{subset}_seed{seed}_H{H}"
    path = out / "runs" / f"{tag}_series.csv"
    if path.exists():
        return
    world = load_world(subset, seed, H, loader=CMapssLoader())
    blind, meta = build_series(world, seed, n_ctrl, n_reps, n_faults, max_engines)
    print(f"[{tag}] {int((meta['role'] == 'falha').sum())} faults, {int((meta['role'] == 'controle').sum())} controls", flush=True)
    tr, te = world.regimes.transform(world.train), world.regimes.transform(blind)
    parts = []
    for mode in MODES:
        for est in estimators:
            if mode == "cross_sensor" and est == BASELINE:
                continue  # the regime mean ignores its inputs: identical in both modes
            print(f"[{tag}] {mode} / {est}", flush=True)
            parts.append(score_estimator(world, tr, te, meta, est, mode, seed, mewma_offset=mewma_offset))
    path.parent.mkdir(parents=True, exist_ok=True)
    meta.assign(Subset=subset, Seed=seed, H=H).to_csv(out / "runs" / f"{tag}_manifesto.csv", index=False)
    pd.concat(parts, ignore_index=True).assign(Subset=subset, Seed=seed, H=H).to_csv(path, index=False)


# ============================================================================
# aggregation: power by family/magnitude and paired difference vs the regime mean (cluster bootstrap)
# ============================================================================

def _load(out: Path, kind: str) -> pd.DataFrame:
    files = sorted((out / "runs").glob(f"*_{kind}.csv"))
    if not files:
        raise SystemExit(f"nenhum resultado em {out / 'runs'}")
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


def paired_delta(series: pd.DataFrame, manifest: pd.DataFrame, rng: np.random.Generator, n_boot: int = BOOT) -> pd.DataFrame:
    """Mean paired (estimator - regime mean) detection difference in points, 95 % CI by bootstrap over recipient engines."""
    base = series[series["Estimator"] == BASELINE][["Subset", "Seed", "blind_id", "Chart", "detected"]].rename(columns={"detected": "base"})
    d = series[series["Estimator"] != BASELINE].merge(base, on=["Subset", "Seed", "blind_id", "Chart"], how="inner")
    d = d.merge(manifest[["Subset", "Seed", "blind_id", "recipient", "family", "mag"]], on=["Subset", "Seed", "blind_id"])
    d["diff"] = 100.0 * (d["detected"].astype(float) - d["base"].astype(float))
    d["cluster"] = d["Subset"] + "|" + d["Seed"].astype(str) + "|" + d["recipient"].astype(str)
    rows = []
    for keys, g in d.groupby(["Mode", "Estimator", "Chart", "family"]):
        cl = g.groupby("cluster")["diff"].agg(["sum", "count"]).to_numpy()
        idx = rng.integers(0, len(cl), (n_boot, len(cl)))
        boot = cl[idx, 0].sum(axis=1) / cl[idx, 1].sum(axis=1)
        rows.append({"Mode": keys[0], "Estimator": keys[1], "Chart": keys[2], "family": keys[3], "n_faults": len(g), "n_clusters": len(cl),
                     "delta_pp": float(g["diff"].mean()), "ci_lo": float(np.percentile(boot, 2.5)), "ci_hi": float(np.percentile(boot, 97.5))})
    return pd.DataFrame(rows)


def power_by_bin(series: pd.DataFrame, manifest: pd.DataFrame) -> pd.DataFrame:
    d = series.merge(manifest[["Subset", "Seed", "blind_id", "family", "mag", "shape", "gain"]], on=["Subset", "Seed", "blind_id"])
    parts = []
    for fam, edges in BINS.items():
        x = d[d["family"] == fam].copy()
        x["bin"] = pd.cut(x["mag"], edges, include_lowest=True)
        x["mag_mid"] = x["bin"].apply(lambda b: float(np.sqrt(b.left * b.right)) if pd.notna(b) else np.nan)
        parts.append(x)
    x = pd.concat(parts)
    return x.groupby(["Mode", "Estimator", "Chart", "family", "mag_mid"])["detected"].agg(power_pct=lambda s: 100 * s.mean(), n="size").reset_index()


def aggregate(out: Path, seed: int = 0) -> dict[str, pd.DataFrame]:
    tabs = out / "tabelas"
    tabs.mkdir(parents=True, exist_ok=True)
    series, manifest = _load(out, "series"), _load(out, "manifesto")
    delta = paired_delta(series, manifest, np.random.default_rng(seed))
    byb = power_by_bin(series, manifest)
    by_shape = series.merge(manifest[["Subset", "Seed", "blind_id", "family", "shape"]], on=["Subset", "Seed", "blind_id"]).groupby(
        ["Mode", "Estimator", "Chart", "family", "shape"])["detected"].agg(power_pct=lambda s: 100 * s.mean(), n="size").reset_index()
    delta.to_csv(tabs / "diferenca_vs_media_regime.csv", index=False)
    byb.to_csv(tabs / "poder_por_magnitude.csv", index=False)
    by_shape.to_csv(tabs / "poder_por_forma.csv", index=False)
    return {"delta": delta, "bin": byb, "shape": by_shape}


def fig_delta(delta: pd.DataFrame, path: Path) -> None:
    viz_style.apply_style()
    cases = [(c, f) for c in ("MEWMA", "sensor afetado") for f in ("processo", "sensor") if not delta[(delta["Chart"] == c) & (delta["family"] == f)].empty]
    fig, axes = plt.subplots(1, len(cases), figsize=(4.6 * len(cases), 4.2), squeeze=False, sharex=True)
    colors = {"exogenous": viz_style.SERIES[0], "cross_sensor": viz_style.SERIES[1]}
    for ax, (chart, fam) in zip(axes[0], cases):
        d = delta[(delta["Chart"] == chart) & (delta["family"] == fam)].sort_values(["Mode", "Estimator"])
        labels = [f"{e.replace(' (padrão)', '')} · {m.replace('exogenous', 'exóg.').replace('cross_sensor', 'cross')}" for e, m in zip(d["Estimator"], d["Mode"])]
        y = np.arange(len(d))
        ax.barh(y, d["delta_pp"], xerr=[d["delta_pp"] - d["ci_lo"], d["ci_hi"] - d["delta_pp"]], color=[colors[m] for m in d["Mode"]], height=0.65, capsize=2)
        ax.axvline(0, color=viz_style.INK, lw=1)
        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=8)
        ax.invert_yaxis()
        ax.set_title(f"{chart} · falhas de {fam}", fontsize=10)
        ax.set_xlabel("poder − poder da média do regime (pontos)", fontsize=8)
        viz_style.style_axes(ax, grid="x")
    fig.suptitle("Ganho de detecção sobre a média do regime (sem ML), IC 95 % por bootstrap de motores. Azul = exógeno, laranja = cross-sensor", fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=140)
    plt.close(fig)


def fig_power(byb: pd.DataFrame, path: Path) -> None:
    viz_style.apply_style()
    cases = [(c, f) for c in ("MEWMA", "sensor afetado") for f in ("processo", "sensor") if not byb[(byb["Chart"] == c) & (byb["family"] == f)].empty]
    fig, axes = plt.subplots(1, len(cases), figsize=(4.6 * len(cases), 3.9), squeeze=False, sharey=True)
    ests = list(dict.fromkeys(byb["Estimator"]))
    pal = dict(zip(ests, [*viz_style.SERIES, viz_style.STATUS[3], viz_style.INK_2, viz_style.MUTED]))
    for ax, (chart, fam) in zip(axes[0], cases):
        for (est, mode), g in byb[(byb["Chart"] == chart) & (byb["family"] == fam)].groupby(["Estimator", "Mode"]):
            g = g.sort_values("mag_mid")
            ax.plot(g["mag_mid"], g["power_pct"], marker="o", ms=4, lw=viz_style.LINE_W, color=pal[est], ls="-" if mode == "exogenous" else "--", label=f"{est.replace(' (padrão)', '')} · {mode}")
        ax.set_xscale("log")
        ax.set_ylim(-3, 103)
        ax.set_title(f"{chart} · {fam}", fontsize=10)
        ax.set_xlabel("magnitude (fração da assinatura)" if fam == "processo" else "magnitude (σ)", fontsize=8)
        viz_style.style_axes(ax)
    axes[0, 0].set_ylabel("poder (%)")
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=3, frameon=False, fontsize=7)
    fig.suptitle("Poder de detecção ao longo de faltas sorteadas (falso alarme ≤ 0,2 por 100 ciclos); cheio = exógeno, tracejado = cross-sensor", fontsize=10)
    fig.tight_layout(rect=(0, 0.1, 1, 0.94))
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=140)
    plt.close(fig)


def main(argv: list[str] | None = None) -> int:
    p = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    p.add_argument("--H", type=int, default=25)
    p.add_argument("--subsets", nargs="+", default=["FD004", "FD002", "FD003", "FD001"])
    p.add_argument("--seeds", nargs="+", type=int, default=[42, 7, 123])
    p.add_argument("--estimators", nargs="+", default=DEFAULT_ESTIMATORS, choices=ESTIMATORS)
    p.add_argument("--n-ctrl", type=int, default=4)
    p.add_argument("--n-reps", type=int, default=2)
    p.add_argument("--n-faults", type=int, default=12, help="random faults per engine and replicate")
    p.add_argument("--max-engines", type=int, default=None)
    p.add_argument("--jobs", type=int, default=3)
    p.add_argument("--output-dir", type=Path, default=None, help=f"padrão: {OUT_DEFAULT} (protocolo principal, com o deslocamento do motor em todos os detectores); --no-mewma-offset grava em {OUT_DEFAULT}_offset_so_t50")
    p.add_argument("--mewma-offset", action=argparse.BooleanOptionalAction, default=True, help="(padrão ligado desde 2026-11-18; --no-mewma-offset reproduz o protocolo anterior, só o T50) deslocamento do motor (por motor e regime) removido do z de todos os sensores, no T² e nos gráficos por sensor")
    p.add_argument("--aggregate-only", action="store_true")
    a = p.parse_args(argv)
    if a.output_dir is None:
        a.output_dir = OUT_DEFAULT if a.mewma_offset else OUT_DEFAULT.with_name(OUT_DEFAULT.name + "_offset_so_t50")
    if not a.aggregate_only:
        jobs = [(s, seed) for s in a.subsets for seed in a.seeds]
        args = lambda s, seed: (s, seed, a.H, a.output_dir, a.n_ctrl, a.n_reps, a.n_faults, a.max_engines, a.estimators, a.mewma_offset)  # noqa: E731
        if a.jobs > 1:
            from joblib import Parallel, delayed
            Parallel(n_jobs=a.jobs)(delayed(run_one)(*args(s, seed)) for s, seed in jobs)
        else:
            for s, seed in jobs:
                run_one(*args(s, seed))
    res = aggregate(a.output_dir)
    fig_delta(res["delta"], a.output_dir / "figuras" / "fig_ganho_vs_media_regime.png")
    fig_power(res["bin"], a.output_dir / "figuras" / "fig_poder_faltas_sorteadas.png")
    with pd.option_context("display.width", 220, "display.max_columns", 20):
        print(res["delta"].round(2).sort_values(["Chart", "family", "Mode", "Estimator"]).to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/fault_injection.py
"""Blind fault-injection benchmark: do the detectors and RUL models react to a fault only WE know about?

WHAT: takes held-out (test) engines, replaces them by a healthy surrogate, injects a fault at a cycle
t0 chosen by us, and scores the unchanged detectors / RUL models, which only ever see sensor columns.
Three stages that talk ONLY through files, so the "answer" cannot leak into the models:

  generate : writes dados_cegos/*.csv (schema = raw C-MAPSS columns, nothing else) and the sealed
             answer key selado/manifesto_*.csv (+ its sha256 printed next to the blind data)
  detect   : reads ONLY dados_cegos/; fits detectors/models on TRAINING engines; writes alarms/predictions
  score    : reads the sealed manifest + the stage-2 outputs; writes tables and figures

TWO KINDS OF FAULT (they answer different questions)
  1. SYNTHETIC (detector power): step / ramp / one-regime / variance / sensor-bias, with magnitude as a
     fraction of the fleet's own end-of-life signature. The true RUL is not defined, so this only scores
     detection (power, delay) against a paired un-injected control (the SAME surrogate engine).
  2. TRANSPLANT (RUL models): the last N cycles of a DONOR test engine, in regime-standardized sensor
     units, are added to a recipient's healthy trajectory from t0 on. The donor's remaining life gives a
     true RUL by construction: label(c) = min(EOL - c, 125) with EOL = t0 + N - 1.

[LEAKAGE] detectors/models are fit on training engines only; donors and recipients are test engines;
the blind files hold no RUL, no scenario, no t0. [CAUSALITY] injection touches cycles >= t0 only, so a
control and its faulty twin are identical before t0 (checked by the tests).
[ASSUMPTION] healthy surrogate: the recipient's real cycles with true RUL >= 125 (natural degradation is
negligible there), extended by resampling its own healthy rows when the horizon is longer; this keeps
the settings/regime process but drops its temporal autocorrelation beyond the real prefix.
[ASSUMPTION] only the 13 MEWMA sensors are modified (never the exogenous settings/ambient channels).
Fault shapes are plausible, not physically simulated: "signature" = mean regime-standardized sensor
deviation of the last 10 cycles of the training engines.

Run:  .venv/bin/python -m pdm_cmaps.experiments.fault_injection --subset FD004 [--include-deep] [--stage all|generate|detect|score]
Output: results/injecao_de_falhas/<subset>_seed<seed>/{dados_cegos,selado,saidas,tabelas,figuras}
"""

from __future__ import annotations

from pdm_cmaps.config import DEFAULT_RESULTS_DIR

import argparse
import hashlib
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pdm_cmaps.domain.schema import COLUMNS, DEFAULT_HEALTHY_CYCLE_LIMIT, DEFAULT_RUL_CAP
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.data.features import CausalFeatureEngineer
from pdm_cmaps.domain.rul import compute_piecewise_rul
from pdm_cmaps.data.splits import split_engine_units
from pdm_cmaps.detection.virtual_sensor import CANONICAL_EXOGENOUS_FEATURES, VirtualSensor
from pdm_cmaps.detection.multivariate import MULTIVARIATE_SENSORS, MultivariateResidualMonitor
from pdm_cmaps.detection.regime import RegimeDecoupler
from pdm_cmaps.reporting import viz_style

OUT_DEFAULT = DEFAULT_RESULTS_DIR / "injecao_de_falhas"
INJECT_SENSORS = tuple(s for s in MULTIVARIATE_SENSORS if s not in CANONICAL_EXOGENOUS_FEATURES)
EOL_WINDOW = 10          # cycles used for the end-of-life signature
POST_CYCLES = 60         # observation window after t0 for the synthetic faults
RAMP_LEN = 40            # cycles to reach full magnitude for ramp-type faults
TRANSPLANT_N = 80        # donor cycles transplanted (= true remaining life at t0)
BIAS_SENSOR = "T50"
# magnitude = fraction of the EOL signature (signature kinds) or sigma units (variance / bias)
SCENARIOS = {
    "degrau": (0.05, 0.10, 0.25, 0.50),
    "rampa": (0.05, 0.10, 0.25, 0.50),
    "um_regime": (0.10, 0.25, 0.50, 1.00),
    "variancia": (0.5, 1.0, 2.0),
    "vies_sensor": (1.0, 2.0, 4.0),
}
DEFAULT_MODELS = ["Ridge", "LightGBM", "XGBoost", "LightGBM (+temporal)", "XGBoost (+temporal)"]
DEEP_MODELS = ["PyTorch CNN", "PyTorch RNN (LSTM)", "PyTorch Transformer"]
DETECTORS = ("T50 EWMA", "MEWMA (13 resíduos)")


# ============================================================================
# world: split, regimes, standardization, signature
# ============================================================================

@dataclass
class World:
    subset: str
    seed: int
    H: int
    train: pd.DataFrame
    test: pd.DataFrame
    regimes: RegimeDecoupler
    ref_mean: pd.DataFrame   # regime_id x INJECT_SENSORS (healthy training window)
    ref_std: pd.DataFrame
    signature: np.ndarray    # EOL deviation in regime-standardized units, one per INJECT_SENSORS
    main_regime: int


def load_world(subset: str, seed: int, H: int, loader: CMapssLoader | None = None) -> World:
    loader = loader or CMapssLoader()
    frame = loader.load(subset)
    frame["RUL"] = compute_piecewise_rul(frame)
    frame["rul_true"] = frame.groupby("unit_id")["cycle"].transform("max") - frame["cycle"]
    train_units, test_units = split_engine_units(frame, test_ratio=0.25, random_state=seed)
    train = frame.loc[frame["unit_id"].isin(train_units)]
    test = frame.loc[frame["unit_id"].isin(test_units)]
    n_regimes = 6 if subset.upper() in {"FD002", "FD004"} else 1
    regimes = RegimeDecoupler(n_clusters=n_regimes, random_state=seed).fit(train)
    tr, te = regimes.transform(train), regimes.transform(test)
    healthy = tr.loc[tr["cycle"] <= H]
    ref_mean = healthy.groupby("regime_id")[list(INJECT_SENSORS)].mean()
    ref_std = healthy.groupby("regime_id")[list(INJECT_SENSORS)].std().fillna(1.0).clip(lower=1e-8)
    world = World(subset, seed, H, tr, te, regimes, ref_mean, ref_std, np.zeros(len(INJECT_SENSORS)), 0)
    z = standardize(world, tr)
    base = z[(tr["cycle"] <= H).to_numpy()].mean(axis=0)
    end = z[(tr["rul_true"] <= EOL_WINDOW).to_numpy()].mean(axis=0)
    world.signature = end - base
    world.main_regime = int(tr["regime_id"].value_counts().idxmax())
    return world


def _lookup(table: pd.DataFrame, regime_id: np.ndarray) -> np.ndarray:
    fallback = table.mean()
    return table.reindex(regime_id).fillna(fallback).to_numpy(dtype=float)


def standardize(world: World, frame: pd.DataFrame) -> np.ndarray:
    """Regime-standardized INJECT_SENSORS (rows x sensors) using the healthy TRAINING window."""
    rid = frame["regime_id"].to_numpy(dtype=int)
    x = frame[list(INJECT_SENSORS)].to_numpy(dtype=float)
    return (x - _lookup(world.ref_mean, rid)) / _lookup(world.ref_std, rid)


# ============================================================================
# healthy surrogate + fault shapes
# ============================================================================

def healthy_surrogate(engine: pd.DataFrame, horizon: int, H: int, rng: np.random.Generator) -> pd.DataFrame | None:
    """Healthy trajectory of ``horizon`` cycles built from the engine's own cycles with true RUL >= cap."""
    healthy = engine.loc[engine["rul_true"] >= DEFAULT_RUL_CAP].sort_values("cycle")
    if len(healthy) < H + 10:
        return None
    if len(healthy) >= horizon:
        base = healthy.iloc[:horizon]
    else:
        pool = healthy.iloc[H // 2:]
        extra = pool.iloc[rng.integers(0, len(pool), horizon - len(healthy))]
        base = pd.concat([healthy, extra], ignore_index=True)
    base = base.reset_index(drop=True).copy()
    base["cycle"] = np.arange(1, horizon + 1)
    return base


def fault_profile(kind: str, cycles: np.ndarray, t0: int, ramp_len: int = RAMP_LEN) -> np.ndarray:
    """0 before t0; 1 after (degrau) or a linear ramp reaching 1 after ``ramp_len`` cycles."""
    on = cycles >= t0
    if kind == "degrau":
        return on.astype(float)
    return np.clip((cycles - t0 + 1) / ramp_len, 0.0, 1.0) * on


def delta_z(world: World, base: pd.DataFrame, kind: str, magnitude: float, t0: int, rng: np.random.Generator) -> np.ndarray:
    """Fault added to the INJECT_SENSORS, in regime-standardized units (rows x sensors)."""
    cycles = base["cycle"].to_numpy()
    n, k = len(base), len(INJECT_SENSORS)
    if kind in ("degrau", "rampa", "um_regime"):
        g = fault_profile("degrau" if kind == "degrau" else "rampa", cycles, t0)
        if kind == "um_regime":
            g = g * (base["regime_id"].to_numpy() == world.main_regime)
        return magnitude * g[:, None] * world.signature[None, :]
    out = np.zeros((n, k))
    on = (cycles >= t0)
    if kind == "variancia":
        out[on] = rng.normal(0.0, magnitude, (int(on.sum()), k))
    elif kind == "vies_sensor":
        out[on, INJECT_SENSORS.index(BIAS_SENSOR)] = magnitude
    else:
        raise ValueError(f"unknown fault kind {kind!r}")
    return out


def inject(world: World, base: pd.DataFrame, dz: np.ndarray) -> pd.DataFrame:
    """Return ``base`` with ``dz`` (standardized units) added to the INJECT_SENSORS in raw units."""
    out = base.copy()
    sd = _lookup(world.ref_std, base["regime_id"].to_numpy(dtype=int))
    out[list(INJECT_SENSORS)] = base[list(INJECT_SENSORS)].to_numpy(dtype=float) + dz * sd
    return out


def donor_tail(world: World, donor: pd.DataFrame, n_cycles: int) -> np.ndarray | None:
    """Last ``n_cycles`` of a donor in standardized units, minus the donor's own healthy-window mean."""
    donor = donor.sort_values("cycle")
    if len(donor) < n_cycles + world.H:
        return None
    z = standardize(world, donor)
    own_base = z[(donor["cycle"] <= world.H).to_numpy()].mean(axis=0)
    return z[-n_cycles:] - own_base


def blind_columns(frame: pd.DataFrame, unit_id: int) -> pd.DataFrame:
    """Only the raw C-MAPSS schema: no RUL, no regime, no scenario."""
    return frame.assign(unit_id=unit_id)[COLUMNS]


def manifest_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


# ============================================================================
# stage 1: generate
# ============================================================================

def generate(world: World, run_dir: Path, max_engines: int | None = None) -> dict:
    rng = np.random.default_rng(world.seed)
    t0 = world.H + 10
    horizon_det = t0 + POST_CYCLES
    horizon_rul = t0 + TRANSPLANT_N - 1  # EOL = last transplanted cycle
    engines = {int(u): g.sort_values("cycle") for u, g in world.test.groupby("unit_id")}
    recipients = [u for u in sorted(engines) if (engines[u]["rul_true"] >= DEFAULT_RUL_CAP).sum() >= world.H + 10]
    if max_engines:
        recipients = recipients[:max_engines]
    det_frames, det_meta, rul_frames, rul_meta = [], [], [], []
    det_id = rul_id = 0
    for u in recipients:
        base = healthy_surrogate(engines[u], horizon_det, world.H, rng)
        det_id += 1
        det_frames.append(blind_columns(base, det_id))
        det_meta.append({"blind_id": det_id, "recipient": u, "role": "controle", "kind": "", "magnitude": np.nan, "t0": t0})
        for kind, mags in SCENARIOS.items():
            if kind == "um_regime" and world.regimes.n_clusters < 2:
                continue
            for m in mags:
                det_id += 1
                det_frames.append(blind_columns(inject(world, base, delta_z(world, base, kind, m, t0, rng)), det_id))
                det_meta.append({"blind_id": det_id, "recipient": u, "role": "falha", "kind": kind, "magnitude": m, "t0": t0})
        base_r = healthy_surrogate(engines[u], horizon_rul, world.H, rng)
        donors = [d for d in sorted(engines) if d != u and len(engines[d]) >= TRANSPLANT_N + world.H]
        if not donors:
            continue
        donor = int(rng.choice(donors))
        tail = donor_tail(world, engines[donor], TRANSPLANT_N)
        dz = np.zeros((horizon_rul, len(INJECT_SENSORS)))
        dz[t0 - 1:] = tail
        rul_id += 1
        rul_frames.append(blind_columns(base_r, rul_id))
        rul_meta.append({"blind_id": rul_id, "recipient": u, "donor": -1, "role": "controle", "t0": t0, "eol_cycle": horizon_rul})
        rul_id += 1
        rul_frames.append(blind_columns(inject(world, base_r, dz), rul_id))
        rul_meta.append({"blind_id": rul_id, "recipient": u, "donor": donor, "role": "transplante", "t0": t0, "eol_cycle": horizon_rul})
    blind_dir, sealed_dir = run_dir / "dados_cegos", run_dir / "selado"
    blind_dir.mkdir(parents=True, exist_ok=True)
    sealed_dir.mkdir(parents=True, exist_ok=True)
    pd.concat(det_frames, ignore_index=True).to_csv(blind_dir / "cego_deteccao.csv", index=False)
    pd.concat(rul_frames, ignore_index=True).to_csv(blind_dir / "cego_rul.csv", index=False)
    pd.DataFrame(det_meta).to_csv(sealed_dir / "manifesto_deteccao.csv", index=False)
    pd.DataFrame(rul_meta).to_csv(sealed_dir / "manifesto_rul.csv", index=False)
    digest = {p.name: manifest_sha256(p) for p in sorted(sealed_dir.glob("manifesto_*.csv"))}
    (blind_dir / "manifesto_sha256.txt").write_text("".join(f"{h}  {n}\n" for n, h in digest.items()))
    print(f"[generate] {len(recipients)} recipients | {det_id} detector series | {rul_id} RUL series | t0={t0}", flush=True)
    return {"recipients": len(recipients), "t0": t0}


# ============================================================================
# stage 2: detect / predict (reads ONLY dados_cegos/)
# ============================================================================

def detect(world: World, run_dir: Path) -> pd.DataFrame:
    blind = pd.read_csv(run_dir / "dados_cegos" / "cego_deteccao.csv")
    tr, te = world.regimes.transform(world.train), world.regimes.transform(blind)
    vs = VirtualSensor(target_sensor="T50", mode="exogenous", ewma_alpha=0.10, min_consecutive_breaches=2,
                       healthy_cycle_limit=world.H, random_state=world.seed)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _, per_engine, _ = vs.evaluate(tr, te)
        # engine_offset=False de propósito: este estudo reporta a detecção no limite NOMINAL (sem falso alarme igualado), e a MEWMA com o deslocamento do motor
        # tem limite nominal mal calibrado (falso alarme nos controles de até 86 % em H = 25); só vale a falso alarme igualado (fault_detection_study).
        mv = MultivariateResidualMonitor(healthy_cycle_limit=world.H, random_state=world.seed, engine_offset=False).fit(tr)
        mv_alarms, _, _ = mv.alarms(mv.score(te))
    t50 = {int(r.unit_id): r.first_alarm_cycle for r in per_engine.itertuples()}
    out = pd.DataFrame({"blind_id": sorted(t50)})
    out[DETECTORS[0]] = out["blind_id"].map(t50)
    out[DETECTORS[1]] = out["blind_id"].map(mv_alarms)
    saidas = run_dir / "saidas"
    saidas.mkdir(parents=True, exist_ok=True)
    out.to_csv(saidas / "alarmes.csv", index=False)
    return out


def _model_recipes(names: list[str], seed: int):
    from pdm_cmaps.pipeline.extra_models import build_extra_models
    from pdm_cmaps.modeling.minirocket import MiniRocketRULModel
    from pdm_cmaps.modeling.registry import get_model_pool

    pool, extras = get_model_pool(seed), build_extra_models(seed)
    suffix = " (+temporal)"
    for name in names:
        if name in extras:
            yield name, extras[name][0], extras[name][1], False
        elif name in pool:
            yield name, "sklearn", pool[name], False
        elif name.endswith(suffix) and name[: -len(suffix)] in pool:
            yield name, "sklearn", pool[name[: -len(suffix)]], True
        elif name == "MiniRocket + RidgeCV":
            yield name, "deep", MiniRocketRULModel(seq_length=15, num_kernels=1000, random_state=seed), False
        else:
            print(f"[WARN] no recipe for model {name!r}; omitted", flush=True)


def predict_rul(world: World, run_dir: Path, names: list[str]) -> pd.DataFrame:
    from pdm_cmaps.pipeline.extra_models import fit_predict

    blind = pd.read_csv(run_dir / "dados_cegos" / "cego_rul.csv").sort_values(["unit_id", "cycle"]).reset_index(drop=True)
    loader = CMapssLoader()
    sensors = loader.informative_sensors(world.train)
    features = [*sensors, "setting_1", "setting_2", "setting_3", "cycle"]
    engineer = CausalFeatureEngineer(sensors, windows=(5, 10)).fit(world.train, world.H)
    fe_train, fe_blind = engineer.transform(world.train), engineer.transform(blind)
    fe_features = [*features, *[c for c in fe_train.columns if c not in world.train.columns]]
    out = blind[["unit_id", "cycle"]].rename(columns={"unit_id": "blind_id"}).copy()
    for name, m_type, model, temporal in _model_recipes(names, world.seed):
        print(f"[predict] {name}", flush=True)
        tr_f, ev_f, cols = (fe_train, fe_blind, fe_features) if temporal else (world.train, blind, features)
        pred = fit_predict(m_type, model, tr_f, cols, ev_f)
        out[name] = np.clip(pred, 0.0, float(DEFAULT_RUL_CAP))
    saidas = run_dir / "saidas"
    saidas.mkdir(parents=True, exist_ok=True)
    out.to_csv(saidas / "predicoes_rul.csv", index=False)
    return out


# ============================================================================
# stage 3: score (reads the sealed manifest)
# ============================================================================

def score_detection(manifest: pd.DataFrame, alarms: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Per-series outcome and per (kind, magnitude, detector) summary. Pre-t0 alarms are false alarms."""
    rows = []
    for det in DETECTORS:
        merged = manifest.merge(alarms[["blind_id", det]], on="blind_id").rename(columns={det: "alarm"})
        merged["Detector"] = det
        rows.append(merged)
    per = pd.concat(rows, ignore_index=True)
    per["has_alarm"] = per["alarm"].notna()
    per["false_alarm"] = per["has_alarm"] & (per["alarm"] < per["t0"])
    per["detected"] = per["has_alarm"] & (per["alarm"] >= per["t0"])
    per["delay"] = np.where(per["detected"], per["alarm"] - per["t0"], np.nan)
    ctrl = per[per["role"] == "controle"].groupby("Detector")["has_alarm"].mean().mul(100).rename("far_controle_pct")
    faults = per[per["role"] == "falha"]
    elig = faults[~faults["false_alarm"]]  # a series already alarmed before t0 cannot show its own detection
    g = elig.groupby(["kind", "magnitude", "Detector"])
    summary = g.agg(n=("detected", "size"), poder_pct=("detected", lambda s: 100 * s.mean()),
                    atraso_mediano=("delay", "median"), atraso_p90=("delay", lambda s: s.quantile(0.9))).reset_index()
    pre = faults.groupby(["kind", "magnitude", "Detector"])["false_alarm"].mean().mul(100).rename("pre_t0_alarme_pct").reset_index()
    summary = summary.merge(pre, on=["kind", "magnitude", "Detector"]).merge(ctrl.reset_index(), on="Detector")
    return per, summary


def _first_run_below(values: np.ndarray, cycles: np.ndarray, threshold: float, after: int, window: int = 3) -> int | None:
    smooth = pd.Series(values).rolling(window, min_periods=window).mean().to_numpy()
    hit = np.flatnonzero((smooth <= threshold) & (cycles > after))
    return int(cycles[hit[0]]) if hit.size else None


def score_rul(manifest: pd.DataFrame, preds: pd.DataFrame, H: int, alert_rul: float) -> tuple[pd.DataFrame, pd.DataFrame]:
    models = [c for c in preds.columns if c not in ("blind_id", "cycle")]
    rows = []
    for meta in manifest.itertuples():
        eng = preds[preds["blind_id"] == meta.blind_id].sort_values("cycle")
        cyc = eng["cycle"].to_numpy()
        truth = np.minimum(meta.eol_cycle - cyc, DEFAULT_RUL_CAP)
        fault = cyc >= meta.t0
        for model in models:
            p = eng[model].to_numpy()
            alert = _first_run_below(p, cyc, alert_rul, H)
            row = {"blind_id": meta.blind_id, "recipient": meta.recipient, "role": meta.role, "Model": model,
                   "alert_cycle": alert, "alert_before_t0": alert is not None and alert < meta.t0,
                   "alert_lead": (meta.eol_cycle - alert) if alert is not None and alert >= meta.t0 else np.nan,
                   "pred_min": float(p[cyc > H].min())}
            if meta.role == "transplante":
                row["RMSE_falha"] = float(np.sqrt(np.mean((p[fault] - truth[fault]) ** 2)))
                row["Bias_falha"] = float(np.mean(p[fault] - truth[fault]))
                for cp in (60, 30, 10):
                    j = np.flatnonzero(truth == cp)
                    row[f"pred_em_RUL{cp}"] = float(p[j[0]]) if j.size else np.nan
            rows.append(row)
    per = pd.DataFrame(rows)
    ctrl = per[per["role"] == "controle"].groupby("Model")["alert_cycle"].apply(lambda s: 100 * s.notna().mean()).rename("alerta_falso_controle_pct")
    tp = per[per["role"] == "transplante"]
    summary = tp.groupby("Model").agg(
        n=("blind_id", "size"), RMSE_falha=("RMSE_falha", "mean"), Bias_falha=("Bias_falha", "mean"),
        pred_RUL60=("pred_em_RUL60", "mean"), pred_RUL30=("pred_em_RUL30", "mean"), pred_RUL10=("pred_em_RUL10", "mean"),
        alerta_pct=("alert_lead", lambda s: 100 * s.notna().mean()), antecedencia_mediana=("alert_lead", "median"),
        alerta_antes_t0_pct=("alert_before_t0", lambda s: 100 * s.mean()),
    ).join(ctrl).reset_index().sort_values("RMSE_falha")
    return per, summary


# ============================================================================
# figures
# ============================================================================

def plot_detection_power(summary: pd.DataFrame, path: Path) -> None:
    viz_style.apply_style()
    kinds = [k for k in SCENARIOS if k in set(summary["kind"])]
    fig, axes = plt.subplots(2, len(kinds), figsize=(3.6 * len(kinds), 6.2), squeeze=False)
    colors = dict(zip(DETECTORS, viz_style.SERIES[:2]))
    for j, kind in enumerate(kinds):
        for det in DETECTORS:
            d = summary[(summary["kind"] == kind) & (summary["Detector"] == det)].sort_values("magnitude")
            axes[0, j].plot(d["magnitude"], d["poder_pct"], marker="o", color=colors[det], lw=viz_style.LINE_W, label=det)
            axes[1, j].plot(d["magnitude"], d["atraso_mediano"], marker="o", color=colors[det], lw=viz_style.LINE_W)
        unit = "× assinatura de fim de vida" if kind in ("degrau", "rampa", "um_regime") else "σ"
        axes[0, j].set_title(f"{kind.replace('_', ' ')}", fontsize=10)
        axes[1, j].set_xlabel(f"magnitude ({unit})", fontsize=8)
        axes[0, j].set_ylim(-3, 103)
        for i in range(2):
            viz_style.style_axes(axes[i, j])
    axes[0, 0].set_ylabel("poder de detecção (%)")
    axes[1, 0].set_ylabel("atraso mediano após t0 (ciclos)")
    axes[0, 0].legend(frameon=False, fontsize=8, loc="lower right")
    far = summary.drop_duplicates("Detector").set_index("Detector")["far_controle_pct"]
    fig.suptitle("Falhas injetadas (t0 conhecido só por nós): poder e atraso de detecção. Falso alarme nos controles: "
                 + " · ".join(f"{d} {far[d]:.1f} %" for d in DETECTORS), fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=140)
    plt.close(fig)
    summary.to_csv(path.with_suffix(".csv"), index=False)


NOMES_RUL_PT = {
    "XGBoost (NASA Loss)": "XGBoost (perda NASA)",
    "Support Vector Machine (SVM)": "SVM linear",
    "MiniRocket + RidgeCV": "MiniRocket + Ridge",
    "PyTorch RNN (LSTM)": "LSTM (PyTorch)",
    "PyTorch CNN": "CNN (PyTorch)",
    "PyTorch Transformer": "Transformer (PyTorch)",
}


def plot_rul_transplant(manifest: pd.DataFrame, preds: pd.DataFrame, path: Path, recipient: int | None = None) -> None:
    viz_style.apply_style()
    models = [c for c in preds.columns if c not in ("blind_id", "cycle")]
    recipient = recipient if recipient is not None else int(manifest["recipient"].iloc[0])
    pair = manifest[manifest["recipient"] == recipient].set_index("role")
    f_id, c_id = int(pair.loc["transplante", "blind_id"]), int(pair.loc["controle", "blind_id"])
    t0, eol = int(pair.loc["transplante", "t0"]), int(pair.loc["transplante", "eol_cycle"])
    fault, ctrl = (preds[preds["blind_id"] == i].sort_values("cycle") for i in (f_id, c_id))
    # grade de 3 colunas com uma célula livre para a legenda (5 modelos -> 2 x 3), legível na largura da página
    ncol = min(len(models) + 1, 3)
    nrow = int(np.ceil((len(models) + 1) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 2.9 * nrow), sharex=True, sharey=True, squeeze=False)
    cyc = fault["cycle"].to_numpy()
    truth = np.minimum(eol - cyc, DEFAULT_RUL_CAP)
    for ax, model in zip(axes.flat, models):
        ax.plot(cyc, truth, color=viz_style.INK, lw=1.6, label="RUL verdadeira (por construção)")
        ax.plot(ctrl["cycle"], ctrl[model], color=viz_style.MUTED, lw=1.4, ls="--", label="controle (sem falha)")
        ax.plot(cyc, fault[model], color=viz_style.SERIES[0], lw=1.6, label="previsto com falha")
        ax.axvline(t0, color=viz_style.STATUS[3], lw=1.4, ls=":")
        ax.set_title(NOMES_RUL_PT.get(model, model), fontsize=11, loc="left")
        ax.tick_params(labelsize=9.5)
        viz_style.style_axes(ax)
    for ax in list(axes.flat)[len(models):]:
        ax.axis("off")
    h, l = axes[0, 0].get_legend_handles_labels()
    h.append(plt.Line2D([], [], color=viz_style.STATUS[3], lw=1.4, ls=":"))
    l.append(f"início da falha (t0 = {t0},\noculto dos modelos)")
    list(axes.flat)[len(models)].legend(h, l, frameon=False, fontsize=10, loc="center")
    fig.suptitle(f"Transplante de degradação no motor receptor {recipient}", fontsize=12, x=0.01, ha="left")
    fig.supxlabel("ciclo", fontsize=11)
    fig.supylabel("RUL (ciclos)", fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=170)
    plt.close(fig)


def score(world: World, run_dir: Path, alert_rul: float) -> dict[str, pd.DataFrame]:
    sealed, out, tabs, figs = run_dir / "selado", run_dir / "saidas", run_dir / "tabelas", run_dir / "figuras"
    tabs.mkdir(parents=True, exist_ok=True)
    expected = (run_dir / "dados_cegos" / "manifesto_sha256.txt").read_text().split()
    for h, name in zip(expected[0::2], expected[1::2]):
        if manifest_sha256(sealed / name) != h:
            raise RuntimeError(f"sealed manifest {name} changed after generation")
    res = {}
    det_manifest = pd.read_csv(sealed / "manifesto_deteccao.csv")
    per, summ = score_detection(det_manifest, pd.read_csv(out / "alarmes.csv"))
    per.to_csv(tabs / "deteccao_por_serie.csv", index=False)
    summ.to_csv(tabs / "deteccao_resumo.csv", index=False)
    plot_detection_power(summ, figs / "fig_poder_deteccao.png")
    res["deteccao"] = summ
    rul_path = out / "predicoes_rul.csv"
    if rul_path.exists():
        rul_manifest, preds = pd.read_csv(sealed / "manifesto_rul.csv"), pd.read_csv(rul_path)
        per_r, summ_r = score_rul(rul_manifest, preds, world.H, alert_rul)
        per_r.to_csv(tabs / "rul_por_serie.csv", index=False)
        summ_r.to_csv(tabs / "rul_resumo.csv", index=False)
        plot_rul_transplant(rul_manifest, preds, figs / "fig_rul_transplante.png")
        res["rul"] = summ_r
    return res


def main(argv: list[str] | None = None) -> int:
    p = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    p.add_argument("--subset", default="FD004")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--healthy-cycle-limit", type=int, default=DEFAULT_HEALTHY_CYCLE_LIMIT)
    p.add_argument("--stage", choices=["all", "generate", "detect", "score"], default="all")
    p.add_argument("--include-deep", action="store_true", help="also CNN / LSTM / Transformer / MiniRocket (slow)")
    p.add_argument("--alert-rul", type=float, default=60.0, help="a model 'alerts' when its 3-cycle mean RUL <= this")
    p.add_argument("--max-engines", type=int, default=None)
    p.add_argument("--output-dir", type=Path, default=OUT_DEFAULT)
    a = p.parse_args(argv)
    subset = a.subset.upper()
    run_dir = a.output_dir / f"{subset}_seed{a.seed}"
    world = load_world(subset, a.seed, a.healthy_cycle_limit)
    if a.stage in ("all", "generate"):
        generate(world, run_dir, a.max_engines)
    if a.stage in ("all", "detect"):
        detect(world, run_dir)
        predict_rul(world, run_dir, DEFAULT_MODELS + (DEEP_MODELS if a.include_deep else []))
    if a.stage in ("all", "score"):
        res = score(world, run_dir, a.alert_rul)
        with pd.option_context("display.width", 220, "display.max_columns", 30):
            print(res["deteccao"].round(2).to_string(index=False))
            if "rul" in res:
                print(res["rul"].round(2).to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/fault_model_study.py
"""Which learning model should be the virtual sensor? Detection of injected faults per sensor and per estimator.

WHAT: the detectors compare each sensor (T50, Nc, P30, ...) with what a model expects from the operating
conditions. Here the model inside every virtual sensor is swapped (same engines, same injected faults, same
detector rule) and the power / delay of each chart is measured at a MATCHED false-alarm rate:
    Extra Trees (padrão) · Random Forest · LightGBM · XGBoost · Ridge · MLP · Média do regime (sem ML)
Charts scored per estimator: one EWMA chart per sensor (13 sensors, regime-conditioned, no raw-3sigma gate)
and the 13-residual MEWMA built from the same residuals.
WHY: virtual-sensor inputs are settings + near-constant ambient channels, so within a regime the model is close
to a regime mean. The "Média do regime (sem ML)" row tests that directly: if the learning models
do not beat it, the gain of the detector comes from the regime conditioning, not from the learner.

[LEAKAGE] estimators fit on TRAINING engines (healthy window H) only; injected series come from TEST engines
and reach the charts as raw columns; the answer key is used only by the scoring functions.
[ASSUMPTION] estimators are used with the repo defaults (no hyperparameter search); same series for all
estimators within a (subset, seed); false-alarm budget 0.2 per 100 monitored cycles.

Run:  .venv/bin/python -m pdm_cmaps.experiments.fault_model_study --H 50 [--subsets FD004] [--jobs 3]   /  --aggregate-only
Output: results/injecao_modelos/{runs,tabelas,figuras}
"""

from __future__ import annotations

from pdm_cmaps.config import DEFAULT_RESULTS_DIR

import argparse
import warnings
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.dummy import DummyRegressor

from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.detection.multivariate import MultivariateResidualMonitor
from pdm_cmaps.experiments.fault_detection_study import BUDGETS, analyse_detector, build_series
from pdm_cmaps.experiments.fault_injection import World, load_world
from pdm_cmaps.modeling.registry import get_model
from pdm_cmaps.reporting import viz_style

OUT_DEFAULT = DEFAULT_RESULTS_DIR / "injecao_modelos"
MEWMA = "MEWMA (13 resíduos)"
DIAGNOSTIC_ESTIMATORS = ["Mistura ET 25%", "Mistura ET 50%", "Extra Trees regularizado"]  # só para o diagnóstico do FD003 (ruído de estimação)
DEFAULT_ESTIMATORS = ["Extra Trees (padrão)", "Random Forest", "LightGBM", "XGBoost", "Ridge", "MLP", "Média do regime (sem ML)"]  # os do estudo principal
ESTIMATORS = [*DEFAULT_ESTIMATORS, *DIAGNOSTIC_ESTIMATORS]  # os aceitos na linha de comando
BASELINE = "Média do regime (sem ML)"


class BlendWithMean(BaseEstimator, RegressorMixin):
    """weight * base(x) + (1 - weight) * mean(y): encolhe um modelo aprendido na direção da média do regime (diagnóstico de ruído de estimação)."""

    def __init__(self, base=None, weight: float = 0.5):
        self.base = base
        self.weight = weight

    def fit(self, X, y):
        self.base_ = clone(self.base).fit(X, y)
        self.mean_ = float(np.mean(y))
        return self

    def predict(self, X):
        return self.weight * np.asarray(self.base_.predict(X), dtype=float) + (1.0 - self.weight) * self.mean_
KEY_FAULTS = (("degrau", 0.25), ("rampa", 0.25), ("variancia", 1.0), ("vies_sensor", 2.0))


def make_estimator(name: str, seed: int):
    """Fresh, untuned estimator with the repo defaults (same family/size as the benchmark pool)."""
    if name == "Extra Trees (padrão)":
        return get_model("extra_trees", seed, n_estimators=60, max_depth=12, n_jobs=1)
    if name in ("Mistura ET 25%", "Mistura ET 50%"):
        return BlendWithMean(get_model("extra_trees", seed, n_estimators=60, max_depth=12, n_jobs=1), 0.25 if "25" in name else 0.5)
    if name == "Extra Trees regularizado":
        return get_model("extra_trees", seed, n_estimators=60, max_depth=12, min_samples_leaf=100, n_jobs=1)
    if name == "Random Forest":
        return get_model("rf", seed, n_estimators=60, max_depth=12, n_jobs=1)
    if name == "LightGBM":
        return get_model("lightgbm", seed, n_estimators=80, max_depth=6, n_jobs=1)
    if name == "XGBoost":
        return get_model("xgboost", seed, n_estimators=80, max_depth=6, n_jobs=1)
    if name == "Ridge":
        return get_model("ridge", seed, alpha=10)
    if name == "MLP":
        return get_model("mlp", seed, hidden_layer_sizes=(64, 32), max_iter=180)
    if name == BASELINE:
        return DummyRegressor(strategy="mean")  # ignores the inputs: the regime mean
    raise ValueError(name)


def model_scores(mv: MultivariateResidualMonitor, te: pd.DataFrame, ids_control: set[int], H: int) -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    """Per-sensor EWMA ratios (regime-conditioned, no gate), the MEWMA ratio and the healthy residual sd on controls."""
    scores, zcols, resid = {}, {}, []
    index = None
    for sensor, vs in mv.virtual_.items():
        pred = vs.predict(te).sort_values(["unit_id", "cycle"]).reset_index(drop=True)
        index = pred[["unit_id", "cycle"]]
        zcols[sensor] = mv.residual_z(vs, pred)  # entra no T² (com o deslocamento do motor removido, se o monitor o usa)
        parts = []
        for uid, g in pred.assign(z=mv.chart_z(vs, pred)).groupby("unit_id", sort=True):
            e = vs.ewma_detector.ewma_abs_z(g["z"].to_numpy())
            parts.append(pd.DataFrame({"blind_id": uid, "cycle": g["cycle"].to_numpy(), "ratio": e / vs.ewma_detector.ewma_upper_limits(len(e)), "gate": True}))
        scores[sensor] = pd.concat(parts, ignore_index=True)
        ctrl = pred[pred["unit_id"].isin(ids_control) & (pred["cycle"] > H)]
        resid.append({"Detector": sensor, "resid_sd": float(ctrl["residual"].std())})
    out = index.copy()
    zmat = np.column_stack([zcols[s] for s in mv.virtual_])
    t2 = mv._t2(zmat, out["unit_id"].to_numpy())
    scores[MEWMA] = pd.DataFrame({"blind_id": out["unit_id"].to_numpy(), "cycle": out["cycle"].to_numpy(), "ratio": t2 / mv.ucl_, "gate": True})
    return scores, pd.DataFrame(resid)


def run_one(subset: str, seed: int, H: int, out: Path, n_ctrl: int = 6, n_rep: int = 2, max_engines: int | None = None, estimators: list[str] | None = None,
            mewma_offset: bool = True) -> None:
    tag = f"{subset}_seed{seed}_H{H}"
    path = out / "runs" / f"{tag}_pareado.csv"
    if path.exists():
        return
    world: World = load_world(subset, seed, H, loader=CMapssLoader())
    blind, meta = build_series(world, seed, n_ctrl, n_rep, max_engines)
    tr, te = world.regimes.transform(world.train), world.regimes.transform(blind)
    ctrl_ids = set(meta.loc[meta["role"] == "controle", "blind_id"].astype(int))
    rows, resid_rows = [], []
    for est in estimators or DEFAULT_ESTIMATORS:
        print(f"[{tag}] {est}", flush=True)
        mv = MultivariateResidualMonitor(healthy_cycle_limit=H, random_state=seed, estimator=make_estimator(est, seed), engine_offset=mewma_offset)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mv.fit(tr)
            scores, resid = model_scores(mv, te, ctrl_ids, H)
        resid_rows.append(resid.assign(Estimator=est))
        for name, sc in scores.items():
            rows.append(analyse_detector(name, sc, meta, H, BUDGETS)["pareado"].assign(Estimator=est))
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.concat(resid_rows).assign(Subset=subset, Seed=seed, H=H).to_csv(out / "runs" / f"{tag}_residuos.csv", index=False)
    pd.concat(rows).assign(Subset=subset, Seed=seed, H=H).to_csv(path, index=False)


def _load(out: Path, kind: str) -> pd.DataFrame:
    files = sorted((out / "runs").glob(f"*_{kind}.csv"))
    if not files:
        raise SystemExit(f"nenhum resultado em {out / 'runs'}: rode sem --aggregate-only")
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


def aggregate(out: Path) -> dict[str, pd.DataFrame]:
    tabs = out / "tabelas"
    tabs.mkdir(parents=True, exist_ok=True)
    par, res = _load(out, "pareado"), _load(out, "residuos")
    keys = ["Subset", "H", "Estimator", "Detector", "kind", "magnitude", "budget"]
    g = par.groupby(keys)[["power_pct", "delay_med", "far_ctrl_per100"]]
    summ = g.mean().add_suffix("_mean").join(g.std().add_suffix("_std")).reset_index()
    base = res[res["Estimator"] == BASELINE].set_index(["Subset", "Seed", "H", "Detector"])["resid_sd"]
    res = res.join(base.rename("resid_sd_baseline"), on=["Subset", "Seed", "H", "Detector"])
    res["resid_sd_ratio"] = res["resid_sd"] / res["resid_sd_baseline"]
    rsum = res.groupby(["Subset", "H", "Estimator", "Detector"])["resid_sd_ratio"].mean().reset_index()
    ok = par[par["power_pct"] >= 80.0]
    mdm_keys = ["Subset", "Seed", "H", "Estimator", "Detector", "kind", "budget"]
    mdm = ok.groupby(mdm_keys)["magnitude"].min().rename("mdm").reset_index()
    mdm = par[mdm_keys].drop_duplicates().merge(mdm, on=mdm_keys, how="left").groupby(["Subset", "H", "Estimator", "Detector", "kind", "budget"])["mdm"].median().reset_index()
    summ.to_csv(tabs / "modelos_pareado_resumo.csv", index=False)
    rsum.to_csv(tabs / "modelos_residuo_vs_media_regime.csv", index=False)
    mdm.to_csv(tabs / "modelos_magnitude_minima.csv", index=False)
    return {"resumo": summ, "residuo": rsum, "mdm": mdm}


def fig_mewma_by_estimator(summ: pd.DataFrame, path: Path, H: int) -> None:
    viz_style.apply_style()
    subsets = sorted(set(summ["Subset"]))
    fig, axes = plt.subplots(len(KEY_FAULTS), len(subsets), figsize=(3.6 * len(subsets), 2.7 * len(KEY_FAULTS)), squeeze=False, sharey=True)
    for i, (kind, mag) in enumerate(KEY_FAULTS):
        for j, sub in enumerate(subsets):
            d = summ[(summ["Subset"] == sub) & (summ["Detector"] == MEWMA) & (summ["kind"] == kind) & np.isclose(summ["magnitude"], mag) & (summ["H"] == H)].set_index("Estimator").reindex(DEFAULT_ESTIMATORS)
            colors = [viz_style.INK_2 if e == BASELINE else viz_style.SERIES[1] for e in DEFAULT_ESTIMATORS]
            axes[i, j].barh(range(len(DEFAULT_ESTIMATORS)), d["power_pct_mean"], xerr=d["power_pct_std"].fillna(0), color=colors, height=0.65, capsize=2)
            axes[i, j].set_yticks(range(len(DEFAULT_ESTIMATORS)))
            axes[i, j].set_yticklabels(DEFAULT_ESTIMATORS if j == 0 else [], fontsize=8)
            axes[i, j].invert_yaxis()
            axes[i, j].set_xlim(0, 105)
            viz_style.style_axes(axes[i, j], grid="x")
            if i == 0:
                axes[i, j].set_title(sub, fontsize=10)
            if j == 0:
                axes[i, j].set_ylabel(f"{kind} {mag:g}", fontsize=9)
            if i == len(KEY_FAULTS) - 1:
                axes[i, j].set_xlabel("poder de detecção (%)", fontsize=8)
    fig.suptitle(f"MEWMA com cada modelo como sensor virtual (H = {H}, falso alarme ≤ 0,2 por 100 ciclos; cinza = média do regime, sem ML)", fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=140)
    plt.close(fig)


def fig_sensor_heatmap(summ: pd.DataFrame, path: Path, H: int, kind: str = "degrau", mag: float = 0.25) -> None:
    viz_style.apply_style()
    subsets = sorted(set(summ["Subset"]))
    fig, axes = plt.subplots(1, len(subsets), figsize=(4.6 * len(subsets), 5.2), squeeze=False)
    for ax, sub in zip(axes[0], subsets):
        d = summ[(summ["Subset"] == sub) & (summ["kind"] == kind) & np.isclose(summ["magnitude"], mag) & (summ["H"] == H) & (summ["Detector"] != MEWMA)]
        p = d.pivot(index="Detector", columns="Estimator", values="power_pct_mean").reindex(columns=DEFAULT_ESTIMATORS)
        im = ax.imshow(p.values, vmin=0, vmax=100, cmap="Blues", aspect="auto")
        ax.set_xticks(range(len(DEFAULT_ESTIMATORS)))
        ax.set_xticklabels([e.replace(" (padrão)", "").replace(" (sem ML)", "") for e in DEFAULT_ESTIMATORS], rotation=40, ha="right", fontsize=8)
        ax.set_yticks(range(len(p.index)))
        ax.set_yticklabels(p.index, fontsize=8)
        for (r, c), v in np.ndenumerate(p.values):
            if np.isfinite(v):
                ax.text(c, r, f"{v:.0f}", ha="center", va="center", fontsize=7, color="white" if v > 55 else viz_style.INK)
        ax.set_title(sub, fontsize=10)
    fig.colorbar(im, ax=axes[0].tolist(), shrink=0.6, label="poder (%)")
    fig.suptitle(f"Poder de detecção por sensor e modelo ({kind} {mag:g}, H = {H}, falso alarme ≤ 0,2 por 100 ciclos)", fontsize=10)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=140, bbox_inches="tight")
    plt.close(fig)


def fig_residual_ratio(rsum: pd.DataFrame, path: Path, H: int) -> None:
    viz_style.apply_style()
    d = rsum[(rsum["H"] == H) & (rsum["Detector"] != MEWMA)].groupby(["Subset", "Estimator"])["resid_sd_ratio"].median().unstack("Estimator").reindex(columns=DEFAULT_ESTIMATORS)
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    w = 0.8 / len(d.index)
    for k, sub in enumerate(d.index):
        ax.bar(np.arange(len(DEFAULT_ESTIMATORS)) + k * w, d.loc[sub], w, label=sub, color=[*viz_style.SERIES, viz_style.INK_2][k % 4])
    ax.axhline(1.0, color=viz_style.INK, lw=1)
    ax.set_xticks(np.arange(len(DEFAULT_ESTIMATORS)) + 0.4 - w / 2)
    ax.set_xticklabels([e.replace(" (padrão)", "").replace(" (sem ML)", "") for e in DEFAULT_ESTIMATORS], rotation=25, ha="right", fontsize=8)
    ax.set_ylabel("desvio do resíduo saudável / o da média do regime\n(mediana dos sensores; < 1 = modelo melhor que a média)")
    viz_style.style_axes(ax)
    ax.legend(frameon=False, fontsize=8)
    ax.set_title(f"O modelo prevê o sensor melhor que a média do regime? (controles saudáveis, H = {H})", fontsize=10)
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=140)
    plt.close(fig)


def make_figures(res: dict[str, pd.DataFrame], out: Path, H: int) -> None:
    fig_mewma_by_estimator(res["resumo"], out / "figuras" / "fig_mewma_por_modelo.png", H)
    fig_sensor_heatmap(res["resumo"], out / "figuras" / "fig_sensor_x_modelo.png", H)
    fig_residual_ratio(res["residuo"], out / "figuras" / "fig_residuo_vs_media_regime.png", H)


def main(argv: list[str] | None = None) -> int:
    p = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    p.add_argument("--H", type=int, required=True, help="healthy window (use the best one from fault_detection_study)")
    p.add_argument("--subsets", nargs="+", default=["FD001", "FD002", "FD003", "FD004"])
    p.add_argument("--seeds", nargs="+", type=int, default=[42, 7, 123])
    p.add_argument("--estimators", nargs="+", default=None, choices=ESTIMATORS)
    p.add_argument("--n-ctrl", type=int, default=6)
    p.add_argument("--n-rep", type=int, default=2)
    p.add_argument("--max-engines", type=int, default=None)
    p.add_argument("--jobs", type=int, default=3)
    p.add_argument("--output-dir", type=Path, default=None, help=f"padrão: {OUT_DEFAULT} (protocolo principal, com o deslocamento do motor em todos os detectores); --no-mewma-offset grava em {OUT_DEFAULT}_offset_so_t50")
    p.add_argument("--mewma-offset", action=argparse.BooleanOptionalAction, default=True, help="(padrão ligado desde 2026-11-18; --no-mewma-offset reproduz o protocolo anterior, só o T50) deslocamento do motor (por motor e regime) removido do z de todos os sensores, no T² e nos gráficos por sensor")
    p.add_argument("--aggregate-only", action="store_true")
    a = p.parse_args(argv)
    if a.output_dir is None:
        a.output_dir = OUT_DEFAULT if a.mewma_offset else OUT_DEFAULT.with_name(OUT_DEFAULT.name + "_offset_so_t50")
    if not a.aggregate_only:
        jobs = [(s, seed) for s in a.subsets for seed in a.seeds]
        if a.jobs > 1:
            from joblib import Parallel, delayed
            Parallel(n_jobs=a.jobs)(delayed(run_one)(s, seed, a.H, a.output_dir, a.n_ctrl, a.n_rep, a.max_engines, a.estimators, a.mewma_offset) for s, seed in jobs)
        else:
            for s, seed in jobs:
                run_one(s, seed, a.H, a.output_dir, a.n_ctrl, a.n_rep, a.max_engines, a.estimators, a.mewma_offset)
    res = aggregate(a.output_dir)
    make_figures(res, a.output_dir, a.H)
    s = res["resumo"]
    top = s[(s["H"] == a.H) & (s["Detector"] == MEWMA) & (s["kind"] == "degrau") & np.isclose(s["magnitude"], 0.25)]
    with pd.option_context("display.width", 200):
        print(top.pivot(index="Estimator", columns="Subset", values="power_pct_mean").reindex(DEFAULT_ESTIMATORS).round(1).to_string())
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/healthy_window_study.py
"""Healthy-window sensitivity study: how many initial cycles should calibrate the SPC baseline?

WHAT: repeats the T50 virtual-sensor EWMA chart and the 13-residual MEWMA chart for several
healthy windows H (50, 75, 100 and a relative window), over subsets x seeds, on the held-out
engines of the same seeded engine split as monitoring/layer.py / pipeline.run().

WHY: a longer window should give less noisy control limits (more calibration rows) and fewer false
alarms, but (i) monitoring only starts after cycle H (blind period) and (ii) the calibration set
starts to include early degradation, which inflates sigma / the UCL and delays detection.

[ASSUMPTION] "Relative" window: 25 % of each engine's own life would use the end of life of the
engine being monitored (unknown online). The causal version used here is
    H_rel = round(0.25 x median life of the TRAINING engines)   (one integer per subset and seed).

Metrics that do NOT depend on H (the detector's own `healthy_far` / `premature_alarm` are measured
on cycles <= H, so they change window and denominator with H and cannot be compared across H):
  * early_alarm_pct     first alarm at a cycle whose true RUL (EOL - cycle) >= 125 (piecewise cap)
  * far_ref_pct         per-cycle breach rate on monitored cycles with RUL >= 125
  * far_common_pct      same, restricted to cycles > 100 so every H is scored on the SAME cycles
  * detection, lead time (EOL - first alarm), blind period (= H), engines with EOL <= H + k
  * contamination_pct   % of training calibration rows (cycle <= H) with true RUL < 125
  * calibration noise   engine-cluster bootstrap CV of the T50 per-regime residual sigma and of
                        the MEWMA UCL (quantile of OOF healthy T2, resampled by engine)

[LEAKAGE] everything is fit on training engines only; RUL_true is used only for scoring.
[GROUPED-CV] bootstrap resamples whole engines.

Run:  .venv/bin/python -m pdm_cmaps.experiments.healthy_window_study            (4 subsets x 3 seeds x 4 windows)
      .venv/bin/python -m pdm_cmaps.experiments.healthy_window_study --aggregate-only
Output: results/healthy_window_study/{runs,tables,figures}
"""

from __future__ import annotations

from pdm_cmaps.config import DEFAULT_RESULTS_DIR

import argparse
import warnings
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pdm_cmaps.domain.schema import DEFAULT_RUL_CAP
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.data.splits import split_engine_units
from pdm_cmaps.detection.multivariate import MultivariateResidualMonitor
from pdm_cmaps.detection.regime import RegimeDecoupler
from pdm_cmaps.detection.virtual_sensor import VirtualSensor
from pdm_cmaps.reporting import viz_style

OUT_DEFAULT = DEFAULT_RESULTS_DIR / "healthy_window_study"
FIXED_WINDOWS = (50, 75, 100)
REL_FRACTION = 0.25
COMMON_START = 100  # far_common_pct is scored only on cycles > this (largest fixed H)
N_BOOT = 200


def _windows(train_df: pd.DataFrame) -> dict[str, int]:
    life = train_df.groupby("unit_id")["cycle"].max()
    h_rel = int(round(REL_FRACTION * float(life.median())))
    return {**{f"H={h}": h for h in FIXED_WINDOWS}, f"H=25% vida (={h_rel})": h_rel}


def _cluster_boot_cv(values_by_unit: pd.DataFrame, stat, rng: np.random.Generator) -> tuple[float, float]:
    """Engine-cluster bootstrap of ``stat(frame)``; returns (point estimate, coefficient of variation %)."""
    units = values_by_unit["unit_id"].unique()
    groups = {u: g for u, g in values_by_unit.groupby("unit_id")}
    point = float(stat(values_by_unit))
    reps = []
    for _ in range(N_BOOT):
        pick = rng.choice(units, size=units.size, replace=True)
        reps.append(stat(pd.concat([groups[u] for u in pick], ignore_index=True)))
    reps = np.asarray(reps, dtype=float)
    return point, float(100 * reps.std(ddof=1) / abs(reps.mean())) if reps.mean() else float("nan")


def _reference_metrics(per_cycle: pd.DataFrame, first_alarm: dict, H: int, k: int) -> dict:
    """H-independent false-alarm metrics. ``per_cycle``: unit_id, cycle, breach, rul_true."""
    pc = per_cycle.copy()
    mon = pc["cycle"] > H
    ref = mon & (pc["rul_true"] >= DEFAULT_RUL_CAP)
    common = (pc["cycle"] > COMMON_START) & (pc["rul_true"] >= DEFAULT_RUL_CAP)
    eol = pc.groupby("unit_id")["cycle"].max()
    early = [a is not None and (eol[u] - a) >= DEFAULT_RUL_CAP for u, a in first_alarm.items()]
    engines_with_ref = int(pc.loc[ref, "unit_id"].nunique())
    return {
        "early_alarm_pct": 100 * float(np.mean(early)),
        "far_ref_pct": 100 * float(pc.loc[ref, "breach"].mean()) if ref.any() else float("nan"),
        "far_common_pct": 100 * float(pc.loc[common, "breach"].mean()) if common.any() else float("nan"),
        "engines_with_ref_cycles": engines_with_ref,
        "engines_unmonitorable": int((eol <= H + k).sum()),
        "monitored_cycles_pct": 100 * float(mon.mean()),
    }


def run_one(subset: str, seed: int, out: Path, mewma_offset: bool = True) -> pd.DataFrame:
    """All windows for one (subset, seed); writes runs/<subset>_seed<seed>.csv and returns it."""
    path = out / "runs" / f"{subset}_seed{seed}.csv"
    if path.exists():
        return pd.read_csv(path)
    loader = CMapssLoader()
    frame = loader.load(subset)
    frame["rul_true"] = frame.groupby("unit_id")["cycle"].transform("max") - frame["cycle"]
    train_units, test_units = split_engine_units(frame, test_ratio=0.25, random_state=seed)
    train_df = frame.loc[frame["unit_id"].isin(train_units)]
    test_df = frame.loc[frame["unit_id"].isin(test_units)]
    n_regimes = 6 if subset.upper() in {"FD002", "FD004"} else 1
    regimes = RegimeDecoupler(n_clusters=n_regimes, random_state=seed).fit(train_df)
    tr, te = regimes.transform(train_df), regimes.transform(test_df)
    rul_lookup = te.set_index(["unit_id", "cycle"])["rul_true"]
    rng = np.random.default_rng(seed)
    rows = []
    for label, H in _windows(train_df).items():
        print(f"[window] {subset} seed={seed} {label}", flush=True)
        calib = tr.loc[tr["cycle"] <= H]
        base = {"Subset": subset, "Seed": seed, "Window": label, "H": H,
                "calibration_rows": int(len(calib)),
                "contamination_pct": 100 * float((calib["rul_true"] < DEFAULT_RUL_CAP).mean())}

        # --- T50 univariate chart (same config as monitoring.compute_fleet_alarms) ---
        vs = VirtualSensor(target_sensor="T50", mode="exogenous", ewma_alpha=0.10, min_consecutive_breaches=2,
                           healthy_cycle_limit=H, random_state=seed)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            trace, per_engine, summary = vs.evaluate(tr, te)
        alarms = {int(r.unit_id): (None if pd.isna(r.first_alarm_cycle) else int(r.first_alarm_cycle)) for r in per_engine.itertuples()}
        breach = (trace["ewma_abs_z"] >= trace["ewma_ucl"]) & (trace["z_score"].abs() >= vs.threshold_sigma)
        pc = trace[["unit_id", "cycle"]].assign(breach=breach.to_numpy())
        pc["rul_true"] = rul_lookup.reindex(pd.MultiIndex.from_frame(pc[["unit_id", "cycle"]])).to_numpy()
        cal = vs.calibration_predictions_[["unit_id", "regime_id", "residual"]]
        sigma, sigma_cv = _cluster_boot_cv(cal, lambda f: f.groupby("regime_id")["residual"].std().mean(), rng)
        rows.append({**base, "Detector": "T50 EWMA", **summary, **_reference_metrics(pc, alarms, H, 2),
                     "calib_stat": sigma, "calib_stat_name": "sigma_residuo_T50", "calib_cv_pct": sigma_cv})

        # --- MEWMA on 13 residuals ---
        mv = MultivariateResidualMonitor(healthy_cycle_limit=H, random_state=seed, engine_offset=mewma_offset).fit(tr)
        scored = mv.score(te)
        mv_alarms, _, mv_summary = mv.alarms(scored)
        pc = scored[["unit_id", "cycle", "breach"]].copy()
        pc["rul_true"] = rul_lookup.reindex(pd.MultiIndex.from_frame(pc[["unit_id", "cycle"]])).to_numpy()
        ucl, ucl_cv = _cluster_boot_cv(mv.calibration_t2_, lambda f: np.quantile(f["t2"], 1 - mv.alpha), rng)
        mv_summary.pop("n_sensors", None)
        rows.append({**base, "Detector": "MEWMA (13 resíduos)", **mv_summary, **_reference_metrics(pc, mv_alarms, H, mv.k),
                     "calib_stat": ucl, "calib_stat_name": "UCL_MEWMA", "calib_cv_pct": ucl_cv})
    df = pd.DataFrame(rows)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return df


METRICS = ["H", "calibration_rows", "contamination_pct", "calib_stat", "calib_cv_pct", "detection_rate_pct",
           "mean_lead_time_detected_cycles", "early_alarm_pct", "far_ref_pct", "far_common_pct",
           "mean_healthy_false_alarm_pct", "premature_alarm_pct", "engines_unmonitorable", "monitored_cycles_pct"]


def aggregate(out: Path) -> pd.DataFrame:
    runs = pd.concat([pd.read_csv(p) for p in sorted((out / "runs").glob("*.csv"))], ignore_index=True)
    runs["Window_kind"] = runs["Window"].str.replace(r" \(=\d+\)", "", regex=True)
    g = runs.groupby(["Subset", "Detector", "Window_kind"], sort=False)[METRICS]
    summary = g.mean().add_suffix("_mean").join(g.std().add_suffix("_std")).reset_index()
    summary["n_seeds"] = g.size().to_numpy()
    tables = out / "tables"
    tables.mkdir(parents=True, exist_ok=True)
    runs.to_csv(tables / "healthy_window_runs.csv", index=False)
    summary.to_csv(tables / "healthy_window_summary.csv", index=False)
    plot_tradeoff(summary, out / "figures" / "fig_healthy_window_tradeoff.png")
    return summary


def plot_tradeoff(summary: pd.DataFrame, path: Path) -> None:
    """Rows = detector, columns = metric; one line per subset, x = window (ordered by mean H)."""
    viz_style.apply_style()
    panels = [("calib_cv_pct", "Ruído de calibração\n(CV bootstrap, %)"),
              ("contamination_pct", "Contaminação da calibração\n(% linhas com RUL < 125)"),
              ("far_common_pct", "Falso alarme por ciclo\n(ciclos > 100 e RUL ≥ 125, %)"),
              ("early_alarm_pct", "Motores com alarme precoce\n(1º alarme com RUL ≥ 125, %)"),
              ("mean_lead_time_detected_cycles", "Antecedência média\n(ciclos antes da falha)"),
              ("detection_rate_pct", "Taxa de detecção (%)")]
    detectors = list(dict.fromkeys(summary["Detector"]))
    subsets = list(dict.fromkeys(summary["Subset"]))
    colors = dict(zip(subsets, [*viz_style.SERIES, viz_style.INK_2]))
    markers = dict(zip(subsets, ["o", "s", "^", "D"]))
    fig, axes = plt.subplots(len(detectors), len(panels), figsize=(4.0 * len(panels), 3.6 * len(detectors)), squeeze=False)
    for i, det in enumerate(detectors):
        for j, (col, title) in enumerate(panels):
            ax = axes[i, j]
            for sub in subsets:
                d = summary[(summary["Detector"] == det) & (summary["Subset"] == sub)].sort_values("H_mean")
                ax.errorbar(d["H_mean"], d[f"{col}_mean"], yerr=d[f"{col}_std"], color=colors[sub], marker=markers[sub],
                            lw=viz_style.LINE_W, ms=6, capsize=3, label=sub)
            viz_style.style_axes(ax)
            ax.set_title(title if i == 0 else "", fontsize=10)
            ax.set_xticks(sorted(set(FIXED_WINDOWS)))
            if j == 0:
                ax.set_ylabel(det, fontsize=10, fontweight="bold")
            if i == len(detectors) - 1:
                ax.set_xlabel("Janela saudável H (ciclos)")
    axes[0, 0].legend(frameon=False, fontsize=8)
    fig.suptitle("Janela saudável: ruído de calibração × contaminação × falso alarme × antecedência "
                 "(média ± dp de 3 seeds; ponto não inteiro em x = janela relativa 25 % da vida mediana)", fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150)
    plt.close(fig)
    summary.to_csv(path.with_suffix(".csv"), index=False)


def main(argv: list[str] | None = None) -> int:
    p = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    p.add_argument("--subsets", nargs="+", default=["FD001", "FD002", "FD003", "FD004"])
    p.add_argument("--seeds", nargs="+", type=int, default=[42, 7, 123])
    p.add_argument("--jobs", type=int, default=4)
    p.add_argument("--output-dir", type=Path, default=None, help=f"padrão: {OUT_DEFAULT} (protocolo principal, com o deslocamento do motor em todos os detectores); --no-mewma-offset grava em {OUT_DEFAULT}_offset_so_t50")
    p.add_argument("--mewma-offset", action=argparse.BooleanOptionalAction, default=True, help="(padrão ligado desde 2026-11-18; --no-mewma-offset reproduz o protocolo anterior, só o T50) deslocamento do motor (janela = H de cada linha) também na MEWMA; o T50 EWMA já o usa por padrão; falso alarme fora da janela: far_ref_pct e far_common_pct")
    p.add_argument("--aggregate-only", action="store_true")
    a = p.parse_args(argv)
    if a.output_dir is None:
        a.output_dir = OUT_DEFAULT if a.mewma_offset else OUT_DEFAULT.with_name(OUT_DEFAULT.name + "_offset_so_t50")
    if not a.aggregate_only:
        jobs = [(s, seed) for s in a.subsets for seed in a.seeds]
        if a.jobs > 1:
            from joblib import Parallel, delayed
            Parallel(n_jobs=a.jobs)(delayed(run_one)(s, seed, a.output_dir, a.mewma_offset) for s, seed in jobs)
        else:
            for s, seed in jobs:
                run_one(s, seed, a.output_dir, a.mewma_offset)
    summary = aggregate(a.output_dir)
    with pd.option_context("display.width", 250, "display.max_columns", 40):
        cols = ["Subset", "Detector", "Window_kind", "H_mean", "contamination_pct_mean", "calib_cv_pct_mean",
                "far_common_pct_mean", "early_alarm_pct_mean", "mean_lead_time_detected_cycles_mean", "detection_rate_pct_mean"]
        print(summary[cols].round(2).to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/mewma_calibracao_sintetica.py
"""Calibração do limite do T² da MEWMA com séries saudáveis sintéticas: ela corrige o falso alarme nominal?

PROBLEMA (protocolo principal): o limite é o quantil (1 − α) do T² medido nos próprios ciclos da janela, já com o deslocamento do motor tirado
DELA, onde a média do z é zero por construção. O T² sai subestimado, o limite fica baixo e, depois da janela, o erro de estimar o deslocamento
faz o falso alarme nominal subir (0,20 a 2,64 por 100 ciclos contra 0,01 a 0,06 sem a correção).

TRATAMENTOS (MEWMA de 13 resíduos, mesma divisão por motor e mesmos regimes da camada de monitoramento, H = 50):
    H, limite empírico                  deslocamento nos 50 primeiros ciclos; limite do protocolo principal
    H, limite sintético                 o mesmo ajuste, só o limite muda (calibration="synthetic", W = 50)
    janela 25, empírico dentro          deslocamento nos 25 primeiros ciclos, limite medido nos ciclos 1 a 50
    janela 25, empírico fora            idem, mas μ, Σ e limite só nos ciclos 26 a 50 (a opção que já existia)
    janela 25, limite sintético         W = 25 com o limite sintético

VALIDAÇÃO NÃO CIRCULAR (critério revisado *post hoc*, com o motivo abaixo): os controles independentes (reamostragem iid dos ciclos saudáveis) compartilham a premissa da calibração sintética e
passariam por construção. A validação é feita em DADOS REAIS: nos tratamentos de janela 25, os ciclos 26 a 50 dos motores de teste são saudáveis
pelo protocolo e NÃO entraram na estimativa do deslocamento. Para α em {0,05; 0,01; 0,0027} mede-se a fração desses ciclos acima do limite do
próprio α (curva de calibração): calibrado = fração observada perto de α. Critério fixado ANTES de rodar: razão observado/nominal entre 0,5 e 2
em α = 0,05 e α = 0,01, em cada subconjunto, para o limite sintético; e mais perto de 1 (em escala log) que o melhor limite empírico em ao menos
3 dos 4 subconjuntos. Em α = 0,0027 o número de ciclos é pequeno (poucos eventos esperados): só se reporta, com IC por bootstrap de motores.
REVISÃO POST HOC (2026-11-18, depois do primeiro resultado, FD001 semente 42): o limite sintético pooleia 150 ciclos pós-janela, dominados pelo
regime estacionário, e saiu 55% a 76% acima do empírico, com só 0,3% dos ciclos reais 26 a 50 acima dele em α = 0,05. O diagnóstico mostrou que a
média do T² sintético (21,9) bate com a teoria para ruído i.i.d. (22,9), mas a média real nesses ciclos é 17,0: os ciclos 26 a 50 são o INÍCIO depois
da janela de 25, em que a EWMA (λ = 0,10) ainda carrega a memória da janela (z centrado por construção) e só chega ao estacionário depois de 20 a
30 passos. Validar um transitório contra um limite estacionário reprova um gerador correto. Por isso o estudo reporta DOIS limites sintéticos:
    horizonte   quantil dos 150 ciclos pós-janela: o que o monitoramento longo usa (calibra o falso alarme médio por ciclo ao longo do horizonte)
    casado      quantil só dos mesmos ciclos pós-janela da validação (1 a H − W): compara igual com igual e testa o GERADOR em dados reais
O critério (razão observado/nominal entre 0,5 e 2 em α = 0,05 e 0,01, em cada subconjunto, e mais perto de 1 que o melhor empírico em ao menos 3 de 4)
passa a valer para o limite CASADO; o limite do horizonte é reportado e se espera mais folgado nesses ciclos iniciais. A mudança foi feita
depois de ver o resultado e está registrada aqui por isso.

Nos tratamentos de janela H os ciclos 26 a 50 ESTÃO dentro da janela do deslocamento (centrados nela): esses números não validam nada e não entram.

O QUE ISTO NÃO RESOLVE: calibrar o limite conserta o falso alarme nominal. A antecedência nativa continua sem ser verdade-terreno, porque RUL ≥ 125
não é um rótulo de saúde (a degradação inicial real e o falso alarme se misturam depois da janela).

Run:  .venv/bin/python -m pdm_cmaps.experiments.mewma_calibracao_sintetica [--subsets FD001 ...] [--seeds 42 7 123]
Saída: results/mewma_calibracao_sintetica/{runs/*.csv,motores/*.csv,por_semente.csv,curva_calibracao.csv}
"""
from __future__ import annotations

import argparse
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from pdm_cmaps.config import DEFAULT_RESULTS_DIR
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.data.splits import split_engine_units
from pdm_cmaps.detection.multivariate import MultivariateResidualMonitor
from pdm_cmaps.detection.regime import RegimeDecoupler
from pdm_cmaps.experiments.native_offset_fora_janela import (CTRL_ID0, H_NATIVE, W_SPLIT, _control_far, _controls, _late_metrics)

OUT_DEFAULT = DEFAULT_RESULTS_DIR / "mewma_calibracao_sintetica"
ALPHAS = (0.05, 0.01, 0.0027)
# (rótulo, offset_window, calibrate_out_of_window, qual limite: "empirical" | "synthetic", vale como validação em dados reais?)
TREATMENTS = (
    ("H, limite empírico", None, False, "empirical", False),
    ("H, limite sintético", None, False, "synthetic", False),
    ("janela 25, empírico dentro", W_SPLIT, False, "empirical", True),
    ("janela 25, empírico fora", W_SPLIT, True, "empirical", True),
    ("janela 25, limite sintético", W_SPLIT, False, "synthetic", True),
)
FITS = ((None, False), (W_SPLIT, False), (W_SPLIT, True))  # (offset_window, calibrate_out_of_window): três ajustes cobrem os cinco tratamentos


def _ucl_for(mv: MultivariateResidualMonitor, which: str, alpha: float, n_post: int | None = None) -> float:
    """Limite para `alpha`. `n_post` (só no sintético) restringe aos primeiros n ciclos pós-janela (limite "casado")."""
    if which == "synthetic":
        sample = mv.synthetic_t2_ if n_post is None else mv.synthetic_t2_cycles_[:, :n_post].ravel()
        return float(np.quantile(sample, 1 - alpha))
    return float(np.quantile(mv.calibration_t2_["t2"], 1 - alpha))


def run_one(subset: str, seed: int, out: Path, H: int = H_NATIVE) -> pd.DataFrame:
    path = out / "runs" / f"{subset}_seed{seed}.csv"
    if path.exists():
        return pd.read_csv(path)
    frame = CMapssLoader().load(subset)
    frame["rul_true"] = frame.groupby("unit_id")["cycle"].transform("max") - frame["cycle"]
    train_units, test_units = split_engine_units(frame, test_ratio=0.25, random_state=seed)
    train_df, test_df = frame.loc[frame["unit_id"].isin(train_units)], frame.loc[frame["unit_id"].isin(test_units)]
    regimes = RegimeDecoupler(n_clusters=6 if subset.upper() in {"FD002", "FD004"} else 1, random_state=seed).fit(train_df)
    tr, te = regimes.transform(train_df), regimes.transform(test_df)
    rul = te.set_index(["unit_id", "cycle"])["rul_true"]
    eol = te.groupby("unit_id")["cycle"].max()
    ctrl = _controls(test_df, regimes, seed)
    te_all = pd.concat([te, ctrl], ignore_index=True)
    rows, per_engine = [], []
    for window, out_of_window in FITS:
        mv = MultivariateResidualMonitor(healthy_cycle_limit=H, random_state=seed, engine_offset=True, offset_window=window,
                                         calibrate_out_of_window=out_of_window,
                                         calibration="synthetic" if (window, out_of_window) != (W_SPLIT, True) else "empirical")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mv.fit(tr)
            scored_all = mv.score(te_all)
        real = scored_all["unit_id"] < CTRL_ID0
        scored, c = scored_all[real].copy(), scored_all[~real].sort_values(["unit_id", "cycle"])
        t2_ctrl = c["t2"].to_numpy()
        for label, w, oow, which, valid in TREATMENTS:
            if (w, oow) != (window, out_of_window):
                continue
            if which == "synthetic" and mv.synthetic_t2_ is None:
                continue
            ucl = _ucl_for(mv, which, mv.alpha)
            scored["ucl"], scored["breach"] = ucl, scored["t2"].to_numpy() >= ucl
            mv.ucl_ = ucl  # alarms() usa o limite do monitor
            alarms, _, summ = mv.alarms(scored)
            pc = scored[["unit_id", "cycle", "breach"]].copy()
            pc["rul_true"] = rul.reindex(pd.MultiIndex.from_frame(pc[["unit_id", "cycle"]])).to_numpy()
            summ.pop("n_sensors", None)
            far_ctrl = _control_far(t2_ctrl / ucl, np.ones(len(c), bool), c["unit_id"].to_numpy(), c["cycle"].to_numpy(), H)
            row = {"Subset": subset, "Seed": seed, "Tratamento": label, "valida_em_dados_reais": valid, "ucl": round(ucl, 3),
                   "ucl_ratio_vs_empirico": round(ucl / _ucl_for(mv, "empirical", mv.alpha), 3), **summ, **_late_metrics(pc, alarms, H, eol), "far_ctrl_s1": far_ctrl}
            # curva de calibração nos ciclos reais 26 a 50 (limite do próprio alpha)
            win = (scored["cycle"] > W_SPLIT) & (scored["cycle"] <= H)
            limits = [("horizonte", None)] + ([("casado", H - W_SPLIT)] if which == "synthetic" else [])
            for a in ALPHAS:
                for lim, n_post in limits:
                    u = _ucl_for(mv, which, a, n_post)
                    hit = scored.loc[win, "t2"].to_numpy() >= u
                    row[f"obs_{lim}_a{a}"] = float(hit.mean())
                    g = scored.loc[win, ["unit_id"]].assign(h=hit).groupby("unit_id")["h"].agg(["sum", "size"])
                    per_engine += [{"Subset": subset, "Seed": seed, "Tratamento": label, "limite": lim, "alpha": a, "unit_id": int(u_), "hits": int(r["sum"]),
                                    "cycles": int(r["size"])} for u_, r in g.iterrows()]
            rows.append(row)
        print(f"[{subset} seed {seed}] ajuste janela={window} fora={out_of_window} ok", flush=True)
    df = pd.DataFrame(rows)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    (out / "motores").mkdir(parents=True, exist_ok=True)
    pd.DataFrame(per_engine).to_csv(out / "motores" / f"{subset}_seed{seed}.csv", index=False)
    return df


def calibration_curve(out: Path, n_boot: int = 2000, seed: int = 0) -> pd.DataFrame:
    """Taxa observada de ciclos 26 a 50 acima do limite de cada alpha, por subconjunto e tratamento, com IC por bootstrap de motores
    (o cluster é o motor de teste de cada divisão; as sementes são pooladas). Só os tratamentos que validam em dados reais."""
    runs = pd.concat([pd.read_csv(p) for p in sorted((out / "runs").glob("*.csv"))], ignore_index=True)
    runs.to_csv(out / "por_semente.csv", index=False)
    eng = pd.concat([pd.read_csv(p) for p in sorted((out / "motores").glob("*.csv"))], ignore_index=True)
    valid = set(runs.loc[runs["valida_em_dados_reais"], "Tratamento"])
    rng = np.random.default_rng(seed)
    rows = []
    for (sub, trat, lim, a), g in eng[eng["Tratamento"].isin(valid)].groupby(["Subset", "Tratamento", "limite", "alpha"]):
        h, n = g["hits"].to_numpy(float), g["cycles"].to_numpy(float)
        obs = h.sum() / n.sum()
        idx = rng.integers(0, len(h), size=(n_boot, len(h)))
        boot = h[idx].sum(axis=1) / n[idx].sum(axis=1)
        lo, hi = np.quantile(boot, [0.025, 0.975])
        rows.append({"Subset": sub, "Tratamento": trat, "limite": lim, "alpha": a, "observado": obs, "razao_obs_nominal": obs / a, "ic95_lo": lo, "ic95_hi": hi,
                     "eventos": int(h.sum()), "ciclos": int(n.sum())})
    res = pd.DataFrame(rows).round(5)
    res.to_csv(out / "curva_calibracao.csv", index=False)
    g = runs.groupby(["Subset", "Tratamento"], sort=False).agg(
        limite_ucl=("ucl", "mean"), razao_ucl_vs_empirico=("ucl_ratio_vs_empirico", "mean"), deteccao_pct=("detection_rate_pct", "mean"),
        antecedencia_ciclos=("mean_lead_time_detected_cycles", "mean"), falsos_alarmes_100ciclos_controles=("far_ctrl_s1", "mean"),
        far_ref_pct=("far_ref_pct", "mean"), alarme_prematuro_pct=("early_alarm_pct", "mean"), sementes=("Seed", "nunique")).round(3).reset_index()
    g.to_csv(out / "resumo.csv", index=False)
    return res


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--subsets", nargs="+", default=["FD001", "FD002", "FD003", "FD004"])
    ap.add_argument("--seeds", nargs="+", type=int, default=[42, 7, 123])
    ap.add_argument("--jobs", type=int, default=1)
    ap.add_argument("--output-dir", type=Path, default=OUT_DEFAULT)
    ap.add_argument("--aggregate-only", action="store_true")
    a = ap.parse_args(argv)
    if not a.aggregate_only:
        jobs = [(s, seed) for s in a.subsets for seed in a.seeds]
        if a.jobs > 1:
            from joblib import Parallel, delayed
            Parallel(n_jobs=a.jobs)(delayed(run_one)(s, seed, a.output_dir) for s, seed in jobs)
        else:
            for s, seed in jobs:
                run_one(s, seed, a.output_dir)
    res = calibration_curve(a.output_dir)
    with pd.option_context("display.width", 250, "display.max_columns", 30):
        print(res.to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/native_mewma_offset.py
"""Detecção da MEWMA nos motores nativos, com e sem o deslocamento do motor.

WHAT: para cada subconjunto e semente (mesma divisão por motor e mesmos regimes da camada de monitoramento, H = 50), ajusta a MEWMA de
13 resíduos com ``engine_offset`` desligado e ligado e mede detecção, antecedência, falso alarme na janela saudável e alarmes prematuros,
como em ``monitoring_repeated_spc_mv_summary.csv``. Sem a RUL, o conformal nem o custo: só a parte da MEWMA.
WHY: o deslocamento do motor já vale no T50; esta é a medida da mesma correção na MEWMA (extensão natural do estudo do T50).

Run:  .venv/bin/python -m pdm_cmaps.experiments.native_mewma_offset [--subsets FD001 ...] [--seeds 42 7 123]
Saída: results/native_mewma_offset/{por_semente.csv,resumo.csv}
"""
from __future__ import annotations

import argparse
import warnings
from pathlib import Path

import pandas as pd

from pdm_cmaps.config import DEFAULT_RESULTS_DIR
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.data.splits import split_engine_units
from pdm_cmaps.detection.multivariate import MultivariateResidualMonitor
from pdm_cmaps.detection.regime import RegimeDecoupler
from pdm_cmaps.domain.rul import compute_piecewise_rul

OUT_DEFAULT = DEFAULT_RESULTS_DIR / "native_mewma_offset"
H_NATIVE = 50


def run_one(subset: str, seed: int, H: int = H_NATIVE) -> list[dict]:
    loader = CMapssLoader()
    frame = loader.load(subset)
    frame["RUL"] = compute_piecewise_rul(frame)
    train_units, test_units = split_engine_units(frame, test_ratio=0.25, random_state=seed)
    train_df, test_df = frame.loc[frame["unit_id"].isin(train_units)], frame.loc[frame["unit_id"].isin(test_units)]
    regimes = RegimeDecoupler(n_clusters=6 if subset.upper() in {"FD002", "FD004"} else 1, random_state=seed).fit(train_df)
    tr, te = regimes.transform(train_df), regimes.transform(test_df)
    rows = []
    for offset in (False, True):
        mv = MultivariateResidualMonitor(healthy_cycle_limit=H, random_state=seed, engine_offset=offset)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mv.fit(tr)
            _, _, summary = mv.alarms(mv.score(te))
        rows.append({"Subset": subset, "Seed": seed, "H": H, "engine_offset": offset, **summary})
    return rows


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--subsets", nargs="+", default=["FD001", "FD002", "FD003", "FD004"])
    ap.add_argument("--seeds", nargs="+", type=int, default=[42, 7, 123])
    ap.add_argument("--output-dir", type=Path, default=OUT_DEFAULT)
    a = ap.parse_args(argv)
    a.output_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for s in a.subsets:
        for seed in a.seeds:
            rows += run_one(s, seed)
            print(f"[{s} seed {seed}] ok", flush=True)
    per = pd.DataFrame(rows)
    per.to_csv(a.output_dir / "por_semente.csv", index=False)
    cols = ["detection_rate_pct", "mean_lead_time_detected_cycles", "mean_healthy_false_alarm_pct", "premature_alarm_pct"]
    res = per.groupby(["Subset", "engine_offset"])[cols].mean().round(2).reset_index()
    res.to_csv(a.output_dir / "resumo.csv", index=False)
    print(res.to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/native_offset_fora_janela.py
"""Deslocamento do motor estimado FORA da janela de calibração: o falso alarme nativo deixa de ser otimista.

WHAT: nos motores nativos (H = 50, mesma divisão por motor e mesmos regimes da camada de monitoramento) compara, para o T50 EWMA e para a
MEWMA de 13 resíduos, quatro tratamentos do deslocamento do motor:
    sem               nenhum deslocamento removido
    janela H          deslocamento = média do z nos primeiros 50 ciclos (o do texto); o falso alarme da janela saudável é medido nos
                      mesmos ciclos que estimam o deslocamento, onde a média é nula por construção (otimista)
    janela 25, dentro deslocamento nos primeiros 25 ciclos, mas calibração (escala, μ, Σ, limite) e falso alarme em todos os 50
    janela 25, fora   deslocamento nos primeiros 25 ciclos; calibração e falso alarme só nos ciclos 26 a 50 (a calibração vê o mesmo erro de
                      estimativa do deslocamento que o monitoramento depois de H verá)
WHY: calibrar o deslocamento dentro da própria janela de calibração é otimista (ressalva metodológica). As medidas honestas, iguais para os quatro tratamentos, são:
    far_26_50_pct   falso alarme por ciclo nos ciclos 26 a 50 (fora da janela de 25 ciclos, dentro da janela saudável)
    far_ref_pct     falso alarme por ciclo depois de H nos ciclos de motor saudável (RUL verdadeira >= 125)
    early_alarm_pct motores cujo 1º alarme vem com RUL verdadeira >= 125 (alarme prematuro depois da janela)
    far_ctrl_s1     falsos alarmes por 100 ciclos monitorados em controles independentes (reamostragem dos ciclos saudáveis do próprio motor, sem
                    deriva), com o limite NOMINAL de cada detector: mede a calibração sem a confusão da degradação inicial real
Também conta quantas linhas monitoradas de motor multirregime ficam sem correção (menos de 3 linhas do regime na janela do deslocamento).

Run:  .venv/bin/python -m pdm_cmaps.experiments.native_offset_fora_janela [--subsets FD001 ...] [--seeds 42 7 123] [--jobs 3]
Saída: results/native_offset_fora_janela/{runs/*.csv,por_semente.csv,resumo.csv}
"""
from __future__ import annotations

import argparse
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from pdm_cmaps.config import DEFAULT_RESULTS_DIR
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.data.splits import split_engine_units
from pdm_cmaps.detection.multivariate import MultivariateResidualMonitor
from pdm_cmaps.detection.regime import RegimeDecoupler
from pdm_cmaps.detection.virtual_sensor import VirtualSensor
from pdm_cmaps.domain.schema import DEFAULT_RUL_CAP
from pdm_cmaps.experiments.fault_detection_study import HORIZON_CTRL, MIN_POOL, alarm_cycles, bootstrap_series, control_far
from pdm_cmaps.experiments.fault_injection import blind_columns

OUT_DEFAULT = DEFAULT_RESULTS_DIR / "native_offset_fora_janela"
H_NATIVE = 50
W_SPLIT = 25
K = 2
N_CTRL = 6              # controles independentes por motor de teste
CTRL_ID0 = 1_000_000    # unit_id dos controles (não colide com os motores reais)
# (rótulo, engine_offset, offset_window, calibrate_out_of_window)
TREATMENTS = (
    ("sem deslocamento", False, None, False),
    ("deslocamento, janela H", True, None, False),
    ("deslocamento, janela 25, calibração dentro", True, W_SPLIT, False),
    ("deslocamento, janela 25, calibração fora", True, W_SPLIT, True),
)


def _late_metrics(per_cycle: pd.DataFrame, first_alarm: dict, H: int, eol: pd.Series) -> dict:
    """Medidas de falso alarme iguais para todos os tratamentos. ``per_cycle``: unit_id, cycle, breach, rul_true."""
    win = (per_cycle["cycle"] > W_SPLIT) & (per_cycle["cycle"] <= H)
    ref = (per_cycle["cycle"] > H) & (per_cycle["rul_true"] >= DEFAULT_RUL_CAP)
    early = [a is not None and (eol[u] - a) >= DEFAULT_RUL_CAP for u, a in first_alarm.items()]
    return {"far_26_50_pct": 100 * float(per_cycle.loc[win, "breach"].mean()),
            "far_ref_pct": 100 * float(per_cycle.loc[ref, "breach"].mean()) if ref.any() else float("nan"),
            "early_alarm_pct": 100 * float(np.mean(early))}


def _control_far(ratio: np.ndarray, gate: np.ndarray, ids: np.ndarray, cycles: np.ndarray, H: int) -> float:
    """Falsos alarmes por 100 ciclos monitorados (> H) no limite nominal (s = 1) das séries de controle."""
    alarms, last = [], []
    for u in np.unique(ids):
        m = ids == u
        alarms.append(alarm_cycles(ratio[m], gate[m], cycles[m], H + 1, np.array([1.0]), K))
        last.append(cycles[m].max())
    return float(control_far(np.vstack(alarms), np.array(last), H)[0][0])


def _controls(test_df: pd.DataFrame, regimes, seed: int) -> pd.DataFrame:
    """Séries saudáveis independentes (bootstrap iid dos ciclos de cada motor de teste com RUL >= 125), no esquema bruto e já com regime."""
    rng = np.random.default_rng(seed)
    frames, bid = [], CTRL_ID0
    for _, g in test_df.sort_values(["unit_id", "cycle"]).groupby("unit_id"):
        if (g["rul_true"] >= DEFAULT_RUL_CAP).sum() < MIN_POOL:
            continue
        for _ in range(N_CTRL):
            bid += 1
            frames.append(blind_columns(bootstrap_series(g, HORIZON_CTRL, rng), bid))
    return regimes.transform(pd.concat(frames, ignore_index=True))


def run_one(subset: str, seed: int, out: Path, H: int = H_NATIVE) -> pd.DataFrame:
    path = out / "runs" / f"{subset}_seed{seed}.csv"
    if path.exists():
        return pd.read_csv(path)
    frame = CMapssLoader().load(subset)
    frame["rul_true"] = frame.groupby("unit_id")["cycle"].transform("max") - frame["cycle"]
    train_units, test_units = split_engine_units(frame, test_ratio=0.25, random_state=seed)
    train_df, test_df = frame.loc[frame["unit_id"].isin(train_units)], frame.loc[frame["unit_id"].isin(test_units)]
    regimes = RegimeDecoupler(n_clusters=6 if subset.upper() in {"FD002", "FD004"} else 1, random_state=seed).fit(train_df)
    tr, te = regimes.transform(train_df), regimes.transform(test_df)
    rul = te.set_index(["unit_id", "cycle"])["rul_true"]
    eol = te.groupby("unit_id")["cycle"].max()
    ctrl = _controls(test_df, regimes, seed)
    te_all = pd.concat([te, ctrl], ignore_index=True)  # motores reais + controles, pontuados pelo mesmo detector ajustado
    rows = []
    for label, offset, window, out_of_window in TREATMENTS:
        base = {"Subset": subset, "Seed": seed, "H": H, "Tratamento": label}
        vs = VirtualSensor(target_sensor="T50", mode="exogenous", ewma_alpha=0.10, min_consecutive_breaches=K, healthy_cycle_limit=H,
                           random_state=seed, engine_offset=offset, offset_window=window, calibrate_out_of_window=out_of_window)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            trace_all, per_all, _ = vs.evaluate(tr, te_all)
        real = trace_all["unit_id"] < CTRL_ID0
        trace, per_engine = trace_all[real], per_all[per_all["unit_id"] < CTRL_ID0]
        summary = _summary(per_engine)
        c = trace_all[~real].sort_values(["unit_id", "cycle"])
        far_ctrl_t50 = _control_far((c["ewma_abs_z"] / c["ewma_ucl"]).to_numpy(), (c["z_score"].abs() >= vs.threshold_sigma).to_numpy(),
                                    c["unit_id"].to_numpy(), c["cycle"].to_numpy(), H)
        alarms = {int(r.unit_id): (None if pd.isna(r.first_alarm_cycle) else int(r.first_alarm_cycle)) for r in per_engine.itertuples()}
        breach = (trace["ewma_abs_z"] >= trace["ewma_ucl"]) & (trace["z_score"].abs() >= vs.threshold_sigma)
        pc = trace[["unit_id", "cycle"]].assign(breach=breach.to_numpy())
        pc["rul_true"] = rul.reindex(pd.MultiIndex.from_frame(pc[["unit_id", "cycle"]])).to_numpy()
        rows.append({**base, "Detector": "T50 EWMA", **summary, **_late_metrics(pc, alarms, H, eol), "far_ctrl_s1": far_ctrl_t50})

        mv = MultivariateResidualMonitor(healthy_cycle_limit=H, random_state=seed, engine_offset=offset, offset_window=window,
                                         calibrate_out_of_window=out_of_window)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mv.fit(tr)
            scored_all = mv.score(te_all)
        real = scored_all["unit_id"] < CTRL_ID0
        scored = scored_all[real]
        c = scored_all[~real].sort_values(["unit_id", "cycle"])
        far_ctrl_mv = _control_far((c["t2"] / c["ucl"]).to_numpy(), np.ones(len(c), bool), c["unit_id"].to_numpy(), c["cycle"].to_numpy(), H)
        mv_alarms, _, mv_summary = mv.alarms(scored)
        pc = scored[["unit_id", "cycle", "breach"]].copy()
        pc["rul_true"] = rul.reindex(pd.MultiIndex.from_frame(pc[["unit_id", "cycle"]])).to_numpy()
        mv_summary.pop("n_sensors", None)
        rows.append({**base, "Detector": "MEWMA (13 resíduos)", **mv_summary, **_late_metrics(pc, mv_alarms, H, eol),
                     "far_ctrl_s1": far_ctrl_mv, "linhas_sem_correcao_pct": mv.offset_uncovered_pct(te) if offset else 0.0})
        print(f"[{subset} seed {seed}] {label}", flush=True)
    df = pd.DataFrame(rows)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return df


def _summary(per_engine: pd.DataFrame) -> dict:
    """Mesmo resumo que ``VirtualSensor.evaluate`` devolve, só nos motores reais."""
    det = per_engine.loc[per_engine["detected"]]
    return {"total_engines": len(per_engine), "detection_rate_pct": float(per_engine["detected"].mean() * 100.0),
            "mean_lead_time_cycles": float(per_engine["lead_time_cycles"].mean()),
            "mean_lead_time_detected_cycles": float(det["lead_time_cycles"].mean()) if len(det) else 0.0,
            "mean_healthy_false_alarm_pct": float(per_engine["healthy_far"].mean() * 100.0),
            "premature_alarm_pct": float(per_engine["premature_alarm"].mean() * 100.0)}


METRICS = ["detection_rate_pct", "mean_lead_time_detected_cycles", "mean_healthy_false_alarm_pct", "premature_alarm_pct",
           "far_26_50_pct", "far_ref_pct", "early_alarm_pct", "far_ctrl_s1", "linhas_sem_correcao_pct"]


def aggregate(out: Path) -> pd.DataFrame:
    per = pd.concat([pd.read_csv(p) for p in sorted((out / "runs").glob("*.csv"))], ignore_index=True)
    per.to_csv(out / "por_semente.csv", index=False)
    order = {t[0]: i for i, t in enumerate(TREATMENTS)}
    g = per.groupby(["Subset", "Detector", "Tratamento"], sort=False)[METRICS]
    res = g.mean().round(3).add_suffix("_mean").join(g.std().round(3).add_suffix("_std")).reset_index()
    res["n_seeds"] = g.size().to_numpy()
    res = res.sort_values(["Subset", "Detector", "Tratamento"], key=lambda c: c.map(order) if c.name == "Tratamento" else c).reset_index(drop=True)
    res.to_csv(out / "resumo.csv", index=False)
    return res


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--subsets", nargs="+", default=["FD001", "FD002", "FD003", "FD004"])
    ap.add_argument("--seeds", nargs="+", type=int, default=[42, 7, 123])
    ap.add_argument("--jobs", type=int, default=3)
    ap.add_argument("--output-dir", type=Path, default=OUT_DEFAULT)
    ap.add_argument("--aggregate-only", action="store_true")
    a = ap.parse_args(argv)
    if not a.aggregate_only:
        jobs = [(s, seed) for s in a.subsets for seed in a.seeds]
        if a.jobs > 1:
            from joblib import Parallel, delayed
            Parallel(n_jobs=a.jobs)(delayed(run_one)(s, seed, a.output_dir) for s, seed in jobs)
        else:
            for s, seed in jobs:
                run_one(s, seed, a.output_dir)
    res = aggregate(a.output_dir)
    cols = ["Subset", "Detector", "Tratamento", "detection_rate_pct_mean", "mean_lead_time_detected_cycles_mean", "mean_healthy_false_alarm_pct_mean",
            "far_26_50_pct_mean", "far_ref_pct_mean", "early_alarm_pct_mean", "far_ctrl_s1_mean"]
    with pd.option_context("display.width", 250, "display.max_columns", 30):
        print(res[cols].to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/poder_bootstrap.py
"""Intervalos de confiança do poder de detecção por bootstrap sobre os motores (e das diferenças pareadas entre detectores).

WHAT: refaz as séries cegas do estudo de injeção (mesmas sementes, mesmo ``build_series``, mesmos detectores) e GUARDA, para cada detector, a
matriz de alarmes [séries x limiares] e o manifesto (motor receptor, papel, forma, magnitude, t0). Com isso:
  * ``collect`` grava ``alarmes/<subconj>_seed<s>_H<H>.npz`` + ``_manifesto.csv`` (e confere o poder pareado com o versionado);
  * ``analyse`` faz o bootstrap em cluster por motor receptor (controles e falhas reamostrados JUNTOS, dentro de cada semente), RECALCULA o limiar
    de falso alarme igualado (s*) em cada réplica a partir dos controles reamostrados e tira a média das sementes, como a tabela de poder;
  * as diferenças entre detectores usam as MESMAS réplicas (pareadas), com valor-p do bootstrap e correção de Holm por família de afirmações.
WHY: o texto comparava diferenças de poder com o desvio entre três sementes, que não é a régua certa para uma diferença pareada, e o limiar
estimado em 6 controles por motor tinha incerteza que não era propagada. Aqui a incerteza vem dos motores (e do limiar).
LIMITES: o IC condiciona às três divisões treino/teste (não reamostra a divisão) e trata as sementes como independentes, embora reaproveitem motores;
cobre o erro de amostragem de motores e de limiar, não o modelo da falha injetada.

Run:  .venv/bin/python -m pdm_cmaps.experiments.poder_bootstrap collect [--config principal|so_t50] [--jobs 3]
      .venv/bin/python -m pdm_cmaps.experiments.poder_bootstrap analyse [--config ...] [--boot 4000]
Saída: results/injecao_deteccao_ic/<config>/{alarmes/,poder_ic.csv,contrastes_ic.csv}
"""
from __future__ import annotations

import argparse
import re
from pathlib import Path

import numpy as np
import pandas as pd

from pdm_cmaps.config import DEFAULT_RESULTS_DIR
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.experiments.fault_detection_study import (ADOPTED_H, BUDGETS, DET_IFOREST, DET_MV, DET_RAW_MV, DET_RAW_T50, DET_T50_NOGATE, DET_T50_SIGNED,
                                                         DETECTORS, EXTRA_DETECTORS, alarm_matrix, build_series, extra_scores, score_all)
from pdm_cmaps.experiments.fault_injection import load_world

OUT_DEFAULT = DEFAULT_RESULTS_DIR / "injecao_deteccao_ic"
BUDGET = BUDGETS[0]
CONFIGS = {"principal": dict(engine_offset=True, mewma_offset=True, versioned="injecao_deteccao"),
           "so_t50": dict(engine_offset=True, mewma_offset=False, versioned="injecao_deteccao_offset_so_t50")}
N_BOOT = 4000
SEED_BOOT = 20261118

# Contrastes (A - B) e a família de afirmações de cada um; Holm corrige dentro da família e da configuração.
CONTRASTS = {
    "regime na MEWMA (com - sem)": (DET_MV, DET_RAW_MV),
    "T50 sem portão - MEWMA com regime": (DET_T50_NOGATE, DET_MV),
    "T50 sem portão - T50 bruto": (DET_T50_NOGATE, DET_RAW_T50),
}
# config "extras": a MEWMA contra a EWMA clássica com sinal e o Isolation Forest (degraus e rampas pequenos) e a EWMA clássica contra a do módulo, onde a
# do módulo deve ganhar (variância) e onde deve perder (degrau, rampa, viés de um sensor)
CONTRASTS_EXTRAS = {
    "MEWMA com regime - EWMA clássica com sinal": (DET_MV, DET_T50_SIGNED),
    "MEWMA com regime - Isolation Forest": (DET_MV, DET_IFOREST),
    "EWMA clássica com sinal - EWMA do módulo (T50 sem portão)": (DET_T50_SIGNED, DET_T50_NOGATE),
}
FAMILIES_EXTRAS = {
    "MEWMA com regime - EWMA clássica com sinal": (("degrau", 0.10), ("degrau", 0.05), ("rampa", 0.10)),
    "MEWMA com regime - Isolation Forest": (("degrau", 0.10), ("degrau", 0.05), ("rampa", 0.10)),
    "EWMA clássica com sinal - EWMA do módulo (T50 sem portão)": (("degrau", 0.10), ("rampa", 0.10), ("variancia", 0.5), ("vies_sensor", 1.0)),
}
FAMILIES = {  # contraste -> células (forma, magnitude) da família
    "regime na MEWMA (com - sem)": (("degrau", 0.10), ("degrau", 0.05), ("rampa", 0.10)),
    "T50 sem portão - MEWMA com regime": (("degrau", 0.25), ("rampa", 0.25), ("vies_sensor", 1.0), ("variancia", 1.0)),
    "T50 sem portão - T50 bruto": (("degrau", 0.25), ("vies_sensor", 1.0)),
}


def _slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


# ============================================================================
# collect
# ============================================================================

def collect_one(subset: str, seed: int, H: int, out: Path, config: str, n_ctrl: int = 6, n_rep: int = 2) -> Path:
    path = out / config / "alarmes" / f"{subset}_seed{seed}_H{H}.npz"
    if path.exists():
        return path
    cfg = CONFIGS[config]
    world = load_world(subset, seed, H, loader=CMapssLoader())
    blind, meta = build_series(world, seed, n_ctrl, n_rep, None, False)
    scores = score_all(world, blind, seed, cfg["engine_offset"], cfg["mewma_offset"])
    if cfg.get("extras"):
        scores.update(extra_scores(world, blind, seed, cfg["engine_offset"], cfg["mewma_offset"]))
    midx = meta.set_index("blind_id")
    arrays: dict[str, np.ndarray] = {}
    for name, sc in scores.items():
        ids, mat, last = alarm_matrix(sc, np.maximum(midx["t0"], H + 1))
        arrays[f"{_slug(name)}__ids"], arrays[f"{_slug(name)}__mat"], arrays[f"{_slug(name)}__last"] = ids, mat.astype(np.float32), last
    path.parent.mkdir(parents=True, exist_ok=True)
    meta.assign(Subset=subset, Seed=seed, H=H).to_csv(path.with_name(path.stem + "_manifesto.csv"), index=False)
    np.savez_compressed(path, **arrays)
    print(f"[{subset} seed {seed}] {len(meta)} séries, {len(scores)} detectores guardados", flush=True)
    return path


# ============================================================================
# bootstrap
# ============================================================================

class Run:
    """Uma execução (subconjunto, semente) com tudo que o bootstrap precisa, por detector."""

    def __init__(self, npz: Path):
        self.subset, self.seed, self.H = re.match(r"(FD\d{3})_seed(\d+)_H(\d+)", npz.stem).groups()
        self.seed, self.H = int(self.seed), int(self.H)
        meta = pd.read_csv(npz.with_name(npz.stem + "_manifesto.csv")).set_index("blind_id")
        data = np.load(npz)
        self.det: dict[str, dict] = {}
        self.names = [n for n in (*DETECTORS, *EXTRA_DETECTORS) if f"{_slug(n)}__mat" in data.files]
        for name in self.names:
            slug = _slug(name)
            ids, mat, last = data[f"{slug}__ids"], data[f"{slug}__mat"].astype(float), data[f"{slug}__last"]
            m = meta.loc[ids]
            ctrl = (m["role"] == "controle").to_numpy()
            has = ~np.isnan(mat)
            expo = np.where(has, mat - self.H, (last - self.H)[:, None])
            self.det[name] = {"ctrl_has": has[ctrl], "ctrl_expo": expo[ctrl], "fault_has": has[~ctrl], "ctrl_rec": m["recipient"].to_numpy()[ctrl],
                              "fault_rec": m["recipient"].to_numpy()[~ctrl], "groups": m.loc[~ctrl, ["kind", "magnitude", "t0"]].reset_index(drop=True)}
        ref = self.det[DETECTORS[0]]
        self.recipients = np.unique(np.concatenate([ref["ctrl_rec"], ref["fault_rec"]]))
        code = {r: i for i, r in enumerate(self.recipients)}
        keys = ref["groups"].astype(str).agg("|".join, axis=1)
        self.group_keys = pd.unique(keys)
        gcode = {k: i for i, k in enumerate(self.group_keys)}
        self.group_table = ref["groups"].assign(_k=keys).drop_duplicates("_k").drop(columns="_k").reset_index(drop=True)
        for d in self.det.values():
            d["ctrl_code"] = np.array([code[r] for r in d["ctrl_rec"]])
            d["fault_code"] = np.array([code[r] for r in d["fault_rec"]])
            d["gcode"] = np.array([gcode[k] for k in d["groups"].astype(str).agg("|".join, axis=1)])

    def power(self, name: str, counts: np.ndarray, budget: float = BUDGET) -> np.ndarray:
        """Poder (%) por grupo de falha no s* mais baixo cujo falso alarme dos controles (ponderados por ``counts``) cabe no orçamento; NaN se nenhum."""
        d = self.det[name]
        wc = counts[d["ctrl_code"]].astype(float)
        events, exposure = wc @ d["ctrl_has"], wc @ d["ctrl_expo"]
        with np.errstate(divide="ignore", invalid="ignore"):
            far = 100 * events / exposure
        ok = np.flatnonzero(far <= budget)
        n_groups = len(self.group_keys)
        if not ok.size:
            return np.full(n_groups, np.nan)
        wf = counts[d["fault_code"]].astype(float)
        det = d["fault_has"][:, ok[0]].astype(float)
        with np.errstate(divide="ignore", invalid="ignore"):
            return 100 * np.bincount(d["gcode"], wf * det, n_groups) / np.bincount(d["gcode"], wf, n_groups)


def point_power(runs: list[Run], name: str) -> pd.DataFrame:
    """Poder pontual (todos os motores, peso 1) por (subconjunto, semente, forma, magnitude, t0)."""
    rows = []
    for r in runs:
        p = r.power(name, np.ones(len(r.recipients)))
        rows.append(r.group_table.assign(Subset=r.subset, Seed=r.seed, Detector=name, power_pct=p[np.arange(len(r.group_keys))]))
    return pd.concat(rows, ignore_index=True)


def check_against_versioned(runs: list[Run], versioned_dir: Path) -> float:
    """Maior diferença absoluta de poder (pontos) entre as matrizes guardadas e os ``*_pareado.csv`` versionados (esperado: 0)."""
    worst = 0.0
    for name in runs[0].names:
        if name in EXTRA_DETECTORS:
            continue  # não existem no estudo versionado
        mine = point_power(runs, name)
        for (subset, seed), g in mine.groupby(["Subset", "Seed"]):
            old = pd.read_csv(versioned_dir / "runs" / f"{subset}_seed{seed}_H{ADOPTED_H}_pareado.csv")
            old = old[(old["Detector"] == name) & (old["budget"] == BUDGET)][["kind", "magnitude", "t0", "power_pct"]]
            merged = g.merge(old, on=["kind", "magnitude", "t0"], suffixes=("", "_old"))
            assert len(merged) == len(g), f"grupos diferentes em {subset} seed {seed} {name}"
            worst = max(worst, float(np.nanmax(np.abs(merged["power_pct"] - merged["power_pct_old"]))))
    return worst


def bootstrap(runs: list[Run], n_boot: int = N_BOOT, seed: int = SEED_BOOT) -> dict[str, np.ndarray]:
    """Réplicas do poder médio das sementes, por subconjunto: {subset: array [n_boot, n_detectores, n_grupos]} (grupos na ordem de ``group_table``)."""
    rng = np.random.default_rng(seed)
    by_subset: dict[str, list[Run]] = {}
    for r in runs:
        by_subset.setdefault(r.subset, []).append(r)
    out = {}
    for subset, rs in by_subset.items():
        ref_keys = list(rs[0].group_keys)
        assert all(list(r.group_keys) == ref_keys for r in rs), "os grupos de falha diferem entre as sementes"
        names = rs[0].names
        boot = np.full((n_boot, len(names), len(ref_keys)), np.nan)
        for b in range(n_boot):
            per_seed = []
            for r in rs:
                counts = np.bincount(rng.integers(0, len(r.recipients), len(r.recipients)), minlength=len(r.recipients))
                per_seed.append(np.vstack([r.power(name, counts) for name in names]))
            with np.errstate(all="ignore"):
                boot[b] = np.nanmean(np.stack(per_seed), axis=0)
        out[subset] = boot
    return out


def holm(p: np.ndarray) -> np.ndarray:
    order = np.argsort(p)
    adj = np.empty_like(p, dtype=float)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj


def summarize(runs: list[Run], boot: dict[str, np.ndarray], contrasts: dict | None = None, families: dict | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """(poder com IC por detector e célula, contrastes pareados com IC, p do bootstrap e p de Holm por família)."""
    power_rows, contrast_rows = [], []
    contrasts = CONTRASTS if contrasts is None else contrasts
    families = FAMILIES if families is None else families
    names = runs[0].names
    didx = {n: i for i, n in enumerate(names)}
    for subset, b in boot.items():
        rs = [r for r in runs if r.subset == subset]
        table = rs[0].group_table
        pts = {n: np.nanmean(np.vstack([r.power(n, np.ones(len(r.recipients))) for r in rs]), axis=0) for n in names}
        for g, cell in table.iterrows():
            for n in names:
                lo, hi = np.nanpercentile(b[:, didx[n], g], [2.5, 97.5])
                power_rows.append({"Subset": subset, "Detector": n, "kind": cell["kind"], "magnitude": cell["magnitude"], "t0": cell["t0"],
                                   "poder_pct": pts[n][g], "ic_lo": lo, "ic_hi": hi, "n_seeds": len(rs)})
            for label, (a, c) in contrasts.items():
                d = b[:, didx[a], g] - b[:, didx[c], g]
                d = d[np.isfinite(d)]
                if not d.size:
                    continue
                p = float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))
                p = max(p, 1.0 / (len(d) + 1))
                contrast_rows.append({"Subset": subset, "Contraste": label, "kind": cell["kind"], "magnitude": cell["magnitude"], "t0": cell["t0"],
                                      "diferenca_pp": pts[a][g] - pts[c][g], "ic_lo": np.percentile(d, 2.5), "ic_hi": np.percentile(d, 97.5), "p_boot": p})
    power, contr = pd.DataFrame(power_rows), pd.DataFrame(contrast_rows)
    # Holm dentro da família (contraste x células listadas x subconjuntos)
    contr["familia"] = False
    contr["p_holm"] = np.nan
    for label, cells in families.items():
        sel = contr["Contraste"].eq(label) & contr[["kind", "magnitude"]].apply(tuple, axis=1).isin([(k, m) for k, m in cells]) & (contr["t0"] == ADOPTED_H + 10)
        contr.loc[sel, "familia"] = True
        if sel.any():
            contr.loc[sel, "p_holm"] = holm(contr.loc[sel, "p_boot"].to_numpy())
    return power, contr


# ============================================================================
# CLI
# ============================================================================

def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("action", choices=["collect", "analyse"])
    ap.add_argument("--config", choices=list(CONFIGS), default="principal")
    ap.add_argument("--subsets", nargs="+", default=["FD001", "FD002", "FD003", "FD004"])
    ap.add_argument("--seeds", nargs="+", type=int, default=[42, 7, 123])
    ap.add_argument("--jobs", type=int, default=3)
    ap.add_argument("--boot", type=int, default=N_BOOT)
    ap.add_argument("--output-dir", type=Path, default=OUT_DEFAULT)
    a = ap.parse_args(argv)
    if a.action == "collect":
        jobs = [(s, seed) for s in a.subsets for seed in a.seeds]
        if a.jobs > 1:
            from joblib import Parallel, delayed
            Parallel(n_jobs=a.jobs)(delayed(collect_one)(s, seed, ADOPTED_H, a.output_dir, a.config) for s, seed in jobs)
        else:
            for s, seed in jobs:
                collect_one(s, seed, ADOPTED_H, a.output_dir, a.config)
        return 0
    runs = [Run(p) for p in sorted((a.output_dir / a.config / "alarmes").glob("*.npz"))]
    versioned = DEFAULT_RESULTS_DIR / CONFIGS[a.config]["versioned"]
    if (versioned / "runs").exists():
        print(f"conferência com {versioned.name}: maior diferença de poder = {check_against_versioned(runs, versioned):.6f} ponto")
    ex = a.config == "extras"
    power, contr = summarize(runs, bootstrap(runs, a.boot), CONTRASTS_EXTRAS if ex else None, FAMILIES_EXTRAS if ex else None)
    power.round(3).to_csv(a.output_dir / a.config / "poder_ic.csv", index=False)
    contr.round(4).to_csv(a.output_dir / a.config / "contrastes_ic.csv", index=False)
    with pd.option_context("display.width", 250, "display.max_columns", 30):
        print(contr[contr["familia"]].round(3).to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/rul_prediction_plots.py
"""Predicted vs real RUL for every model and every held-out engine, with maintenance markers.

WHAT: reads `results/maintenance_guide_<subset>/tables/rul_holdout_predictions.csv` (written by
pipeline.run(): seed-42 75/25 engine split, every cycle of each held-out engine, all models, H=50)
and writes one figure per (model, engine), one all-models grid per engine and one pooled
predicted-vs-real scatter per model:

  results/rul_predicoes/<subset>/
      modelos/<model>/motor_XXX.png          real vs predicted RUL along the engine life
      modelos/<model>/_dispersao.png         pooled predicted vs real (all held-out engines)
      motores/motor_XXX.png                  all models for one engine (small multiples)
      manutencao/motor_XXX.png               same grid, only engines with a maintenance event / candidate
      metricas_por_motor.csv                 RMSE / MAE per (model, engine)
      candidatos_descontinuidade.csv         step-drop scan of the core speed Nc on ALL engines

WHY: the holdout tables only give fleet averages; per-engine traces show WHERE a model fails
(early plateau, late life, a reset). C-MAPSS is run-to-failure: no maintenance is recorded. The one
verified exception is FD004 engine 179, whose core (Nc, T30, T50, phi, P30) drops back to the
healthy level at cycle 242 while the fan side (Nf, T24) does not move
(found by scan_discontinuities below and confirmed by inspecting the five core channels).

[ASSUMPTION] Candidate scan: Nc standardized per regime with the cycles <= 50 of all engines;
drop(t) = mean(z[t:t+10]) - mean(z[t-10:t]); an engine is a CANDIDATE when min drop < -3 sigma.
Candidates are drawn as grey dotted lines and are NOT called maintenance.
[CAUSALITY] the RUL label is piecewise from the final failure (cap 125); it does NOT reset at a
maintenance event, so after a reset the "real" curve is the label convention, not physics.

Run:  .venv/bin/python -m pdm_cmaps.experiments.rul_prediction_plots [--subsets FD004] [--jobs 8]
"""

from __future__ import annotations

from pdm_cmaps.config import DEFAULT_RESULTS_DIR

import argparse
import re
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.detection.regime import RegimeDecoupler
from pdm_cmaps.reporting import viz_style

OUT_DEFAULT = DEFAULT_RESULTS_DIR / "rul_predicoes"
VERIFIED_MAINTENANCE = {("FD004", 179): 242}  # core reset of engine 179; scan_discontinuities flags it as verified_maintenance
STEP_WINDOW = 10
STEP_SIGMA = 3.0
REAL_COLOR, PRED_COLOR, EVENT_COLOR, CAND_COLOR = viz_style.INK, viz_style.SERIES[0], viz_style.STATUS[3], viz_style.MUTED


def slug(name: str) -> str:
    return re.sub(r"[^0-9A-Za-z]+", "_", name).strip("_")


def scan_discontinuities(subset: str, loader: CMapssLoader | None = None) -> pd.DataFrame:
    """Largest sustained drop of the regime-standardized core speed Nc per engine (all train engines)."""
    frame = (loader or CMapssLoader()).load(subset)
    k = 6 if subset.upper() in {"FD002", "FD004"} else 1
    d = RegimeDecoupler(n_clusters=k, random_state=42).fit(frame).transform(frame)
    ref = d.loc[d["cycle"] <= 50].groupby("regime_id")["Nc"].agg(["mean", "std"])
    d["z"] = (d["Nc"] - d["regime_id"].map(ref["mean"])) / d["regime_id"].map(ref["std"])
    rows = []
    for unit, g in d.sort_values(["unit_id", "cycle"]).groupby("unit_id"):
        z, c, n = g["z"].to_numpy(), g["cycle"].to_numpy(), len(g)
        cs = np.concatenate([[0.0], np.cumsum(z)])
        t = np.arange(STEP_WINDOW, n - STEP_WINDOW + 1)
        if t.size == 0:
            continue
        after = (cs[t + STEP_WINDOW] - cs[t]) / STEP_WINDOW
        before = (cs[t] - cs[t - STEP_WINDOW]) / STEP_WINDOW
        i = int(np.argmin(after - before))
        rows.append({"Subset": subset, "unit_id": int(unit), "cycle": int(c[t[i]]), "eol_cycle": int(c.max()),
                     "Nc_z_before": float(before[i]), "Nc_z_after": float(after[i]), "drop_sigma": float(after[i] - before[i]),
                     "candidate": bool(after[i] - before[i] < -STEP_SIGMA),
                     "verified_maintenance": VERIFIED_MAINTENANCE.get((subset, int(unit))) == int(c[t[i]])})
    return pd.DataFrame(rows).sort_values("drop_sigma").reset_index(drop=True)


def _events(subset: str, unit: int, cands: pd.DataFrame) -> list[tuple[int, str]]:
    ev = []
    if (subset, unit) in VERIFIED_MAINTENANCE:
        ev.append((VERIFIED_MAINTENANCE[(subset, unit)], "verified"))
    for r in cands.loc[(cands["unit_id"] == unit) & cands["candidate"] & ~cands["verified_maintenance"]].itertuples():
        ev.append((int(r.cycle), "candidate"))
    return ev


def _draw_events(ax, events, label: bool) -> None:
    for cyc, kind in events:
        if kind == "verified":
            ax.axvline(cyc, color=EVENT_COLOR, lw=1.6, ls="--", zorder=1)
            if label:
                ax.annotate(f"manutenção / reset\ndo núcleo (ciclo {cyc})", xy=(cyc, 1), xycoords=("data", "axes fraction"),
                            xytext=(4, -4), textcoords="offset points", va="top", fontsize=8, color=viz_style.INK)
        else:
            ax.axvline(cyc, color=CAND_COLOR, lw=1.2, ls=":", zorder=1)
            if label:
                ax.annotate(f"queda brusca de Nc (ciclo {cyc})\ncandidato, não verificado", xy=(cyc, 1),
                            xycoords=("data", "axes fraction"), xytext=(4, -4), textcoords="offset points",
                            va="top", fontsize=8, color=viz_style.INK_2)


def _engine_figure(eng: pd.DataFrame, model: str, subset: str, unit: int, events, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(7.5, 4.0))
    ax.plot(eng["cycle"], eng["RUL_actual"], color=REAL_COLOR, lw=viz_style.LINE_W, label="RUL real (rótulo, teto 125)")
    ax.plot(eng["cycle"], eng["pred"], color=PRED_COLOR, lw=viz_style.LINE_W, label="RUL predito")
    _draw_events(ax, events, label=True)
    err = eng["pred"] - eng["RUL_actual"]
    rmse, mae = float(np.sqrt(np.mean(err ** 2))), float(np.mean(np.abs(err)))
    viz_style.style_axes(ax)
    ax.set_xlabel("Ciclo")
    ax.set_ylabel("RUL (ciclos)")
    ax.set_ylim(bottom=min(0, float(eng["pred"].min()) - 5))
    ax.set_title(f"{subset} · motor {unit} · {model}\nRMSE {rmse:.1f} · MAE {mae:.1f} ciclos (holdout, seed 42, H=50)", fontsize=10)
    ax.legend(frameon=False, fontsize=8, loc="lower left")
    fig.tight_layout()
    fig.savefig(path, dpi=90)
    plt.close(fig)


NOMES_PT = {
    "XGBoost (NASA Loss)": "XGBoost (perda NASA)",
    "Support Vector Machine (SVM)": "SVM linear",
    "MiniRocket + RidgeCV": "MiniRocket + Ridge",
    "PyTorch RNN (LSTM)": "LSTM (PyTorch)",
    "PyTorch CNN": "CNN (PyTorch)",
    "PyTorch Transformer": "Transformer (PyTorch)",
}


def _grid_figure(eng: pd.DataFrame, models: list[str], subset: str, unit: int, events, path: Path) -> None:
    ncol = 5
    nrow = int(np.ceil(len(models) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 2.5 * nrow), sharex=True, sharey=True, squeeze=False)
    ymax = float(eng["RUL_actual"].max())
    for ax, model in zip(axes.flat, models):
        pred = eng[f"RUL_predicted_{model}"]
        rmse = float(np.sqrt(np.mean((pred - eng["RUL_actual"]) ** 2)))
        ax.plot(eng["cycle"], eng["RUL_actual"], color=REAL_COLOR, lw=1.5)
        ax.plot(eng["cycle"], pred, color=PRED_COLOR, lw=1.5)
        _draw_events(ax, events, label=False)
        for cyc, kind in events:
            if kind == "verified":  # o rótulo vai em todos os painéis: a figura é lida reduzida, e a legenda do título é pequena
                ax.text(cyc + 8, ymax * 1.13, f"descontinuidade (ciclo {cyc})", ha="left", va="center", fontsize=8.5, color=EVENT_COLOR, fontweight="bold")
        viz_style.style_axes(ax)
        label = NOMES_PT.get(model, model)
        ax.set_title(f"{label}\n" + f"RMSE {rmse:.1f}".replace(".", ","), fontsize=9.5)
    for ax in list(axes.flat)[len(models):]:
        ax.axis("off")
    note = ""
    if any(k == "verified" for _, k in events):
        note = " · linha vermelha tracejada = descontinuidade no ciclo 242"
    elif events:
        note = " · linha cinza pontilhada = queda brusca de Nc (candidato, não verificado)"
    fig.suptitle(f"{subset} · motor {unit}: RUL real (preto) × prevista (azul), todos os modelos{note}", fontsize=13)
    fig.supxlabel("Ciclo", fontsize=11)
    fig.supylabel("RUL (ciclos)", fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(path, dpi=130)
    plt.close(fig)


def _scatter_figure(df: pd.DataFrame, model: str, subset: str, path: Path) -> None:
    from sklearn.metrics import r2_score

    truth, pred = df["RUL_actual"].to_numpy(), df[f"RUL_predicted_{model}"].to_numpy()
    fig, ax = plt.subplots(figsize=(5.8, 5.4))
    ax.scatter(truth, pred, s=6, alpha=0.2, color=PRED_COLOR, edgecolors="none", label="todos os ciclos, motores do holdout")
    for (sub, unit), cyc in VERIFIED_MAINTENANCE.items():
        if sub != subset:
            continue
        m = df["unit_id"] == unit
        if m.any():
            after = m & (df["cycle"] >= cyc)
            ax.scatter(truth[m & ~after], pred[m & ~after], s=14, color=viz_style.SERIES[1], label=f"motor {unit}, antes do ciclo {cyc}")
            ax.scatter(truth[after], pred[after], s=14, color=EVENT_COLOR, label=f"motor {unit}, após reset (ciclo ≥ {cyc})")
    lo, hi = float(min(truth.min(), pred.min())), float(max(truth.max(), pred.max()))
    ax.plot([lo, hi], [lo, hi], color=viz_style.INK_2, ls="--", lw=1.2, label="predição perfeita")
    viz_style.style_axes(ax, grid="both")
    ax.set_xlabel("RUL real (ciclos)")
    ax.set_ylabel("RUL predito (ciclos)")
    ax.set_title(f"{subset} · {model}\nR² holdout = {r2_score(truth, pred):.3f} · n = {len(truth)} ciclos", fontsize=10)
    ax.legend(frameon=False, fontsize=7.5, loc="upper left")
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)


def _model_job(df: pd.DataFrame, model: str, subset: str, cands: pd.DataFrame, out: Path) -> list[dict]:
    viz_style.apply_style()
    folder = out / "modelos" / slug(model)
    folder.mkdir(parents=True, exist_ok=True)
    rows = []
    for unit, eng in df.groupby("unit_id"):
        eng = eng.sort_values("cycle").assign(pred=lambda e: e[f"RUL_predicted_{model}"])
        _engine_figure(eng, model, subset, int(unit), _events(subset, int(unit), cands), folder / f"motor_{int(unit):03d}.png")
        err = eng["pred"] - eng["RUL_actual"]
        rows.append({"Subset": subset, "Model": model, "unit_id": int(unit), "n_cycles": len(eng),
                     "RMSE": float(np.sqrt(np.mean(err ** 2))), "MAE": float(np.mean(np.abs(err))), "Bias": float(err.mean())})
    _scatter_figure(df, model, subset, folder / "_dispersao.png")
    return rows


def _engine_job(eng: pd.DataFrame, models, subset: str, unit: int, events, out: Path) -> None:
    viz_style.apply_style()
    _grid_figure(eng.sort_values("cycle"), models, subset, unit, events, out / "motores" / f"motor_{unit:03d}.png")
    if events:
        _grid_figure(eng.sort_values("cycle"), models, subset, unit, events, out / "manutencao" / f"motor_{unit:03d}.png")


def run_subset(subset: str, out_root: Path, jobs: int) -> pd.DataFrame:
    from joblib import Parallel, delayed

    src = DEFAULT_RESULTS_DIR / f"maintenance_guide_{subset.lower()}" / "tables" / "rul_holdout_predictions.csv"
    df = pd.read_csv(src)
    models = [c.replace("RUL_predicted_", "") for c in df.columns if c.startswith("RUL_predicted_")]
    out = out_root / subset
    for sub in ("modelos", "motores", "manutencao"):
        (out / sub).mkdir(parents=True, exist_ok=True)
    cands = scan_discontinuities(subset)
    cands.to_csv(out / "candidatos_descontinuidade.csv", index=False)
    print(f"[{subset}] {df['unit_id'].nunique()} holdout engines x {len(models)} models; "
          f"{int(cands['candidate'].sum())} Nc-drop candidates in the whole fleet", flush=True)
    per_model = Parallel(n_jobs=jobs)(delayed(_model_job)(df, m, subset, cands, out) for m in models)
    Parallel(n_jobs=jobs)(delayed(_engine_job)(eng, models, subset, int(u), _events(subset, int(u), cands), out)
                          for u, eng in df.groupby("unit_id"))
    metrics = pd.DataFrame([r for rows in per_model for r in rows])
    metrics.to_csv(out / "metricas_por_motor.csv", index=False)
    return metrics


def main(argv: list[str] | None = None) -> int:
    p = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    p.add_argument("--subsets", nargs="+", default=["FD001", "FD002", "FD003", "FD004"])
    p.add_argument("--jobs", type=int, default=6)
    p.add_argument("--output-dir", type=Path, default=OUT_DEFAULT)
    a = p.parse_args(argv)
    for s in a.subsets:
        run_subset(s.upper(), a.output_dir, a.jobs)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/selecao_hiperparametros.py
"""Busca de hiperparâmetros (inclui a janela) por validação cruzada agrupada por motor, só nos motores de treino.

WHAT   Para cada (subconjunto, modelo), um estudo Optuna (TPE, semente fixa) escolhe os hiperparâmetros, o tamanho da janela, se há atributos
       temporais e se os sensores são padronizados por regime. O objetivo é o RMSE (RUL truncada em 125) nos motores de validação de um
       GroupKFold por motor. A janela (`window`) é tamanho máximo dos atributos temporais (5, 10 e `window`) ou `seq_length` das redes.
WHY    Substituir hiperparâmetros fixos e escolhas de janela feitas à mão por uma seleção em validação, sem tocar o teste oficial.
       Regra do erro-padrão (1-SE): entre as tentativas com RMSE médio dentro de 1 erro-padrão (das dobras) do melhor, escolhe-se a de
       MENOR janela (0 = sem atributos temporais), isto é, a mais simples; evita escolher a janela por ruído.
[LEAKAGE] Tudo é ajustado só nos motores de treino de cada dobra (regimes, estatísticas por regime, escalonadores, atributos causais).
          O teste oficial não é lido aqui.

Rodar (da raiz; no Colab, com `--device cuda` para as redes e o XGBoost/CatBoost):
    python -m pdm_cmaps.experiments.selecao_hiperparametros --subsets FD004 --models XGBoost "PyTorch RNN (LSTM)" --output-dir results/hiperparametros
Saída: <output-dir>/<subconjunto>/<modelo>/{melhores.json, tentativas.csv, estudo.db}. Retomável (o estudo é um SQLite).
"""

from __future__ import annotations

import argparse
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

from pdm_cmaps.data.features import CausalFeatureEngineer
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.domain.rul import compute_piecewise_rul
from pdm_cmaps.domain.schema import DEFAULT_HEALTHY_CYCLE_LIMIT, SETTING_COLUMNS
from pdm_cmaps.experiments.benchmark_oficial_completo import _regime_frames
from pdm_cmaps.modeling.registry import get_model
from pdm_cmaps.pipeline.extra_models import fit_predict

WINDOWS = (10, 15, 20, 30, 40, 50, 60)
NETS = ("PyTorch RNN (LSTM)", "PyTorch CNN", "PyTorch Transformer")
MODELS = ("Ridge", "Random Forest", "XGBoost", "XGBoost (NASA Loss)", "LightGBM", "CatBoost", "MLP", *NETS)
# tentativas por modelo (escalonadas por --trials-scale); redes são caras, ficam com menos
BUDGET = {"Ridge": 30, "MLP": 30, "Random Forest": 40, "XGBoost": 60, "XGBoost (NASA Loss)": 60, "LightGBM": 60, "CatBoost": 40,
          "PyTorch RNN (LSTM)": 20, "PyTorch CNN": 20, "PyTorch Transformer": 20}
N_SPLITS = {m: (3 if m in NETS or m.startswith("MiniRocket") else 5) for m in MODELS}


def slug(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")


# --------------------------------------------------------------------------------------------------------------------
# espaço de busca e construção do modelo
# --------------------------------------------------------------------------------------------------------------------
def sugerir(trial, model: str) -> dict:
    """Hiperparâmetros de uma tentativa. `window` = 0 quando o modelo não usa janela."""
    p: dict = {"representation": trial.suggest_categorical("representation", ["global", "regime"])}
    sequence = model in NETS or model.startswith("MiniRocket")
    if sequence:
        p["window"] = trial.suggest_categorical("window", [w for w in WINDOWS])
        p["temporal"] = False
    else:
        p["temporal"] = trial.suggest_categorical("temporal", [False, True])
        p["window"] = trial.suggest_categorical("window", list(WINDOWS)) if p["temporal"] else 0
    h: dict = {}
    if model == "Ridge":
        h["alpha"] = trial.suggest_float("alpha", 1e-2, 1e3, log=True)
    elif model == "Random Forest":
        h.update(n_estimators=trial.suggest_int("n_estimators", 50, 300, step=50), max_depth=trial.suggest_int("max_depth", 6, 24),
                 min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10), max_features=trial.suggest_float("max_features", 0.3, 1.0))
    elif model in ("XGBoost", "XGBoost (NASA Loss)"):
        h.update(n_estimators=trial.suggest_int("n_estimators", 50, 600, step=50), max_depth=trial.suggest_int("max_depth", 3, 10),
                 learning_rate=trial.suggest_float("learning_rate", 0.02, 0.3, log=True), subsample=trial.suggest_float("subsample", 0.5, 1.0),
                 colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0), min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
                 reg_lambda=trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True))
    elif model == "LightGBM":
        h.update(n_estimators=trial.suggest_int("n_estimators", 50, 600, step=50), max_depth=trial.suggest_int("max_depth", 3, 12),
                 learning_rate=trial.suggest_float("learning_rate", 0.02, 0.3, log=True), num_leaves=trial.suggest_int("num_leaves", 15, 127),
                 subsample=trial.suggest_float("subsample", 0.5, 1.0), subsample_freq=1, colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
                 min_child_samples=trial.suggest_int("min_child_samples", 5, 100), reg_lambda=trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True))
    elif model == "CatBoost":
        h.update(iterations=trial.suggest_int("iterations", 100, 800, step=100), depth=trial.suggest_int("depth", 4, 10),
                 learning_rate=trial.suggest_float("learning_rate", 0.02, 0.3, log=True), l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True))
    elif model == "MLP":
        layers = {"64-32": (64, 32), "128-64": (128, 64), "128-64-32": (128, 64, 32), "256-128": (256, 128)}
        h.update(hidden_layer_sizes=layers[trial.suggest_categorical("hidden", list(layers))], alpha=trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
                 learning_rate_init=trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True))
    elif model == "MiniRocket + RidgeCV":
        h["num_kernels"] = trial.suggest_categorical("num_kernels", [500, 1000, 2000])
    elif model in NETS:
        h.update(learning_rate=trial.suggest_float("learning_rate", 5e-4, 5e-3, log=True), batch_size=trial.suggest_categorical("batch_size", [128, 256, 512]),
                 dropout=trial.suggest_float("dropout", 0.0, 0.4))
        if model == "PyTorch RNN (LSTM)":
            h.update(hidden_dim=trial.suggest_categorical("hidden_dim", [32, 64, 128]), num_layers=trial.suggest_int("num_layers", 1, 3))
        elif model == "PyTorch CNN":
            h.update(num_filters=trial.suggest_categorical("num_filters", [32, 64, 128]), kernel_size=trial.suggest_categorical("kernel_size", [3, 5]))
        else:
            h.update(d_model=trial.suggest_categorical("d_model", [32, 64, 128]), nhead=trial.suggest_categorical("nhead", [2, 4]),
                     num_layers=trial.suggest_int("num_layers", 1, 3), dim_feedforward=trial.suggest_categorical("dim_feedforward", [64, 128, 256]))
    else:
        raise ValueError(f"modelo desconhecido: {model!r}")
    p["hyper"] = h
    return p


def montar_modelo(model: str, params: dict, seed: int, device: str = "cpu", n_jobs: int = 2):
    """Modelo novo, com os hiperparâmetros de `params`; (tipo, estimador) como em `pipeline.extra_models.fit_predict`."""
    h = dict(params["hyper"])
    gpu = device != "cpu"
    if model == "Ridge":
        return "sklearn", get_model("ridge", seed, **h)
    if model == "Random Forest":
        return "sklearn", get_model("rf", seed, n_jobs=n_jobs, **h)
    if model == "XGBoost":
        extra = {"device": "cuda", "tree_method": "hist"} if gpu else {}
        return "sklearn", get_model("xgboost", seed, n_jobs=n_jobs, **h, **extra)
    if model == "XGBoost (NASA Loss)":
        return "sklearn", get_model("xgboost_nasa_loss", seed, n_jobs=n_jobs, **h)
    if model == "LightGBM":
        return "sklearn", get_model("lightgbm", seed, n_jobs=n_jobs, **h)
    if model == "CatBoost":
        extra = {"task_type": "GPU"} if gpu else {}
        return "sklearn", get_model("catboost", seed, n_jobs=n_jobs, **h, **extra)
    if model == "MLP":
        return "sklearn", get_model("mlp", seed, **h)
    if model == "MiniRocket + RidgeCV":
        from pdm_cmaps.modeling.minirocket import MiniRocketRULModel
        return "deep", MiniRocketRULModel(seq_length=params["window"], random_state=seed, device=device, **h)
    from pdm_cmaps.modeling.torch_models import PyTorchCNNModel, PyTorchRNNModel, PyTorchTransformerModel
    cls = {"PyTorch RNN (LSTM)": PyTorchRNNModel, "PyTorch CNN": PyTorchCNNModel, "PyTorch Transformer": PyTorchTransformerModel}[model]
    return "deep", cls(seq_length=params["window"], random_state=seed, device=device, **h)


# --------------------------------------------------------------------------------------------------------------------
# dados: representação (global ou por regime) e atributos temporais, ajustados só no treino
# --------------------------------------------------------------------------------------------------------------------
def montar_dados(train: pd.DataFrame, evalf: pd.DataFrame, sensors: list[str], n_regimes: int, params: dict):
    """(treino, avaliação, colunas) prontos para `fit_predict`; tudo ajustado em `train`. A ordem das linhas de `evalf` pode mudar (ordenada por motor e ciclo)."""
    if params["representation"] == "regime":
        tr, ev, cols = _regime_frames(train, evalf, sensors, n_regimes)
        sens = [c for c in cols if c.endswith("_norm")]
    else:
        tr, ev, cols, sens = train, evalf, [*sensors, *SETTING_COLUMNS, "cycle"], sensors
    if params["temporal"]:
        eng = CausalFeatureEngineer(sens, windows=tuple(sorted({5, 10, int(params["window"])}))).fit(tr, DEFAULT_HEALTHY_CYCLE_LIMIT)
        tr_fe, ev_fe = eng.transform(tr), eng.transform(ev)
        cols = [*cols, *[c for c in tr_fe.columns if c not in tr.columns]]
        return tr_fe, ev_fe, cols
    return tr.sort_values(["unit_id", "cycle"]).reset_index(drop=True), ev.sort_values(["unit_id", "cycle"]).reset_index(drop=True), cols


def preparar(subset: str):
    """(quadro de treino com RUL, sensores informativos, nº de regimes) de um subconjunto; só os motores de treino do C-MAPSS."""
    loader = CMapssLoader()
    frame = loader.load(subset)
    frame["RUL"] = compute_piecewise_rul(frame)
    return frame, loader.informative_sensors(frame), (6 if subset in ("FD002", "FD004") else 1)


def validar(frame: pd.DataFrame, sensors: list[str], n_regimes: int, model: str, params: dict, seed: int, n_splits: int, device: str = "cpu",
            n_jobs: int = 2, trial=None) -> list[float]:
    """RMSE (RUL truncada) em cada dobra de um GroupKFold por motor, só nos motores de treino. Poda após cada dobra se `trial` for dado."""
    from sklearn.model_selection import GroupKFold

    import optuna

    scores: list[float] = []
    for k, (tr_idx, va_idx) in enumerate(GroupKFold(n_splits=n_splits).split(frame, groups=frame["unit_id"])):
        tr_df, va_df = frame.iloc[tr_idx].reset_index(drop=True), frame.iloc[va_idx].reset_index(drop=True)
        kind, est = montar_modelo(model, params, seed, device)
        tr, ev, cols = montar_dados(tr_df, va_df, sensors, n_regimes, params)
        pred = fit_predict(kind, est, tr, cols, ev)
        scores.append(float(np.sqrt(np.mean((np.asarray(pred, dtype=float) - ev["RUL"].to_numpy(dtype=float)) ** 2))))
        if trial is not None:
            trial.report(float(np.mean(scores)), k)
            if trial.should_prune():
                raise optuna.TrialPruned()
    return scores


# --------------------------------------------------------------------------------------------------------------------
# escolha final: regra do erro-padrão
# --------------------------------------------------------------------------------------------------------------------
def escolher_um_se(tentativas: pd.DataFrame) -> pd.Series:
    """Entre as tentativas dentro de 1 erro-padrão do melhor RMSE médio, a de MENOR janela (desempate: menor RMSE médio).

    `tentativas` tem as colunas `rmse_mean`, `rmse_se` e `window` (0 = sem janela). Só entram tentativas completas (sem poda)."""
    d = tentativas.dropna(subset=["rmse_mean"])
    if d.empty:
        raise ValueError("nenhuma tentativa completa")
    best = d.loc[d["rmse_mean"].idxmin()]
    limite = float(best["rmse_mean"] + (best["rmse_se"] if np.isfinite(best["rmse_se"]) else 0.0))
    dentro = d[d["rmse_mean"] <= limite]
    return dentro.sort_values(["window", "rmse_mean"]).iloc[0]


def tabela_de_tentativas(study) -> pd.DataFrame:
    rows = []
    for t in study.trials:
        folds = t.user_attrs.get("fold_rmse")
        if t.state.name != "COMPLETE" or not folds:
            continue
        a = np.asarray(folds, dtype=float)
        rows.append({"trial": t.number, "rmse_mean": float(a.mean()), "rmse_se": float(a.std(ddof=1) / np.sqrt(len(a))) if len(a) > 1 else float("nan"),
                     "window": int(t.user_attrs["params"]["window"]), "params": json.dumps(t.user_attrs["params"], ensure_ascii=False)})
    return pd.DataFrame(rows)


def buscar(subset: str, model: str, out: Path, trials: int | None = None, seed: int = 42, device: str = "cpu", n_jobs: int = 2) -> dict:
    """Roda (ou retoma) o estudo de um (subconjunto, modelo) e grava `melhores.json`."""
    import optuna

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    target = out / subset / slug(model)
    target.mkdir(parents=True, exist_ok=True)
    frame, sensors, n_regimes = preparar(subset)
    study = optuna.create_study(study_name=f"{subset}_{slug(model)}", storage=f"sqlite:///{target / 'estudo.db'}", load_if_exists=True, direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=seed), pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=1))
    alvo = trials if trials is not None else BUDGET[model]
    feitas = len([t for t in study.trials if t.state.name in ("COMPLETE", "PRUNED")])
    t0 = time.time()

    def objetivo(trial):
        params = sugerir(trial, model)
        trial.set_user_attr("params", params)
        scores = validar(frame, sensors, n_regimes, model, params, seed, N_SPLITS[model], device, n_jobs, trial)
        trial.set_user_attr("fold_rmse", scores)
        return float(np.mean(scores))

    if alvo - feitas > 0:
        study.optimize(objetivo, n_trials=alvo - feitas, gc_after_trial=True)
    tent = tabela_de_tentativas(study)
    tent.to_csv(target / "tentativas.csv", index=False)
    escolhida = escolher_um_se(tent)
    melhor_media = tent.loc[tent["rmse_mean"].idxmin()]
    resultado = {"subset": subset, "model": model, "params": json.loads(escolhida["params"]), "cv_rmse": float(escolhida["rmse_mean"]), "cv_se": float(escolhida["rmse_se"]),
                 "melhor_rmse_medio": float(melhor_media["rmse_mean"]), "melhor_trial": int(melhor_media["trial"]), "trial_escolhida": int(escolhida["trial"]),
                 "n_tentativas_completas": int(len(tent)), "n_tentativas_total": int(len(study.trials)), "n_splits": N_SPLITS[model], "seed": seed,
                 "segundos": round(time.time() - t0, 1)}
    (target / "melhores.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=2))
    print(f"[{subset}] {model}: CV RMSE {resultado['cv_rmse']:.2f} ± {resultado['cv_se']:.2f} (janela {resultado['params']['window']}, "
          f"{resultado['params']['representation']}), {resultado['n_tentativas_completas']} completas de {resultado['n_tentativas_total']} ({resultado['segundos']:.0f} s)", flush=True)
    return resultado


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--subsets", nargs="+", default=["FD001", "FD002", "FD003", "FD004"])
    ap.add_argument("--models", nargs="+", default=list(MODELS), choices=list(MODELS))
    ap.add_argument("--trials", type=int, default=None, help="tentativas por estudo (padrão: orçamento do modelo)")
    ap.add_argument("--trials-scale", type=float, default=1.0, help="multiplica o orçamento padrão de cada modelo (ignorado com --trials)")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--device", default="cpu", help="cpu ou cuda (só na busca; a refit final do teste oficial é sempre na CPU)")
    ap.add_argument("--n-jobs", type=int, default=2, help="threads dos modelos de árvores")
    ap.add_argument("--output-dir", type=Path, default=Path("results/hiperparametros"))
    a = ap.parse_args(argv)
    for subset in a.subsets:
        for model in a.models:
            n = a.trials if a.trials is not None else max(1, int(round(BUDGET[model] * a.trials_scale)))
            buscar(subset, model, a.output_dir, n, a.seed, a.device, a.n_jobs)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/experiments/virtual_sensor_models.py
"""Virtual sensor for EVERY informative sensor, EVERY model and EVERY held-out engine (feeds the dashboards).

WHAT: the pipeline fits one virtual sensor (default ExtraTrees) for four primary sensors and, historically,
saved the multi-model comparison only for engine 179. This study repeats the same protocol for all
informative sensors and a pool of regressors, on the same engine split, regimes and healthy window as the
pipeline, and writes the full trace of the held-out engines so any dashboard filter (dataset, engine,
sensor, model) has data.

[LEAKAGE] same discipline as the pipeline: engine split first; sensor filter, regimes and every regressor
are fit on TRAINING engines (healthy window only); held-out engines are only scored.
[ASSUMPTION] the regressors use the exogenous inputs (settings + ambient channels), exactly like the
pipeline's virtual sensor, so the comparison isolates the regressor, not the feature set.

Run:  .venv/bin/python -m pdm_cmaps.experiments.virtual_sensor_models [--subset FD004 | --all] [--models ...]
Output: results/virtual_sensor_models/<subset>/{trace.csv.gz, per_engine_alarms.csv, model_scores.csv}
"""

from __future__ import annotations

import argparse
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin, clone

from pdm_cmaps.config import DEFAULT_RESULTS_DIR
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.data.splits import partition_healthy_and_lifecycle, split_engine_units
from pdm_cmaps.detection.regime import RegimeDecoupler
from pdm_cmaps.detection.virtual_sensor import VirtualSensor
from pdm_cmaps.detection.virtual_sensor_metrics import virtual_sensor_r2
from pdm_cmaps.domain.rul import compute_piecewise_rul
from pdm_cmaps.domain.schema import DEFAULT_HEALTHY_CYCLE_LIMIT
from pdm_cmaps.modeling.registry import get_model

OUT_DEFAULT = DEFAULT_RESULTS_DIR / "virtual_sensor_models"
SUBSETS = ("FD001", "FD002", "FD003", "FD004")
# display name -> registry key and kwargs (small, untuned, same spirit as the RUL pool)
MODELS = {
    # n_jobs=1: os dados são pequenos (janela saudável); com várias threads o custo é só sincronização
    "Extra Trees": ("extra_trees", {"n_estimators": 60, "max_depth": 12, "n_jobs": 1}),
    "Random Forest": ("random_forest", {"n_estimators": 60, "max_depth": 12, "n_jobs": 1}),
    "XGBoost": ("xgboost", {"n_estimators": 80, "max_depth": 6, "n_jobs": 1}),
    "LightGBM": ("lightgbm", {"n_estimators": 80, "max_depth": 6, "verbose": -1, "n_jobs": 1}),
    "CatBoost": ("catboost", {"iterations": 100, "depth": 6, "n_jobs": 1}),
    "HistGradientBoosting": ("gradient_boosting", {"max_iter": 80, "max_depth": 6}),
    "Ridge": ("ridge", {"alpha": 10}),
}
TRACE_COLS = ["unit_id", "cycle", "regime_id", "actual", "prediction", "z_score", "ewma_abs_z", "ewma_ucl"]


class SafeRegressor(BaseEstimator, RegressorMixin):
    """Envolve um regressor e devolve a constante quando o alvo é constante (alguns sensores não variam dentro de um regime;
    CatBoost e outros recusam ajustar). Fora disso, é o próprio regressor."""

    def __init__(self, base=None):
        self.base = base

    def fit(self, X, y):
        y = np.asarray(y, dtype=float)
        self.const_ = float(y[0]) if y.size and np.ptp(y) == 0 else None
        self.model_ = None if self.const_ is not None else clone(self.base).fit(X, y)
        return self

    def predict(self, X):
        n = len(X)
        return np.full(n, self.const_) if self.const_ is not None else np.asarray(self.model_.predict(X), dtype=float)


def run_subset(subset: str, out_dir: Path, models: list[str], seed: int = 42, H: int = DEFAULT_HEALTHY_CYCLE_LIMIT,
               min_consecutive: int = 2, engine_offset: bool = True) -> None:
    t0 = time.time()
    loader = CMapssLoader()
    frame = loader.load(subset)
    frame["RUL"] = compute_piecewise_rul(frame)
    train_units, test_units = split_engine_units(frame, test_ratio=0.25, random_state=seed)
    sensors = loader.informative_sensors(frame.loc[frame["unit_id"].isin(train_units)])
    parts = partition_healthy_and_lifecycle(frame, train_units, test_units, H)
    n_regimes = 6 if subset in {"FD002", "FD004"} else 1
    regimes = RegimeDecoupler(n_clusters=n_regimes, random_state=seed).fit(parts["full_lifecycle_train"])
    healthy_train = regimes.transform(parts["healthy_train"])
    test_full = regimes.transform(parts["full_lifecycle_test"])
    print(f"[{subset}] {len(test_units)} held-out engines | {len(sensors)} sensors | {len(models)} models", flush=True)

    traces, alarms, scores = [], [], []
    for sensor in sensors:
        for name in models:
            key, kw = MODELS[name]
            vs = VirtualSensor(target_sensor=sensor, mode="exogenous", ewma_alpha=0.10, healthy_cycle_limit=H,
                               min_consecutive_breaches=min_consecutive, random_state=seed,
                               estimator=SafeRegressor(get_model(key, seed, **kw)),
                               engine_offset=engine_offset, offset_window=H // 2 if engine_offset else None)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                scored, per_engine, summary = vs.evaluate(healthy_train, test_full)
            t = scored.rename(columns={sensor: "actual"})[TRACE_COLS].copy()
            t.insert(0, "sensor", sensor)
            t.insert(1, "model", name)
            traces.append(t)
            per_engine.insert(0, "sensor", sensor)
            per_engine.insert(1, "model", name)
            alarms.append(per_engine)
            resid = (scored["residual"].to_numpy())
            healthy = scored["cycle"].to_numpy() <= H
            scores.append({"sensor": sensor, "model": name, "rmse_holdout": float(np.sqrt(np.mean(resid ** 2))),
                           "rmse_healthy_window": float(np.sqrt(np.mean(resid[healthy] ** 2))) if healthy.any() else np.nan,
                           **{k: v for k, v in summary.items()}})
        print(f"  [{subset}] {sensor} ok ({time.time() - t0:.0f}s)", flush=True)

    target = out_dir / subset
    target.mkdir(parents=True, exist_ok=True)
    pd.concat(traces, ignore_index=True).round({"actual": 3, "prediction": 3, "z_score": 3, "ewma_abs_z": 4, "ewma_ucl": 4}).to_csv(
        target / "trace.csv.gz", index=False)
    pd.concat(alarms, ignore_index=True).to_csv(target / "per_engine_alarms.csv", index=False)
    pd.DataFrame(scores).to_csv(target / "model_scores.csv", index=False)
    add_r2_columns(target, H)
    print(f"[{subset}] done in {time.time() - t0:.0f}s -> {target}", flush=True)


R2_COLUMNS = ["mae_holdout", "nrmse_within_regime_holdout", "r2_holdout", "r2_healthy", "r2_late_model", "r2_late_motor_base", "r2_late_model_offset", "pct_between_engine",
              "r2_late_within_regime", "r2_holdout_within_regime", "noise_ceiling_late", "frac_ceiling"]


def add_r2_columns(target: Path, H: int = DEFAULT_HEALTHY_CYCLE_LIMIT) -> pd.DataFrame:
    """Calcula as métricas de R² (ver detection/virtual_sensor_metrics.py) a partir do trace gravado e grava em model_scores.csv.

    Não reajusta nenhum modelo: serve também para atualizar resultados antigos (`--recompute-r2`)."""
    scores = pd.read_csv(target / "model_scores.csv").drop(columns=R2_COLUMNS, errors="ignore")
    r2 = virtual_sensor_r2(pd.read_csv(target / "trace.csv.gz"), H)
    out = scores.merge(r2, on=["sensor", "model"], how="left")
    out.to_csv(target / "model_scores.csv", index=False)
    return out


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--subset", choices=SUBSETS, default=None)
    ap.add_argument("--all", action="store_true")
    ap.add_argument("--models", nargs="*", default=list(MODELS), choices=list(MODELS))
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--out", type=Path, default=None, help=f"padrão: {OUT_DEFAULT} (ou {OUT_DEFAULT}_sem_offset com --no-engine-offset)")
    ap.add_argument("--engine-offset", action=argparse.BooleanOptionalAction, default=True, help="remove o deslocamento de cada motor (média do z nos ciclos 1 a H//2) do resíduo; padrão ligado; --no-engine-offset reproduz o comportamento anterior")
    ap.add_argument("--recompute-r2", action="store_true", help="só recalcula as colunas de R² de model_scores.csv a partir do trace existente (não reajusta modelos)")
    a = ap.parse_args(argv)
    if a.out is None:
        a.out = OUT_DEFAULT if a.engine_offset else OUT_DEFAULT.with_name(OUT_DEFAULT.name + "_sem_offset")
    for s in (SUBSETS if a.all or a.subset is None else (a.subset,)):
        if a.recompute_r2:
            add_r2_columns(a.out / s)
            print(f"[{s}] R² recalculado", flush=True)
        else:
            run_subset(s, a.out, a.models, a.seed, engine_offset=a.engine_offset)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

### 2.2 Pacote `pdm_cmaps`: reporting

gráficos e relatórios

5 arquivos.

In [ ]:
%%writefile projeto/src/pdm_cmaps/reporting/__init__.py
"""Figures only (matplotlib and Plotly). Nothing here computes a result; it draws tables other layers produced."""

In [ ]:
%%writefile projeto/src/pdm_cmaps/reporting/monitoring_figures.py
"""Thesis figures of the monitoring layer (dataviz method; tokens and mark specs in viz_style.py).

Each function draws ONE message and has a CSV as its table view:
  plot_regime_variance          regime_variance_by_subset.csv           why regime decoupling is needed
  plot_residual_control_chart   alert_timeline / spc residual trace     what an alarm looks like on one engine
  plot_rul_trajectories         alert_timeline.csv                      RUL, uncertainty and criticality over time
  plot_strategy_tradeoff        criticality_policy_sensitivity.csv      missed-critical vs false-alarm trade-off
  plot_strategy_tradeoff_repeated  monitoring_repeated_strategies_raw.csv   same, mean +- std over seeds
  plot_model_ci                 thesis_bootstrap_ci.csv                 model comparison with uncertainty
"""

from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import FuncFormatter

from pdm_cmaps.reporting.plots import save_figure
from pdm_cmaps.reporting.viz_style import INK, INK_2, LEVEL_NAMES_PT, LINE_W, MARKER_PT, MUTED, SERIES, STATUS, SURFACE, apply_style, ring_marker_kwargs, style_axes

apply_style()

STRATEGY_ORDER = ("point", "interval", "interval+anomaly")
STRATEGY_LABEL = {
    "point": "Ponto (RUL prevista)",
    "interval": "Limite inferior do intervalo",
    "interval+anomaly": "Intervalo + anomalia SPC",
}
STRATEGY_MARKER = {"point": "o", "interval": "s", "interval+anomaly": "s"}
STRATEGY_COLOR = dict(zip(STRATEGY_ORDER, SERIES))


def _policy_label(policy: str) -> str:
    return policy.replace("critical<=", "crítico ≤ ")


def _level_runs(levels: np.ndarray):
    start = 0
    for i in range(1, len(levels) + 1):
        if i == len(levels) or levels[i] != levels[start]:
            yield start, i - 1, int(levels[start])
            start = i


# ---------------------------------------------------------------------------
# regime variance (the thesis' strongest evidence)
# ---------------------------------------------------------------------------

def plot_regime_variance(table: pd.DataFrame, path: Path):
    """One series: share of T50 variance explained by flight regime, per C-MAPSS subset."""
    d = table.groupby(["Subset", "Regimes"], as_index=False)["Between_regime_pct"].mean().sort_values("Subset", ascending=False)
    fig, ax = plt.subplots(figsize=(7.2, 0.62 * len(d) + 1.7))
    y = np.arange(len(d))
    ax.barh(y, d["Between_regime_pct"], height=0.26, color=SERIES[0])
    ax.set_yticks(y, [f"{s}  ·  {int(k)} regime{'s' if k > 1 else ''}" for s, k in zip(d["Subset"], d["Regimes"])], color=INK_2)
    for yi, v in zip(y, d["Between_regime_pct"]):
        ax.text(v + 1.2, yi, f"{v:.1f}%".replace(".", ","), va="center", ha="left", color=INK, fontsize=10)
    ax.set_xlim(0, 108)
    ax.set_xticks([0, 25, 50, 75, 100], ["0", "25%", "50%", "75%", "100%"])
    style_axes(ax, "x")
    ax.spines["bottom"].set_visible(False)
    ax.set_xlabel("Variância do T50 explicada pelo regime de voo (motores de teste)")
    ax.set_title("Em voo com vários regimes, o regime domina o sinal bruto")
    fig.tight_layout()
    return save_figure(fig, path)


# ---------------------------------------------------------------------------
# detection on one engine
# ---------------------------------------------------------------------------

def plot_residual_control_chart(trace: pd.DataFrame, unit_id: int, first_alarm_cycle, path: Path, healthy_cycle_limit: int):
    """Residual z-score and EWMA control chart for ONE engine; the alarm is marked on the EWMA."""
    eng = trace.loc[trace["unit_id"] == unit_id].sort_values("cycle")
    cycles = eng["cycle"].to_numpy()
    eol = int(cycles.max())
    fig, axes = plt.subplots(2, 1, figsize=(10, 6.6), sharex=True, gridspec_kw={"hspace": 0.28})
    for ax in axes:
        style_axes(ax, "y")
        ax.axvspan(0.5, healthy_cycle_limit + 0.5, color=MUTED, alpha=0.10, lw=0)
    top, bottom = axes
    top.plot(cycles, eng["z_score"], color=SERIES[0], lw=LINE_W)
    for level in (3, -3):
        top.axhline(level, color=INK_2, lw=1.2, ls=(0, (4, 3)))
    top.text(healthy_cycle_limit + 6, 3.0, "limite ±3σ", ha="left", va="bottom", color=INK_2, fontsize=8.5)
    top.text(healthy_cycle_limit / 2, top.get_ylim()[1] * 0.95, f"calibração saudável\n(ciclos 1–{healthy_cycle_limit})", ha="center", va="top", color=INK_2, fontsize=8.5)
    top.set_ylabel("z do resíduo")
    top.set_title("Resíduo do sensor virtual (real − esperado em condição saudável), padronizado")

    bottom.plot(cycles, eng["ewma_abs_z"], color=SERIES[0], lw=LINE_W, label="EWMA de |z| (λ = 0,10)")
    bottom.plot(cycles, eng["ewma_ucl"], color=INK_2, lw=1.2, ls=(0, (4, 3)), label="Limite de controle (3σ)")
    handles = [Line2D([], [], color=SERIES[0], lw=LINE_W), Line2D([], [], color=INK_2, lw=1.2, ls=(0, (4, 3)))]
    bottom.legend(handles, ["EWMA de |z| (λ = 0,10)", "Limite de controle (3σ)"], loc="upper left", ncol=2, bbox_to_anchor=(0.0, 1.02))
    if first_alarm_cycle is not None and not pd.isna(first_alarm_cycle):
        a = int(first_alarm_cycle)
        y_alarm = float(eng.loc[eng["cycle"] == a, "ewma_abs_z"].iloc[0])
        bottom.vlines(a, 0, y_alarm, color=INK_2, lw=1.0)
        bottom.plot([a], [y_alarm], marker="o", **ring_marker_kwargs(STATUS[3], 10))
        bottom.annotate(f"Primeiro alarme persistente: ciclo {a}\nantecedência de {eol - a} ciclos até o fim de vida",
                        xy=(a, y_alarm), xytext=(-12, 6), textcoords="offset points", ha="right", va="bottom", color=INK, fontsize=9)
    bottom.set_ylabel("EWMA de |z|")
    bottom.set_xlabel("Ciclo operacional (o último ciclo é o fim de vida)")
    bottom.set_title("Carta de controle")
    fig.suptitle(f"Ativo {unit_id} (motor do holdout)", x=0.01, ha="left", fontsize=12, fontweight="semibold")
    fig.subplots_adjust(top=0.88)
    return save_figure(fig, path)


# ---------------------------------------------------------------------------
# RUL with uncertainty and criticality
# ---------------------------------------------------------------------------

def plot_rul_trajectories(timeline: pd.DataFrame, unit_ids: list[int], path: Path, strategy: str = "interval+anomaly", alpha: float = 0.10):
    """RUL real vs prevista com intervalo conformal; o fundo mostra o nível de criticidade ao longo do tempo."""
    n = len(unit_ids)
    cols = 2
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(11, 3.3 * rows + 0.9), squeeze=False, sharey=True)
    ymax, strip_lo, strip_hi = 146, 131, 143
    for ax, unit in zip(axes.ravel(), unit_ids):
        style_axes(ax, "y")
        eng = timeline.loc[timeline["unit_id"] == unit].sort_values("cycle")
        c = eng["cycle"].to_numpy()
        span = max(c.max() - c.min(), 1)
        ax.fill_between(c, eng["RUL_lower"], eng["RUL_upper"], color=SERIES[1], alpha=0.18, lw=0)
        ax.plot(c, eng["RUL_true"], color=INK, lw=LINE_W)
        ax.plot(c, eng["RUL_pred"], color=SERIES[1], lw=LINE_W)
        # criticality strip: status color + printed level name (never color alone); 2 px surface gap between runs
        for s_i, e_i, lvl in _level_runs(eng[f"level_{strategy}"].to_numpy()):
            ax.add_patch(Rectangle((c[s_i] - 0.5, strip_lo), c[e_i] - c[s_i] + 1, strip_hi - strip_lo,
                                   facecolor=STATUS[lvl], edgecolor=SURFACE, linewidth=1.5, clip_on=False))
            if (c[e_i] - c[s_i] + 1) / span >= 0.14:  # name only where it fits; the legend carries the rest
                ax.text((c[s_i] + c[e_i]) / 2, (strip_lo + strip_hi) / 2, LEVEL_NAMES_PT[lvl], ha="center", va="center",
                        color="#ffffff" if lvl in (0, 3) else INK, fontsize=8)
        ax.set_ylim(0, ymax)
        ax.set_yticks([0, 25, 50, 75, 100, 125])
        ax.set_title(f"Ativo {unit}")
        ax.set_xlabel("Ciclo")
    for ax in axes[:, 0]:
        ax.set_ylabel("RUL (ciclos)")
    for ax in axes.ravel()[n:]:
        ax.axis("off")
    handles = [
        Line2D([], [], color=INK, lw=LINE_W), Line2D([], [], color=SERIES[1], lw=LINE_W),
        Patch(color=SERIES[1], alpha=0.30),
        *[Patch(color=STATUS[i]) for i in range(4)],
    ]
    labels = ["RUL real", "RUL prevista", f"Intervalo de {int(round((1 - alpha) * 100))}%", *[f"Nível {name}" for name in LEVEL_NAMES_PT]]
    fig.legend(handles, labels, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle("RUL com incerteza e nível de criticidade ao longo da vida (motores do holdout)", x=0.01, ha="left", fontsize=12, fontweight="semibold")
    fig.tight_layout(rect=(0, 0.09, 1, 0.95))
    return save_figure(fig, path)


# ---------------------------------------------------------------------------
# criticality trade-off
# ---------------------------------------------------------------------------

def _strategy_handles() -> list[Line2D]:
    out = []
    for s in STRATEGY_ORDER:
        hollow = s == "interval+anomaly"
        out.append(Line2D([], [], marker=STRATEGY_MARKER[s], **ring_marker_kwargs(STRATEGY_COLOR[s], 14 if hollow else MARKER_PT + 1, hollow)))
    return out


def plot_strategy_tradeoff(sensitivity: pd.DataFrame, path: Path):
    """Críticos não alertados × falsos alertas; cada ponto é uma política de limiares."""
    fig, ax = plt.subplots(figsize=(7.2, 5.0))
    style_axes(ax, "both")
    d = sensitivity.copy()
    d["x"], d["y"] = 100 * d["False_alarm_rate_normal"], 100 * d["Missed_critical_rate"]
    for s in ("interval+anomaly", "point", "interval"):  # hollow ring first so the filled marker on top does not hide it
        g = d.loc[d["Strategy"] == s]
        hollow = s == "interval+anomaly"
        ax.plot(g["x"], g["y"], marker=STRATEGY_MARKER[s], clip_on=False, **ring_marker_kwargs(STRATEGY_COLOR[s], 14 if hollow else MARKER_PT + 1, hollow))
    for row in d.loc[d["Strategy"] == "interval"].itertuples():
        ax.annotate(_policy_label(row.Policy), (row.x, row.y), xytext=(7, 7), textcoords="offset points", color=INK_2, fontsize=8.5)
    a = d.loc[d["Strategy"] == "interval", ["x", "y"]].to_numpy()
    b = d.loc[d["Strategy"] == "interval+anomaly", ["x", "y"]].to_numpy()
    if a.shape == b.shape and np.allclose(a, b):
        ax.text(0.98, 0.04, "“Limite inferior” e “intervalo + anomalia” coincidem:\no alarme SPC não mudou nenhum nível.",
                transform=ax.transAxes, ha="right", va="bottom", color=INK_2, fontsize=8.5)
    ax.set_xlim(left=-0.06 * max(d["x"].max(), 1e-9))
    ax.set_ylim(bottom=-0.08 * max(d["y"].max(), 1e-9))
    ax.set_xlabel("Falsos alertas em estado normal (% dos ciclos)")
    ax.set_ylabel("Estados críticos não alertados (% dos ciclos)")
    ax.set_title("Compromisso entre estratégias de criticidade\nmais perto da origem é melhor; cada ponto é uma política de limiares")
    ax.legend(_strategy_handles(), [STRATEGY_LABEL[s] for s in STRATEGY_ORDER], loc="upper right")
    fig.tight_layout()
    return save_figure(fig, path)


def plot_strategy_tradeoff_repeated(strat: pd.DataFrame, path: Path, policy: str = "critical<=15"):
    """Um painel por subconjunto: média ± desvio entre sementes, política fixa."""
    default = strat.loc[strat["Policy"] == policy]
    subsets = sorted(default["Subset"].unique())
    fig, axes = plt.subplots(1, len(subsets), figsize=(3.4 * len(subsets) + 0.6, 4.3), sharey=True, squeeze=False)
    for ax, subset in zip(axes.ravel(), subsets):
        style_axes(ax, "both")
        d = default.loc[default["Subset"] == subset]
        for s in ("interval+anomaly", "point", "interval"):
            g = d.loc[d["Strategy"] == s]
            hollow = s == "interval+anomaly"
            ax.errorbar(100 * g["False_alarm_rate_normal"].mean(), 100 * g["Missed_critical_rate"].mean(),
                        xerr=100 * g["False_alarm_rate_normal"].std(), yerr=100 * g["Missed_critical_rate"].std(),
                        fmt=STRATEGY_MARKER[s], color=STRATEGY_COLOR[s], ecolor=STRATEGY_COLOR[s], elinewidth=1.4, capsize=3,
                        markersize=14 if hollow else MARKER_PT + 1, clip_on=False,
                        markerfacecolor=SURFACE if hollow else STRATEGY_COLOR[s],
                        markeredgecolor=STRATEGY_COLOR[s] if hollow else SURFACE, markeredgewidth=2.0)
        ax.set_title(subset)
        ax.set_xlabel("Falsos alertas (%)")
    # limits from mean + std of every subset/strategy, set once after plotting: with sharey, a
    # one-sided set_ylim inside the loop froze the top at the first panel and clipped later ones
    g = default.groupby(["Subset", "Strategy"])
    ytop = 100 * (g["Missed_critical_rate"].mean() + g["Missed_critical_rate"].std().fillna(0)).max()
    ytop = max(ytop, 1.0)
    axes.ravel()[0].set_ylim(-0.06 * ytop, 1.08 * ytop)
    for ax, subset in zip(axes.ravel(), subsets):
        gx = default.loc[default["Subset"] == subset].groupby("Strategy")["False_alarm_rate_normal"]
        xtop = max(100 * (gx.mean() + gx.std().fillna(0)).max(), 1.0)
        ax.set_xlim(-0.08 * xtop, 1.08 * xtop)
    axes.ravel()[0].set_ylabel("Críticos não alertados (%)")
    fig.legend(_strategy_handles(), [STRATEGY_LABEL[s] for s in STRATEGY_ORDER], loc="lower center", ncol=3, bbox_to_anchor=(0.5, 0.0))
    fig.suptitle(f"Compromisso das estratégias (média ± desvio entre sementes; {_policy_label(policy)} ciclos)", x=0.01, ha="left", fontsize=12, fontweight="semibold")
    fig.tight_layout(rect=(0, 0.10, 1, 0.94))
    return save_figure(fig, path)


# ---------------------------------------------------------------------------
# model comparison
# ---------------------------------------------------------------------------

def plot_model_ci(boot: pd.DataFrame, path: Path, metric: str = "R2"):
    """Modelos ordenados pelo valor observado, com IC 95% por bootstrap de motores (uma série, sem legenda)."""
    d = boot.sort_values(f"{metric}_Observed").reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8, 0.33 * len(d) + 1.9))
    style_axes(ax, "x")
    y = np.arange(len(d))
    obs = d[f"{metric}_Observed"].to_numpy()
    lo, hi = d[f"{metric}_CI_Lower"].to_numpy(), d[f"{metric}_CI_Upper"].to_numpy()
    ax.hlines(y, lo, hi, color=SERIES[0], lw=LINE_W, alpha=0.5)
    ax.plot(obs, y, marker="o", **ring_marker_kwargs(SERIES[0]))
    ax.set_yticks(y, d["Model"], color=INK_2)
    best = len(d) - 1
    ax.annotate(f"{obs[best]:.3f}".replace(".", ","), (hi[best], best), xytext=(8, 0), textcoords="offset points", va="center", color=INK, fontsize=9)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.2f}".replace(".", ",")))
    ax.set_xlabel(f"{metric.replace('R2', 'R²')} nos motores do holdout 75/25 (IC 95%, bootstrap por motor)")
    ax.set_title("Modelos de RUL: intervalos marginais sobrepostos exigem teste pareado")
    fig.tight_layout()
    return save_figure(fig, path)

In [ ]:
%%writefile projeto/src/pdm_cmaps/reporting/plotly_hub.py
"""Interactive (Plotly) figures for the TCC, behind one clickable index page.

WHAT: builds every interactive figure from the CSVs already written by the pipeline and groups them in
    results/plotly/index.html        <- the only file you need to open
    results/plotly/tcc/*.html        main figures (one per planned TCC figure; see TCC_ESQUELETO.md)
    results/plotly/exploracao/*.html extra figures to dig into the data
    results/plotly/assets/plotly.min.js   shared offline copy of Plotly (works from file://, no internet)
WHY: the PNGs are fixed pictures; here you can hover for exact values, toggle detectors/models in the
legend, pick the subset / fault / engine in dropdowns, and export each figure as a PNG (camera icon in the
modebar, 3x scale) for the text.

Run (from the repo root):
    .venv/bin/python -m pdm_cmaps.reporting.plotly_hub --open        # build everything and open the index in the browser
    .venv/bin/python -m pdm_cmaps.reporting.plotly_hub --only t05 e01 # rebuild a few figures
A figure whose data is missing is skipped and listed in the index with the command that creates the data.

Every figure is a pure function  data (CSV) -> plotly Figure  or an "explorer" page (a generic
dropdown/checkbox time-series viewer driven by embedded JSON). Colors come from viz_style (categorical
slots blue/orange/aqua, status palette only for criticality), text in ink tokens.
"""

from __future__ import annotations

from pdm_cmaps.config import DEFAULT_RESULTS_DIR

import argparse
import datetime as dt
import html
import json
import re
import subprocess
import unicodedata
import webbrowser
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.offline as pyo
from plotly.subplots import make_subplots

from pdm_cmaps.reporting import viz_style

SUBSETS = ("FD001", "FD002", "FD003", "FD004")
SUBSET_COLORS = dict(zip(SUBSETS, [*viz_style.SERIES, viz_style.INK_2]))
FONT = "system-ui, -apple-system, 'Segoe UI', Roboto, sans-serif"
ADOPTED_H = 25  # healthy window adopted in the TCC (detection sweep 25/35/50)
PLOT_CONFIG = {"displaylogo": False, "responsive": True,
               "toImageButtonOptions": {"format": "png", "scale": 3, "filename": "figura_tcc"}}
VERIFIED_RESET = {("FD004", 179): 242}  # core reset found by rul_prediction_plots.scan_discontinuities (see its docstring)
DET_COLORS = {"T50 bruto (sem regime)": (viz_style.SERIES[0], "dash"), "T50 EWMA (com regime)": (viz_style.SERIES[0], "solid"),
              "MEWMA bruto (sem regime)": (viz_style.SERIES[1], "dash"), "MEWMA (13 resíduos, com regime)": (viz_style.SERIES[1], "solid")}
DET_COLORS["T50 EWMA (com regime, sem portão)"] = (viz_style.SERIES[2], "solid")
KINDS_PT = {"degrau": "degrau", "rampa": "rampa", "um_regime": "um regime", "variancia": "variância", "vies_sensor": "viés de sensor"}

TEMPLATE = go.layout.Template(layout=dict(
    font=dict(family=FONT, color=viz_style.INK, size=13), paper_bgcolor=viz_style.SURFACE, plot_bgcolor=viz_style.SURFACE,
    colorway=list(viz_style.SERIES) + [viz_style.INK_2, viz_style.MUTED],
    xaxis=dict(gridcolor=viz_style.GRID, linecolor=viz_style.AXIS, zeroline=False, ticks="", automargin=True),
    yaxis=dict(gridcolor=viz_style.GRID, linecolor=viz_style.AXIS, zeroline=False, ticks="", automargin=True),
    legend=dict(bgcolor="rgba(0,0,0,0)"), hoverlabel=dict(font=dict(family=FONT)), margin=dict(l=70, r=30, t=90, b=70)))


class MissingData(Exception):
    """Raised by a builder when an input table does not exist yet; the message says how to create it."""


@dataclass
class Ctx:
    results: Path
    out: Path
    repo: Path = Path(".")

    def csv(self, rel: str, hint: str) -> pd.DataFrame:
        path = self.results / rel
        if not path.exists():
            raise MissingData(f"falta {path}: {hint}")
        return pd.read_csv(path)

    def glob_csv(self, pattern: str, hint: str) -> pd.DataFrame:
        files = sorted(self.results.glob(pattern))
        if not files:
            raise MissingData(f"falta {self.results / pattern}: {hint}")
        return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


@dataclass
class Chart:
    id: str
    section: str  # "tcc" | "exploracao"
    title: str
    desc: str
    ref: str      # where it is used in the TCC
    builder: Callable[[Ctx], "go.Figure | str"]
    tags: list[str] = field(default_factory=list)

    @property
    def filename(self) -> str:
        ascii_title = unicodedata.normalize("NFKD", self.title).encode("ascii", "ignore").decode()
        return f"{self.id}_{re.sub(r'[^a-z0-9]+', '-', ascii_title.lower()).strip('-')[:48]}.html"


# ============================================================================
# page helpers
# ============================================================================

def _layout(fig: go.Figure, title: str, subtitle: str = "", height: int = 560) -> go.Figure:
    fig.update_layout(template=TEMPLATE, height=height, title=dict(text=f"<b>{title}</b>" + (f"<br><sup>{subtitle}</sup>" if subtitle else ""), x=0.01, xanchor="left", y=0.97, yanchor="top"))
    return fig


def _nav(title: str, back: str = "../index.html") -> str:
    return (f'<div style="font-family:{FONT};padding:12px 20px;border-bottom:1px solid {viz_style.GRID};background:{viz_style.SURFACE}">'
            f'<a href="{back}" style="color:{viz_style.SERIES[0]};text-decoration:none;font-weight:600">← Índice dos gráficos</a>'
            f'<span style="color:{viz_style.MUTED};margin-left:14px">{html.escape(title)}</span></div>')


def write_figure(fig: go.Figure, path: Path, title: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    page = fig.to_html(full_html=True, include_plotlyjs="../assets/plotly.min.js", config=PLOT_CONFIG)
    page = page.replace("<head>", f"<head><title>{html.escape(title)}</title>", 1)
    page = page.replace("<body>", f'<body style="margin:0;background:{viz_style.SURFACE}">{_nav(title)}', 1)
    path.write_text(page, encoding="utf-8")


def _target(ctx: "Ctx", cid: str) -> Path:
    chart = next(c for c in CHARTS if c.id == cid)
    return ctx.out / chart.section / chart.filename


def dropdown_figure(fig: go.Figure, options: list[tuple[str, list[tuple[go.BaseTraceType, int, int]]]], label: str = "", x: float = 0.0) -> go.Figure:
    """options = [(label, [(trace, row, col), ...])]; one option visible at a time (buttons toggle trace visibility)."""
    owner, k = [], 0
    for oi, (_, traces) in enumerate(options):
        for tr, row, col in traces:
            tr.visible = oi == 0
            if getattr(fig, "_grid_ref", None) is None:
                fig.add_trace(tr)
            else:
                fig.add_trace(tr, row=row, col=col)
            owner.append(oi)
            k += 1
    buttons = [dict(label=(f"{label}: {lab}" if label else lab), method="update", args=[{"visible": [o == oi for o in owner]}]) for oi, (lab, _) in enumerate(options)]
    has_grid = getattr(fig, "_grid_ref", None) is not None
    fig.update_layout(margin=dict(l=70, r=30, t=200, b=70),
                      updatemenus=[dict(buttons=buttons, direction="down", x=x, xanchor="left", y=1.0, yanchor="bottom", pad=dict(b=40 if has_grid else 12),
                                        showactive=True, bgcolor=viz_style.SURFACE, bordercolor=viz_style.AXIS, font=dict(size=13))])
    return fig


# ============================================================================
# generic explorer (dropdowns + checkboxes, embedded JSON)
# ============================================================================

EXPLORER_TEMPLATE = r"""<!doctype html><html lang="pt-br"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>__TITLE__</title><script src="../assets/plotly.min.js"></script>
<style>
:root{--surface:#fcfcfb;--ink:#0b0b0b;--ink2:#52514e;--muted:#898781;--grid:#e1e0d9;--axis:#c3c2b7;--blue:#2a78d6}
body{margin:0;background:var(--surface);color:var(--ink);font-family:system-ui,-apple-system,'Segoe UI',Roboto,sans-serif}
nav{padding:12px 20px;border-bottom:1px solid var(--grid)} nav a{color:var(--blue);text-decoration:none;font-weight:600}
header{padding:14px 20px 0} h1{margin:0 0 4px;font-size:20px} p.sub{margin:0;color:var(--ink2);font-size:14px;max-width:1100px}
#controls{display:flex;flex-wrap:wrap;gap:14px 22px;padding:14px 20px;align-items:flex-end}
label{display:flex;flex-direction:column;font-size:12px;color:var(--ink2);gap:4px}
select,button{font:inherit;font-size:14px;padding:6px 9px;border:1px solid var(--axis);background:var(--surface);color:var(--ink);border-radius:6px}
button{cursor:pointer} #models{display:flex;flex-wrap:wrap;gap:6px 14px;padding:0 20px 8px}
#models label{flex-direction:row;align-items:center;gap:6px;font-size:13px;color:var(--ink)} .sw{width:12px;height:3px;display:inline-block}
#plot{height:620px;padding:0 10px} #note{padding:4px 20px 24px;color:var(--ink2);font-size:13px;max-width:1100px}
</style></head><body>
<nav><a href="../index.html">← Índice dos gráficos</a></nav>
<header><h1>__TITLE__</h1><p class="sub">__SUBTITLE__</p></header>
<div id="controls">
 <label>__SETLABEL__<select id="set"></select></label>
 <label>__UNITLABEL__<select id="unit"></select></label>
 <label id="modeWrap">Modo<select id="mode"><option value="series">Série ao longo dos ciclos</option><option value="scatter">Dispersão predito × real (todos os motores)</option></select></label>
 <button id="all">marcar todos</button><button id="none">desmarcar</button><button id="reset">padrão</button>
</div>
<div id="models"></div><div id="plot"></div><div id="note">__NOTE__</div>
<script>
const DATA=__DATA__, CFG=__CONFIG__;
const PAL=["#2a78d6","#eb6834","#1baf7a","#7c3aed","#d03b3b","#b8860b","#0f766e","#be185d","#4b5563","#0891b2","#65a30d","#9333ea"];
const $=id=>document.getElementById(id);
const base=n=>n.split(" · ")[0], isCtrl=n=>n.endsWith(" · controle");
let checked=new Set(CFG.defaultOn);
const setSel=$("set"), unitSel=$("unit"), modeSel=$("mode");
if(!CFG.scatter) $("modeWrap").style.display="none";
Object.keys(DATA).forEach(k=>{const o=document.createElement("option");o.value=k;o.textContent=k;setSel.appendChild(o)});
function names(){return DATA[setSel.value].names}
function colorOf(n){const b=names().filter(x=>!isCtrl(x)||true).map(base);const u=[...new Set(b)];return PAL[u.indexOf(base(n))%PAL.length]}
function fillUnits(){const U=DATA[setSel.value].units;unitSel.innerHTML="";
 Object.keys(U).sort((a,b)=>a-b).forEach(u=>{const o=document.createElement("option");o.value=u;
  const mk=U[u].markers||[];o.textContent=(mk.some(m=>m.kind==="verified")?"★ ":mk.some(m=>m.kind==="candidate")?"◆ ":"")+CFG.unitLabel+" "+u;unitSel.appendChild(o)});
 if(CFG.defaultUnit&&DATA[setSel.value].units[CFG.defaultUnit]&&setSel.value===CFG.defaultSet)unitSel.value=CFG.defaultUnit}
function fillModels(){const box=$("models");box.innerHTML="";
 names().forEach(n=>{const l=document.createElement("label"),c=document.createElement("input");c.type="checkbox";c.checked=checked.has(n);
  c.onchange=()=>{c.checked?checked.add(n):checked.delete(n);draw()};
  const sw=document.createElement("span");sw.className="sw";sw.style.background=colorOf(n);if(isCtrl(n))sw.style.opacity=.45;
  l.appendChild(c);l.appendChild(sw);l.appendChild(document.createTextNode(n));box.appendChild(l)})}
function draw(){const S=DATA[setSel.value],U=S.units[unitSel.value];const mode=CFG.scatter?modeSel.value:"series";let tr=[],shapes=[],ann=[];
 if(mode==="series"){
  if(U.truth)tr.push({x:U.x,y:U.truth,name:CFG.truthName,mode:"lines",line:{color:"#0b0b0b",width:3},hovertemplate:"%{y:.1f}"});
  names().filter(n=>checked.has(n)).forEach(n=>tr.push({x:U.x,y:U.series[n],name:n,mode:"lines",line:{color:colorOf(n),width:2,dash:isCtrl(n)?"dot":"solid"},hovertemplate:"%{y:.1f}"}));
  (U.markers||[]).forEach(m=>{const col=(m.kind==="verified"||m.kind==="fault")?"#d03b3b":"#898781";
   shapes.push({type:"line",x0:m.x,x1:m.x,y0:0,y1:1,yref:"paper",line:{color:col,width:2,dash:(m.kind==="candidate"||m.kind==="eol")?"dot":"dash"}});
   ann.push({x:m.x,y:1,yref:"paper",text:m.label,showarrow:false,xanchor:"left",yanchor:"top",font:{size:11,color:col},bgcolor:"rgba(252,252,251,.85)"})});
  Plotly.react("plot",tr,{height:600,margin:{l:70,r:30,t:30,b:60},paper_bgcolor:"#fcfcfb",plot_bgcolor:"#fcfcfb",hovermode:"x unified",
   xaxis:{title:CFG.xlabel,gridcolor:"#e1e0d9",linecolor:"#c3c2b7"},yaxis:{title:CFG.ylabel,gridcolor:"#e1e0d9",linecolor:"#c3c2b7"},
   legend:{orientation:"h",y:-0.18},shapes:shapes,annotations:ann},CFG.plotConfig);
 } else {
  const all=Object.values(S.units),truth=[].concat(...all.map(u=>u.truth)),sel=U;let lo=0,hi=130;
  names().filter(n=>checked.has(n)).forEach(n=>{const y=[].concat(...all.map(u=>u.series[n]));
   tr.push({x:truth,y:y,name:n,type:"scattergl",mode:"markers",marker:{size:4,opacity:.25,color:colorOf(n)},hoverinfo:"skip"});
   tr.push({x:sel.truth,y:sel.series[n],name:n+" (motor "+unitSel.value+")",type:"scattergl",mode:"markers",marker:{size:7,color:colorOf(n),line:{color:"#0b0b0b",width:1}}})});
  tr.push({x:[lo,hi],y:[lo,hi],name:"predição perfeita",mode:"lines",line:{color:"#52514e",dash:"dash",width:1.5}});
  Plotly.react("plot",tr,{height:600,margin:{l:70,r:30,t:30,b:60},paper_bgcolor:"#fcfcfb",plot_bgcolor:"#fcfcfb",
   xaxis:{title:"RUL real (ciclos)",gridcolor:"#e1e0d9",linecolor:"#c3c2b7"},yaxis:{title:"RUL predito (ciclos)",gridcolor:"#e1e0d9",linecolor:"#c3c2b7"},legend:{orientation:"h",y:-0.18}},CFG.plotConfig)}}
setSel.onchange=()=>{fillUnits();fillModels();draw()};unitSel.onchange=draw;modeSel.onchange=draw;
$("all").onclick=()=>{names().forEach(n=>checked.add(n));fillModels();draw()};
$("none").onclick=()=>{checked.clear();fillModels();draw()};
$("reset").onclick=()=>{checked=new Set(CFG.defaultOn);fillModels();draw()};
if(CFG.defaultSet)setSel.value=CFG.defaultSet;fillUnits();fillModels();draw();
</script></body></html>"""


def _round(a, nd: int = 1) -> list:
    return [None if (v is None or (isinstance(v, float) and not np.isfinite(v))) else round(float(v), nd) for v in a]


def explorer_page(path: Path, title: str, subtitle: str, sets: dict, cfg: dict, set_label: str = "Subconjunto", note: str = "") -> None:
    """sets = {set: {"names": [...], "units": {unit: {"x": [...], "truth": [...]|None, "series": {name: [...]}, "markers": [...]}}}}"""
    base_cfg = {"truthName": "RUL real", "xlabel": "Ciclo", "ylabel": "RUL (ciclos)", "unitLabel": "Motor", "scatter": False,
                "defaultOn": [], "plotConfig": PLOT_CONFIG}
    page = (EXPLORER_TEMPLATE.replace("__TITLE__", html.escape(title)).replace("__SUBTITLE__", html.escape(subtitle))
            .replace("__SETLABEL__", set_label).replace("__UNITLABEL__", cfg.get("unitLabel", "Motor")).replace("__NOTE__", html.escape(note))
            .replace("__DATA__", json.dumps(sets, separators=(",", ":"))).replace("__CONFIG__", json.dumps({**base_cfg, **cfg}, separators=(",", ":"))))
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(page, encoding="utf-8")


# ============================================================================
# TCC figures
# ============================================================================

def t01_regime_variance(ctx: Ctx) -> go.Figure:
    d = ctx.csv("monitoring_repeated/tables/regime_variance_by_subset.csv", "python -m pdm_cmaps.monitoring.layer --regime-variance-only")
    fig = go.Figure(go.Bar(x=d["Subset"], y=d["Between_regime_pct_mean"], error_y=dict(type="data", array=d["Between_regime_pct_std"].fillna(0)),
                           marker_color=[SUBSET_COLORS[s] for s in d["Subset"]], text=[f"{v:.2f} %" for v in d["Between_regime_pct_mean"]], textposition="outside",
                           customdata=np.c_[d["Regimes"], d["OOS_R2_mean"]], hovertemplate="%{x}<br>variância entre regimes: %{y:.3f} %<br>regimes: %{customdata[0]}<br>R² fora da amostra: %{customdata[1]:.3f}<extra></extra>"))
    fig.update_yaxes(title="Variância de T50 explicada pelo regime (%)", range=[0, 112])
    return _layout(fig, "O regime de operação domina a variância do T50", "Média de 3 seeds nos motores de teste; FD001/FD003 têm 1 regime e servem de controle", 520)


def t02_benchmark(ctx: Ctx) -> go.Figure:
    d = ctx.csv("benchmark_repeated/benchmark_repeated_summary.csv", "python -m pdm_cmaps.experiments.benchmark_repeated")
    order = d.groupby("Model")["RMSE_mean"].mean().sort_values().index.tolist()
    fig = make_subplots(rows=1, cols=1)
    opts = []
    for proto, lab in (("official", "Protocolo oficial (último ciclo do test_FDxxx)"), ("holdout", "Holdout 75/25 por motor (todos os ciclos)")):
        tr = []
        for s in SUBSETS:
            x = d[(d["Protocol"] == proto) & (d["Subset"] == s)].set_index("Model").reindex(order)
            tr.append((go.Bar(x=order, y=x["RMSE_mean"], error_y=dict(type="data", array=x["RMSE_std"].fillna(0)), name=s, marker_color=SUBSET_COLORS[s],
                              hovertemplate="%{x}<br>RMSE %{y:.2f} ciclos<extra>" + s + "</extra>"), 1, 1))
        opts.append((lab, tr))
    dropdown_figure(fig, opts, "Protocolo")
    fig.update_layout(barmode="group", legend_title="Subconjunto")
    fig.update_yaxes(title="RMSE (ciclos), menor é melhor")
    fig.update_xaxes(tickangle=-35)
    return _layout(fig, "Modelos de RUL: RMSE por subconjunto", "Média ± desvio de 3 seeds. No protocolo oficial o desvio é só de inicialização (modelos determinísticos têm 0)", 620)


def t03_detection(ctx: Ctx) -> go.Figure:
    a = ctx.csv("monitoring_repeated/tables/monitoring_repeated_spc_summary.csv", "python -m pdm_cmaps.monitoring.layer")
    b = ctx.csv("monitoring_repeated/tables/monitoring_repeated_spc_mv_summary.csv", "python -m pdm_cmaps.monitoring.layer")
    metrics = [("detection_rate_pct", "Detecção (% dos motores)"), ("mean_lead_time_cycles", "Antecedência média (ciclos)"),
               ("mean_healthy_false_alarm_pct", "Falso alarme na janela saudável (%)"), ("premature_alarm_pct", "Motores com alarme prematuro (%)")]
    fig = make_subplots(rows=1, cols=4, subplot_titles=[m[1] for m in metrics], horizontal_spacing=0.07)
    for i, (col, _) in enumerate(metrics, start=1):
        for df, name, color in ((a, "T50 EWMA (1 resíduo)", viz_style.SERIES[0]), (b, "MEWMA (13 resíduos)", viz_style.SERIES[1])):
            fig.add_trace(go.Bar(x=df["Subset"], y=df[f"{col}_mean"], error_y=dict(type="data", array=df[f"{col}_std"].fillna(0)), name=name, marker_color=color,
                                 showlegend=i == 1, legendgroup=name, hovertemplate="%{x}: %{y:.1f}<extra>" + name + "</extra>"), row=1, col=i)
    fig.update_layout(barmode="group")
    return _layout(fig, "Detecção antecipada: T50 sozinho × MEWMA de 13 resíduos", "Sensor virtual calibrado na janela saudável (H = 50, pipeline principal); média ± desvio de 3 seeds", 480)


def _power_traces(d: pd.DataFrame, subset: str, kinds: list[str], metric: str, ysd: str | None, show_legend: bool):
    out = []
    for j, kind in enumerate(kinds, start=1):
        for det, (col, dash) in DET_COLORS.items():
            x = d[(d["Subset"] == subset) & (d["kind"] == kind) & (d["Detector"] == det)].sort_values("magnitude")
            if x.empty:
                continue
            out.append((go.Scatter(x=x["magnitude"], y=x[metric], error_y=dict(type="data", array=x[ysd].fillna(0), thickness=1) if ysd else None, mode="lines+markers", name=det,
                                   line=dict(color=col, dash=dash, width=2.5), marker=dict(size=7), legendgroup=det, showlegend=show_legend and j == 1,
                                   hovertemplate="magnitude %{x}<br>poder %{y:.0f} %<extra>" + det + "</extra>"), 1, j))
    return out


def t05_ablation(ctx: Ctx) -> go.Figure:
    d = ctx.csv("injecao_deteccao/tabelas/pareado_far_resumo.csv", "python -m pdm_cmaps.experiments.fault_detection_study")
    d = d[(d["H"] == ADOPTED_H) & (d["budget"] == 0.2) & (d["t0"] == ADOPTED_H + 10)]
    kinds = [k for k in KINDS_PT if k in set(d["kind"])]
    fig = make_subplots(rows=1, cols=len(kinds), subplot_titles=[KINDS_PT[k] for k in kinds], horizontal_spacing=0.05, shared_yaxes=True)
    opts = [(s, _power_traces(d, s, kinds, "power_pct_mean", "power_pct_std", True)) for s in SUBSETS if s in set(d["Subset"])]
    dropdown_figure(fig, opts, "Subconjunto")
    fig.update_yaxes(range=[-3, 103], title_text="Poder de detecção (%)", col=1)
    fig.update_xaxes(title_text="magnitude da falha", matches=None)
    return _layout(fig, "Ablação do desacoplamento de regime: poder de detecção a falso alarme igualado", "Tracejado = sem remover o regime, cheio = com regime. Falso alarme ≤ 0,2 por 100 ciclos, H = 25, média ± dp de 3 seeds. Clique na legenda para ocultar um detector", 520)


def _curve_options(ctx: Ctx, kind: str) -> list:
    raw = ctx.glob_csv(f"injecao_deteccao/runs/*_H{ADOPTED_H}_{kind}.csv", "python -m pdm_cmaps.experiments.fault_detection_study")
    return raw


def t06_operating(ctx: Ctx) -> go.Figure:
    c = _curve_options(ctx, "curvas")
    c = c[(c["H"] == ADOPTED_H) & (c["t0"] == ADOPTED_H + 10)]
    m = c.groupby(["Subset", "kind", "magnitude", "Detector", "s"])[["far_ctrl_per100", "power_pct"]].mean().reset_index()
    fig = go.Figure()
    opts = []
    for (s, kind, mag), g in m.groupby(["Subset", "kind", "magnitude"], sort=True):
        tr = []
        for det, (col, dash) in DET_COLORS.items():
            x = g[g["Detector"] == det].sort_values("s")
            tr.append((go.Scatter(x=x["far_ctrl_per100"].clip(lower=1e-3), y=x["power_pct"], mode="lines", name=det, line=dict(color=col, dash=dash, width=2.5),
                                  customdata=x["s"], hovertemplate="falso alarme %{x:.3f}/100 ciclos<br>poder %{y:.0f} %<br>limite s = %{customdata:.2f}<extra>" + det + "</extra>"), 1, 1))
        opts.append((f"{s} · {KINDS_PT[kind]} {mag:g}", tr))
    dropdown_figure(fig, opts, "Subconjunto · falha · magnitude")
    fig.add_vline(x=0.2, line=dict(color=viz_style.MUTED, dash="dot"), annotation_text="orçamento 0,2", annotation_position="top")
    fig.update_xaxes(type="log", title="Falsos alarmes por 100 ciclos nos controles (escala log; 0 aparece em 0,001)")
    fig.update_yaxes(title="Poder de detecção (%)", range=[-3, 103])
    return _layout(fig, "Curvas operacionais: poder × falso alarme (limite varrido)", "Cada ponto da curva é um limite de controle diferente; escolha a falha no menu", 560)


def t07_cumulative(ctx: Ctx) -> go.Figure:
    d = ctx.glob_csv(f"injecao_deteccao/runs/*_H{ADOPTED_H}_atrasos.csv", "python -m pdm_cmaps.experiments.fault_detection_study")
    d = d[(d["H"] == ADOPTED_H) & (d["t0"] == ADOPTED_H + 10) & (d["budget"] == 0.2)]
    grid = np.arange(0, 81)
    fig = go.Figure()
    opts = []
    for (s, kind, mag), g in d.groupby(["Subset", "kind", "magnitude"], sort=True):
        tr = []
        for det, (col, dash) in DET_COLORS.items():
            x = g.loc[g["Detector"] == det, "delay"].to_numpy(dtype=float)
            y = [100 * np.mean(x <= v) if len(x) else 0 for v in grid]
            tr.append((go.Scatter(x=grid, y=y, mode="lines", line=dict(color=col, dash=dash, width=2.5, shape="hv"), name=det,
                                  hovertemplate="até %{x} ciclos: %{y:.0f} % detectadas<extra>" + det + "</extra>"), 1, 1))
        opts.append((f"{s} · {KINDS_PT[kind]} {mag:g}", tr))
    dropdown_figure(fig, opts, "Subconjunto · falha · magnitude")
    fig.update_xaxes(title="Ciclos após o início da falha (t0)")
    fig.update_yaxes(title="Séries detectadas até o atraso (%)", range=[-3, 103])
    return _layout(fig, "Detecção acumulada por atraso", "Falso alarme igualado a 0,2 por 100 ciclos, H = 25; séries nunca detectadas ficam censuradas no fim", 540)


def t08_min_magnitude(ctx: Ctx) -> go.Figure:
    d = ctx.csv("injecao_deteccao/tabelas/magnitude_minima_detectavel.csv", "python -m pdm_cmaps.experiments.fault_detection_study")
    d = d[(d["H"] == ADOPTED_H) & (d["budget"] == 0.2) & (d["t0"] == ADOPTED_H + 10)]
    dets = list(DET_COLORS)
    rows = d.pivot_table(index=["Subset", "kind"], columns="Detector", values="mdm_mediana").reindex(columns=dets).reset_index()
    cells = [rows["Subset"], rows["kind"].map(KINDS_PT)] + [[("não atinge" if pd.isna(v) else f"{v:g}") for v in rows[c]] for c in dets]
    fig = go.Figure(go.Table(header=dict(values=["Subconjunto", "Falha", *dets], fill_color=viz_style.GRID, align="left", font=dict(color=viz_style.INK, size=13)),
                             cells=dict(values=cells, align="left", fill_color=viz_style.SURFACE, height=26)))
    return _layout(fig, "Magnitude mínima detectável (poder ≥ 80 %)", "Mediana entre seeds; magnitude em fração da assinatura de fim de vida (degrau, rampa, um regime) ou em σ (variância, viés). Falso alarme ≤ 0,2 por 100 ciclos", 190 + 27 * len(rows))


def t04_window(ctx: Ctx) -> go.Figure:
    d = ctx.csv("injecao_deteccao/tabelas/pareado_far_resumo.csv", "python -m pdm_cmaps.experiments.fault_detection_study --windows 25 35 50")
    d = d[(d["budget"] == 0.2) & (d["t0"] == d["H"] + 10)]
    if d["H"].nunique() < 2:
        raise MissingData("a varredura de H não foi rodada: python -m pdm_cmaps.experiments.fault_detection_study --windows 25 35 50")
    options = [(("degrau", 0.25), "degrau 0,25"), (("rampa", 0.25), "rampa 0,25"), (("degrau", 0.5), "degrau 0,5")]
    fig = make_subplots(rows=2, cols=4, subplot_titles=list(SUBSETS), shared_xaxes=True, vertical_spacing=0.1, horizontal_spacing=0.05)
    opts = []
    for (kind, mag), lab in options:
        tr = []
        for j, s in enumerate(SUBSETS, start=1):
            for r, (m, sd) in enumerate((("power_pct_mean", "power_pct_std"), ("delay_med_mean", "delay_med_std")), start=1):
                for det, (col, dash) in DET_COLORS.items():
                    x = d[(d["Subset"] == s) & (d["kind"] == kind) & np.isclose(d["magnitude"], mag) & (d["Detector"] == det)].sort_values("H")
                    tr.append((go.Scatter(x=x["H"], y=x[m], error_y=dict(type="data", array=x[sd].fillna(0), thickness=1), mode="lines+markers", name=det,
                                          line=dict(color=col, dash=dash, width=2.5), legendgroup=det, showlegend=r == 1 and j == 1,
                                          hovertemplate="H=%{x}: %{y:.1f}<extra>" + det + "</extra>"), r, j))
        opts.append((lab, tr))
    dropdown_figure(fig, opts, "Falha")
    fig.update_yaxes(title_text="Poder (%)", range=[-3, 103], row=1, col=1)
    fig.update_yaxes(title_text="Atraso mediano (ciclos)", row=2, col=1)
    fig.update_xaxes(title_text="Janela saudável H (ciclos)", tickvals=sorted(d["H"].unique()), row=2)
    return _layout(fig, "Aumentar a janela saudável reduz a detecção", "A falso alarme igualado (0,2 por 100 ciclos): o poder cai e o atraso sobe com H. Adotado: H = 25 (a diferença para 50 é pequena). Média ± dp de 3 seeds", 720)


def t09_conformal(ctx: Ctx) -> go.Figure:
    d = ctx.csv("monitoring_repeated/tables/monitoring_repeated_uncertainty_summary.csv", "python -m pdm_cmaps.monitoring.layer")
    fig = go.Figure()
    for _, r in d.iterrows():
        fig.add_trace(go.Scatter(x=[r["mean_width_mean"]], y=[r["coverage_mean"]], mode="markers+text", text=[r["Subset"]], textposition="top center", name=r["Subset"],
                                 marker=dict(size=14, color=SUBSET_COLORS[r["Subset"]]), error_x=dict(type="data", array=[r["mean_width_std"]]), error_y=dict(type="data", array=[r["coverage_std"]]),
                                 hovertemplate=f"{r['Subset']}<br>cobertura {r['coverage_mean']:.3f}<br>largura {r['mean_width_mean']:.1f} ciclos<br>"
                                               f"cobertura p10 por motor {r['Per_engine_coverage_p10_mean']:.2f}<extra></extra>", showlegend=False))
    fig.add_hline(y=0.9, line=dict(color=viz_style.STATUS[3], dash="dash"), annotation_text="nominal 90 %", annotation_position="bottom right")
    fig.update_xaxes(title="Largura média do intervalo (ciclos)")
    fig.update_yaxes(title="Cobertura empírica", range=[0.8, 0.98])
    return _layout(fig, "Incerteza da RUL: cobertura do intervalo conformal", "Média ± dp de 3 seeds; FD003 fica abaixo do nominal (reportado, não corrigido)", 520)


def t10_cost(ctx: Ctx) -> go.Figure:
    d = ctx.csv("monitoring_repeated/tables/monitoring_repeated_costs_summary.csv", "python -m pdm_cmaps.monitoring.layer")
    policies = sorted(d["Policy"].unique(), key=lambda p: int(p.split("<=")[1]))
    strategies = ["point", "interval", "interval+anomaly", "interval+anomaly_mv"]
    colors = dict(zip(strategies, [viz_style.INK_2, viz_style.SERIES[0], viz_style.SERIES[1], viz_style.SERIES[2]]))
    fig = make_subplots(rows=1, cols=len(policies), subplot_titles=policies, shared_yaxes=True, horizontal_spacing=0.04)
    opts = []
    for s in SUBSETS:
        for lead in sorted(d["Lead"].unique()):
            tr = []
            for j, p in enumerate(policies, start=1):
                for st in strategies:
                    x = d[(d["Subset"] == s) & (d["Lead"] == lead) & (d["Policy"] == p) & (d["Strategy"] == st)].sort_values("Cost_ratio")
                    tr.append((go.Scatter(x=x["Cost_ratio"], y=x["Relative_to_oracle_mean"], mode="lines+markers", name=st, line=dict(color=colors[st], width=2.5), legendgroup=st,
                                          showlegend=j == 1, hovertemplate="razão de custo %{x}:1<br>%{y:.2f}× o oráculo<extra>" + st + "</extra>"), 1, j))
            opts.append((f"{s} · antecedência logística {lead}", tr))
    dropdown_figure(fig, opts, "Subconjunto · prazo de logística")
    fig.update_xaxes(type="log", title_text="Razão custo falha : custo preventivo")
    fig.update_yaxes(type="log", title_text="Custo por ciclo relativo ao oráculo (1 = ideal)", col=1)
    return _layout(fig, "Custo da decisão por estratégia de criticidade", "Três políticas de limiar (crítico ≤ 10/15/20). A estratégia 'point' só recebe o limiar ≤ 20, o que confunde a comparação com a margem de segurança", 560)


def t11_tradeoff(ctx: Ctx) -> go.Figure:
    d = ctx.csv("monitoring_repeated/tables/monitoring_repeated_summary.csv", "python -m pdm_cmaps.monitoring.layer")
    sym = {"critical<=10": "circle", "critical<=15": "square", "critical<=20": "diamond"}
    strategies = ["point", "interval", "interval+anomaly", "interval+anomaly_mv"]
    colors = dict(zip(strategies, [viz_style.INK_2, viz_style.SERIES[0], viz_style.SERIES[1], viz_style.SERIES[2]]))
    fig = go.Figure()
    opts = []
    for s in (*SUBSETS,):
        tr = []
        for st in strategies:
            x = d[(d["Subset"] == s) & (d["Strategy"] == st)]
            tr.append((go.Scatter(x=100 * x["False_alarm_rate_normal_mean"], y=100 * x["Missed_critical_rate_mean"], mode="markers", name=st,
                                  marker=dict(size=13, color=colors[st], symbol=[sym.get(p, "circle") for p in x["Policy"]], line=dict(color=viz_style.SURFACE, width=2)),
                                  error_x=dict(type="data", array=100 * x["False_alarm_rate_normal_std"].fillna(0), thickness=1), error_y=dict(type="data", array=100 * x["Missed_critical_rate_std"].fillna(0), thickness=1),
                                  text=x["Policy"], hovertemplate="%{text}<br>falsos alertas %{x:.1f} %<br>críticos perdidos %{y:.1f} %<extra>" + st + "</extra>"), 1, 1))
        opts.append((s, tr))
    dropdown_figure(fig, opts, "Subconjunto")
    fig.update_xaxes(title="Falsos alertas em estado normal (%)")
    fig.update_yaxes(title="Críticos não alertados (%)", rangemode="tozero")
    return _layout(fig, "Criticidade: críticos perdidos × falsos alertas", "Formas = política de limiar (círculo ≤ 10, quadrado ≤ 15, losango ≤ 20). Mais perto da origem é melhor", 560)


def _holdout_sets(ctx: Ctx, with_markers: bool = True) -> dict:
    sets = {}
    for s in SUBSETS:
        p = ctx.results / f"maintenance_guide_{s.lower()}" / "tables" / "rul_holdout_predictions.csv"
        if not p.exists():
            continue
        df = pd.read_csv(p)
        models = [c for c in df.columns if c.startswith("RUL_predicted_")]
        names = [c.replace("RUL_predicted_", "") for c in models]
        cand = ctx.results / "rul_predicoes" / s / "candidatos_descontinuidade.csv"
        cand_df = pd.read_csv(cand) if (cand.exists() and with_markers) else pd.DataFrame()
        units = {}
        for u, g in df.sort_values(["unit_id", "cycle"]).groupby("unit_id"):
            mk = []
            if with_markers and (s, int(u)) in VERIFIED_RESET:
                mk.append({"x": VERIFIED_RESET[(s, int(u))], "label": "manutenção / reset do núcleo (ciclo %d)" % VERIFIED_RESET[(s, int(u))], "kind": "verified"})
            if len(cand_df):
                for r in cand_df[(cand_df["unit_id"] == u) & cand_df["candidate"] & ~cand_df["verified_maintenance"]].itertuples():
                    mk.append({"x": int(r.cycle), "label": "queda brusca de Nc (candidato, não verificado)", "kind": "candidate"})
            units[str(int(u))] = {"x": g["cycle"].astype(int).tolist(), "truth": _round(g["RUL_actual"]), "markers": mk,
                                  "series": {n: _round(g[c]) for n, c in zip(names, models)}}
        sets[s] = {"names": names, "units": units}
    if not sets:
        raise MissingData("falta results/maintenance_guide_*/tables/rul_holdout_predictions.csv: python -m pdm_cmaps --all")
    return sets


def t12_pred_vs_real(ctx: Ctx) -> str:
    sets = _holdout_sets(ctx)
    path = _target(ctx, "t12")
    explorer_page(path, "RUL predito × real, por modelo e por motor", "Holdout (seed 42, modelos treinados com H = 50). Escolha subconjunto, motor e modelos; ★ = manutenção verificada, ◆ = queda brusca candidata.", sets,
                  {"scatter": True, "defaultOn": ["Ridge", "XGBoost (+temporal)", "PyTorch RNN (LSTM)"], "defaultSet": "FD004", "defaultUnit": "179", "truthName": "RUL real (rótulo, teto 125)"},
                  note="O rótulo de RUL é linear por partes a partir da falha final (teto 125) e não reinicia na manutenção; após o reset do motor 179 (ciclo 242) os modelos saltam de volta, o rótulo não.")
    return str(path)


def t13_motor179(ctx: Ctx) -> go.Figure:
    from pdm_cmaps.data.loader import CMapssLoader
    from pdm_cmaps.detection.regime import RegimeDecoupler

    raw = ctx.repo / "data" / "CMaps"
    if not (raw / "train_FD004.txt").exists():
        raise MissingData(f"falta {raw}/train_FD004.txt (dados do C-MAPSS)")
    d = RegimeDecoupler(n_clusters=6, random_state=42).fit_transform(CMapssLoader().load("FD004"))
    m = d[d["unit_id"] == 179].sort_values("cycle").reset_index(drop=True)
    sensors = {"Nc": "Nc (rotação do núcleo)", "T30": "T30 (saída HPC)", "T50": "T50 (escape)", "P30": "P30 (saída HPC)", "phi": "phi (combustível/Ps30)",
               "Nf": "Nf (rotação do fan)", "T24": "T24 (saída LPC)", "BPR": "BPR (razão de bypass)"}
    core = ["Nc", "T30", "T50", "P30", "phi"]
    pre, post, healthy = (m["cycle"].between(210, 241)), (m["cycle"].between(242, 270)), (m["cycle"] <= 50)
    z, delta = {}, []
    for s in sensors:
        base = m.groupby("regime_id")[s].transform("mean")
        res = m[s] - base
        z[s] = (res / (res[healthy].std() or 1.0)).rolling(5, min_periods=1).mean()
        delta.append((sensors[s], s in core, float((100 * res / base)[post].mean() - (100 * res / base)[pre].mean()), float((res[post].mean() - res[pre].mean()) / (res[healthy].std() or 1.0))))
    fig = make_subplots(rows=1, cols=2, column_widths=[0.64, 0.36], horizontal_spacing=0.09,
                        subplot_titles=("Resíduo por regime, em desvios-padrão da janela saudável (média móvel de 5)", "Mudança do resíduo em 242 (média 242–270 menos 210–241), em % do valor"))
    pal = [viz_style.SERIES[0], viz_style.SERIES[1], viz_style.SERIES[2], viz_style.STATUS[3], viz_style.STATUS[1], "#7c3aed", viz_style.INK_2, "#0f766e"]
    for i, s in enumerate(sensors):
        fig.add_trace(go.Scatter(x=m["cycle"], y=z[s], name=sensors[s], mode="lines", line=dict(width=2.2, color=pal[i], dash="solid" if s in core else "dash"), legendgroup=s), row=1, col=1)
    fig.add_vrect(x0=1, x1=50, fillcolor=viz_style.STATUS[0], opacity=0.10, line_width=0, row=1, col=1)
    fig.add_vline(x=242, line=dict(color=viz_style.STATUS[3], dash="dash", width=2), row=1, col=1)
    fig.add_annotation(x=242, y=1, xref="x", yref="paper", text=" ciclo 242", showarrow=False, xanchor="left", yanchor="top", font=dict(color=viz_style.STATUS[3]))
    delta.sort(key=lambda t: abs(t[2]))
    fig.add_trace(go.Bar(y=[t[0] for t in delta], x=[t[2] for t in delta], orientation="h", marker_color=[viz_style.STATUS[3] if t[1] else viz_style.SERIES[0] for t in delta],
                         text=[f"{t[2]:+.2f} %" for t in delta], textposition="outside", customdata=[t[3] for t in delta], showlegend=False,
                         hovertemplate="%{y}<br>%{x:+.3f} % do valor<br>%{customdata:+.1f} σ saudável<extra></extra>"), row=1, col=2)
    fig.update_xaxes(title_text="Ciclo do motor 179", row=1, col=1)
    fig.update_xaxes(title_text="Mudança (% do valor do sensor)", range=[-3.4, 2.6], row=1, col=2)
    return _layout(fig, "Motor 179 (FD004): queda abrupta em quase todos os sensores no ciclo 242",
                   "Em σ da janela saudável todos mudam (Nf, com variância saudável minúscula, até mais que o núcleo); em % do valor o núcleo (vermelho) muda 0,6 a 2,4 %, Nf e T24 menos de 0,13 %, e o BPR 1,5 %", 600)


# ============================================================================
# exploration figures
# ============================================================================

def e01_control_chart(ctx: Ctx) -> str:
    sets = {}
    for s in SUBSETS:
        base = ctx.results / f"maintenance_guide_{s.lower()}" / "tables"
        if not (base / "virtual_sensor_residuals_t50.csv").exists():
            continue
        r = pd.read_csv(base / "virtual_sensor_residuals_t50.csv")
        al = pd.read_csv(base / "fleet_anomalies_t50.csv").set_index("unit_id") if (base / "fleet_anomalies_t50.csv").exists() else pd.DataFrame()
        units = {}
        for u, g in r.sort_values(["unit_id", "cycle"]).groupby("unit_id"):
            mk = []
            if len(al) and u in al.index and pd.notna(al.loc[u, "first_alarm_cycle"]):
                mk.append({"x": int(al.loc[u, "first_alarm_cycle"]), "label": f"1º alarme (ciclo {int(al.loc[u, 'first_alarm_cycle'])})", "kind": "fault"})
            mk.append({"x": int(g["cycle"].max()), "label": "fim de vida (EOL)", "kind": "eol"})
            units[str(int(u))] = {"x": g["cycle"].astype(int).tolist(), "truth": None, "markers": mk,
                                  "series": {"EWMA de |z| do resíduo de T50": _round(g["ewma_abs_z"], 3), "Limite de controle (UCL)": _round(g["ewma_ucl"], 3)}}
        sets[s] = {"names": ["EWMA de |z| do resíduo de T50", "Limite de controle (UCL)"], "units": units}
    if not sets:
        raise MissingData("falta results/maintenance_guide_*/tables/virtual_sensor_residuals_t50.csv: python -m pdm_cmaps --all")
    path = _target(ctx, "e01")
    explorer_page(path, "Carta de controle do sensor virtual (T50), por motor", "EWMA do resíduo padronizado contra o limite de controle, nos motores de teste (seed 42, H = 50).", sets,
                  {"defaultOn": ["EWMA de |z| do resíduo de T50", "Limite de controle (UCL)"], "ylabel": "EWMA de |z|", "defaultSet": "FD004"})
    return str(path)


def e02_benchmark_heatmap(ctx: Ctx) -> go.Figure:
    d = ctx.csv("benchmark_repeated/benchmark_repeated_summary.csv", "python -m pdm_cmaps.experiments.benchmark_repeated")
    order = d.groupby("Model")["RMSE_mean"].mean().sort_values().index.tolist()
    fig = go.Figure()
    opts = []
    for proto in ("official", "holdout"):
        for metric, lab, scale in (("RMSE_mean", "RMSE (ciclos)", "Blues"), ("R2_mean", "R²", "Blues_r")):
            p = d[d["Protocol"] == proto].pivot(index="Model", columns="Subset", values=metric).reindex(order)
            opts.append((f"{proto} · {lab}", [(go.Heatmap(z=p.values, x=list(p.columns), y=list(p.index), colorscale=scale, text=np.round(p.values, 2), texttemplate="%{text}",
                                                          hovertemplate="%{y}<br>%{x}: %{z:.3f}<extra></extra>", colorbar=dict(title=lab)), 1, 1)]))
    dropdown_figure(fig, opts, "Protocolo · métrica")
    fig.update_yaxes(autorange="reversed")
    return _layout(fig, "Todos os modelos × subconjuntos", "Média de 3 seeds; modelos ordenados pelo RMSE médio", 640)


def e03_error_by_engine(ctx: Ctx) -> go.Figure:
    d = ctx.glob_csv("rul_predicoes/*/metricas_por_motor.csv", "python -m pdm_cmaps.experiments.rul_prediction_plots")
    fig = go.Figure()
    order = d.groupby("Model")["RMSE"].median().sort_values().index.tolist()
    opts = []
    for s in SUBSETS:
        x = d[d["Subset"] == s]
        if x.empty:
            continue
        opts.append((s, [(go.Box(x=g["Model"], y=g["RMSE"], name=m, marker_color=viz_style.SERIES[0], boxpoints="outliers", customdata=g["unit_id"], showlegend=False,
                                 hovertemplate="motor %{customdata}<br>RMSE %{y:.1f}<extra>" + m + "</extra>"), 1, 1) for m, g in x.groupby("Model") if m in order]))
    dropdown_figure(fig, opts, "Subconjunto")
    fig.update_xaxes(categoryorder="array", categoryarray=order, tickangle=-35)
    fig.update_yaxes(title="RMSE por motor (ciclos)")
    return _layout(fig, "Erro de RUL por motor e por modelo (holdout)", "Cada caixa resume os motores do holdout; passe o mouse nos pontos para ver o número do motor", 620)


def e04_window_calibration(ctx: Ctx) -> go.Figure:
    d = ctx.csv("healthy_window_study/tables/healthy_window_summary.csv", "python -m pdm_cmaps.experiments.healthy_window_study")
    metrics = {"contamination_pct_mean": "Contaminação da calibração (% linhas com RUL < 125)", "calib_cv_pct_mean": "Ruído de calibração (CV bootstrap, %)",
               "far_common_pct_mean": "Falso alarme por ciclo (ciclos > 100 com RUL ≥ 125, %)", "early_alarm_pct_mean": "Motores com alarme precoce (%)",
               "mean_lead_time_detected_cycles_mean": "Antecedência média (ciclos)", "detection_rate_pct_mean": "Taxa de detecção (%)"}
    fig = go.Figure()
    opts = []
    for col, lab in metrics.items():
        tr = []
        for det in d["Detector"].unique():
            for s in SUBSETS:
                x = d[(d["Detector"] == det) & (d["Subset"] == s)].sort_values("H_mean")
                tr.append((go.Scatter(x=x["H_mean"], y=x[col], mode="lines+markers", name=f"{s} · {det}", line=dict(color=SUBSET_COLORS[s], dash="solid" if det.startswith("MEWMA") else "dash", width=2.2),
                                      text=x["Window_kind"], hovertemplate="%{text}: %{y:.2f}<extra>" + f"{s} · {det}" + "</extra>"), 1, 1))
        opts.append((lab, tr))
    dropdown_figure(fig, opts, "Métrica")
    fig.update_xaxes(title="Janela saudável H (ciclos; 25 % da vida ≈ 50 a 58)")
    return _layout(fig, "Estudo da janela saudável: calibração e métricas independentes de H", "Cheio = MEWMA, tracejado = T50 EWMA; média de 3 seeds", 620)


def e05_power_grid(ctx: Ctx) -> go.Figure:
    d = ctx.csv("injecao_deteccao/tabelas/pareado_far_resumo.csv", "python -m pdm_cmaps.experiments.fault_detection_study --windows 50 75 100 --late-t0 --budgets 0.2 1.0")
    fig = make_subplots(rows=1, cols=4, subplot_titles=list(SUBSETS), shared_yaxes=True, horizontal_spacing=0.04)
    opts = []
    for (H, t0, budget, kind), g in d.groupby(["H", "t0", "budget", "kind"], sort=True):
        tr = []
        for j, s in enumerate(SUBSETS, start=1):
            for det, (col, dash) in DET_COLORS.items():
                x = g[(g["Subset"] == s) & (g["Detector"] == det)].sort_values("magnitude")
                if x.empty:
                    continue
                tr.append((go.Scatter(x=x["magnitude"], y=x["power_pct_mean"], error_y=dict(type="data", array=x["power_pct_std"].fillna(0), thickness=1), mode="lines+markers", name=det,
                                      line=dict(color=col, dash=dash, width=2.5), legendgroup=det, showlegend=j == 1, hovertemplate="magnitude %{x}: %{y:.0f} %<extra>" + det + "</extra>"), 1, j))
        opts.append((f"H={H} · t0={t0} · FAR≤{budget:g} · {KINDS_PT[kind]}", tr))
    dropdown_figure(fig, opts, "H · t0 · falso alarme · falha")
    fig.update_yaxes(range=[-3, 103], title_text="Poder (%)", col=1)
    fig.update_xaxes(title_text="magnitude")
    return _layout(fig, "Grade completa de poder de detecção", "Todas as combinações de janela H, início t0, orçamento de falso alarme e tipo de falha que foram rodadas", 560)


def e06_default_rule(ctx: Ctx) -> go.Figure:
    d = ctx.csv("injecao_deteccao/tabelas/regra_padrao_s1.csv", "python -m pdm_cmaps.experiments.fault_detection_study")
    d = d[(d["H"] == ADOPTED_H)].groupby(["Subset", "Detector"])[["far_ctrl_per100", "arl0"]].mean().reset_index()
    fig = go.Figure()
    for det, (col, dash) in DET_COLORS.items():
        x = d[d["Detector"] == det]
        fig.add_trace(go.Bar(x=x["Subset"], y=x["far_ctrl_per100"], name=det, marker_color=col, marker_pattern_shape="/" if dash == "dash" else "",
                             customdata=x["arl0"], hovertemplate="%{x}<br>%{y:.3f} por 100 ciclos<br>ARL0 ≈ %{customdata:.0f} ciclos<extra>" + det + "</extra>"))
    fig.update_layout(barmode="group")
    fig.update_yaxes(title="Falsos alarmes por 100 ciclos (controles)")
    return _layout(fig, "Regra padrão (limite s = 1): falso alarme nos controles saudáveis", "ARL0 infinito (nenhum alarme observado) aparece como barra zero; é um limite inferior, não uma medida", 500)


def e07_transplant(ctx: Ctx) -> str:
    sets = {}
    for s in SUBSETS:
        base = ctx.results / "injecao_de_falhas" / f"{s}_seed42"
        if not (base / "saidas" / "predicoes_rul.csv").exists():
            continue
        pred, man = pd.read_csv(base / "saidas" / "predicoes_rul.csv"), pd.read_csv(base / "selado" / "manifesto_rul.csv")
        models = [c for c in pred.columns if c not in ("blind_id", "cycle")]
        names = [f"{m} · {k}" for m in models for k in ("falha", "controle")]
        units = {}
        for rec, g in man.groupby("recipient"):
            f, c = g[g["role"] == "transplante"].iloc[0], g[g["role"] == "controle"].iloc[0]
            pf, pc = (pred[pred["blind_id"] == int(i)].sort_values("cycle") for i in (f["blind_id"], c["blind_id"]))
            cyc = pf["cycle"].to_numpy()
            series = {}
            for m in models:
                series[f"{m} · falha"], series[f"{m} · controle"] = _round(pf[m]), _round(pc[m])
            units[str(int(rec))] = {"x": cyc.tolist(), "truth": _round(np.minimum(int(f["eol_cycle"]) - cyc, 125)), "series": series,
                                    "markers": [{"x": int(f["t0"]), "label": f"início da falha (t0 = {int(f['t0'])}), só nós sabíamos", "kind": "fault"}]}
        sets[s] = {"names": names, "units": units}
    if not sets:
        raise MissingData("falta results/injecao_de_falhas: python -m pdm_cmaps.experiments.fault_injection --subset FD004 --include-deep")
    path = _target(ctx, "e07")
    explorer_page(path, "Transplante de degradação: como cada modelo de RUL reage", "Motor receptor (saudável) com a degradação de outro motor a partir de t0. Cheio = com falha; pontilhado = o mesmo motor sem falha (controle).", sets,
                  {"defaultOn": ["Ridge · falha", "Ridge · controle", "XGBoost (+temporal) · falha", "PyTorch RNN (LSTM) · falha"], "truthName": "RUL verdadeira (por construção)", "unitLabel": "Receptor"},
                  note="Análise secundária: o foco do TCC é a detecção antecipada. A assinatura transplantada vem de outro motor, plausível mas não física.")
    return str(path)


def e08_candidates(ctx: Ctx) -> go.Figure:
    d = ctx.glob_csv("rul_predicoes/*/candidatos_descontinuidade.csv", "python -m pdm_cmaps.experiments.rul_prediction_plots")
    fig = go.Figure()
    opts = []
    for s in SUBSETS:
        x = d[d["Subset"] == s].sort_values("drop_sigma").head(25)
        if x.empty:
            continue
        colors = [viz_style.STATUS[3] if v else (viz_style.INK_2 if c else viz_style.MUTED) for v, c in zip(x["verified_maintenance"], x["candidate"])]
        opts.append((s, [(go.Bar(x=[f"motor {u}" for u in x["unit_id"]], y=x["drop_sigma"], marker_color=colors, customdata=np.c_[x["cycle"], x["eol_cycle"]],
                                 hovertemplate="%{x}<br>queda de %{y:.1f}σ no ciclo %{customdata[0]} (EOL %{customdata[1]})<extra></extra>"), 1, 1)]))
    dropdown_figure(fig, opts, "Subconjunto")
    fig.add_hline(y=-3, line=dict(color=viz_style.MUTED, dash="dot"), annotation_text="limiar de candidato (−3σ)")
    fig.update_yaxes(title="Maior queda sustentada de Nc (σ da janela saudável)")
    return _layout(fig, "Busca de manutenção/reset: maior queda de Nc por motor", "Vermelho = manutenção verificada (motor 179, ciclo 242); cinza escuro = candidato não verificado. Apenas o FD004 tem candidatos", 520)


def t14_ml_gain(ctx: Ctx) -> go.Figure:
    d = ctx.csv("falhas_dificeis/tabelas/diferenca_vs_media_regime.csv", "python -m pdm_cmaps.experiments.fault_hard_study")
    cases = [(c, f) for c in ("MEWMA", "sensor afetado") for f in ("processo", "sensor") if not d[(d["Chart"] == c) & (d["family"] == f)].empty]
    fig = make_subplots(rows=1, cols=len(cases), subplot_titles=[f"{c} · falhas de {f}" for c, f in cases], shared_xaxes=True, horizontal_spacing=0.18)
    colors = {"exogenous": viz_style.SERIES[0], "cross_sensor": viz_style.SERIES[1]}
    for j, (chart, fam) in enumerate(cases, start=1):
        x = d[(d["Chart"] == chart) & (d["family"] == fam)].sort_values(["Mode", "Estimator"])
        for mode in ("exogenous", "cross_sensor"):
            g = x[x["Mode"] == mode]
            fig.add_trace(go.Bar(y=g["Estimator"].str.replace(" (padrão)", "", regex=False), x=g["delta_pp"], orientation="h", name="exógeno (settings + ambiente)" if mode == "exogenous" else "cross-sensor (vê os outros sensores)",
                                 marker_color=colors[mode], legendgroup=mode, showlegend=j == 1,
                                 error_x=dict(type="data", symmetric=False, array=g["ci_hi"] - g["delta_pp"], arrayminus=g["delta_pp"] - g["ci_lo"]),
                                 customdata=np.c_[g["n_faults"], g["n_clusters"]], hovertemplate="%{y}<br>%{x:+.1f} pontos<br>%{customdata[0]} falhas, %{customdata[1]} motores<extra></extra>"), row=1, col=j)
        fig.add_vline(x=0, line=dict(color=viz_style.INK, width=1), row=1, col=j)
    fig.update_xaxes(title_text="poder − poder da média do regime (pontos)")
    fig.update_layout(barmode="group")
    return _layout(fig, "Quanto o modelo de ML ganha sobre a média do regime (sem ML)?", "Falhas sorteadas, falso alarme ≤ 0,2 por 100 ciclos, H = 25; barras de erro = IC 95 % por bootstrap sobre motores. Ganho só vale se o IC exclui 0", 520)


def e09_hard_power(ctx: Ctx) -> go.Figure:
    d = ctx.csv("falhas_dificeis/tabelas/poder_por_magnitude.csv", "python -m pdm_cmaps.experiments.fault_hard_study")
    fig = go.Figure()
    opts = []
    ests = list(dict.fromkeys(d["Estimator"]))
    pal = dict(zip(ests, [*viz_style.SERIES, viz_style.STATUS[3], viz_style.INK_2, viz_style.MUTED]))
    for chart in ("MEWMA", "sensor afetado"):
        for fam in ("processo", "sensor"):
            x = d[(d["Chart"] == chart) & (d["family"] == fam)]
            if x.empty:
                continue
            tr = []
            for (est, mode), g in x.groupby(["Estimator", "Mode"]):
                g = g.sort_values("mag_mid")
                tr.append((go.Scatter(x=g["mag_mid"], y=g["power_pct"], mode="lines+markers", name=f"{est} · {mode}", line=dict(color=pal[est], dash="solid" if mode == "exogenous" else "dash", width=2.5),
                                      customdata=g["n"], hovertemplate="magnitude %{x:.2f}: %{y:.0f} % (n=%{customdata})<extra>" + f"{est} · {mode}" + "</extra>"), 1, 1))
            opts.append((f"{chart} · falhas de {fam}", tr))
    dropdown_figure(fig, opts, "Detector · família")
    fig.update_xaxes(type="log", title="Magnitude (fração da assinatura para processo; σ para sensor)")
    fig.update_yaxes(title="Poder (%)", range=[-3, 103])
    return _layout(fig, "Falhas sorteadas: poder por magnitude, modelo e modo", "Cheio = exógeno, tracejado = cross-sensor; falso alarme ≤ 0,2 por 100 ciclos", 560)


def e10_envelope_cases(ctx: Ctx) -> str:
    from pdm_cmaps.experiments import envelope_ignored_cases as eic

    frames = []
    for sub in SUBSETS:
        try:
            frames.append(eic.fleet_envelope(ctx.results, sub)[0])
        except FileNotFoundError:
            continue
    if not frames:
        raise MissingData("falta results/maintenance_guide_*/tables/envelope_scored_trace_t50.csv: python -m pdm_cmaps --all")
    r = pd.concat(frames, ignore_index=True)
    runs = eic.breach_runs(r)
    sets = {}
    for sub, g in r.groupby("Subset"):
        units = {}
        for u, gu in g.groupby("unit_id"):
            mk = []
            for rr in runs[(runs["Subset"] == sub) & (runs["unit_id"] == u) & runs["monitored"]].itertuples():
                if rr.alert and not rr.first_alert:
                    continue  # only the first alert of an engine is drawn, to keep the plot readable
                mk.append({"x": int(rr.start), "kind": "fault" if rr.alert else "candidate",
                           "label": f"1º alerta (ciclo {int(rr.alert_cycle)})" if rr.alert else f"ignorado ({rr.max_excess_sigma:.1f}σ)"})
            units[str(int(u))] = {"x": gu["cycle"].astype(int).tolist(), "truth": None, "markers": mk,
                                  "series": {"resíduo de T50 (σ)": _round(gu["res_sigma"], 2), "limite superior (+3σ)": [3.0] * len(gu), "limite inferior (−3σ)": [-3.0] * len(gu)}}
        sets[sub] = {"names": ["resíduo de T50 (σ)", "limite superior (+3σ)", "limite inferior (−3σ)"], "units": units}
    path = _target(ctx, "e10")
    explorer_page(path, "Envelope do sensor virtual: desvios isolados ignorados e alertas", "Cada motor de teste (seed 42, H = 50). Linhas cinza pontilhadas = desvios de 1 ciclo ignorados pela regra de persistência; vermelha = primeiro alerta (2 ciclos seguidos fora do ±3σ).", sets,
                  {"defaultOn": ["resíduo de T50 (σ)", "limite superior (+3σ)", "limite inferior (−3σ)"], "ylabel": "resíduo de T50 (σ saudável)", "defaultSet": "FD004", "truthName": ""},
                  note="O envelope é previsão ± 3σ, com σ do resíduo saudável fora da amostra. Um ciclo fora do envelope não gera alerta; são necessários 2 ciclos seguidos (regra de persistência de Western Electric).")
    return str(path)


CHARTS = [
    Chart("t01", "tcc", "O regime domina a variância do T50", "Fração da variância explicada pelo regime, por subconjunto.", "5.1", t01_regime_variance, ["regime", "tese"]),
    Chart("t02", "tcc", "RMSE dos modelos de RUL", "Benchmark em 2 protocolos, 4 subconjuntos, 3 seeds.", "5.2", t02_benchmark, ["RUL", "modelos"]),
    Chart("t03", "tcc", "Detecção: T50 × MEWMA", "Detecção, antecedência, falso alarme e alarme prematuro.", "5.3", t03_detection, ["detecção"]),
    Chart("t04", "tcc", "Janela saudável: H maior reduz a detecção", "Poder e atraso × H (25, 35, 50), a falso alarme igualado. Adotado H = 25.", "5.4", t04_window, ["janela", "injeção"]),
    Chart("t05", "tcc", "Ablação do desacoplamento de regime", "Poder de detecção com e sem regime, 5 tipos de falha injetada.", "5.7", t05_ablation, ["injeção", "regime"]),
    Chart("t06", "tcc", "Curvas operacionais (poder × falso alarme)", "Limite varrido; escolha o subconjunto e a falha.", "5.7", t06_operating, ["injeção"]),
    Chart("t07", "tcc", "Detecção acumulada por atraso", "Fração de falhas detectadas até cada atraso.", "5.7", t07_cumulative, ["injeção"]),
    Chart("t08", "tcc", "Magnitude mínima detectável", "Tabela por subconjunto, falha e detector (poder ≥ 80 %).", "5.7", t08_min_magnitude, ["injeção"]),
    Chart("t09", "tcc", "Cobertura do intervalo conformal", "Cobertura empírica × largura do intervalo.", "5.5", t09_conformal, ["incerteza"]),
    Chart("t10", "tcc", "Custo da decisão por estratégia", "Custo relativo ao oráculo; subconjunto e prazo no menu.", "5.6", t10_cost, ["criticidade", "custo"]),
    Chart("t11", "tcc", "Críticos perdidos × falsos alertas", "Trade-off das estratégias e políticas de limiar.", "5.6", t11_tradeoff, ["criticidade"]),
    Chart("t12", "tcc", "RUL predito × real (explorador)", "Por motor e modelo, com a manutenção verificada do motor 179; modo dispersão.", "5.8", t12_pred_vs_real, ["RUL", "explorador"]),
    Chart("t13", "tcc", "Motor 179: queda abrupta no ciclo 242", "Todos os sensores em σ e a mudança em % do valor (núcleo × fan).", "5.1", t13_motor179, ["regime", "manutenção"]),
    Chart("t14", "tcc", "Ganho do ML sobre a média do regime", "Diferença pareada de poder com IC 95 %, exógeno × cross-sensor.", "5.8", t14_ml_gain, ["modelos", "injeção"]),
    Chart("e01", "exploracao", "Carta de controle por motor", "EWMA do resíduo de T50 e limite, com o 1º alarme.", "—", e01_control_chart, ["detecção", "explorador"]),
    Chart("e02", "exploracao", "Mapa de calor: modelos × subconjuntos", "RMSE e R² em ambos os protocolos.", "—", e02_benchmark_heatmap, ["RUL"]),
    Chart("e03", "exploracao", "Erro por motor e modelo", "Distribuição do RMSE por motor do holdout.", "—", e03_error_by_engine, ["RUL"]),
    Chart("e04", "exploracao", "Janela saudável: calibração", "Contaminação, ruído de calibração e métricas independentes de H.", "—", e04_window_calibration, ["janela"]),
    Chart("e05", "exploracao", "Grade completa de poder", "Todas as combinações de H, t0, falso alarme e falha.", "—", e05_power_grid, ["injeção"]),
    Chart("e06", "exploracao", "Regra padrão: falso alarme nos controles", "Falso alarme e ARL0 com o limite s = 1.", "—", e06_default_rule, ["injeção"]),
    Chart("e07", "exploracao", "Transplante de degradação (RUL)", "Reação dos modelos de RUL a uma falha com início conhecido (análise secundária).", "apêndice", e07_transplant, ["RUL", "injeção", "explorador"]),
    Chart("e09", "exploracao", "Falhas sorteadas: poder por magnitude", "Poder por modelo e modo ao longo das magnitudes sorteadas.", "—", e09_hard_power, ["injeção", "modelos"]),
    Chart("e10", "exploracao", "Envelope: desvios ignorados × alertas", "Motor a motor: quais desvios do envelope viram alerta e quais são ignorados.", "5.3", e10_envelope_cases, ["detecção", "envelope", "explorador"]),
    Chart("e08", "exploracao", "Busca de reset: quedas de Nc", "Candidatos a manutenção em todos os motores.", "—", e08_candidates, ["manutenção"]),
]


# ============================================================================
# index + build
# ============================================================================

INDEX_CSS = f"""
:root{{--surface:{viz_style.SURFACE};--ink:{viz_style.INK};--ink2:{viz_style.INK_2};--muted:{viz_style.MUTED};--grid:{viz_style.GRID};--axis:{viz_style.AXIS};--blue:{viz_style.SERIES[0]}}}
*{{box-sizing:border-box}} body{{margin:0;background:var(--surface);color:var(--ink);font-family:{FONT}}}
main{{max-width:1180px;margin:0 auto;padding:28px 20px 60px}} h1{{margin:0 0 6px;font-size:26px}} p.lead{{color:var(--ink2);margin:0 0 18px;max-width:820px;line-height:1.5}}
h2{{margin:34px 0 4px;font-size:19px}} h2 small{{font-weight:400;color:var(--muted);font-size:13px;margin-left:8px}}
input{{font:inherit;padding:9px 12px;border:1px solid var(--axis);border-radius:8px;width:100%;max-width:420px;background:var(--surface);color:var(--ink)}}
.grid{{display:grid;grid-template-columns:repeat(auto-fill,minmax(300px,1fr));gap:14px;margin-top:12px}}
a.card{{display:block;border:1px solid var(--grid);border-radius:10px;padding:14px 16px;text-decoration:none;color:inherit;background:var(--surface);transition:border-color .12s,transform .12s}}
a.card:hover{{border-color:var(--blue);transform:translateY(-1px)}} a.card b{{display:block;font-size:15px;margin-bottom:4px}} a.card span{{display:block;color:var(--ink2);font-size:13px;line-height:1.4}} a.card span.tag{{display:inline-block}}
.meta{{margin-top:8px;font-size:12px;color:var(--muted)}} .tag{{display:inline-block;margin:6px 6px 0 0;padding:1px 8px;border:1px solid var(--grid);border-radius:99px;font-size:11px;color:var(--ink2)}}
.miss{{border:1px dashed var(--axis);border-radius:10px;padding:12px 14px;color:var(--ink2);font-size:13px;margin-top:10px}} code{{background:var(--grid);padding:1px 6px;border-radius:4px}}
footer{{margin-top:40px;color:var(--muted);font-size:12px}}
"""


def build_index(out: Path, built: list[Chart], skipped: list[tuple[Chart, str]]) -> None:
    def cards(section: str) -> str:
        items = []
        for c in built:
            if c.section != section:
                continue
            tags = "".join(f'<span class="tag">{html.escape(t)}</span>' for t in c.tags)
            items.append(f'<a class="card" href="{section}/{c.filename}" data-q="{html.escape((c.title + " " + c.desc + " " + " ".join(c.tags)).lower())}"><b>{html.escape(c.title)}</b>'
                         f'<span>{html.escape(c.desc)}</span><div class="meta">Seção do TCC: {html.escape(c.ref)}</div>{tags}</a>')
        return f'<div class="grid">{"".join(items)}</div>'

    missing = "".join(f'<div class="miss"><b>{html.escape(c.title)}</b> indisponível. {html.escape(msg)}</div>' for c, msg in skipped)
    try:
        commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, timeout=5).stdout.strip()
    except Exception:
        commit = ""
    page = f"""<!doctype html><html lang="pt-br"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Gráficos interativos do TCC</title><style>{INDEX_CSS}</style></head>
<body><main><h1>Gráficos interativos do TCC</h1>
<p class="lead">Manutenção preditiva em turbofans (NASA C-MAPSS). Clique em um gráfico para abrir; passe o mouse para ver os valores, clique na legenda para ocultar séries e use o ícone da câmera para exportar a figura em PNG.</p>
<input id="q" placeholder="Filtrar: regime, injeção, RUL, custo, janela…" autofocus>
<h2>Figuras principais do TCC <small>pasta <code>tcc/</code>: uma por figura planejada</small></h2>{cards("tcc")}
<h2>Exploração <small>pasta <code>exploracao/</code>: o resto, para investigar os dados</small></h2>{cards("exploracao")}
{('<h2>Indisponíveis</h2>' + missing) if missing else ''}
<footer>Gerado em {dt.datetime.now():%Y-%m-%d %H:%M}{(' · commit ' + commit) if commit else ''} · <code>python -m pdm_cmaps.reporting.plotly_hub --open</code> reconstrói tudo a partir de <code>results/</code>.</footer></main>
<script>const q=document.getElementById('q');q.oninput=()=>{{const t=q.value.toLowerCase().trim();document.querySelectorAll('a.card').forEach(a=>a.style.display=!t||a.dataset.q.includes(t)?'':'none')}}</script></body></html>"""
    (out / "index.html").write_text(page, encoding="utf-8")


def build(ctx: Ctx, only: list[str] | None = None) -> tuple[list[Chart], list[tuple[Chart, str]]]:
    (ctx.out / "assets").mkdir(parents=True, exist_ok=True)
    (ctx.out / "assets" / "plotly.min.js").write_text(pyo.get_plotlyjs(), encoding="utf-8")
    built, skipped = [], []
    for c in CHARTS:
        if only and c.id not in only:
            if (ctx.out / c.section / c.filename).exists():
                built.append(c)
            continue
        for old in (ctx.out / c.section).glob(f"{c.id}_*.html"):  # drop stale files from a previous title
            if old.name != c.filename:
                old.unlink()
        try:
            res = c.builder(ctx)
            if not isinstance(res, str):  # explorer pages write their own file
                write_figure(res, ctx.out / c.section / c.filename, c.title)
            built.append(c)
            print(f"[ok]   {c.id} {c.title}", flush=True)
        except MissingData as e:
            skipped.append((c, str(e)))
            print(f"[skip] {c.id} {c.title}: {e}", flush=True)
    build_index(ctx.out, built, skipped)
    return built, skipped


def main(argv: list[str] | None = None) -> int:
    p = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    p.add_argument("--results", type=Path, default=DEFAULT_RESULTS_DIR)
    p.add_argument("--output-dir", type=Path, default=None, help="default: <results>/plotly")
    p.add_argument("--repo", type=Path, default=Path("."), help="folder with data/CMaps (for the motor 179 figure)")
    p.add_argument("--only", nargs="+", help="figure ids to rebuild (t01 … e09)")
    p.add_argument("--open", action="store_true", help="open the index in the default browser")
    a = p.parse_args(argv)
    out = a.output_dir or a.results / "plotly"
    ctx = Ctx(results=a.results, out=out, repo=a.repo)
    built, skipped = build(ctx, a.only)
    index = (out / "index.html").resolve()
    print(f"\n{len(built)} figuras, {len(skipped)} indisponíveis.\nAbra: {index.as_uri()}")
    if a.open:
        webbrowser.open(index.as_uri())
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/reporting/plots.py
"""All figures: pipeline plots and thesis diagnostic figures.

Plots never compute results; they only draw tables/arrays that the
other modules produced, so you can ignore this file until you want to change a figure.
Useful reading guide for the figures:
  parity plot       predicted vs actual; points on the diagonal = perfect; the +-3 sigma band
                    shows the calibrated tolerance.
  envelope plots    healthy-calibrated corridor; a sustained exit from it is the alarm.
  figure_1..4       FD004 thesis figures (noise vs damage, physical cascade, autocorrelation,
                    fleet persistence trade-off); they read tables written earlier in the run.
"""

from __future__ import annotations

from pdm_cmaps.config import DEFAULT_PATHS

from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from sklearn.metrics import r2_score
from pdm_cmaps.domain.schema import SENSOR_UNITS


# ============================================================================
# plotting
# ============================================================================

matplotlib.use("Agg")

MIN_SENSOR_R2_FOR_PLOT = 0.75


def save_figure(figure, path: Path) -> Path:
    """Write a figure after ensuring its destination directory exists."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        figure.savefig(path, dpi=180, bbox_inches="tight")
    except FileNotFoundError:
        path.parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(figure)
    return path


def plot_baseline(summary: pd.DataFrame, path: Path) -> Path:
    """Plot pooled grouped-CV R² with fold standard deviation bars."""
    ordered = summary.sort_values("R2_pooled_OOF", ascending=False)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar(ordered["Subpart"], ordered["R2_pooled_OOF"], yerr=ordered["R2_fold_std"], capsize=4, color="#2a7a8c")
    for index, value in enumerate(ordered["R2_pooled_OOF"]):
        ax.text(index, value + 0.012, f"{value:.3f}", ha="center")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Pooled out-of-fold R² (error bars: fold SD)")
    ax.set_title("C-MAPSS capped-RUL grouped-CV baseline")
    return save_figure(fig, path)


def plot_regime_scores(scores: pd.DataFrame, path: Path) -> Path:
    """Plot healthy-row virtual-sensor R² for each learned regime."""
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar(scores["Regime_ID"].astype(str), scores["R2"], color="#c9742a")
    ax.set_xlabel("Learned flight regime")
    ax.set_ylabel("Held-out healthy-row R²")
    ax.set_title("FD004 healthy T50 virtual sensor by regime")
    return save_figure(fig, path)


def plot_learning_curves(summary: pd.DataFrame, path: Path) -> Path:
    """Plot repeated grouped-holdout R² as training-engine count grows."""
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for subset, group in summary.groupby("Subpart"):
        ax.errorbar(group["Train_engine_fraction"], group["Mean_R2"], yerr=group["Std_R2"], marker="o", capsize=3, label=subset)
    ax.set_xlabel("Fraction of training engines")
    ax.set_ylabel("Held-out R²")
    ax.set_title("Learning curves by subset")
    ax.legend()
    return save_figure(fig, path)


def plot_sensor_variance(results: pd.DataFrame, path: Path) -> Path:
    """Plot between- and within-regime sensor variance shares."""
    fig, ax = plt.subplots(figsize=(8, 4.5))
    positions = np.arange(len(results))
    ax.bar(positions, results["Between_regime_pct"], label="Between regime", color="#2a7a8c")
    ax.bar(positions, results["Within_regime_pct"], bottom=results["Between_regime_pct"], label="Within regime", color="#c9742a")
    ax.set_xticks(positions, results["Sensor"])
    ax.set_ylim(0, 100)
    ax.set_ylabel("Share of total variance (%)")
    ax.set_title("FD004 sensor variance by operating regime")
    ax.legend(frameon=False)
    return save_figure(fig, path)


def plot_failure_detection(
    scored: pd.DataFrame,
    engines: pd.DataFrame,
    path: Path,
    healthy_cycle_limit: int = 50,
    sensor_name: str = "T50",
) -> Path:
    """Plot a representative engine's sensor residual alarm and first threshold crossing with calibrated baseline."""
    detected = engines.loc[engines["detected"]]
    if detected.empty:
        engine_id = int(engines.iloc[0]["unit_id"])
    else:
        engine_id = int(detected.sort_values("lead_time_cycles", ascending=False).iloc[0]["unit_id"])
    trace = scored.loc[scored["unit_id"] == engine_id].sort_values("cycle")
    summary = engines.loc[engines["unit_id"] == engine_id].iloc[0]
    figure, (sensor_ax, alarm_ax) = plt.subplots(2, 1, figsize=(11, 7.2), sharex=True)

    unit_label = SENSOR_UNITS.get(sensor_name, "")
    unit_str = f" ({unit_label})" if unit_label else ""

    max_cycle = float(trace["cycle"].max())
    # Calibrated shading covering the full interval from 0 to healthy_cycle_limit and post-calibration
    for axis in (sensor_ax, alarm_ax):
        axis.axvspan(0, healthy_cycle_limit, color="#55b982", alpha=0.15, label=f"Healthy calibration (cycles 0–{healthy_cycle_limit})")
        axis.axvspan(healthy_cycle_limit, max_cycle + 5, color="#e46f70", alpha=0.08, label="Online monitoring (post-cal)")
        axis.axvline(healthy_cycle_limit, color="#2e7d32", linestyle="--", linewidth=1.2, alpha=0.8, label=f"Calibration limit (cycle {healthy_cycle_limit})")

    # If multi-regime, plot points by regime to avoid false oscillatory continuous lines
    has_regimes = "regime_id" in trace.columns and trace["regime_id"].nunique() > 1
    if has_regimes:
        cmap = plt.get_cmap("tab10")
        for r_id, r_rows in trace.groupby("regime_id"):
            sensor_ax.scatter(
                r_rows["cycle"], r_rows[sensor_name],
                color=cmap(int(r_id) % 10), s=18, alpha=0.75, label=f"Regime {r_id}" if int(r_id) < 6 else "",
                zorder=3
            )
        sensor_ax.plot(trace["cycle"], trace["prediction"], color="#326bff", linewidth=1.5, linestyle="--", label="Virtual sensor (Exogenous)")
    else:
        sensor_ax.plot(trace["cycle"], trace[sensor_name], color="#26364a", linewidth=1.6, label=f"Measured {sensor_name}")
        sensor_ax.plot(trace["cycle"], trace["prediction"], color="#326bff", linewidth=1.5, linestyle="--", label="Virtual sensor")

    sensor_ax.set_ylabel(f"{sensor_name}{unit_str}")
    sensor_ax.set_title(f"Held-out engine {engine_id}: measured and virtual {sensor_name} with calibrated baseline")
    sensor_ax.legend(loc="upper left", ncol=2, frameon=True, fontsize=9)
    sensor_ax.set_xlim(0, max_cycle + 5)
    sensor_ax.grid(True, linestyle=":", alpha=0.5)

    alarm_ax.plot(trace["cycle"], trace["ewma_abs_z"], color="#7444ed", linewidth=1.6, label="EWMA absolute Z-score (α = 0.10)")
    if "ewma_ucl" in trace.columns:
        alarm_ax.plot(trace["cycle"], trace["ewma_ucl"], color="#d62728", linestyle=":", linewidth=1.8, label="Dynamic finite-horizon UCL (3σ)")
    else:
        alarm_ax.axhline(3.0, color="#d62728", linestyle=":", linewidth=1.8, label="Alarm threshold (3σ)")
    alarm_cycle = summary["first_alarm_cycle"]
    if pd.notna(alarm_cycle):
        alarm_row = trace.loc[trace["cycle"] == alarm_cycle]
        if not alarm_row.empty:
            alarm_val = float(alarm_row["ewma_abs_z"].iloc[0])
            alarm_ax.scatter([alarm_cycle], [alarm_val], marker="*", s=180, color="#bd1e1e", zorder=5, label=f"First alarm (cycle {int(alarm_cycle)})")
            alarm_ax.annotate(
                f"First alarm: cycle {int(alarm_cycle)}\nLead time: {int(summary['lead_time_cycles'])} cycles",
                xy=(alarm_cycle, alarm_val),
                xytext=(max(10, alarm_cycle - 50), min(alarm_val + 0.4, 4.3)),
                arrowprops=dict(arrowstyle="->", color="#bd1e1e", lw=1.3),
                fontsize=9, fontweight="bold", color="#8b0000",
                bbox=dict(boxstyle="round,pad=0.3", fc="#fff0f0", ec="#bd1e1e", alpha=0.9),
            )
    alarm_ax.set_xlabel("Operational cycle")
    alarm_ax.set_ylabel("EWMA |Z|")
    alarm_ax.set_title(
        f"First persistent alarm: cycle {int(alarm_cycle)}; lead time {int(summary['lead_time_cycles'])} cycles"
        if pd.notna(alarm_cycle) else "No alarm crossed the threshold"
    )
    alarm_ax.legend(loc="upper left", ncol=2, frameon=True, fontsize=9)
    alarm_ax.set_xlim(0, max_cycle + 5)
    alarm_ax.set_ylim(bottom=0.0)
    alarm_ax.grid(True, linestyle=":", alpha=0.5)
    figure.tight_layout()
    return save_figure(figure, path)


def plot_actual_vs_predicted(y_true, y_pred, path: Path, title: str, value_label: str) -> Path:
    """Plot held-out actual-versus-predicted values with identity line and R²."""
    truth = np.asarray(y_true, dtype=float)
    prediction = np.asarray(y_pred, dtype=float)
    score = float(r2_score(truth, prediction))
    lower = float(min(truth.min(), prediction.min()))
    upper = float(max(truth.max(), prediction.max()))
    padding = (upper - lower) * 0.04 or 1.0
    figure, axis = plt.subplots(figsize=(6.5, 6.0))
    axis.scatter(truth, prediction, s=10, alpha=0.25, color="#326bff", edgecolors="none")
    axis.plot([lower - padding, upper + padding], [lower - padding, upper + padding], color="#26364a", linestyle="--", linewidth=1.5, label="Perfect prediction")
    axis.set_xlim(lower - padding, upper + padding)
    axis.set_ylim(lower - padding, upper + padding)
    axis.set_xlabel(f"Actual {value_label}")
    axis.set_ylabel(f"Predicted {value_label}")
    axis.set_title(f"{title}\nHeld-out R² = {score:.3f}")
    axis.legend(frameon=False)
    return save_figure(figure, path)


def plot_model_r2(summary: pd.DataFrame, path: Path, score_column: str = "R2", min_r2_cutoff: float = 0.75) -> Path:
    """Rank model holdout R² values, filtering for models meeting the high-fidelity cutoff (R² >= 0.75)."""
    eligible = summary.loc[summary[score_column] >= min_r2_cutoff]
    if eligible.empty:
        eligible = summary
    ordered = eligible.sort_values(score_column, ascending=True)
    best_model = ordered.iloc[-1]["Model"]
    colors = ["#16a34a" if name == best_model else "#3b82f6" for name in ordered["Model"]]
    figure, axis = plt.subplots(figsize=(8.5, max(4.0, 0.52 * len(ordered))))
    positions = np.arange(len(ordered))
    axis.barh(positions, ordered[score_column], color=colors, height=0.65)
    axis.axvline(min_r2_cutoff, color="#dc2626", linestyle="--", linewidth=1.4, label=f"Limiar de Qualidade (R² ≥ {min_r2_cutoff:.2f})")
    axis.set_yticks(positions, ordered["Model"], fontsize=10, fontweight="bold")
    axis.set_xlabel("Held-out R²", fontsize=10, fontweight="bold")
    axis.set_xlim(left=min(0.70, float(ordered[score_column].min()) - 0.03), right=max(1.0, float(ordered[score_column].max()) + 0.05))
    axis.set_title(f"Modelos de RUL com Alta Fidelidade (R² ≥ {min_r2_cutoff:.2f}) | Melhor: {best_model}", fontsize=11, fontweight="bold")
    axis.legend(loc="lower right", frameon=True, fontsize=9)
    axis.grid(True, linestyle=":", alpha=0.5, axis="x")
    for position, value in zip(positions, ordered[score_column]):
        axis.text(value + 0.005, position, f"{value:.4f}", va="center", ha="left", fontsize=9.5, fontweight="bold")
    figure.tight_layout()
    return save_figure(figure, path)


def plot_sensor_r2(results: pd.DataFrame, path: Path) -> Path | None:
    """Rank held-out virtual-sensor R² scores across physical channels."""
    eligible = results.loc[results["R2"] >= MIN_SENSOR_R2_FOR_PLOT]
    if eligible.empty:
        return None
    ordered = eligible.sort_values("R2", ascending=True)
    best_sensor = ordered.iloc[-1]["Sensor"]
    colors = ["#c9742a" if sensor == best_sensor else "#ddb18a" for sensor in ordered["Sensor"]]
    figure, axis = plt.subplots(figsize=(8, max(6.0, 0.36 * len(ordered))))
    positions = np.arange(len(ordered))
    axis.barh(positions, ordered["R2"], color=colors)
    axis.axvline(0.0, color="#26364a", linewidth=0.8)
    axis.axvline(MIN_SENSOR_R2_FOR_PLOT, color="#bd1e1e", linestyle=":", linewidth=1.2, label=f"Display cutoff ({MIN_SENSOR_R2_FOR_PLOT:.2f})")
    axis.set_yticks(positions, ordered["Sensor"])
    axis.set_xlabel("Held-out R²")
    axis.set_title("Pooled full-lifecycle R² by sensor (all regimes)")
    axis.legend(frameon=False)
    return save_figure(figure, path)


def plot_sensor_parity_grid(predictions: pd.DataFrame, path: Path) -> Path | None:
    """Plot actual-vs-predicted held-out values, colored by operating regime."""
    sensor_scores = {
        sensor: float(r2_score(rows["Actual"], rows["Predicted"]))
        for sensor, rows in predictions.groupby("Sensor", sort=False)
        if rows["Actual"].var() > 1e-12
    }
    sensors = [sensor for sensor in dict.fromkeys(predictions["Sensor"]) if sensor_scores.get(sensor, -np.inf) >= MIN_SENSOR_R2_FOR_PLOT]
    if not sensors:
        return None
    columns = 4
    rows = int(np.ceil(len(sensors) / columns))
    figure, axes = plt.subplots(rows, columns, figsize=(15, 3.2 * rows), squeeze=False)
    best_sensor = max(sensor_scores, key=sensor_scores.get) if sensor_scores else None
    for axis, sensor in zip(axes.flat, sensors):
        rows_for_sensor = predictions.loc[predictions["Sensor"] == sensor]
        truth = rows_for_sensor["Actual"].to_numpy(dtype=float)
        estimate = rows_for_sensor["Predicted"].to_numpy(dtype=float)
        score = float(r2_score(truth, estimate)) if np.var(truth) > 1e-12 else np.nan
        lower = float(min(truth.min(), estimate.min()))
        upper = float(max(truth.max(), estimate.max()))
        padding = (upper - lower) * 0.04 or 1.0
        if "regime_id" in rows_for_sensor:
            for regime_id, regime_rows in rows_for_sensor.groupby("regime_id", sort=True):
                axis.scatter(
                    regime_rows["Actual"], regime_rows["Predicted"], s=5, alpha=0.35,
                    color=plt.get_cmap("tab10")(int(regime_id) % 10), edgecolors="none",
                    rasterized=True,
                )
        else:
            axis.scatter(truth, estimate, s=5, alpha=0.2, color="#326bff", edgecolors="none", rasterized=True)
        axis.plot([lower - padding, upper + padding], [lower - padding, upper + padding], color="#26364a", linestyle="--", linewidth=1)
        axis.set_xlim(lower - padding, upper + padding)
        axis.set_ylim(lower - padding, upper + padding)
        score_label = f"{score:.3f}" if np.isfinite(score) else "n/a"
        axis.set_title(f"{sensor}{' (best)' if sensor == best_sensor else ''}: R² {score_label}")
        axis.set_xlabel("Actual")
        axis.set_ylabel("Predicted")
    for axis in axes.flat[len(sensors):]:
        axis.set_visible(False)
    regime_ids = sorted(predictions["regime_id"].dropna().unique()) if "regime_id" in predictions else []
    if regime_ids:
        handles = [Patch(facecolor=plt.get_cmap("tab10")(int(regime_id) % 10), label=f"Regime {regime_id}") for regime_id in regime_ids]
        figure.legend(handles=handles, loc="lower center", ncol=min(len(handles), 6), frameon=False)
    figure.suptitle("Held-out actual vs predicted sensor values by operating regime", y=1.0)
    figure.tight_layout(rect=(0, 0.04 if regime_ids else 0, 1, 0.98))
    return save_figure(figure, path)


def plot_sensor_parity_by_channel(
    predictions: pd.DataFrame,
    directory: Path,
    scope_label: str = "held-out",
) -> list[Path]:
    """Save a separate actual-vs-predicted panel for each regime and sensor."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    paths = []
    for sensor, rows in predictions.groupby("Sensor", sort=False):
        regime_rows = [
            (regime_id, rows.loc[rows["regime_id"] == regime_id])
            for regime_id in sorted(rows["regime_id"].dropna().unique())
        ] if "regime_id" in rows else [(None, rows)]
        regimes = []
        for regime_id, regime_frame in regime_rows:
            if len(regime_frame) < 2 or np.var(regime_frame["Actual"].to_numpy(dtype=float)) <= 1e-12:
                continue
            if r2_score(regime_frame["Actual"], regime_frame["Predicted"]) >= MIN_SENSOR_R2_FOR_PLOT:
                regimes.append(regime_id)
        if not regimes:
            continue
        columns = min(3, len(regimes))
        figure, axes = plt.subplots(
            int(np.ceil(len(regimes) / columns)), columns,
            figsize=(5.5 * columns, 4.8 * int(np.ceil(len(regimes) / columns))),
            squeeze=False,
        )
        for axis, regime_id in zip(axes.flat, regimes):
            regime_rows = rows if regime_id is None else rows.loc[rows["regime_id"] == regime_id]
            regime_truth = regime_rows["Actual"].to_numpy(dtype=float)
            regime_prediction = regime_rows["Predicted"].to_numpy(dtype=float)
            score = float(r2_score(regime_truth, regime_prediction)) if len(regime_truth) > 1 and np.var(regime_truth) > 1e-12 else np.nan
            if not np.isfinite(score) or score < MIN_SENSOR_R2_FOR_PLOT:
                axis.set_visible(False)
                continue
            lower = float(min(regime_truth.min(), regime_prediction.min()))
            upper = float(max(regime_truth.max(), regime_prediction.max()))
            padding = (upper - lower) * 0.06 or 1.0
            axis.scatter(regime_truth, regime_prediction, s=8, alpha=0.25, color="#326bff", edgecolors="none", rasterized=True)
            axis.plot([lower - padding, upper + padding], [lower - padding, upper + padding], color="#26364a", linestyle="--", linewidth=1.2)
            axis.set_xlim(lower - padding, upper + padding)
            axis.set_ylim(lower - padding, upper + padding)
            axis.set_xlabel(f"Actual {sensor}")
            axis.set_ylabel(f"Predicted {sensor}")
            score_label = f"R² {score:.3f}" if np.isfinite(score) else "R² n/a"
            axis.set_title(f"Regime {regime_id}: {score_label}, n={len(regime_rows)}")
        for axis in axes.flat[len(regimes):]:
            axis.set_visible(False)
        figure.suptitle(f"{sensor}: {scope_label} actual vs predicted, split by regime")
        figure.tight_layout()
        paths.append(save_figure(figure, Path(directory) / f"{sensor.lower()}_actual_vs_predicted.png"))
    return paths


def plot_sensor_cycle_windows_by_regime(
    predictions: pd.DataFrame,
    directory: Path,
    window_size: int = 50,
    ) -> tuple[list[Path], pd.DataFrame]:
    """Plot per-regime sensor timelines in short cycle windows with engine spread bands."""
    if window_size < 1:
        raise ValueError("window_size must be positive")
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    sensors = list(dict.fromkeys(predictions["Sensor"]))
    windowed = predictions.copy()
    windowed["Cycle_start"] = ((windowed["cycle"] - 1) // window_size) * window_size + 1
    windowed["Cycle_end"] = windowed["Cycle_start"] + window_size - 1
    paths = []
    metric_rows = []
    columns = 4
    rows = int(np.ceil(len(sensors) / columns))

    for (regime_id, cycle_start, cycle_end), window_rows in windowed.groupby(
        ["regime_id", "Cycle_start", "Cycle_end"], sort=True
    ):
        figure, axes = plt.subplots(rows, columns, figsize=(16, 3.0 * rows), squeeze=False)
        plotted_sensors = 0
        for axis, sensor in zip(axes.flat, sensors):
            sensor_rows = window_rows.loc[window_rows["Sensor"] == sensor]
            if sensor_rows.empty:
                axis.set_visible(False)
                continue
            actual = sensor_rows["Actual"].to_numpy(dtype=float)
            predicted = sensor_rows["Predicted"].to_numpy(dtype=float)
            score = float(r2_score(actual, predicted)) if len(actual) > 1 and np.var(actual) > 1e-12 else np.nan
            errors = actual - predicted
            metrics = {
                "R2": score,
                "RMSE": float(np.sqrt(np.mean(errors ** 2))),
                "MAE": float(np.mean(np.abs(errors))),
                "target_variance": float(np.var(actual)),
            }
            metric_rows.append({
                "Regime_ID": int(regime_id),
                "Sensor": sensor,
                "Cycle_start": int(cycle_start),
                "Cycle_end": int(cycle_end),
                "Rows": len(sensor_rows),
                "Engines": int(sensor_rows["unit_id"].nunique()),
                **metrics,
            })
            if not np.isfinite(score) or score < MIN_SENSOR_R2_FOR_PLOT:
                axis.set_visible(False)
                continue

            cycle_summary = sensor_rows.groupby("cycle").agg(
                Actual_mean=("Actual", "mean"),
                Actual_std=("Actual", "std"),
                Predicted_mean=("Predicted", "mean"),
                Predicted_std=("Predicted", "std"),
            ).fillna(0.0)
            cycles = cycle_summary.index.to_numpy(dtype=int)
            actual_mean = cycle_summary["Actual_mean"].to_numpy(dtype=float)
            actual_std = cycle_summary["Actual_std"].to_numpy(dtype=float)
            predicted_mean = cycle_summary["Predicted_mean"].to_numpy(dtype=float)
            predicted_std = cycle_summary["Predicted_std"].to_numpy(dtype=float)
            axis.plot(cycles, actual_mean, color="#26364a", linewidth=1.4, label="Actual")
            axis.fill_between(cycles, actual_mean - actual_std, actual_mean + actual_std, color="#26364a", alpha=0.12)
            axis.plot(cycles, predicted_mean, color="#326bff", linewidth=1.4, linestyle="--", label="Predicted")
            axis.fill_between(cycles, predicted_mean - predicted_std, predicted_mean + predicted_std, color="#326bff", alpha=0.14)
            score_label = f"R² {score:.3f}" if np.isfinite(score) else "R² n/a"
            axis.set_title(f"{sensor}: {score_label}")
            axis.set_xlim(cycle_start, cycle_end)
            axis.set_xlabel("Cycle")
            axis.set_ylabel("Sensor value")
            axis.grid(alpha=0.2)
            plotted_sensors += 1

        for axis in axes.flat[len(sensors):]:
            axis.set_visible(False)
        if plotted_sensors == 0:
            plt.close(figure)
            continue
        figure.suptitle(f"Regime {int(regime_id)} sensor actual vs predicted | cycles {int(cycle_start)}-{int(cycle_end)}")
        figure.legend(
            handles=[
                Line2D([0], [0], color="#26364a", label="Actual mean"),
                Line2D([0], [0], color="#326bff", linestyle="--", label="Predicted mean"),
            ],
            loc="lower center", ncol=2, frameon=False,
        )
        figure.tight_layout(rect=(0, 0.035, 1, 0.97))
        paths.append(save_figure(
            figure,
            directory / f"regime_{int(regime_id)}_cycles_{int(cycle_start):03d}-{int(cycle_end):03d}.png",
        ))

    return paths, pd.DataFrame(metric_rows)


def plot_subset_rul_parity(datasets: dict[str, pd.DataFrame], predictions: dict[str, np.ndarray], path: Path) -> Path:
    """Plot pooled grouped-CV RUL actual-vs-predicted scores by subset."""
    subsets = list(datasets)
    columns = 2
    rows = int(np.ceil(len(subsets) / columns))
    figure, axes = plt.subplots(rows, columns, figsize=(12, 5.5 * rows), squeeze=False)
    for axis, subset in zip(axes.flat, subsets):
        truth = datasets[subset]["RUL"].to_numpy(dtype=float)
        estimate = np.asarray(predictions[subset], dtype=float)
        score = float(r2_score(truth, estimate))
        lower = float(min(truth.min(), estimate.min()))
        upper = float(max(truth.max(), estimate.max()))
        padding = (upper - lower) * 0.04 or 1.0
        axis.scatter(truth, estimate, s=5, alpha=0.15, color="#326bff", edgecolors="none", rasterized=True)
        axis.plot([lower - padding, upper + padding], [lower - padding, upper + padding], color="#26364a", linestyle="--", linewidth=1.2)
        axis.set_xlim(lower - padding, upper + padding)
        axis.set_ylim(lower - padding, upper + padding)
        axis.set_xlabel("Actual RUL (cycles)")
        axis.set_ylabel("Predicted RUL (cycles)")
        axis.set_title(f"{subset}: grouped OOF R² {score:.3f}")
    figure.suptitle("C-MAPSS RUL actual vs predicted by subset", y=1.0)
    figure.tight_layout()
    return save_figure(figure, path)


def plot_signal_filtering_comparison(
    frame: pd.DataFrame,
    unit_id: int,
    sensor: str,
    path: Path,
    alphas: tuple[float, ...] = (0.1, 0.35, 0.7),
) -> Path:
    """Plot raw sensor data vs EWMA low-pass filtering at multiple alphas, mirroring Padua's slide 17."""
    trace = frame.loc[frame["unit_id"] == unit_id].sort_values("cycle").copy().reset_index(drop=True)
    cycles = trace["cycle"].to_numpy()
    raw = trace[sensor].to_numpy(dtype=float)

    figure, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)

    # Panel 1: Sinal cru (Sem Filtro)
    axes[0].plot(cycles, raw, color="#bd1e1e", linewidth=1.5, label=f"{sensor} cru")
    axes[0].set_title(f"{sensor} cru (Sem filtro) - Motor {unit_id}", fontsize=11, fontweight="bold")
    axes[0].set_xlabel("Ciclo operacional")
    axes[0].set_ylabel(f"{sensor} medido")
    axes[0].grid(True, linestyle=":", alpha=0.5)
    axes[0].legend(loc="upper left", frameon=True)

    # Panel 2: EWMA com alpha = 0.1 (alta inércia / filtro suave)
    ewma_slow = pd.Series(raw).ewm(alpha=alphas[0], adjust=False).mean().to_numpy()
    axes[1].plot(cycles, ewma_slow, color="#2e7d32", linewidth=1.8, label=f"EWMA (α = {alphas[0]})")
    axes[1].set_title(f"{sensor} EWMA (α = {alphas[0]}) - Suave", fontsize=11, fontweight="bold")
    axes[1].set_xlabel("Ciclo operacional")
    axes[1].grid(True, linestyle=":", alpha=0.5)
    axes[1].legend(loc="upper left", frameon=True)

    # Panel 3: EWMA com alpha = 0.7 (baixa inércia / mais responsivo)
    ewma_fast = pd.Series(raw).ewm(alpha=alphas[-1], adjust=False).mean().to_numpy()
    axes[2].plot(cycles, ewma_fast, color="#326bff", linewidth=1.8, label=f"EWMA (α = {alphas[-1]})")
    axes[2].set_title(f"{sensor} EWMA (α = {alphas[-1]}) - Responsivo", fontsize=11, fontweight="bold")
    axes[2].set_xlabel("Ciclo operacional")
    axes[2].grid(True, linestyle=":", alpha=0.5)
    axes[2].legend(loc="upper left", frameon=True)

    figure.suptitle(
        f"Enriquecimento Temporal: Comparativo com e sem Filtros EWMA ({sensor})",
        fontsize=12, fontweight="bold", y=1.02,
    )
    figure.tight_layout()
    return save_figure(figure, path)


def plot_minirocket_feature_filtering(
    frame: pd.DataFrame,
    unit_id: int,
    path: Path,
    sensor: str = "T50",
    random_state: int = 42,
) -> Path:
    """Plot MiniRocket convolutional kernel response, PPV thresholding, and feature trajectory (slides 19 & 21)."""
    trace = frame.loc[frame["unit_id"] == unit_id].sort_values("cycle").copy().reset_index(drop=True)
    cycles = trace["cycle"].to_numpy()
    raw = trace[sensor].to_numpy(dtype=float)
    rul = trace["RUL"].to_numpy(dtype=float) if "RUL" in trace else None

    figure, axes = plt.subplots(1, 3, figsize=(16, 4.8))

    # Panel 1 (Slide 21 left): Sinal Cru e RUL
    ax0_twin = axes[0].twinx()
    p1 = axes[0].plot(cycles, raw, color="#bd1e1e", linewidth=1.4, label=f"{sensor} cru")
    p2 = []
    if rul is not None:
        p2 = ax0_twin.plot(cycles, rul, color="#7444ed", linestyle="--", linewidth=1.4, label="RUL real")
        ax0_twin.set_ylabel("RUL (ciclos)", color="#7444ed")
        ax0_twin.tick_params(axis="y", labelcolor="#7444ed")
    axes[0].set_title(f"{sensor} cru e Degradação (RUL) - Motor {unit_id}", fontsize=11, fontweight="bold")
    axes[0].set_xlabel("Ciclo operacional")
    axes[0].set_ylabel(f"{sensor} medido")
    axes[0].grid(True, linestyle=":", alpha=0.5)
    lines = p1 + p2
    axes[0].legend(lines, [l.get_label() for l in lines], loc="upper right", frameon=True)

    # Panel 2 (Slide 19): Convoluções & Limiar PPV (demonstração com kernel determinístico L=9)
    np.random.seed(random_state)
    mid = len(raw) // 2
    window = raw[max(0, mid - 15): min(len(raw), mid + 15)]
    kernel = np.array([-1, -1, 2, -1, 2, -1, -1, 2, -1], dtype=float)
    conv_resp = np.convolve(window, kernel, mode="valid")
    conv_resp = (conv_resp - conv_resp.mean()) / (conv_resp.std() or 1.0)
    m_resp = len(conv_resp)
    b = 0.5
    colors = ["#2e7d32" if c > b else "#888888" for c in conv_resp]
    axes[1].bar(range(1, m_resp + 1), conv_resp, color=colors, edgecolor="black", linewidth=0.6, width=0.7)
    axes[1].axhline(b, color="#d95f02", linestyle="--", linewidth=1.8, label=f"Limiar b = {b}")
    pos_count = int((conv_resp > b).sum())
    ppv_val = pos_count / m_resp
    axes[1].set_title(f"MiniRocket PPV: {pos_count}/{m_resp} respostas > b (PPV = {ppv_val:.2f})", fontsize=11, fontweight="bold")
    axes[1].set_xlabel("Posição do filtro convolucional (j)")
    axes[1].set_ylabel("Resposta convolucional (Cj)")
    axes[1].grid(True, linestyle=":", alpha=0.5)
    axes[1].legend(loc="upper right", frameon=True)

    # Panel 3 (Slide 21 right): Trajetória de Ativação PPV convolucional
    window_size = 15
    all_sub_c = []
    for end in range(window_size, len(raw) + 1):
        sub_w = raw[end - window_size:end]
        sub_c = np.convolve(sub_w, kernel, mode="valid")
        all_sub_c.append(sub_c)
    all_sub_c = np.vstack(all_sub_c)
    b_healthy = float(np.quantile(all_sub_c[:min(35, len(all_sub_c))], 0.45))
    ppv_trajectory = (all_sub_c > b_healthy).mean(axis=1)
    ppv_smooth = pd.Series(ppv_trajectory).ewm(alpha=0.15, adjust=False).mean().to_numpy()

    cycles_w = cycles[window_size - 1:]
    axes[2].plot(cycles_w, ppv_trajectory, color="#02818a", alpha=0.35, linewidth=1.2, label="PPV instantâneo")
    axes[2].plot(cycles_w, ppv_smooth, color="#01665e", linewidth=2.2, label="PPV suavizado (tendência)")
    axes[2].set_title(f"Atributo PPV Convolucional vs Degradação - Motor {unit_id}", fontsize=11, fontweight="bold")
    axes[2].set_xlabel("Ciclo operacional")
    axes[2].set_ylabel(f"PPV(C, b={b_healthy:.1f}) ∈ [0, 1]")
    axes[2].grid(True, linestyle=":", alpha=0.5)
    axes[2].legend(loc="upper left", frameon=True)

    figure.suptitle(
        "Enriquecimento Convolucional MiniRocket: Resposta de Filtros e PPV",
        fontsize=12, fontweight="bold", y=1.02,
    )
    figure.tight_layout()
    return save_figure(figure, path)


def plot_time_series_envelope_detection(
    scored_trace: pd.DataFrame,
    summary: dict,
    path: Path,
    title: str = "Detecção de Anomalia por Envelope de Série Temporal (Estilo Power BI)",
    sensor_name: str = "T50",
    healthy_cycle_limit: int = 50,
) -> Path:
    """Plot time-series prediction envelope ('embrulho'), highlighting breaches, first alarm, and product breakage."""
    trace = scored_trace.sort_values("cycle").copy().reset_index(drop=True)
    figure, (ax_main, ax_breach) = plt.subplots(2, 1, figsize=(12, 8.0), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})

    cycles = trace["cycle"].to_numpy()
    actual = trace[sensor_name].to_numpy()
    pred = trace["prediction"].to_numpy()
    upper = trace["envelope_upper"].to_numpy()
    lower = trace["envelope_lower"].to_numpy()
    anomalies = trace["out_of_envelope"].to_numpy(dtype=bool)
    breach_dist = trace["breach_distance"].to_numpy()

    max_cycle = float(cycles.max())
    eol_cycle = int(summary["eol_cycle"])
    alarm_cycle = summary["first_alarm_cycle"]
    lead_time = summary["lead_time_cycles"]

    # Shading for healthy calibration and monitoring
    for ax in (ax_main, ax_breach):
        ax.axvspan(0, healthy_cycle_limit, color="#10b981", alpha=0.12, label=f"Calibração Saudável (0–{healthy_cycle_limit})")
        ax.axvspan(healthy_cycle_limit, max_cycle + 5, color="#f43f5e", alpha=0.06, label="Monitoramento Online")
        ax.axvline(healthy_cycle_limit, color="#059669", linestyle="--", linewidth=1.2, alpha=0.85, label="Limite de Calibração")

    # Main plot
    unit_label = SENSOR_UNITS.get(sensor_name, "")
    unit_str = f" ({unit_label})" if unit_label else ""

    has_regimes = "regime_id" in trace.columns and trace["regime_id"].nunique() > 1
    if has_regimes:
        cmap = plt.get_cmap("tab10")
        for r_id, r_rows in trace.groupby("regime_id"):
            ax_main.scatter(
                r_rows["cycle"], r_rows[sensor_name],
                color=cmap(int(r_id) % 10), s=16, alpha=0.75, label=f"Regime {r_id}" if int(r_id) < 6 else "",
                zorder=3
            )
        ax_main.plot(cycles, pred, color="#2563eb", linewidth=1.6, linestyle="--", label="Sensor Virtual (Predição)", zorder=2)
    else:
        ax_main.plot(cycles, actual, color="#1e293b", linewidth=1.7, label=f"Sinal Medido ({sensor_name})", zorder=3)
        ax_main.plot(cycles, pred, color="#2563eb", linewidth=1.6, linestyle="--", label="Sensor Virtual (Predição)", zorder=3)

    # Shaded Envelope ("Embrulho")
    ax_main.fill_between(
        cycles, lower, upper,
        color="#94a3b8", alpha=0.35, label=f"Envelope de Predição ('Embrulho' ±3σ: {summary.get('envelope_margin', 0):.2f})", zorder=2,
    )
    ax_main.plot(cycles, upper, color="#64748b", linestyle=":", linewidth=1.0, alpha=0.7)
    ax_main.plot(cycles, lower, color="#64748b", linestyle=":", linewidth=1.0, alpha=0.7)

    # Highlight anomalous points outside the envelope
    if np.any(anomalies):
        ax_main.scatter(
            cycles[anomalies], actual[anomalies],
            color="#ef4444", s=32, zorder=4, label=f"Anomalia Estrutural (Fora do Embrulho, n={int(np.sum(anomalies))})",
            edgecolors="#991b1b", linewidth=0.6,
        )

    # Mark First Alarm
    if alarm_cycle is not None:
        alarm_idx = np.where(cycles == alarm_cycle)[0]
        if len(alarm_idx):
            alarm_val = actual[alarm_idx[0]]
            ax_main.scatter([alarm_cycle], [alarm_val], marker="*", s=260, color="#dc2626", edgecolors="#7f1d1d", linewidth=1.2, zorder=6, label="PRIMEIRO ALARME")
            ax_main.annotate(
                f"PRIMEIRO ALARME: Ciclo {int(alarm_cycle)}\nLead Time: {int(lead_time)} ciclos de antecedência",
                xy=(alarm_cycle, alarm_val),
                xytext=(max(15, alarm_cycle - 55), alarm_val + (upper.max() - lower.min()) * 0.18),
                arrowprops=dict(arrowstyle="->", color="#dc2626", lw=1.8),
                fontsize=9.5, fontweight="bold", color="#991b1b",
                bbox=dict(boxstyle="round,pad=0.35", fc="#fef2f2", ec="#dc2626", lw=1.2),
            )

    # Mark Product Breakage (Quebra do Produto / EOL)
    eol_idx = np.where(cycles == eol_cycle)[0]
    if len(eol_idx):
        eol_val = actual[eol_idx[0]]
        ax_main.scatter([eol_cycle], [eol_val], marker="X", s=220, color="#1e1b4b", edgecolors="#dc2626", linewidth=1.5, zorder=6, label="QUEBRA DO PRODUTO (EOL)")
        ax_main.annotate(
            f"QUEBRA DO PRODUTO (EOL)\nCiclo {int(eol_cycle)} (Falha do Motor)",
            xy=(eol_cycle, eol_val),
            xytext=(max(20, eol_cycle - 45), eol_val - (upper.max() - lower.min()) * 0.22),
            arrowprops=dict(arrowstyle="->", color="#1e1b4b", lw=1.8),
            fontsize=9.5, fontweight="bold", color="#1e1b4b",
            bbox=dict(boxstyle="round,pad=0.35", fc="#e0e7ff", ec="#3730a3", lw=1.2),
        )

    ax_main.set_ylabel(f"{sensor_name}{unit_str}", fontsize=10, fontweight="bold")
    coverage_pct = summary.get("in_envelope_pct", 100.0)
    ax_main.set_title(
        f"{title} - Motor {summary.get('unit_id', 'Holdout')}\n"
        f"Cobertura dentro do embrulho: {coverage_pct:.1f}% | Alarme no Ciclo {alarm_cycle} (Antecedência: {lead_time} ciclos da Quebra)",
        fontsize=11, fontweight="bold",
    )
    ax_main.legend(loc="upper left", ncol=2, frameon=True, fontsize=8.5)
    ax_main.grid(True, linestyle=":", alpha=0.5)
    ax_main.set_xlim(0, max_cycle + 5)

    # Breach distance plot
    ax_breach.plot(cycles, breach_dist, color="#dc2626", linewidth=1.4, label="Distância Fora do Envelope (|Δ| > Margem)")
    ax_breach.fill_between(cycles, 0, breach_dist, color="#ef4444", alpha=0.25)
    ax_breach.axhline(0.0, color="#64748b", linestyle="-", linewidth=0.8)

    if alarm_cycle is not None and len(alarm_idx):
        ax_breach.scatter([alarm_cycle], [breach_dist[alarm_idx[0]]], marker="*", s=200, color="#dc2626", zorder=5)
    if len(eol_idx):
        ax_breach.scatter([eol_cycle], [breach_dist[eol_idx[0]]], marker="X", s=180, color="#1e1b4b", edgecolors="#dc2626", zorder=5)

    ax_breach.set_xlabel("Ciclo Operacional", fontsize=10, fontweight="bold")
    ax_breach.set_ylabel("Rompimento do Envelope", fontsize=10, fontweight="bold")
    ax_breach.set_ylim(bottom=0.0)
    ax_breach.grid(True, linestyle=":", alpha=0.5)
    ax_breach.legend(loc="upper left", frameon=True, fontsize=8.5)

    figure.tight_layout()
    return save_figure(figure, path)


def plot_geometric_envelope_detection(
    scored_geom: pd.DataFrame,
    summary: dict,
    path: Path,
    title: str = "Envelope Geométrico de Sinal Digital (Rolling Peak/Valley Hull)",
    sensor_name: str = "T50",
    healthy_cycle_limit: int = 50,
) -> Path:
    """Plot digital signal geometric envelope wrapping peaks and valleys, highlighting modulation and breakage."""
    df = scored_geom.sort_values("cycle").copy().reset_index(drop=True)
    figure, (ax_main, ax_amp) = plt.subplots(2, 1, figsize=(12, 7.8), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})

    cycles = df["cycle"].to_numpy()
    signal = df["geom_signal"].to_numpy() if "geom_signal" in df.columns else df[sensor_name].to_numpy()
    upper_hull = df["geom_upper_hull"].to_numpy()
    lower_hull = df["geom_lower_hull"].to_numpy()
    amp = df["geom_amplitude"].to_numpy()

    unit_label = SENSOR_UNITS.get(sensor_name, "")
    unit_str = f" ({unit_label})" if unit_label else ""

    max_cycle = float(cycles.max())
    eol_cycle = int(summary["eol_cycle"])
    alarm_cycle = summary["first_alarm_cycle"]
    lead_time = summary["lead_time_cycles"]
    amp_thresh = summary["amp_threshold"]
    upper_ceil = summary["upper_ceiling"]

    for ax in (ax_main, ax_amp):
        ax.axvspan(0, healthy_cycle_limit, color="#10b981", alpha=0.12, label=f"Calibração (0–{healthy_cycle_limit})")
        ax.axvspan(healthy_cycle_limit, max_cycle + 5, color="#f43f5e", alpha=0.06, label="Operação Contínua")
        ax.axvline(healthy_cycle_limit, color="#059669", linestyle="--", linewidth=1.2, alpha=0.85)

    # Main plot: Signal and geometric envelope
    signal_label = f"Resíduo Desacoplado ({sensor_name})" if "geom_signal" in df.columns else f"Sinal Bruto ({sensor_name})"
    ax_main.plot(cycles, signal, color="#334155", linewidth=1.2, alpha=0.75, label=signal_label)
    ax_main.plot(cycles, upper_hull, color="#0284c7", linewidth=1.6, label="Envelope Geométrico Superior (Upper Hull)")
    ax_main.plot(cycles, lower_hull, color="#0284c7", linewidth=1.6, linestyle="-.", label="Envelope Geométrico Inferior (Lower Hull)")
    ax_main.fill_between(cycles, lower_hull, upper_hull, color="#38bdf8", alpha=0.22, label="Embrulho Geométrico")
    ax_main.axhline(upper_ceil, color="#b91c1c", linestyle=":", linewidth=1.5, label=f"Teto Superior ({upper_ceil:.2f})")

    # Mark First Alarm and Breakage
    if alarm_cycle is not None:
        a_idx = np.where(cycles == alarm_cycle)[0]
        if len(a_idx):
            val = signal[a_idx[0]]
            ax_main.scatter([alarm_cycle], [val], marker="*", s=260, color="#dc2626", edgecolors="#7f1d1d", zorder=6, label="PRIMEIRO ALARME")
            ax_main.annotate(
                f"PRIMEIRO ALARME: Ciclo {int(alarm_cycle)}\nModulação anômala do envelope ({lead_time} ciclos antes da falha)",
                xy=(alarm_cycle, val),
                xytext=(max(15, alarm_cycle - 50), val + (upper_hull.max() - lower_hull.min()) * 0.15),
                arrowprops=dict(arrowstyle="->", color="#dc2626", lw=1.6),
                fontsize=9.5, fontweight="bold", color="#991b1b",
                bbox=dict(boxstyle="round,pad=0.35", fc="#fef2f2", ec="#dc2626", lw=1.2),
            )

    e_idx = np.where(cycles == eol_cycle)[0]
    if len(e_idx):
        e_val = signal[e_idx[0]]
        ax_main.scatter([eol_cycle], [e_val], marker="X", s=220, color="#1e1b4b", edgecolors="#dc2626", linewidth=1.5, zorder=6, label="QUEBRA DO PRODUTO (EOL)")
        ax_main.annotate(
            f"QUEBRA DO PRODUTO (EOL)\nCiclo {int(eol_cycle)}",
            xy=(eol_cycle, e_val),
            xytext=(max(20, eol_cycle - 40), e_val - (upper_hull.max() - lower_hull.min()) * 0.20),
            arrowprops=dict(arrowstyle="->", color="#1e1b4b", lw=1.6),
            fontsize=9.5, fontweight="bold", color="#1e1b4b",
            bbox=dict(boxstyle="round,pad=0.35", fc="#e0e7ff", ec="#3730a3", lw=1.2),
        )

    ax_main.set_ylabel(f"{sensor_name}{unit_str}", fontsize=10, fontweight="bold")
    ax_main.set_title(
        f"{title} - Motor {summary.get('unit_id', 'Holdout')}\n"
        f"Alarme no Ciclo {alarm_cycle} | Quebra no Ciclo {eol_cycle} | Lead Time: {lead_time} ciclos",
        fontsize=11, fontweight="bold",
    )
    ax_main.legend(loc="upper left", ncol=2, frameon=True, fontsize=8.5)
    ax_main.grid(True, linestyle=":", alpha=0.5)
    ax_main.set_xlim(0, max_cycle + 5)

    # Amplitude modulation plot
    ax_amp.plot(cycles, amp, color="#0369a1", linewidth=1.6, label="Largura do Envelope Geométrico (Pico - Vale)")
    ax_amp.axhline(amp_thresh, color="#dc2626", linestyle="--", linewidth=1.5, label=f"Limiar de Modulação ({amp_thresh:.2f})")
    ax_amp.fill_between(cycles, amp_thresh, np.maximum(amp, amp_thresh), where=(amp > amp_thresh), color="#ef4444", alpha=0.35)

    if alarm_cycle is not None and len(a_idx):
        ax_amp.scatter([alarm_cycle], [amp[a_idx[0]]], marker="*", s=200, color="#dc2626", zorder=5)
    if len(e_idx):
        ax_amp.scatter([eol_cycle], [amp[e_idx[0]]], marker="X", s=180, color="#1e1b4b", edgecolors="#dc2626", zorder=5)

    ax_amp.set_xlabel("Ciclo Operacional", fontsize=10, fontweight="bold")
    ax_amp.set_ylabel("Largura do Envelope", fontsize=10, fontweight="bold")
    ax_amp.grid(True, linestyle=":", alpha=0.5)
    ax_amp.legend(loc="upper left", frameon=True, fontsize=8.5)

    figure.tight_layout()
    return save_figure(figure, path)


def plot_actual_vs_predicted_with_envelope(
    y_true,
    y_pred,
    path: Path,
    title: str,
    value_label: str = "RUL (ciclos)",
    k_sigma: float = 3.0,
    cal_sigma: float | None = None,
) -> Path:
    """Plot actual-vs-predicted scatter with identity line and shaded prediction envelope corridor ('embrulho')."""
    truth = np.asarray(y_true, dtype=float)
    prediction = np.asarray(y_pred, dtype=float)
    residuals = truth - prediction
    score = float(r2_score(truth, prediction))
    rmse = float(np.sqrt(np.mean(residuals ** 2)))
    mae = float(np.mean(np.abs(residuals)))

    sigma = cal_sigma if cal_sigma is not None and cal_sigma > 0 else float(np.std(residuals))
    margin = k_sigma * sigma

    out_of_envelope = np.abs(residuals) > margin
    in_envelope_pct = float(np.mean(~out_of_envelope) * 100.0)
    out_pct = float(np.mean(out_of_envelope) * 100.0)

    lower = float(min(truth.min(), prediction.min()))
    upper = float(max(truth.max(), prediction.max()))
    padding = (upper - lower) * 0.05 or 1.0

    figure, ax = plt.subplots(figsize=(7.5, 7.0))
    grid = np.linspace(lower - padding, upper + padding, 200)

    # Shaded Envelope Corridor along identity line
    ax.fill_between(
        grid, grid - margin, grid + margin,
        color="#cbd5e1", alpha=0.55, label=f"Embrulho de Tolerância (±{k_sigma:.1f}σ: ±{margin:.1f} {value_label})",
    )
    ax.plot(grid, grid + margin, color="#94a3b8", linestyle=":", linewidth=1.1)
    ax.plot(grid, grid - margin, color="#94a3b8", linestyle=":", linewidth=1.1)
    ax.plot(grid, grid, color="#1e293b", linestyle="--", linewidth=1.6, label="Paridade Perfeita (y = x)")

    # Scatter nominal vs anomalous
    ax.scatter(
        truth[~out_of_envelope], prediction[~out_of_envelope],
        s=16, alpha=0.45, color="#2563eb", edgecolors="none", label=f"Nominal no Embrulho ({in_envelope_pct:.1f}%)",
    )
    if np.any(out_of_envelope):
        ax.scatter(
            truth[out_of_envelope], prediction[out_of_envelope],
            s=28, alpha=0.85, color="#ef4444", edgecolors="#991b1b", linewidth=0.5,
            label=f"Anomalia / Fora do Embrulho ({out_pct:.1f}%, n={int(np.sum(out_of_envelope))})",
        )

    ax.set_xlim(lower - padding, upper + padding)
    ax.set_ylim(lower - padding, upper + padding)
    ax.set_xlabel(f"Real {value_label}", fontsize=10, fontweight="bold")
    ax.set_ylabel(f"Predito {value_label}", fontsize=10, fontweight="bold")
    ax.set_title(
        f"{title}\nR² = {score:.4f} | RMSE = {rmse:.2f} | MAE = {mae:.2f}\n"
        f"Confiabilidade do Embrulho: {in_envelope_pct:.1f}% dentro da banda",
        fontsize=11, fontweight="bold",
    )
    ax.legend(loc="upper left", frameon=True, fontsize=8.5)
    ax.grid(True, linestyle=":", alpha=0.5)

    figure.tight_layout()
    return save_figure(figure, path)


# ============================================================================
# generate_thesis_figures
# ============================================================================

#!/usr/bin/env python3




def compute_acf(x: np.ndarray, nlags: int = 25) -> np.ndarray:
    """Compute sample autocorrelation function using numpy."""
    x = np.asarray(x, dtype=float)
    x = x - np.mean(x)
    var = np.sum(x**2)
    if var == 0:
        return np.ones(nlags + 1)
    r = np.correlate(x, x, mode="full")
    mid = len(x) - 1
    acf_vals = r[mid:mid + nlags + 1] / var
    return acf_vals

# Formatting
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Liberation Sans", "Helvetica", "Arial"]
plt.rcParams["axes.edgecolor"] = "#334155"
plt.rcParams["axes.linewidth"] = 0.8

PALETTE = {
    "bg_dark": "#0b0f19",
    "surface": "#131c2e",
    "card": "#1e293b",
    "border": "#334155",
    "cyan": "#06b6d4",
    "cyan_light": "#38bdf8",
    "amber": "#f59e0b",
    "amber_light": "#fbbf24",
    "red_alert": "#ef4444",
    "red_dark": "#b91c1c",
    "emerald": "#10b981",
    "purple": "#8b5cf6",
    "gray_text": "#94a3b8",
    "white": "#f8fafc",
}

FIGURES_DIR = DEFAULT_PATHS.guide_dir("FD004") / "figures"
TABLES_DIR = DEFAULT_PATHS.guide_dir("FD004") / "tables"


def _thesis_style():
    """Light print style shared with monitoring_figures (tokens in viz_style.py)."""
    from pdm_cmaps.reporting.viz_style import apply_style
    apply_style()


def _first_persistent(mask: np.ndarray, cycles: np.ndarray, k: int, after: int) -> int | None:
    """First cycle > ``after`` that starts ``k`` consecutive True values (Western-Electric style persistence)."""
    run = 0
    for i, flag in enumerate(mask):
        run = run + 1 if (flag and cycles[i] > after) else 0
        if run >= k:
            return int(cycles[i - k + 1])
    return None


def figure_1_noise_vs_damage(output_dir: Path | None = None, healthy_cycle_limit: int = 50, k: int = 2) -> Path:
    """Figure 1: an isolated envelope breach (noise) vs the first sustained run of breaches, on the sample engine.

    Every cycle and value printed in the figure is computed from envelope_scored_trace_t50_h50.csv.
    """
    from pdm_cmaps.reporting.viz_style import INK, INK_2, LINE_W, SERIES, STATUS, style_axes
    _thesis_style()
    figures_dir = Path(output_dir) / "figures" if output_dir else FIGURES_DIR
    figures_dir.mkdir(parents=True, exist_ok=True)
    tables_dir = Path(output_dir) / "tables" if output_dir else TABLES_DIR
    trace = pd.read_csv(tables_dir / "envelope_scored_trace_t50_h50.csv").sort_values("cycle").reset_index(drop=True)
    unit = int(trace["unit_id"].iloc[0])
    cyc = trace["cycle"].to_numpy()
    out = trace["out_of_envelope"].to_numpy(dtype=bool)
    sustained = _first_persistent(out, cyc, k, healthy_cycle_limit)
    prev_ = np.r_[False, out[:-1]]
    next_ = np.r_[out[1:], False]
    isolated = np.flatnonzero(out & ~prev_ & ~next_ & (cyc > healthy_cycle_limit))
    if sustained is not None:
        before = isolated[cyc[isolated] < sustained]
        isolated = before if before.size else isolated
    iso_cycle = int(cyc[isolated[0]]) if isolated.size else None

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
    panels = [(iso_cycle, "Violação isolada", "um ciclo fora do envelope, sem continuidade"),
              (sustained, f"Primeira sequência de {k}+ violações", "o padrão que a regra de persistência procura")]
    for ax, (center, title, subtitle) in zip(axes, panels):
        style_axes(ax, "y")
        if center is None:
            ax.text(0.5, 0.5, "não ocorreu neste motor", transform=ax.transAxes, ha="center", color=INK_2)
            ax.set_title(title)
            continue
        win = trace.loc[(trace["cycle"] >= center - 8) & (trace["cycle"] <= center + 12)]
        ax.fill_between(win["cycle"], win["envelope_lower"], win["envelope_upper"], color=SERIES[0], alpha=0.15, linewidth=0)
        ax.plot(win["cycle"], win["prediction"], color=SERIES[0], lw=LINE_W, linestyle=(0, (4, 2)))
        ax.plot(win["cycle"], win["T50"], color=INK, lw=1.4, marker="o", markersize=4)
        hit = win.loc[win["out_of_envelope"]]
        ax.plot(hit["cycle"], hit["T50"], linestyle="none", marker="o", markersize=11, markerfacecolor="none",
                markeredgecolor=STATUS[3], markeredgewidth=2)
        row = trace.loc[trace["cycle"] == center].iloc[0]
        excess = float(max(row["T50"] - row["envelope_upper"], row["envelope_lower"] - row["T50"]))
        ax.set_title(f"{title} (ciclo {center})\n{subtitle}; excesso de {excess:.2f} °R".replace(".", ","), fontsize=10)
        ax.set_xlabel("Ciclo")
        ax.set_ylabel("T50 (°R)")
    handles = [Line2D([], [], color=INK, lw=1.4, marker="o", markersize=4), Line2D([], [], color=SERIES[0], lw=LINE_W, linestyle=(0, (4, 2))),
               Patch(color=SERIES[0], alpha=0.15), Line2D([], [], linestyle="none", marker="o", markersize=10, markerfacecolor="none", markeredgecolor=STATUS[3], markeredgewidth=2)]
    fig.legend(handles, ["T50 medido", "Linha de base saudável (sensor virtual)", "Envelope ±3σ", "Fora do envelope"],
               loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.02))
    fig.suptitle(f"Ativo {unit} (FD004, motor de teste): ruído isolado × violação sustentada", x=0.01, ha="left", fontsize=12, fontweight="semibold")
    fig.tight_layout(rect=(0, 0.07, 1, 0.95))
    out_path = figures_dir / "thesis_anomaly_noise_vs_damage.png"
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print(f"Saved: {out_path}")
    return out_path


def figure_2_physical_cascade(output_dir: Path | None = None, healthy_cycle_limit: int = 50, k: int = 2) -> Path:
    """Figure 2: residuals of four sensors on the sample engine and the first persistent envelope alarm of each.

    Alarm cycles are computed (k consecutive envelope breaches after the healthy window). The order is a
    temporal sequence on ONE engine, not evidence of causality.
    """
    from pdm_cmaps.reporting.viz_style import INK, MUTED, SERIES, STATUS, style_axes
    _thesis_style()
    figures_dir = Path(output_dir) / "figures" if output_dir else FIGURES_DIR
    figures_dir.mkdir(parents=True, exist_ok=True)
    tables_dir = Path(output_dir) / "tables" if output_dir else TABLES_DIR
    meta = [("nc", "Nc, rotação do núcleo (rpm)"), ("p30", "P30, pressão de saída do HPC (psia)"),
            ("bpr", "BPR, razão de bypass"), ("t50", "T50, temperatura de saída da LPT (°R)")]
    fig, axes = plt.subplots(4, 1, figsize=(11, 9.5), sharex=True)
    rows, unit, eol = [], None, None
    for ax, (key, label) in zip(axes, meta):
        style_axes(ax, "y")
        df = pd.read_csv(tables_dir / f"envelope_scored_trace_{key}_h50.csv").sort_values("cycle")
        unit, eol = int(df["unit_id"].iloc[0]), int(df["cycle"].max())
        cyc = df["cycle"].to_numpy()
        res = df["residual"].to_numpy()
        half = (df["envelope_upper"] - df["prediction"]).to_numpy()
        lo = (df["envelope_lower"] - df["prediction"]).to_numpy()
        ax.fill_between(cyc, lo, half, color=SERIES[0], alpha=0.12, linewidth=0)
        ax.plot(cyc, res, color=SERIES[0], lw=1.2)
        ax.axhline(0, color=MUTED, lw=0.8)
        alarm = _first_persistent(df["out_of_envelope"].to_numpy(dtype=bool), cyc, k, healthy_cycle_limit)
        if alarm is not None:
            ax.axvline(alarm, color=STATUS[3], lw=1.6, linestyle=(0, (4, 2)))
            ax.text(alarm + 3, ax.get_ylim()[1] * 0.85, f"1º alarme persistente: ciclo {alarm}", color=INK, fontsize=9, va="top")
        ax.set_title(label, fontsize=10)
        ax.set_ylabel("Resíduo")
        rows.append({"sensor": key, "first_persistent_alarm": alarm})
    axes[-1].set_xlabel(f"Ciclo (fim de vida no ciclo {eol})")
    names = {"nc": "Nc", "p30": "P30", "bpr": "BPR", "t50": "T50"}
    order = " → ".join(f"{names[r['sensor']]} ({r['first_persistent_alarm']})" for r in sorted(rows, key=lambda r: (r["first_persistent_alarm"] is None, r["first_persistent_alarm"] or 0)))
    fig.suptitle(f"Ativo {unit}: resíduo de quatro sensores e envelope ±3σ\nOrdem do primeiro alarme: {order}\n(um motor; é uma sequência no tempo, não prova causalidade)",
                 x=0.01, ha="left", fontsize=11.5, fontweight="semibold")
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    out_path = figures_dir / "thesis_multisensor_physical_cascade.png"
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    pd.DataFrame(rows).to_csv(tables_dir / "thesis_multisensor_first_alarms.csv", index=False)
    print(f"Saved: {out_path}")
    return out_path


def figure_3_autocorrelation_evidence(output_dir: Path | None = None, healthy_cycle_limit: int = 50, degraded_window: int = 50) -> Path:
    """Figure 3: residual distribution and ACF in the healthy window vs the last cycles of life (sample engine).

    Windows match hypothesis tests H2/H3: healthy = cycle <= H; degraded = last ``degraded_window`` cycles.
    """
    from pdm_cmaps.reporting.viz_style import INK, MUTED, SERIES, style_axes
    _thesis_style()
    figures_dir = Path(output_dir) / "figures" if output_dir else FIGURES_DIR
    figures_dir.mkdir(parents=True, exist_ok=True)
    tables_dir = Path(output_dir) / "tables" if output_dir else TABLES_DIR
    trace = pd.read_csv(tables_dir / "envelope_scored_trace_t50_h50.csv").sort_values("cycle")
    unit, eol = int(trace["unit_id"].iloc[0]), int(trace["cycle"].max())
    start_deg = max(healthy_cycle_limit, eol - degraded_window)
    healthy = trace.loc[trace["cycle"] <= healthy_cycle_limit, "residual"].dropna().to_numpy()
    degraded = trace.loc[trace["cycle"] > start_deg, "residual"].dropna().to_numpy()
    fig, axes = plt.subplots(2, 2, figsize=(12, 7.5))
    lags = 15
    stats_rows = []
    for col, (vals, name, color) in enumerate([(healthy, f"fase saudável (ciclos 1–{healthy_cycle_limit})", SERIES[0]),
                                               (degraded, f"fim de vida (ciclos {start_deg + 1}–{eol})", SERIES[1])]):
        ax = axes[0, col]
        style_axes(ax, "y")
        bins = np.linspace(min(healthy.min(), degraded.min()), max(healthy.max(), degraded.max()), 26)
        ax.hist(vals, bins=bins, color=color, edgecolor="#ffffff", linewidth=1.0)
        mu, sd = float(np.mean(vals)), float(np.std(vals))
        ax.axvline(mu, color=INK, lw=1.2)
        ax.set_title(f"Resíduo do T50, {name}\nmédia {mu:.1f} °R, desvio {sd:.1f} °R, n = {len(vals)}".replace(".", ","), fontsize=10)
        ax.set_xlabel("Resíduo (°R)")
        ax.set_ylabel("Ciclos")
        ax = axes[1, col]
        style_axes(ax, "y")
        acf = compute_acf(vals, nlags=lags)
        conf = 1.96 / np.sqrt(len(vals))
        ax.bar(range(1, lags + 1), acf[1:], color=color, width=0.6)
        ax.axhspan(-conf, conf, color=MUTED, alpha=0.15, linewidth=0)
        ax.axhline(0, color=MUTED, lw=0.8)
        outside = int(np.sum(np.abs(acf[1:]) > conf))
        ax.set_title(f"Autocorrelação, {name}\nr₁ = {acf[1]:.2f}; {outside} de {lags} defasagens fora da faixa de 95 %".replace(".", ","), fontsize=10)
        ax.set_xlabel("Defasagem (ciclos)")
        ax.set_ylabel("r_k")
        ax.set_ylim(-0.4, 1.0)
        stats_rows.append({"window": name, "n": len(vals), "mean": mu, "std": sd, "r1": float(acf[1]), "lags_outside_95": outside})
    fig.suptitle(f"Ativo {unit}: o resíduo muda de nível e ganha memória no fim de vida\n(um motor; o teste de frota é o H3, Ljung–Box por motor + Fisher)",
                 x=0.01, ha="left", fontsize=11.5, fontweight="semibold")
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    out_path = figures_dir / "thesis_residual_autocorrelation_evidence.png"
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    pd.DataFrame(stats_rows).to_csv(tables_dir / "thesis_residual_window_stats.csv", index=False)
    print(f"Saved: {out_path}")
    return out_path


def figure_4_fleet_persistence_tradeoff(output_dir: Path | None = None, healthy_cycle_limit: int = 50, k_values=(1, 2, 3, 5)) -> Path:
    """Figure 4: effect of the persistence rule k on the fleet, COMPUTED from virtual_sensor_residuals_t50.csv.

    A cycle is a breach when |z| >= 3 and EWMA(|z|) >= UCL_t (same rule as StatisticalAnomalyDetector).
    For each k: detection = % engines with k consecutive breaches after the healthy window; false alarm =
    % engines with k consecutive breaches INSIDE the healthy window; lead time = EOL - first alarm.
    """
    from pdm_cmaps.reporting.viz_style import INK_2, MARKER_PT, SERIES, SURFACE, style_axes
    _thesis_style()
    figures_dir = Path(output_dir) / "figures" if output_dir else FIGURES_DIR
    figures_dir.mkdir(parents=True, exist_ok=True)
    tables_dir = Path(output_dir) / "tables" if output_dir else TABLES_DIR
    res = pd.read_csv(tables_dir / "virtual_sensor_residuals_t50.csv").sort_values(["unit_id", "cycle"])
    res["breach"] = (res["z_score"].abs() >= 3.0) & (res["ewma_abs_z"] >= res["ewma_ucl"])
    rows = []
    for k in k_values:
        det, fa, leads = 0, 0, []
        engines = 0
        for _, eng in res.groupby("unit_id"):
            engines += 1
            cyc = eng["cycle"].to_numpy()
            b = eng["breach"].to_numpy(dtype=bool)
            alarm = _first_persistent(b, cyc, k, healthy_cycle_limit)
            healthy_alarm = _first_persistent(b & (cyc <= healthy_cycle_limit), np.where(cyc <= healthy_cycle_limit, cyc, -1), k, 0)
            if alarm is not None:
                det += 1
                leads.append(int(cyc.max()) - alarm)
            if healthy_alarm is not None:
                fa += 1
        rows.append({"k": k, "engines": engines, "detection_pct": 100 * det / engines,
                     "healthy_false_alarm_engines_pct": 100 * fa / engines,
                     "mean_lead_time_cycles": float(np.mean(leads)) if leads else float("nan")})
    table = pd.DataFrame(rows)
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharex=True)
    specs = [("detection_pct", "Detecção (% dos motores)"), ("healthy_false_alarm_engines_pct", "Motores com alarme na fase saudável (%)"),
             ("mean_lead_time_cycles", "Antecedência média (ciclos)")]
    for ax, (col, title) in zip(axes, specs):
        style_axes(ax, "y")
        ax.plot(table["k"], table[col], color=SERIES[0], lw=2.0, marker="o", markersize=MARKER_PT, markeredgecolor=SURFACE, markeredgewidth=2)
        for x, y in zip(table["k"], table[col]):
            ax.annotate(f"{y:.1f}".replace(".", ","), (x, y), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8.5, color=INK_2)
        ax.set_title(title, fontsize=10)
        ax.set_xticks(list(k_values))
        ax.set_xlabel("k (violações consecutivas exigidas)")
        ax.margins(y=0.25)
    n = int(table["engines"].iloc[0])
    fig.suptitle(f"Regra de persistência na frota de teste ({n} motores, semente desta execução): k maior troca antecedência por menos alarmes",
                 x=0.01, ha="left", fontsize=11, fontweight="semibold")
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    out_path = figures_dir / "thesis_persistence_tradeoff_fleet.png"
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    table.to_csv(tables_dir / "thesis_persistence_tradeoff_fleet.csv", index=False)
    print(f"Saved: {out_path}")
    return out_path


def main() -> int:
    print("[thesis_figures] Generating high-resolution thesis proof figures...")
    figure_1_noise_vs_damage()
    figure_2_physical_cascade()
    figure_3_autocorrelation_evidence()
    figure_4_fleet_persistence_tradeoff()
    print("[thesis_figures] All 4 thesis figures successfully generated!")
    return 0


if __name__ == "__main__":
    import sys
    sys.exit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/reporting/viz_style.py
"""Shared figure style for the thesis figures (tokens + helpers), following the dataviz method.

Only colors, line/marker specs and axis styling live here.

Rules encoded here
  * Categorical series use fixed slots in fixed order (blue, orange, aqua). The first three validate
    all-pairs (scatter / small multiples). Aqua is below 3:1 contrast on the surface, so every figure
    that uses it also prints direct labels and has its CSV as the table equivalent.
  * Criticality levels use the STATUS palette (good / warning / serious / critical). Status colors
    never carry meaning alone: the level NAME is always printed next to the color.
  * Marks: 2 px lines, markers >= 8 px with a 2 px surface ring, hairline SOLID gridlines, no
    top/right spines. Text uses ink tokens, never a series color.
  * Print-only figures: no hover or dark mode. The CSV next to each figure is its table view.
"""

from __future__ import annotations

import matplotlib.pyplot as plt

SURFACE = "#ffffff"
INK = "#0b0b0b"
INK_2 = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
AXIS = "#c3c2b7"

SERIES = ("#2a78d6", "#eb6834", "#1baf7a")  # categorical slots 1-3 (blue, orange, aqua)
STATUS = ("#0ca30c", "#fab219", "#ec835a", "#d03b3b")  # good, warning, serious, critical
LEVEL_NAMES_PT = ("Normal", "Atenção", "Alerta", "Crítico")

LINE_W = 2.0
MARKER_PT = 8.0  # points; >= 8 px at the 100 dpi reference
RING_W = 2.0


def apply_style() -> None:
    """Global rcParams: system sans, surface background, ink text tokens."""
    plt.rcParams.update({
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "font.family": "sans-serif",
        "font.sans-serif": ["Inter", "Segoe UI", "Helvetica Neue", "Arial", "DejaVu Sans"],
        "text.color": INK,
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.titlesize": 11,
        "axes.titleweight": "semibold",
        "axes.titlelocation": "left",
        "axes.labelsize": 9.5,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.frameon": False,
        "legend.fontsize": 9,
        "axes.axisbelow": True,
    })


def style_axes(ax, grid: str = "y") -> None:
    """Hairline solid recessive grid and axes; no top/right spines."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(AXIS)
        ax.spines[side].set_linewidth(1.0)
    ax.tick_params(length=0, pad=4)
    ax.grid(False)
    if grid in ("y", "both"):
        ax.grid(axis="y", color=GRID, linewidth=0.8, linestyle="-")
    if grid in ("x", "both"):
        ax.grid(axis="x", color=GRID, linewidth=0.8, linestyle="-")


def ring_marker_kwargs(color: str, size: float = MARKER_PT, hollow: bool = False) -> dict:
    """Marker with a 2 px surface ring so it stays legible over lines and other markers."""
    return {
        "markersize": size,
        "markerfacecolor": SURFACE if hollow else color,
        "markeredgecolor": color if hollow else SURFACE,
        "markeredgewidth": RING_W,
        "linestyle": "none",
    }

### 2.2 Pacote `pdm_cmaps`: cli

pontos de entrada (`python -m pdm_cmaps`, `update_all`)

3 arquivos.

In [ ]:
%%writefile projeto/src/pdm_cmaps/cli/__init__.py
"""Command-line entry points: main.py (`python -m pdm_cmaps`) and update_all.py (extra studies + table sync)."""

In [ ]:
%%writefile projeto/src/pdm_cmaps/cli/main.py
#!/usr/bin/env python3
"""Command-line entry point: `python -m pdm_cmaps` (or the `pdm-cmaps` console script).

parse_arguments() -> PipelineConfig -> PdMPipeline.execute(); with --all, then cli.update_all runs the extra studies.
"""

from __future__ import annotations



import argparse
from pathlib import Path
import sys

from pdm_cmaps.config import DEFAULT_PATHS
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.detection.regime import RegimeDecoupler
from pdm_cmaps.modeling.benchmark import RULBenchmark
from pdm_cmaps.pipeline.pipeline import PdMPipeline
from pdm_cmaps.pipeline.config import PipelineConfig
from pdm_cmaps.progress import ProgressTracker


def parse_arguments(args: list[str] | None = None) -> PipelineConfig:
    """Parse command-line arguments into a strongly-typed PipelineConfig."""
    parser = argparse.ArgumentParser(
        description="NASA C-MAPSS Turbofan Predictive Maintenance Pipeline",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument(
        "--subset",
        choices=("FD001", "FD002", "FD003", "FD004", "ALL"),
        default="ALL",
        help="C-MAPSS dataset subset to process (or ALL for all four)",
    )
    parser.add_argument(
        "--all",
        dest="run_all",
        action="store_true",
        help="Execute pipeline sequentially across all subsets (FD001-FD004)",
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=42,
        help="Random seed for reproducibility and engine unit partitioning",
    )
    parser.add_argument(
        "--healthy-cycle-limit",
        type=int,
        default=50,
        help="Healthy operating cycle threshold (H) for virtual sensor calibration",
    )
    parser.add_argument(
        "--unit-id",
        type=int,
        default=179,
        help="Sample engine unit identifier for detailed trace visualizations",
    )
    parser.add_argument(
        "--include-minirocket",
        dest="include_minirocket",
        action="store_true",
        help="Also run the MiniRocket convolutional model (off by default; it is not part of the monograph)",
    )
    parser.add_argument(
        "--no-minirocket",
        dest="include_minirocket",
        action="store_false",
        help="Disable MiniRocket (the default)",
    )
    parser.set_defaults(include_minirocket=False)
    parser.add_argument(
        "--include-lstm",
        action="store_true",
        help="Enable PyTorch LSTM sequential model training",
    )
    parser.add_argument(
        "--no-clean",
        dest="clean_output",
        action="store_false",
        help="Do not clean existing output directories before execution",
    )
    parser.set_defaults(clean_output=True)
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=None,
        help="Custom output directory for results (tables and figures)",
    )

    parser.add_argument(
        "--only-pipeline",
        action="store_true",
        help="With --all: run only the per-subset pipeline, without the extra studies and table sync (update_all.py)",
    )

    parsed = parser.parse_args(args)
    return PipelineConfig(
        subset=parsed.subset,
        seed=parsed.seed,
        healthy_cycle_limit=parsed.healthy_cycle_limit,
        unit_id=parsed.unit_id,
        include_minirocket=parsed.include_minirocket,
        include_lstm=parsed.include_lstm,
        clean_output=parsed.clean_output,
        run_all=parsed.run_all,
        output_dir=parsed.output_dir,
    )


def main(args: list[str] | None = None) -> int:
    """Parse arguments, run the pipeline for each requested subset (with progress/ETA) and return the exit code."""
    # 1. Configuration
    config = parse_arguments(args)

    # 2. Determine target subsets
    target_subsets = ["FD001", "FD002", "FD003", "FD004"] if (config.subset == "ALL" or getattr(config, "run_all", False)) else [config.subset]

    # 3. Component instantiation and execution loop
    loader = CMapssLoader()
    exit_code = 0
    multi = len(target_subsets) > 1  # a single subset needs no progress file
    progress = ProgressTracker(len(target_subsets), log_file=DEFAULT_PATHS.logs_dir / "_progresso.txt" if multi else None)

    for sub in target_subsets:
        progress.start(f"pipeline {sub}")
        n_regimes = 6 if sub in {"FD002", "FD004"} else 1
        decoupler = RegimeDecoupler(n_clusters=n_regimes, random_state=config.seed)
        benchmark = RULBenchmark(random_state=config.seed)

        # Clone config for current subset
        sub_config = PipelineConfig(
            subset=sub,
            seed=config.seed,
            healthy_cycle_limit=config.healthy_cycle_limit,
            unit_id=config.unit_id if sub in {"FD002", "FD004"} else 45,
            include_minirocket=config.include_minirocket,
            include_lstm=config.include_lstm,
            clean_output=config.clean_output,
            output_dir=config.output_dir,
        )

        pipeline = PdMPipeline(
            config=sub_config,
            loader=loader,
            decoupler=decoupler,
            benchmark=benchmark,
        )
        res = pipeline.execute()
        progress.finish(f"pipeline {sub}", res)
        if res != 0:
            exit_code = res

    # `--all` atualiza tudo de uma vez: estudos adicionais, tabelas versionadas e painéis
    argv = args if args is not None else sys.argv[1:]
    if "--all" in argv and "--only-pipeline" not in argv:
        from pdm_cmaps.cli.update_all import run_studies_and_sync

        exit_code = run_studies_and_sync() or exit_code

    return exit_code


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile projeto/src/pdm_cmaps/cli/update_all.py
"""Reexecuta os estudos além do pipeline por dataset e sincroniza as tabelas da monografia (tcc-writing/tables/) e os gráficos Plotly.
O site de painéis fica em outro repositório; depois desta atualização, copie os resultados para ele com
`python fonte/atualizar_dados.py` (no repositório do dashboard).

Chamado por `python -m pdm_cmaps --all` (depois do pipeline FD001 a FD004). Também pode rodar sozinho:
    python -m pdm_cmaps.cli.update_all                    # estudos + sincronização
    python -m pdm_cmaps.cli.update_all --so-sincronizar
Cada estudo roda em um processo, um por vez, com log em results/logs/<nome>.log.
"""

from __future__ import annotations

import argparse
import os
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass, field

import pandas as pd

from pdm_cmaps.config import DEFAULT_PATHS, ProjectPaths
from pdm_cmaps.progress import ProgressTracker


@dataclass(frozen=True)
class StudySpec:
    """A study = a runnable module (`python -m <module> <args>`) with a log name."""

    name: str
    module: str
    args: tuple[str, ...] = ()


E_ = "pdm_cmaps.experiments"
STUDIES: tuple[StudySpec, ...] = (
    StudySpec("monitoring", "pdm_cmaps.monitoring.layer", ("--jobs", "3")),
    StudySpec("benchmark_repeated", f"{E_}.benchmark_repeated"),
    StudySpec("healthy_window", f"{E_}.healthy_window_study", ("--jobs", "3")),
    StudySpec("fault_detection", f"{E_}.fault_detection_study", ("--jobs", "3")),
    StudySpec("fault_model_diag13", f"{E_}.fault_model_study",
              ("--H", "25", "--subsets", "FD001", "FD003", "--jobs", "3", "--output-dir", "results/injecao_modelos_diag")),
    StudySpec("fault_model_diag24", f"{E_}.fault_model_study",
              ("--H", "25", "--subsets", "FD002", "FD004", "--jobs", "3", "--output-dir", "results/injecao_modelos_diag24")),
    StudySpec("fault_hard", f"{E_}.fault_hard_study", ("--jobs", "3")),
    StudySpec("fault_injection", f"{E_}.fault_injection", ("--subset", "FD004")),
    StudySpec("rul_predicoes", f"{E_}.rul_prediction_plots", ("--jobs", "3")),
    StudySpec("envelope_cases", f"{E_}.envelope_ignored_cases", ("--results", "results")),
)

# tabela versionada -> arquivo de origem em results/
TABLE_COPIES: dict[str, str] = {
    "benchmark_repeated_summary.csv": "benchmark_repeated/benchmark_repeated_summary.csv",
    "benchmark_repeated_tests.csv": "benchmark_repeated/benchmark_repeated_tests.csv",
    "benchmark_repeated_tests_por_motor.csv": "benchmark_repeated/benchmark_repeated_tests_por_motor.csv",
    "diferenca_vs_media_regime.csv": "falhas_dificeis/tabelas/diferenca_vs_media_regime.csv",
    "poder_por_forma.csv": "falhas_dificeis/tabelas/poder_por_forma.csv",
    "poder_por_magnitude.csv": "falhas_dificeis/tabelas/poder_por_magnitude.csv",
    "healthy_window_summary.csv": "healthy_window_study/tables/healthy_window_summary.csv",
    "magnitude_minima_detectavel.csv": "injecao_deteccao/tabelas/magnitude_minima_detectavel.csv",
    "pareado_far_resumo.csv": "injecao_deteccao/tabelas/pareado_far_resumo.csv",
    "regra_padrao_s1.csv": "injecao_deteccao/tabelas/regra_padrao_s1.csv",
    "thesis_hypothesis_tests.csv": "maintenance_guide_fd004/tables/thesis_hypothesis_tests.csv",
    **{f: f"monitoring_repeated/tables/{f}" for f in (
        "margin_check_point_thresholds_raw.csv", "monitoring_repeated_costs_summary.csv", "monitoring_repeated_spc_mv_summary.csv",
        "monitoring_repeated_spc_summary.csv", "monitoring_repeated_summary.csv", "monitoring_repeated_uncertainty_summary.csv",
        "monitoring_repeated_uncertainty_variants_summary.csv", "regime_variance_by_subset.csv")},
}

class StudyRunner:
    """Runs studies as subprocesses, one at a time, logging each to results/logs/<name>.log."""

    def __init__(self, paths: ProjectPaths = DEFAULT_PATHS, python: str = sys.executable) -> None:
        self.paths = paths
        self.python = python
        self._env = {**os.environ, "PYTHONPATH": os.pathsep.join(filter(None, [str(paths.root / "src"), os.environ.get("PYTHONPATH", "")]))}

    def _run(self, name: str, command: list[str]) -> int:
        self.paths.logs_dir.mkdir(parents=True, exist_ok=True)
        t0 = time.time()
        print(f"[{time.strftime('%H:%M:%S')}] {name}", flush=True)
        with open(self.paths.logs_dir / f"{name}.log", "w") as log:
            rc = subprocess.run(command, cwd=self.paths.root, env=self._env, stdout=log, stderr=subprocess.STDOUT).returncode
        line = f"[{time.strftime('%H:%M:%S')}] {name} exit={rc} ({(time.time() - t0) / 60:.1f} min)"
        print(line, flush=True)
        with open(self.paths.logs_dir / "_resumo.txt", "a") as f:
            f.write(line + "\n")
        return rc

    def run_study(self, study: StudySpec) -> int:
        return self._run(study.name, [self.python, "-m", study.module, *study.args])

    def run_module(self, name: str, module: str, *args: str) -> int:
        return self._run(name, [self.python, "-m", module, *args])

    def run_all(self, studies: tuple[StudySpec, ...] = STUDIES) -> int:
        rc = 0
        progress = ProgressTracker(len(studies), log_file=self.paths.logs_dir / "_progresso.txt")
        for study in studies:
            progress.start(study.name)
            study_rc = self.run_study(study)
            progress.finish(study.name, study_rc)
            rc = study_rc or rc
        return rc


@dataclass
class TableSynchronizer:
    """Copies result tables into the versioned `tcc-writing/tables/` and rebuilds dashboards that depend on them."""

    runner: StudyRunner
    paths: ProjectPaths = DEFAULT_PATHS
    copies: dict[str, str] = field(default_factory=lambda: dict(TABLE_COPIES))

    def sync(self) -> None:
        res, tab = self.paths.results_dir, self.paths.tables_dir
        tab.mkdir(parents=True, exist_ok=True)
        missing = []
        for dest, origin in self.copies.items():
            src = res / origin
            if src.exists():
                shutil.copy2(src, tab / dest)
            else:
                missing.append(origin)
        diag = [res / d / "tabelas" / "modelos_pareado_resumo.csv" for d in ("injecao_modelos_diag", "injecao_modelos_diag24")]
        if all(p.exists() for p in diag):
            pd.concat([pd.read_csv(p) for p in diag]).to_csv(tab / "decomposicao_regime_modelo.csv", index=False)
        else:
            missing.append("injecao_modelos_diag*/tabelas/modelos_pareado_resumo.csv")
        if (res / "falhas_dificeis" / "runs").exists():
            subprocess.run(
                [self.runner.python, "scripts/hard_by_subset.py", "--results", "results/falhas_dificeis",
                 "--out", str(tab / "falhas_dificeis_por_subconjunto.csv")],
                cwd=self.paths.root, env=self.runner._env, check=False, stdout=subprocess.DEVNULL)
        self.runner.run_module("sync_plotly_hub", "pdm_cmaps.reporting.plotly_hub",
                               "--results", "results", "--output-dir", "results/plotly", "--repo", ".")
        if missing:
            print("Atenção: fontes ausentes (não copiadas):", *missing, sep="\n  ")
        print("Sincronização concluída. Revise os números da monografia (Capítulo 5) contra tcc-writing/tables/.")


def run_studies_and_sync(paths: ProjectPaths = DEFAULT_PATHS) -> int:
    """Used by `--all`: every study followed by the table/dashboard synchronization."""
    runner = StudyRunner(paths)
    rc = runner.run_all()
    TableSynchronizer(runner, paths).sync()
    return rc


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--so-sincronizar", action="store_true")
    args = ap.parse_args(argv)
    runner = StudyRunner()
    rc = 0 if args.so_sincronizar else runner.run_all()
    TableSynchronizer(runner).sync()
    return rc


if __name__ == "__main__":
    sys.exit(main())

### 2.3 Planos de execução e scripts

`scripts/rodar.py` roda os planos (retomável); os demais são estudos pontuais.

5 arquivos.

In [ ]:
%%writefile projeto/scripts/hard_by_subset.py
"""Ganho do ML sobre a média do regime por subconjunto (falhas sorteadas; MEWMA e sensor afetado; IC por bootstrap de motores).
Uso (da raiz): python scripts/hard_by_subset.py --results results/falhas_dificeis --out tcc-writing/tables/falhas_dificeis_por_subconjunto.csv"""
import argparse
from pathlib import Path
import numpy as np, pandas as pd
from pdm_cmaps.experiments import fault_hard_study as fh

ap = argparse.ArgumentParser(); ap.add_argument("--results", type=Path, required=True); ap.add_argument("--out", type=Path, required=True)
a = ap.parse_args()
series, manifest = fh._load(a.results, "series"), fh._load(a.results, "manifesto")
rows = []
for sub in sorted(series["Subset"].unique()):
    d = fh.paired_delta(series[series["Subset"] == sub], manifest[manifest["Subset"] == sub], np.random.default_rng(0))
    rows.append(d.assign(Subset=sub))
out = pd.concat(rows)[["Subset", "Mode", "Estimator", "Chart", "family", "n_faults", "n_clusters", "delta_pp", "ci_lo", "ci_hi"]]
a.out.parent.mkdir(parents=True, exist_ok=True); out.to_csv(a.out, index=False)
pd.set_option("display.width", 220)
print(out[(out.Mode == "exogenous") & (out.Chart == "MEWMA")].round(1).to_string(index=False))

In [ ]:
%%writefile projeto/scripts/planos.py
"""Planos de execução para `rodar.py`: o que rodar, em que ordem, como saber que deu certo e quanto deve demorar.

Um plano é só dados (`Plan` com `Step`s). Para criar um novo, escreva uma função que devolve um `Plan` e registre em PLANOS.

Uso cotidiano:   tudo, rul_oficial, hiperparametros, diagnostico_fd003
Registro histórico (como cada conjunto de números do TCC foi regenerado; não são necessários para usar o projeto):
    engine_offset, varredura_janelas, multirregime, offset_todos, protocolo_principal,
    ic_bootstrap, injecao_h25, injecao_h35_50, mewma_nativo, deteccao_fd004
"""
from __future__ import annotations

import os
import sys

import pandas as pd
from rodar import ROOT, Context, Plan, Step

PY = sys.executable
SUBSETS = ("FD001", "FD002", "FD003", "FD004")
# Repositório do site de painéis, irmão deste; os passos "site" só existem nos planos que o atualizam.
DASHBOARD = (ROOT / os.environ.get("DASHBOARD_DIR", "../cmapss-turbofan-predictive-maintenance-dashboard")).resolve()


# ── verificações de sucesso ───────────────────────────────────────────────────
# O comando pode sair com código 0 sem ter gerado o resultado; cada passo confere o que deveria existir.
def existe(*paths: str):
    return lambda ctx: all((ctx.root / p).exists() for p in paths)


def n_arquivos(padrao: str, minimo: int):
    from glob import glob
    return lambda ctx: len(glob(str(ctx.root / padrao))) >= minimo


def pipeline_ok(fd: str):
    """Tabela principal existe e alguma tabela é mais nova que o início do plano (não vale resultado de uma execução antiga)."""
    tables = f"results/maintenance_guide_{fd.lower()}/tables"
    return lambda ctx: (ctx.root / tables / "rul_holdout_predictions.csv").exists() and any(
        p.stat().st_mtime > ctx.started for p in (ctx.root / tables).glob("*.csv"))


def manifesto_t0(fd: str, t0: int = 35):
    """Injeção de falhas: todas as séries cegas devem ter a falha no ciclo t0 (H = 25 -> t0 = 35)."""
    def check(ctx: Context) -> bool:
        m = pd.read_csv(ctx.root / f"results/injecao_de_falhas/{fd}_seed42/selado/manifesto_deteccao.csv")
        return set(m.t0) == {t0}
    return check


def sensor_virtual_ok(fd: str, estrito: bool = False):
    def check(ctx: Context) -> bool:
        d = pd.read_csv(ctx.root / f"results/virtual_sensor_models/{fd}/model_scores.csv")
        ok = "r2_late_motor_base" in d and d["model"].nunique() == 7 and len(d) >= 7 * 14
        return ok and (not estrito or d["r2_holdout"].min() > -5)
    return check


def linhas_minimas(path: str, minimo: int):
    return lambda ctx: sum(1 for _ in open(ctx.root / path)) >= minimo


def passo(name: str, modulo: str, *args: str, est: int = 600, check=lambda ctx: True, clean: tuple[str, ...] = (), after: tuple[str, ...] = ()) -> Step:
    """Passo que roda `python -m modulo args...`."""
    return Step(name, (PY, "-m", modulo, *args), check, est, clean, after)


def passos_do_site(tabela_ok) -> list[Step]:
    """Final comum das reexecuções: sincroniza as tabelas, roda os testes, copia os resultados para o dashboard e regenera o site."""
    return [
        passo("sincronizacao", "pdm_cmaps.cli.update_all", "--so-sincronizar", est=400, check=tabela_ok),
        passo("pytest", "pytest", "-q", est=60),
        Step("dados_site", (PY, str(DASHBOARD / "fonte/atualizar_dados.py")), lambda c: (DASHBOARD / "dados/tables/pareado_far_resumo.csv").exists(), 120),
        Step("site", (PY, str(DASHBOARD / "fonte/build_site.py")), lambda c: (DASHBOARD / "site/sensores_FD003.html").exists(), 200),
    ]


def sensor_virtual(estimativas: dict[str, int], estrito: bool = False) -> list[Step]:
    return [passo(f"vs_{fd}", "pdm_cmaps.experiments.virtual_sensor_models", "--subset", fd, est=est, check=sensor_virtual_ok(fd, estrito))
            for fd, est in estimativas.items()]


def injecoes(estimativas: dict[str, int], *args: str) -> list[Step]:
    return [passo(f"injecao_{fd}", "pdm_cmaps.experiments.fault_injection", "--subset", fd, *args, est=est, check=manifesto_t0(fd))
            for fd, est in estimativas.items()]


# ── uso cotidiano ─────────────────────────────────────────────────────────────
def plano_tudo() -> Plan:
    """pytest, pipeline dos 4 subconjuntos, todos os estudos e a sincronização das tabelas, um passo por vez (equivale a `pdm_cmaps --all`, mas retomável)."""
    from pdm_cmaps.cli.update_all import STUDIES  # a lista de estudos fica em um lugar só

    estimativas = {"monitoring": 1500, "benchmark_repeated": 6000, "healthy_window": 1500, "fault_detection": 600, "fault_model_diag13": 1800,
                   "fault_model_diag24": 2400, "fault_hard": 5400, "fault_injection": 120, "rul_predicoes": 600, "envelope_cases": 120}
    pipeline = [passo(f"pipeline_{fd}", "pdm_cmaps", "--subset", fd, est=est, check=pipeline_ok(fd))
                for fd, est in {"FD001": 1100, "FD002": 1300, "FD003": 700, "FD004": 1300}.items()]
    estudos = [passo(s.name, s.module, *s.args, est=estimativas.get(s.name, 600)) for s in STUDIES]
    return Plan("tudo", "pytest + pipeline dos 4 subconjuntos + todos os estudos + sincronização das tabelas e gráficos (horas)",
                lanes={"A": [passo("pytest", "pytest", "-q", est=60), *pipeline, *estudos]},
                final=[passo("sincronizacao", "pdm_cmaps.cli.update_all", "--so-sincronizar", est=400)])


def plano_rul_oficial() -> Plan:
    """Teste oficial de RUL: 12 modelos x 5 sementes, redes com normalização por regime e janela de 30 ciclos (3 sementes) e boosting com o mesmo tratamento."""
    of = "pdm_cmaps.experiments.benchmark_oficial_completo"
    saida = lambda nome: f"results/benchmark_oficial/{nome}"  # noqa: E731
    pronto = lambda nome: existe(f"{saida(nome)}/previsoes_oficial.csv")  # noqa: E731
    principais = [passo(f"main_{fd}_s{seed}", of, "--subsets", fd, "--part", "main", "--seeds", str(seed), "--output-dir", saida(f"main_{fd}_s{seed}"),
                        est=1500, check=pronto(f"main_{fd}_s{seed}")) for fd in SUBSETS for seed in (42, 7, 123, 1, 2)]
    redes = [passo(f"extras_{fd}_s{seed}", of, "--subsets", fd, "--part", "extras", "--seeds", str(seed), "--output-dir", saida(f"{fd}_s{seed}"),
                   est=1800, check=pronto(f"{fd}_s{seed}")) for fd in SUBSETS for seed in (42, 7, 123)]
    boosting = [passo(f"boosting_{fd}", of, "--subsets", fd, "--part", "boosting", "--output-dir", saida(f"boost_{fd}"),
                      est=400, check=pronto(f"boost_{fd}")) for fd in SUBSETS]
    return Plan("rul_oficial", "teste oficial de RUL + redes com regime e janela 30 + boosting com o mesmo tratamento, agregado (horas, 4 raias)",
                lanes={"A": [], "B": [], "C": [], "D": []}, pool=[*redes, *principais, *boosting],
                final=[passo("agregar", of, "--aggregate-only", "--output-dir", "results/benchmark_oficial", est=60,
                             check=existe("results/benchmark_oficial/resumo_oficial.csv"))])


def plano_diagnostico_fd003() -> Plan:
    """Por que o Extra Trees fica abaixo da média do regime no FD003? Mistura com a média, ET regularizado e janela saudável de 50 (FD003 e FD001)."""
    hard = "pdm_cmaps.experiments.fault_hard_study"
    base = ("Média do regime (sem ML)",)
    e25 = ("Extra Trees (padrão)", "Mistura ET 25%", "Mistura ET 50%", "Extra Trees regularizado", *base)
    e50 = ("Extra Trees (padrão)", *base)
    por_subconjunto = lambda h: Step(f"por_subconjunto_{h}", (PY, "scripts/hard_by_subset.py", "--results", f"results/diagnostico_fd003/{h}",  # noqa: E731
                                                              "--out", f"results/diagnostico_fd003/por_subconjunto_{h}.csv"), existe(f"results/diagnostico_fd003/por_subconjunto_{h}.csv"), 60)
    return Plan("diagnostico_fd003", "diagnóstico do ganho negativo do Extra Trees no FD003 (~10 min)",
                lanes={"A": [passo("h25", hard, "--subsets", "FD003", "FD001", "--H", "25", "--jobs", "6", "--estimators", *e25, "--output-dir", "results/diagnostico_fd003/H25",
                                   est=300, check=n_arquivos("results/diagnostico_fd003/H25/runs/*_series.csv", 6)),
                       passo("h50", hard, "--subsets", "FD003", "FD001", "--H", "50", "--jobs", "6", "--estimators", *e50, "--output-dir", "results/diagnostico_fd003/H50",
                             est=300, check=n_arquivos("results/diagnostico_fd003/H50/runs/*_series.csv", 6)),
                       por_subconjunto("H25"), por_subconjunto("H50")]})



# ── registro histórico das reexecuções ────────────────────────────────────────
def plano_engine_offset() -> Plan:
    """Reexecução completa dos estudos afetados por VirtualSensor(engine_offset=True)."""
    limpar_runs = ("results/injecao_deteccao/runs",)  # senão os runs antigos são reaproveitados (cache por arquivo)
    return Plan(
        "engine_offset", "reexecução completa depois de o deslocamento por motor virar padrão do sensor virtual (~1,5 h, 3 raias)",
        lanes={
            "A": [*[passo(f"pipeline_{fd}", "pdm_cmaps", "--subset", fd, est=est, check=pipeline_ok(fd)) for fd, est in {"FD001": 1100, "FD002": 1300, "FD003": 700, "FD004": 1300}.items()],
                  passo("rul_predicoes", "pdm_cmaps.experiments.rul_prediction_plots", "--jobs", "2", est=600, check=existe("results/rul_predicoes")),
                  passo("envelope_cases", "pdm_cmaps.experiments.envelope_ignored_cases", "--results", "results", est=120)],
            "B": [passo("monitoring", "pdm_cmaps.monitoring.layer", "--jobs", "2", est=1500, check=existe("results/monitoring_repeated/tables")),
                  passo("healthy_window", "pdm_cmaps.experiments.healthy_window_study", "--jobs", "2", est=1500, check=existe("results/healthy_window_study/tables/healthy_window_summary.csv")),
                  passo("fault_detection", "pdm_cmaps.experiments.fault_detection_study", "--jobs", "2", est=600, check=n_arquivos("results/injecao_deteccao/runs/*_pareado.csv", 12), clean=limpar_runs),
                  *injecoes({"FD001": 30, "FD002": 100, "FD003": 30, "FD004": 100}, "--healthy-cycle-limit", "25")],
            "C": [],  # sem passos próprios: pega tarefas do pool
        },
        pool=sensor_virtual({"FD002": 2200, "FD004": 2200, "FD001": 1300, "FD003": 1320}),
        final=passos_do_site(existe("tcc-writing/tables/benchmark_repeated_summary.csv")))


def plano_varredura_janelas() -> Plan:
    return Plan(
        "varredura_janelas", "detecção a falso alarme igualado com as janelas H = 25, 35 e 50 + sincronização e site (~20 min)",
        lanes={"A": [passo("fault_detection", "pdm_cmaps.experiments.fault_detection_study", "--windows", "25", "35", "50", "--jobs", "2",
                           est=1100, check=n_arquivos("results/injecao_deteccao/runs/*_pareado.csv", 36))]},
        final=[x for x in passos_do_site(existe("tcc-writing/tables/pareado_far_resumo.csv")) if x.name != "pytest"])


def plano_multirregime() -> Plan:
    """Refaz FD002 e FD004 com o deslocamento POR REGIME (FD001 e FD003 têm regime único e não mudam)."""
    return Plan(
        "multirregime", "refaz FD002 e FD004 com o deslocamento do motor por regime (~1,5 h, 4 raias)",
        lanes={
            "A": [passo("pipeline_FD002", "pdm_cmaps", "--subset", "FD002", est=1800, check=pipeline_ok("FD002")),
                  passo("rul_predicoes", "pdm_cmaps.experiments.rul_prediction_plots", "--jobs", "2", est=600, check=existe("results/rul_predicoes"), after=("pipeline_FD004",)),
                  passo("envelope_cases", "pdm_cmaps.experiments.envelope_ignored_cases", "--results", "results", est=30)],
            "B": [passo("pipeline_FD004", "pdm_cmaps", "--subset", "FD004", est=1300, check=pipeline_ok("FD004")),
                  passo("fault_detection", "pdm_cmaps.experiments.fault_detection_study", "--subsets", "FD002", "FD004", "--windows", "25", "35", "50", "--jobs", "2",
                        est=700, check=n_arquivos("results/injecao_deteccao/runs/*_pareado.csv", 36), clean=("results/injecao_deteccao/runs/FD002_*", "results/injecao_deteccao/runs/FD004_*"))],
            "C": [passo("monitoring", "pdm_cmaps.monitoring.layer", "--subsets", "FD002", "FD004", "--jobs", "2", est=700,
                        check=n_arquivos("results/monitoring_repeated/runs/FD00[24]_seed*", 6), clean=("results/monitoring_repeated/runs/FD002_seed*", "results/monitoring_repeated/runs/FD004_seed*")),
                  passo("healthy_window", "pdm_cmaps.experiments.healthy_window_study", "--subsets", "FD002", "FD004", "--jobs", "2", est=900,
                        check=n_arquivos("results/healthy_window_study/runs/FD00[24]_seed*.csv", 6), clean=("results/healthy_window_study/runs/FD002_seed*.csv", "results/healthy_window_study/runs/FD004_seed*.csv")),
                  *injecoes({"FD002": 100, "FD004": 150}, "--healthy-cycle-limit", "25")],
            "D": [],
        },
        pool=sensor_virtual({"FD002": 1100, "FD004": 1100}, estrito=True),
        final=passos_do_site(lambda c: len(open(c.root / "tcc-writing/tables/pareado_far_resumo.csv").readlines()) >= 961
                             and len(open(c.root / "tcc-writing/tables/magnitude_minima_detectavel.csv").readlines()) >= 271))


def plano_offset_todos() -> Plan:
    """Estudos longos com o deslocamento em todos os detectores (hoje o padrão) e o falso alarme nativo com o deslocamento fora da janela."""
    return Plan(
        "offset_todos", "falhas sorteadas, decomposição por modelo, janela saudável e estudo nativo com o deslocamento (~1 h, 3 raias)",
        lanes={
            "A": [passo("fm_diag13", "pdm_cmaps.experiments.fault_model_study", "--H", "25", "--subsets", "FD001", "FD003", "--jobs", "2", "--output-dir", "results/injecao_modelos_diag",
                        est=1800, check=n_arquivos("results/injecao_modelos_diag/runs/*_pareado.csv", 6)),
                  passo("fm_diag24", "pdm_cmaps.experiments.fault_model_study", "--H", "25", "--subsets", "FD002", "FD004", "--jobs", "2", "--output-dir", "results/injecao_modelos_diag24",
                        est=2400, check=n_arquivos("results/injecao_modelos_diag24/runs/*_pareado.csv", 6))],
            "B": [passo("hard", "pdm_cmaps.experiments.fault_hard_study", "--H", "25", "--jobs", "2", est=5400, check=n_arquivos("results/falhas_dificeis/runs/*_series.csv", 9))],
            "C": [passo("healthy_window", "pdm_cmaps.experiments.healthy_window_study", "--jobs", "2", est=1500, check=n_arquivos("results/healthy_window_study/runs/*.csv", 12)),
                  passo("nativo_fora_janela", "pdm_cmaps.experiments.native_offset_fora_janela", "--jobs", "2", est=1200, check=n_arquivos("results/native_offset_fora_janela/runs/*.csv", 12))],
        })


def plano_protocolo_principal() -> Plan:
    return Plan(
        "protocolo_principal", "camada de monitoramento (H = 50, conformal, custo) e injeção de RUL com a MEWMA corrigida do deslocamento (~25 min)",
        lanes={"A": [passo("monitoramento", "pdm_cmaps.monitoring.layer", "--jobs", "2", est=1500, check=n_arquivos("results/monitoring_repeated/runs/*/uncertainty.csv", 12))],
               "B": injecoes({"FD001": 60, "FD002": 120, "FD003": 60, "FD004": 120}, "--healthy-cycle-limit", "25")})


def plano_ic_bootstrap() -> Plan:
    """Coleta as matrizes de alarme da injeção (H = 25, 3 sementes, 4 subconjuntos) e analisa o poder por bootstrap de motores."""
    coleta = [passo(f"coleta_{cfg}", "pdm_cmaps.experiments.poder_bootstrap", "collect", "--config", cfg, "--jobs", "2", est=360,
                    check=n_arquivos(f"results/injecao_deteccao_ic/{cfg}/alarmes/*.npz", 12)) for cfg in ("principal", "so_t50")]
    analise = [passo(f"analise_{cfg}", "pdm_cmaps.experiments.poder_bootstrap", "analyse", "--config", cfg, est=60) for cfg in ("principal", "so_t50")]
    return Plan("ic_bootstrap", "matrizes de alarme da injeção + IC por bootstrap do poder dos detectores (~12 min)", lanes={"A": [*coleta, *analise]})


def plano_injecao_h25() -> Plan:
    """Injeção a falso alarme igualado (H = 25, 4 subconjuntos, 3 sementes). O protocolo anterior (só T50) sai com --no-mewma-offset."""
    return Plan("injecao_h25", "estudo de injeção a falso alarme igualado, H = 25 (~10 min)",
                lanes={"A": [passo("fault_detection", "pdm_cmaps.experiments.fault_detection_study", "--windows", "25", "--jobs", "3", est=600,
                                   check=n_arquivos("results/injecao_deteccao/runs/*_pareado.csv", 12))]})


def plano_injecao_h35_50() -> Plan:
    return Plan("injecao_h35_50", "varredura da janela saudável H = 35 e 50 na injeção (o H = 25 já existe; ~30 min)",
                lanes={"A": [passo("fault_detection", "pdm_cmaps.experiments.fault_detection_study", "--windows", "35", "50", "--jobs", "2", est=1800,
                                   check=n_arquivos("results/injecao_deteccao/runs/*_pareado.csv", 36))]})


def plano_mewma_nativo() -> Plan:
    return Plan("mewma_nativo", "MEWMA nos motores nativos (H = 50) com e sem o deslocamento do motor (~17 min)",
                lanes={"A": [passo("native_mewma_offset", "pdm_cmaps.experiments.native_mewma_offset", est=1000,
                                   check=lambda c: (c.root / "results/native_mewma_offset/resumo.csv").stat().st_size > 0)]})


def plano_deteccao_fd004() -> Plan:
    return Plan("deteccao_fd004", "estudo de injeção cega só para o FD004 (H = 25, 3 sementes), usado nas figuras 5.12 e 5.13 (~10 a 20 min)",
                lanes={"A": [passo("fault_detection_FD004", "pdm_cmaps.experiments.fault_detection_study", "--subsets", "FD004", "--windows", "25", "--jobs", "3", est=900,
                                   check=n_arquivos("results/injecao_deteccao/runs/*", 9))]})


def plano_hiperparametros() -> Plan:
    """Escolha de hiperparâmetros e da janela por validação cruzada (Optuna, só nos motores de treino) e teste oficial com o conjunto escolhido.

    10 modelos x 4 subconjuntos, um estudo por passo (retomável: o banco SQLite guarda as tentativas). Depois, `--part tuned` refaz o ajuste final
    de cada modelo na CPU com os melhores parâmetros e lê o teste oficial uma única vez (3 sementes)."""
    from pdm_cmaps.experiments.selecao_hiperparametros import BUDGET, MODELS, slug
    sel, of, saida = "pdm_cmaps.experiments.selecao_hiperparametros", "pdm_cmaps.experiments.benchmark_oficial_completo", "results/hiperparametros"
    # segundos por tentativa medidos no Colab (8 núcleos); só orientam a previsão de tempo
    por_tentativa = {"Random Forest": 400, "XGBoost": 280, "XGBoost (NASA Loss)": 300, "CatBoost": 330, "LightGBM": 40, "MLP": 160, "Ridge": 30}
    estudos = [passo(f"busca_{fd}_{slug(m)}", sel, "--subsets", fd, "--models", m, "--output-dir", saida,
                     est=int(por_tentativa.get(m, 50) * BUDGET[m]), check=existe(f"{saida}/{fd}/{slug(m)}/melhores.json"))
               for m in sorted(MODELS, key=lambda m: -por_tentativa.get(m, 50) * BUDGET[m]) for fd in SUBSETS]
    tuned = [passo(f"tuned_{fd}_s{seed}", of, "--subsets", fd, "--part", "tuned", "--seeds", str(seed), "--params-dir", saida,
                   "--output-dir", f"results/benchmark_oficial_tuned/{fd}_s{seed}", est=1500,
                   check=existe(f"results/benchmark_oficial_tuned/{fd}_s{seed}/previsoes_oficial.csv")) for fd in SUBSETS for seed in (42, 7, 123)]
    return Plan("hiperparametros", "busca de hiperparâmetros e janela (Optuna, validação cruzada por motor) + teste oficial ajustado (muitas horas; 4 raias)",
                lanes={"A": [], "B": [], "C": [], "D": []}, pool=[*estudos, *tuned],
                final=[passo("agregar", of, "--aggregate-only", "--output-dir", "results/benchmark_oficial_tuned", est=60,
                             check=existe("results/benchmark_oficial_tuned/resumo_oficial.csv"))])


PLANOS = {
    "tudo": plano_tudo,
    "rul_oficial": plano_rul_oficial,
    "hiperparametros": plano_hiperparametros,
    "diagnostico_fd003": plano_diagnostico_fd003,
    "engine_offset": plano_engine_offset,
    "varredura_janelas": plano_varredura_janelas,
    "multirregime": plano_multirregime,
    "offset_todos": plano_offset_todos,
    "protocolo_principal": plano_protocolo_principal,
    "ic_bootstrap": plano_ic_bootstrap,
    "injecao_h25": plano_injecao_h25,
    "injecao_h35_50": plano_injecao_h35_50,
    "mewma_nativo": plano_mewma_nativo,
    "deteccao_fd004": plano_deteccao_fd004,
}

In [ ]:
%%writefile projeto/scripts/rodar.py
#!/usr/bin/env python3
"""Executa um *plano* de estudos longos: passos em sequência ou em raias paralelas, retomável, com tentativas e progresso.

    python scripts/rodar.py lista                      # planos disponíveis
    python scripts/rodar.py rodar tudo                 # executa (retoma de onde parou se interrompido)
    python scripts/rodar.py rodar tudo --dry-run       # só mostra os comandos, nada é executado
    python scripts/rodar.py status tudo                # o que já terminou, o que está rodando, o que falhou

Os planos são dados em `planos.py`; este arquivo é só o motor. Cada passo tem um comando, uma verificação de sucesso
(o comando pode sair com código 0 sem ter gerado o resultado) e uma duração estimada, usada para a previsão do tempo que falta.
Para deixar rodando em segundo plano:  nohup python scripts/rodar.py rodar tudo > /dev/null 2>&1 &
"""
from __future__ import annotations

import argparse
import os
import queue
import shutil
import subprocess
import sys
import threading
import time
from dataclasses import dataclass, field
from glob import glob
from pathlib import Path
from typing import Callable

ROOT = Path(__file__).resolve().parents[1]
sys.path[:0] = [str(ROOT / "src"), str(Path(__file__).resolve().parent)]

from pdm_cmaps.progress import format_duration  # noqa: E402

# Cada processo usa poucos threads: com várias raias e joblib, OpenMP/BLAS ficam em espera ativa e a máquina
# passa de 20 de carga (medido: mais de 7x mais lento). 2 por processo mantém as raias paralelas eficientes.
THREAD_ENV = {k: "2" for k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS")} | {"OMP_WAIT_POLICY": "PASSIVE"}


@dataclass
class Context:
    """O que as verificações podem consultar."""

    root: Path
    started: float  # quando o plano começou pela 1ª vez (sobrevive a retomadas); serve para exigir arquivos mais novos que isso


@dataclass(frozen=True)
class Step:
    name: str
    cmd: tuple[str, ...]
    check: Callable[[Context], bool] = lambda ctx: True
    est: int = 60                      # segundos; só para a previsão de tempo
    clean: tuple[str, ...] = ()        # padrões (glob) apagados antes de cada tentativa, p.ex. runs antigos que seriam reaproveitados
    after: tuple[str, ...] = ()        # passos (de outras raias) que precisam ter terminado antes


@dataclass
class Plan:
    name: str
    description: str
    lanes: dict[str, list[Step]] = field(default_factory=dict)  # raias rodam em paralelo; os passos de cada uma, em sequência
    pool: list[Step] = field(default_factory=list)             # tarefas independentes: cada raia que acabar pega a próxima livre
    final: list[Step] = field(default_factory=list)            # só rodam se nada falhou
    env: dict[str, str] = field(default_factory=dict)

    def all_steps(self) -> list[Step]:
        return [s for lane in self.lanes.values() for s in lane] + self.pool + self.final


class Runner:
    def __init__(self, plan: Plan, tries: int = 3, dry_run: bool = False, redo: bool = False, root: Path = ROOT):
        self.plan, self.tries, self.dry_run, self.root = plan, tries, dry_run, root
        self.dir = root / "results" / "logs" / plan.name
        self.done_dir, self.events = self.dir / "done", self.dir / "progresso.tsv"
        self.lock, self.cond = threading.Lock(), threading.Condition()
        self.done: set[str] = set()
        self.failed: set[str] = set()
        self.actual: dict[str, float] = {}
        self.t0 = time.time()
        steps = plan.all_steps()
        names = [s.name for s in steps]
        assert len(set(names)) == len(names), f"passos repetidos em {plan.name}"
        for s in steps:
            assert all(a in names for a in s.after), f"{s.name}: 'after' aponta para passo inexistente"
        if redo and not dry_run:
            shutil.rmtree(self.done_dir, ignore_errors=True)
        if not dry_run:
            self.done_dir.mkdir(parents=True, exist_ok=True)
            self.done |= {p.name for p in self.done_dir.iterdir()}
            start = self.dir / "inicio"
            start.touch(exist_ok=True)
            self.ctx = Context(root, start.stat().st_mtime)
        else:
            self.ctx = Context(root, self.t0)

    # ── execução ──────────────────────────────────────────────────────────
    def run(self) -> int:
        if self.dry_run:
            return self._dry_run()
        pool = queue.Queue()
        for s in self.plan.pool:
            pool.put(s)
        lanes = [threading.Thread(target=self._lane, args=(name, steps, pool), name=name) for name, steps in self.plan.lanes.items()]
        self._event("-", "geral", "inicio" if not self.done else "retomada", "", f"{len(self.done)} passos já concluídos")
        for t in lanes:
            t.start()
        for t in lanes:
            t.join()
        if not self.failed:
            for s in self.plan.final:
                self._step("final", s)
        self._event("-", "geral", "fim_com_falhas" if self.failed else "fim_ok", "", ", ".join(sorted(self.failed)))
        return 1 if self.failed else 0

    def _lane(self, lane: str, steps: list[Step], pool: queue.Queue) -> None:
        for s in steps:
            self._step(lane, s)
        while True:
            try:
                s = pool.get_nowait()
            except queue.Empty:
                return
            self._step(lane, s)

    def _step(self, lane: str, s: Step) -> None:
        if s.name in self.done:
            self._event(lane, s.name, "pulada", "", "já concluída")
            return
        if s.after and not self._wait_for(s.after):
            self._finish(s.name, ok=False)
            self._event(lane, s.name, "cancelada", "", f"depende de {', '.join(s.after)}, que falhou")
            return
        env = os.environ | THREAD_ENV | self.plan.env | {"PYTHONPATH": os.pathsep.join(filter(None, [str(self.root / "src"), os.environ.get("PYTHONPATH", "")]))}
        log = self.root / "results" / "logs" / f"{self.plan.name}_{s.name}.log"
        log.parent.mkdir(parents=True, exist_ok=True)
        for attempt in range(1, self.tries + 1):
            for pattern in s.clean:
                for p in glob(str(self.root / pattern)):
                    shutil.rmtree(p) if Path(p).is_dir() else Path(p).unlink()
            t = time.time()
            self._event(lane, s.name, "inicio", str(attempt), f"est={format_duration(s.est)}")
            with open(log, "w") as f:
                rc = subprocess.run(s.cmd, cwd=self.root, env=env, stdout=f, stderr=subprocess.STDOUT, preexec_fn=lambda: os.nice(5)).returncode
            secs = time.time() - t
            if rc == 0 and s.check(self.ctx):
                self.actual[s.name] = secs
                (self.done_dir / s.name).touch()
                self._finish(s.name, ok=True)
                self._event(lane, s.name, "ok", str(attempt), format_duration(secs))
                return
            self._event(lane, s.name, "falha", str(attempt), f"codigo={rc}{'' if rc else ', verificação do resultado falhou'} ({format_duration(secs)}); ver {log.relative_to(self.root)}")
        self._finish(s.name, ok=False)
        self._event(lane, s.name, "desistiu", str(self.tries), "")

    def _finish(self, name: str, ok: bool) -> None:
        with self.cond:
            (self.done if ok else self.failed).add(name)
            self.cond.notify_all()

    def _wait_for(self, names: tuple[str, ...]) -> bool:
        with self.cond:
            self.cond.wait_for(lambda: all(n in self.done or n in self.failed for n in names))
            return all(n in self.done for n in names)

    # ── observabilidade ───────────────────────────────────────────────────
    def _eta(self) -> str:
        steps = self.plan.all_steps()
        finished = [s for s in steps if s.name in self.actual]
        pending = sum(s.est for s in steps if s.name not in self.done and s.name not in self.failed)
        if not finished:
            return "restante: estimado após o 1º passo"
        ratio = sum(self.actual[s.name] for s in finished) / max(1, sum(s.est for s in finished))  # real / estimado, dos passos concluídos
        left = pending * ratio / max(1, len(self.plan.lanes))
        return f"restante ~{format_duration(left)}" if pending else "concluído"

    def _event(self, lane: str, step: str, event: str, attempt: str, detail: str) -> None:
        total = len(self.plan.all_steps())
        with self.lock:
            line = (f"[{time.strftime('%H:%M:%S')}] [{len(self.done)}/{total}] {step} ... {event}"
                    f"{f' (tentativa {attempt})' if attempt else ''}{f' {detail}' if detail else ''}"
                    f" | decorrido {format_duration(time.time() - self.t0)} | {self._eta()}")
            print(line, flush=True)
            self.dir.mkdir(parents=True, exist_ok=True)
            with open(self.events, "a") as f:
                f.write("\t".join([time.strftime("%FT%T"), lane, step, event, attempt, detail]) + "\n")

    def _dry_run(self) -> int:
        print(f"Plano '{self.plan.name}': {self.plan.description}\n")
        for lane, steps in [*self.plan.lanes.items(), ("pool (qualquer raia livre)", self.plan.pool), ("final (só se nada falhar)", self.plan.final)]:
            if steps:
                print(f"raia {lane}:")
            for s in steps:
                dep = f"  [depois de {', '.join(s.after)}]" if s.after else ""
                clean = f"  [apaga antes: {', '.join(s.clean)}]" if s.clean else ""
                print(f"  - {s.name} (~{format_duration(s.est)}){dep}{clean}\n      {' '.join(s.cmd)}")
        total = sum(s.est for s in self.plan.all_steps())
        print(f"\n{len(self.plan.all_steps())} passos, ~{format_duration(total)} de trabalho somado (menos com raias paralelas).")
        return 0


def status(plan: Plan, root: Path = ROOT) -> int:
    """Mostra o andamento lendo só os arquivos que o Runner deixou (não cria nada)."""
    run_dir = root / "results" / "logs" / plan.name
    events, done_dir = run_dir / "progresso.tsv", run_dir / "done"
    steps = plan.all_steps()
    if not events.exists():
        print(f"Plano '{plan.name}' ainda não foi executado.")
        return 0
    lines = events.read_text().splitlines()
    last = {}
    for line in lines:
        _, _lane, step, event, *_ = line.split("\t")
        last[step] = event
    done = {p.name for p in done_dir.iterdir()} if done_dir.exists() else set()
    print(f"Plano '{plan.name}': {len(done & {s.name for s in steps})}/{len(steps)} passos concluídos")
    labels = {"inicio": "RODANDO", "falha": "falhou (vai tentar de novo)", "desistiu": "DESISTIU", "cancelada": "cancelada"}
    for s in steps:
        print(f"  {'ok' if s.name in done else labels.get(last.get(s.name, ''), 'pendente'):28} {s.name}")
    print(f"\nÚltimos eventos ({events.relative_to(root)}):")
    for line in lines[-5:]:
        print("  " + line.replace("\t", " | "))
    return 0


def main(argv: list[str] | None = None) -> int:
    from planos import PLANOS

    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    sub = ap.add_subparsers(dest="cmd", required=True)
    sub.add_parser("lista", help="lista os planos")
    for name, help_ in (("rodar", "executa um plano (retoma se interrompido)"), ("status", "mostra o andamento de um plano")):
        p = sub.add_parser(name, help=help_)
        p.add_argument("plano", choices=sorted(PLANOS))
        if name == "rodar":
            p.add_argument("--dry-run", action="store_true", help="mostra os comandos sem executar")
            p.add_argument("--refazer", action="store_true", help="ignora os passos já concluídos")
            p.add_argument("--tentativas", type=int, default=3, help="tentativas por passo (padrão: 3)")
    a = ap.parse_args(argv)
    if a.cmd == "lista":
        for name in sorted(PLANOS):
            print(f"{name:22} {PLANOS[name]().description}")
        return 0
    plan = PLANOS[a.plano]()
    if a.cmd == "status":
        return status(plan)
    return Runner(plan, tries=a.tentativas, dry_run=a.dry_run, redo=a.refazer).run()


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile projeto/scripts/taxa_fallback_deslocamento.py
"""Taxa de recuo do deslocamento do motor (o_{e,r} = 0): fração de pares (motor, regime) com menos de três ciclos nos primeiros H ciclos.

Uso (da raiz): python scripts/taxa_fallback_deslocamento.py [--out tcc-writing/tables/taxa_fallback_deslocamento.csv]
Responde: com H = 25 e seis regimes, quantos motores caem no recuo (o_{e,r} = 0) em algum regime, e o quanto isso muda com H = 50.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from pdm_cmaps.data.loader import CMapssLoader  # noqa: E402
from pdm_cmaps.detection.regime import RegimeDecoupler  # noqa: E402


def taxa(subset: str, H: int, min_rows: int = 3) -> dict:
    frame = CMapssLoader().load(subset)
    n = 6 if subset in ("FD002", "FD004") else 1
    f = RegimeDecoupler(n_clusters=n).fit(frame).transform(frame) if n > 1 else frame.assign(regime_id=0)
    first = f[f["cycle"] <= H]
    cnt = first.groupby(["unit_id", "regime_id"]).size().unstack(fill_value=0)
    pairs = cnt.size
    low = int((cnt < min_rows).sum().sum())
    return {"Subset": subset, "H": H, "motores": int(cnt.shape[0]), "regimes": int(cnt.shape[1]), "pares_motor_regime": int(pairs),
            "pares_com_menos_de_3_ciclos_pct": 100 * low / pairs, "motores_com_algum_recuo_pct": 100 * float((cnt < min_rows).any(axis=1).mean()),
            "ciclos_por_regime_media": float(cnt.to_numpy().mean())}


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--out", type=Path, default=ROOT / "tcc-writing" / "tables" / "taxa_fallback_deslocamento.csv")
    a = ap.parse_args()
    d = pd.DataFrame([taxa(s, H) for s in ("FD002", "FD004") for H in (25, 50)])
    a.out.parent.mkdir(parents=True, exist_ok=True)
    d.round(2).to_csv(a.out, index=False)
    print(d.round(2).to_string(index=False))

In [ ]:
%%writefile projeto/scripts/teste_bloco_injecao.py
"""Teste de robustez da série saudável da injeção cega (revisão da banca, item 3).

O protocolo principal (fault_detection_study.bootstrap_series) gera a série saudável por bootstrap i.i.d.
dos ciclos saudáveis do próprio motor (RUL >= 125). Este script compara-o com um bootstrap em blocos
contíguos de 25 ciclos, que preserva a ordem local (autocorrelação e deriva lenta). Todo o resto fica igual:
detectores, limiar s* a falso alarme igualado (0,2 por 100 ciclos), falhas injetadas, H = 25.

Modos:
  diag  SUBSET SEED            autocorrelação e desvio do T50 padronizado: ciclos saudáveis reais,
                               reamostragem i.i.d. e reamostragem em blocos
  run   METODO SUBSET SEED     injeção cega completa; METODO = iid (protocolo atual), bloco
                               (série inteira em blocos) ou bloco_cal (calibração i.i.d., resto em blocos)
  resumir                      junta as duas execuções em tabelas (poder, falso alarme, s*), pareadas por semente

Uso, a partir da raiz do projeto (no Colab, com o mesmo layout):
  python scripts/teste_bloco_injecao.py diag FD004 42
  python scripts/teste_bloco_injecao.py run bloco FD004 42
  python scripts/teste_bloco_injecao.py resumir
"""

from __future__ import annotations

import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

from pdm_cmaps.config import DEFAULT_RESULTS_DIR  # noqa: E402
from pdm_cmaps.data.loader import CMapssLoader  # noqa: E402
from pdm_cmaps.domain.schema import DEFAULT_RUL_CAP  # noqa: E402
from pdm_cmaps.experiments import fault_detection_study as fds  # noqa: E402
from pdm_cmaps.experiments.fault_injection import INJECT_SENSORS, load_world, standardize  # noqa: E402

H = 25
BLOCK = 25
LAGS = (1, 5, 25)
SERIES_LEN = 300  # mesmo HORIZON_CTRL do protocolo
RESULTS = DEFAULT_RESULTS_DIR / "teste_bloco"
TABLES = ROOT / "tcc-writing" / "tables"
_ORIG_BOOTSTRAP = fds.bootstrap_series


def bloco_bootstrap(engine: pd.DataFrame, horizon: int, rng: np.random.Generator) -> pd.DataFrame | None:
    """Bootstrap em blocos contíguos de BLOCK ciclos sobre os ciclos saudáveis (RUL >= cap), em ordem."""
    pool = engine.loc[engine["rul_true"] >= DEFAULT_RUL_CAP].sort_values("cycle")
    n = len(pool)
    if n < fds.MIN_POOL:
        return None
    if n < BLOCK:
        return _ORIG_BOOTSTRAP(engine, horizon, rng)
    n_blocks = int(np.ceil(horizon / BLOCK))
    starts = rng.integers(0, n - BLOCK + 1, n_blocks)
    idx = np.concatenate([np.arange(s, s + BLOCK) for s in starts])[:horizon]
    base = pool.iloc[idx].reset_index(drop=True).copy()
    base["cycle"] = np.arange(1, horizon + 1)
    return base


def bloco_cal_bootstrap(engine: pd.DataFrame, horizon: int, rng: np.random.Generator) -> pd.DataFrame | None:
    """Variante de controle: a janela de calibração (primeiros H ciclos) continua i.i.d. sobre todo o histórico
    saudável, como no protocolo; só o restante da série usa blocos contíguos de BLOCK ciclos."""
    pool = engine.loc[engine["rul_true"] >= DEFAULT_RUL_CAP].sort_values("cycle")
    n = len(pool)
    if n < fds.MIN_POOL:
        return None
    if n < BLOCK or horizon <= H:
        return _ORIG_BOOTSTRAP(engine, horizon, rng)
    cal = pool.iloc[rng.integers(0, n, H)]
    rest_len = horizon - H
    n_blocks = int(np.ceil(rest_len / BLOCK))
    starts = rng.integers(0, n - BLOCK + 1, n_blocks)
    idx = np.concatenate([np.arange(s, s + BLOCK) for s in starts])[:rest_len]
    base = pd.concat([cal, pool.iloc[idx]], ignore_index=True)
    base["cycle"] = np.arange(1, horizon + 1)
    return base


def _acf(x: np.ndarray, k: int) -> float:
    x = x - x.mean()
    denom = float(np.sum(x * x))
    if denom == 0.0 or len(x) <= k:
        return np.nan
    return float(np.sum(x[:-k] * x[k:]) / denom)


def diag(subset: str, seed: int) -> pd.DataFrame:
    world = load_world(subset, seed, H, loader=CMapssLoader())
    rng = np.random.default_rng(seed)
    t50 = INJECT_SENSORS.index("T50")
    engines = {int(u): g.sort_values("cycle") for u, g in world.test.groupby("unit_id")}
    rows = []
    for fonte, sampler in (("real", None), ("iid", _ORIG_BOOTSTRAP), ("bloco", bloco_bootstrap)):
        acfs = {k: [] for k in LAGS}
        sds, n_motores = [], 0
        for _, eng in engines.items():
            if sampler is None:
                frame = eng.loc[eng["rul_true"] >= DEFAULT_RUL_CAP].sort_values("cycle")
                if len(frame) < 60:
                    continue
            else:
                frame = sampler(eng, SERIES_LEN, rng)
                if frame is None:
                    continue
            frame = world.regimes.transform(frame)
            z = standardize(world, frame)[:, t50]
            n_motores += 1
            sds.append(float(np.std(z)))
            for k in LAGS:
                acfs[k].append(_acf(z, k))
        for k in LAGS:
            rows.append({"subset": subset, "seed": seed, "fonte": fonte, "lag": k,
                         "acf_medio": float(np.nanmean(acfs[k])), "n_motores": n_motores})
        rows.append({"subset": subset, "seed": seed, "fonte": fonte, "lag": 0,
                     "acf_medio": np.nan, "n_motores": n_motores, "desvio_padrao_z": float(np.mean(sds))})
    return pd.DataFrame(rows)


def run(metodo: str, subset: str, seed: int) -> None:
    if metodo == "bloco":
        fds.bootstrap_series = bloco_bootstrap
    elif metodo == "bloco_cal":
        fds.bootstrap_series = bloco_cal_bootstrap
    elif metodo != "iid":
        raise SystemExit(f"metodo desconhecido: {metodo}")
    out = RESULTS / metodo
    fds.run_one(subset, seed, H, out)
    print(f"FIM {metodo} {subset} seed {seed}", flush=True)


def resumir() -> None:
    frames = []
    for metodo in ("iid", "bloco", "bloco_cal"):
        out = RESULTS / metodo
        if not (out / "runs").exists():
            continue
        par = fds._load(out, "pareado")
        par = par.assign(metodo=metodo)
        frames.append(par)
    par = pd.concat(frames, ignore_index=True)
    keys = ["metodo", "Subset", "Detector", "kind", "magnitude"]
    g = par.groupby(keys)[["s_star", "far_ctrl_per100", "power_pct"]]
    resumo = g.mean().add_suffix("_media").join(g.std().add_suffix("_dp")).reset_index()
    resumo["n_sementes"] = g.size().to_numpy()
    TABLES.mkdir(parents=True, exist_ok=True)
    resumo.to_csv(TABLES / "teste_bloco_injecao_resumo.csv", index=False)
    par.to_csv(TABLES / "teste_bloco_injecao_por_semente.csv", index=False)
    print(resumo.to_string(), flush=True)


if __name__ == "__main__":
    modo = sys.argv[1]
    if modo == "diag":
        df = diag(sys.argv[2], int(sys.argv[3]))
        RESULTS.mkdir(parents=True, exist_ok=True)
        df.to_csv(RESULTS / f"diag_{sys.argv[2]}_seed{sys.argv[3]}.csv", index=False)
        print(df.to_string(), flush=True)
    elif modo == "run":
        run(sys.argv[2], sys.argv[3], int(sys.argv[4]))
    elif modo == "resumir":
        resumir()
    else:
        raise SystemExit(__doc__)

### 2.4 Tabelas da monografia

Monta `tcc-writing/tables/` a partir de `results/`.

1 arquivos.

In [ ]:
%%writefile projeto/tcc-writing/montar_tabelas.py
"""Monta tcc-writing/tables/ a partir de results/ (cópias das tabelas dos estudos e tabelas derivadas das previsões e das execuções).

Uso (da raiz): python tcc-writing/montar_tabelas.py [--diag results/diagnostico_fd003] [--boost results/benchmark_oficial_boosting]
Cópias: as de `pdm_cmaps.cli.update_all.TABLE_COPIES`, mais as do protocolo só-T50, do bootstrap, dos nativos e das falhas sorteadas.
Derivadas: testes pareados por motor do RUL oficial, boosting por regime, largura do conformal na zona crítica, diagnóstico do FD003 e, da busca de
hiperparâmetros, `hiperparametros_escolhidos`, `hiperparametros_janela_por_modelo`, `rul_ajustado_vs_padrao` e `rul_ajustado_metricas_estendidas`.
"""
from __future__ import annotations

import argparse
import glob
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from pdm_cmaps.cli.update_all import TABLE_COPIES  # noqa: E402

RES, TAB = ROOT / "results", ROOT / "tcc-writing" / "tables"
ZONA_CRITICA = 50.0


def _nasa(d):
    return np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)


def copiar() -> list[str]:
    faltando = []
    pares = {**TABLE_COPIES,
             "contrastes_ic_principal.csv": "injecao_deteccao_ic/principal/contrastes_ic.csv", "poder_ic_principal.csv": "injecao_deteccao_ic/principal/poder_ic.csv",
             "contrastes_ic_so_t50.csv": "injecao_deteccao_ic/so_t50/contrastes_ic.csv", "poder_ic_so_t50.csv": "injecao_deteccao_ic/so_t50/poder_ic.csv",
             "contrastes_ic_extras.csv": "injecao_deteccao_ic/extras/contrastes_ic.csv", "poder_ic_extras.csv": "injecao_deteccao_ic/extras/poder_ic.csv",
             "pareado_far_resumo_offset_so_t50.csv": "injecao_deteccao_offset_so_t50/tabelas/pareado_far_resumo.csv",
             "magnitude_minima_detectavel_offset_so_t50.csv": "injecao_deteccao_offset_so_t50/tabelas/magnitude_minima_detectavel.csv",
             "regra_padrao_s1_offset_so_t50.csv": "injecao_deteccao_offset_so_t50/tabelas/regra_padrao_s1.csv",
             "native_mewma_offset_resumo.csv": "native_mewma_offset/resumo.csv", "native_offset_fora_janela_resumo.csv": "native_offset_fora_janela/resumo.csv",
             "falhas_dificeis_por_subconjunto.csv": "falhas_dificeis/por_subconjunto.csv",
             "rul_oficial_resumo.csv": "benchmark_oficial/resumo_oficial.csv", "rul_oficial_metricas_por_semente.csv": "benchmark_oficial/metricas_por_semente.csv",
             "rul_ajustado_resumo.csv": "benchmark_oficial_tuned/resumo_oficial.csv", "rul_ajustado_metricas_por_semente.csv": "benchmark_oficial_tuned/metricas_por_semente.csv"}
    derivadas = {"margin_check_point_thresholds_raw.csv"}  # montada em `limiares_previsao_pontual`
    for dest, origem in pares.items():
        if dest in derivadas:
            continue
        src = RES / origem
        if src.exists():
            shutil.copy2(src, TAB / dest)
        else:
            faltando.append(f"{dest} <- {origem}")
    diag = [RES / d / "tabelas" / "modelos_pareado_resumo.csv" for d in ("injecao_modelos_diag", "injecao_modelos_diag24")]
    if all(p.exists() for p in diag):
        from pdm_cmaps.experiments.fault_model_study import DIAGNOSTIC_ESTIMATORS  # os estimadores só do diagnóstico do FD003 não entram aqui

        d = pd.concat([pd.read_csv(p) for p in diag])
        d[~d.Estimator.isin(DIAGNOSTIC_ESTIMATORS)].to_csv(TAB / "decomposicao_regime_modelo.csv", index=False)
    else:
        faltando.append("decomposicao_regime_modelo.csv")
    return faltando


def testes_por_motor() -> pd.DataFrame:
    """Erro de cada motor = média sobre as sementes do erro absoluto; diferença (rede - XGBoost com atributos temporais) por motor; Wilcoxon pareado."""
    p = pd.concat([pd.read_csv(f) for f in glob.glob(str(RES / "benchmark_oficial/*/previsoes_oficial.csv"))]).drop_duplicates(["Subset", "Seed", "Model", "Config", "unit_id"])
    p["ae"] = (p.pred - p.rul_cap).abs()
    e = p.groupby(["Subset", "Model", "Config", "unit_id"]).ae.mean().reset_index()
    ref = e[(e.Model == "XGBoost (+temporal)") & (e.Config == "global_w15")].set_index(["Subset", "unit_id"]).ae
    linhas = []
    for (s, m, c), g in e[e.Model.str.startswith(("PyTorch", "MiniRocket"))].groupby(["Subset", "Model", "Config"]):
        a = g.set_index("unit_id").ae.sort_index()
        r = ref.loc[s].sort_index()
        d = a - r
        linhas.append(dict(Subset=s, Model=m, Config=c, n_motores=len(a), RMSE_rede_menos_xgb_mediana_erro=float(d.median()), p_wilcoxon=float(wilcoxon(a, r).pvalue)))
    return pd.DataFrame(linhas)


def boosting_regime(boost: Path) -> pd.DataFrame:
    linhas = []
    for f in sorted(boost.glob("*/previsoes_oficial.csv")):
        for (s, m, c), g in pd.read_csv(f).groupby(["Subset", "Model", "Config"]):
            d, dr = g.pred - g.rul_cap, g.pred - g.rul_raw
            linhas.append(dict(Subset=s, Model=m, Config=c, RMSE_cap=float(np.sqrt((d ** 2).mean())), RMSE_raw=float(np.sqrt((dr ** 2).mean())), NASA_cap=float(_nasa(d).mean())))
    return pd.DataFrame(linhas)


def limiares_previsao_pontual() -> pd.DataFrame:
    """Custo relativo ao oráculo (razão 10) da previsão pontual (limiar c de 10 a 50) e do limite inferior (c de 10 a 20), com a persistência de dois ciclos."""
    from pdm_cmaps.monitoring.alerts import CRITICAL, NORMAL, debounce, maintenance_cost

    linhas = []
    for f in sorted(glob.glob(str(RES / "monitoring_repeated/runs/*/timelines.csv.gz"))):
        t = pd.read_csv(f)
        s, seed = t.Subset.iloc[0], int(t.Seed.iloc[0])
        t = t[t.Policy == "critical<=10"].sort_values(["unit_id", "cycle"]).reset_index(drop=True)  # a previsão e o limite inferior não dependem da política
        for strat, col, cs in (("point", "RUL_pred", (10, 15, 20, 25, 30, 35, 40, 50)), ("interval", "RUL_lower", (10, 15, 20))):
            for c in cs:
                tl = t[["unit_id", "cycle"]].copy()
                tl["level_x"] = np.concatenate([debounce(np.where(g[col].to_numpy() <= c, CRITICAL, NORMAL), 2) for _, g in t.groupby("unit_id", sort=False)])
                for L in (0, 5, 10):
                    r = maintenance_cost(tl, "x", 10, L)
                    linhas.append(dict(S=s, seed=seed, strat=strat, c=c, L=L, rel=r["Relative_to_oracle"], fail=float(r["Failures"])))
    return pd.DataFrame(linhas)


def largura_conformal() -> pd.DataFrame:
    d = pd.concat([pd.read_csv(f) for f in glob.glob(str(RES / "monitoring_repeated/runs/*/uncertainty_variants.csv"))])
    m = d[d.Method.str.contains("padrão")]
    g = m.groupby("Subset")[["mean_width", "mean_width_critical_zone"]].agg(["mean", "std"])
    g.columns = ["_".join(c) for c in g.columns]
    g["Runs"] = m.groupby("Subset").size()
    return g.reset_index().rename(columns={"mean_width_mean": "mean_width_mean", "mean_width_critical_zone_mean": "mean_width_critical_zone_mean"})


def diagnostico(diag: Path) -> pd.DataFrame:
    partes = []
    for h in (25, 50):
        f = diag / f"por_subconjunto_H{h}.csv"
        partes.append(pd.read_csv(f).assign(H=h))
    d = pd.concat(partes)
    return d[(d.Mode == "exogenous") & (d.Chart == "MEWMA")]


def hiperparametros_escolhidos() -> pd.DataFrame:
    """Uma linha por (subconjunto, modelo): janela, representação, RMSE e erro-padrão da validação cruzada, tentativas e os parâmetros do modelo."""
    import json

    linhas = []
    for f in sorted(glob.glob(str(RES / "hiperparametros/FD00*/*/melhores.json"))):
        d = json.loads(Path(f).read_text())
        p = d["params"]
        linhas.append(dict(Subset=d["subset"], Model=d["model"], Janela=p["window"], Temporal=p["temporal"], Representacao=p["representation"],
                           CV_RMSE=d["cv_rmse"], CV_EP=d["cv_se"], Melhor_RMSE_medio=d["melhor_rmse_medio"], Tentativas_completas=d["n_tentativas_completas"],
                           Tentativas_total=d["n_tentativas_total"], Dobras=d["n_splits"], Parametros=json.dumps(p["hyper"], ensure_ascii=False, sort_keys=True)))
    return pd.DataFrame(linhas).sort_values(["Subset", "Model"])


def ajustado_vs_padrao(boost: Path) -> pd.DataFrame:
    """RMSE (truncado e sem truncar) e pontuação NASA média do teste oficial: ajustado (3 sementes) contra o padrão (janela 15, global) e contra a melhor
    configuração anterior (escolhida à mão: janela 30, regime), por modelo. A melhor anterior sai do RMSE truncado entre todas as configurações já rodadas."""
    aj = pd.read_csv(RES / "benchmark_oficial_tuned/resumo_oficial.csv")
    ant = pd.read_csv(RES / "benchmark_oficial/resumo_oficial.csv")
    pool = ant[["Subset", "Model", "Config", "RMSE_cap_mean", "RMSE_raw_mean", "Mean_NASA_cap_mean"]].rename(
        columns={"RMSE_cap_mean": "rmse", "RMSE_raw_mean": "rmse_raw", "Mean_NASA_cap_mean": "nasa"})
    bo = boosting_regime(boost)
    if len(bo):
        bo = bo.assign(Model=bo.Model.str.replace(" (+temporal)", "", regex=False)).rename(columns={"RMSE_cap": "rmse", "RMSE_raw": "rmse_raw", "NASA_cap": "nasa"})
        pool = pd.concat([pool, bo[["Subset", "Model", "Config", "rmse", "rmse_raw", "nasa"]]])
    pad = pool[pool.Config == "global_w15"].set_index(["Subset", "Model"])
    melhor = pool.sort_values("rmse").groupby(["Subset", "Model"]).head(1).set_index(["Subset", "Model"])
    linhas = []
    for _, r in aj.iterrows():
        k = (r.Subset, r.Model)
        linhas.append(dict(Subset=r.Subset, Model=r.Model,
                           Ajustado_RMSE=r.RMSE_cap_mean, Ajustado_RMSE_dp=r.RMSE_cap_std, Ajustado_RMSE_sem_truncar=r.RMSE_raw_mean, Ajustado_NASA_medio=r.Mean_NASA_cap_mean,
                           Padrao_RMSE=pad.rmse.get(k, np.nan), Padrao_RMSE_sem_truncar=pad.rmse_raw.get(k, np.nan), Padrao_NASA_medio=pad.nasa.get(k, np.nan),
                           Melhor_anterior_config=melhor.Config.get(k, ""), Melhor_anterior_RMSE=melhor.rmse.get(k, np.nan), Melhor_anterior_NASA_medio=melhor.nasa.get(k, np.nan),
                           Ganho_sobre_padrao=pad.rmse.get(k, np.nan) - r.RMSE_cap_mean, Ganho_sobre_melhor_anterior=melhor.rmse.get(k, np.nan) - r.RMSE_cap_mean))
    return pd.DataFrame(linhas).sort_values(["Subset", "Ajustado_RMSE"])


def metricas_estendidas_ajustado() -> pd.DataFrame:
    """Métricas além de RMSE/NASA no teste oficial ajustado (média de 3 sementes): viés, erro mediano e P90, acerto em +-10 e +-20 ciclos e, para
    c = 10, 15 e 20 ciclos, precisão, revocação, F1 e PR-AUC do estado crítico (positivo = RUL verdadeira <= c; previsto crítico = previsão <= c)."""
    t = pd.read_csv(RES / "benchmark_oficial_tuned/resumo_oficial.csv")
    cols = ["Subset", "Model", "Bias_cap_mean", "Median_AE_cap_mean", "P90_AE_cap_mean", "Hit_within_10_cap_mean", "Hit_within_20_cap_mean",
            *[f"{m}_c{c}_cap_mean" for c in (10, 15, 20) for m in ("Precision", "Recall", "F1", "PR_AUC")]]
    return t[cols].rename(columns=lambda c: c.replace("_cap_mean", "")).sort_values(["Subset", "Model"])


def janela_por_modelo(h: pd.DataFrame) -> pd.DataFrame:
    """Janela escolhida por modelo nos quatro subconjuntos e quantas vezes bateu no limite do espaço de busca (60 ciclos)."""
    g = h.pivot(index="Model", columns="Subset", values="Janela")
    g["No_limite_60"] = (g[["FD001", "FD002", "FD003", "FD004"]] == 60).sum(axis=1)
    return g.reset_index()


def sensor_virtual_metricas(subsets=("FD001", "FD002", "FD003", "FD004")) -> tuple[pd.DataFrame, pd.DataFrame]:
    """MAE, NRMSE (RMSE dividido pelo desvio dentro do regime) e ganho sobre a média do regime (= R² dentro do regime) de cada sensor virtual e modelo,
    recalculados do `trace.csv.gz` de `virtual_sensor_models` (sem refazer os modelos). Devolve (por sensor e modelo, resumo por modelo = mediana dos sensores)."""
    from pdm_cmaps.detection.virtual_sensor_metrics import virtual_sensor_r2
    from pdm_cmaps.domain.schema import DEFAULT_HEALTHY_CYCLE_LIMIT

    partes = []
    for fd in subsets:
        f = RES / "virtual_sensor_models" / fd / "trace.csv.gz"
        if f.exists():
            partes.append(virtual_sensor_r2(pd.read_csv(f), DEFAULT_HEALTHY_CYCLE_LIMIT).assign(Subset=fd))
    por = pd.concat(partes, ignore_index=True)[["Subset", "sensor", "model", "mae_holdout", "nrmse_within_regime_holdout", "r2_holdout", "r2_holdout_within_regime"]]
    por = por.rename(columns={"r2_holdout_within_regime": "ganho_sobre_media_do_regime"})
    resumo = por.groupby(["Subset", "model"])[["nrmse_within_regime_holdout", "ganho_sobre_media_do_regime", "r2_holdout"]].median().reset_index()
    resumo["n_sensores"] = por.groupby(["Subset", "model"]).size().to_numpy()
    return por, resumo


def alertas_f1() -> pd.DataFrame:
    """Precisão, revocação e F1 de cada estratégia de criticidade, linha a linha nos motores de teste (positivo = nível verdadeiro >= ALERTA; alerta =
    nível previsto >= ALERTA), por política e subconjunto, média das sementes. Recalculado das linhas do tempo (`timelines.csv.gz`)."""
    from pdm_cmaps.monitoring.alerts import CriticalityPolicy, evaluate_strategies

    linhas = []
    for f in sorted(glob.glob(str(RES / "monitoring_repeated/runs/*/timelines.csv.gz"))):
        t = pd.read_csv(f)
        for pol, g in t.groupby("Policy"):
            e = evaluate_strategies(g.drop(columns=["Policy", "Subset", "Seed"]), CriticalityPolicy())
            linhas.append(e[["Strategy", "Alert_precision", "Alert_recall", "Alert_F1"]].assign(Subset=t.Subset.iloc[0], Seed=int(t.Seed.iloc[0]), Policy=pol))
    d = pd.concat(linhas, ignore_index=True)
    return d.groupby(["Subset", "Policy", "Strategy"])[["Alert_precision", "Alert_recall", "Alert_F1"]].mean().reset_index()


def detectores_comparacao() -> pd.DataFrame:
    """Poder e atraso, a falso alarme igualado (0,2 por 100 ciclos, H = 25, início em t0 = 35), da MEWMA com regime e do T50 com regime e sem portão
    (protocolo principal) contra a EWMA clássica de Roberts sobre o resíduo com sinal e o Isolation Forest sobre os 13 resíduos, nas mesmas séries cegas."""
    from pdm_cmaps.experiments.fault_detection_study import DET_MV, DET_T50_NOGATE, EXTRA_DETECTORS

    principal = pd.read_csv(TAB / "pareado_far_resumo.csv")
    extras = pd.read_csv(RES / "injecao_deteccao_extras" / "tabelas" / "pareado_far_resumo.csv")
    d = pd.concat([principal[principal.Detector.isin([DET_MV, DET_T50_NOGATE])], extras[extras.Detector.isin(EXTRA_DETECTORS)]], ignore_index=True)
    d = d[(d.H == 25) & np.isclose(d.budget, 0.2) & (d.t0 == 35) & d.kind.isin(["degrau", "rampa"]) & d.magnitude.round(2).isin([0.05, 0.10, 0.25])]
    cols = ["Subset", "Detector", "kind", "magnitude", "power_pct_mean", "power_pct_std", "delay_med_mean", "far_ctrl_per100_mean", "n_seeds"]
    return d[cols].sort_values(["Subset", "kind", "magnitude", "Detector"]).reset_index(drop=True)


def sem_minirocket() -> None:
    """O MiniRocket saiu da monografia: remove de todas as tabelas as linhas que o citam (os resultados em results/ continuam completos)."""
    for f in sorted(TAB.glob("*.csv")):
        linhas = f.read_text(encoding="utf-8").split("\n")
        manter = [ln for ln in linhas if "ocket" not in ln]
        if len(manter) != len(linhas):
            f.write_text("\n".join(manter), encoding="utf-8")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--diag", type=Path, default=RES / "diagnostico_fd003")
    ap.add_argument("--boost", type=Path, default=RES / "benchmark_oficial_boosting")
    ap.add_argument("--so-metricas-extras", action="store_true", help="só os sensores virtuais (MAE, NRMSE) e o F1 dos alertas, recalculados dos traces e linhas do tempo (Colab)")
    a = ap.parse_args()
    if a.so_metricas_extras:
        TAB.mkdir(parents=True, exist_ok=True)
        por, resumo = sensor_virtual_metricas()
        por.to_csv(TAB / "sensor_virtual_metricas_por_sensor.csv", index=False)
        resumo.to_csv(TAB / "sensor_virtual_metricas_resumo.csv", index=False)
        alertas_f1().to_csv(TAB / "alertas_precisao_f1.csv", index=False)
        return
    TAB.mkdir(parents=True, exist_ok=True)
    faltando = copiar()
    testes_por_motor().to_csv(TAB / "rul_oficial_testes_por_motor.csv", index=False)
    boosting_regime(a.boost).to_csv(TAB / "rul_oficial_boosting_regime.csv", index=False)
    limiares_previsao_pontual().sort_values(["S", "seed", "strat", "c", "L"]).to_csv(TAB / "margin_check_point_thresholds_raw.csv", index=False)
    largura_conformal().to_csv(TAB / "conformal_largura_zona_critica.csv", index=False)
    diagnostico(a.diag).to_csv(TAB / "diagnostico_fd003_ruido_estimacao.csv", index=False)
    h = hiperparametros_escolhidos()
    h.to_csv(TAB / "hiperparametros_escolhidos.csv", index=False)
    janela_por_modelo(h).to_csv(TAB / "hiperparametros_janela_por_modelo.csv", index=False)
    ajustado_vs_padrao(a.boost).to_csv(TAB / "rul_ajustado_vs_padrao.csv", index=False)
    metricas_estendidas_ajustado().to_csv(TAB / "rul_ajustado_metricas_estendidas.csv", index=False)
    if (RES / "injecao_deteccao_extras" / "tabelas" / "pareado_far_resumo.csv").exists():
        detectores_comparacao().to_csv(TAB / "detectores_comparacao.csv", index=False)
    sem_minirocket()
    print(f"{len(list(TAB.glob('*.csv')))} tabelas em {TAB}")
    if faltando:
        print("sem origem:", *faltando, sep="\n  ")


if __name__ == "__main__":
    main()

### 2.5 Gráficos da monografia

Figuras F* dos capítulos 4 e 5 e diagramas explicativos de física (Brayton e atmosfera).

10 arquivos.

In [ ]:
%%writefile projeto/tcc-writing/make_figuras_consistencia.py
"""Figuras da injeção cega e da criticidade refeitas a partir das tabelas de `tcc-writing/tables/`, sem rodar estudos.

F23: decomposição do ganho no degrau de 0,10 (sensores brutos, média do regime, Extra Trees), com o deslocamento do motor em todos os detectores.
F10a/F10b: a ablação do regime em duas figuras legíveis no papel (falhas coerentes; demais formas), quatro subconjuntos.
F10: a mesma ablação em uma só grade (versão do estudo, usada nos slides e painéis), com os quatro subconjuntos.
F08: o compromisso de criticidade com as quatro estratégias do texto (a versão anterior não desenhava a MEWMA).
Uso (da raiz): .venv/bin/python tcc-writing/make_figuras_consistencia.py
"""
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from pdm_cmaps.experiments import fault_detection_study as fds  # noqa: E402
from pdm_cmaps.reporting import viz_style as vs  # noqa: E402

HERE = Path(__file__).resolve().parent
FIG, TAB = HERE / "figures", HERE / "tables"
SUBSETS = ("FD001", "FD002", "FD003", "FD004")
POLICY = "critical<=15"
# (rótulo como na tabela de custo, marcador, cor, vazado)
STRATEGIES = {
    "point": ("Previsão pontual", "o", vs.SERIES[0], False),
    "interval": ("Limite inferior", "s", vs.SERIES[1], False),
    "interval+anomaly": ("Limite inferior + T50", "s", vs.SERIES[2], True),
    "interval+anomaly_mv": ("Limite inferior + MEWMA", "D", "#52514e", False),
}


H, BUDGET = 25, 0.2
DESC = {"FD001": "1 regime\n1 modo de falha", "FD002": "6 regimes\n1 modo de falha", "FD003": "1 regime\n2 modos de falha", "FD004": "6 regimes\n2 modos de falha"}
# mesmos nomes, cores e traços das curvas operacionais (make_figuras_deteccao.py): tracejado = sem regime
DET = {fds.DET_MV: ("MEWMA com regime", vs.SERIES[1], "-", "o"), fds.DET_RAW_MV: ("MEWMA sem regime", vs.SERIES[1], "--", "s"),
       fds.DET_T50_NOGATE: ("T50 com regime, sem portão", vs.SERIES[2], "-", "^"), fds.DET_T50: ("T50 com regime, com portão", vs.SERIES[0], "-", "v"),
       fds.DET_RAW_T50: ("T50 sem regime", vs.SERIES[0], "--", "D")}
SIG = "magnitude (fração da assinatura de fim de vida)"
KINDS = {"degrau": ("degrau", SIG), "rampa": ("rampa de 40 ciclos", SIG), "variancia": ("variância aumentada", "magnitude (σ)"),
         "vies_sensor": ("viés em um sensor (T50)", "magnitude (σ)"), "um_regime": ("falha restrita a um regime", SIG)}


def _virgula(v: float) -> str:
    return f"{v:g}".replace(".", ",")


def _grade_ablacao(summ: pd.DataFrame, kinds: tuple[str, ...], titulo: str, nome: str) -> None:
    vs.apply_style()
    base = summ[(summ["H"] == H) & (summ["budget"] == BUDGET) & (summ["t0"] == H + 10)]
    fig, axes = plt.subplots(len(kinds), len(SUBSETS), figsize=(8.4, 2.45 * len(kinds) + 1.2), sharey=True, squeeze=False)
    for i, kind in enumerate(kinds):
        nome_kind, xlabel = KINDS[kind]
        for j, sub in enumerate(SUBSETS):
            ax = axes[i, j]
            d0 = base[(base["Subset"] == sub) & (base["kind"] == kind)]
            if i == 0:
                ax.set_title(sub, fontsize=10.5, loc="left", fontweight="bold")
            if d0.empty:  # falha restrita a um regime não existe onde há um regime só
                ax.text(0.5, 0.5, "não se aplica\n(regime único)", transform=ax.transAxes, ha="center", va="center", fontsize=9, color=vs.MUTED)
                ax.set_xticks([]), ax.set_ylim(-3, 103), ax.set_yticks([0, 25, 50, 75, 100]), ax.tick_params(labelsize=9)
                vs.style_axes(ax)
                continue
            for det, (lab, col, ls, mk) in DET.items():
                d = d0[d0["Detector"] == det].sort_values("magnitude")
                ax.errorbar(d["magnitude"], d["power_pct_mean"], yerr=d["power_pct_std"].fillna(0), color=col, ls=ls, marker=mk, ms=4.5,
                            lw=1.8, capsize=2, elinewidth=1, label=lab)
            mags = sorted(d0["magnitude"].unique())
            ticks = mags[1:] if len(mags) == 4 else mags  # os dois menores se sobrepõem no eixo; o ponto continua desenhado
            ax.set_xticks(ticks), ax.set_xticklabels([_virgula(m) for m in ticks])
            ax.set_ylim(-3, 103), ax.set_yticks([0, 25, 50, 75, 100])
            ax.tick_params(labelsize=9)
            vs.style_axes(ax)
        axes[i, 0].set_ylabel(f"{nome_kind}\npoder (%)", fontsize=9.5)
        next(ax for ax in axes[i] if ax.lines).set_xlabel(xlabel, fontsize=8.5, loc="left")  # primeiro painel com dados
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=9.5, frameon=False, handlelength=3.2)
    fig.suptitle(titulo, fontsize=10.5, fontweight="bold", x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0.11 if len(kinds) < 3 else 0.075, 1, 0.94 if len(kinds) < 3 else 0.955))
    fig.savefig(FIG / nome, dpi=200)
    plt.close(fig)


def ablacao() -> None:
    summ = pd.read_csv(TAB / "pareado_far_resumo.csv")
    fds.fig_ablation(summ, FIG / "F10_ablacao_regime_injecao.png", subsets=SUBSETS)
    sub = "\nmesmo falso alarme (0,2 por 100 ciclos); tracejado = sem regime; barras = desvio entre sementes"
    _grade_ablacao(summ, ("degrau", "rampa"), f"Falhas coerentes: poder por magnitude{sub}", "F10a_ablacao_coerentes.png")
    _grade_ablacao(summ, ("variancia", "vies_sensor", "um_regime"), f"Demais formas de falha: poder por magnitude{sub}", "F10b_ablacao_outras_formas.png")


def decomposicao() -> None:
    """Poder da MEWMA no degrau de 0,10 (o de 0,25 satura em 100 %) com três versões do sinal, com o deslocamento do motor em todos os detectores."""
    par = pd.read_csv(TAB / "pareado_far_resumo.csv")
    dec = pd.read_csv(TAB / "decomposicao_regime_modelo.csv")
    sel = lambda d: d[(d["H"] == H) & (d["budget"] == BUDGET) & (d["kind"] == "degrau") & (d["magnitude"] == 0.10)]
    par, dec = sel(par[par["t0"] == H + 10]), sel(dec)
    series = [("Sensores brutos (sem regime)", par[par["Detector"] == fds.DET_RAW_MV], "#b9b7ae"),
              ("Média do regime", dec[(dec["Estimator"] == "Média do regime (sem ML)") & dec["Detector"].str.startswith("MEWMA")], vs.SERIES[0]),
              ("Extra Trees (sensor virtual)", dec[(dec["Estimator"] == "Extra Trees (padrão)") & dec["Detector"].str.startswith("MEWMA")], vs.SERIES[1])]
    vs.apply_style()
    fig, ax = plt.subplots(figsize=(8.4, 4.3))
    w = 0.26
    for k, (lab, d, col) in enumerate(series):
        d = d.set_index("Subset").loc[list(SUBSETS)]
        x = [i + (k - 1) * w for i in range(len(SUBSETS))]
        ax.bar(x, d["power_pct_mean"], width=w * 0.92, color=col, label=lab, yerr=d["power_pct_std"], error_kw=dict(ecolor=vs.INK_2, elinewidth=1, capsize=2.5))
        for xi, v, e in zip(x, d["power_pct_mean"], d["power_pct_std"]):
            ax.text(xi, v + e + 1.5, f"{v:.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold", color=vs.INK)
    ax.set_xticks(range(len(SUBSETS))), ax.set_xticklabels([f"{s}\n{DESC[s]}" for s in SUBSETS], fontsize=9.5)
    ax.set_ylim(0, 118), ax.set_yticks([0, 20, 40, 60, 80, 100]), ax.set_ylabel("poder de detecção (%)", fontsize=10)
    ax.tick_params(axis="y", labelsize=9.5)
    vs.style_axes(ax)
    ax.legend(loc="upper center", ncol=3, fontsize=9.5, frameon=False, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle("De onde vem o ganho: poder da MEWMA de 13 sinais em um degrau de 0,10\nnível de cada motor corrigido; mesmo falso alarme (0,2/100 ciclos); barras = desvio entre sementes",
                 fontsize=10.2, fontweight="bold", x=0.015, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    fig.savefig(FIG / "F23_decomposicao_ganho.png", dpi=200)
    plt.close(fig)


def criticidade() -> None:
    d = pd.read_csv(TAB / "monitoring_repeated_summary.csv")
    d = d[d["Policy"] == POLICY]
    vs.apply_style()
    # grade 2 x 2: na largura da página, quatro painéis lado a lado ficavam pequenos demais para ler
    fig, axes = plt.subplots(2, 2, figsize=(9.5, 7.4), sharex=True, sharey=True)
    axes = axes.flatten()
    for ax, sub in zip(axes, SUBSETS):
        vs.style_axes(ax, grid="both")
        for key in ("interval+anomaly", "interval+anomaly_mv", "point", "interval"):  # o vazado primeiro, para não esconder o cheio
            _, mk, col, hollow = STRATEGIES[key]
            r = d[(d["Subset"] == sub) & (d["Strategy"] == key)].iloc[0]
            ax.errorbar(100 * r["False_alarm_rate_normal_mean"], 100 * r["Missed_critical_rate_mean"],
                        xerr=100 * r["False_alarm_rate_normal_std"], yerr=100 * r["Missed_critical_rate_std"],
                        fmt=mk, color=col, ecolor=col, elinewidth=1.4, capsize=3, markersize=14 if hollow else 9, clip_on=False,
                        markerfacecolor=vs.SURFACE if hollow else col, markeredgecolor=col if hollow else vs.SURFACE, markeredgewidth=2.0)
        ax.set_title(sub, fontsize=12, loc="left")
        ax.tick_params(labelsize=10.5)
    for ax in axes[2:]:
        ax.set_xlabel("falsos alertas em ciclos normais (%)", fontsize=11)
    for ax in axes[::2]:
        ax.set_ylabel("críticos não alertados (%)", fontsize=11)
    xtop = 100 * (d["False_alarm_rate_normal_mean"] + d["False_alarm_rate_normal_std"]).max()
    ytop = 100 * (d["Missed_critical_rate_mean"] + d["Missed_critical_rate_std"]).max()
    axes[0].set_xlim(-0.06 * xtop, 1.08 * xtop), axes[0].set_ylim(-0.08 * ytop, 1.08 * ytop)
    handles = [Line2D([], [], ls="", marker=mk, markersize=11 if hollow else 9, markerfacecolor=vs.SURFACE if hollow else col,
                      markeredgecolor=col if hollow else vs.SURFACE, markeredgewidth=2.0, label=lab) for lab, mk, col, hollow in STRATEGIES.values()]
    fig.legend(handles=handles, loc="lower center", ncol=2, fontsize=11, frameon=False)
    fig.suptitle("Compromisso das estratégias: mais perto do canto inferior esquerdo é melhor\n(média ± desvio entre sementes; crítico ≤ 15 ciclos)",
                 fontsize=12.5, fontweight="bold", x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0.09, 1, 0.92))
    fig.savefig(FIG / "F08_criticidade_tradeoff.png", dpi=170)
    plt.close(fig)


RES = ROOT / "results"
NOMES_MODELOS = {"PyTorch CNN": "CNN", "PyTorch RNN (LSTM)": "LSTM", "PyTorch Transformer": "Transformer", "XGBoost (NASA Loss)": "XGBoost (perda NASA)",
                 "Support Vector Machine (SVM)": "SVM linear", "MiniRocket + RidgeCV": "MiniRocket + Ridge", "Random Forest": "Random forest"}


def vocabulario() -> None:
    """F06 e F07 pelas funções do pacote, só com os rótulos no vocabulário do texto (nada muda em `src/`)."""
    from pdm_cmaps.reporting import monitoring_figures as mf

    vs.apply_style()
    tabs = RES / "maintenance_guide_fd004" / "tables"
    boot = pd.read_csv(tabs / "thesis_bootstrap_ci.csv")
    boot["Model"] = boot["Model"].replace(NOMES_MODELOS)
    boot = boot.nlargest(9, "R2_Observed")  # só os nove de maior R² (0,82 a 0,85); a legenda da monografia cita a faixa dos demais
    mf.plot_model_ci(boot, FIG / "F06_comparacao_modelos_ic_fd004.png")

    timeline = pd.read_csv(tabs / "alert_timeline.csv")
    eol = timeline.groupby("unit_id")["cycle"].max().sort_values()
    picks = list(dict.fromkeys(int(eol.index[int(q * (len(eol) - 1))]) for q in (0.1, 0.4, 0.7, 0.95)))  # mesma escolha de `monitoring.layer`
    salvar = mf.save_figure

    def _com_motor(fig, path):
        for ax in fig.axes:
            for loc in ("left", "center", "right"):
                if ax.get_title(loc):
                    t = ax.get_title(loc).replace("Ativo", "Motor")
                    if "Motor " in t and "FD004" not in t:
                        t = f"{t} (FD004)"
                    ax.set_title(t, loc=loc, pad=11)  # logo acima da faixa de criticidade
        if fig._suptitle is not None:
            fig._suptitle.set_text("RUL com incerteza e nível de criticidade ao longo da vida (FD004, motores 178, 204, 224 e 174 do holdout)")
        return salvar(fig, path)

    mf.save_figure = _com_motor
    try:
        mf.plot_rul_trajectories(timeline, picks, FIG / "F07_trajetorias_intervalo_conformal.png")
    finally:
        mf.save_figure = salvar


MODOS = {"exogenous": ("modo exógeno", vs.SERIES[0], "-"), "cross_sensor": ("modo entre sensores", vs.SERIES[1], "--")}
CASOS = [("MEWMA", "processo", "MEWMA\nfalhas de processo"), ("MEWMA", "sensor", "MEWMA\nfalhas de sensor"), ("sensor afetado", "sensor", "Carta do sensor afetado\nfalhas de sensor")]
ESTIMADORES = ("Extra Trees (padrão)", "LightGBM", "MLP", "Ridge")


def ganho_ml() -> None:
    """Ganho de poder de cada modelo sobre a média do regime, com IC 95 % (tabela `diferenca_vs_media_regime.csv`)."""
    d = pd.read_csv(TAB / "diferenca_vs_media_regime.csv")
    vs.apply_style()
    fig, axes = plt.subplots(1, len(CASOS), figsize=(8.4, 4.0), sharex=True, sharey=True)
    for ax, (chart, fam, titulo) in zip(axes, CASOS):
        for k, (modo, (lab, col, _)) in enumerate(MODOS.items()):
            g = d[(d["Chart"] == chart) & (d["family"] == fam) & (d["Mode"] == modo)].set_index("Estimator").loc[list(ESTIMADORES)]
            y = [i + (k - 0.5) * 0.38 for i in range(len(ESTIMADORES))]
            ax.barh(y, g["delta_pp"], height=0.34, color=col, label=lab, xerr=[g["delta_pp"] - g["ci_lo"], g["ci_hi"] - g["delta_pp"]],
                    error_kw=dict(ecolor=vs.INK_2, elinewidth=1, capsize=2.5))
        ax.axvline(0, color=vs.INK, lw=1)
        ax.set_yticks(range(len(ESTIMADORES))), ax.set_yticklabels([e.replace(" (padrão)", "") for e in ESTIMADORES], fontsize=10)
        ax.invert_yaxis(), ax.tick_params(axis="x", labelsize=9.5)
        ax.set_title(titulo, fontsize=10, loc="left", fontweight="bold")
        vs.style_axes(ax, grid="x")
    fig.supxlabel("ganho de poder sobre a média do regime (pontos percentuais)", fontsize=9.5, y=0.075)
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=2, fontsize=10, frameon=False)
    fig.suptitle("Ganho de detecção de cada modelo sobre a média do regime nos quatro subconjuntos juntos\nbarras finas = IC 95 % por bootstrap sobre motores", fontsize=10.5, fontweight="bold", x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0.11, 1, 0.91))
    fig.savefig(FIG / "F18_ganho_ml_vs_media_regime.png", dpi=200)
    plt.close(fig)


def poder_sorteadas() -> None:
    """Poder por magnitude (F19a) e por forma (F19b) nas falhas sorteadas: cor = modelo, traço/marcador = modo."""
    d = pd.read_csv(TAB / "poder_por_magnitude.csv")
    cores = dict(zip(ESTIMADORES, (vs.SERIES[1], vs.SERIES[0], vs.SERIES[2], "#8a5fbf")))
    base = "Média do regime (sem ML)"
    vs.apply_style()
    fig, axes = plt.subplots(1, len(CASOS), figsize=(8.4, 4.4), sharey=True)
    for ax, (chart, fam, titulo) in zip(axes, CASOS):
        g0 = d[(d["Chart"] == chart) & (d["family"] == fam)]
        for est in ESTIMADORES:
            for modo, (_, _, ls) in MODOS.items():
                g = g0[(g0["Estimator"] == est) & (g0["Mode"] == modo)].sort_values("mag_mid")
                ax.plot(g["mag_mid"], g["power_pct"], color=cores[est], ls=ls, lw=1.7, marker="o", ms=3.5)
        g = g0[g0["Estimator"] == base].sort_values("mag_mid")
        ax.plot(g["mag_mid"], g["power_pct"], color=vs.INK, lw=2.4, marker="s", ms=4.5, zorder=5)
        ax.set_xscale("log"), ax.set_ylim(-3, 103), ax.set_yticks([0, 25, 50, 75, 100])
        ticks = sorted(g["mag_mid"].unique())
        ax.set_xticks(ticks), ax.set_xticklabels([f"{v:.2f}".replace(".", ",") if v < 1 else f"{v:.1f}".replace(".", ",") for v in ticks])
        ax.minorticks_off(), ax.tick_params(labelsize=9)
        ax.set_xlabel("magnitude (fração da assinatura)" if fam == "processo" else "magnitude (σ)", fontsize=9, loc="left")
        ax.set_title(titulo, fontsize=10, loc="left", fontweight="bold")
        vs.style_axes(ax)
    axes[0].set_ylabel("poder (%)", fontsize=10)
    handles = [Line2D([], [], color=vs.INK, lw=2.4, marker="s", ms=4.5, label="média do regime (sem modelo)")]
    handles += [Line2D([], [], color=c, lw=2, label=e.replace(" (padrão)", "")) for e, c in cores.items()]
    handles += [Line2D([], [], color=vs.INK_2, lw=1.7, ls=ls, label=lab) for lab, _, ls in MODOS.values()]
    fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=9.5, frameon=False, handlelength=2.8)
    fig.suptitle("Poder de detecção nas falhas sorteadas, por magnitude\nfalso alarme de até 0,2 por 100 ciclos; média dos quatro subconjuntos", fontsize=10.5, fontweight="bold", x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0.13, 1, 0.92))
    fig.savefig(FIG / "F19a_poder_sorteadas_magnitude.png", dpi=200)
    plt.close(fig)

    # F19b: estratificado por forma da falha (`poder_por_forma.csv`)
    df = pd.read_csv(TAB / "poder_por_forma.csv")
    shapes_order = ["degrau", "rampa", "saturante", "intermitente", "variancia"]
    shape_labels = {"degrau": "degrau", "rampa": "rampa", "saturante": "saturante", "intermitente": "intermitente", "variancia": "variância"}
    fig, axes = plt.subplots(1, len(CASOS), figsize=(8.8, 4.4), sharex=True)
    for idx_ax, (ax, (chart, fam, titulo)) in enumerate(zip(axes, CASOS)):
        g0 = df[(df["Chart"] == chart) & (df["family"] == fam)]
        shs = [s for s in shapes_order if s in set(g0["shape"])]
        for yi, sh in enumerate(shs):
            gs = g0[g0["shape"] == sh]
            gb = gs[gs["Estimator"] == base]
            if not gb.empty:
                ax.scatter([gb["power_pct"].iloc[0]], [yi], color=vs.INK, marker="s", s=52, zorder=6)
            offsets = np.linspace(-0.22, 0.22, len(ESTIMADORES))
            for k, est in enumerate(ESTIMADORES):
                for modo, (_, _, ls) in MODOS.items():
                    gm = gs[(gs["Estimator"] == est) & (gs["Mode"] == modo)]
                    if gm.empty:
                        continue
                    mk = "o" if modo == "exogenous" else "^"
                    fc = cores[est] if modo == "exogenous" else vs.SURFACE
                    ax.scatter([gm["power_pct"].iloc[0]], [yi + offsets[k]], color=cores[est], facecolors=fc, edgecolors=cores[est],
                               marker=mk, s=42, lw=1.4, zorder=4)
        ax.set_yticks(range(len(shs)))
        ax.set_yticklabels([shape_labels[s] for s in shs], fontsize=9.5)
        ax.invert_yaxis()
        ax.set_xlim(-3, 103)
        ax.set_xticks([0, 25, 50, 75, 100])
        ax.set_title(titulo, fontsize=10, loc="left", fontweight="bold")
        vs.style_axes(ax, grid="x")
    fig.supxlabel("poder de detecção (%)", fontsize=9.5, y=0.145)
    handles_f = [Line2D([], [], color=vs.INK, ls="", marker="s", ms=6.5, label="média do regime (sem modelo)")]
    handles_f += [Line2D([], [], color=c, ls="", marker="o", ms=6.5, label=e.replace(" (padrão)", "")) for e, c in cores.items()]
    handles_f += [
        Line2D([], [], color=vs.INK_2, ls="", marker="o", ms=6.5, label="modo exógeno"),
        Line2D([], [], color=vs.INK_2, ls="", marker="^", markerfacecolor=vs.SURFACE, ms=6.5, label="modo entre sensores"),
    ]
    fig.legend(handles=handles_f, loc="lower center", ncol=4, fontsize=9, frameon=False)
    fig.suptitle("Poder de detecção nas falhas sorteadas, por forma\nfalso alarme de até 0,2 por 100 ciclos; média dos quatro subconjuntos", fontsize=10.5, fontweight="bold", x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0.165, 1, 0.91))
    fig.savefig(FIG / "F19b_poder_sorteadas_forma.png", dpi=200)
    plt.close(fig)


def conformal_ajustado() -> None:
    """F27: compara Mondrian sobre LightGBM (dente de serra) vs XGBoost ajustado + CQR + histerese."""
    from matplotlib.patches import Patch, Rectangle

    traj_path = RES / "estudos_complementares" / "exp_a_trajetorias_seed42.csv"
    if not traj_path.exists():
        print("F27 ignorada (rode Exp. A antes)")
        return
    d = pd.read_csv(traj_path)
    # Escolhe 2 motores representativos com forte efeito dente-de-serra no baseline
    ref = d[(d["Model"] == "LightGBM (ref.)") & (d["Conformal_method"] == "Mondrian 3 tercis")].copy()
    stats = []
    for u, g in ref.groupby("unit_id"):
        g = g.sort_values("cycle")
        jumps = int(((np.abs(np.diff(g["RUL_lower"])) >= 15) & (np.abs(np.diff(g["RUL_pred"])) <= 5)).sum())
        revs = int((np.diff(g["level_interval"]) < 0).sum())
        stats.append((u, jumps, revs, int(g["cycle"].max())))
    st_df = pd.DataFrame(stats, columns=["unit_id", "jumps", "revs", "eol"]).sort_values(["jumps", "revs"], ascending=False)
    picks = [int(u) for u in st_df["unit_id"].head(2)]
    if 204 in set(ref["unit_id"]) and 204 not in picks:
        picks[1] = 204

    panels = [
        ("LightGBM (ref.)", "Mondrian 3 tercis", "LightGBM (ref.) + Mondrian (3 tercis)"),
        ("XGBoost (ajustado)", "CQR + histerese", "XGBoost (ajustado) + CQR + histerese"),
    ]
    vs.apply_style()
    fig, axes = plt.subplots(len(picks), 2, figsize=(11.2, 6.2), sharex=False, sharey=True)
    ymax, strip_lo, strip_hi = 146, 131, 143
    for row_i, uid in enumerate(picks):
        for col_j, (mod, conf, label) in enumerate(panels):
            ax = axes[row_i, col_j]
            vs.style_axes(ax, "y")
            eng = d[(d["unit_id"] == uid) & (d["Model"] == mod) & (d["Conformal_method"] == conf)].sort_values("cycle")
            c = eng["cycle"].to_numpy()
            span = max(int(c.max() - c.min()), 1)
            ax.fill_between(c, eng["RUL_lower"], eng["RUL_upper"], color=vs.SERIES[1], alpha=0.18, lw=0)
            ax.plot(c, eng["RUL_true"], color=vs.INK, lw=vs.LINE_W)
            ax.plot(c, eng["RUL_pred"], color=vs.SERIES[1], lw=vs.LINE_W)
            lvls = eng["level_interval"].to_numpy(dtype=int)
            # Runs
            cuts = np.flatnonzero(lvls[1:] != lvls[:-1]) + 1
            for s_i, e_i in zip(np.r_[0, cuts], np.r_[cuts, len(lvls)] - 1):
                lvl = int(lvls[s_i])
                ax.add_patch(Rectangle((c[s_i] - 0.5, strip_lo), c[e_i] - c[s_i] + 1, strip_hi - strip_lo,
                                       facecolor=vs.STATUS[lvl], edgecolor=vs.SURFACE, linewidth=1.3, clip_on=False))
                if (c[e_i] - c[s_i] + 1) / span >= 0.16:
                    ax.text((c[s_i] + c[e_i]) / 2, (strip_lo + strip_hi) / 2, vs.LEVEL_NAMES_PT[lvl], ha="center", va="center",
                            color="#ffffff" if lvl in (0, 3) else vs.INK, fontsize=7.5)
            ax.set_ylim(0, ymax)
            ax.set_yticks([0, 25, 50, 75, 100, 125])
            ax.set_title(f"Motor {uid} · {label}", fontsize=10, loc="left", pad=9)
            if row_i == len(picks) - 1:
                ax.set_xlabel("Ciclo")
        axes[row_i, 0].set_ylabel("RUL (ciclos)")
    handles = [
        Line2D([], [], color=vs.INK, lw=vs.LINE_W), Line2D([], [], color=vs.SERIES[1], lw=vs.LINE_W),
        Patch(color=vs.SERIES[1], alpha=0.30), *[Patch(color=vs.STATUS[i]) for i in range(4)],
    ]
    labels_leg = ["RUL real", "RUL prevista", "Intervalo de 90%", *[f"Nível {n}" for n in vs.LEVEL_NAMES_PT]]
    fig.legend(handles, labels_leg, loc="lower center", ncol=4, frameon=False, fontsize=9)
    fig.suptitle("Eliminação do dente de serra e estreitamento do intervalo: Mondrian vs. XGBoost ajustado + CQR + histerese (FD004, holdout)",
                 x=0.01, ha="left", fontsize=11, fontweight="semibold")
    fig.tight_layout(rect=(0, 0.08, 1, 0.94))
    fig.savefig(FIG / "F27_conformal_ajustado_cqr.png", dpi=170)
    plt.close(fig)


if __name__ == "__main__":
    vocabulario()
    ganho_ml()
    poder_sorteadas()
    ablacao()
    decomposicao()
    criticidade()
    conformal_ajustado()
    print("ok")

In [ ]:
%%writefile projeto/tcc-writing/make_figuras_deteccao.py
"""Figuras 5.12 e 5.13 refeitas com destaques e rótulos diretos (curvas operacionais e detecção acumulada), fundo branco.

Lê as saídas do estudo `fault_detection_study` (results/injecao_deteccao/runs), que precisam ser regeradas para o FD004:
    .venv/bin/python -m pdm_cmaps.experiments.fault_detection_study --subsets FD004 --windows 25 --jobs 3
Uso (da raiz): .venv/bin/python tcc-writing/make_figuras_deteccao.py [--runs results/injecao_deteccao]
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from pdm_cmaps.experiments import fault_detection_study as fds  # noqa: E402
from pdm_cmaps.reporting import viz_style as vs  # noqa: E402

FIG = Path(__file__).resolve().parent / "figures"
H, SUBSET, BUDGET = 25, "FD004", 0.2
SHORT = {fds.DET_MV: "MEWMA com regime", fds.DET_RAW_MV: "MEWMA sem regime", fds.DET_T50_NOGATE: "T50 com regime, sem portão",
         fds.DET_RAW_T50: "T50 sem regime", fds.DET_T50: "T50 com regime, com portão"}
STY = {fds.DET_MV: (vs.SERIES[1], "-", "o"), fds.DET_RAW_MV: (vs.SERIES[1], "--", "s"), fds.DET_T50_NOGATE: (vs.SERIES[2], "-", "^"),
       fds.DET_RAW_T50: (vs.SERIES[0], "--", "D"), fds.DET_T50: (vs.SERIES[0], "-", "v")}
CASES = [("degrau", 0.10), ("rampa", 0.10), ("degrau", 0.25)]  # com o deslocamento em todos os detectores o degrau de 0,25 satura na MEWMA: os casos informativos são os de 0,10


def operating(cur: pd.DataFrame, par: pd.DataFrame) -> None:
    vs.apply_style()
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 4.9), sharey=True)
    for ax, (kind, mag) in zip(axes, CASES):
        for det in fds.DETECTORS:
            d = cur[(cur["Subset"] == SUBSET) & (cur["H"] == H) & (cur["kind"] == kind) & np.isclose(cur["magnitude"], mag) & (cur["t0"] == H + 10) & (cur["Detector"] == det)]
            m = d.groupby("s")[["far_ctrl_per100", "power_pct"]].mean().sort_index()
            col, ls, mk = STY[det]
            ax.plot(m["far_ctrl_per100"].clip(lower=1e-3), m["power_pct"], color=col, ls=ls, lw=2.2, label=SHORT[det])
            row = par[(par["Subset"] == SUBSET) & (par["H"] == H) & (par["kind"] == kind) & np.isclose(par["magnitude"], mag) & (par["Detector"] == det) & (par["budget"] == BUDGET) & (par["t0"] == H + 10)]
            if len(row):
                pw = float(row["power_pct"].mean())
                ax.scatter([BUDGET], [pw], s=70, color=col, marker=mk, zorder=6, edgecolor=vs.SURFACE, linewidth=1.5)
        ax.axvline(BUDGET, color=vs.INK_2, lw=1.4, ls=":")
        ax.set_xscale("log"), ax.set_xlim(2e-3, 10), ax.set_ylim(0, 102)
        ax.set_title(f"{kind} de {f"{mag:.2f}".replace('.', ',')} da assinatura", fontsize=12.5, loc="left")
        ax.set_xlabel("falsos alarmes por 100 ciclos (controles)", fontsize=11)
        ax.tick_params(labelsize=10.5)
        vs.style_axes(ax, grid="both")
        if ax is axes[0]:  # a seta de "melhor" só no primeiro painel: nos outros cruzaria as curvas
            ax.annotate("melhor", xy=(0.0028, 97), xytext=(0.0034, 66), fontsize=11.5, color="#12805a", fontweight="bold", ha="left", arrowprops=dict(arrowstyle="-|>", color=vs.STATUS[0], lw=2))
    axes[0].set_ylabel("poder de detecção (%)", fontsize=11)
    # a legenda mostra também o marcador de cada detector (o ponto grande no orçamento de falso alarme)
    h = [plt.Line2D([], [], color=STY[d][0], ls=STY[d][1], lw=2.2, marker=STY[d][2], ms=8, markeredgecolor=vs.SURFACE) for d in fds.DETECTORS]
    l = [SHORT[d] for d in fds.DETECTORS]
    fig.legend(h, l, loc="lower center", ncol=5, fontsize=10.5, frameon=False)
    fig.suptitle("Curvas operacionais: cada ponto grande é o detector operado no orçamento de falso alarme", fontsize=13, fontweight="bold", x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0.08, 1, 0.93))
    fig.savefig(FIG / "F11_curvas_operacionais.png", dpi=170, bbox_inches="tight", pad_inches=0.2)
    plt.close(fig)


def cumulative(dl: pd.DataFrame) -> None:
    vs.apply_style()
    cases = CASES  # mesmos casos e mesma ordem das curvas operacionais
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 4.9), sharey=True)
    grid = np.arange(0, fds.POST_DET + 1)
    for ax, (kind, mag) in zip(axes, cases):
        for det in fds.DETECTORS:
            d = dl[(dl["Subset"] == SUBSET) & (dl["H"] == H) & (dl["kind"] == kind) & np.isclose(dl["magnitude"], mag) & (dl["t0"] == H + 10) & (dl["budget"] == BUDGET) & (dl["Detector"] == det)]
            if d.empty:
                continue
            x = d["delay"].to_numpy()
            col, ls, _ = STY[det]
            curve = np.array([100 * np.mean(x <= g) for g in grid])
            ax.step(grid, curve, where="post", color=col, ls=ls, lw=2.2, label=SHORT[det])
            hit = np.flatnonzero(curve >= 50)
            if len(hit):
                ax.scatter([grid[hit[0]]], [50], s=50, color=col, zorder=6, edgecolor=vs.SURFACE, linewidth=1.5)
        ax.axhline(50, color=vs.INK_2, lw=1, ls=":")
        # o rótulo fica à direita, na faixa livre acima da linha de 50%; só na rampa de 0,10 a curva da MEWMA com regime sobe por ali, então vai à esquerda
        if not (kind.startswith('ramp') and mag < 0.2):
            ax.text(fds.POST_DET - 1, 51.5, "50% detectadas", ha="right", va="bottom", fontsize=10.5, color=vs.INK_2)
        else:
            ax.text(1, 52.5, "50% detectadas", ha="left", va="bottom", fontsize=10.5, color=vs.INK_2)
        ax.set_title(f"{kind} de {f"{mag:.2f}".replace('.', ',')} da assinatura", fontsize=12.5, loc="left")
        ax.set_xlabel("ciclos após o início da falha (t0)", fontsize=11), ax.set_ylim(0, 102)
        ax.tick_params(labelsize=10.5)
        vs.style_axes(ax, grid="both")
    axes[0].set_ylabel("séries detectadas até esse atraso (%)", fontsize=11)
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=5, fontsize=10.5, frameon=False)
    fig.suptitle("Detecção acumulada: quanto mais rápida a curva sobe, menor o atraso (0,2 falsos alarmes por 100 ciclos)", fontsize=13, fontweight="bold", x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0.08, 1, 0.93))
    fig.savefig(FIG / "F12_deteccao_acumulada.png", dpi=170, bbox_inches="tight", pad_inches=0.2)
    plt.close(fig)


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--runs", type=Path, default=ROOT / "results" / "injecao_deteccao")
    a = ap.parse_args()
    cur, par, dl = fds._load(a.runs, "curvas"), fds._load(a.runs, "pareado"), fds._load(a.runs, "atrasos")
    operating(cur, par)
    cumulative(dl)
    print("ok")

In [ ]:
%%writefile projeto/tcc-writing/make_figuras_injecao.py
"""Exemplos visuais da injeção cega (Seção 4.9): série saudável reamostrada e formas de falha.

  F21a  de um motor real a uma série saudável (bootstrap dos ciclos com RUL >= 125) e ao seu gêmeo com falha
  F21b  as cinco formas de falha, como o deslocamento somado ao T50 (em σ saudável)
Usa as funções de `fault_injection` e `fault_detection_study` só para leitura (sem alterá-las).
Uso (da raiz): .venv/bin/python tcc-writing/make_figuras_injecao.py
"""
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from pdm_cmaps.experiments import fault_detection_study as fds  # noqa: E402
from pdm_cmaps.experiments import fault_injection as fi  # noqa: E402
from pdm_cmaps.reporting import viz_style as vs  # noqa: E402

FIG = Path(__file__).resolve().parent / "figures"
H, T0, SUBSET, SEED = 25, 35, "FD004", 42


def main() -> None:
    vs.apply_style()
    world = fi.load_world(SUBSET, SEED, H)
    j = fi.INJECT_SENSORS.index("T50")
    test = world.test
    unit = int(sorted(test["unit_id"].unique())[3])
    eng = test[test.unit_id == unit].sort_values("cycle")
    rng = np.random.default_rng(SEED)
    base = fds.bootstrap_series(eng, 120, rng)
    ctrl = base.copy()
    dz = fi.delta_z(world, base, "degrau", 0.25, T0, rng)
    faulty = fi.inject(world, base, dz)

    # ---- F21a ----
    fig, axes = plt.subplots(3, 1, figsize=(10, 7.6))
    ax = axes[0]
    ax.scatter(eng["cycle"], eng["T50"], s=8, color=vs.MUTED)
    healthy = eng[eng["rul_true"] >= 125]
    ax.scatter(healthy["cycle"], healthy["T50"], s=10, color=vs.SERIES[0], label="ciclos saudáveis (RUL real ≥ 125)")
    ax.set_title(f"1. Motor {unit} do {SUBSET} (holdout): T50 bruto; em azul, a vida saudável", fontsize=10.5, loc="left")
    ax.set_xlabel("ciclo original"), ax.legend(loc="lower left", fontsize=9)
    ax = axes[1]
    ax.scatter(ctrl["cycle"], ctrl["T50"], s=8, color=vs.SERIES[0])
    ax.set_title(f"2. Série saudável (motor {unit} do {SUBSET}): ciclos saudáveis sorteados com reposição e renumerados (controle)", fontsize=10.5, loc="left")
    ax.set_xlabel("ciclo da série")
    ax = axes[2]
    diff = faulty["T50"].to_numpy() - ctrl["T50"].to_numpy()
    ax.plot(ctrl["cycle"], diff, color=vs.STATUS[3], lw=2)
    ax.axvline(T0, color=vs.INK_2, lw=1.4, ls="--")
    ax.axvspan(0, H, color=vs.GRID, alpha=0.6)
    ax.text(T0 + 1, diff.max() * 0.55, f"início da falha t0 = {T0}\n(oculto dos detectores)", va="center", fontsize=9.5, color=vs.INK_2, fontweight="bold")
    ax.text(H / 2, diff.max() * 0.9, "calibração", ha="center", fontsize=9, color=vs.INK_2)
    ax.set_title(f"3. Gêmeo com falha menos controle (motor {unit} do {SUBSET}): zero antes de t0; depois, deslocamento pequeno", fontsize=10.5, loc="left")
    ax.set_xlabel("ciclo da série")
    axes[2].set_ylabel("diferença no T50 (°R)")
    for a in axes[:2]:
        a.set_ylabel("T50 (°R)")
    for a in axes:
        vs.style_axes(a)
    axes[0].sharey = None
    fig.tight_layout()
    fig.savefig(FIG / "F21a_injecao_serie_saudavel.png", dpi=170)
    plt.close(fig)

    # ---- F21b ----
    cyc = base["cycle"].to_numpy()
    shapes = [("degrau", 0.25, "Degrau\n(25% da assinatura)"), ("rampa", 0.25, "Rampa de 40 ciclos\n(25% da assinatura)"), ("vies_sensor", 2.0, "Viés em um sensor (T50)\n(2σ)"),
              ("variancia", 1.0, "Variância aumentada\n(1σ, todos os sensores)"), ("um_regime", 0.50, "Rampa restrita a um regime\n(50% da assinatura)")]
    fig, axes = plt.subplots(1, 5, figsize=(11.5, 3.6), sharey=True)
    for ax, (kind, mag, title) in zip(axes, shapes):
        d = fi.delta_z(world, base, kind, mag, T0, np.random.default_rng(1))[:, j]
        ax.plot(cyc, d, color=vs.STATUS[3] if kind != "variancia" else vs.SERIES[1], lw=1.6)
        ax.axvline(T0, color=vs.INK_2, lw=1, ls="--")
        ax.set_title(title, fontsize=10.5, loc="left"), ax.set_xlabel("ciclo", fontsize=10.5)
        ax.tick_params(labelsize=9.5)
        vs.style_axes(ax, grid="y")
    axes[0].set_ylabel("deslocamento somado\nao T50 (σ saudável)", fontsize=10.5)
    fig.suptitle("As cinco formas de falha injetadas (tracejado = início t0)", fontsize=12, fontweight="bold", x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    fig.savefig(FIG / "F21b_injecao_formas_de_falha.png", dpi=170)
    plt.close(fig)
    print("ok", unit)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile projeto/tcc-writing/make_figuras_janela.py
"""Figuras da janela saudável (Seção 5.4), divididas por pergunta e com fundo branco, a partir das tabelas versionadas.

  F09a  o custo de uma janela maior: ciclos monitorados e contaminação da calibração (motores nativos, H = 50, 75, 100)
  F09b  falsos alarmes e alarmes precoces por H (motores nativos)
  F09c  antecedência por H (motores nativos)
  F13a  poder de detecção por H = 25, 35, 50 (falhas injetadas, MEWMA, degrau de 0,10, falso alarme igualado; o de 0,25 satura)
  F13b  atraso mediano por H (falhas injetadas)
Fontes: tcc-writing/tables/healthy_window_summary.csv e pareado_far_resumo.csv.  Uso (da raiz): .venv/bin/python tcc-writing/make_figuras_janela.py
"""
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import FuncFormatter

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from pdm_cmaps.reporting import viz_style as vs  # noqa: E402

FIG = Path(__file__).resolve().parent / "figures"
TAB = Path(__file__).resolve().parent / "tables"
COL = {"FD001": vs.SERIES[0], "FD002": vs.SERIES[1], "FD003": vs.SERIES[2], "FD004": vs.INK}
MK = {"FD001": "o", "FD002": "s", "FD003": "^", "FD004": "D"}


def lines(ax, df, x, y, title, ylabel, xticks, ylim=None, legend=True):
    for sub in COL:
        d = df[df.Subset == sub].sort_values(x)
        ax.plot(d[x], d[y], color=COL[sub], lw=2, marker=MK[sub], ms=7, markeredgecolor=vs.SURFACE, markeredgewidth=1.5, label=sub)
    if ylim:
        ax.set_ylim(*ylim)
    if legend:
        ax.legend(loc="best", fontsize=8.5, ncol=2)
    ax.set_title(title, fontsize=11, loc="left")
    ax.set_ylabel(ylabel), ax.set_xlabel("janela saudável H (ciclos)")
    ax.set_xticks(xticks), ax.margins(x=0.12)
    vs.style_axes(ax, grid="y")


def main() -> None:
    vs.apply_style()
    h = pd.read_csv(TAB / "healthy_window_summary.csv")
    h = h[h.Window_kind.isin(["H=50", "H=75", "H=100"])].assign(H=lambda d: d.H_mean.round().astype(int))
    mv, t50 = h[h.Detector.str.startswith("MEWMA")], h[h.Detector == "T50 EWMA"]
    xt = [50, 75, 100]
    # F09a
    fig, (a, b) = plt.subplots(1, 2, figsize=(10.5, 4.2))
    lines(a, mv, "H", "monitored_cycles_pct_mean", "Ciclos que sobram para monitorar", "% dos ciclos da vida monitorados", xt)
    lines(b, mv, "H", "contamination_pct_mean", "Degradação dentro da janela de calibração", "% das linhas da janela já em degradação", xt)
    fig.suptitle("Uma janela maior deixa menos ciclos monitorados e pode calibrar com motor já degradado", fontsize=12, fontweight="bold", x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.92)), fig.savefig(FIG / "F09a_janela_custo.png", dpi=170), plt.close(fig)
    # F09b
    fig, (a, b) = plt.subplots(1, 2, figsize=(10.5, 4.2))
    lines(a, mv, "H", "mean_healthy_false_alarm_pct_mean", "Falso alarme em ciclos saudáveis (MEWMA)", "% dos ciclos saudáveis com alarme", xt)
    a.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:.2f}".rstrip("0").rstrip(",").replace(".", ",") if "." in f"{y}" else f"{y:g}".replace(".", ",")))
    lines(b, mv, "H", "early_alarm_pct_mean", "Alarmes precoces (MEWMA)", "% dos motores com alarme muito cedo", xt)
    fig.suptitle("Falsos alarmes e alarmes precoces por tamanho da janela (motores nativos)", fontsize=12, fontweight="bold", x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.92)), fig.savefig(FIG / "F09b_janela_falsos_alarmes.png", dpi=170), plt.close(fig)
    # F09c
    fig, (a, b) = plt.subplots(1, 2, figsize=(10.5, 4.2), sharey=False)
    lines(a, mv, "H", "mean_lead_time_detected_cycles_mean", "MEWMA de 13 resíduos", "antecedência média (ciclos antes do fim)", xt)
    lines(b, t50, "H", "mean_lead_time_detected_cycles_mean", "Carta EWMA do T50", "antecedência média (ciclos antes do fim)", xt)
    fig.suptitle("Janelas maiores reduzem a antecedência da detecção (motores nativos)", fontsize=12, fontweight="bold", x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.92)), fig.savefig(FIG / "F09c_janela_antecedencia.png", dpi=170), plt.close(fig)
    # F13 (falhas injetadas)
    p = pd.read_csv(TAB / "pareado_far_resumo.csv")
    p = p[(p.kind == "degrau") & (p.magnitude == 0.10) & (p.budget == 0.2) & (p.Detector == "MEWMA (13 resíduos, com regime)")]
    p = p.assign(H=p["H"].astype(int))
    for name, col, title, yl, out in (("F13a_janela_poder.png", "power_pct_mean", "Poder de detecção por H", "poder (%)", "F13a_janela_poder.png"),
                                      ("F13b_janela_atraso.png", "delay_med_mean", "Atraso mediano da detecção por H", "atraso mediano (ciclos)", "F13b_janela_atraso.png")):
        fig, ax = plt.subplots(figsize=(6.4, 4.3))
        lines(ax, p, "H", col, title, yl, [25, 35, 50], ylim=(0, 100) if col == "power_pct_mean" else (0, 15))
        ax.set_title(f"{title}\n(MEWMA, degrau de 0,10, mesmo falso alarme)", fontsize=10.5, loc="left")
        fig.tight_layout(), fig.savefig(FIG / out, dpi=170), plt.close(fig)
    print("ok")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile projeto/tcc-writing/make_figuras_mewma.py
"""Figuras do passo a passo EWMA -> MEWMA (Seção 2.7) com dados reais do FD004 e uma falha injetada de 25% da assinatura.

  F20a  passo 1: o resíduo de um sensor, em σ saudável: a falha (início no ciclo 61) fica escondida no ruído
  F20b  passo 2: a EWMA do mesmo sensor suaviza, mas ainda não cruza o limiar
  F20c  passo 3: 13 sensores, cada um dentro do ruído, e a estatística T² da MEWMA, que cruza o limiar
Também imprime o exemplo numérico (dois sensores) usado no texto.
Uso (da raiz): .venv/bin/python tcc-writing/make_figuras_mewma.py
"""
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from pdm_cmaps.domain.schema import COLUMNS, SETTING_COLUMNS  # noqa: E402
from pdm_cmaps.reporting import viz_style as vs  # noqa: E402

FIG = Path(__file__).resolve().parent / "figures"
S13 = ["T24", "T30", "T50", "P30", "Ps30", "phi", "Nf", "Nc", "NRc", "BPR", "htBleed", "W31", "W32"]
H, T0, MAG, LAM, UNIT = 25, 60, 0.25, 0.10, 1


def build():
    df = pd.read_csv(ROOT / "data/CMaps/train_FD004.txt", sep=r"\s+", header=None, names=COLUMNS)
    st = df[SETTING_COLUMNS]
    df["regime"] = KMeans(6, n_init=10, random_state=42).fit((st - st.mean()) / st.std()).labels_
    calib = df[df.unit_id != UNIT]
    healthy = calib[calib.cycle <= H]
    r_mu = healthy.groupby("regime")[S13].mean()

    def resid(d):
        return (d[S13] - r_mu.loc[d.regime].to_numpy()).to_numpy()

    def center(x):
        return x - x[:H].mean(axis=0)

    sd = np.concatenate([center(resid(g.sort_values("cycle")))[:H] for _, g in calib.groupby("unit_id")]).std(axis=0)
    last = calib.sort_values(["unit_id", "cycle"]).groupby("unit_id").tail(10)
    w = (resid(last) / sd).mean(axis=0)

    def z_of(d):
        return center(resid(d)) / sd

    zh = np.concatenate([z_of(g.sort_values("cycle"))[:H] for _, g in calib.groupby("unit_id")])
    cov_inv = np.linalg.inv(np.cov(zh.T) + 1e-6 * np.eye(len(S13)))
    eol = calib.groupby("unit_id")["cycle"].transform("max")
    ctrl = calib[(calib.cycle <= eol - 125) & (calib.cycle <= 150)]

    def mewma(z):
        wv, out = np.zeros(z.shape[1]), np.zeros(len(z))
        for t in range(len(z)):
            wv = LAM * z[t] + (1 - LAM) * wv
            out[t] = wv @ cov_inv @ wv / (LAM / (2 - LAM) * (1 - (1 - LAM) ** (2 * (t + 1))))
        return out

    def ewma_abs(z):
        wv, out = 0.0, np.zeros(len(z))
        for t in range(len(z)):
            wv = LAM * abs(z[t]) + (1 - LAM) * wv
            out[t] = wv
        return out

    t2c = np.concatenate([mewma(z_of(g.sort_values("cycle")))[H:] for _, g in ctrl.groupby("unit_id") if len(g) > H + 5])
    thr_t2 = float(np.quantile(t2c, 0.9973))
    j = S13.index("T50")
    ec = np.concatenate([ewma_abs(z_of(g.sort_values("cycle"))[:, j])[H:] for _, g in ctrl.groupby("unit_id") if len(g) > H + 5])
    thr_e = float(np.quantile(ec, 0.9973))
    e = df[df.unit_id == UNIT].sort_values("cycle").iloc[:150]
    z = z_of(e)
    zf = z.copy()
    zf[T0:] += MAG * w
    return zf, mewma(zf), ewma_abs(zf[:, j]), thr_t2, thr_e, j


def first_alarm(x, thr, k=2):
    run = 0
    for t in range(H, len(x)):
        run = run + 1 if x[t] > thr else 0
        if run >= k:
            return t
    return None


def main():
    vs.apply_style()
    zf, t2, ew, thr_t2, thr_e, j = build()
    n = len(zf)
    cyc = np.arange(1, n + 1)
    a_e, a_t = first_alarm(ew, thr_e), first_alarm(t2, thr_t2)

    def base(ax, title):
        ax.axvspan(0, H, color=vs.GRID, alpha=0.6)
        ax.axvline(T0 + 1, color=vs.STATUS[3], lw=2)
        ax.set_xlim(0, n + 1)
        ax.set_xlabel("ciclo (voo)")
        ax.set_title(title, fontsize=11.5, loc="left")
        vs.style_axes(ax)

    # passo 1
    fig, ax = plt.subplots(figsize=(10, 3.9))
    ax.plot(cyc, zf[:, j], color=vs.INK_2, lw=1.2)
    ax.axhspan(-3, 3, color=vs.SERIES[0], alpha=0.07)
    base(ax, f"Passo 1 (motor {UNIT} do FD004): o resíduo do T50 (em σ saudável) com a falha escondida no ruído")
    ax.text(T0 + 2, 3.6, f"início da falha: ciclo {T0 + 1}", color=vs.STATUS[3], fontsize=10, fontweight="bold")
    ax.text(H / 2, 3.6, "calibração", ha="center", fontsize=9, color=vs.INK_2)
    ax.text(n - 1, -3.5, "faixa de ±3σ: o ruído de um motor saudável", ha="right", fontsize=9, color=vs.SERIES[0])
    ax.set_ylabel("resíduo (σ)"), ax.set_ylim(-4.2, 4.3)
    fig.tight_layout(), fig.savefig(FIG / "F20a_mewma_passo1_bruto.png", dpi=170), plt.close(fig)
    # passo 2
    fig, ax = plt.subplots(figsize=(10, 3.9))
    ax.plot(cyc, np.abs(zf[:, j]), color=vs.MUTED, lw=1, alpha=0.7, label="|resíduo| bruto")
    ax.plot(cyc, ew, color=vs.SERIES[0], lw=2.6, label="EWMA do |resíduo| (λ = 0,10)")
    ax.axhline(thr_e, color=vs.INK_2, lw=1.5, ls="--")
    ax.text(H + 2, thr_e + 0.08, "limiar da EWMA do T50", ha="left", fontsize=9, color=vs.INK_2)
    base(ax, f"Passo 2 (motor {UNIT} do FD004): a EWMA suaviza e acumula, mas um sensor só ainda não é suficiente")
    msg = "a EWMA do T50 não cruza o limiar" if a_e is None else f"a EWMA cruza o limiar só no ciclo {a_e + 1}"
    ax.text(T0 + 2, ax.get_ylim()[1] * 0.92, msg, color=vs.SERIES[0], fontsize=10, fontweight="bold")
    ax.set_ylabel("|z| e EWMA"), ax.legend(loc="upper left", fontsize=9)
    fig.tight_layout(), fig.savefig(FIG / "F20b_mewma_passo2_ewma.png", dpi=170), plt.close(fig)
    # passo 3
    fig = plt.figure(figsize=(10, 6.2))
    gs = fig.add_gridspec(2, 1, height_ratios=[1.15, 1], hspace=0.38)
    sub = gs[0].subgridspec(3, 5, hspace=0.5, wspace=0.25)
    for k, name in enumerate(S13):
        ax = fig.add_subplot(sub[k // 5, k % 5])
        ax.plot(cyc, zf[:, k], color=vs.MUTED, lw=0.8)
        ax.axhspan(-3, 3, color=vs.SERIES[0], alpha=0.08)
        ax.axvline(T0 + 1, color=vs.STATUS[3], lw=1)
        ax.set_title(name, fontsize=8.5, loc="left"), ax.set_ylim(-4, 4), ax.set_xticks([]), ax.set_yticks([])
        for sp in ("top", "right"):
            ax.spines[sp].set_visible(False)
    fig.text(0.01, 0.955, f"Passo 3 (motor {UNIT} do FD004): cada um dos 13 sensores sozinho parece ruído...", fontsize=11.5, fontweight="bold", color=vs.INK)
    ax = fig.add_subplot(gs[1])
    ax.plot(cyc, t2 / thr_t2, color=vs.SERIES[1], lw=2.6)
    ax.axhline(1, color=vs.INK_2, lw=1.5, ls="--")
    ax.text(n - 1, 1.12, "limiar da MEWMA (1,0)", ha="right", fontsize=9, color=vs.INK_2)
    base(ax, f"...mas a MEWMA (motor {UNIT} do FD004) soma as mudanças pequenas e coerentes dos 13 e passa do limiar")
    ax.set_ylabel("T² ÷ limiar"), ax.set_ylim(0, 6)
    if a_t is not None:
        ax.scatter([a_t + 1], [min(t2[a_t] / thr_t2, 5.9)], s=90, color=vs.STATUS[3], zorder=5, edgecolor=vs.SURFACE, linewidth=2)
        ax.annotate(f"alarme: ciclo {a_t + 1}\n({a_t - T0} ciclos após o início)", (a_t + 1, min(t2[a_t] / thr_t2, 5.9)), xytext=(10, 18), textcoords="offset points", fontsize=9.5, color=vs.STATUS[3], fontweight="bold")
    fig.savefig(FIG / "F20c_mewma_passo3_t2.png", dpi=170, bbox_inches="tight"), plt.close(fig)
    print("alarmes: EWMA T50", None if a_e is None else a_e + 1, "| MEWMA", None if a_t is None else a_t + 1)
    # exemplo numerico
    z1, z2 = [0.3, -0.5, 0.9, 1.1], [0.2, 0.8, 0.6, 1.0]
    w1 = w2 = 0.0
    print("t  z1   z2   W1     W2     fator   T2")
    for t, (a, b) in enumerate(zip(z1, z2), 1):
        w1, w2 = LAM * a + (1 - LAM) * w1, LAM * b + (1 - LAM) * w2
        f = LAM / (2 - LAM) * (1 - (1 - LAM) ** (2 * t))
        print(f"{t}  {a:4.1f} {b:4.1f}  {w1:6.3f} {w2:6.3f}  {f:.4f}  {(w1**2 + w2**2) / f:.2f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile projeto/tcc-writing/make_figuras_novas.py
"""Figuras novas da revisão de 2026-11-16, feitas só com dados já versionados (nenhum estudo é rodado).

F24  Capítulo 1   T50 bruto e resíduo do sensor virtual do mesmo motor (results/maintenance_guide_fd004/tables)
F25  Seção 5.7    cobertura do intervalo conformal por motor, semente 42 (results/maintenance_guide_fd00x/tables/alert_timeline.csv)
F26  Seção 5.8    custo relativo ao oráculo por limiar crítico (tcc-writing/tables/margin_check_point_thresholds_raw.csv)
Uso (da raiz): .venv/bin/python tcc-writing/make_figuras_novas.py
"""
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from pdm_cmaps.reporting import viz_style as vs  # noqa: E402

HERE = Path(__file__).resolve().parent
FIG, TAB, RES = HERE / "figures", HERE / "tables", ROOT / "results"
SUBSETS = ("FD001", "FD002", "FD003", "FD004")
MOTOR, JANELA = 88, 50  # o mesmo motor da figura dos desvios do envelope; janela saudável do pipeline nativo


def _virgula(v: float, nd: int = 2) -> str:
    return f"{v:.{nd}f}".replace(".", ",")


def bruto_vs_residuo() -> None:
    d = pd.read_csv(RES / "maintenance_guide_fd004" / "tables" / "virtual_sensor_residuals_t50_h50.csv")
    d = d[d["unit_id"] == MOTOR].sort_values("cycle")
    vs.apply_style()
    fig, (a, b) = plt.subplots(2, 1, figsize=(8.4, 5.4), sharex=True)
    a.scatter(d["cycle"], d["T50"], s=9, color=vs.SERIES[0])
    a.set_ylabel("T50 medido (°R)", fontsize=10)
    a.set_title("Sinal bruto: o que se vê é o regime de voo (cada faixa é um regime)", fontsize=10.5, loc="left", fontweight="bold")
    b.axvspan(d["cycle"].min(), JANELA, color="#ecebe5", lw=0)
    b.text(JANELA / 2, 0.93, "calibração", transform=b.get_xaxis_transform(), ha="center", va="top", fontsize=9, color=vs.INK_2)
    b.axhline(0, color=vs.INK_2, lw=1)
    b.scatter(d["cycle"], d["residual"], s=9, color="#b9b7ae", label="resíduo de cada voo")
    b.plot(d["cycle"], d["residual"].rolling(15, min_periods=5).mean(), color=vs.SERIES[1], lw=2.4, label="média dos 15 voos anteriores")
    b.set_ylabel("T50 medido − esperado (°R)", fontsize=10), b.set_xlabel("ciclo (voo)", fontsize=10)
    b.set_title("Depois de remover o regime: aparece o desgaste (escala vertical ampliada)", fontsize=10.5, loc="left", fontweight="bold")
    b.legend(loc="upper left", bbox_to_anchor=(0.16, 0.99), fontsize=9, frameon=False)
    for ax in (a, b):
        ax.tick_params(labelsize=9.5)
        vs.style_axes(ax)
    fig.suptitle(f"Motor {MOTOR} do FD004 (holdout, sem descontinuidade)", fontsize=11.5, fontweight="bold", x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(FIG / "F24_bruto_vs_residuo.png", dpi=200)
    plt.close(fig)


def cobertura_por_motor() -> None:
    rng = np.random.default_rng(0)
    vs.apply_style()
    fig, ax = plt.subplots(figsize=(8.4, 4.0))
    for i, sub in enumerate(SUBSETS):
        t = pd.read_csv(RES / f"maintenance_guide_{sub.lower()}" / "tables" / "alert_timeline.csv")
        dentro = (t["RUL_lower"] <= t["RUL_true"]) & (t["RUL_true"] <= t["RUL_upper"])
        por_motor = dentro.groupby(t["unit_id"]).mean()
        y = len(SUBSETS) - 1 - i
        ax.scatter(por_motor, y + rng.uniform(-0.22, 0.22, len(por_motor)), s=22, color=vs.SERIES[0], alpha=0.55, linewidths=0)
        p10 = por_motor.quantile(0.10)
        ax.plot([p10, p10], [y - 0.34, y + 0.34], color=vs.STATUS[3], lw=2.2)
        ax.text(p10 - 0.008, y + 0.36, f"P10 = {_virgula(p10)}", ha="right", va="center", fontsize=9, color=vs.STATUS[3])
        ax.scatter([dentro.mean()], [y], marker="D", s=62, color=vs.INK, zorder=5, edgecolor=vs.SURFACE, linewidth=1.2)
    ax.axvline(0.90, color=vs.INK_2, lw=1.2, ls="--")
    ax.text(0.903, len(SUBSETS) - 0.42, "nominal: 0,90", fontsize=9, color=vs.INK_2, ha="left", va="center")
    ax.set_yticks(range(len(SUBSETS))), ax.set_yticklabels(SUBSETS[::-1], fontsize=10)
    ax.set_ylim(-0.6, len(SUBSETS) - 0.25), ax.set_xlim(0.25, 1.02)
    ax.set_xticks(np.arange(0.3, 1.01, 0.1)), ax.set_xticklabels([_virgula(v, 1) for v in np.arange(0.3, 1.01, 0.1)], fontsize=9.5)
    ax.set_xlabel("cobertura do intervalo de 90 % em cada motor do holdout (fração dos ciclos com a RUL real dentro do intervalo)", fontsize=9.5)
    vs.style_axes(ax, grid="x")
    handles = [Line2D([], [], ls="", marker="o", color=vs.SERIES[0], alpha=0.55, markersize=6, label="um motor do holdout"),
               Line2D([], [], ls="", marker="D", color=vs.INK, markersize=7, label="cobertura no conjunto (todos os ciclos)"),
               Line2D([], [], color=vs.STATUS[3], lw=2.2, label="décimo percentil dos motores (P10)")]
    fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=9.5, frameon=False)
    fig.suptitle("A cobertura vale no conjunto, não em cada motor (semente 42)", fontsize=10.5, fontweight="bold", x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0.08, 1, 0.95))
    fig.savefig(FIG / "F25_cobertura_por_motor.png", dpi=200)
    plt.close(fig)


def custo_por_limiar() -> None:
    d = pd.read_csv(TAB / "margin_check_point_thresholds_raw.csv")
    m = d.groupby(["S", "L", "strat", "c"])["rel"].mean().reset_index()
    est = {"point": ("Previsão pontual", vs.SERIES[0], "o"), "interval": ("Limite inferior do intervalo", vs.SERIES[1], "s")}
    prazos = (0, 10)
    vs.apply_style()
    fig, axes = plt.subplots(len(prazos), len(SUBSETS), figsize=(8.4, 5.6), sharex=True, sharey=True)
    for i, L in enumerate(prazos):
        for j, sub in enumerate(SUBSETS):
            ax = axes[i, j]
            for k, (lab, col, mk) in est.items():
                g = m[(m["S"] == sub) & (m["L"] == L) & (m["strat"] == k)].sort_values("c")
                ax.plot(g["c"], g["rel"], color=col, marker=mk, ms=4.5, lw=1.8, label=lab)
                best = g.loc[g["rel"].idxmin()]
                ax.scatter([best["c"]], [best["rel"]], s=95, facecolor="none", edgecolor=col, linewidth=1.8, zorder=5)
                acima = k == "interval"
                ax.annotate(_virgula(best["rel"]), (best["c"], best["rel"]), xytext=(0, 8 if acima else -8), textcoords="offset points",
                            ha="center", va="bottom" if acima else "top", fontsize=8.5, color=col, fontweight="bold", zorder=7,
                            bbox=dict(boxstyle="round,pad=0.12", facecolor="white", edgecolor="none", alpha=0.9))  # fundo branco: a curva não cobre o rótulo
            ax.set_yscale("log"), ax.set_ylim(0.8, 10), ax.set_xlim(3, 54)
            ax.set_yticks([1, 1.5, 2, 3, 5, 8]), ax.set_yticklabels(["1", "1,5", "2", "3", "5", "8"])
            ax.minorticks_off()
            ax.set_xticks([10, 20, 30, 40, 50]), ax.tick_params(labelsize=9)
            if i == 0:
                ax.set_title(sub, fontsize=10.5, loc="left", fontweight="bold")
            vs.style_axes(ax)
        axes[i, 0].set_ylabel(("sem prazo de logística" if L == 0 else f"prazo de {L} ciclos") + "\ncusto relativo ao oráculo", fontsize=9.5)
    axes[-1, 0].set_xlabel("limiar crítico c (ciclos de RUL)", fontsize=9, loc="left")
    h, l = axes[0, 0].get_legend_handles_labels()
    h.append(Line2D([], [], ls="", marker="o", markersize=10, markerfacecolor="none", markeredgecolor=vs.INK_2, markeredgewidth=1.6))
    l.append("menor custo de cada estratégia")
    fig.legend(h, l, loc="lower center", ncol=3, fontsize=9.5, frameon=False)
    fig.suptitle("Custo por limiar crítico (razão de custo 10; escala logarítmica; 1 = oráculo)\no limite inferior só foi avaliado com limiares de 10 a 20",
                 fontsize=10.5, fontweight="bold", x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0.07, 1, 0.93))
    fig.savefig(FIG / "F26_custo_por_limiar.png", dpi=200)
    plt.close(fig)


if __name__ == "__main__":
    bruto_vs_residuo()
    cobertura_por_motor()
    custo_por_limiar()
    print("ok")

In [ ]:
%%writefile projeto/tcc-writing/make_figuras_revisao.py
"""Figuras refeitas ou divididas a pedido do orientador (revisao de 2026-11-14), sem alterar o pacote `pdm_cmaps`.

  F15  Figura 5.4  RUL prevista x real sem a coluna em RUL real = 125 (teto da verdade truncada)
  F17a/b/c  Figura 5.6  desvios isolados do envelope, dividida em tres figuras (fundo branco)
  F05  Figura 5.5  carta de controle do motor 179 com a manutencao (ciclo 242) rotulada

Usa os dados de results/maintenance_guide_fd00x/tables (versionados) e as funcoes de
`pdm_cmaps.experiments.envelope_ignored_cases` e `rul_prediction_plots` apenas para leitura.
Uso (da raiz): .venv/bin/python tcc-writing/make_figuras_revisao.py [--so f15|f17]   (segundos)
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from pdm_cmaps.experiments import envelope_ignored_cases as env  # noqa: E402
from pdm_cmaps.experiments.rul_prediction_plots import EVENT_COLOR, PRED_COLOR, VERIFIED_MAINTENANCE  # noqa: E402
from pdm_cmaps.reporting import viz_style as vs  # noqa: E402

FIG = Path(__file__).resolve().parent / "figures"
CAP = 125


def f15(results: Path, model: str = "XGBoost (+temporal)", subset: str = "FD004") -> None:
    from sklearn.metrics import r2_score

    vs.apply_style()
    df = pd.read_csv(results / f"maintenance_guide_{subset.lower()}" / "tables" / "rul_holdout_predictions.csv")
    col = next(c for c in df.columns if c.startswith("RUL_predicted_") and "xgboost" in c.lower() and "temporal" in c.lower() and "nasa" not in c.lower())
    truth, pred = df["RUL_actual"].to_numpy(), df[col].to_numpy()
    r2_all = r2_score(truth, pred)
    keep = truth < CAP
    fig, ax = plt.subplots(figsize=(6.4, 6.0))
    ax.scatter(truth[keep], pred[keep], s=7, alpha=0.22, color=PRED_COLOR, edgecolors="none", label="ciclos do holdout")
    for (sub, unit), cyc in VERIFIED_MAINTENANCE.items():
        if sub != subset:
            continue
        m = (df["unit_id"] == unit).to_numpy() & keep
        after = m & (df["cycle"] >= cyc).to_numpy()
        if (m & ~after).any():
            ax.scatter(truth[m & ~after], pred[m & ~after], s=15, color=vs.SERIES[1], label=f"motor {unit}, antes do ciclo {cyc}")
        if after.any():
            ax.scatter(truth[after], pred[after], s=15, color=EVENT_COLOR, label=f"motor {unit}, após a descontinuidade (ciclo ≥ {cyc})")
    lo, hi = 0, CAP + 3
    ax.plot([lo, hi], [lo, hi], color=vs.INK_2, ls="--", lw=1.4, label="previsão perfeita")
    ax.set_xlim(lo, hi), ax.set_ylim(lo - 2, 150)
    vs.style_axes(ax, grid="both")
    ax.set_xlabel("RUL real (ciclos)"), ax.set_ylabel("RUL prevista (ciclos)")
    n_fmt = f"{len(truth):,}".replace(",", " ")
    r2_fmt = f"{r2_all:.3f}".replace(".", ",")
    ax.set_title(f"{subset} · XGBoost com atributos temporais\nR² no holdout = {r2_fmt} (todos os ciclos, n = {n_fmt})", fontsize=10.5)
    ax.legend(loc="upper left", fontsize=8)
    n_omit = f"{int((~keep).sum()):,}".replace(",", " ")
    ax.text(0.99, 0.02, f"{n_omit} ciclos com RUL real truncada em {CAP} não são mostrados", transform=ax.transAxes, ha="right", va="bottom", fontsize=8, color=vs.INK_2)
    fig.tight_layout()
    fig.savefig(FIG / "F15_dispersao_predito_real_xgboost_temporal.png", dpi=150)
    plt.close(fig)
    print("F15 ok:", model, "omitidos", int((~keep).sum()))


def f05(results: Path, unit: int = 179, maint_cycle: int = 242) -> None:
    """Carta de controle do motor 179 (Figura 5.5) com a manutenção realizada no ciclo 242 rotulada nos dois painéis."""
    vs.apply_style()
    trace = pd.read_csv(results / "maintenance_guide_fd004" / "tables" / "envelope_scored_trace_h50.csv")
    eng = trace[trace.unit_id == unit].sort_values("cycle")
    cyc = eng["cycle"].to_numpy()
    eol = int(cyc.max())
    flag = ((eng["ewma_abs_z"] > eng["ewma_ucl"]) & (eng["z_score"].abs() >= 3.0) & (eng["cycle"] > 50)).to_numpy()
    first, run = None, 0
    for i, f in enumerate(flag):
        run = run + 1 if f else 0
        if run >= 2:
            first = int(cyc[i - 1])
            break
    gold = "#c98500"
    fig, axes = plt.subplots(2, 1, figsize=(10, 6.8), sharex=True, gridspec_kw={"hspace": 0.3})
    for ax in axes:
        vs.style_axes(ax, "y")
        ax.axvspan(0.5, 50.5, color=vs.MUTED, alpha=0.10, lw=0)
        ax.axvline(maint_cycle, color=gold, lw=2.2, ls=(0, (5, 3)))
    top, bot = axes
    top.plot(cyc, eng["z_score"], color=vs.SERIES[0], lw=vs.LINE_W)
    for lv in (3, -3):
        top.axhline(lv, color=vs.INK_2, lw=1.2, ls=(0, (4, 3)))
    top.text(56, 3.05, "limite ±3σ", color=vs.INK_2, fontsize=8.5)
    top.text(25, top.get_ylim()[1] * 0.95, "calibração saudável\n(ciclos 1–50)", ha="center", va="top", color=vs.INK_2, fontsize=8.5)
    top.set_ylabel("z do resíduo")
    top.set_title("Resíduo do sensor virtual (real − esperado em condição saudável), padronizado")
    top.text(maint_cycle + 5, -2.3, "descontinuidade\n(ciclo 242)", color=gold, fontsize=9.5, fontweight="bold", va="center")
    bot.plot(cyc, eng["ewma_abs_z"], color=vs.SERIES[0], lw=vs.LINE_W, label="EWMA de |z| (λ = 0,10)")
    bot.plot(cyc, eng["ewma_ucl"], color=vs.INK_2, lw=1.2, ls=(0, (4, 3)), label="Limite de controle (3σ)")
    bot.legend(loc="upper left", ncol=2, bbox_to_anchor=(0.0, 1.02))
    if first is not None:
        y = float(eng.loc[eng["cycle"] == first, "ewma_abs_z"].iloc[0])
        bot.vlines(first, 0, y, color=vs.INK_2, lw=1.0)
        bot.plot([first], [y], marker="o", **vs.ring_marker_kwargs(vs.STATUS[3], 10))
        bot.annotate(f"Primeiro alarme persistente: ciclo {first}\nantecedência de {eol - first} ciclos até o fim de vida", xy=(first, y), xytext=(-12, 6), textcoords="offset points", ha="right", va="bottom", color=vs.INK, fontsize=9)
    bot.annotate("descontinuidade\n(ciclo 242)", xy=(maint_cycle, 4.0), xytext=(maint_cycle + 18, 5.4), ha="left", color=gold, fontsize=9.5, fontweight="bold", arrowprops=dict(arrowstyle="->", color=gold, lw=1.8))
    bot.set_ylabel("EWMA de |z|"), bot.set_xlabel("Ciclo operacional (o último ciclo é o fim de vida)"), bot.set_title("Carta de controle")
    fig.suptitle(f"Motor {unit} do FD004 (holdout, com descontinuidade no ciclo {maint_cycle})", x=0.01, ha="left", fontsize=12, fontweight="bold")
    fig.subplots_adjust(top=0.88)
    fig.savefig(FIG / "F05_carta_controle_residuo_fd004.png", dpi=164)
    plt.close(fig)
    print("F05 ok: primeiro alarme", first, "fim de vida", eol)


def _cases(r, runs, subset="FD004"):
    rs, ru = r[r["Subset"] == subset], runs[runs["Subset"] == subset]
    mon = ru[ru["monitored"]]
    ign_by_unit = mon[~mon["alert"]].groupby("unit_id").size()
    firsts = ru[ru["first_alert"]].set_index("unit_id")["start"]
    cand = [(int(u), int(n)) for u, n in ign_by_unit.items() if u in firsts.index and (subset, int(u)) not in env.RESET_ENGINES]
    top = max(cand, key=lambda t: (t[1], t[0]))[0] if cand else int(rs["unit_id"].iloc[0])
    runs_top = ru[ru["unit_id"] == top]
    healthy_ign = mon[(~mon["alert"]) & mon["phase"].str.startswith("vida saudável") & ~mon["unit_id"].map(lambda u: (subset, int(u)) in env.RESET_ENGINES)]
    deg_ign = mon[(~mon["alert"]) & mon["phase"].str.startswith("degradação")]
    case_b = healthy_ign.iloc[(healthy_ign["max_excess_sigma"] - healthy_ign["max_excess_sigma"].median()).abs().argsort()].iloc[0]
    case_c = deg_ign.iloc[(deg_ign["max_excess_sigma"] - deg_ign["max_excess_sigma"].median()).abs().argsort()].iloc[0] if len(deg_ign) else case_b
    case_d = runs_top[runs_top["first_alert"]].iloc[0]
    return rs, ru, mon, top, runs_top, case_b, case_c, case_d


def f17(results: Path, subset: str = "FD004") -> None:
    vs.apply_style()
    frames = [env.fleet_envelope(results, s)[0] for s in env.SUBSETS]
    r = pd.concat(frames, ignore_index=True)
    runs = env.breach_runs(r)
    summ = env.summarize(r, runs)
    rs, ru, mon, top, runs_top, case_b, case_c, case_d = _cases(r, runs, subset)
    g_top = rs[rs["unit_id"] == top]

    # ---- F17a: um motor inteiro, com todos os desvios ----
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    env._band(ax)
    ax.plot(g_top["cycle"], g_top["res_sigma"], color=vs.INK, lw=1.0, label="resíduo do T50 (em σ)")
    env._mark(ax, g_top, runs_top[runs_top["monitored"]], 70)
    fa = int(runs_top[runs_top["first_alert"]]["alert_cycle"].iloc[0])
    ax.axvline(fa, color=vs.STATUS[3], lw=1.8, ls="--")
    ax.axvspan(g_top["cycle"].min(), env.H, color=vs.STATUS[0], alpha=0.10, label=f"janela de calibração (até o ciclo {env.H}, não monitorada)")
    ymax = g_top["res_sigma"].max() + 1.5
    ax.annotate(f"primeiro alerta: ciclo {fa}\n(2 ciclos seguidos fora do envelope)", (fa, ymax - 0.4), xytext=(-12, 0), textcoords="offset points", ha="right", va="top", fontsize=9.5, color=vs.STATUS[3], fontweight="bold")
    n_ign = int(((~runs_top["alert"]) & runs_top["monitored"]).sum())
    h, l = ax.get_legend_handles_labels()
    ax.legend(dict(zip(l, h)).values(), dict(zip(l, h)).keys(), fontsize=8.5, ncol=2, loc="lower left")
    ax.set_title(f"Motor {top} do {subset} (holdout, sem descontinuidade): {n_ign} desvios isolados ignorados; alerta só quando persiste", fontsize=10.8)
    ax.set_xlabel("ciclo (voo)"), ax.set_ylabel("resíduo do T50 (σ saudável)")
    ax.set_ylim(min(-4.8, g_top["res_sigma"].min() - 0.5), ymax)
    vs.style_axes(ax)
    fig.tight_layout()
    fig.savefig(FIG / "F17a_envelope_motor_completo.png", dpi=150)
    plt.close(fig)

    # ---- F17b: tres exemplos de zoom ----
    fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.4))
    specs = [(case_b, f"Desvio isolado, vida saudável\nmotor {int(case_b['unit_id'])} ({subset}): 1 ciclo, {case_b['max_excess_sigma']:.2f}σ além", vs.INK),
             (case_c, f"Desvio isolado, em degradação\nmotor {int(case_c['unit_id'])} ({subset}): 1 ciclo, {case_c['max_excess_sigma']:.2f}σ além", vs.INK),
             (case_d, f"Desvio persistente: vira alerta\nmotor {top} ({subset}): ≥ 2 ciclos seguidos", vs.STATUS[3])]
    for ax, (run, ttl, color) in zip(axes, specs):
        gz = rs[rs["unit_id"] == int(run["unit_id"])]
        env._zoom(ax, gz, run, ttl, color, ru[(ru["unit_id"] == int(run["unit_id"])) & ru["monitored"] & (ru["end"] >= run["start"] - 14) & (ru["start"] <= run["end"] + 14)])
        ax.set_title(ttl, fontsize=10.5, color=color, loc="left", fontweight="semibold")
        ax.set_ylabel("resíduo do T50 (σ)")
        leg = ax.get_legend()
        if leg:
            leg.remove()
    h, l = axes[2].get_legend_handles_labels()
    allh, alll = [], []
    for ax in axes:
        hh, ll = ax.get_legend_handles_labels()
        allh += hh
        alll += ll
    uniq = dict(zip(alll, allh))
    fig.legend(uniq.values(), uniq.keys(), loc="lower center", ncol=4, fontsize=9, frameon=False)
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    fig.savefig(FIG / "F17b_envelope_desvios_isolados.png", dpi=150)
    plt.close(fig)

    # ---- F17c: o que a regra ensina na frota ----
    fig, (axe, axf) = plt.subplots(1, 2, figsize=(12.5, 4.6))
    pool_ign = runs[runs["monitored"] & ~runs["alert"]]["max_excess_sigma"]
    pool_per = runs[runs["monitored"] & runs["alert"]]["max_excess_sigma"]
    bins = np.linspace(0, max(8, pool_per.quantile(0.9)), 22)
    axe.hist(pool_ign, bins=bins, alpha=0.9, color=vs.STATUS[1], label=f"ignorados (n = {len(pool_ign)})", edgecolor=vs.SURFACE)
    axe.hist(pool_per, bins=bins, alpha=0.85, color=vs.STATUS[3], label=f"viraram alerta (n = {len(pool_per)})", edgecolor=vs.SURFACE)
    axe.set_xlabel("excesso máximo além do envelope (σ do resíduo saudável)"), axe.set_ylabel("sequências de desvio")
    axe.set_title("Os desvios ignorados são pequenos;\nos persistentes são maiores", fontsize=10.5)
    axe.legend(), vs.style_axes(axe)
    x = np.arange(len(summ))
    a_, b_ = summ["false_alert_per100_without_persistence"], summ["false_alert_per100_with_persistence"]
    axf.bar(x - 0.2, a_, 0.4, color=vs.INK_2, label="alertar em todo desvio")
    axf.bar(x + 0.2, b_, 0.4, color=vs.SERIES[0], label="exigir 2 ciclos seguidos")
    axf.scatter(x + 0.2, b_, marker="_", s=420, color=vs.SERIES[0], linewidths=3.5, zorder=4, clip_on=False)  # barras de valor zero continuam visíveis
    for xi, (a, b) in enumerate(zip(a_, b_)):
        axf.text(xi - 0.2, a + 0.004, f"{a:.2f}".replace(".", ","), ha="center", va="bottom", fontsize=9)
        axf.text(xi + 0.2, b + 0.004, f"{b:.2f}".replace(".", ","), ha="center", va="bottom", fontsize=9)
    axf.set_xticks(x), axf.set_xticklabels(summ["Subset"])
    axf.set_ylabel("alertas falsos por 100 ciclos saudáveis")
    axf.set_title("O que a persistência evita: alertas falsos\nna vida saudável (RUL ≥ 125, após a calibração)", fontsize=10.5)
    axf.legend(), vs.style_axes(axf)
    fig.tight_layout()
    fig.savefig(FIG / "F17c_envelope_persistencia_frota.png", dpi=150)
    plt.close(fig)
    print("F17a/b/c ok: motor", top, "casos", int(case_b["unit_id"]), int(case_c["unit_id"]))


def f14_linhas(results: Path, unit: int = 179, subset: str = "FD004", maint_cycle: int = 242) -> None:
    """Gera F14_linha1..4.png com cabeçalho explícito do motor 179 do FD004 em cada faixa."""
    from pdm_cmaps.experiments.rul_prediction_plots import NOMES_PT, REAL_COLOR

    vs.apply_style()
    df = pd.read_csv(results / f"maintenance_guide_{subset.lower()}" / "tables" / "rul_holdout_predictions.csv")
    eng = df[df["unit_id"] == unit].sort_values("cycle")
    models = [c.replace("RUL_predicted_", "") for c in df.columns if c.startswith("RUL_predicted_") and "MiniRocket" not in c]  # MiniRocket fora da monografia
    ymax = float(eng["RUL_actual"].max())
    ncol = 5
    nrow = int(np.ceil(len(models) / ncol))
    for r_idx in range(nrow):
        row_models = models[r_idx * ncol : (r_idx + 1) * ncol]
        fig, axes = plt.subplots(1, ncol, figsize=(3.4 * ncol, 2.50), sharex=True, sharey=True)
        for ax in axes[len(row_models):]:
            ax.set_visible(False)
        for j, (ax, model) in enumerate(zip(axes, row_models)):
            pred = eng[f"RUL_predicted_{model}"]
            rmse = float(np.sqrt(np.mean((pred - eng["RUL_actual"]) ** 2)))
            ax.plot(eng["cycle"], eng["RUL_actual"], color=REAL_COLOR, lw=1.5)
            ax.plot(eng["cycle"], pred, color=PRED_COLOR, lw=1.5)
            ax.axvline(maint_cycle, color=EVENT_COLOR, lw=1.6, ls="--", zorder=1)
            ax.text(maint_cycle + 8, ymax * 1.13, f"descontinuidade (ciclo {maint_cycle})", ha="left", va="center", fontsize=8.5, color=EVENT_COLOR, fontweight="bold")
            vs.style_axes(ax)
            label = NOMES_PT.get(model, model)
            ax.set_title(f"{label}\n" + f"RMSE {rmse:.1f}".replace(".", ","), fontsize=9.5)
            ax.set_xlabel("Ciclo", fontsize=8.5)
            if j == 0:
                ax.set_ylabel("RUL (ciclos)", fontsize=9)
        fig.suptitle(f"Motor {unit} do {subset} (holdout, linha {r_idx + 1} de {nrow}): RUL real (preto) × prevista (azul) — descontinuidade no ciclo {maint_cycle}",
                     fontsize=11, fontweight="bold", x=0.01, ha="left", y=0.99)
        fig.tight_layout(rect=(0, 0, 1, 0.91))
        fig.savefig(FIG / f"F14_linha{r_idx + 1}.png", dpi=300)
        plt.close(fig)
    print("F14_linha1..4 ok: motor", unit)


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--results", type=Path, default=ROOT / "results")
    ap.add_argument("--so", choices=("f05", "f14", "f15", "f17"))
    a = ap.parse_args()
    if a.so in (None, "f05"):
        f05(a.results)
    if a.so in (None, "f14"):
        f14_linhas(a.results)
    if a.so in (None, "f15"):
        f15(a.results)
    if a.so in (None, "f17"):
        f17(a.results)

In [ ]:
%%writefile projeto/tcc-writing/diagrams/explicativos/make_fisica.py
"""Diagramas de física do turbofan para a Seção 2.4: ciclo de Brayton (T-s), atmosfera e voo, balanço de energia.

Os valores numéricos vêm das Tabelas 2.2 e 2.3 da monografia (dados do C-MAPSS); os desenhos são esquemáticos, sem escala.
Uso (da raiz): .venv/bin/python tcc-writing/diagrams/explicativos/make_fisica.py
"""
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

HERE = Path(__file__).resolve().parent
ROOT = HERE.parents[2]
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(HERE))
from pdm_cmaps.reporting import viz_style as vs  # noqa: E402

BLUE, ORANGE, AQUA, RED = vs.SERIES[0], vs.SERIES[1], vs.SERIES[2], vs.STATUS[3]


def brayton() -> None:
    """Figura do ciclo de Brayton: agora em make_brayton.py (versão refeita)."""
    import make_brayton

    make_brayton.main()


def atmosfera() -> None:
    vs.apply_style()
    alt = np.array([0, 10, 20, 25, 35, 42])
    ts = np.array([518.67, 483.01, 447.35, 429.52, 393.86, 389.97])
    t50 = np.array([1405.2, 1307.4, 1249.6, 1046.8, 1128.2, 1124.5])
    label = ["0 pés\nMach 0", "10 mil\n0,25", "20 mil\n0,70", "25 mil\n0,62\nmanete 60", "35 mil\n0,84", "42 mil\n0,84"]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4.9))
    h = np.linspace(0, 42, 100)
    a1.plot(h, np.where(h < 36.089, 518.67 - 3.566 * h, 389.97), color=vs.INK_2, lw=2, label="atmosfera padrão (T estática)")
    a1.scatter(alt, ts, s=70, color=BLUE, zorder=5, edgecolor=vs.SURFACE, linewidth=2, label="seis condições de voo do C-MAPSS")
    a1.axvline(36.089, color=vs.MUTED, lw=1, ls="--")
    a1.text(36.4, 505, "tropopausa", fontsize=9, color=vs.MUTED, rotation=90, va="top")
    a1.set_xlabel("altitude (mil pés)"), a1.set_ylabel("temperatura estática do ar (°R)")
    a1.set_title("O ar esfria com a altitude", fontsize=11.5)
    a1.legend(loc="lower left"), vs.style_axes(a1, grid="both")
    cols = [BLUE] * 3 + [ORANGE] + [BLUE] * 2
    a2.scatter(alt, t50, s=80, color=cols, zorder=5, edgecolor=vs.SURFACE, linewidth=2)
    for i, (x, y, lab) in enumerate(zip(alt, t50, label)):
        ha, dx = ("right", 6) if i == 4 else (("left", -6) if i == 5 else ("center", 0))
        a2.annotate(lab.replace("\n", " · "), (x, y), xytext=(dx, 12), textcoords="offset points", ha=ha, fontsize=8.5, color=vs.INK_2)
    # barra de escala: o desgaste de uma vida inteira (30 °R), no canto vazio
    a2.plot([49, 49], [1340, 1370], color=RED, lw=4, solid_capstyle="butt")
    a2.text(47.6, 1355, "desgaste de uma vida\ninteira: 20 a 30 °R", fontsize=9, color=RED, va="center", ha="right", fontweight="bold")
    a2.set_xlim(-3, 52), a2.set_ylim(950, 1480)
    a2.set_xlabel("altitude (mil pés)"), a2.set_ylabel("T50 médio na saída da turbina de baixa (°R)")
    a2.set_title("O T50 médio muda centenas de graus com o regime", fontsize=11.5)
    vs.style_axes(a2, grid="y")
    fig.suptitle("Por que o regime de voo pesa mais que o desgaste", fontsize=13, fontweight="bold", x=0.01, ha="left")
    fig.text(0.01, 0.01, "Médias dos 249 motores de treino do FD004 (multirregime, Tabela 4.1); a escala vermelha compara o desgaste de uma vida inteira com a variação entre regimes.", fontsize=8.5, color=vs.MUTED)
    fig.tight_layout(rect=(0, 0.03, 1, 0.94))
    fig.savefig(HERE / "atmosfera-e-voo.png", dpi=200, facecolor=vs.SURFACE)
    plt.close(fig)


def balanco() -> None:
    import make_svgs as m
    W, H = 820, 470
    s = m.Svg(W, H)
    INK, INK2, MUTED = m.INK, m.INK2, m.MUTED
    s.text(30, 34, "Balanço de energia: por que um compressor desgastado esquenta o T50", 15, INK, "start", "700")
    # eixo
    s.rect(130, 116, 560, 14, fill="#c3c2b7", stroke=INK2, sw=1.2, rx=6)
    s.text(410, 108, "eixo de alta pressão", 11.5, INK2, "middle", "600")
    # compressor e turbina
    s.poly([(60, 78), (60, 168), (130, 150), (130, 96)], fill="#f6c9a3", stroke=INK2, sw=1.5)
    s.text(95, 190, "compressor (HPC)", 12, INK, "middle", "700")
    s.poly([(690, 96), (690, 150), (760, 168), (760, 78)], fill="#f2a67e", stroke=INK2, sw=1.5)
    s.text(725, 190, "turbina (HPT)", 12, INK, "middle", "700")
    s.rect(300, 208, 220, 40, fill="#fde5da", stroke=m.ORANGE, sw=2, rx=8)
    s.text(410, 232, "câmara: queima de combustível", 12, INK, "middle", "700")
    s.arrow(95, 60, 95, 76, "#5f9be8", 3, 8)
    s.text(95, 54, "ar entra", 11, INK2, "middle")
    s.arrow(130, 160, 296, 220, m.ORANGE, 3, 8)
    s.arrow(522, 226, 690, 160, m.ORANGE, 3, 8)
    s.text(410, 276, "trabalho do compressor  =  trabalho da turbina   (regime permanente)", 14, INK, "middle", "700")
    s.math(410, 300, "$c_p\\,(T_3 − T_2) = c_{p,\\r{gás}}\\,(T_4 − T_5)$", 16, INK2, "middle")
    # cadeia causal
    steps = [("compressor menos\neficiente ($η_c$ 0,886 → 0,872)", "#f6c9a3"), ("trabalho por massa de ar\n+1,4 % (256 → 260 BTU/lbm)", "#fde5da"),
             ("o controle injeta mais\ncombustível (mantém Nf)", "#fde5da"), ("gás sai mais quente:\nT30 +15 °R, T50 +26 °R", "#f2a67e")]
    bw, gap, y0 = 172, 28, 330
    for i, (t, c) in enumerate(steps):
        x = 30 + i * (bw + gap)
        s.rect(x, y0, bw, 76, fill=c, stroke=m.ORANGE, sw=1.8, rx=9)
        for k, ln in enumerate(t.split("\n")):
            s.math(x + bw / 2, y0 + 30 + k * 18, ln, 11.5, INK, "middle", "700" if k == 0 else "400")
        if i:
            s.arrow(x - gap + 2, y0 + 38, x - 2, y0 + 38, INK2, 2.5, 8)
    s.text(30, 440, "Por isso o T50 tem o maior deslocamento e a margem de EGT é o indicador de saúde mais usado na operação.", 12, INK2)
    s.text(30, 458, "Médias dos 100 motores de treino do FD001 (regime único, Tabela 4.1); esquema didático, sem escala.", 11, MUTED)
    s.h = H
    s.save("balanco-de-energia")


if __name__ == "__main__":
    brayton()
    atmosfera()
    balanco()
    print("ok")

In [ ]:
%%writefile projeto/tcc-writing/diagrams/explicativos/make_brayton.py
"""Figura 2.8 (no PDF atual, 2.7): ciclo de Brayton do núcleo (diagrama T-s esquemático).
Números: médias dos 100 motores de treino do FD001. Gera brayton-ts.png ao lado deste script.
Uso (no Colab, da pasta do script): python make_brayton.py
"""
from pathlib import Path

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

HERE = Path(__file__).resolve().parent

plt.rcParams.update({
    "font.family": "serif", "mathtext.fontset": "stix",
    "font.serif": ["STIX Two Text", "STIXGeneral", "DejaVu Serif"],
    "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
})

# Dados (°R)
T2, T3s = 518.67, 1465.0
T3 = {"novo": 1587.0, "fim": 1602.0}
T5 = {"novo": 1403.0, "fim": 1429.0}
T4 = 2150.0  # não medido no C-MAPSS: só ilustrativo
# eficiência do compressor, valores da monografia (Tabelas 2.1 e 2.2)
eta = {"novo": 0.886, "fim": 0.872}
virg = lambda x, d=3: f"{x:.{d}f}".replace(".", ",")

# Entropia esquemática (sem escala)
s2 = 0.0
s3 = {"novo": 0.22, "fim": 0.27}
s4 = 1.00
s5 = {"novo": 1.10, "fim": 1.16}

cor = {"novo": "#1f5fa8", "fim": "#c0392b"}
estilo = {"novo": "-", "fim": "--"}
fmt = lambda x: f"{x:,.0f}".replace(",", ".")


def main() -> None:
    fig, ax = plt.subplots(figsize=(6.3, 4.2))

    # Compressão ideal (vertical: entropia constante)
    ax.plot([s2, s2], [T2, T3s], color="0.45", lw=1.4, ls=":")
    ax.plot(s2, T3s, "o", ms=6, mfc="white", mec="0.35", mew=1.4, zorder=5)

    for k in ("novo", "fim"):
        c, ls = cor[k], estilo[k]
        # compressão real 2 -> 3
        ax.plot([s2, s3[k]], [T2, T3[k]], color=c, ls=ls, lw=1.8)
        # combustão 3 -> 4 (pressão quase constante: curva exponencial)
        s = np.linspace(s3[k], s4, 60)
        T = T3[k] * np.exp(np.log(T4 / T3[k]) * (s - s3[k]) / (s4 - s3[k]))
        ax.plot(s, T, color=c, ls=ls, lw=1.2, alpha=0.6)
        # expansão nas turbinas 4 -> 5
        ax.plot([s4, s5[k]], [T4, T5[k]], color=c, ls=ls, lw=1.8)
        ax.plot(s3[k], T3[k], "o", color=c, ms=6, zorder=5)
        ax.plot(s5[k], T5[k], "o", color=c, ms=6, zorder=5)

    ax.plot(s2, T2, "o", color="black", ms=6, zorder=5)
    ax.plot(s4, T4, "o", ms=6, mfc="white", mec="0.35", mew=1.4, zorder=5)

    # Rótulos dos pontos
    ax.annotate("2  entrada do fan\n$T_2$ = 518,67 °R", (s2, T2), xytext=(0.06, T2 + 10),
                va="center", fontsize=9)
    ax.annotate(f"3 ideal (sem perdas)\n$T_{{3,\\mathrm{{ideal}}}}$ = {fmt(T3s)} °R",
                (s2, T3s), xytext=(-0.33, T3s + 70), fontsize=9, ha="left")
    ax.text(s3["fim"] + 0.06, 1500,
            "3  saída do compressor (T30)\n"
            f"novo: {fmt(T3['novo'])} °R  ($\\eta_c$ = {virg(eta['novo'])})\n"
            f"fim de vida: {fmt(T3['fim'])} °R  ($\\eta_c$ = {virg(eta['fim'])})",
            fontsize=9, va="top")
    ax.text(s4 - 0.02, T4 + 45, "4  saída da câmara\n(não medido)", fontsize=9,
            ha="center", va="bottom", color="0.35")
    ax.text(s5["fim"] - 0.04, 1330 - 140,
            "5  saída da turbina de baixa (T50)\n"
            f"novo: {fmt(T5['novo'])} °R\nfim de vida: {fmt(T5['fim'])} °R",
            fontsize=9, va="top", ha="center")
    ax.text(0.55, 1960, "combustão\n(pressão quase constante)", fontsize=8.5,
            color="0.35", ha="center", rotation=12)

    # Legenda
    leg = [Line2D([], [], color=cor["novo"], ls="-", marker="o", label="motor novo"),
           Line2D([], [], color=cor["fim"], ls="--", marker="o", label="fim de vida"),
           Line2D([], [], color="0.45", ls=":", marker="o", mfc="white", mec="0.35",
                  label="compressão ideal (isentrópica)")]
    ax.legend(handles=leg, loc="upper left", frameon=False, fontsize=9)

    ax.set_xlim(-0.36, 1.45)
    ax.set_ylim(350, 2450)
    ax.set_xticks([])
    ax.set_xlabel("entropia, $s$ (esquemático, sem escala) $\\rightarrow$ mais perdas")
    ax.set_ylabel("temperatura total (°R)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: fmt(v)))

    fig.tight_layout()
    fig.savefig(HERE / "brayton-ts.png", dpi=260)
    plt.close(fig)
    print("ok")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile projeto/tcc-writing/diagrams/explicativos/make_leitura_curvas.py
"""Diagramas explicativos da injeção de falhas para a Seção 5.5: como ler uma curva operacional e a linha do tempo de uma falha.

Esquemáticos, sem dados do estudo (as curvas reais estão nas Figuras de resultados). Uso (da raiz): .venv/bin/python tcc-writing/diagrams/explicativos/make_leitura_curvas.py
"""
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE.parents[2] / "src"))
from pdm_cmaps.reporting import viz_style as vs  # noqa: E402

BLUE, ORANGE, RED = vs.SERIES[0], vs.SERIES[1], vs.STATUS[3]


def curva() -> None:
    vs.apply_style()
    fig, ax = plt.subplots(figsize=(8.2, 5.2))
    far = np.geomspace(0.01, 10, 200)
    good = 100 * (1 - np.exp(-1.6 * (np.log10(far) + 2.2))) ** 1
    good = np.clip(good, 0, 100)
    weak = np.clip(100 * (1 - np.exp(-0.9 * (np.log10(far) + 2.0))), 0, 95) * 0.78
    ax.plot(far, good, color=ORANGE, lw=3, label="detector A (melhor)")
    ax.plot(far, weak, color=BLUE, lw=3, ls="--", label="detector B")
    ax.axvline(0.2, color=vs.INK_2, lw=1.5, ls=":")
    ax.text(0.2, 6, "  orçamento: 0,2 falsos alarmes\n  por 100 ciclos", fontsize=10, color=vs.INK_2, va="bottom")
    pa, pb = float(np.interp(0.2, far, good)), float(np.interp(0.2, far, weak))
    for y, c in ((pa, ORANGE), (pb, BLUE)):
        ax.scatter([0.2], [y], s=110, color=c, zorder=5, edgecolor=vs.SURFACE, linewidth=2)
    ax.annotate(f"no mesmo orçamento,\no detector A detecta {pa:.0f}%", (0.2, pa), xytext=(0.45, 79), fontsize=10, color=ORANGE, fontweight="bold", arrowprops=dict(arrowstyle="->", color=ORANGE))
    ax.annotate(f"e o detector B, só {pb:.0f}%", (0.2, pb), xytext=(0.45, pb - 28), fontsize=10, color=BLUE, fontweight="bold", arrowprops=dict(arrowstyle="->", color=BLUE))
    ax.annotate("", xy=(0.013, 96), xytext=(0.06, 70), arrowprops=dict(arrowstyle="-|>", color=vs.STATUS[0], lw=3))
    ax.text(0.014, 62, "melhor: mais poder\ncom menos falsos alarmes", fontsize=10.5, color="#12805a", fontweight="bold")
    ax.set_xscale("log"), ax.set_xlim(0.01, 10), ax.set_ylim(0, 102)
    ax.set_xlabel("falsos alarmes por 100 ciclos saudáveis (escala logarítmica)", color=vs.INK_2)
    ax.set_ylabel("poder de detecção: % das falhas detectadas", color=vs.INK_2)
    ax.set_title("Como ler uma curva operacional (cada ponto é um limiar)", fontsize=12, loc="left")
    ax.legend(loc="lower right"), vs.style_axes(ax, grid="both")
    fig.tight_layout()  # o aviso "curvas inventadas" fica na legenda da figura, não na imagem
    fig.savefig(HERE / "como-ler-curva-operacional.png", dpi=200, facecolor=vs.SURFACE, bbox_inches="tight", pad_inches=0.2)  # margem: o rótulo 10^1 não pode ser cortado
    plt.close(fig)


def linha_do_tempo() -> None:
    vs.apply_style()
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.set_xlim(0, 100), ax.set_ylim(0, 10), ax.axis("off")
    ax.plot([2, 98], [4, 4], color=vs.INK_2, lw=3)
    ax.axvspan(2, 20, ymin=0.3, ymax=0.5, color=vs.STATUS[0], alpha=0.25)
    ax.text(11, 5.6, "janela de\ncalibração (H)", ha="center", fontsize=10, color="#12805a", fontweight="bold")
    marks = [(35, "t0: início da falha\n(oculto dos detectores)", RED, 7.0), (62, "primeiro alarme\n(detector)", ORANGE, 7.0), (92, "fim da série", vs.MUTED, 7.0)]
    for x, lab, c, y in marks:
        ax.plot([x, x], [3.4, 4.6], color=c, lw=3)
        ax.text(x, y, lab, ha="center", fontsize=10.5, color=c, fontweight="bold")
    ax.annotate("", xy=(62, 2.4), xytext=(35, 2.4), arrowprops=dict(arrowstyle="<->", color=vs.INK, lw=2))
    ax.text(48.5, 1.3, "atraso = ciclo do alarme − t0", ha="center", fontsize=10.5, color=vs.INK, fontweight="bold")
    ax.text(2, 9.4, "poder = fração das séries com falha em que sai um alarme depois de t0", fontsize=10.5, color=vs.INK_2)
    ax.text(2, 0.15, "alarmes antes de t0 são falsos alarmes e são cobrados nos controles (séries sem falha)", fontsize=10, color=vs.MUTED)
    fig.savefig(HERE / "linha-do-tempo-da-falha.png", dpi=200, facecolor=vs.SURFACE, bbox_inches="tight")
    plt.close(fig)


if __name__ == "__main__":
    curva()
    linha_do_tempo()
    print("ok")

### 2.6 Testes

Suíte de testes (sem os estudos longos).

21 arquivos.

In [ ]:
%%writefile projeto/tests/conftest.py
"""Pytest configuration and global hooks."""



def pytest_configure(config):
    config.addinivalue_line("markers", "requirement(id): Traceability marker for plan requirement IDs")
    config.addinivalue_line("markers", "docs: Documentation synchronization and freshness checks")
    config.addinivalue_line("markers", "slow: Slow end-to-end pipeline execution checks")

In [ ]:
%%writefile projeto/tests/factories.py
"""Construtores de dados sintéticos compartilhados entre os arquivos de teste."""

import numpy as np
import pandas as pd
from pdm_cmaps.domain.schema import COLUMNS, SENSOR_COLUMNS


def sample_frame(engine_count: int = 10) -> pd.DataFrame:
    rows = []
    for unit_id in range(1, engine_count + 1):
        for cycle in range(1, 6):
            row = {column: 0.0 for column in COLUMNS}
            row.update({"unit_id": unit_id, "cycle": cycle, "setting_1": unit_id % 3, "setting_2": 0.2, "setting_3": 100.0})
            for index, sensor in enumerate(SENSOR_COLUMNS):
                row[sensor] = float(unit_id + cycle * (index + 1))
            rows.append(row)
    return pd.DataFrame(rows)

def make_synthetic_cmapss(
    n_engines: int = 12,
    min_cycles: int = 60,
    max_cycles: int = 120,
    seed: int = 42,
) -> pd.DataFrame:
    """Generate realistic synthetic C-MAPSS dataset with diverse EOLs and 6 flight regimes."""
    rng = np.random.default_rng(seed)
    rows = []
    regime_settings = [
        (0.0, 0.0, 100.0),
        (10.0, 0.25, 100.0),
        (20.0, 0.70, 100.0),
        (25.0, 0.62, 60.0),
        (35.0, 0.84, 100.0),
        (42.0, 0.84, 100.0),
    ]
    for unit_id in range(1, n_engines + 1):
        eol = rng.integers(min_cycles, max_cycles + 1)
        for cycle in range(1, eol + 1):
            r_idx = rng.integers(0, len(regime_settings))
            s1, s2, s3 = regime_settings[r_idx]
            deg_factor = (cycle / eol) ** 2
            row = {
                "unit_id": unit_id,
                "cycle": cycle,
                "setting_1": s1 + rng.normal(0, 0.01),
                "setting_2": s2 + rng.normal(0, 0.005),
                "setting_3": s3,
            }
            # Physical sensors: T2, P2, P15, epr are boundary/inlet conditions
            row["T2"] = 518.67 - 3.5 * s1 + rng.normal(0, 0.2)
            row["P2"] = 14.696 * (1.0 - 0.02 * s1) + rng.normal(0, 0.05)
            row["P15"] = 14.0 + 0.1 * s2 + rng.normal(0, 0.02)
            row["epr"] = 1.0 + 0.3 * s2 + rng.normal(0, 0.01)
            row["Nf_dmd"] = 2388.0
            row["PCNfR_dmd"] = 100.0
            # Internal degraded sensors: T50, T24, P30 drift with deg_factor
            row["T24"] = 640.0 + 5.0 * s1 + 15.0 * deg_factor + rng.normal(0, 0.5)
            row["T30"] = 1580.0 + 10.0 * s1 + 25.0 * deg_factor + rng.normal(0, 1.0)
            row["T50"] = 1400.0 + 8.0 * s1 + 35.0 * deg_factor + rng.normal(0, 0.8)
            row["P30"] = 550.0 - 2.0 * s1 - 10.0 * deg_factor + rng.normal(0, 0.5)
            # Remaining sensor columns
            for sensor in SENSOR_COLUMNS:
                if sensor not in row:
                    row[sensor] = 100.0 + rng.normal(0, 1.0)
            rows.append(row)
    df = pd.DataFrame(rows)
    return df

def _toy_timeline(n_cycles: int = 80, anomaly_from: int | None = 40):
    import numpy as np
    import pandas as pd
    from pdm_cmaps.monitoring.alerts import CriticalityPolicy, build_alert_timeline

    cycles = np.arange(1, n_cycles + 1)
    rul_true = np.clip(n_cycles - cycles, 0, 125).astype(float)
    rng = np.random.default_rng(0)
    pred = np.clip(rul_true + rng.normal(0, 4, n_cycles), 0, 125)
    frame = pd.DataFrame({
        "unit_id": 7, "cycle": cycles, "RUL_true": rul_true, "RUL_pred": pred,
        "RUL_lower": np.clip(pred - 10, 0, 125), "RUL_upper": np.clip(pred + 10, 0, 125),
    })
    policy = CriticalityPolicy()
    return build_alert_timeline(frame, policy, {7: anomaly_from}), policy

In [ ]:
%%writefile projeto/tests/test_analysis.py
"""Estatística: testes de hipótese, diagnósticos e importância de atributos."""

import warnings

import numpy as np
import pandas as pd
from pdm_cmaps.analysis.stats import (
    ThesisHypothesisAnalyzer,
    distribution_shift,
    feature_signal,
    permutation_importances,
)
from pdm_cmaps.domain.rul import (
    compute_piecewise_rul,
)

from factories import make_synthetic_cmapss, sample_frame


def test_feature_signal_handles_constant_features_without_runtime_warning():
    frame = sample_frame(engine_count=3)
    frame["RUL"] = compute_piecewise_rul(frame)
    frame["constant"] = 1.0
    with warnings.catch_warnings():
        warnings.simplefilter("error", RuntimeWarning)
        result = feature_signal(frame, ["constant", "cycle"], "test")
    constant_correlation = result.loc[result["Feature"] == "constant", "Abs_correlation"].iloc[0]
    assert np.isnan(constant_correlation)
    assert np.isfinite(result.loc[result["Feature"] == "cycle", "Abs_correlation"]).all()


def test_thesis_hypothesis_rigor():
    """Verify H1 engine-block Friedman, H2 n=62 relative window, H3 fleet Ljung-Box, H4/H5 Wilcoxon."""
    analyzer = ThesisHypothesisAnalyzer(random_state=42)
    df = make_synthetic_cmapss(n_engines=12, min_cycles=60, max_cycles=100, seed=42)
    df["regime_id"] = (df["setting_1"] // 10).astype(int)

    # H1: Engine-block repeated measures
    h1 = analyzer.test_regime_variance_dominance(df, sensor="T50", regime_col="regime_id")
    assert h1.is_rejected or h1.p_value <= 0.05
    assert "Friedman" in h1.test_name or "Alexander" in h1.test_name or "ANOVA" in h1.test_name

    # H2: Survivor-bias-free relative window
    df["residual"] = df["T50"] - (1400.0 + 8.0 * df["setting_1"])
    h2 = analyzer.test_physical_degradation_shift(df, healthy_cutoff=30, residual_col="residual")
    assert h2.is_rejected
    assert h2.p_value < 0.05
    assert "n=12" in h2.test_name or h2.effect_size is not None

    # H3: Fleet Ljung-Box
    h3 = analyzer.test_white_noise_residual_autocorrelation(df["residual"].to_numpy())
    assert h3.test_statistic > 0

    # H4 & H5: Wilcoxon tests
    y_true = np.linspace(100, 0, 100)
    pred_base = np.full(100, 50.0)
    pred_model = y_true + np.random.normal(0, 5.0, 100)
    unit_ids = np.repeat(np.arange(1, 11), 10)
    h4 = analyzer.test_model_superiority_vs_baseline(y_true, pred_model, pred_base, "ModelX", unit_ids=unit_ids)
    assert h4.is_rejected

    # BCa Confidence Intervals export observed metric
    test_eval = pd.DataFrame({
        "unit_id": unit_ids,
        "RUL_actual": y_true,
        "ModelX": pred_model,
    })
    ci_df = analyzer.compute_bootstrap_confidence_intervals(test_eval, {"ModelX": "ModelX"}, target_col="RUL_actual", n_bootstraps=50)
    assert "R2_Observed" in ci_df.columns or "R2_Mean" in ci_df.columns


def test_diagnostics_subsystem_permutation_and_wasserstein():
    """Verify grouped subsystem permutation importance and Wasserstein-1 distance."""
    df = make_synthetic_cmapss(n_engines=10, seed=42)
    df["RUL"] = compute_piecewise_rul(df)
    features = ["setting_1", "setting_2", "T2", "P2", "T50", "P30"]

    # Subsystem permutation importances
    p_imp = permutation_importances(df, features, "FD004", random_state=42)
    assert isinstance(p_imp, pd.DataFrame)
    assert "R2_drop_mean" in p_imp.columns

    # Distribution shift with Wasserstein distance
    df_high = df.iloc[: len(df) // 2]
    df_low = df.iloc[len(df) // 2 :]
    shift = distribution_shift(df_high, df_low, ["T50", "P30"], "High", "Low")
    assert "Wasserstein_distance" in shift.columns or "KS_statistic" in shift.columns

In [ ]:
%%writefile projeto/tests/test_benchmark_estatistica.py
"""Teste pareado por motor do benchmark de RUL (experiments/benchmark_repeated.engine_level_tests)."""
from __future__ import annotations

import numpy as np
import pandas as pd

from pdm_cmaps.experiments.benchmark_repeated import engine_level_tests, merge_lanes


def _errors(n_seeds, n_engines=30, delta=2.0, seed=0):
    rng = np.random.default_rng(seed)
    rows = []
    for sd in range(n_seeds):
        for u in range(n_engines):
            base = rng.gamma(4, 2)
            for model, add in (("XGBoost (+temporal)", 0.0), ("PyTorch CNN", delta)):
                rows.append({"Protocol": "official", "Subset": "FDX", "Seed": sd, "Model": model, "unit_id": u, "abs_err": base + add + rng.normal(0, 0.5)})
    return pd.DataFrame(rows)


def test_each_engine_counts_once_however_many_seeds():
    """Repetir as sementes não pode inflar o número de unidades nem encolher o p-valor: um ponto por motor."""
    few, many = engine_level_tests(_errors(1)), engine_level_tests(_errors(5))
    for t in (few, many):
        row = t[t["Comparison"].str.endswith("PyTorch CNN")].iloc[0]
        assert row["n_engines"] == 30
    assert many.iloc[0]["Seeds_per_engine_mean"] == 5 and few.iloc[0]["Seeds_per_engine_mean"] == 1
    assert many.iloc[0]["Median_err_diff_other_minus_ref"] > 1.5          # a diferença real (2,0) aparece
    assert few.iloc[0]["p_wilcoxon_engines"] < 1e-3 and many.iloc[0]["p_wilcoxon_engines"] < 1e-3


def test_no_difference_gives_a_large_p_value():
    t = engine_level_tests(_errors(3, delta=0.0), others=["PyTorch CNN"])
    assert t.iloc[0]["p_wilcoxon_engines"] > 0.05


def test_merge_lanes_requires_every_lane(tmp_path):
    for sub in ("FD001", "FD002"):
        (tmp_path / sub).mkdir()
        for name in ("benchmark_rows.csv", "benchmark_engine_errors.csv"):
            pd.DataFrame({"Subset": [sub], "x": [1]}).to_csv(tmp_path / sub / name, index=False)
    merge_lanes(tmp_path, ("FD001", "FD002"))
    assert len(pd.read_csv(tmp_path / "benchmark_rows.csv")) == 2
    try:
        merge_lanes(tmp_path, ("FD001", "FD002", "FD003"))
    except FileNotFoundError as e:
        assert "FD003" in str(e)
    else:
        raise AssertionError("faltou erro para raia ausente")


def test_merge_lanes_accepts_several_folders_per_subset_and_rejects_duplicates(tmp_path):
    def write(folder, seed, model="M"):
        (tmp_path / folder).mkdir(exist_ok=True)
        pd.DataFrame({"Subset": ["FD002"], "Seed": [seed], "Protocol": ["holdout"], "Model": [model]}).to_csv(tmp_path / folder / "benchmark_rows.csv", index=False)
        pd.DataFrame({"Subset": ["FD002"], "Seed": [seed], "Protocol": ["holdout"], "Model": [model], "unit_id": [1]}).to_csv(tmp_path / folder / "benchmark_engine_errors.csv", index=False)
    write("FD002", 42)
    write("FD002_s12", 1)
    merge_lanes(tmp_path, ("FD002",))
    assert sorted(pd.read_csv(tmp_path / "benchmark_rows.csv")["Seed"]) == [1, 42]
    write("FD002_s12", 42)  # mesma semente nas duas pastas
    import pytest
    with pytest.raises(ValueError, match="duplicadas"):
        merge_lanes(tmp_path, ("FD002",))

In [ ]:
%%writefile projeto/tests/test_data.py
"""Leitura, divisão por motor (sem vazamento) e atributos causais."""

import numpy as np
import pandas as pd
import pytest
from pdm_cmaps.data.features import (
    CausalFeatureEngineer,
    engineer_causal_temporal_features,
)
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.data.splits import split_engine_units
from pdm_cmaps.domain.schema import COLUMNS
from pdm_cmaps.modeling.registry import get_model

from factories import make_synthetic_cmapss, sample_frame


def test_loader_reads_named_fd001_schema():
    frame = CMapssLoader().load("FD001")
    assert list(frame.columns) == COLUMNS
    assert frame["unit_id"].nunique() == 100


def test_engine_holdout_has_no_shared_units():
    frame = sample_frame()
    train, test = split_engine_units(frame, test_ratio=0.3, random_state=4)
    assert not set(train).intersection(test)
    assert len(train) + len(test) == frame["unit_id"].nunique()


def test_temporal_features_do_not_cross_engine_boundaries():
    frame = sample_frame(engine_count=2)
    features = engineer_causal_temporal_features(frame, ["T50"], windows=(3,))
    first_rows = features.groupby("unit_id", sort=False).head(1)
    assert (first_rows["T50_lag1"] == first_rows["T50"]).all()
    assert (first_rows["T50_delta"] == 0.0).all()


def test_loader_rejects_duplicate_engine_cycle_rows():
    frame = sample_frame(engine_count=2)
    duplicated = pd.concat([frame, frame.iloc[[0]]], ignore_index=True)
    with pytest.raises(ValueError, match="Duplicate engine-cycle"):
        CMapssLoader.validate(duplicated)


def test_split_engine_units_eol_stratified():
    """Verify EOL-stratified holdout split produces matching EOL distributions without leakage."""
    df = make_synthetic_cmapss(n_engines=24, min_cycles=50, max_cycles=150, seed=42)
    train_units, test_units = split_engine_units(df, test_ratio=0.25, random_state=42, stratify_eol=True)
    assert len(set(train_units).intersection(test_units)) == 0
    assert len(train_units) + len(test_units) == df["unit_id"].nunique()

    # Compare EOL medians between train and test
    eol_map = df.groupby("unit_id")["cycle"].max()
    train_eols = eol_map.loc[train_units]
    test_eols = eol_map.loc[test_units]
    # KS-test or median difference should confirm similar distributions
    assert abs(train_eols.median() - test_eols.median()) < 25.0


def test_lag2_boundary_fix_at_cycle_2():
    """Verify lag2 at t=2 is filled with lag1 (x at t=1), not current x at t=2."""
    df = pd.DataFrame({
        "unit_id": [1, 1, 1],
        "cycle": [1, 2, 3],
        "T50": [100.0, 110.0, 120.0],
    })
    res = engineer_causal_temporal_features(df, ["T50"], windows=(3,))
    # At t=1: lag1=100, lag2=100
    assert res.loc[0, "T50_lag1"] == 100.0
    assert res.loc[0, "T50_lag2"] == 100.0
    # At t=2: lag1=100, lag2 must be 100 (filled with lag1), NOT 110!
    assert res.loc[1, "T50_lag1"] == 100.0
    assert res.loc[1, "T50_lag2"] == 100.0
    # At t=3: lag1=110, lag2=100
    assert res.loc[2, "T50_lag1"] == 110.0
    assert res.loc[2, "T50_lag2"] == 100.0


def test_causal_feature_engineer_startup_and_imputation():
    """Verify CausalFeatureEngineer provides is_startup indicator and healthy-median rollstd imputation."""
    train_df = make_synthetic_cmapss(n_engines=6, seed=42)
    test_df = make_synthetic_cmapss(n_engines=2, seed=99)
    engineer = CausalFeatureEngineer(sensor_cols=["T50", "T24"], windows=(3, 5))
    engineer.fit(train_df, healthy_cycle_limit=30)
    transformed = engineer.transform(test_df)
    assert "is_startup" in transformed.columns
    # cycles 1 and 2 should have is_startup == 1.0
    assert (transformed.loc[transformed["cycle"] <= 2, "is_startup"] == 1.0).all()
    assert (transformed.loc[transformed["cycle"] > 2, "is_startup"] == 0.0).all()
    # rollstd at cycle 1 should be imputed with healthy median, not rigid 0.0
    cycle1_std = transformed.loc[transformed["cycle"] == 1, "T50_rollstd_3"].to_numpy()
    assert (cycle1_std > 0.0).all()


def test_data_loader_load_test():
    """Verify CMapssLoader.load_test loads official test and RUL files."""
    loader = CMapssLoader()
    # Check if FD004 test exists on disk
    test_file = loader.data_dir / "test_FD004.txt"
    rul_file = loader.data_dir / "RUL_FD004.txt"
    if test_file.is_file() and rul_file.is_file():
        test_df, rul_series = loader.load_test("FD004")
        assert list(test_df.columns) == COLUMNS
        assert len(rul_series) == test_df["unit_id"].nunique()
        assert (rul_series >= 0).all()


def test_notebook_insights_slopes_and_nasa_objective():
    """Notebook Insights: Verify rolling causal slopes and XGBoost NASA asymmetric objective."""
    df = pd.DataFrame({
        "unit_id": [1] * 25,
        "cycle": np.arange(1, 26),
        "T50": np.linspace(1400.0, 1450.0, 25),
    })
    feat_eng = CausalFeatureEngineer(sensor_cols=["T50"], windows=(5, 10))
    feat_eng.fit(df, healthy_cycle_limit=10)
    trans = feat_eng.transform(df)

    assert "T50_slope_5" in trans.columns
    assert "T50_slope_10" in trans.columns
    # T50 is linearly increasing, slope should be positive
    assert (trans.loc[trans["cycle"] > 10, "T50_slope_5"] > 0).all()

    # Test get_model("xgboost_nasa_loss")
    xgb_nasa = get_model("xgboost_nasa_loss", n_estimators=5, max_depth=2, random_state=42)
    X = np.random.randn(30, 4)
    y = np.random.uniform(10, 100, 30)
    xgb_nasa.fit(X, y)
    preds = xgb_nasa.predict(X)
    assert len(preds) == len(y)
    assert np.isfinite(preds).all()

In [ ]:
%%writefile projeto/tests/test_detection_regime.py
"""Desacoplamento de regimes operacionais."""


import numpy as np
import pandas as pd
from pdm_cmaps.detection.regime import RegimeDecoupler

from factories import make_synthetic_cmapss, sample_frame


def test_variance_decomposition_adds_to_total():
    frame = sample_frame()
    frame["regime_id"] = frame["unit_id"] % 3
    result = RegimeDecoupler.variance_decomposition(frame, "T50")
    assert np.isclose(result["total_var"], result["between_regime_var"] + result["within_regime_var"])
    assert np.isclose(result["pct_between_regime"] + result["pct_within_regime"], 100.0)
    assert result["Between_regime_pct"] == result["pct_between_regime"]
    assert result["Within_regime_pct"] == result["pct_within_regime"]
    assert 0.0 <= result["regime_baseline_r2"] <= 1.0


def test_regime_k_sweep_diagnostics():
    """Verify empirical k-selection evaluation (Silhouette, Calinski-Harabasz, BIC)."""
    df = make_synthetic_cmapss(n_engines=8, seed=42)
    decoupler = RegimeDecoupler()
    sweep = decoupler.evaluate_k_sweep(df, k_range=range(2, 6))
    assert isinstance(sweep, pd.DataFrame)
    assert "k" in sweep.columns
    assert "silhouette" in sweep.columns
    assert "calinski_harabasz" in sweep.columns
    assert "bic" in sweep.columns
    assert len(sweep) == 4
    assert (sweep["silhouette"] >= -1.0).all() and (sweep["silhouette"] <= 1.0).all()
    assert (sweep["calinski_harabasz"] > 0).all()


def test_regime_lexicographical_centroid_sorting():
    """Verify cluster ordering is physical and lexicographically stable across runs."""
    df = make_synthetic_cmapss(n_engines=10, seed=42)
    r1 = RegimeDecoupler(n_clusters=6, random_state=42).fit(df)
    r2 = RegimeDecoupler(n_clusters=6, random_state=123).fit(df)
    # Centroid coordinates sorted by (altitude, mach, TRA) should be strictly monotonic in altitude
    centers1 = r1.kmeans.cluster_centers_
    centers_phys1 = r1.scaler.inverse_transform(centers1)
    # Check that regime_ids map to lexicographically ordered physical centers
    ordered_centers = np.zeros_like(centers_phys1)
    for cl_idx, r_id in r1.cluster_to_regime_id.items():
        ordered_centers[r_id] = centers_phys1[cl_idx]
    # Altitude (setting_1) should be non-decreasing across ordered regime IDs
    assert np.all(np.diff(ordered_centers[:, 0]) >= -1e-3)
    # ... and the same must hold for another random seed: the ordering is a property of the data, not of the initialisation
    phys2 = r2.scaler.inverse_transform(r2.kmeans.cluster_centers_)
    ordered2 = np.zeros_like(phys2)
    for cl_idx, r_id in r2.cluster_to_regime_id.items():
        ordered2[r_id] = phys2[cl_idx]
    assert np.all(np.diff(ordered2[:, 0]) >= -1e-3)


def test_regime_distance_margin_and_relative_margin():
    """Verify distance_margin includes absolute and dimensionless relative margins."""
    df = make_synthetic_cmapss(n_engines=4, seed=42)
    decoupler = RegimeDecoupler(n_clusters=6, random_state=42).fit(df)
    margins = decoupler.distance_margin(df)
    assert "nearest_distance" in margins.columns
    assert "runner_up_distance" in margins.columns
    assert "distance_margin" in margins.columns
    assert "relative_margin" in margins.columns
    assert (margins["nearest_distance"] <= margins["runner_up_distance"] + 1e-6).all()
    assert (margins["distance_margin"] >= -1e-6).all()
    assert (margins["relative_margin"] >= -1e-6).all() and (margins["relative_margin"] <= 1.0 + 1e-6).all()


def test_regime_standardization_and_out_of_sample_variance():
    """Verify regime-wise standardization and clean separation of descriptive vs OOS variance."""
    train_df = make_synthetic_cmapss(n_engines=8, seed=42)
    test_df = make_synthetic_cmapss(n_engines=4, seed=99)
    decoupler = RegimeDecoupler(n_clusters=6, random_state=42).fit(train_df)
    train_trans = decoupler.transform(train_df)
    test_trans = decoupler.transform(test_df)

    # Standardize by regime
    decoupler.fit_sensor_scaler(train_trans, ["T50", "P30"])
    std_train = decoupler.standardize_sensors(train_trans, ["T50", "P30"])
    # Per-regime training mean of standardized features should be ~0.0
    for r_id in train_trans["regime_id"].unique():
        sub = std_train.loc[std_train["regime_id"] == r_id, "T50_norm"]
        assert abs(sub.mean()) < 1e-3
        assert abs(sub.std() - 1.0) < 1e-2 or sub.std() == 0.0

    # Variance decomposition separating descriptive vs OOS
    decomp = RegimeDecoupler.variance_decomposition(test_trans, "T50", reference_frame=train_trans)
    assert "descriptive_between_pct" in decomp or "pct_between_regime" in decomp
    assert "oos_r2" in decomp or "regime_baseline_r2" in decomp
    assert decomp["baseline_evaluation"] == "out_of_sample"

In [ ]:
%%writefile projeto/tests/test_detection_spc.py
"""Cartas de controle (EWMA, 3σ, envelope), MEWMA e índice de saúde."""


import numpy as np
import pandas as pd
from pdm_cmaps.detection.health_index import HealthIndexFuser
from pdm_cmaps.detection.regime import RegimeDecoupler
from pdm_cmaps.detection.spc import EnvelopeAnomalyDetector, StatisticalAnomalyDetector
from pdm_cmaps.detection.virtual_sensor import VirtualSensor

from factories import make_synthetic_cmapss


def test_envelope_anomaly_detector():
    from pdm_cmaps.detection.spc import EnvelopeAnomalyDetector

    detector = EnvelopeAnomalyDetector(k_sigma=2.5, healthy_cycle_limit=30, window_size=4, min_consecutive_breaches=2)
    np.random.seed(42)
    healthy_actual = 1400.0 + np.random.randn(30) * 1.5
    healthy_pred = 1400.0 + np.random.randn(30) * 0.2
    detector.calibrate(healthy_actual, healthy_pred)

    assert detector.sigma_e > 0
    assert detector.mu_amp > 0

    cycles = np.arange(1, 101)
    actual = 1400.0 + np.random.randn(100) * 1.5
    # Add severe thermal runaway / degradation starting at cycle 75
    actual[74:] += np.linspace(4.0, 30.0, 26)
    pred = np.full(100, 1400.0)

    df = pd.DataFrame({"unit_id": 1, "cycle": cycles, "T50": actual, "pred": pred})
    scored_env, summary_env = detector.score_time_series_envelope(df, "T50", "pred")

    assert "envelope_upper" in scored_env.columns
    assert "envelope_lower" in scored_env.columns
    assert summary_env["detected"] is True
    assert summary_env["first_alarm_cycle"] is not None
    assert summary_env["first_alarm_cycle"] >= 70
    assert summary_env["eol_cycle"] == 100
    assert summary_env["lead_time_cycles"] == 100 - summary_env["first_alarm_cycle"]

    scored_geom, summary_geom = detector.score_geometric_envelope(df, "T50")
    assert "geom_upper_hull" in scored_geom.columns
    assert "geom_lower_hull" in scored_geom.columns
    assert summary_geom["detected"] is True
    assert summary_geom["first_alarm_cycle"] is not None


def test_online_alarm_restriction_and_disaggregated_metrics():
    """Verify operational alarm is restricted to online monitoring phase (cycle > H)."""
    detector = StatisticalAnomalyDetector(healthy_cycle_limit=50, threshold_sigma=3.0)
    # Calibrate on healthy residuals
    np.random.seed(42)
    healthy_res = np.random.randn(200) * 0.5
    detector.calibrate(healthy_res)

    # Engine with early spike at cycle 10, but true failure starting at cycle 80
    cycles = np.arange(1, 101)
    res = np.random.randn(100) * 0.5
    res[9] = 10.0  # Spike at cycle 10 (healthy phase, cycle <= 50)
    res[79:] += np.linspace(2.0, 10.0, 21)  # Degraded phase (cycle > 50)

    engine_df = pd.DataFrame({"unit_id": 1, "cycle": cycles, "residual": res})
    scored, summary = detector.score_engine(engine_df)

    assert "first_alarm_cycle" in summary
    assert "premature_alarm_pct" in summary or "premature_alarm" in summary
    # The operational first alarm must occur during online monitoring (> 50)
    assert summary["first_alarm_cycle"] > 50
    assert summary["detected"] is True
    assert summary["healthy_far"] > 0.0 or summary["healthy_far_pct"] > 0.0


def test_envelope_regime_standardized_corridor():
    """Verify EnvelopeAnomalyDetector handles multi-regime FD004 without corridor collapse."""
    detector = EnvelopeAnomalyDetector(healthy_cycle_limit=30, k_sigma=3.0)
    # Calibrate
    actual = 1400.0 + np.random.randn(50) * 2.0
    pred = 1400.0 + np.random.randn(50) * 0.5
    detector.calibrate(actual, pred)

    # Test with regime-normalized residuals
    cycles = np.arange(1, 80)
    res = np.random.randn(79) * 0.5
    res[50:] += np.linspace(1.0, 8.0, 29)
    df = pd.DataFrame({"unit_id": 1, "cycle": cycles, "residual": res, "T50": 1400.0 + res, "pred": 1400.0})
    scored, summary = detector.score_time_series_envelope(df, "T50", "pred")
    assert summary["detected"] is True
    assert summary["first_alarm_cycle"] > 30


def test_todo1_spc_persistence_eliminates_transient_spikes():
    """TODO-1 & F2: Isolated transient spikes do not trigger first_alarm_cycle with min_consecutive_breaches=3."""
    detector = StatisticalAnomalyDetector(healthy_cycle_limit=50, threshold_sigma=3.0, ewma_alpha=0.10, min_consecutive_breaches=3)
    np.random.seed(42)
    healthy_res = np.random.randn(200) * 0.2
    detector.calibrate(healthy_res)

    cycles = np.arange(1, 101)
    res = np.random.randn(100) * 0.2
    # Single isolated massive transient spike at cycle 65 (cycle > 50)
    res[64] = 15.0

    engine_df = pd.DataFrame({"unit_id": 1, "cycle": cycles, "residual": res})
    scored, summary = detector.score_engine(engine_df)
    # The isolated spike at cycle 65 should NOT trigger an alarm because it lasts only 1 cycle
    assert summary["detected"] is False
    assert summary["first_alarm_cycle"] is None

    # Now add 3 consecutive breaches at cycles 80, 81, 82
    res[79:83] = 12.0
    engine_df2 = pd.DataFrame({"unit_id": 1, "cycle": cycles, "residual": res})
    scored2, summary2 = detector.score_engine(engine_df2)
    assert summary2["detected"] is True
    assert summary2["first_alarm_cycle"] == 80  # Offsets to the start of the 3-cycle sequence


def test_todo2_ewma_alpha_default_and_inertia():
    """TODO-2 & F3: Verify default ewma_alpha is 0.10 across detector classes."""
    det = StatisticalAnomalyDetector()
    assert det.ewma_alpha == 0.10

    vs = VirtualSensor()
    assert vs.ewma_alpha == 0.10
    assert vs.min_consecutive_breaches == 3


def test_todo3_geometric_envelope_decoupled_multiregime():
    """TODO-3 & F1: Geometric envelope over decoupled signal restores lead time in multi-regime jumps."""
    detector = EnvelopeAnomalyDetector(healthy_cycle_limit=30, k_sigma=3.0, min_consecutive_breaches=2)
    # Calibrate on healthy residual variance
    actual = 1400.0 + np.random.randn(50) * 0.5
    pred = np.full(50, 1400.0)
    detector.calibrate(actual, pred)

    # Multi-regime jumps: BPR oscillates between 8.4 and 10.8 due to altitude jumps
    cycles = np.arange(1, 101)
    altitude_jump = np.tile([0.0, 2.4], 50)
    base_bpr = 8.4 + altitude_jump
    # Add real mechanical degradation starting at cycle 75
    deg = np.zeros(100)
    deg[74:] = np.linspace(0.5, 3.0, 26)
    actual_bpr = base_bpr + deg
    # Virtual sensor predicts the altitude jump accurately
    pred_bpr = base_bpr

    df = pd.DataFrame({
        "unit_id": 179,
        "cycle": cycles,
        "BPR": actual_bpr,
        "prediction": pred_bpr,
        "residual": actual_bpr - pred_bpr,
    })

    # Decoupled mode detects the true degradation
    scored, summary = detector.score_geometric_envelope(df, "BPR", use_decoupled_signal=True)
    assert summary["detected"] is True
    assert summary["first_alarm_cycle"] is not None
    assert summary["first_alarm_cycle"] >= 74
    assert summary["lead_time_cycles"] > 0
    assert summary["decoupled_mode"] == "residual"


def test_todo5_health_index_fuser():
    """TODO-5 & F7: HealthIndexFuser synthesizes multi-sensor degradation into monotonic [1.0 -> 0.0] HI."""
    df = make_synthetic_cmapss(n_engines=8, seed=42)
    decoupler = RegimeDecoupler(n_clusters=6, random_state=42).fit(df)
    df_trans = decoupler.transform(df)
    decoupler.fit_sensor_scaler(df_trans, ["T50", "P30", "T24"])
    df_norm = decoupler.standardize_sensors(df_trans, ["T50", "P30", "T24"])

    fuser = HealthIndexFuser(healthy_cycle_limit=30, ewma_alpha=0.10, random_state=42)
    scored, per_engine, summary = fuser.evaluate(df_norm, df_norm)

    assert "health_index" in scored.columns
    assert (scored["health_index"] >= 0.0).all() and (scored["health_index"] <= 1.0).all()

    # Verify healthy phase has high HI and end-of-life has lower HI
    for _, eng in scored.groupby("unit_id"):
        healthy_hi = eng.loc[eng["cycle"] <= 30, "health_index"].mean()
        eol_hi = eng.tail(5)["health_index"].mean()
        assert healthy_hi > eol_hi


def test_multivariate_monitor_alarms_on_drift_not_on_healthy_engines():
    import numpy as np
    import pandas as pd
    from pdm_cmaps.detection.multivariate import MultivariateResidualMonitor

    rng = np.random.default_rng(3)
    rows = []
    for unit in range(1, 13):
        drift = unit > 8  # engines 9-12 degrade after cycle 70
        for cycle in range(1, 121):
            base = {"unit_id": unit, "cycle": cycle, "regime_id": 0, "setting_1": 0.0, "setting_2": 0.0, "setting_3": 100.0,
                    "T2": 518.67, "P2": 14.62, "P15": 21.6, "epr": 1.3, "Nf_dmd": 2388.0, "PCNfR_dmd": 100.0}
            shift = 0.08 * max(0, cycle - 70) if drift else 0.0
            for j, s in enumerate(("T24", "T30", "T50", "P30")):
                base[s] = 100.0 + j + rng.normal(0, 1.0) + (shift if s in ("T50", "P30") else 0.0)
            rows.append(base)
    df = pd.DataFrame(rows)
    train = df.loc[df["unit_id"] <= 8]
    test = df.loc[df["unit_id"] > 4]
    # engine_offset=False: aqui se testa o limite NOMINAL. Com o deslocamento ligado (padrão do protocolo principal) o limite calibrado em janelas já
    # centradas fica baixo e motores saudáveis alarmam (3 de 4 neste exemplo); a MEWMA com o deslocamento só vale a falso alarme igualado.
    mon = MultivariateResidualMonitor(sensors=("T24", "T30", "T50", "P30"), healthy_cycle_limit=50, engine_offset=False).fit(train)
    first, per, summary = mon.alarms(mon.score(test))
    assert all(first[u] is not None for u in (9, 10, 11, 12))
    assert sum(first[u] is not None for u in (5, 6, 7, 8)) <= 1


def test_first_persistent_requires_k_consecutive_breaches_after_window():
    import numpy as np
    from pdm_cmaps.reporting.plots import _first_persistent

    cycles = np.arange(1, 11)
    mask = np.array([1, 1, 0, 0, 1, 0, 1, 1, 1, 0], dtype=bool)
    assert _first_persistent(mask, cycles, k=2, after=0) == 1
    assert _first_persistent(mask, cycles, k=2, after=2) == 7
    assert _first_persistent(mask, cycles, k=4, after=0) is None


def test_envelope_runs_ignore_isolated_breaches_and_alert_on_persistent_ones():
    from pdm_cmaps.experiments import envelope_ignored_cases as eic

    cyc = np.arange(1, 121)
    breach = np.zeros(120, bool)
    breach[[10, 60, 90]] = True        # cycle 11 inside the calibration window; 61 and 91 isolated after it
    breach[100:103] = True              # persistent run (3 cycles) -> alert
    r = pd.DataFrame({"Subset": "FD001", "unit_id": 1, "cycle": cyc, "breach": breach, "excess_sigma": np.where(breach, 0.5, 0.0), "RUL": 125.0})
    runs = eic.breach_runs(r, persist=2, h=50)
    assert runs["length"].tolist() == [1, 1, 1, 3]
    assert runs["monitored"].tolist() == [False, True, True, True]                       # first one is in the calibration window
    assert runs["alert"].tolist() == [False, False, False, True]                         # only the persistent run alerts
    assert runs.loc[runs["alert"], "alert_cycle"].iloc[0] == 101 + 1                     # issued when the 2nd breach arrives (run starts at cycle 101)
    summ = eic.summarize(r.assign(T50=0.0), runs, persist=2, h=50)
    assert summ.loc[0, "runs_ignored"] == 2 and summ.loc[0, "runs_alert"] == 1 and summ.loc[0, "alert_delay_cycles_by_rule"] == 1

In [ ]:
%%writefile projeto/tests/test_detectores_comparacao.py
"""Detectores de comparação do estudo de injeção (EWMA clássica com sinal e Isolation Forest): fórmulas e propriedades."""
from __future__ import annotations

import numpy as np

from pdm_cmaps.experiments.fault_detection_study import EXTRA_DETECTORS, iforest_ratio, signed_ewma_ratio


def test_signed_ewma_is_zero_without_residual_and_symmetric_in_sign():
    assert np.allclose(signed_ewma_ratio(np.zeros(30)), 0.0)
    z = np.random.default_rng(0).normal(size=200)
    assert np.allclose(signed_ewma_ratio(z), signed_ewma_ratio(-z))


def test_signed_ewma_matches_the_closed_form_limit_for_a_constant_shift():
    lam, L, c = 0.10, 3.0, 1.5
    r = signed_ewma_ratio(np.full(80, c), lam, L)
    t = np.arange(1, 81)
    w = c * (1 - (1 - lam) ** t)                      # EWMA de uma constante partindo de zero
    limit = L * np.sqrt(lam / (2 - lam) * (1 - (1 - lam) ** (2 * t)))
    assert np.allclose(r, w / limit)
    assert r[-1] > 1.0 > r[0]                         # um deslocamento de 1,5 sigma acaba fora de controle, mas não no primeiro ciclo


def test_signed_ewma_rarely_alarms_on_white_noise():
    r = signed_ewma_ratio(np.random.default_rng(1).normal(size=5000))
    assert (r >= 1.0).mean() < 0.02


def test_iforest_ratio_is_zero_at_the_center_and_large_for_an_outlier():
    from sklearn.ensemble import IsolationForest

    rng = np.random.default_rng(2)
    cal = rng.normal(size=(2000, 3))
    model = IsolationForest(n_estimators=100, random_state=0).fit(cal)
    r = iforest_ratio(model, cal, np.array([[0.0, 0.0, 0.0], [8.0, 8.0, 8.0]]))
    assert r[0] < 0.2 and r[1] > 1.0
    assert (iforest_ratio(model, cal, cal) >= 1.0).mean() < 0.01   # quantil 1 - 0,0027 da própria calibração


def test_extra_detectors_stay_out_of_the_main_protocol():
    from pdm_cmaps.experiments.fault_detection_study import DETECTORS

    assert set(EXTRA_DETECTORS).isdisjoint(DETECTORS) and len(EXTRA_DETECTORS) == 2


def test_bootstrap_handles_the_extra_detectors_and_their_contrasts(tmp_path):
    """Corrida sintética com os 7 detectores: os contrastes da configuração `extras` saem com diferença 0 e p = 1 (todos iguais)."""
    import pandas as pd

    from pdm_cmaps.experiments.fault_detection_study import DETECTORS, S_GRID
    from pdm_cmaps.experiments.poder_bootstrap import CONTRASTS_EXTRAS, FAMILIES_EXTRAS, Run, _slug, bootstrap, summarize

    H, rows, bid = 25, [], 0
    for rec in range(1, 9):
        for _ in range(4):
            bid += 1
            rows.append({"blind_id": bid, "recipient": rec, "role": "controle", "kind": "", "magnitude": np.nan, "t0": 0})
        for kind, mag in (("degrau", 0.10), ("variancia", 0.5)):
            bid += 1
            rows.append({"blind_id": bid, "recipient": rec, "role": "falha", "kind": kind, "magnitude": mag, "t0": H + 10})
    meta = pd.DataFrame(rows)
    arrays, ctrl = {}, (meta["role"] == "controle").to_numpy()
    for d in (*DETECTORS, *EXTRA_DETECTORS):
        mat = np.full((len(meta), len(S_GRID)), np.nan)
        mat[np.ix_(ctrl, S_GRID < 0.5)] = H + 50.0
        mat[np.ix_(~ctrl, S_GRID < 3.0)] = H + 15.0
        slug = _slug(d)
        arrays[f"{slug}__ids"], arrays[f"{slug}__mat"], arrays[f"{slug}__last"] = meta["blind_id"].to_numpy(), mat, np.full(len(meta), H + 100.0)
    runs = []
    for seed in (1, 2):
        path = tmp_path / f"FD004_seed{seed}_H{H}.npz"
        meta.assign(Subset="FD004", Seed=seed, H=H).to_csv(tmp_path / f"FD004_seed{seed}_H{H}_manifesto.csv", index=False)
        np.savez_compressed(path, **arrays)
        runs.append(Run(path))
    assert len(runs[0].names) == len(DETECTORS) + len(EXTRA_DETECTORS)
    power, contr = summarize(runs, bootstrap(runs, n_boot=40, seed=5), CONTRASTS_EXTRAS, FAMILIES_EXTRAS)
    assert set(contr["Contraste"]) == set(CONTRASTS_EXTRAS) and set(power["Detector"]) >= set(EXTRA_DETECTORS)
    assert (contr["diferenca_pp"] == 0.0).all() and (contr["p_boot"] == 1.0).all()
    assert contr.loc[contr["familia"], "p_holm"].eq(1.0).all()

In [ ]:
%%writefile projeto/tests/test_determinismo_redes.py
"""As redes PyTorch repetem as previsões bit a bit, qualquer que seja a contagem de threads do processo (ver modeling/determinism.py).

Medido antes da correção (FD001, semente 42): CNN 18,43 → 18,77 só por mudar de 2 para 4 threads, e LSTM 19,18 contra 16,68 em duas
execuções idênticas. Estes testes usam dados sintéticos pequenos e poucas épocas (rápidos); a verificação entre processos, com
OMP_NUM_THREADS diferentes e dados reais, está em results/logs/determinismo/.
"""
from __future__ import annotations

import numpy as np
import pytest
import torch

from pdm_cmaps.modeling.determinism import deterministic_torch
from pdm_cmaps.modeling.torch_models import PyTorchCNNModel, PyTorchRNNModel, PyTorchTransformerModel
from factories import make_synthetic_cmapss

FEATURES = ["T24", "T30", "T50", "P30", "Nf", "Nc"]


def _frame():
    df = make_synthetic_cmapss(n_engines=8, min_cycles=40, max_cycles=60, seed=3)
    df["RUL"] = df.groupby("unit_id")["cycle"].transform("max") - df["cycle"]
    return df


def _models(seed):
    kw = dict(seq_length=5, max_epochs=3, patience=2, random_state=seed, device="cpu")
    return {"CNN": PyTorchCNNModel(**kw), "LSTM": PyTorchRNNModel(hidden_dim=16, num_layers=1, **kw),
            "Transformer": PyTorchTransformerModel(**kw)}


@pytest.mark.parametrize("name", ["CNN", "LSTM", "Transformer"])
def test_network_predictions_do_not_depend_on_the_process_thread_count(name):
    df = _frame()
    out = []
    prev = torch.get_num_threads()
    for threads in (1, 3):
        torch.set_num_threads(threads)  # o que o processo tinha antes; fit e predict devem ignorar
        m = _models(5)[name].fit(df, FEATURES, "RUL")
        out.append(m.predict(df))
    torch.set_num_threads(prev)
    assert np.array_equal(out[0], out[1]), f"{name}: previsões diferem entre 1 e 3 threads (máx {np.abs(out[0] - out[1]).max():.3g})"


@pytest.mark.parametrize("name", ["CNN", "LSTM", "Transformer"])
def test_network_predictions_repeat_with_the_same_seed_and_differ_across_seeds(name):
    df = _frame()
    a = _models(7)[name].fit(df, FEATURES, "RUL").predict(df)
    b = _models(7)[name].fit(df, FEATURES, "RUL").predict(df)
    c = _models(8)[name].fit(df, FEATURES, "RUL").predict(df)
    assert np.array_equal(a, b)
    assert not np.array_equal(a, c), "sementes diferentes deram o mesmo resultado: a semente não está sendo usada"


def test_deterministic_context_restores_the_process_state():
    prev_threads, prev_det = torch.get_num_threads(), torch.are_deterministic_algorithms_enabled()
    with deterministic_torch(1, threads=1):
        assert torch.get_num_threads() == 1 and torch.are_deterministic_algorithms_enabled()
    assert torch.get_num_threads() == prev_threads
    assert torch.are_deterministic_algorithms_enabled() == prev_det


def test_minirocket_predictions_do_not_depend_on_the_process_thread_count():
    """O MiniRocket daqui é em PyTorch (convoluções): somas multi-thread mudam a PPV quando cruzam um viés."""
    from pdm_cmaps.modeling.minirocket import MiniRocketRULModel
    df = _frame()
    out = []
    prev = torch.get_num_threads()
    for threads in (1, 3):
        torch.set_num_threads(threads)
        m = MiniRocketRULModel(seq_length=8, num_kernels=200, random_state=4).fit(df, FEATURES, "RUL")
        out.append(m.predict_values(df))
    torch.set_num_threads(prev)
    assert np.array_equal(out[0], out[1]), f"MiniRocket: diferença máxima {np.abs(out[0] - out[1]).max():.3g}"

In [ ]:
%%writefile projeto/tests/test_domain.py
"""Regras puras do domínio: alvo de RUL, métricas e esquema."""

import numpy as np
from pdm_cmaps.domain.rul import (
    clip_predictions,
    compute_piecewise_rul,
    evaluate_rul_predictions,
    nasa_scoring_function,
)
from pdm_cmaps.domain.schema import DEFAULT_RUL_CAP, SENSOR_COLUMNS

from factories import sample_frame


def test_capped_rul_reaches_zero_and_caps_at_limit():
    frame = sample_frame(engine_count=2)
    rul = compute_piecewise_rul(frame, rul_max=3)
    assert rul.max() == 3.0
    assert rul.min() == 0.0


def test_late_rul_error_has_larger_nasa_penalty():
    truth = np.array([50.0])
    early, _ = nasa_scoring_function(truth, np.array([45.0]))
    late, _ = nasa_scoring_function(truth, np.array([55.0]))
    assert late > early


def test_nasa_scoring_function_overflow_guard():
    """Verify nasa_scoring_function does not trigger RuntimeWarning: overflow on extreme differences."""
    y_true = np.array([10.0, 10.0])
    y_pred = np.array([600.0, -600.0])  # Extreme deviations that would overflow exp(60)
    score, mean_score = nasa_scoring_function(y_true, y_pred)
    assert np.isfinite(score)
    assert np.isfinite(mean_score)
    assert score > 0.0


def test_clip_predictions_and_stratified_evaluation():
    """Verify prediction clipping in [0, 125] and stratified critical-zone scoring."""
    y_true = np.array([125.0, 80.0, 30.0, 10.0, 0.0])
    y_pred = np.array([150.0, 90.0, 35.0, -5.0, -20.0])
    clipped = clip_predictions(y_pred, lower=0.0, upper=DEFAULT_RUL_CAP)
    assert clipped.max() <= DEFAULT_RUL_CAP
    assert clipped.min() >= 0.0

    # Evaluation with clipping
    metrics_unclipped = evaluate_rul_predictions(y_true, y_pred, clip=False)
    metrics_clipped = evaluate_rul_predictions(y_true, y_pred, clip=True)
    assert metrics_clipped["RMSE"] <= metrics_unclipped["RMSE"] + 1e-6

    # Critical zone evaluation (RUL < 50)
    from pdm_cmaps.domain.rul import evaluate_stratified_rul
    strat = evaluate_stratified_rul(y_true, y_pred, critical_threshold=50.0)
    assert "overall" in strat
    assert "critical_zone" in strat
    assert strat["critical_zone"]["N_samples"] == 3


def test_todo6_physical_directionality_schema():
    """TODO-6: Positive drift, negative drift and exogenous sensors form disjoint partitions with valid units."""
    from pdm_cmaps.domain.schema import (
        EXOGENOUS_CONSTANT_SENSORS,
        NEGATIVE_DRIFT_SENSORS,
        POSITIVE_DRIFT_SENSORS,
        SENSOR_UNITS,
    )
    s_pos = set(POSITIVE_DRIFT_SENSORS)
    s_neg = set(NEGATIVE_DRIFT_SENSORS)
    s_exo = set(EXOGENOUS_CONSTANT_SENSORS)

    assert len(s_pos.intersection(s_neg)) == 0
    assert len(s_pos.intersection(s_exo)) == 0
    assert len(s_neg.intersection(s_exo)) == 0

    # Every sensor column has a declared physical unit
    for sensor in SENSOR_COLUMNS:
        assert sensor in SENSOR_UNITS
        assert isinstance(SENSOR_UNITS[sensor], str) and len(SENSOR_UNITS[sensor]) > 0

In [ ]:
%%writefile projeto/tests/test_estudos_complementares.py
"""Testes unitários dos estudos complementares da revisão de banca (Exp. A, B, C e D)."""
from __future__ import annotations

import numpy as np
import pandas as pd

from pdm_cmaps.detection.multivariate import MULTIVARIATE_SENSORS, MultivariateResidualMonitor
from pdm_cmaps.domain.schema import SETTING_COLUMNS
from pdm_cmaps.experiments.estudos_complementares import (
    IForestEwmaMonitor,
    PCAEwmaMonitor,
    _strip_endogenous_exog,
    apply_cqr_interval,
    fit_cqr_from_quantiles,
    fit_failure_mode_kmeans,
    flicker_metrics,
    levels_with_hysteresis,
)
from pdm_cmaps.monitoring.alerts import CriticalityPolicy


def test_cqr_interval_calibration_and_monotonicity():
    rng = np.random.default_rng(7)
    x = rng.uniform(0, 125, 1000)
    noise = rng.normal(0, 5.0 + 0.08 * x)
    y = np.clip(x + noise, 0.0, 125.0)
    q_lo = x - 1.2 * (5.0 + 0.08 * x)
    q_hi = x + 1.2 * (5.0 + 0.08 * x)
    corr = fit_cqr_from_quantiles(y, q_lo, q_hi, alpha=0.10)
    lo, hi = apply_cqr_interval(q_lo, q_hi, corr, rul_cap=125.0)
    cov = float(np.mean((y >= lo) & (y <= hi)))
    assert np.all(lo <= hi)
    assert np.all((lo >= 0.0) & (hi <= 125.0))
    assert cov >= 0.89


def test_hysteresis_eliminates_oscillations_inside_band():
    policy = CriticalityPolicy(attention=60.0, warning=30.0, critical=15.0, persistence=2)
    # Sequência oscila ao redor do limiar crítico 15 (entre 14 e 18): sem histerese oscila toda hora; com banda 5 fica estável
    rul = np.array([25, 22, 14, 18, 14, 18, 14, 18, 13, 19, 25], dtype=float)
    raw_no_hyst = policy.level_from_rul(rul)
    with_hyst = levels_with_hysteresis(rul, policy, band=5.0)
    assert np.sum(np.diff(with_hyst) < 0) == 1  # só sai de CRITICAL quando passa de 15 + 5 = 20
    assert np.sum(np.diff(raw_no_hyst) < 0) == 4


def test_flicker_metrics_detects_sawtooth_jumps():
    df = pd.DataFrame({
        "unit_id": [1] * 6,
        "cycle": [1, 2, 3, 4, 5, 6],
        "RUL_pred": [40.0, 41.0, 42.0, 40.0, 39.0, 38.0],
        "RUL_lower": [12.0, 35.0, 11.0, 10.0, 36.0, 10.0],
        "level_interval": [3, 1, 3, 3, 1, 3],
    })
    m = flicker_metrics(df, "interval")
    assert m["Lower_jumps_ge15_per_engine"] == 4.0
    assert m["Reversals_per_engine"] == 2.0


def test_pca_ewma_and_iforest_ewma_respond_to_shift():
    from sklearn.ensemble import IsolationForest

    rng = np.random.default_rng(42)
    units_cal = np.repeat(np.arange(1, 21), 50)
    z_cal = rng.normal(0.0, 1.0, size=(len(units_cal), 13))

    pca_mon = PCAEwmaMonitor(lam=0.10, variance_ratio=0.90, alpha=0.0027, random_state=42).fit(z_cal, units_cal)
    forest = IsolationForest(n_estimators=100, random_state=42).fit(z_cal)
    if_mon = IForestEwmaMonitor(forest, lam=0.10, alpha=0.0027).fit(z_cal, units_cal)

    # motor de teste saudável até ciclo 30, depois degrau em 4 sensores
    units_te = np.ones(80, dtype=int)
    z_te = rng.normal(0.0, 1.0, size=(80, 13))
    z_te[30:, :4] += 2.5

    r_pca = pca_mon.score_ratio(z_te, units_te)
    r_if = if_mon.score_ratio(z_te, units_te)
    assert r_pca[:25].mean() < 0.8 and r_pca[-15:].mean() > 1.2
    assert r_if[:25].mean() < 0.8 and r_if[-15:].mean() > 1.0


def test_strip_endogenous_removes_p15_and_epr_but_preserves_monitored_sensors():
    rng = np.random.default_rng(1)
    n_units, T = 12, 80
    rows = []
    for u in range(1, n_units + 1):
        for c in range(1, T + 1):
            row = {"unit_id": u, "cycle": c, "regime_id": 0, "setting_1": 0.0, "setting_2": 0.0, "setting_3": 100.0,
                   "T2": 518.67, "P2": 14.62, "P15": 21.61 + rng.normal(0, 0.01), "epr": 1.3 + rng.normal(0, 0.005),
                   "Nf_dmd": 2388.0, "PCNfR_dmd": 100.0}
            for s in MULTIVARIATE_SENSORS:
                row[s] = 100.0 + rng.normal(0, 0.5)
            rows.append(row)
    df = pd.DataFrame(rows)
    stripped = _strip_endogenous_exog(df)
    assert "P15" not in stripped.columns and "epr" not in stripped.columns
    mv = MultivariateResidualMonitor(healthy_cycle_limit=25, random_state=1).fit(stripped)
    assert len(mv.sensors_) == 13
    assert "P15" not in mv.virtual_["T50"].feature_cols and "epr" not in mv.virtual_["T50"].feature_cols


def test_failure_mode_kmeans_recovers_hpc_and_fan_modes():
    rng = np.random.default_rng(9)
    rows = []
    for u in range(1, 21):
        is_fan = u > 10
        for c in range(1, 91):
            prog = max(0.0, (c - 50) / 40.0)
            row = {"unit_id": u, "cycle": c, "setting_1": 0.0, "setting_2": 0.0, "setting_3": 100.0}
            for s in MULTIVARIATE_SENSORS:
                row[s] = 100.0 + rng.normal(0, 0.2)
            row["T50"] += 5.0 * prog
            row["P30"] += (6.0 if is_fan else -3.0) * prog
            row["phi"] += (5.0 if is_fan else -2.5) * prog
            row["BPR"] += (-0.2 if is_fan else 0.15) * prog
            rows.append(row)
    df = pd.DataFrame(rows)
    _, _, _, z_tr, _ = fit_failure_mode_kmeans(df, n_regimes=1, sensors=MULTIVARIATE_SENSORS, seed=42)
    assert (z_tr.loc[1:10, "modo"] == "HPC").all()
    assert (z_tr.loc[11:20, "modo"] == "Fan").all()

In [ ]:
%%writefile projeto/tests/test_experiments.py
"""Estudos executáveis: injeção de falhas, detecção, janela saudável, poder e bootstrap."""

import numpy as np
import pandas as pd
import pytest
from pdm_cmaps.domain.schema import COLUMNS, SENSOR_COLUMNS


def _fi_world(seed=0):
    from pdm_cmaps.experiments import fault_injection as fi

    rng = np.random.default_rng(seed)
    rows = []
    for unit in range(1, 9):
        life = 260
        for c in range(1, life + 1):
            row = {col: 0.0 for col in COLUMNS}
            row.update(unit_id=unit, cycle=c, setting_1=0.0, setting_2=0.0, setting_3=100.0)
            for s in SENSOR_COLUMNS:
                row[s] = 100.0 + rng.normal(0, 1.0) + (0.1 * max(0, c - (life - 100)) if s in ("T50", "Nc") else 0.0)
            rows.append(row)

    class FakeLoader:
        data_dir = None

        def load(self, subset):
            return pd.DataFrame(rows)

    return fi, fi.load_world("FD001", seed, 50, loader=FakeLoader())


def _write_fake_alarm_run(tmp_path, subset="FD001", seed=1, H=25, n_rec=8, detect_all=True):
    """Corrida sintética para o bootstrap do poder: n_rec motores, 4 controles (alarme só em s baixo) e 2 falhas (alarme em todo s <= 20) por motor."""
    import numpy as np
    import pandas as pd
    from pdm_cmaps.experiments.fault_detection_study import DETECTORS, S_GRID
    from pdm_cmaps.experiments.poder_bootstrap import _slug

    rows, bid = [], 0
    for rec in range(1, n_rec + 1):
        for _ in range(4):
            bid += 1
            rows.append({"blind_id": bid, "recipient": rec, "role": "controle", "kind": "", "magnitude": np.nan, "t0": 0, "T": 300})
        for _ in range(2):
            bid += 1
            rows.append({"blind_id": bid, "recipient": rec, "role": "falha", "kind": "degrau", "magnitude": 0.25, "t0": H + 10, "T": H + 90})
    meta = pd.DataFrame(rows)
    arrays = {}
    for d in DETECTORS:
        mat = np.full((len(meta), len(S_GRID)), np.nan)
        ctrl = (meta["role"] == "controle").to_numpy()
        low = S_GRID < 0.5
        mat[np.ix_(ctrl, low)] = H + 50.0                      # controles alarmam só com limiar baixo
        if detect_all:
            mat[np.ix_(~ctrl, S_GRID < 3.0)] = H + 15.0       # falhas detectadas com qualquer limiar abaixo de 3
        slug = _slug(d)
        arrays[f"{slug}__ids"], arrays[f"{slug}__mat"], arrays[f"{slug}__last"] = meta["blind_id"].to_numpy(), mat, np.full(len(meta), H + 100.0)
    path = tmp_path / f"{subset}_seed{seed}_H{H}.npz"
    meta.assign(Subset=subset, Seed=seed, H=H).to_csv(tmp_path / f"{subset}_seed{seed}_H{H}_manifesto.csv", index=False)
    np.savez_compressed(path, **arrays)
    return path


def test_healthy_window_reference_metrics_do_not_depend_on_h():
    """early alarm / FAR are scored against true RUL >= 125, the same cycles for every H."""
    from pdm_cmaps.experiments.healthy_window_study import _reference_metrics, _windows

    cyc = np.arange(1, 301)
    pc = pd.DataFrame({"unit_id": 1, "cycle": cyc, "rul_true": 300 - cyc, "breach": cyc == 120})
    for H in (50, 100):
        m = _reference_metrics(pc, {1: 120}, H, k=2)
        assert m["early_alarm_pct"] == 100.0  # alarm at RUL 180 >= 125
        assert m["far_common_pct"] == pytest.approx(100 / 75)  # cycles 101..175, one breach
    assert _reference_metrics(pc, {1: 200}, 50, 2)["early_alarm_pct"] == 0.0
    train = pd.DataFrame({"unit_id": [1, 1, 2, 2], "cycle": [1, 200, 1, 240]})
    assert _windows(train)["H=25% vida (=55)"] == 55  # 0.25 x median(200, 240)


def test_nc_step_scan_flags_a_core_reset(monkeypatch):
    """A synthetic core-speed reset (high Nc, then back to the healthy level) is found at its cycle."""
    from pdm_cmaps.experiments import rul_prediction_plots as rpp

    rng = np.random.default_rng(0)
    rows = []
    for unit in (1, 2, 3):
        for c in range(1, 201):
            nc = 9000 + rng.normal(0, 1.0) + (8.0 if unit == 1 and 100 <= c < 150 else 0.0)
            rows.append({"unit_id": unit, "cycle": c, "setting_1": 0.0, "setting_2": 0.0, "setting_3": 100.0, "Nc": nc})

    class FakeLoader:
        def load(self, subset):
            return pd.DataFrame(rows)

    scan = rpp.scan_discontinuities("FD001", loader=FakeLoader())
    top = scan.iloc[0]
    assert int(top["unit_id"]) == 1 and int(top["cycle"]) == 150 and bool(top["candidate"])
    assert not scan.loc[scan["unit_id"] != 1, "candidate"].any()


def test_fault_injection_is_causal_blind_and_paired(tmp_path):
    fi, world = _fi_world()
    assert world.signature[fi.INJECT_SENSORS.index("T50")] > 3  # EOL signature recovered in sigma units
    eng = world.test[world.test["unit_id"] == int(world.test["unit_id"].iloc[0])].sort_values("cycle")
    rng = np.random.default_rng(1)
    base = fi.healthy_surrogate(eng, 120, 50, rng)
    assert len(base) == 120 and base["cycle"].tolist() == list(range(1, 121))
    t0 = 60
    for kind in ("degrau", "rampa", "variancia", "vies_sensor"):
        twin = fi.inject(world, base, fi.delta_z(world, base, kind, 1.0, t0, rng))
        pre = base["cycle"] < t0
        assert np.allclose(twin.loc[pre, list(fi.INJECT_SENSORS)], base.loc[pre, list(fi.INJECT_SENSORS)])  # identical before t0
        assert not np.allclose(twin.loc[~pre, list(fi.INJECT_SENSORS)], base.loc[~pre, list(fi.INJECT_SENSORS)])
        assert np.allclose(twin[["setting_1", "setting_2", "setting_3"]], base[["setting_1", "setting_2", "setting_3"]])
    # step of magnitude 1 adds exactly the signature in sigma units
    step = fi.inject(world, base, fi.delta_z(world, base, "degrau", 1.0, t0, rng))
    got = (fi.standardize(world, step) - fi.standardize(world, base))[-1]
    assert np.allclose(got, world.signature)
    # blind files carry the raw schema only
    run_dir = tmp_path / "run"
    fi.generate(world, run_dir, max_engines=2)
    assert list(pd.read_csv(run_dir / "dados_cegos" / "cego_deteccao.csv").columns) == COLUMNS
    assert list(pd.read_csv(run_dir / "dados_cegos" / "cego_rul.csv").columns) == COLUMNS
    # tampering with the sealed key is detected by score()
    key = run_dir / "selado" / "manifesto_deteccao.csv"
    key.write_text(key.read_text() + "\n")
    (run_dir / "saidas").mkdir()
    with pytest.raises(RuntimeError):
        fi.score(world, run_dir, 60.0)


def test_fault_injection_transplant_label_ends_at_last_cycle(tmp_path):
    fi, world = _fi_world()
    fi.generate(world, tmp_path / "r", max_engines=2)
    man = pd.read_csv(tmp_path / "r" / "selado" / "manifesto_rul.csv")
    blind = pd.read_csv(tmp_path / "r" / "dados_cegos" / "cego_rul.csv")
    for row in man[man["role"] == "transplante"].itertuples():
        assert blind.loc[blind["unit_id"] == row.blind_id, "cycle"].max() == row.eol_cycle == row.t0 + fi.TRANSPLANT_N - 1


def test_alarm_cycles_are_causal_monotone_and_start_at_min_cycle():
    from pdm_cmaps.experiments import fault_detection_study as fd

    rng = np.random.default_rng(0)
    cycles = np.arange(1, 201)
    ratio = np.abs(rng.normal(0.5, 0.2, 200))
    ratio[120:] += 1.5
    gate = np.ones(200, bool)
    full = fd.alarm_cycles(ratio, gate, cycles, 51)
    for cut in (110, 140, 180):  # truncation: an alarm inside the prefix is identical, a later one is not seen
        part = fd.alarm_cycles(ratio[:cut], gate[:cut], cycles[:cut], 51)
        inside = full <= cut - fd.K_CONSEC + 1
        assert np.array_equal(part[inside], full[inside])
        assert np.isnan(part[~inside]).all() or (part[~inside] >= full[~inside]).all()
    has = ~np.isnan(full)
    assert np.all(np.diff(full[has]) >= 0) and (np.diff(has.astype(int)) <= 0).all()  # higher s: later or none
    assert np.nanmin(fd.alarm_cycles(ratio, gate, cycles, 130)) >= 130  # monitoring starts at min_cycle
    assert np.isnan(fd.alarm_cycles(ratio[:1], gate[:1], cycles[:1], 1)).all()


def test_control_far_and_arl0_use_exposure_until_first_alarm():
    from pdm_cmaps.experiments import fault_detection_study as fd

    alarms = np.array([[60.0, np.nan], [np.nan, np.nan]])
    far, arl0 = fd.control_far(alarms, np.array([150, 150]), 50)
    assert far[0] == pytest.approx(100 * 1 / (10 + 100)) and arl0[0] == pytest.approx(110)
    assert far[1] == 0 and np.isinf(arl0[1])


def test_detection_study_series_are_blind_and_paired():
    from pdm_cmaps.domain.schema import COLUMNS
    from pdm_cmaps.experiments import fault_detection_study as fd

    fi, world = _fi_world()
    blind, meta = fd.build_series(world, seed=0, n_ctrl=2, n_rep=1)
    assert list(blind.columns) == COLUMNS
    assert {"controle", "falha"} == set(meta["role"]) and meta["blind_id"].is_unique
    ctrl = meta[meta["role"] == "controle"].iloc[0]
    assert blind.loc[blind["unit_id"] == ctrl["blind_id"], "cycle"].max() == fd.HORIZON_CTRL
    sc = fd.raw_t50_scores(world, blind)
    mv = fd.raw_mewma_scores(world, blind, ["T24", "T30", "T50", "P30"])
    assert set(sc["blind_id"]) == set(meta["blind_id"]) == set(mv["blind_id"]) and np.isfinite(sc["ratio"]).all()
    # a large injected step is detected by the raw T50 chart only after t0
    big = meta[(meta["kind"] == "degrau") & (meta["magnitude"] == 0.5)]
    fid = int(big.iloc[0]["blind_id"])
    t0 = int(big.iloc[0]["t0"])
    one = sc[sc["blind_id"] == fid]
    a = fd.alarm_cycles(one["ratio"].to_numpy(), one["gate"].to_numpy(), one["cycle"].to_numpy(), t0)
    assert np.isnan(a).all() or np.nanmin(a) >= t0


def test_hard_fault_family_is_causal_documented_and_paired_delta_is_signed():
    from pdm_cmaps.detection.multivariate import MultivariateResidualMonitor
    from pdm_cmaps.experiments import fault_hard_study as fh

    fi, world = _fi_world()
    rng = np.random.default_rng(3)
    eng = world.test[world.test["unit_id"] == int(world.test["unit_id"].iloc[0])].sort_values("cycle")
    base = fh.bootstrap_series(eng, 160, rng)
    specs = [fh.draw_fault(world, rng, 50) for _ in range(60)]
    assert {s["family"] for s in specs} == {"processo", "sensor"} and {s["shape"] for s in specs} >= {"degrau", "rampa", "intermitente"}
    for s in specs:
        dz = fh.fault_dz(s, base, rng)
        assert np.allclose(dz[(base["cycle"] < s["t0"]).to_numpy()], 0)          # nothing before t0
        assert 50 + fh.T0_RANGE[0] <= s["t0"] <= 50 + fh.T0_RANGE[1]
        if s["family"] == "sensor":
            assert (s["weights"] != 0).sum() == 1 and fh.SENS_MAG[0] <= s["mag"] <= fh.SENS_MAG[1]
        else:
            assert fh.PROC_MAG[0] <= s["mag"] <= fh.PROC_MAG[1]
    # paired difference: estimator detects 1 of 2 more faults than the baseline in the same cluster
    keys = {"Subset": "FD001", "Seed": 1, "Chart": "MEWMA", "Mode": "exogenous"}
    series = pd.DataFrame([{**keys, "Estimator": fh.BASELINE, "blind_id": i, "detected": d} for i, d in ((1, False), (2, True))] +
                          [{**keys, "Estimator": "Ridge", "blind_id": i, "detected": d} for i, d in ((1, True), (2, True))])
    man = pd.DataFrame({"Subset": "FD001", "Seed": 1, "blind_id": [1, 2], "recipient": [10, 11], "family": "processo", "mag": 0.1})
    out = fh.paired_delta(series, man, np.random.default_rng(0), n_boot=50)
    assert out.loc[0, "delta_pp"] == pytest.approx(50.0)
    with pytest.raises(ValueError):
        MultivariateResidualMonitor(mode="nonsense")


def test_compare_engine_offset_power_table_pairs_cells(tmp_path):
    from pdm_cmaps.experiments.compare_engine_offset import DETECTORS, power_table

    def write(root, power):
        (root / "runs").mkdir(parents=True)
        pd.DataFrame({"Detector": [DETECTORS[0]] * 2, "kind": ["degrau"] * 2, "magnitude": [0.1, 0.25], "Subset": "FD003", "Seed": 42,
                      "power_pct": power, "far_ctrl_per100": 0.1, "delay_med": 10.0}).to_csv(root / "runs" / "FD003_seed42_H25_pareado.csv", index=False)

    write(tmp_path / "base", [10.0, 40.0])
    write(tmp_path / "offset", [30.0, 35.0])
    out = power_table([tmp_path / "base"], [tmp_path / "offset"]).iloc[0]
    assert out["celulas"] == 2 and out["celulas_melhores"] == 1 and out["celulas_piores"] == 1
    assert out["ganho_medio"] == 7.5 and out["poder_base"] == 25.0 and out["poder_offset"] == 32.5


def test_poder_bootstrap_matches_point_power_and_pairs_detectors(tmp_path):
    import numpy as np
    from pdm_cmaps.experiments.fault_detection_study import DETECTORS
    from pdm_cmaps.experiments.poder_bootstrap import Run, bootstrap, summarize

    runs = [Run(_write_fake_alarm_run(tmp_path, seed=s)) for s in (1, 2)]
    # o s* escolhido deixa os controles fora do orçamento de 0,2 e as falhas dentro: poder 100 % em todos os detectores
    point = runs[0].power(DETECTORS[0], np.ones(len(runs[0].recipients)))
    assert np.allclose(point, 100.0)
    boot = bootstrap(runs, n_boot=60, seed=3)
    power, contr = summarize(runs, boot)
    assert (power["poder_pct"] == 100.0).all() and (power["ic_lo"] == 100.0).all()
    assert (contr["diferenca_pp"] == 0.0).all() and (contr["p_boot"] == 1.0).all()
    # caso sem nenhuma detecção: poder 0
    (tmp_path / "sem").mkdir()
    run0 = Run(_write_fake_alarm_run(tmp_path / "sem", seed=9, detect_all=False))
    assert np.allclose(run0.power(DETECTORS[0], np.ones(len(run0.recipients))), 0.0)


def test_holm_adjusts_monotonically():
    import numpy as np
    from pdm_cmaps.experiments.poder_bootstrap import holm

    p = np.array([0.001, 0.04, 0.03, 0.5])
    adj = holm(p)
    assert np.allclose(adj, [0.004, 0.09, 0.09, 0.5])
    assert (adj >= p).all() and adj.max() <= 1.0

In [ ]:
%%writefile projeto/tests/test_mewma_calibracao.py
"""Calibração sintética do limite do T² da MEWMA (detection/multivariate.py, calibration="synthetic")."""
from __future__ import annotations

import numpy as np
import pandas as pd
import pytest

from pdm_cmaps.detection.multivariate import MultivariateResidualMonitor

P, W, ENG, CYC = 4, 50, 40, 50


def _setup(seed=0, offset_sd=0.5):
    rng = np.random.default_rng(seed)
    rows = []
    for u in range(1, ENG + 1):
        off = rng.normal(0, offset_sd, P)  # nível próprio do motor, que a previsão não conhece
        for c in range(1, CYC + 1):
            rows.append([u, c, *(off + rng.normal(0, 1, P))])
    df = pd.DataFrame(rows, columns=["unit_id", "cycle", *"abcd"]).set_index(["unit_id", "cycle"])
    regime = pd.Series(0, index=df.index)
    mv = MultivariateResidualMonitor(sensors=tuple("abcd"), healthy_cycle_limit=W, engine_offset=True, calibration="synthetic",
                                     synthetic_reps=10, synthetic_horizon=120, random_state=1)
    # o que o fit faria: z centrado em cada motor DENTRO da janela, depois mu e Sigma
    centered = df - df.groupby(level="unit_id").transform("mean")
    mv.mu_ = centered.to_numpy().mean(axis=0)
    mv.sigma_inv_ = np.linalg.inv(np.cov(centered.to_numpy(), rowvar=False))
    return mv, df, regime, centered


def test_default_calibration_is_the_previous_protocol():
    assert MultivariateResidualMonitor().calibration == "empirical"
    with pytest.raises(ValueError):
        MultivariateResidualMonitor(calibration="outra")


def test_synthetic_limit_is_higher_than_the_in_window_empirical_limit():
    """No protocolo anterior o T² da calibração é medido onde a média do z é zero por construção, então o limite sai baixo demais; a calibração
    sintética mede o T² depois da janela, com o erro de estimar o deslocamento, e dá um limite maior."""
    mv, df, regime, centered = _setup()
    t2_in_window = mv._t2(centered.to_numpy(), centered.index.get_level_values("unit_id").to_numpy())
    synth = mv._synthetic_post_window_t2(df, regime)
    assert synth.mean() > t2_in_window.mean() * 1.02
    assert np.quantile(synth, 0.9973) > np.quantile(t2_in_window, 0.9973)


def test_synthetic_calibration_shape_and_determinism():
    mv, df, regime, _ = _setup()
    a = mv._synthetic_post_window_t2(df, regime)
    b = mv._synthetic_post_window_t2(df, regime)
    assert a.shape == (ENG * 10 * 120,)          # motores x réplicas x ciclos depois da janela
    assert np.array_equal(a, b) and np.isfinite(a).all()


def test_synthetic_mean_t2_matches_the_theory_for_offset_estimation_error():
    """Com ruído i.i.d. e o deslocamento estimado de W ciclos, o erro de estimação (variância 1/W) soma-se à variância estacionária da EWMA
    (λ/(2−λ)): E[T²] ≈ p · (1 + (1/W) / (λ/(2−λ))). Para p=4, λ=0,1 e W=50 são 5,5; a média em janela, centrada nela mesma, fica ABAIXO de p."""
    mv, df, regime, centered = _setup(seed=2)
    lam = mv.lam
    expected = P * (1 + (1 / W) / (lam / (2 - lam)))
    synth = mv._synthetic_post_window_t2(df, regime)
    assert abs(synth.mean() / expected - 1) < 0.15, (synth.mean(), expected)
    t2_in = mv._t2(centered.to_numpy(), centered.index.get_level_values("unit_id").to_numpy())
    assert t2_in.mean() < P, "a média em janela deveria ficar abaixo de p (centrada nela mesma)"


def test_synthetic_t2_has_a_transient_right_after_the_window():
    """Logo depois da janela a EWMA ainda carrega a memória da janela (z centrado por construção), então o T² médio cresce até o estacionário.
    Por isso a validação em dados reais nos primeiros ciclos pós-janela exige o limite sintético RESTRITO a esses ciclos."""
    mv, df, regime, _ = _setup()
    m = mv._synthetic_post_window_t2(df, regime, by_cycle=True)
    assert m.shape == (ENG * 10, 120)
    assert np.array_equal(m.ravel(), mv._synthetic_post_window_t2(df, regime))
    assert m[:, :10].mean() < m[:, -50:].mean() * 0.95
    assert np.quantile(m[:, :25].ravel(), 0.99) < np.quantile(m.ravel(), 0.99)

In [ ]:
%%writefile projeto/tests/test_modeling.py
"""Modelos de RUL: pool, validação cruzada agrupada, MiniRocket e redes PyTorch."""

import numpy as np
import pandas as pd
import pytest
import torch
from pdm_cmaps.domain.rul import (
    compute_piecewise_rul,
)
from pdm_cmaps.modeling.benchmark import RULBenchmark
from pdm_cmaps.modeling.minirocket import MiniRocketRULModel, MiniRocketTransformer
from pdm_cmaps.modeling.registry import get_model, get_model_pool, grouped_oof
from pdm_cmaps.modeling.torch_models import PyTorchRNNModel
from sklearn.ensemble import ExtraTreesRegressor

from factories import make_synthetic_cmapss, sample_frame


def test_grouped_oof_covers_rows_without_engine_leakage():
    frame = sample_frame(engine_count=10)
    frame["RUL"] = compute_piecewise_rul(frame)
    features = ["cycle", "setting_1", "T2"]
    estimator = ExtraTreesRegressor(n_estimators=5, random_state=2)
    folds, prediction = grouped_oof(frame, features, estimator=estimator, n_splits=5)
    assert len(prediction) == len(frame)
    assert np.isfinite(prediction).all()
    assert len(folds) == 5


def test_model_pool_includes_padua_algorithms():

    pool = get_model_pool()
    assert "CatBoost" in pool
    assert "XGBoost" in pool
    assert "LightGBM" in pool

    X = np.random.randn(20, 4)
    y = np.random.randn(20)

    for name in ["catboost", "xgboost", "lightgbm"]:
        model = get_model(name, n_estimators=5, max_depth=3, iterations=5, depth=3)
        model.fit(X, y)
        pred = model.predict(X)
        assert len(pred) == len(y)
        assert np.isfinite(pred).all()


def test_minirocket_kernel_properties():
    """Validate that base kernels and multi-channel kernels satisfy length 9, zero sum, and {-1, 2} weights."""
    base_kernels = MiniRocketTransformer.generate_base_kernels()
    assert base_kernels.shape == (84, 9)
    assert np.all(np.isin(base_kernels, [-1.0, 2.0]))
    assert np.allclose(base_kernels.sum(axis=1), 0.0)
    for row in base_kernels:
        assert (row == 2.0).sum() == 3
        assert (row == -1.0).sum() == 6

    # Test multi-channel kernel assignment
    transformer = MiniRocketTransformer(num_kernels=84, seq_length=15, random_state=42)
    dummy_x = np.random.randn(10, 5, 15).astype(np.float32)
    transformer.fit(dummy_x)

    assert hasattr(transformer, "weights_")
    for d, weights in transformer.weights_.items():
        assert weights.shape[1] == 5  # channels
        assert weights.shape[2] == 9  # length
        for k in range(weights.shape[0]):
            kernel = weights[k]
            # Kernel sum must be zero
            assert np.isclose(kernel.sum(), 0.0)
            active_channels = (np.abs(kernel).sum(axis=1) > 0)
            assert 1 <= active_channels.sum() <= 3
            for c_idx in np.where(active_channels)[0]:
                c_weights = kernel[c_idx]
                assert set(np.unique(c_weights)) == {-1.0, 2.0}
                assert np.isclose(c_weights.sum(), 0.0)


def test_minirocket_causality():
    """Ensure that causal left-padding guarantees that output at time t does not depend on future timestamps."""
    transformer = MiniRocketTransformer(num_kernels=50, seq_length=15, dilations=(1, 2), random_state=42)
    # Fit on initial dummy data
    init_x = np.random.randn(20, 4, 15).astype(np.float32)
    transformer.fit(init_x)

    # Construct two series: identical up to t=9, diverging from t=10 onwards
    x1 = np.random.randn(1, 4, 15).astype(np.float32)
    x2 = x1.copy()
    x2[:, :, 10:] = np.random.randn(1, 4, 5).astype(np.float32) + 10.0

    conv1 = transformer.convolve(x1)
    conv2 = transformer.convolve(x2)

    for d in transformer.active_dilations_:
        out1 = conv1[d]
        out2 = conv2[d]
        # Timestamps 0 to 9 must be bitwise/numerically identical
        assert torch.allclose(out1[..., :10], out2[..., :10], atol=1e-6), (
            f"Future leakage detected for dilation {d} at timestamps before t=10"
        )
        # Timestamps 10 to 14 must differ because future values changed
        assert not torch.allclose(out1[..., 10:], out2[..., 10:]), (
            f"Expected difference at t >= 10 for dilation {d}"
        )


def test_minirocket_shape_and_determinism():
    """Validate output dimensions, bounded [0, 1] range, and reproducibility with random_state."""
    n_samples, n_channels, seq_len = 25, 6, 15
    num_kernels = 120
    x = np.random.randn(n_samples, n_channels, seq_len).astype(np.float32)

    transformer1 = MiniRocketTransformer(num_kernels=num_kernels, seq_length=seq_len, random_state=42)
    ppv1 = transformer1.fit_transform(x)

    assert ppv1.shape == (n_samples, num_kernels)
    assert np.all(ppv1 >= 0.0) and np.all(ppv1 <= 1.0)
    assert np.isfinite(ppv1).all()

    # Reproducibility check with identical random_state
    transformer2 = MiniRocketTransformer(num_kernels=num_kernels, seq_length=seq_len, random_state=42)
    ppv2 = transformer2.fit_transform(x)
    assert np.allclose(ppv1, ppv2)

    # Different random_state should yield different feature activations
    transformer3 = MiniRocketTransformer(num_kernels=num_kernels, seq_length=seq_len, random_state=999)
    ppv3 = transformer3.fit_transform(x)
    assert not np.allclose(ppv1, ppv3)


def test_minirocket_fit_predict_pipeline():
    """Test the complete MiniRocketRULModel fit/predict pipeline on sample engine cycles."""
    frame = sample_frame(engine_count=6)
    frame["RUL"] = compute_piecewise_rul(frame)
    features = ["setting_1", "setting_2", "setting_3", "T50", "T24", "P30"]

    model = MiniRocketRULModel(seq_length=4, num_kernels=60, random_state=42)

    # Ensure calling predict before fit raises RuntimeError
    with pytest.raises(RuntimeError, match="Fit the MiniRocketRULModel before calling predict"):
        model.predict(frame)

    model.fit(frame, features, "RUL")
    assert model.fit_time > 0.0
    assert model.regressor is not None

    y_true, y_pred = model.predict(frame)
    assert len(y_true) == len(y_pred)
    assert np.isfinite(y_pred).all()

    values = model.predict_values(frame)
    assert np.array_equal(y_pred, values)


def test_cross_validate_pooled_vs_fold_weighted_and_mean_nasa():
    """Verify cross_validate reports pooled OOF, fold-weighted metrics, and Mean_NASA_Score."""
    df = make_synthetic_cmapss(n_engines=10, seed=42)
    df["RUL"] = compute_piecewise_rul(df)
    features = ["setting_1", "setting_2", "T2", "T50"]
    benchmark = RULBenchmark(verbose=False)
    cv_results = benchmark.cross_validate(df, features, target_col="RUL", n_splits=3)
    assert isinstance(cv_results, pd.DataFrame)
    # Check that pooled OOF and fold-weighted metrics are distinctly reported
    assert "CV_OOF_Pooled_R2" in cv_results.columns or "CV_R2_Mean" in cv_results.columns
    assert "CV_Mean_NASA_Score" in cv_results.columns or "CV_NASA_Score_Mean" in cv_results.columns


def test_pytorch_sequence_model_group_early_stopping():
    """Verify PyTorch sequence model uses engine-grouped validation for early stopping."""
    df = make_synthetic_cmapss(n_engines=8, seed=42)
    df["RUL"] = compute_piecewise_rul(df)
    features = ["setting_1", "T2", "T50"]
    rnn = PyTorchRNNModel(seq_length=5, hidden_dim=16, num_layers=1, max_epochs=5, patience=2, random_state=42)
    rnn.fit(df, features, "RUL")
    assert rnn.fit_time > 0.0
    preds = rnn.predict(df)
    assert len(preds) == len(df)
    assert np.isfinite(preds).all()


def test_minirocket_replicate_padding_and_grouped_ridge():
    """Verify MiniRocket uses replicate padding and GroupKFold in RidgeCV."""
    x = np.random.randn(10, 4, 10).astype(np.float32)
    transformer = MiniRocketTransformer(num_kernels=50, seq_length=10, random_state=42)
    transformer.fit(x)
    features = transformer.transform(x)
    assert features.shape == (10, 50)
    assert np.isfinite(features).all()

    df = make_synthetic_cmapss(n_engines=6, seed=42)
    df["RUL"] = compute_piecewise_rul(df)
    model = MiniRocketRULModel(seq_length=5, num_kernels=50, random_state=42)
    model.fit(df, ["setting_1", "T2", "T50"], "RUL")
    assert model.regressor is not None


def test_xgboost_nasa_loss_starts_from_training_mean_and_beats_mean_baseline():
    """[BASELINE] The NASA-loss XGBoost must start at the training-mean RUL (base_score), not 0.5."""
    import numpy as np
    from pdm_cmaps.modeling.gbdt import XGBoostNasaRegressor

    rng = np.random.default_rng(0)
    x = rng.normal(size=(400, 4))
    y = 90.0 + 25.0 * x[:, 0] + rng.normal(scale=2.0, size=400)
    model = XGBoostNasaRegressor(n_estimators=60, max_depth=4).fit(x, y)
    pred = model.predict(x)
    assert abs(float(model.model_.get_params()["base_score"]) - y.mean()) < 1e-6
    assert np.mean((pred - y) ** 2) < np.var(y)  # better than predicting the mean


def test_minirocket_fit_and_transform_use_same_padding():
    """Biases learned in fit() must come from the same convolution that transform() thresholds."""
    import numpy as np
    from pdm_cmaps.modeling.minirocket import MiniRocketTransformer

    rng = np.random.default_rng(1)
    x = rng.normal(size=(64, 3, 15)).astype(np.float32)
    transformer = MiniRocketTransformer(num_kernels=84, seq_length=15, random_state=0).fit(x)
    ppv = transformer.transform(x)
    assert ppv.shape == (64, 84)
    assert 0.05 < float(ppv.mean()) < 0.95  # quantile-calibrated biases give mid-range PPV

In [ ]:
%%writefile projeto/tests/test_monitoring.py
"""Camada de monitoramento: conformal, criticidade, alertas, custo e relatórios."""

import numpy as np
import pandas as pd

from factories import _toy_timeline


def test_conformal_interval_reaches_nominal_coverage_on_heldout_data():
    """[CALIBRATION] Split-conformal coverage must be close to nominal on fresh data."""
    from pdm_cmaps.monitoring.uncertainty import ConformalRULInterval, interval_metrics

    rng = np.random.default_rng(1)

    def draw(n):
        y = rng.uniform(0, 125, n)
        pred = y + rng.normal(0, 3 + 0.15 * y)  # heteroscedastic: larger error early in life
        return y, np.clip(pred, 0, 125)

    y_cal, p_cal = draw(6000)
    y_te, p_te = draw(6000)
    lo, hi = ConformalRULInterval(alpha=0.10).fit(y_cal, p_cal).predict_interval(p_te)
    metrics = interval_metrics(y_te, lo, hi)
    assert 0.86 <= metrics["coverage"] <= 0.94
    assert np.all(lo >= 0) and np.all(hi <= 125) and np.all(lo <= hi)


def test_debounce_is_causal_and_requires_persistence():
    from pdm_cmaps.monitoring.alerts import debounce

    raw = np.array([0, 3, 0, 2, 2, 2, 3])
    out = debounce(raw, 2)
    assert list(out) == [0, 0, 0, 0, 2, 2, 2]
    # the output at t depends only on raw[:t+1]
    for t in range(len(raw)):
        assert debounce(raw[: t + 1], 2)[-1] == out[t]


def test_anomaly_escalates_only_when_interval_already_shows_risk():
    from pdm_cmaps.monitoring.alerts import (
        NORMAL,
        CriticalityPolicy,
        build_alert_timeline,
    )

    # healthy engine (RUL high) with an SPC alarm must NOT create a maintenance-level alert
    frame = pd.DataFrame({
        "unit_id": 1, "cycle": range(1, 11), "RUL_true": 120.0, "RUL_pred": 120.0, "RUL_lower": 100.0, "RUL_upper": 125.0,
    })
    tl = build_alert_timeline(frame, CriticalityPolicy(), {1: 3})
    assert (tl["level_interval+anomaly"] == NORMAL).all()
    assert tl.loc[tl["cycle"] >= 3, "anomaly_active"].all()


def test_strategy_evaluation_reports_missed_critical_and_false_alarm_rates():
    from pdm_cmaps.monitoring.alerts import evaluate_strategies

    tl, policy = _toy_timeline()
    table = evaluate_strategies(tl, policy)
    assert set(table["Strategy"]) == {"point", "interval", "interval+anomaly"}
    # the conservative lower bound must not miss more critical cycles than the point estimate
    point = table.set_index("Strategy").loc["point", "Missed_critical_rate"]
    interval = table.set_index("Strategy").loc["interval", "Missed_critical_rate"]
    assert interval <= point


def test_rendered_message_contains_only_numbers_from_facts():
    """The message is a fixed template over the facts: every number it prints must come from the facts."""
    import re

    from pdm_cmaps.monitoring.reports import build_facts, render_report

    tl, policy = _toy_timeline()
    facts = build_facts(tl.loc[tl["cycle"] <= 70], policy)
    text = render_report(facts)
    allowed = {facts["unit_id"], facts["cycle"], facts["rul_estimate"], facts["rul_lower"], facts["rul_upper"],
               facts["interval_coverage_pct"], facts["trend_cycles_per_cycle"], facts["trend_window_cycles"]}
    if facts["cycles_since_anomaly"] is not None:
        allowed.add(facts["cycles_since_anomaly"])
    for token in re.findall(r"-?\d+(?:\.\d+)?", text):
        assert float(token) in {float(a) for a in allowed}, token
    assert facts["level_label_pt"] in text and facts["recommended_action_pt"] in text


def test_facts_use_only_past_rows():
    from pdm_cmaps.monitoring.reports import build_facts

    tl, policy = _toy_timeline()
    past = tl.loc[tl["cycle"] <= 50]
    assert build_facts(past, policy)["cycle"] == 50
    assert build_facts(tl, policy)["cycle"] == int(tl["cycle"].max())


def test_mondrian_groups_fall_back_to_rul_bins_for_small_cells():
    from pdm_cmaps.monitoring.uncertainty import ConformalRULInterval

    rng = np.random.default_rng(0)
    p = rng.uniform(0, 125, 3000)
    y = p + rng.normal(0, 10, p.size)
    groups = np.where(np.arange(p.size) < 2990, 0, 1)  # group 1 has only 10 rows -> falls back
    model = ConformalRULInterval(alpha=0.10).fit(y, p, groups=groups)
    assert all(key[0] == 0 for key in model.group_offsets_)
    lo_g, hi_g = model.predict_interval(p[:5], groups=np.ones(5, dtype=int))
    lo, hi = model.predict_interval(p[:5])
    assert np.allclose(lo_g, lo) and np.allclose(hi_g, hi)


def test_normalized_conformal_keeps_coverage_and_adapts_width():
    from pdm_cmaps.monitoring.uncertainty import (
        NormalizedConformalRULInterval,
        interval_metrics,
    )

    rng = np.random.default_rng(1)
    n = 6000
    p = rng.uniform(20, 110, n)
    spread = rng.uniform(1, 10, n)
    y = p + rng.normal(0, spread)
    cal, te = slice(0, 4000), slice(4000, n)
    model = NormalizedConformalRULInterval(alpha=0.10).fit(y[cal], p[cal], spread[cal])
    lo, hi = model.predict_interval(p[te], spread[te])
    assert interval_metrics(y[te], lo, hi)["coverage"] >= 0.87
    width = hi - lo
    assert width[spread[te] > 8].mean() > 1.5 * width[spread[te] < 3].mean()  # floor = median spread damps the ratio


def test_causal_local_spread_uses_only_the_past_of_each_engine():
    from pdm_cmaps.monitoring.uncertainty import causal_local_spread

    pred = np.array([10.0, 10.0, 10.0, 50.0, 0.0, 0.0])
    units = np.array([1, 1, 1, 1, 2, 2])
    out = causal_local_spread(pred, units, window=3)
    assert out[2] == 0.0          # three equal values so far
    assert out[3] > 0.0           # the jump enters only at its own cycle
    assert out[4] == 0.0 and out[5] == 0.0  # engine 2 does not see engine 1


def test_extra_alarm_maps_add_strategies():
    from pdm_cmaps.monitoring.alerts import evaluate_strategies, strategy_names

    tl, policy = _toy_timeline()
    from pdm_cmaps.monitoring.alerts import build_alert_timeline
    base = tl[["unit_id", "cycle", "RUL_true", "RUL_pred", "RUL_lower", "RUL_upper"]]
    tl2 = build_alert_timeline(base, policy, {}, {"anomaly_mv": {7: 1}})
    assert "interval+anomaly_mv" in strategy_names(tl2)
    table = evaluate_strategies(tl2, policy)
    assert set(table["Strategy"]) == {"point", "interval", "interval+anomaly", "interval+anomaly_mv"}


def test_maintenance_cost_counts_failures_and_lead_time():
    from pdm_cmaps.monitoring.alerts import CRITICAL, NORMAL, maintenance_cost

    # one engine, EOL at cycle 100, strategy reaches CRITICAL at cycle 95
    tl = pd.DataFrame({"unit_id": 1, "cycle": range(1, 101)})
    tl["level_point"] = [CRITICAL if c >= 95 else NORMAL for c in tl["cycle"]]
    planned = maintenance_cost(tl, "point", cost_ratio=10, lead=0)
    assert planned["Failures"] == 0 and planned["Mean_unused_cycles"] == 5
    late = maintenance_cost(tl, "point", cost_ratio=10, lead=10)  # executed at 105 > EOL -> failure
    assert late["Failures"] == 1
    assert late["Relative_to_oracle"] > planned["Relative_to_oracle"]

In [ ]:
%%writefile projeto/tests/test_pipeline_cli.py
"""Configuração do pipeline e linha de comando."""

from pathlib import Path

import pytest

from pdm_cmaps.cli.main import parse_arguments
from pdm_cmaps.data.loader import CMapssLoader
from pdm_cmaps.detection.regime import RegimeDecoupler
from pdm_cmaps.modeling.benchmark import RULBenchmark
from pdm_cmaps.pipeline.config import PipelineConfig
from pdm_cmaps.pipeline.pipeline import PdMPipeline


def test_pipeline_config_defaults():
    config = PipelineConfig()
    assert config.subset == "FD004"
    assert config.seed == 42
    assert config.healthy_cycle_limit == 50
    assert config.unit_id == 179
    assert config.include_minirocket is False
    assert config.include_lstm is False
    assert config.clean_output is True
    assert "maintenance_guide_fd004" in str(config.resolved_output_dir)


def test_pipeline_config_custom_output_dir(tmp_path: Path):
    custom_dir = tmp_path / "custom_results"
    config = PipelineConfig(subset="FD001", output_dir=custom_dir)
    assert config.resolved_output_dir == custom_dir


def test_parse_arguments_cli():
    cli_args = [
        "--subset", "FD002",
        "--seed", "123",
        "--healthy-cycle-limit", "60",
        "--unit-id", "45",
        "--no-minirocket",
        "--include-lstm",
        "--no-clean",
    ]
    config = parse_arguments(cli_args)
    assert config.subset == "FD002"
    assert config.seed == 123
    assert config.healthy_cycle_limit == 60
    assert config.unit_id == 45
    assert config.include_minirocket is False
    assert config.include_lstm is True
    assert config.clean_output is False


def test_pdm_pipeline_instantiation(tmp_path: Path):
    out_dir = tmp_path / "test_out"
    config = PipelineConfig(subset="FD001", output_dir=out_dir)

    loader = CMapssLoader()
    decoupler = RegimeDecoupler(n_clusters=1, random_state=42)
    benchmark = RULBenchmark(random_state=42)

    pipeline = PdMPipeline(
        config=config,
        loader=loader,
        decoupler=decoupler,
        benchmark=benchmark,
    )

    assert pipeline.config == config
    assert pipeline.loader is loader
    assert pipeline.decoupler is decoupler
    assert pipeline.benchmark is benchmark
    assert pipeline.output_dir == out_dir

    # Test directory preparation
    pipeline.prepare_directories()
    assert pipeline.table_dir.exists()
    assert pipeline.figure_dir.exists()


def test_progress_tracker_estimates_time_left_from_finished_steps(tmp_path):
    from pdm_cmaps.progress import ProgressTracker, format_duration

    now = [0.0]
    lines = []
    log = tmp_path / "logs" / "_progresso.txt"
    tracker = ProgressTracker(4, log_file=log, clock=lambda: now[0], emit=lines.append)

    tracker.start("a")
    assert tracker.remaining_seconds() is None and "estimado após o 1º passo" in lines[-1]
    now[0] = 600.0  # first step took 10 min
    tracker.finish("a")
    assert tracker.remaining_seconds() == 3 * 600.0  # mean 10 min x 3 steps left
    assert "[1/4] a" in lines[-1] and "restante ~30min" in lines[-1]
    tracker.start("b")
    now[0] = 1800.0  # second step took 20 min -> mean 15 min x 2 left
    tracker.finish("b", returncode=1)
    assert "FALHOU" in lines[-1] and tracker.remaining_seconds() == 2 * 900.0
    assert log.read_text().count("\n") == len(lines)  # every line is also persisted for `tail -f`
    assert [format_duration(s) for s in (20, 90, 4500)] == ["20s", "1min", "1h15min"]



def test_main_prints_progress_for_each_subset_and_persists_it(tmp_path, monkeypatch, capsys):
    import pdm_cmaps.cli.main as cli_main
    from pdm_cmaps.config import ProjectPaths

    monkeypatch.setattr(cli_main, "DEFAULT_PATHS", ProjectPaths(root=tmp_path))
    monkeypatch.setattr(PdMPipeline, "execute", lambda self: 0)  # the heavy work is not under test here

    assert cli_main.main(["--subset", "ALL"]) == 0
    out = capsys.readouterr().out
    for i, subset in enumerate(["FD001", "FD002", "FD003", "FD004"], start=1):
        assert f"[{i}/4] pipeline {subset}" in out
    assert (tmp_path / "results" / "logs" / "_progresso.txt").read_text().count("\n") == 8  # start + finish per subset


def test_study_runner_run_all_reports_progress_and_keeps_failures(tmp_path, monkeypatch, capsys):
    from pdm_cmaps.cli.update_all import StudyRunner, StudySpec
    from pdm_cmaps.config import ProjectPaths

    runner = StudyRunner(ProjectPaths(root=tmp_path))
    codes = iter([0, 1, 0])
    monkeypatch.setattr(runner, "run_study", lambda study: next(codes))
    studies = tuple(StudySpec(name, "m") for name in ("a", "b", "c"))

    assert runner.run_all(studies) == 1  # one failure is not swallowed by the later successes
    out = capsys.readouterr().out
    assert "[2/3] b ... FALHOU (código 1)" in out and "[3/3] c ... ok" in out
    assert (tmp_path / "results" / "logs" / "_progresso.txt").exists()


@pytest.mark.parametrize("healthy_limit, phases", [(50, 5), (100, 4)])
def test_pipeline_execute_reports_one_progress_line_per_phase(tmp_path, monkeypatch, capsys, healthy_limit, phases):
    for name in ("run_core_workflow", "run_auxiliary_spc_horizon", "run_additional_models_and_statistics",
                 "run_monitoring_layer", "run_figure_generation"):
        monkeypatch.setattr(PdMPipeline, name, lambda self, *a, **k: None)
    config = PipelineConfig(subset="FD001", healthy_cycle_limit=healthy_limit, output_dir=tmp_path / "out")
    pipeline = PdMPipeline(config=config, loader=CMapssLoader(), decoupler=RegimeDecoupler(n_clusters=1, random_state=42),
                           benchmark=RULBenchmark(random_state=42))

    assert pipeline.execute() == 0
    out = capsys.readouterr().out
    assert f"[{phases}/{phases}] fase" in out and f"[{phases + 1}/" not in out  # H=100 skips the auxiliary phase

In [ ]:
%%writefile projeto/tests/test_protocolo.py
"""O protocolo primário congelado (src/pdm_cmaps/protocol.py) bate com os padrões do código e com as tabelas."""
from __future__ import annotations

import ast
import inspect
import re
from pathlib import Path

import pandas as pd
import pytest

from pdm_cmaps import protocol as P

SRC = Path(__file__).resolve().parent.parent / "src" / "pdm_cmaps"
STUDIES = sorted((SRC / "experiments").glob("*.py")) + [SRC / "monitoring" / "layer.py"]


def _argparse_defaults(path: Path, flag: str):
    """Valores `default=` literais de `add_argument("<flag>", ...)` no arquivo (via ast, sem importar nem rodar nada)."""
    out = []
    for node in ast.walk(ast.parse(path.read_text(encoding="utf-8"))):
        if isinstance(node, ast.Call) and getattr(node.func, "attr", "") == "add_argument" and node.args:
            if isinstance(node.args[0], ast.Constant) and node.args[0].value == flag:
                for kw in node.keywords:
                    if kw.arg == "default":
                        try:
                            out.append(ast.literal_eval(kw.value))
                        except ValueError:
                            pass
    return out


def test_class_defaults_match_the_protocol():
    from pdm_cmaps.detection.multivariate import MultivariateResidualMonitor
    from pdm_cmaps.detection.virtual_sensor import VirtualSensor
    mv = MultivariateResidualMonitor()
    assert (mv.lam, mv.alpha, mv.k, mv.healthy_cycle_limit) == (P.MEWMA_LAMBDA, P.MEWMA_ALPHA, P.PERSISTENCE_K, P.H_NATIVE)
    assert mv.engine_offset is P.ENGINE_OFFSET and mv.calibration == P.MEWMA_CALIBRATION
    vs = VirtualSensor()
    assert (vs.ewma_alpha, vs.threshold_sigma, vs.healthy_cycle_limit) == (P.EWMA_ALPHA, P.THRESHOLD_SIGMA, P.H_NATIVE)
    assert vs.engine_offset is P.ENGINE_OFFSET


def test_monitoring_layer_uses_the_native_mewma_calibration():
    """A camada de monitoramento usa o limite sintético (emenda de 2026-11-18); o padrão da classe e a injeção seguem empíricos."""
    from pdm_cmaps.monitoring import layer
    assert P.MEWMA_CALIBRATION_NATIVE == "synthetic" and P.MEWMA_CALIBRATION == "empirical"
    assert layer.MEWMA_CALIBRATION_MONITORING == P.MEWMA_CALIBRATION_NATIVE
    assert inspect.signature(layer.compute_monitoring).parameters["mewma_calibration"].default == P.MEWMA_CALIBRATION_NATIVE
    assert inspect.signature(layer.run_one).parameters["mewma_calibration"].default == P.MEWMA_CALIBRATION_NATIVE
    src = (SRC / "monitoring" / "layer.py").read_text(encoding="utf-8")
    assert re.search(r'"--mewma-calibration".{0,120}default=MEWMA_CALIBRATION_MONITORING', src, re.S), "o padrão da CLI deve ser a constante do protocolo"
    assert re.search(r"MultivariateResidualMonitor\([^)]*calibration=mewma_calibration", src, re.S), "a camada deixou de passar a calibração ao MEWMA"
    for f in ("fault_detection_study.py", "fault_hard_study.py", "fault_model_study.py", "healthy_window_study.py", "fault_injection.py"):
        assert "calibration=" not in (SRC / "experiments" / f).read_text(encoding="utf-8"), f"{f}: a injeção deve manter o limite empírico (padrão da classe)"


def test_schema_constants_match_the_protocol():
    from pdm_cmaps.domain.schema import DEFAULT_HEALTHY_CYCLE_LIMIT, DEFAULT_RUL_CAP
    assert DEFAULT_HEALTHY_CYCLE_LIMIT == P.H_NATIVE and DEFAULT_RUL_CAP == P.RUL_CAP


@pytest.mark.parametrize("path", STUDIES, ids=lambda p: p.name)
def test_every_study_uses_the_protocol_seeds(path):
    for d in _argparse_defaults(path, "--seeds"):
        if path.name in {"exploracao_r2.py", "mewma_calibracao_sintetica.py", "benchmark_oficial_completo.py"}:
            continue  # estudos de apoio declaram as próprias sementes (a exploração usa mais; o benchmark oficial usa 5 no `main` e 3 nas redes extras)
        expected = P.SEEDS_RUL if path.name == "benchmark_repeated.py" else P.SEEDS
        assert tuple(d) == expected, f"{path.name}: --seeds {d} != {expected}"


def test_study_cli_defaults_match_the_protocol():
    hard = SRC / "experiments" / "fault_hard_study.py"
    assert _argparse_defaults(hard, "--H") == [P.H_INJECTION]
    assert _argparse_defaults(hard, "--n-reps") == [P.N_REPS_HARD]
    for f in ("fault_detection_study.py", "fault_hard_study.py", "fault_model_study.py", "healthy_window_study.py"):
        assert _argparse_defaults(SRC / "experiments" / f, "--mewma-offset") in ([True], []), f
    assert _argparse_defaults(SRC / "experiments" / "fault_detection_study.py", "--engine-offset") == [P.ENGINE_OFFSET]


def test_persistence_and_split_constants_in_the_studies():
    from pdm_cmaps.experiments import envelope_ignored_cases, native_offset_fora_janela
    assert native_offset_fora_janela.K == P.PERSISTENCE_K and envelope_ignored_cases.PERSIST == P.PERSISTENCE_K
    assert native_offset_fora_janela.W_SPLIT == P.W_SPLIT and native_offset_fora_janela.H_NATIVE == P.H_NATIVE
    from pdm_cmaps.experiments.virtual_sensor_models import run_subset
    assert inspect.signature(run_subset).parameters["min_consecutive"].default == P.PERSISTENCE_K


def test_every_engine_split_uses_the_protocol_test_ratio():
    ratios = set()
    for path in SRC.rglob("*.py"):
        ratios |= set(re.findall(r"test_ratio\s*=\s*([0-9.]+)", path.read_text(encoding="utf-8")))
    ratios.discard("")
    assert {float(r) for r in ratios} <= {P.TEST_RATIO}, ratios


def test_rul_injection_keeps_the_mewma_without_offset_exception():
    src = (SRC / "experiments" / "fault_injection.py").read_text(encoding="utf-8")
    assert P.ENGINE_OFFSET_MEWMA_IN_RUL_INJECTION is False
    assert re.search(r"MultivariateResidualMonitor\([^)]*engine_offset=False", src, re.S), "a exceção da injeção de RUL deixou de existir no código"

In [ ]:
%%writefile projeto/tests/test_reporting.py
"""Figuras (matplotlib e Plotly)."""

from pathlib import Path

import numpy as np
import pandas as pd

from factories import _toy_timeline


def skipped_ids(skipped):
    return [c for c, _ in skipped]


def test_todo7_todo8_plots_execute_without_error(tmp_path: Path):
    """TODO-7 & TODO-8: Test plots execute cleanly with multi-regime discrete points and physical units."""
    from pdm_cmaps.reporting.plots import (
        plot_failure_detection,
        plot_time_series_envelope_detection,
    )
    cycles = np.arange(1, 40)
    trace = pd.DataFrame({
        "unit_id": 1,
        "cycle": cycles,
        "regime_id": cycles % 6,
        "BPR": 8.4 + (cycles % 6) * 0.4,
        "prediction": 8.4 + (cycles % 6) * 0.4,
        "ewma_abs_z": np.linspace(0.5, 3.5, 39),
        "ewma_ucl": np.full(39, 3.0),
        "envelope_upper": np.full(39, 11.5),
        "envelope_lower": np.full(39, 8.0),
        "out_of_envelope": cycles > 35,
        "breach_distance": np.maximum(0, cycles - 35) * 0.1,
    })
    engines = pd.DataFrame([{
        "unit_id": 1,
        "eol_cycle": 39,
        "first_alarm_cycle": 36,
        "lead_time_cycles": 3,
        "detected": True,
    }])
    summary = {
        "unit_id": 1,
        "eol_cycle": 39,
        "first_alarm_cycle": 36,
        "lead_time_cycles": 3,
        "envelope_margin": 1.5,
        "in_envelope_pct": 90.0,
    }

    p1 = plot_failure_detection(trace, engines, tmp_path / "fail.png", healthy_cycle_limit=20, sensor_name="BPR")
    assert p1.exists()

    p2 = plot_time_series_envelope_detection(trace, summary, tmp_path / "env.png", sensor_name="BPR", healthy_cycle_limit=20)
    assert p2.exists()


def test_monitoring_figures_render_and_follow_style_tokens(tmp_path):
    """Every thesis figure function writes a PNG; viz tokens keep the validated categorical order."""
    import numpy as np
    import pandas as pd
    from pdm_cmaps.reporting import monitoring_figures as mf
    from pdm_cmaps.reporting.viz_style import SERIES, STATUS

    assert SERIES == ("#2a78d6", "#eb6834", "#1baf7a")  # validated all-pairs (see viz_style.py)
    assert len(STATUS) == 4

    tl, _ = _toy_timeline()
    mf.plot_rul_trajectories(tl, [7], tmp_path / "traj.png")
    trace = tl.assign(z_score=np.linspace(-1, 5, len(tl)), ewma_abs_z=np.linspace(0.5, 3, len(tl)), ewma_ucl=1.2)
    mf.plot_residual_control_chart(trace, 7, 45, tmp_path / "control.png", 10)
    sens = pd.DataFrame({
        "Strategy": ["point", "interval", "interval+anomaly"] * 2, "Policy": ["critical<=10"] * 3 + ["critical<=15"] * 3,
        "False_alarm_rate_normal": [0.0, 0.01, 0.01, 0.0, 0.02, 0.02], "Missed_critical_rate": [0.03, 0.0, 0.0, 0.02, 0.0, 0.0],
    })
    mf.plot_strategy_tradeoff(sens, tmp_path / "tradeoff.png")
    mf.plot_strategy_tradeoff_repeated(sens.assign(Subset="FD001", Seed=1), tmp_path / "repeated.png")
    ci = pd.DataFrame({"Model": ["A", "B"], "R2_Observed": [0.8, 0.7], "R2_CI_Lower": [0.7, 0.6], "R2_CI_Upper": [0.85, 0.78]})
    mf.plot_model_ci(ci, tmp_path / "ci.png")
    mf.plot_regime_variance(pd.DataFrame({"Subset": ["FD001", "FD002"], "Regimes": [1, 6], "Between_regime_pct": [0.0, 99.6]}), tmp_path / "regime.png")
    assert len(list(tmp_path.glob("*.png"))) == 6


def test_plotly_hub_builds_index_skips_missing_and_writes_explorer(tmp_path):
    from pdm_cmaps.reporting import plotly_hub as ph

    res, out = tmp_path / "results", tmp_path / "out"
    (res / "monitoring_repeated" / "tables").mkdir(parents=True)
    pd.DataFrame({"Subset": ["FD001", "FD002"], "Regimes": [1, 6], "Between_regime_pct_mean": [0.0, 99.5], "Between_regime_pct_std": [0.0, 0.02],
                  "OOS_R2_mean": [-0.02, 0.99], "Runs": [3, 3]}).to_csv(res / "monitoring_repeated" / "tables" / "regime_variance_by_subset.csv", index=False)
    built, skipped = ph.build(ph.Ctx(results=res, out=out), only=["t01", "t02"])
    assert [c.id for c in built] == ["t01"] and [c.id for c in skipped_ids(skipped)] == ["t02"]
    index = (out / "index.html").read_text(encoding="utf-8")
    assert "t01_" in index and "indisponível" in index and (out / "assets" / "plotly.min.js").stat().st_size > 1_000_000
    page = (out / "tcc" / next(c.filename for c in built)).read_text(encoding="utf-8")
    assert "../assets/plotly.min.js" in page and "Índice dos gráficos" in page
    sets = {"FD004": {"names": ["A"], "units": {"7": {"x": [1, 2], "truth": [125.0, 124.0], "series": {"A": [120.0, 118.0]}, "markers": []}}}}
    ph.explorer_page(out / "tcc" / "x.html", "T", "S", sets, {"scatter": True, "defaultOn": ["A"]})
    text = (out / "tcc" / "x.html").read_text(encoding="utf-8")
    assert '"FD004"' in text and "__DATA__" not in text and "__CONFIG__" not in text

In [ ]:
%%writefile projeto/tests/test_scripts.py
"""O runner de planos (scripts/rodar.py): tentativas, retomada, dependências entre raias, pool e os planos reais."""

import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "scripts"))
from rodar import Plan, Runner, Step, main, status  # noqa: E402

PY = sys.executable


def sh(code: str) -> tuple[str, ...]:
    return (PY, "-c", code)


def append_to(root: Path, name: str) -> str:
    """Código Python que registra uma execução do passo `name` (para contar quantas vezes rodou)."""
    return f"open({str(root / 'execucoes.txt')!r}, 'a').write({name!r} + chr(10))"


def runs(root: Path) -> list[str]:
    f = root / "execucoes.txt"
    return f.read_text().split() if f.exists() else []


def test_runs_steps_in_lane_order_and_marks_them_done(tmp_path):
    plan = Plan("p", "", lanes={"A": [Step("um", sh(append_to(tmp_path, "um"))), Step("dois", sh(append_to(tmp_path, "dois")))]})
    assert Runner(plan, root=tmp_path).run() == 0
    assert runs(tmp_path) == ["um", "dois"]
    assert {p.name for p in (tmp_path / "results/logs/p/done").iterdir()} == {"um", "dois"}


def test_retries_a_failing_step_then_succeeds(tmp_path):
    flaky = f"import pathlib,sys; f=pathlib.Path({str(tmp_path / 'tentativa')!r}); n=int(f.read_text() or 0) if f.exists() else 0; f.write_text(str(n+1)); sys.exit(1 if n < 1 else 0)"
    plan = Plan("p", "", lanes={"A": [Step("instavel", sh(flaky))]})
    assert Runner(plan, tries=3, root=tmp_path).run() == 0
    events = (tmp_path / "results/logs/p/progresso.tsv").read_text()
    assert "falha" in events and "\tok\t2\t" in events  # falhou na 1ª, passou na 2ª


def test_a_step_whose_result_check_fails_counts_as_failure(tmp_path):
    plan = Plan("p", "", lanes={"A": [Step("sem_resultado", sh("pass"), check=lambda ctx: False)]})
    assert Runner(plan, tries=2, root=tmp_path).run() == 1
    assert "verificação do resultado falhou" in (tmp_path / "results/logs/p/progresso.tsv").read_text()


def test_resume_skips_finished_steps_and_redo_runs_them_again(tmp_path):
    plan = Plan("p", "", lanes={"A": [Step("um", sh(append_to(tmp_path, "um")))]})
    Runner(plan, root=tmp_path).run()
    Runner(plan, root=tmp_path).run()  # retomada: nada a fazer
    assert runs(tmp_path) == ["um"]
    Runner(plan, redo=True, root=tmp_path).run()
    assert runs(tmp_path) == ["um", "um"]


def test_failure_gives_up_cancels_dependents_and_skips_final_steps(tmp_path):
    plan = Plan("p", "", lanes={
        "A": [Step("quebra", sh("raise SystemExit(3)"))],
        "B": [Step("depende", sh(append_to(tmp_path, "depende")), after=("quebra",)), Step("independente", sh(append_to(tmp_path, "independente")))]},
        final=[Step("sync", sh(append_to(tmp_path, "sync")))])
    assert Runner(plan, tries=2, root=tmp_path).run() == 1
    assert runs(tmp_path) == ["independente"]  # o dependente foi cancelado, a outra tarefa seguiu, o final não rodou
    assert "cancelada" in (tmp_path / "results/logs/p/progresso.tsv").read_text()


def test_dependency_waits_for_the_step_in_another_lane(tmp_path):
    plan = Plan("p", "", lanes={
        "A": [Step("depois", sh(append_to(tmp_path, "depois")), after=("antes",))],
        "B": [Step("antes", sh(f"import time; time.sleep(0.5); {append_to(tmp_path, 'antes')}"))]})
    assert Runner(plan, root=tmp_path).run() == 0
    assert runs(tmp_path) == ["antes", "depois"]


def test_pool_tasks_run_exactly_once_across_lanes(tmp_path):
    pool = [Step(f"t{i}", sh(append_to(tmp_path, f"t{i}"))) for i in range(6)]
    plan = Plan("p", "", lanes={"A": [], "B": []}, pool=pool)
    assert Runner(plan, root=tmp_path).run() == 0
    assert sorted(runs(tmp_path)) == [f"t{i}" for i in range(6)]


def test_clean_patterns_are_removed_before_each_attempt(tmp_path):
    old = tmp_path / "results" / "runs"
    old.mkdir(parents=True)
    (old / "antigo.csv").write_text("x")
    plan = Plan("p", "", lanes={"A": [Step("limpa", sh("pass"), clean=("results/runs",))]})
    Runner(plan, root=tmp_path).run()
    assert not old.exists()


def test_dry_run_executes_and_writes_nothing(tmp_path, capsys):
    plan = Plan("p", "descrição", lanes={"A": [Step("x", sh(append_to(tmp_path, "x")), est=90)]})
    assert Runner(plan, dry_run=True, root=tmp_path).run() == 0
    assert "1min" in capsys.readouterr().out and runs(tmp_path) == [] and not (tmp_path / "results").exists()


def test_status_reports_done_and_pending_steps(tmp_path, capsys):
    plan = Plan("p", "", lanes={"A": [Step("feito", sh("pass")), Step("quebra", sh("raise SystemExit(1)")), Step("nunca", sh("pass"), after=("quebra",))]})
    Runner(plan, tries=1, root=tmp_path).run()
    status(plan, root=tmp_path)
    out = capsys.readouterr().out
    assert "1/3 passos concluídos" in out and "DESISTIU" in out and "feito" in out


def test_progress_line_has_counter_and_estimate(tmp_path, capsys):
    plan = Plan("p", "", lanes={"A": [Step("um", sh("pass"), est=10), Step("dois", sh("pass"), est=10)]})
    Runner(plan, root=tmp_path).run()
    out = capsys.readouterr().out
    assert "[1/2] um ... ok" in out and "[2/2] dois ... ok" in out and "restante" in out


def test_every_real_plan_is_well_formed(capsys):
    from planos import PLANOS

    for name in PLANOS:
        assert main(["rodar", name, "--dry-run"]) == 0, name
    assert "pytest" in capsys.readouterr().out  # o plano `tudo` roda a suíte


def test_the_everything_plan_uses_the_same_study_list_as_update_all():
    from planos import PLANOS

    from pdm_cmaps.cli.update_all import STUDIES

    steps = {s.name for s in PLANOS["tudo"]().all_steps()}
    assert {s.name for s in STUDIES} <= steps


@pytest.mark.parametrize("cmd", [["lista"], ["status", "tudo"]])
def test_list_and_status_commands_run(cmd, capsys):
    assert main(cmd) == 0
    assert capsys.readouterr().out

In [ ]:
%%writefile projeto/tests/test_selecao_hiperparametros.py
"""Busca de hiperparâmetros (regra do erro-padrão, janela, validação por motor) e as métricas novas de RUL, sensores virtuais e alertas."""
from __future__ import annotations

import inspect

import numpy as np
import pandas as pd
import pytest

from pdm_cmaps.detection.virtual_sensor_metrics import error_metrics
from pdm_cmaps.domain.rul import compute_piecewise_rul, extended_rul_metrics
from pdm_cmaps.monitoring.alerts import CriticalityPolicy, evaluate_strategies
from factories import make_synthetic_cmapss

SENSORS = ["T24", "T30", "T50", "P30", "Nf", "Nc"]


def test_extended_rul_metrics_hand_computed():
    m = extended_rul_metrics([5, 12, 30, 60], [8, 10, 40, 50])  # d = +3, -2, +10, -10
    assert m["Bias"] == pytest.approx(0.25)
    assert m["Median_AE"] == pytest.approx(6.5)
    assert m["P90_AE"] == pytest.approx(10.0)
    assert m["Hit_within_10"] == 1.0 and m["Hit_within_20"] == 1.0
    assert (m["Precision_c15"], m["Recall_c15"], m["F1_c15"]) == (1.0, 1.0, 1.0)
    assert m["Precision_c10"] == pytest.approx(0.5) and m["Recall_c10"] == 1.0 and m["F1_c10"] == pytest.approx(2 / 3)
    assert m["PR_AUC_c15"] == pytest.approx(1.0)  # os dois motores críticos têm as duas menores RUL previstas


def test_extended_rul_metrics_class_without_members_is_nan():
    m = extended_rul_metrics([100, 120, 130], [90, 110, 140])
    assert np.isnan(m["PR_AUC_c10"]) and np.isnan(m["Precision_c10"]) and np.isnan(m["Recall_c10"])


def test_virtual_sensor_error_metrics():
    g = pd.DataFrame({"actual": [1.0, 2.0, 3.0, 4.0], "prediction": [1.0, 2.0, 3.0, 6.0], "regime_id": [0, 0, 1, 1]})
    mae, rmse, nrmse = error_metrics(g)
    assert mae == pytest.approx(0.5) and rmse == pytest.approx(1.0)
    assert nrmse == pytest.approx(2.0)  # desvio em torno da média de cada regime = 0,5


def test_alert_precision_recall_f1_per_strategy():
    t = pd.DataFrame({"unit_id": 1, "cycle": range(1, 7), "RUL_true": [50, 40, 30, 20, 10, 1],
                      "level_true": [0, 0, 1, 2, 3, 3], "level_point": [0, 1, 1, 2, 3, 0]})
    for other in ("level_interval", "level_interval+anomaly", "level_interval+anomaly_mv"):
        t[other] = t["level_point"]
    row = evaluate_strategies(t, CriticalityPolicy()).set_index("Strategy").loc["point"]
    assert row["Alert_precision"] == 1.0
    assert row["Alert_recall"] == pytest.approx(2 / 3)
    assert row["Alert_F1"] == pytest.approx(0.8)


def test_one_standard_error_rule_picks_the_smallest_window_within_one_se():
    optuna = pytest.importorskip("optuna")  # noqa: F841
    from pdm_cmaps.experiments.selecao_hiperparametros import escolher_um_se

    t = pd.DataFrame({"rmse_mean": [14.0, 14.3, 14.4, 16.0], "rmse_se": [0.5, 0.4, 0.4, 0.3], "window": [40, 10, 20, 5], "trial": [0, 1, 2, 3]})
    assert int(escolher_um_se(t)["window"]) == 10  # limite = 14,0 + 0,5; entre 14,0 (40), 14,3 (10) e 14,4 (20) fica a menor janela
    only_best = t.assign(rmse_se=[0.0, 0.0, 0.0, 0.0])
    assert int(escolher_um_se(only_best)["window"]) == 40  # sem erro-padrão, só o melhor


def test_search_space_sets_the_window_by_model_family():
    optuna = pytest.importorskip("optuna")
    from pdm_cmaps.experiments.selecao_hiperparametros import WINDOWS, sugerir

    base = {"representation": "regime", "alpha": 1.0}
    ridge_plain = sugerir(optuna.trial.FixedTrial({**base, "temporal": False}), "Ridge")
    assert ridge_plain["window"] == 0 and ridge_plain["temporal"] is False and ridge_plain["representation"] == "regime"
    ridge_win = sugerir(optuna.trial.FixedTrial({**base, "temporal": True, "window": 30}), "Ridge")
    assert ridge_win["window"] == 30 and 30 in WINDOWS
    lstm = sugerir(optuna.trial.FixedTrial({"representation": "global", "window": 20, "learning_rate": 0.002, "batch_size": 256, "dropout": 0.1,
                                            "hidden_dim": 32, "num_layers": 1}), "PyTorch RNN (LSTM)")
    assert lstm["window"] == 20 and lstm["temporal"] is False and lstm["hyper"]["hidden_dim"] == 32


def test_validation_is_grouped_by_engine_and_runs_with_regime_and_temporal_features():
    pytest.importorskip("optuna")
    from pdm_cmaps.experiments.selecao_hiperparametros import validar

    df = make_synthetic_cmapss(n_engines=9, min_cycles=40, max_cycles=60, seed=5)
    df["RUL"] = compute_piecewise_rul(df)
    params = {"representation": "regime", "temporal": True, "window": 10, "hyper": {"alpha": 10.0}}
    scores = validar(df, SENSORS, 6, "Ridge", params, seed=1, n_splits=3)
    assert len(scores) == 3 and all(np.isfinite(scores)) and all(s > 0 for s in scores)
    again = validar(df, SENSORS, 6, "Ridge", params, seed=1, n_splits=3)
    assert scores == again  # determinístico


def test_model_builders_construct_every_tabular_family():
    pytest.importorskip("optuna")
    from pdm_cmaps.experiments.selecao_hiperparametros import montar_modelo

    for name, hyper in {"XGBoost": {"n_estimators": 10, "max_depth": 3}, "LightGBM": {"n_estimators": 10, "max_depth": 3}, "CatBoost": {"iterations": 10, "depth": 3},
                        "Random Forest": {"n_estimators": 10, "max_depth": 4}, "XGBoost (NASA Loss)": {"n_estimators": 10, "max_depth": 3, "subsample": 0.8}}.items():
        kind, est = montar_modelo(name, {"hyper": hyper, "window": 0}, seed=1)
        assert kind == "sklearn" and hasattr(est, "fit")


def test_the_search_never_reads_the_official_test_file():
    pytest.importorskip("optuna")
    from pdm_cmaps.experiments import selecao_hiperparametros as sh

    assert "load_test" not in inspect.getsource(sh)

In [ ]:
%%writefile projeto/tests/test_virtual_sensor.py
"""Sensor virtual, resíduo e deslocamento por motor (engine offset)."""

import numpy as np
import pandas as pd
import pytest
from pdm_cmaps.detection.virtual_sensor import VirtualSensor

from factories import make_synthetic_cmapss


def _offset_trace(n_engines=6, cycles=120, seed=0, constant=False):
    """Leitura = deslocamento do motor + ruído pequeno + degradação depois do ciclo 60; o modelo prevê a média da frota."""
    rng = np.random.default_rng(seed)
    offsets = rng.normal(0.0, 5.0, n_engines)
    rows = []
    for u in range(n_engines):
        for c in range(1, cycles + 1):
            actual = 100.0 if constant else 100.0 + offsets[u] + rng.normal(0, 0.3) + max(0, c - 60) * 0.5
            rows.append({"sensor": "S", "model": "M", "unit_id": u, "cycle": c, "actual": actual, "prediction": 100.0})
    return pd.DataFrame(rows)


def _mewma_offset_world(n_engines=10, seed=3):
    df = make_synthetic_cmapss(n_engines=n_engines, seed=seed)
    rng = np.random.default_rng(seed)
    for col in ("T50", "T30", "P30"):
        df[col] = df[col] + df["unit_id"].map({u: rng.normal(0, 4.0) for u in df["unit_id"].unique()})
    df["regime_id"] = 0
    return df[df["unit_id"] <= 7], df[df["unit_id"] > 7]


def test_canonical_exogenous_virtual_sensor_and_fallback():
    """Verify VirtualSensor uses canonical exogenous features and handles sparse regimes via fallback."""
    from pdm_cmaps.detection.virtual_sensor import (
        CANONICAL_EXOGENOUS_FEATURES,
        VirtualSensor,
    )

    assert "setting_1" in CANONICAL_EXOGENOUS_FEATURES
    assert "T2" in CANONICAL_EXOGENOUS_FEATURES
    assert "T50" not in CANONICAL_EXOGENOUS_FEATURES  # Target is not exogenous

    df = make_synthetic_cmapss(n_engines=6, seed=42)
    df["regime_id"] = df["cycle"] % 6
    # Fit virtual sensor with canonical features
    vs = VirtualSensor(target_sensor="T50", healthy_cycle_limit=30, random_state=42)
    # Check that it fits without error even if a regime has limited samples
    vs.fit(df)
    assert len(vs.models) > 0
    preds = vs.predict(df)
    assert "prediction" in preds.columns
    assert "residual" in preds.columns
    assert np.isfinite(preds["prediction"]).all()


def test_todo4_virtual_sensor_strictly_exogenous():
    """TODO-4 & F4: Virtual sensor feature_cols in exogenous mode contains only canonical exogenous features."""
    from pdm_cmaps.detection.virtual_sensor import CANONICAL_EXOGENOUS_FEATURES
    train_df = make_synthetic_cmapss(n_engines=4, seed=42)
    train_df["regime_id"] = 0
    vs = VirtualSensor(target_sensor="T50", mode="exogenous")
    vs.fit(train_df)

    # Verify no degraded internal sensors are among feature_cols
    forbidden = {"T24", "T30", "T50", "P30", "Ps30", "BPR", "W31", "W32"}
    assert not set(vs.feature_cols).intersection(forbidden)
    for col in vs.feature_cols:
        assert col in CANONICAL_EXOGENOUS_FEATURES


def test_virtual_sensor_r2_separates_engine_offset_from_degradation():
    from pdm_cmaps.detection.virtual_sensor_metrics import virtual_sensor_r2

    out = virtual_sensor_r2(_offset_trace(), healthy_limit=50).iloc[0]
    assert out["r2_late_model"] < 0.2                 # o modelo é a média da frota: não explica o deslocamento dos motores
    assert out["r2_late_motor_base"] > 0.9            # a média do próprio motor (ciclos 1 a 25) explica quase tudo
    assert out["r2_late_model_offset"] > 0.9          # modelo + deslocamento do motor também
    assert out["pct_between_engine"] > 90.0
    assert out["r2_holdout"] < out["r2_healthy"]      # a degradação afasta a leitura da média saudável


def test_virtual_sensor_r2_constant_sensor_is_nan_and_columns_are_idempotent(tmp_path):
    from pdm_cmaps.detection.virtual_sensor_metrics import virtual_sensor_r2
    from pdm_cmaps.experiments.virtual_sensor_models import R2_COLUMNS, add_r2_columns

    out = virtual_sensor_r2(_offset_trace(constant=True), healthy_limit=50).iloc[0]
    assert np.isnan(out["r2_holdout"]) and np.isnan(out["pct_between_engine"])

    tr = _offset_trace()
    tr.to_csv(tmp_path / "trace.csv.gz", index=False)
    pd.DataFrame([{"sensor": "S", "model": "M", "rmse_holdout": 1.0}]).to_csv(tmp_path / "model_scores.csv", index=False)
    first = add_r2_columns(tmp_path, 50)
    second = add_r2_columns(tmp_path, 50)               # rodar de novo não duplica colunas
    assert list(first.columns) == list(second.columns) == ["sensor", "model", "rmse_holdout"] + R2_COLUMNS


def test_virtual_sensor_engine_offset_removes_engine_to_engine_shift():
    """Com engine_offset=True o z de cada motor sai centrado em zero na janela saudável; sem ele, o deslocamento do motor sobra."""
    from pdm_cmaps.detection.virtual_sensor import VirtualSensor

    df = make_synthetic_cmapss(n_engines=10, seed=3)
    rng = np.random.default_rng(3)
    df["T50"] = df["T50"] + df["unit_id"].map({u: rng.normal(0, 8.0) for u in df["unit_id"].unique()})  # deslocamento por motor
    df["regime_id"] = 0
    train, test = df[df["unit_id"] <= 7], df[df["unit_id"] > 7]
    H = 30

    def window_means(offset):
        vs = VirtualSensor(target_sensor="T50", healthy_cycle_limit=H, random_state=0, engine_offset=offset, min_consecutive_breaches=2)
        scored, _, _ = vs.evaluate(train, test)
        return scored[scored["cycle"] <= H].groupby("unit_id")["z_score"].mean().abs().mean(), vs

    plain, _ = window_means(False)
    centred, vs = window_means(True)
    assert centred < 0.05 < plain            # z médio por motor na janela: ~0 com o deslocamento removido
    assert vs.engine_z_scale_ > 0
    default_vs = VirtualSensor(target_sensor="T50", healthy_cycle_limit=H, random_state=0)
    assert default_vs.engine_offset is True   # padrão adotado; engine_offset=False reproduz o comportamento anterior
    assert VirtualSensor(target_sensor="T50", engine_offset=False).engine_offset is False


def test_virtual_sensor_engine_offset_is_per_regime_and_keeps_single_regime_behaviour():
    """Em multirregime o deslocamento é por (motor, regime): um regime de σ grande não pode distorcer a previsão do outro."""
    from pdm_cmaps.detection.virtual_sensor import VirtualSensor

    vs = VirtualSensor(target_sensor="T50", healthy_cycle_limit=20, engine_offset=True)
    z = pd.DataFrame({"unit_id": [1] * 8, "cycle": [1, 2, 3, 4, 5, 6, 7, 8], "regime_id": [0, 0, 0, 0, 1, 1, 1, 9],
                      "z_score": [2.0, 2.0, 2.0, 2.0, 100.0, 100.0, 100.0, 5.0]})
    off = vs._engine_regime_offsets(z)
    assert off.loc[(1, 0)] == 2.0 and off.loc[(1, 1)] == 100.0     # cada regime com a sua média
    assert off.loc[(1, 9)] == 0.0                                     # menos de 3 linhas: sem correção
    looked = vs._lookup_offsets(z, off)
    assert looked.tolist() == [2.0] * 4 + [100.0] * 3 + [0.0]


def test_mewma_engine_offset_removes_engine_level():
    """Com engine_offset, o z de cada motor tem média 0 nos primeiros H ciclos (por regime); desde 2026-11-18 o padrão é ligado, como no T50."""
    import numpy as np
    import pandas as pd
    from pdm_cmaps.detection.multivariate import MultivariateResidualMonitor

    assert MultivariateResidualMonitor().engine_offset is True   # padrão do protocolo principal desde 2026-11-18
    assert MultivariateResidualMonitor(engine_offset=False).engine_offset is False
    mon = MultivariateResidualMonitor(healthy_cycle_limit=10, engine_offset=True)
    rng = np.random.default_rng(0)
    rows = []
    for unit, level in ((1, 5.0), (2, -3.0)):
        for cycle in range(1, 31):
            rows.append({"unit_id": unit, "cycle": cycle, "regime_id": 0, "z": level + rng.normal(0, 0.1)})
    df = pd.DataFrame(rows)
    out = mon._remove_offset(df["z"].to_numpy(), df)
    for unit in (1, 2):
        m = (df["unit_id"] == unit) & (df["cycle"] <= 10)
        assert abs(out[m.to_numpy()].mean()) < 1e-9
    off = MultivariateResidualMonitor(healthy_cycle_limit=10, engine_offset=False)
    assert (off._remove_offset(df["z"].to_numpy(), df) == df["z"].to_numpy()).all()


def test_model_scores_apply_the_engine_offset_like_score():
    """`model_scores` (fault_model_study e fault_hard_study) precisa dar o mesmo T² que `MultivariateResidualMonitor.score` com o deslocamento ligado."""
    from pdm_cmaps.detection.multivariate import MultivariateResidualMonitor
    from pdm_cmaps.experiments.fault_model_study import MEWMA, model_scores

    train, test = _mewma_offset_world()
    H = 30
    for offset in (False, True):
        mv = MultivariateResidualMonitor(sensors=("T50", "T30", "P30"), healthy_cycle_limit=H, random_state=0, engine_offset=offset).fit(train)
        scored = mv.score(test).sort_values(["unit_id", "cycle"]).reset_index(drop=True)
        scores, _ = model_scores(mv, test, set(), H)
        ratio = scores[MEWMA].sort_values(["blind_id", "cycle"]).reset_index(drop=True)["ratio"].to_numpy()
        assert np.allclose(ratio, (scored["t2"] / scored["ucl"]).to_numpy())
        # gráfico por sensor: o limite da EWMA foi calibrado na escala do z que o alimenta (média de |z| ~ a calibrada)
        z = mv.chart_z(mv.virtual_["T50"], mv.virtual_["T50"].predict(test))
        calib_mean = mv.virtual_["T50"].ewma_detector.abs_z_mean
        assert 0.5 * calib_mean < np.abs(z[test["cycle"].to_numpy() <= H]).mean() < 2.0 * calib_mean


def test_mewma_out_of_window_offset_calibrates_and_measures_after_the_offset_window():
    """Com offset_window < H e calibrate_out_of_window, μ, Σ e o limite saem só dos ciclos (W, H]; o falso alarme da janela também; o padrão não muda."""
    from pdm_cmaps.detection.multivariate import MultivariateResidualMonitor

    train, test = _mewma_offset_world()
    H, W = 30, 15
    common = dict(sensors=("T50", "T30", "P30"), healthy_cycle_limit=H, random_state=0, engine_offset=True)
    inside = MultivariateResidualMonitor(**common, offset_window=W).fit(train)
    outside = MultivariateResidualMonitor(**common, offset_window=W, calibrate_out_of_window=True).fit(train)
    assert MultivariateResidualMonitor().calibrate_out_of_window is False
    assert inside.ucl_ != outside.ucl_
    assert len(outside.calibration_t2_) < len(inside.calibration_t2_)
    # o limite fora da janela vem do quantil dos T² calibrados só nos ciclos (W, H]
    assert np.isclose(outside.ucl_, np.quantile(outside.calibration_t2_["t2"], 1 - outside.alpha))
    scored = outside.score(test)
    _, per, _ = outside.alarms(scored)
    manual = scored.assign(win=(scored["cycle"] > W) & (scored["cycle"] <= H)).query("win").groupby("unit_id")["breach"].mean()
    assert np.allclose(per.set_index("unit_id")["healthy_far"].sort_index().to_numpy(), manual.sort_index().to_numpy())
    assert 0.0 <= outside.offset_uncovered_pct(test) <= 100.0


def test_virtual_sensor_out_of_window_calibration_uses_only_cycles_after_the_offset_window():
    from pdm_cmaps.detection.virtual_sensor import VirtualSensor

    train, test = _mewma_offset_world()
    H, W = 30, 15
    vs = VirtualSensor(target_sensor="T50", healthy_cycle_limit=H, random_state=0, engine_offset=True, offset_window=W, calibrate_out_of_window=True)
    scored, per, _ = vs.evaluate(train, test)
    cp = vs.calibration_predictions_
    off = vs._engine_regime_offsets(cp)
    z_after = (cp["z_score"] - vs._lookup_offsets(cp, off))[cp["cycle"] > W]
    assert np.isclose(vs.engine_z_scale_, z_after.std())
    base = VirtualSensor(target_sensor="T50", healthy_cycle_limit=H, random_state=0, engine_offset=True, offset_window=W)
    base.evaluate(train, test)
    assert not np.isclose(base.engine_z_scale_, vs.engine_z_scale_)
    assert (VirtualSensor(target_sensor="T50", engine_offset=True).calibrate_out_of_window is False)


def test_virtual_sensor_r2_within_regime_equals_raw_in_single_regime():
    from pdm_cmaps.detection.virtual_sensor_metrics import virtual_sensor_r2

    tr = _offset_trace()                                # sem regime_id: regime único
    out = virtual_sensor_r2(tr, healthy_limit=50).iloc[0]
    assert out["r2_late_within_regime"] == pytest.approx(out["r2_late_model"])
    assert out["r2_holdout_within_regime"] == pytest.approx(out["r2_holdout"])


def test_virtual_sensor_r2_within_regime_removes_regime_and_ceiling_ignores_regime_jumps():
    """Multirregime: o regime domina a variância (R² bruto ~1), mas dentro do regime sobra só ruído e deslocamento do motor;
    o teto de ruído NÃO pode contar o salto de regime como ruído (senão sairia muito negativo)."""
    from pdm_cmaps.detection.virtual_sensor_metrics import virtual_sensor_r2

    rng = np.random.default_rng(0)
    rows = []
    for u in range(1, 21):
        off = rng.normal(0, 3.0)                        # deslocamento do motor (a previsão não o conhece)
        for c in range(1, 51):
            reg = int(rng.integers(0, 3))
            level = 100.0 * reg
            rows.append({"sensor": "S", "model": "M", "unit_id": u, "cycle": c, "regime_id": reg,
                         "actual": level + off + rng.normal(0, 1.0), "prediction": level})
    out = virtual_sensor_r2(pd.DataFrame(rows), healthy_limit=50).iloc[0]
    assert out["r2_late_model"] > 0.99                  # bruto: quase perfeito só pelo regime
    assert out["r2_late_within_regime"] < 0.2           # dentro do regime: nada explicado
    assert 0.7 < out["noise_ceiling_late"] < 1.0        # ruído ~1 contra variância ~10 dentro do regime
    assert out["frac_ceiling"] < 0.3

## 3. Dados

### 3.1 Hiperparâmetros e janelas escolhidos (busca Optuna com validação cruzada por motor, já feita)

### Arquivos `data/hiperparametros`

Permitem refazer o teste oficial ajustado sem repetir a busca (dias de máquina).

80 arquivos.

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/CatBoost/melhores.json
{
  "subset": "FD001",
  "model": "CatBoost",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 60,
    "hyper": {
      "iterations": 800,
      "depth": 4,
      "learning_rate": 0.05483455610369595,
      "l2_leaf_reg": 9.570956018768351
    }
  },
  "cv_rmse": 11.506365209868187,
  "cv_se": 0.16947411551912708,
  "melhor_rmse_medio": 11.506365209868187,
  "melhor_trial": 16,
  "trial_escolhida": 16,
  "n_tentativas_completas": 26,
  "n_tentativas_total": 40,
  "n_splits": 5,
  "seed": 42,
  "segundos": 5464.9
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/LightGBM/melhores.json
{
  "subset": "FD001",
  "model": "LightGBM",
  "params": {
    "representation": "global",
    "temporal": true,
    "window": 60,
    "hyper": {
      "n_estimators": 250,
      "max_depth": 5,
      "learning_rate": 0.04062407513338108,
      "num_leaves": 44,
      "subsample": 0.6388007000471191,
      "subsample_freq": 1,
      "colsample_bytree": 0.5984856118921611,
      "min_child_samples": 90,
      "reg_lambda": 0.3041691985549422
    }
  },
  "cv_rmse": 11.297490231468299,
  "cv_se": 0.22665631534558608,
  "melhor_rmse_medio": 11.297490231468299,
  "melhor_trial": 30,
  "trial_escolhida": 30,
  "n_tentativas_completas": 44,
  "n_tentativas_total": 60,
  "n_splits": 5,
  "seed": 42,
  "segundos": 1370.1
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/MLP/melhores.json
{
  "subset": "FD001",
  "model": "MLP",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 50,
    "hyper": {
      "hidden_layer_sizes": [
        256,
        128
      ],
      "alpha": 0.0033674896293976685,
      "learning_rate_init": 0.000967342370900682
    }
  },
  "cv_rmse": 10.912039614875557,
  "cv_se": 0.2373611597104747,
  "melhor_rmse_medio": 10.912039614875557,
  "melhor_trial": 26,
  "trial_escolhida": 26,
  "n_tentativas_completas": 24,
  "n_tentativas_total": 30,
  "n_splits": 5,
  "seed": 42,
  "segundos": 2699.0
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/PyTorch_CNN/melhores.json
{
  "subset": "FD001",
  "model": "PyTorch CNN",
  "params": {
    "representation": "regime",
    "window": 10,
    "temporal": false,
    "hyper": {
      "learning_rate": 0.000520889112475312,
      "batch_size": 256,
      "dropout": 0.28060091877975857,
      "num_filters": 32,
      "kernel_size": 5
    }
  },
  "cv_rmse": 16.38529160007262,
  "cv_se": 1.217152560075133,
  "melhor_rmse_medio": 16.38529160007262,
  "melhor_trial": 14,
  "trial_escolhida": 14,
  "n_tentativas_completas": 19,
  "n_tentativas_total": 20,
  "n_splits": 3,
  "seed": 42,
  "segundos": 246.7
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/PyTorch_RNN_LSTM/melhores.json
{
  "subset": "FD001",
  "model": "PyTorch RNN (LSTM)",
  "params": {
    "representation": "regime",
    "window": 50,
    "temporal": false,
    "hyper": {
      "learning_rate": 0.002552951604697378,
      "batch_size": 256,
      "dropout": 0.08493564427131046,
      "hidden_dim": 128,
      "num_layers": 2
    }
  },
  "cv_rmse": 11.235369964041444,
  "cv_se": 0.5110890308642027,
  "melhor_rmse_medio": 10.359862160677423,
  "melhor_trial": 15,
  "trial_escolhida": 0,
  "n_tentativas_completas": 17,
  "n_tentativas_total": 20,
  "n_splits": 3,
  "seed": 42,
  "segundos": 0.0
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/PyTorch_Transformer/melhores.json
{
  "subset": "FD001",
  "model": "PyTorch Transformer",
  "params": {
    "representation": "global",
    "window": 40,
    "temporal": false,
    "hyper": {
      "learning_rate": 0.0008450942890954814,
      "batch_size": 512,
      "dropout": 0.0027808522124762817,
      "d_model": 32,
      "nhead": 4,
      "num_layers": 3,
      "dim_feedforward": 256
    }
  },
  "cv_rmse": 13.89894163537785,
  "cv_se": 0.3471401760895673,
  "melhor_rmse_medio": 13.89894163537785,
  "melhor_trial": 5,
  "trial_escolhida": 5,
  "n_tentativas_completas": 18,
  "n_tentativas_total": 20,
  "n_splits": 3,
  "seed": 42,
  "segundos": 291.4
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/Random_Forest/melhores.json
{
  "subset": "FD001",
  "model": "Random Forest",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 60,
    "hyper": {
      "n_estimators": 50,
      "max_depth": 22,
      "min_samples_leaf": 6,
      "max_features": 0.5502433808472529
    }
  },
  "cv_rmse": 12.54326683900263,
  "cv_se": 0.3768556934644988,
  "melhor_rmse_medio": 12.54326683900263,
  "melhor_trial": 34,
  "trial_escolhida": 34,
  "n_tentativas_completas": 30,
  "n_tentativas_total": 40,
  "n_splits": 5,
  "seed": 42,
  "segundos": 16061.9
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/Ridge/melhores.json
{
  "subset": "FD001",
  "model": "Ridge",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 50,
    "hyper": {
      "alpha": 53.861200162061195
    }
  },
  "cv_rmse": 15.046012343043424,
  "cv_se": 0.31660171036323903,
  "melhor_rmse_medio": 14.757168740549364,
  "melhor_trial": 26,
  "trial_escolhida": 23,
  "n_tentativas_completas": 23,
  "n_tentativas_total": 30,
  "n_splits": 5,
  "seed": 42,
  "segundos": 0.0
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/XGBoost/melhores.json
{
  "subset": "FD001",
  "model": "XGBoost",
  "params": {
    "representation": "global",
    "temporal": true,
    "window": 60,
    "hyper": {
      "n_estimators": 300,
      "max_depth": 5,
      "learning_rate": 0.05054375608872033,
      "subsample": 0.5985106605947993,
      "colsample_bytree": 0.7155905690443694,
      "min_child_weight": 8,
      "reg_lambda": 0.06630102558709826
    }
  },
  "cv_rmse": 11.437375296966904,
  "cv_se": 0.2893139285933722,
  "melhor_rmse_medio": 11.437375296966904,
  "melhor_trial": 53,
  "trial_escolhida": 53,
  "n_tentativas_completas": 34,
  "n_tentativas_total": 60,
  "n_splits": 5,
  "seed": 42,
  "segundos": 3500.4
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/XGBoost_NASA_Loss/melhores.json
{
  "subset": "FD001",
  "model": "XGBoost (NASA Loss)",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 60,
    "hyper": {
      "n_estimators": 600,
      "max_depth": 10,
      "learning_rate": 0.024751435372469367,
      "subsample": 0.6144549547485179,
      "colsample_bytree": 0.5773599687412635,
      "min_child_weight": 6,
      "reg_lambda": 1.884080422775671
    }
  },
  "cv_rmse": 11.344605138905449,
  "cv_se": 0.3284998674280768,
  "melhor_rmse_medio": 11.344605138905449,
  "melhor_trial": 56,
  "trial_escolhida": 56,
  "n_tentativas_completas": 43,
  "n_tentativas_total": 60,
  "n_splits": 5,
  "seed": 42,
  "segundos": 4606.1
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/CatBoost/melhores.json
{
  "subset": "FD002",
  "model": "CatBoost",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 60,
    "hyper": {
      "iterations": 500,
      "depth": 4,
      "learning_rate": 0.11287044990293908,
      "l2_leaf_reg": 6.375221407140048
    }
  },
  "cv_rmse": 13.163687356788888,
  "cv_se": 0.18070428560744084,
  "melhor_rmse_medio": 13.163687356788888,
  "melhor_trial": 36,
  "trial_escolhida": 36,
  "n_tentativas_completas": 31,
  "n_tentativas_total": 40,
  "n_splits": 5,
  "seed": 42,
  "segundos": 14795.9
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/LightGBM/melhores.json
{
  "subset": "FD002",
  "model": "LightGBM",
  "params": {
    "representation": "global",
    "temporal": true,
    "window": 40,
    "hyper": {
      "n_estimators": 600,
      "max_depth": 9,
      "learning_rate": 0.03697775381245331,
      "num_leaves": 99,
      "subsample": 0.8372562950297838,
      "subsample_freq": 1,
      "colsample_bytree": 0.538829448665663,
      "min_child_samples": 38,
      "reg_lambda": 0.015764753603386868
    }
  },
  "cv_rmse": 16.305490879696855,
  "cv_se": 0.28104292046577267,
  "melhor_rmse_medio": 16.01003985123885,
  "melhor_trial": 30,
  "trial_escolhida": 33,
  "n_tentativas_completas": 49,
  "n_tentativas_total": 60,
  "n_splits": 5,
  "seed": 42,
  "segundos": 12702.6
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/MLP/melhores.json
{
  "subset": "FD002",
  "model": "MLP",
  "params": {
    "representation": "regime",
    "temporal": false,
    "window": 0,
    "hyper": {
      "hidden_layer_sizes": [
        64,
        32
      ],
      "alpha": 0.09441570932564147,
      "learning_rate_init": 0.00038147208642969516
    }
  },
  "cv_rmse": 16.733192759779808,
  "cv_se": 0.35225170542604944,
  "melhor_rmse_medio": 16.733192759779808,
  "melhor_trial": 14,
  "trial_escolhida": 14,
  "n_tentativas_completas": 24,
  "n_tentativas_total": 30,
  "n_splits": 5,
  "seed": 42,
  "segundos": 4807.7
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/PyTorch_CNN/melhores.json
{
  "subset": "FD002",
  "model": "PyTorch CNN",
  "params": {
    "representation": "regime",
    "window": 10,
    "temporal": false,
    "hyper": {
      "learning_rate": 0.0012646019561413444,
      "batch_size": 128,
      "dropout": 0.302962690294729,
      "num_filters": 64,
      "kernel_size": 5
    }
  },
  "cv_rmse": 16.446656721883045,
  "cv_se": 0.5036707985671016,
  "melhor_rmse_medio": 16.446656721883045,
  "melhor_trial": 16,
  "trial_escolhida": 16,
  "n_tentativas_completas": 20,
  "n_tentativas_total": 20,
  "n_splits": 3,
  "seed": 42,
  "segundos": 700.6
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/PyTorch_RNN_LSTM/melhores.json
{
  "subset": "FD002",
  "model": "PyTorch RNN (LSTM)",
  "params": {
    "representation": "regime",
    "window": 50,
    "temporal": false,
    "hyper": {
      "learning_rate": 0.002552951604697378,
      "batch_size": 256,
      "dropout": 0.08493564427131046,
      "hidden_dim": 128,
      "num_layers": 2
    }
  },
  "cv_rmse": 12.3957908042847,
  "cv_se": 0.3679133833505343,
  "melhor_rmse_medio": 12.20347124072149,
  "melhor_trial": 10,
  "trial_escolhida": 0,
  "n_tentativas_completas": 13,
  "n_tentativas_total": 20,
  "n_splits": 3,
  "seed": 42,
  "segundos": 1136.9
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/PyTorch_Transformer/melhores.json
{
  "subset": "FD002",
  "model": "PyTorch Transformer",
  "params": {
    "representation": "regime",
    "window": 60,
    "temporal": false,
    "hyper": {
      "learning_rate": 0.0021481839210591726,
      "batch_size": 512,
      "dropout": 0.35884410398103084,
      "d_model": 32,
      "nhead": 4,
      "num_layers": 3,
      "dim_feedforward": 64
    }
  },
  "cv_rmse": 12.119102810533754,
  "cv_se": 0.3647377003099629,
  "melhor_rmse_medio": 12.119102810533754,
  "melhor_trial": 8,
  "trial_escolhida": 8,
  "n_tentativas_completas": 15,
  "n_tentativas_total": 20,
  "n_splits": 3,
  "seed": 42,
  "segundos": 1061.5
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/Random_Forest/melhores.json
{
  "subset": "FD002",
  "model": "Random Forest",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 50,
    "hyper": {
      "n_estimators": 300,
      "max_depth": 14,
      "min_samples_leaf": 2,
      "max_features": 0.7992713510560965
    }
  },
  "cv_rmse": 15.078941845552956,
  "cv_se": 0.261279489143944,
  "melhor_rmse_medio": 15.078941845552956,
  "melhor_trial": 7,
  "trial_escolhida": 7,
  "n_tentativas_completas": 12,
  "n_tentativas_total": 15,
  "n_splits": 5,
  "seed": 42,
  "segundos": 0.0
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/Ridge/melhores.json
{
  "subset": "FD002",
  "model": "Ridge",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 60,
    "hyper": {
      "alpha": 10.85063770122587
    }
  },
  "cv_rmse": 15.384133041309443,
  "cv_se": 0.16747106568880873,
  "melhor_rmse_medio": 15.384133041309443,
  "melhor_trial": 19,
  "trial_escolhida": 19,
  "n_tentativas_completas": 23,
  "n_tentativas_total": 30,
  "n_splits": 5,
  "seed": 42,
  "segundos": 0.0
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/XGBoost/melhores.json
{
  "subset": "FD002",
  "model": "XGBoost",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 60,
    "hyper": {
      "n_estimators": 550,
      "max_depth": 5,
      "learning_rate": 0.047749049282444445,
      "subsample": 0.6573873408139774,
      "colsample_bytree": 0.556030675310799,
      "min_child_weight": 3,
      "reg_lambda": 9.65558114284457
    }
  },
  "cv_rmse": 13.039928084243394,
  "cv_se": 0.1907529171128372,
  "melhor_rmse_medio": 13.039928084243394,
  "melhor_trial": 52,
  "trial_escolhida": 52,
  "n_tentativas_completas": 45,
  "n_tentativas_total": 60,
  "n_splits": 5,
  "seed": 42,
  "segundos": 22976.9
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/XGBoost_NASA_Loss/melhores.json
{
  "subset": "FD002",
  "model": "XGBoost (NASA Loss)",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 60,
    "hyper": {
      "n_estimators": 600,
      "max_depth": 8,
      "learning_rate": 0.03798768845314897,
      "subsample": 0.5919520858241266,
      "colsample_bytree": 0.5564709544435013,
      "min_child_weight": 9,
      "reg_lambda": 1.396822441773842
    }
  },
  "cv_rmse": 12.989360478759574,
  "cv_se": 0.22137616241704877,
  "melhor_rmse_medio": 12.989360478759574,
  "melhor_trial": 52,
  "trial_escolhida": 52,
  "n_tentativas_completas": 46,
  "n_tentativas_total": 60,
  "n_splits": 5,
  "seed": 42,
  "segundos": 21509.5
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/CatBoost/melhores.json
{
  "subset": "FD003",
  "model": "CatBoost",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 50,
    "hyper": {
      "iterations": 800,
      "depth": 7,
      "learning_rate": 0.027649274676728182,
      "l2_leaf_reg": 8.471354625326555
    }
  },
  "cv_rmse": 10.349834115948497,
  "cv_se": 0.7055385390387926,
  "melhor_rmse_medio": 10.055861139441955,
  "melhor_trial": 25,
  "trial_escolhida": 7,
  "n_tentativas_completas": 32,
  "n_tentativas_total": 40,
  "n_splits": 5,
  "seed": 42,
  "segundos": 10937.6
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/LightGBM/melhores.json
{
  "subset": "FD003",
  "model": "LightGBM",
  "params": {
    "representation": "global",
    "temporal": true,
    "window": 50,
    "hyper": {
      "n_estimators": 350,
      "max_depth": 5,
      "learning_rate": 0.08724162191876421,
      "num_leaves": 88,
      "subsample": 0.9585669471809913,
      "subsample_freq": 1,
      "colsample_bytree": 0.513162093995454,
      "min_child_samples": 6,
      "reg_lambda": 0.39482766598875557
    }
  },
  "cv_rmse": 10.711630532659619,
  "cv_se": 0.6850409917925536,
  "melhor_rmse_medio": 10.20530499302175,
  "melhor_trial": 49,
  "trial_escolhida": 26,
  "n_tentativas_completas": 43,
  "n_tentativas_total": 60,
  "n_splits": 5,
  "seed": 42,
  "segundos": 2847.2
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/MLP/melhores.json
{
  "subset": "FD003",
  "model": "MLP",
  "params": {
    "representation": "global",
    "temporal": true,
    "window": 50,
    "hyper": {
      "hidden_layer_sizes": [
        128,
        64,
        32
      ],
      "alpha": 1.4783198058349258e-05,
      "learning_rate_init": 0.0027219983013791744
    }
  },
  "cv_rmse": 10.218301354183605,
  "cv_se": 0.5865869799547979,
  "melhor_rmse_medio": 9.837639991654722,
  "melhor_trial": 28,
  "trial_escolhida": 15,
  "n_tentativas_completas": 27,
  "n_tentativas_total": 30,
  "n_splits": 5,
  "seed": 42,
  "segundos": 4417.3
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/PyTorch_CNN/melhores.json
{
  "subset": "FD003",
  "model": "PyTorch CNN",
  "params": {
    "representation": "regime",
    "window": 10,
    "temporal": false,
    "hyper": {
      "learning_rate": 0.0008778987571925256,
      "batch_size": 256,
      "dropout": 0.34768971748476474,
      "num_filters": 32,
      "kernel_size": 5
    }
  },
  "cv_rmse": 15.152928185191556,
  "cv_se": 1.253717556377698,
  "melhor_rmse_medio": 15.152928185191556,
  "melhor_trial": 15,
  "trial_escolhida": 15,
  "n_tentativas_completas": 18,
  "n_tentativas_total": 20,
  "n_splits": 3,
  "seed": 42,
  "segundos": 300.0
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/PyTorch_RNN_LSTM/melhores.json
{
  "subset": "FD003",
  "model": "PyTorch RNN (LSTM)",
  "params": {
    "representation": "regime",
    "window": 60,
    "temporal": false,
    "hyper": {
      "learning_rate": 0.0034158985933588444,
      "batch_size": 128,
      "dropout": 0.16783224686740078,
      "hidden_dim": 128,
      "num_layers": 2
    }
  },
  "cv_rmse": 10.2333197689152,
  "cv_se": 0.6661953446761654,
  "melhor_rmse_medio": 10.2333197689152,
  "melhor_trial": 11,
  "trial_escolhida": 11,
  "n_tentativas_completas": 19,
  "n_tentativas_total": 20,
  "n_splits": 3,
  "seed": 42,
  "segundos": 586.4
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/PyTorch_Transformer/melhores.json
{
  "subset": "FD003",
  "model": "PyTorch Transformer",
  "params": {
    "representation": "global",
    "window": 40,
    "temporal": false,
    "hyper": {
      "learning_rate": 0.0008450942890954814,
      "batch_size": 512,
      "dropout": 0.0027808522124762817,
      "d_model": 32,
      "nhead": 4,
      "num_layers": 3,
      "dim_feedforward": 256
    }
  },
  "cv_rmse": 13.91419647400746,
  "cv_se": 1.8985766909776989,
  "melhor_rmse_medio": 13.91419647400746,
  "melhor_trial": 5,
  "trial_escolhida": 5,
  "n_tentativas_completas": 14,
  "n_tentativas_total": 20,
  "n_splits": 3,
  "seed": 42,
  "segundos": 344.0
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/Random_Forest/melhores.json
{
  "subset": "FD003",
  "model": "Random Forest",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 50,
    "hyper": {
      "n_estimators": 200,
      "max_depth": 17,
      "min_samples_leaf": 10,
      "max_features": 0.5193823482565607
    }
  },
  "cv_rmse": 11.011935717512856,
  "cv_se": 0.7609320751745875,
  "melhor_rmse_medio": 10.666978553571838,
  "melhor_trial": 30,
  "trial_escolhida": 23,
  "n_tentativas_completas": 36,
  "n_tentativas_total": 40,
  "n_splits": 5,
  "seed": 42,
  "segundos": 31626.3
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/Ridge/melhores.json
{
  "subset": "FD003",
  "model": "Ridge",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 60,
    "hyper": {
      "alpha": 16.63820832872317
    }
  },
  "cv_rmse": 14.390807249448432,
  "cv_se": 0.34228395266831163,
  "melhor_rmse_medio": 14.390807249448432,
  "melhor_trial": 24,
  "trial_escolhida": 24,
  "n_tentativas_completas": 23,
  "n_tentativas_total": 30,
  "n_splits": 5,
  "seed": 42,
  "segundos": 0.0
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/XGBoost/melhores.json
{
  "subset": "FD003",
  "model": "XGBoost",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 50,
    "hyper": {
      "n_estimators": 350,
      "max_depth": 8,
      "learning_rate": 0.03324142315048949,
      "subsample": 0.573371796425066,
      "colsample_bytree": 0.5076674195028477,
      "min_child_weight": 9,
      "reg_lambda": 3.303855397785506
    }
  },
  "cv_rmse": 10.563874236196057,
  "cv_se": 0.6681012784459822,
  "melhor_rmse_medio": 10.081911710451859,
  "melhor_trial": 53,
  "trial_escolhida": 35,
  "n_tentativas_completas": 43,
  "n_tentativas_total": 60,
  "n_splits": 5,
  "seed": 42,
  "segundos": 9286.5
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/XGBoost_NASA_Loss/melhores.json
{
  "subset": "FD003",
  "model": "XGBoost (NASA Loss)",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 40,
    "hyper": {
      "n_estimators": 550,
      "max_depth": 8,
      "learning_rate": 0.05595699958005902,
      "subsample": 0.5406327017276852,
      "colsample_bytree": 0.5543343713914236,
      "min_child_weight": 8,
      "reg_lambda": 0.5012571977568856
    }
  },
  "cv_rmse": 10.971979716222943,
  "cv_se": 0.8435118844719315,
  "melhor_rmse_medio": 10.17909595890781,
  "melhor_trial": 53,
  "trial_escolhida": 25,
  "n_tentativas_completas": 47,
  "n_tentativas_total": 60,
  "n_splits": 5,
  "seed": 42,
  "segundos": 8406.1
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/CatBoost/melhores.json
{
  "subset": "FD004",
  "model": "CatBoost",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 50,
    "hyper": {
      "iterations": 800,
      "depth": 5,
      "learning_rate": 0.08397983881234862,
      "l2_leaf_reg": 8.335462491675733
    }
  },
  "cv_rmse": 13.18839043377138,
  "cv_se": 0.6495799753393534,
  "melhor_rmse_medio": 12.908515349812282,
  "melhor_trial": 22,
  "trial_escolhida": 12,
  "n_tentativas_completas": 30,
  "n_tentativas_total": 40,
  "n_splits": 5,
  "seed": 42,
  "segundos": 14409.4
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/LightGBM/melhores.json
{
  "subset": "FD004",
  "model": "LightGBM",
  "params": {
    "representation": "regime",
    "temporal": false,
    "window": 0,
    "hyper": {
      "n_estimators": 300,
      "max_depth": 10,
      "learning_rate": 0.024449265853509095,
      "num_leaves": 119,
      "subsample": 0.9023281061722079,
      "subsample_freq": 1,
      "colsample_bytree": 0.6146100590684846,
      "min_child_samples": 53,
      "reg_lambda": 0.023766534262631735
    }
  },
  "cv_rmse": 15.979257107521118,
  "cv_se": 0.6377848430864496,
  "melhor_rmse_medio": 15.979257107521118,
  "melhor_trial": 57,
  "trial_escolhida": 57,
  "n_tentativas_completas": 53,
  "n_tentativas_total": 60,
  "n_splits": 5,
  "seed": 42,
  "segundos": 1455.0
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/MLP/melhores.json
{
  "subset": "FD004",
  "model": "MLP",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 40,
    "hyper": {
      "hidden_layer_sizes": [
        64,
        32
      ],
      "alpha": 0.008324807286421161,
      "learning_rate_init": 0.00015030942695843836
    }
  },
  "cv_rmse": 14.756609021186822,
  "cv_se": 0.5623064394662096,
  "melhor_rmse_medio": 14.756609021186822,
  "melhor_trial": 27,
  "trial_escolhida": 27,
  "n_tentativas_completas": 24,
  "n_tentativas_total": 30,
  "n_splits": 5,
  "seed": 42,
  "segundos": 4017.0
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/PyTorch_CNN/melhores.json
{
  "subset": "FD004",
  "model": "PyTorch CNN",
  "params": {
    "representation": "regime",
    "window": 10,
    "temporal": false,
    "hyper": {
      "learning_rate": 0.0011262698856876726,
      "batch_size": 256,
      "dropout": 0.29174697772868086,
      "num_filters": 32,
      "kernel_size": 5
    }
  },
  "cv_rmse": 16.067694775786208,
  "cv_se": 0.49081911579929166,
  "melhor_rmse_medio": 16.067694775786208,
  "melhor_trial": 12,
  "trial_escolhida": 12,
  "n_tentativas_completas": 18,
  "n_tentativas_total": 20,
  "n_splits": 3,
  "seed": 42,
  "segundos": 643.9
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/PyTorch_RNN_LSTM/melhores.json
{
  "subset": "FD004",
  "model": "PyTorch RNN (LSTM)",
  "params": {
    "representation": "regime",
    "window": 60,
    "temporal": false,
    "hyper": {
      "learning_rate": 0.002100236158351101,
      "batch_size": 128,
      "dropout": 0.13007332881069883,
      "hidden_dim": 128,
      "num_layers": 2
    }
  },
  "cv_rmse": 12.373466281033977,
  "cv_se": 0.6892682971139921,
  "melhor_rmse_medio": 12.373466281033977,
  "melhor_trial": 4,
  "trial_escolhida": 4,
  "n_tentativas_completas": 14,
  "n_tentativas_total": 20,
  "n_splits": 3,
  "seed": 42,
  "segundos": 0.0
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/PyTorch_Transformer/melhores.json
{
  "subset": "FD004",
  "model": "PyTorch Transformer",
  "params": {
    "representation": "regime",
    "window": 60,
    "temporal": false,
    "hyper": {
      "learning_rate": 0.0021481839210591726,
      "batch_size": 512,
      "dropout": 0.35884410398103084,
      "d_model": 32,
      "nhead": 4,
      "num_layers": 3,
      "dim_feedforward": 64
    }
  },
  "cv_rmse": 14.049794343348589,
  "cv_se": 0.8537602372805568,
  "melhor_rmse_medio": 14.049794343348589,
  "melhor_trial": 8,
  "trial_escolhida": 8,
  "n_tentativas_completas": 17,
  "n_tentativas_total": 20,
  "n_splits": 3,
  "seed": 42,
  "segundos": 1132.7
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/Random_Forest/melhores.json
{
  "subset": "FD004",
  "model": "Random Forest",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 50,
    "hyper": {
      "n_estimators": 300,
      "max_depth": 14,
      "min_samples_leaf": 2,
      "max_features": 0.7992713510560965
    }
  },
  "cv_rmse": 14.152711411537258,
  "cv_se": 0.5681527624416998,
  "melhor_rmse_medio": 14.152711411537258,
  "melhor_trial": 7,
  "trial_escolhida": 7,
  "n_tentativas_completas": 13,
  "n_tentativas_total": 15,
  "n_splits": 5,
  "seed": 42,
  "segundos": 0.0
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/Ridge/melhores.json
{
  "subset": "FD004",
  "model": "Ridge",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 60,
    "hyper": {
      "alpha": 10.85063770122587
    }
  },
  "cv_rmse": 16.70789458527721,
  "cv_se": 0.23370656630647144,
  "melhor_rmse_medio": 16.70789458527721,
  "melhor_trial": 19,
  "trial_escolhida": 19,
  "n_tentativas_completas": 22,
  "n_tentativas_total": 30,
  "n_splits": 5,
  "seed": 42,
  "segundos": 0.0
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/XGBoost/melhores.json
{
  "subset": "FD004",
  "model": "XGBoost",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 50,
    "hyper": {
      "n_estimators": 550,
      "max_depth": 6,
      "learning_rate": 0.021377844571975117,
      "subsample": 0.5555576766627507,
      "colsample_bytree": 0.5176803628728603,
      "min_child_weight": 3,
      "reg_lambda": 3.1216635272378905
    }
  },
  "cv_rmse": 13.085661877502414,
  "cv_se": 0.6330602950150462,
  "melhor_rmse_medio": 12.705564387795047,
  "melhor_trial": 54,
  "trial_escolhida": 23,
  "n_tentativas_completas": 45,
  "n_tentativas_total": 60,
  "n_splits": 5,
  "seed": 42,
  "segundos": 14890.3
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/XGBoost_NASA_Loss/melhores.json
{
  "subset": "FD004",
  "model": "XGBoost (NASA Loss)",
  "params": {
    "representation": "regime",
    "temporal": true,
    "window": 50,
    "hyper": {
      "n_estimators": 500,
      "max_depth": 10,
      "learning_rate": 0.04731851459137855,
      "subsample": 0.5550259622638384,
      "colsample_bytree": 0.6139675812709708,
      "min_child_weight": 5,
      "reg_lambda": 2.8447512555118193
    }
  },
  "cv_rmse": 13.095599222539168,
  "cv_se": 0.6648216472468684,
  "melhor_rmse_medio": 12.645182514894135,
  "melhor_trial": 52,
  "trial_escolhida": 8,
  "n_tentativas_completas": 47,
  "n_tentativas_total": 60,
  "n_splits": 5,
  "seed": 42,
  "segundos": 16960.7
}

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/CatBoost/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,16.642469584199496,0.35874528154659396,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""iterations"": 200, ""depth"": 5, ""learning_rate"": 0.023406777359270506, ""l2_leaf_reg"": 13.39433470675048}}"
1,16.141460881127934,0.3965216350034992,10,"{""representation"": ""regime"", ""temporal"": true, ""window"": 10, ""hyper"": {""iterations"": 300, ""depth"": 8, ""learning_rate"": 0.029180149047462245, ""l2_leaf_reg"": 2.399324290681271}}"
2,16.38733707622681,0.3924851331739141,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""iterations"": 500, ""depth"": 8, ""learning_rate"": 0.022680881003614695, ""l2_leaf_reg"": 6.172115948107071}}"
3,16.443788274922397,0.3466526162777059,10,"{""representation"": ""global"", ""temporal"": true, ""window"": 10, ""hyper"": {""iterations"": 100, ""depth"": 10, ""learning_rate"": 0.04030685101059539, ""l2_leaf_reg"": 7.277150634170934}}"
4,17.537292281338665,0.3424288424791628,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""iterations"": 800, ""depth"": 9, ""learning_rate"": 0.2546633740246392, ""l2_leaf_reg"": 14.594768942966839}}"
5,13.218095685574108,0.38309124895292235,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""iterations"": 500, ""depth"": 4, ""learning_rate"": 0.17558477392891916, ""l2_leaf_reg"": 1.2502377950801113}}"
6,17.058588635371443,0.3887457147604169,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""iterations"": 700, ""depth"": 8, ""learning_rate"": 0.14401540623155276, ""l2_leaf_reg"": 10.079659367870727}}"
7,12.226770335244606,0.2819449106514656,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""iterations"": 800, ""depth"": 7, ""learning_rate"": 0.027649274676728182, ""l2_leaf_reg"": 8.471354625326555}}"
10,12.430305497412728,0.33894507165960924,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""iterations"": 800, ""depth"": 8, ""learning_rate"": 0.07287261571692631, ""l2_leaf_reg"": 18.445338876089934}}"
11,12.433760903809922,0.28936461306662303,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""iterations"": 700, ""depth"": 8, ""learning_rate"": 0.05121944772162538, ""l2_leaf_reg"": 17.183790926280032}}"
12,12.51598621511268,0.33775564423817356,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""iterations"": 800, ""depth"": 5, ""learning_rate"": 0.13273723830085388, ""l2_leaf_reg"": 8.253568352966084}}"
14,11.625916011550016,0.2721244131527455,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 700, ""depth"": 6, ""learning_rate"": 0.03700295111196078, ""l2_leaf_reg"": 8.163242449101164}}"
15,11.619023195720896,0.17887999988040462,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 800, ""depth"": 5, ""learning_rate"": 0.02081518810092141, ""l2_leaf_reg"": 19.82027682469185}}"
16,11.506365209868187,0.16947411551912708,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 800, ""depth"": 4, ""learning_rate"": 0.05483455610369595, ""l2_leaf_reg"": 9.570956018768351}}"
17,11.720823890937849,0.1706984097751401,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 700, ""depth"": 4, ""learning_rate"": 0.02948967675512807, ""l2_leaf_reg"": 16.44433966802777}}"
18,11.560758625846173,0.17460918402543746,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 800, ""depth"": 4, ""learning_rate"": 0.06816504198437771, ""l2_leaf_reg"": 13.687400037279852}}"
20,11.602740515585518,0.20840045555856376,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 800, ""depth"": 4, ""learning_rate"": 0.07021102254271089, ""l2_leaf_reg"": 19.18578924472035}}"
21,11.615554006317618,0.16730904335769872,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 700, ""depth"": 4, ""learning_rate"": 0.12150835307941323, ""l2_leaf_reg"": 15.54911337363718}}"
22,11.567079749638804,0.17323497927810946,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 4, ""learning_rate"": 0.048272677510161024, ""l2_leaf_reg"": 3.743535020173426}}"
24,11.546227577097152,0.18887032921003816,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 800, ""depth"": 4, ""learning_rate"": 0.07613921404358207, ""l2_leaf_reg"": 3.864298729266147}}"
25,11.67318535818471,0.22011328186451998,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 800, ""depth"": 4, ""learning_rate"": 0.07975285998303272, ""l2_leaf_reg"": 5.881534767971}}"
29,11.516181826436034,0.20671071661199247,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 500, ""depth"": 5, ""learning_rate"": 0.08543601855078736, ""l2_leaf_reg"": 13.640542011095437}}"
30,11.549170834578174,0.181722706169938,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 500, ""depth"": 4, ""learning_rate"": 0.08187766516719576, ""l2_leaf_reg"": 3.377895720266763}}"
31,11.715694739848317,0.17023999937013135,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 4, ""learning_rate"": 0.11178045301578328, ""l2_leaf_reg"": 3.8109284265293235}}"
33,11.587163587722099,0.20285523854685805,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 5, ""learning_rate"": 0.06581718896688954, ""l2_leaf_reg"": 11.114174607474531}}"
35,11.585390979979433,0.1769365642827595,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 800, ""depth"": 4, ""learning_rate"": 0.031700666612255304, ""l2_leaf_reg"": 5.490251386116665}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/LightGBM/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,17.340132352363742,0.27789844271784186,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 4, ""learning_rate"": 0.023406777359270506, ""num_leaves"": 112, ""subsample"": 0.8005575058716043, ""subsample_freq"": 1, ""colsample_bytree"": 0.8540362888980227, ""min_child_samples"": 6, ""reg_lambda"": 8.123245085588687}}"
1,13.563448335038023,0.09982357302040455,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 250, ""max_depth"": 7, ""learning_rate"": 0.16767510257004176, ""num_leaves"": 37, ""subsample"": 0.7571172192068059, ""subsample_freq"": 1, ""colsample_bytree"": 0.7962072844310213, ""min_child_samples"": 9, ""reg_lambda"": 0.6647135865318028}}"
2,17.3187606672605,0.2487683384680935,10,"{""representation"": ""global"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 50, ""max_depth"": 12, ""learning_rate"": 0.04030685101059539, ""num_leaves"": 89, ""subsample"": 0.6558555380447055, ""subsample_freq"": 1, ""colsample_bytree"": 0.7600340105889054, ""min_child_samples"": 57, ""reg_lambda"": 0.035856126103453984}}"
3,16.557431651502235,0.3661994002858843,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 400, ""max_depth"": 12, ""learning_rate"": 0.025415885998427375, ""num_leaves"": 37, ""subsample"": 0.522613644455269, ""subsample_freq"": 1, ""colsample_bytree"": 0.6626651653816322, ""min_child_samples"": 42, ""reg_lambda"": 0.06516990611177176}}"
4,14.729278484621318,0.3025870658927804,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.14401540623155276, ""num_leaves"": 102, ""subsample"": 0.5370223258670452, ""subsample_freq"": 1, ""colsample_bytree"": 0.6792328642721364, ""min_child_samples"": 16, ""reg_lambda"": 3.8842777547031417}}"
5,14.798466561975479,0.2114326232085243,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""n_estimators"": 500, ""max_depth"": 8, ""learning_rate"": 0.16134583428320007, ""num_leaves"": 70, ""subsample"": 0.7613664146909971, ""subsample_freq"": 1, ""colsample_bytree"": 0.7137705091792748, ""min_child_samples"": 7, ""reg_lambda"": 0.021070472806578238}}"
6,16.885555832285245,0.3369274994692188,10,"{""representation"": ""regime"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 100, ""max_depth"": 12, ""learning_rate"": 0.17842401600508806, ""num_leaves"": 86, ""subsample"": 0.9357302950938589, ""subsample_freq"": 1, ""colsample_bytree"": 0.9018360384495572, ""min_child_samples"": 22, ""reg_lambda"": 4.760767751809501}}"
7,16.394839385795493,0.3692174271564263,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 5, ""learning_rate"": 0.06358398347237385, ""num_leaves"": 107, ""subsample"": 0.9303652916281717, ""subsample_freq"": 1, ""colsample_bytree"": 0.5034760652655954, ""min_child_samples"": 54, ""reg_lambda"": 0.1787446325623842}}"
8,12.882106180992537,0.25211786775521394,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 300, ""max_depth"": 6, ""learning_rate"": 0.043254212679768214, ""num_leaves"": 19, ""subsample"": 0.8047821669899484, ""subsample_freq"": 1, ""colsample_bytree"": 0.7513395116144308, ""min_child_samples"": 9, ""reg_lambda"": 0.06853925708853059}}"
9,16.611567662893965,0.452825865375424,10,"{""representation"": ""global"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 400, ""max_depth"": 9, ""learning_rate"": 0.08533946549577034, ""num_leaves"": 25, ""subsample"": 0.917651247794619, ""subsample_freq"": 1, ""colsample_bytree"": 0.6603900324858679, ""min_child_samples"": 22, ""reg_lambda"": 0.013253342600066534}}"
10,13.218985101591855,0.2435347226944508,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 200, ""max_depth"": 5, ""learning_rate"": 0.03193723880477954, ""num_leaves"": 36, ""subsample"": 0.994615670525111, ""subsample_freq"": 1, ""colsample_bytree"": 0.7768218093259767, ""min_child_samples"": 80, ""reg_lambda"": 0.011734369616928872}}"
11,13.348495108647104,0.2323860563772982,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 250, ""max_depth"": 5, ""learning_rate"": 0.021876520708903467, ""num_leaves"": 50, ""subsample"": 0.9768639858289091, ""subsample_freq"": 1, ""colsample_bytree"": 0.6243740711709365, ""min_child_samples"": 11, ""reg_lambda"": 0.05030322191799626}}"
12,15.012814680111308,0.2289912703848529,20,"{""representation"": ""global"", ""temporal"": true, ""window"": 20, ""hyper"": {""n_estimators"": 200, ""max_depth"": 6, ""learning_rate"": 0.050080195015291123, ""num_leaves"": 59, ""subsample"": 0.9303046499575794, ""subsample_freq"": 1, ""colsample_bytree"": 0.6945364874393392, ""min_child_samples"": 75, ""reg_lambda"": 0.01648851528525295}}"
13,11.760249987041547,0.29956885075639667,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 350, ""max_depth"": 5, ""learning_rate"": 0.03674300857025957, ""num_leaves"": 34, ""subsample"": 0.853031415623392, ""subsample_freq"": 1, ""colsample_bytree"": 0.9087248103886313, ""min_child_samples"": 91, ""reg_lambda"": 0.04928605065294771}}"
14,11.665925049602668,0.26895344564607315,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 350, ""max_depth"": 5, ""learning_rate"": 0.03096022664905448, ""num_leaves"": 45, ""subsample"": 0.7664850663325223, ""subsample_freq"": 1, ""colsample_bytree"": 0.8158455057911022, ""min_child_samples"": 83, ""reg_lambda"": 0.019855339659370638}}"
15,11.611358498463309,0.23190805712264004,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 5, ""learning_rate"": 0.03640328352960014, ""num_leaves"": 58, ""subsample"": 0.6726459422763669, ""subsample_freq"": 1, ""colsample_bytree"": 0.7474367232322943, ""min_child_samples"": 100, ""reg_lambda"": 0.020039680598813347}}"
16,11.711408682386917,0.3054246975626441,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 7, ""learning_rate"": 0.07118345340353874, ""num_leaves"": 57, ""subsample"": 0.7355998976130134, ""subsample_freq"": 1, ""colsample_bytree"": 0.724097411821317, ""min_child_samples"": 90, ""reg_lambda"": 0.047980245917119885}}"
17,12.101608663482418,0.2577574066737938,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 400, ""max_depth"": 3, ""learning_rate"": 0.03152105747579249, ""num_leaves"": 67, ""subsample"": 0.653398002248582, ""subsample_freq"": 1, ""colsample_bytree"": 0.8110218432114722, ""min_child_samples"": 81, ""reg_lambda"": 0.015164307025538308}}"
19,11.461844263432447,0.27730276697139083,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 5, ""learning_rate"": 0.035728140021911485, ""num_leaves"": 45, ""subsample"": 0.6115212921492891, ""subsample_freq"": 1, ""colsample_bytree"": 0.6156767751595634, ""min_child_samples"": 94, ""reg_lambda"": 0.10259959821554286}}"
20,12.09683463669329,0.3005835526808196,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 250, ""max_depth"": 5, ""learning_rate"": 0.03633307943641578, ""num_leaves"": 58, ""subsample"": 0.5303306677859185, ""subsample_freq"": 1, ""colsample_bytree"": 0.5539066640123318, ""min_child_samples"": 95, ""reg_lambda"": 0.033648724257396484}}"
21,12.012194703118306,0.2739397463601367,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 4, ""learning_rate"": 0.029628992347310316, ""num_leaves"": 65, ""subsample"": 0.6730200637782378, ""subsample_freq"": 1, ""colsample_bytree"": 0.907282714341271, ""min_child_samples"": 78, ""reg_lambda"": 0.1013425977250583}}"
22,11.595662277870321,0.282284139087283,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 6, ""learning_rate"": 0.029780394023850028, ""num_leaves"": 24, ""subsample"": 0.655812947338423, ""subsample_freq"": 1, ""colsample_bytree"": 0.8070478731183522, ""min_child_samples"": 79, ""reg_lambda"": 0.018882092650482917}}"
23,11.548031757886285,0.25213969415046583,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 4, ""learning_rate"": 0.042603776809710316, ""num_leaves"": 56, ""subsample"": 0.6477192717457654, ""subsample_freq"": 1, ""colsample_bytree"": 0.5062454063426376, ""min_child_samples"": 86, ""reg_lambda"": 0.07938135760932635}}"
24,12.029220732747408,0.2754759492943122,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 3, ""learning_rate"": 0.04284244998114433, ""num_leaves"": 55, ""subsample"": 0.6283107241332974, ""subsample_freq"": 1, ""colsample_bytree"": 0.6943394265117249, ""min_child_samples"": 72, ""reg_lambda"": 0.0720451308552053}}"
26,12.676808777924093,0.3292749406275023,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 150, ""max_depth"": 4, ""learning_rate"": 0.0416621974263721, ""num_leaves"": 69, ""subsample"": 0.625549654398518, ""subsample_freq"": 1, ""colsample_bytree"": 0.6266671666050533, ""min_child_samples"": 91, ""reg_lambda"": 0.13140329262964115}}"
27,11.751512116983221,0.3014434670678931,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 7, ""learning_rate"": 0.027495358840021404, ""num_leaves"": 27, ""subsample"": 0.7163127164241199, ""subsample_freq"": 1, ""colsample_bytree"": 0.9802426369966042, ""min_child_samples"": 97, ""reg_lambda"": 0.014054073373485182}}"
29,11.71084660829674,0.27731376483799686,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 350, ""max_depth"": 3, ""learning_rate"": 0.05885387054399978, ""num_leaves"": 55, ""subsample"": 0.5620984423847599, ""subsample_freq"": 1, ""colsample_bytree"": 0.5622967634382638, ""min_child_samples"": 74, ""reg_lambda"": 0.0389693600624053}}"
30,11.297490231468299,0.22665631534558608,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 5, ""learning_rate"": 0.04062407513338108, ""num_leaves"": 44, ""subsample"": 0.6388007000471191, ""subsample_freq"": 1, ""colsample_bytree"": 0.5984856118921611, ""min_child_samples"": 90, ""reg_lambda"": 0.3041691985549422}}"
31,11.85854646947946,0.31161370826591506,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 4, ""learning_rate"": 0.034896700411600964, ""num_leaves"": 54, ""subsample"": 0.6445386482935076, ""subsample_freq"": 1, ""colsample_bytree"": 0.5420693408485919, ""min_child_samples"": 84, ""reg_lambda"": 0.015237174624891565}}"
32,11.370207548826603,0.24649785460204068,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 5, ""learning_rate"": 0.044142177633330906, ""num_leaves"": 39, ""subsample"": 0.6842583123738346, ""subsample_freq"": 1, ""colsample_bytree"": 0.6722441895078367, ""min_child_samples"": 99, ""reg_lambda"": 0.1996281520634009}}"
33,11.48231136058467,0.2958279460474542,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 350, ""max_depth"": 4, ""learning_rate"": 0.05010084513742603, ""num_leaves"": 32, ""subsample"": 0.6124269049014588, ""subsample_freq"": 1, ""colsample_bytree"": 0.5565492319923581, ""min_child_samples"": 95, ""reg_lambda"": 0.06402562823428107}}"
34,11.438152537157588,0.24040258659671848,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 400, ""max_depth"": 5, ""learning_rate"": 0.05459908200417522, ""num_leaves"": 38, ""subsample"": 0.678113165372874, ""subsample_freq"": 1, ""colsample_bytree"": 0.6909689590716451, ""min_child_samples"": 84, ""reg_lambda"": 0.23473227501194177}}"
41,11.711080265855736,0.24120058883204465,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 5, ""learning_rate"": 0.058806666745816194, ""num_leaves"": 38, ""subsample"": 0.6760411040317477, ""subsample_freq"": 1, ""colsample_bytree"": 0.7308687473367277, ""min_child_samples"": 87, ""reg_lambda"": 0.3520965444898976}}"
42,11.390152533488893,0.2790344106437572,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 5, ""learning_rate"": 0.04117433219447217, ""num_leaves"": 54, ""subsample"": 0.697865646592798, ""subsample_freq"": 1, ""colsample_bytree"": 0.5755557114420298, ""min_child_samples"": 77, ""reg_lambda"": 0.19255974678044652}}"
43,11.36637060689885,0.24612736580692274,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 5, ""learning_rate"": 0.04822760714486972, ""num_leaves"": 37, ""subsample"": 0.7184821381624301, ""subsample_freq"": 1, ""colsample_bytree"": 0.5831673574592181, ""min_child_samples"": 84, ""reg_lambda"": 0.2151782392973763}}"
44,11.306498177058605,0.23063545523184753,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 5, ""learning_rate"": 0.05406996339902101, ""num_leaves"": 40, ""subsample"": 0.6891766952794444, ""subsample_freq"": 1, ""colsample_bytree"": 0.6081233859359763, ""min_child_samples"": 89, ""reg_lambda"": 0.06564463741783066}}"
47,11.501825820337478,0.2440850920116656,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 150, ""max_depth"": 6, ""learning_rate"": 0.038400059098424935, ""num_leaves"": 32, ""subsample"": 0.7059699093569626, ""subsample_freq"": 1, ""colsample_bytree"": 0.6024727958488068, ""min_child_samples"": 91, ""reg_lambda"": 0.18181856583271402}}"
48,11.313763965674685,0.25974611945369613,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 5, ""learning_rate"": 0.06089294801631826, ""num_leaves"": 42, ""subsample"": 0.7133011248348506, ""subsample_freq"": 1, ""colsample_bytree"": 0.5922303450776443, ""min_child_samples"": 100, ""reg_lambda"": 0.41717063044133923}}"
51,11.350854268851993,0.22204756026609102,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 5, ""learning_rate"": 0.058503289118635254, ""num_leaves"": 30, ""subsample"": 0.7074377730636956, ""subsample_freq"": 1, ""colsample_bytree"": 0.5419406481634167, ""min_child_samples"": 88, ""reg_lambda"": 0.31496050042294965}}"
52,11.41828813403221,0.23079581576243607,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 5, ""learning_rate"": 0.05065142511130537, ""num_leaves"": 33, ""subsample"": 0.7247679348803027, ""subsample_freq"": 1, ""colsample_bytree"": 0.5610476661359849, ""min_child_samples"": 89, ""reg_lambda"": 0.2513980048369073}}"
54,11.301059326632338,0.28334263377100993,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 350, ""max_depth"": 6, ""learning_rate"": 0.04881964347362612, ""num_leaves"": 48, ""subsample"": 0.6992635247391151, ""subsample_freq"": 1, ""colsample_bytree"": 0.5886939419338901, ""min_child_samples"": 88, ""reg_lambda"": 0.25473547325703144}}"
55,11.526213907198006,0.25488203166788337,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 4, ""learning_rate"": 0.056368544803547936, ""num_leaves"": 40, ""subsample"": 0.7690256101896079, ""subsample_freq"": 1, ""colsample_bytree"": 0.6216216309446131, ""min_child_samples"": 92, ""reg_lambda"": 0.1841581452649639}}"
56,11.36713342305298,0.23997229266860232,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 5, ""learning_rate"": 0.042583891681763575, ""num_leaves"": 27, ""subsample"": 0.7336516372864329, ""subsample_freq"": 1, ""colsample_bytree"": 0.5838780737744179, ""min_child_samples"": 78, ""reg_lambda"": 0.18477306759100282}}"
58,11.302936037587363,0.25255253279757117,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 6, ""learning_rate"": 0.06603534401485288, ""num_leaves"": 33, ""subsample"": 0.7008547930105454, ""subsample_freq"": 1, ""colsample_bytree"": 0.5772091487831641, ""min_child_samples"": 94, ""reg_lambda"": 0.34015442358141745}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/MLP/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,16.715319868960478,0.43005966456983713,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.002537815508265664, ""learning_rate_init"": 0.0026070247583707684}}"
1,16.448747685898525,0.4137918497401074,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.0005342937261279777, ""learning_rate_init"": 0.0003823475224675188}}"
2,17.973853966231555,0.44951952838168435,15,"{""representation"": ""global"", ""temporal"": true, ""window"": 15, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.01712337597316398, ""learning_rate_init"": 0.0004066563313514797}}"
3,16.539057012114192,0.4881256661569081,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.004467752817973905, ""learning_rate_init"": 0.0004201672054372534}}"
4,17.54632536473156,0.35321181858966255,15,"{""representation"": ""regime"", ""temporal"": true, ""window"": 15, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.02065142557895925, ""learning_rate_init"": 0.0005170191786366995}}"
5,17.530054777881666,0.3595995061609955,15,"{""representation"": ""regime"", ""temporal"": true, ""window"": 15, ""hyper"": {""hidden_layer_sizes"": [128, 64], ""alpha"": 2.9072088906598435e-05, ""learning_rate_init"": 0.005323617594751502}}"
6,15.149582463450633,0.317287838978684,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.0012329098365270509, ""learning_rate_init"": 0.0007162786999897341}}"
7,15.459824414157083,0.3462223070340679,20,"{""representation"": ""regime"", ""temporal"": true, ""window"": 20, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.01707975034295823, ""learning_rate_init"": 0.0018484491720988621}}"
9,16.422847781267894,0.4790117178825838,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.07076022085822008, ""learning_rate_init"": 0.00031883397351001905}}"
10,14.184772329957884,0.33354040544664465,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.00023815281563458162, ""learning_rate_init"": 0.0035932255004427057}}"
11,13.778315089238887,0.3109146317963315,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.000636180392728198, ""learning_rate_init"": 0.003007384759347854}}"
13,14.081154147557871,0.34175083643529625,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 8.629713172686319e-05, ""learning_rate_init"": 0.003520007788628053}}"
15,11.43340878181706,0.19654559044309658,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 1.4783198058349258e-05, ""learning_rate_init"": 0.0027219983013791744}}"
16,11.711135725208543,0.33213175660653843,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.013066930881586152, ""learning_rate_init"": 0.004420695104907991}}"
18,11.742480553559512,0.20996879893528228,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.005065674572226583, ""learning_rate_init"": 0.007030487967930749}}"
19,11.621181585120649,0.15886466909337207,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 1.1494131334371662e-05, ""learning_rate_init"": 0.002040645402886001}}"
21,11.277861591220692,0.24290431816342417,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 7.422852217088057e-05, ""learning_rate_init"": 0.0018059041910462289}}"
22,11.295341917417593,0.1611070154080419,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 7.743256516480622e-05, ""learning_rate_init"": 0.0012694337530292574}}"
23,11.306572569510674,0.21876946204944436,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.00016955434854726265, ""learning_rate_init"": 0.002126897562945387}}"
24,12.348279016492247,0.21918970592326462,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.00019457485657697212, ""learning_rate_init"": 0.0017857064299775566}}"
25,11.618020107851315,0.1570599994985041,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 2.9363080399139315e-05, ""learning_rate_init"": 0.0012484198701095614}}"
26,10.912039614875557,0.2373611597104747,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.0033674896293976685, ""learning_rate_init"": 0.000967342370900682}}"
27,11.1635551716368,0.24490748547440722,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.0043176443718924915, ""learning_rate_init"": 0.0008049247659962629}}"
28,13.038298765218247,0.173112697522695,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.0016377024956440642, ""learning_rate_init"": 0.0005624750946084906}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/PyTorch_CNN/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,19.505061037422525,0.6732844897168573,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002552951604697378, ""batch_size"": 256, ""dropout"": 0.08493564427131046, ""num_filters"": 128, ""kernel_size"": 3}}"
1,20.64985160343801,0.7247124058061241,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001956070814274847, ""batch_size"": 256, ""dropout"": 0.02602063719411181, ""num_filters"": 64, ""kernel_size"": 3}}"
2,18.660073568173313,0.9681653148207096,30,"{""representation"": ""global"", ""window"": 30, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0016559149440361915, ""batch_size"": 512, ""dropout"": 0.31005312934444584, ""num_filters"": 32, ""kernel_size"": 3}}"
3,18.447669335720438,1.4176646132446817,30,"{""representation"": ""global"", ""window"": 30, ""temporal"": false, ""hyper"": {""learning_rate"": 0.000691662498760998, ""batch_size"": 512, ""dropout"": 0.308897907718663, ""num_filters"": 128, ""kernel_size"": 5}}"
4,17.773811012507853,0.8040955422972846,20,"{""representation"": ""global"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0010571906813317184, ""batch_size"": 512, ""dropout"": 0.18888597006477972, ""num_filters"": 128, ""kernel_size"": 5}}"
5,19.264848109711593,0.8136296641007228,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.004041443189081019, ""batch_size"": 512, ""dropout"": 0.09151926619664899, ""num_filters"": 64, ""kernel_size"": 3}}"
6,19.718836997814716,0.9272980104939769,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0006442017924231741, ""batch_size"": 512, ""dropout"": 0.3442922333025374, ""num_filters"": 64, ""kernel_size"": 3}}"
7,19.249433875212684,1.7361081948070811,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0015711531129544835, ""batch_size"": 128, ""dropout"": 0.24382573359195875, ""num_filters"": 32, ""kernel_size"": 3}}"
8,16.450734072001097,1.2139783791214152,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002144252054813578, ""batch_size"": 128, ""dropout"": 0.3341209982356952, ""num_filters"": 32, ""kernel_size"": 5}}"
9,20.152322493507523,0.9968771683588405,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001096569987677643, ""batch_size"": 256, ""dropout"": 0.10317665108606225, ""num_filters"": 64, ""kernel_size"": 3}}"
10,19.806505517475845,0.8295661538621077,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0007605707018582466, ""batch_size"": 128, ""dropout"": 0.09894249765332985, ""num_filters"": 32, ""kernel_size"": 5}}"
11,18.076796653976377,1.179352555628082,20,"{""representation"": ""global"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0022377922068781047, ""batch_size"": 512, ""dropout"": 0.24234541383816843, ""num_filters"": 128, ""kernel_size"": 5}}"
12,16.69890493808985,1.1929450850543915,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.000991400004320801, ""batch_size"": 256, ""dropout"": 0.26209596575499594, ""num_filters"": 32, ""kernel_size"": 5}}"
13,17.125927868789628,0.7453250034909052,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0042644249940258045, ""batch_size"": 128, ""dropout"": 0.3575312644632453, ""num_filters"": 32, ""kernel_size"": 5}}"
14,16.38529160007262,1.217152560075133,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.000520889112475312, ""batch_size"": 256, ""dropout"": 0.28060091877975857, ""num_filters"": 32, ""kernel_size"": 5}}"
15,16.80152394589278,1.1737105852326104,20,"{""representation"": ""regime"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001248761000929402, ""batch_size"": 128, ""dropout"": 0.37366638431931315, ""num_filters"": 32, ""kernel_size"": 5}}"
16,16.489982346754537,1.2087075030552559,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0005620354063829667, ""batch_size"": 128, ""dropout"": 0.25555445377084673, ""num_filters"": 64, ""kernel_size"": 5}}"
17,17.08203410197386,0.8783615686511893,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0018663671123240314, ""batch_size"": 128, ""dropout"": 0.2809223270458201, ""num_filters"": 64, ""kernel_size"": 5}}"
19,17.966275509652757,0.8992257467914415,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002938752642375606, ""batch_size"": 128, ""dropout"": 0.2575626247886257, ""num_filters"": 32, ""kernel_size"": 5}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/PyTorch_RNN_LSTM/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,11.235369964041444,0.5110890308642027,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002552951604697378, ""batch_size"": 256, ""dropout"": 0.08493564427131046, ""hidden_dim"": 128, ""num_layers"": 2}}"
1,11.467155859862144,0.49927773346766907,50,"{""representation"": ""global"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0016338208828908817, ""batch_size"": 512, ""dropout"": 0.06820964947491662, ""hidden_dim"": 128, ""num_layers"": 3}}"
2,14.157434386677478,1.8276124593846024,50,"{""representation"": ""global"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002298752892366083, ""batch_size"": 512, ""dropout"": 0.07394178221021082, ""hidden_dim"": 32, ""num_layers"": 3}}"
3,13.767009678145923,1.9710988569443244,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0011369027867815917, ""batch_size"": 256, ""dropout"": 0.32087879230161587, ""hidden_dim"": 64, ""num_layers"": 1}}"
4,11.147180627768156,0.7677646989830709,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002100236158351101, ""batch_size"": 128, ""dropout"": 0.13007332881069883, ""hidden_dim"": 128, ""num_layers"": 2}}"
5,16.64324562440682,1.1775872596690693,20,"{""representation"": ""regime"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0006410050209458448, ""batch_size"": 256, ""dropout"": 0.20342827646588113, ""hidden_dim"": 32, ""num_layers"": 3}}"
6,16.509445504778427,1.1677598227513084,20,"{""representation"": ""global"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.000768316328806251, ""batch_size"": 128, ""dropout"": 0.3584365199693973, ""hidden_dim"": 32, ""num_layers"": 2}}"
7,15.631614780576028,1.084426117430718,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001052380849166306, ""batch_size"": 256, ""dropout"": 0.3887128330883843, ""hidden_dim"": 32, ""num_layers"": 1}}"
8,13.87386098088224,1.1921362227155041,40,"{""representation"": ""global"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0015432019542643118, ""batch_size"": 128, ""dropout"": 0.30464784613148704, ""hidden_dim"": 64, ""num_layers"": 2}}"
10,12.211802354155756,0.7697091903459352,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001166641358902415, ""batch_size"": 128, ""dropout"": 0.182290204565902, ""hidden_dim"": 128, ""num_layers"": 1}}"
11,10.852563143370915,0.6884269088471875,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0034158985933588444, ""batch_size"": 128, ""dropout"": 0.16783224686740078, ""hidden_dim"": 128, ""num_layers"": 2}}"
12,12.008839639678186,0.6813218642283552,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.004159471535624509, ""batch_size"": 128, ""dropout"": 0.31279740542841283, ""hidden_dim"": 32, ""num_layers"": 2}}"
13,10.779658383756185,0.4411546636192909,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0028119014015017527, ""batch_size"": 128, ""dropout"": 0.1375070688208361, ""hidden_dim"": 128, ""num_layers"": 2}}"
14,10.660362084430751,0.26787176506580673,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.004546119817091371, ""batch_size"": 128, ""dropout"": 0.12403931089330704, ""hidden_dim"": 32, ""num_layers"": 2}}"
15,10.359862160677423,0.9349348416692347,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.003382547473928405, ""batch_size"": 128, ""dropout"": 0.07545278294979764, ""hidden_dim"": 32, ""num_layers"": 2}}"
17,14.517220813387118,1.720928732251988,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.003218810548839324, ""batch_size"": 128, ""dropout"": 0.20672804462526323, ""hidden_dim"": 32, ""num_layers"": 1}}"
19,12.761509188024766,0.9572501624557763,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0026149750863240866, ""batch_size"": 128, ""dropout"": 0.2679755645647253, ""hidden_dim"": 32, ""num_layers"": 3}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/PyTorch_Transformer/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,15.857527574237702,1.5879332127418329,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002552951604697378, ""batch_size"": 256, ""dropout"": 0.08493564427131046, ""d_model"": 128, ""nhead"": 2, ""num_layers"": 1, ""dim_feedforward"": 64}}"
1,17.524119827722227,1.1028410960272206,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0005807932994623225, ""batch_size"": 256, ""dropout"": 0.12184550766934828, ""d_model"": 64, ""nhead"": 4, ""num_layers"": 1, ""dim_feedforward"": 64}}"
2,17.419334412421946,0.5218426918326946,20,"{""representation"": ""regime"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.004176805377655879, ""batch_size"": 256, ""dropout"": 0.13013213230530574, ""d_model"": 128, ""nhead"": 2, ""num_layers"": 2, ""dim_feedforward"": 128}}"
3,17.43792843936753,0.633326065263102,20,"{""representation"": ""global"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0011413943879952562, ""batch_size"": 256, ""dropout"": 0.13235920994105968, ""d_model"": 128, ""nhead"": 2, ""num_layers"": 3, ""dim_feedforward"": 256}}"
4,17.329422883252118,0.9096337242523121,10,"{""representation"": ""global"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0021646137637899576, ""batch_size"": 512, ""dropout"": 0.09971689165954999, ""d_model"": 64, ""nhead"": 4, ""num_layers"": 1, ""dim_feedforward"": 64}}"
5,13.89894163537785,0.3471401760895673,40,"{""representation"": ""global"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0008450942890954814, ""batch_size"": 512, ""dropout"": 0.0027808522124762817, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 256}}"
6,17.46066360158139,0.8891061134237778,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001590922513078389, ""batch_size"": 512, ""dropout"": 0.09582475626678898, ""d_model"": 128, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 128}}"
7,17.770682002461324,0.964494913999741,20,"{""representation"": ""regime"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002379766606935902, ""batch_size"": 256, ""dropout"": 0.25806911616378, ""d_model"": 64, ""nhead"": 2, ""num_layers"": 2, ""dim_feedforward"": 128}}"
8,18.31168372703029,2.9222234199628114,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0021481839210591726, ""batch_size"": 512, ""dropout"": 0.35884410398103084, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 64}}"
9,18.90911868119453,2.789119472442537,50,"{""representation"": ""global"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0010577175308188558, ""batch_size"": 512, ""dropout"": 0.26304515692013736, ""d_model"": 32, ""nhead"": 2, ""num_layers"": 3, ""dim_feedforward"": 128}}"
11,15.106615321470832,0.7991188509075329,40,"{""representation"": ""global"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0020694942592001664, ""batch_size"": 512, ""dropout"": 0.06760461674802887, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 256}}"
12,16.851626492381513,1.291826983631139,30,"{""representation"": ""global"", ""window"": 30, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0006883254493200618, ""batch_size"": 512, ""dropout"": 0.019661904935623163, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 256}}"
13,14.245928404143733,0.9608452848402697,40,"{""representation"": ""global"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002472635654115835, ""batch_size"": 512, ""dropout"": 0.03828550211524855, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 2, ""dim_feedforward"": 256}}"
15,15.659122527233292,1.9344608775724177,40,"{""representation"": ""global"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0005979408521997784, ""batch_size"": 512, ""dropout"": 0.059507839068351595, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 1, ""dim_feedforward"": 256}}"
16,14.798318341529486,0.5401223464839813,40,"{""representation"": ""global"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0011060710176912544, ""batch_size"": 512, ""dropout"": 0.11325883028216174, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 2, ""dim_feedforward"": 64}}"
17,17.896763993044896,1.5471530162157003,40,"{""representation"": ""global"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.003093290328063297, ""batch_size"": 512, ""dropout"": 0.1312252764942347, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 2, ""dim_feedforward"": 256}}"
18,14.515971242466492,1.1899638550627676,40,"{""representation"": ""global"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0013550883891099543, ""batch_size"": 512, ""dropout"": 0.03607781615609277, ""d_model"": 32, ""nhead"": 2, ""num_layers"": 2, ""dim_feedforward"": 256}}"
19,17.12346270816603,1.729068206946723,60,"{""representation"": ""global"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0009202750223945835, ""batch_size"": 512, ""dropout"": 0.1911297179391442, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 2, ""dim_feedforward"": 256}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/Random_Forest/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,16.76682385128887,0.3812334131591475,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 50, ""max_depth"": 8, ""min_samples_leaf"": 1, ""max_features"": 0.9063233020424546}}"
1,16.56708649796442,0.2724854150108478,10,"{""representation"": ""regime"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 100, ""max_depth"": 17, ""min_samples_leaf"": 2, ""max_features"": 0.5045012539746527}}"
2,16.642134190784283,0.35365036582452863,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 17, ""min_samples_leaf"": 1, ""max_features"": 0.7252813963310067}}"
3,16.694939907551053,0.31906562419990775,10,"{""representation"": ""global"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 50, ""max_depth"": 23, ""min_samples_leaf"": 3, ""max_features"": 0.7637655990477874}}"
4,16.620319249763817,0.3756494185706626,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 300, ""max_depth"": 20, ""min_samples_leaf"": 10, ""max_features"": 0.9263791452993542}}"
5,14.407347579015513,0.22889648958203485,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 200, ""max_depth"": 8, ""min_samples_leaf"": 9, ""max_features"": 0.35218545057583955}}"
6,16.590375740073235,0.3647770502733611,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 250, ""max_depth"": 19, ""min_samples_leaf"": 8, ""max_features"": 0.8398892426801621}}"
7,13.680082010141481,0.3116348085890366,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 14, ""min_samples_leaf"": 2, ""max_features"": 0.7992713510560965}}"
8,16.492690003458115,0.3461402328144594,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 14, ""min_samples_leaf"": 1, ""max_features"": 0.37552399889531307}}"
10,13.912050059290067,0.32351039445748797,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 150, ""max_depth"": 19, ""min_samples_leaf"": 5, ""max_features"": 0.9292785077841608}}"
11,13.92200609245084,0.3278118529567766,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 200, ""max_depth"": 19, ""min_samples_leaf"": 6, ""max_features"": 0.9226072784555683}}"
12,14.025216030844183,0.3119709442489282,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 10, ""min_samples_leaf"": 4, ""max_features"": 0.9208176442899881}}"
13,16.502407926175433,0.31328515773071536,15,"{""representation"": ""regime"", ""temporal"": true, ""window"": 15, ""hyper"": {""n_estimators"": 300, ""max_depth"": 13, ""min_samples_leaf"": 4, ""max_features"": 0.6634774968148336}}"
14,12.972495155060802,0.41882810422789846,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 100, ""max_depth"": 14, ""min_samples_leaf"": 4, ""max_features"": 0.8722058838560489}}"
15,13.157616855215215,0.47587040299062094,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 11, ""min_samples_leaf"": 2, ""max_features"": 0.9764988234657568}}"
16,13.530496007486045,0.48808673183884577,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 150, ""max_depth"": 8, ""min_samples_leaf"": 5, ""max_features"": 0.9735635675255654}}"
17,12.92518769053633,0.46313197158643266,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 11, ""min_samples_leaf"": 2, ""max_features"": 0.8821884912421365}}"
18,12.833809032016166,0.38713341888482633,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 9, ""min_samples_leaf"": 3, ""max_features"": 0.7185632335928457}}"
20,13.997639657271833,0.36416858063584123,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 6, ""min_samples_leaf"": 1, ""max_features"": 0.8705050993935819}}"
21,13.746769998062764,0.34584726448742237,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 6, ""min_samples_leaf"": 5, ""max_features"": 0.7444439768765624}}"
22,12.870625114816661,0.43871374478936687,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 13, ""min_samples_leaf"": 4, ""max_features"": 0.8394703043347376}}"
24,13.132671913471384,0.3785491640017009,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 7, ""min_samples_leaf"": 3, ""max_features"": 0.6210783302787224}}"
25,12.857532558960278,0.42862642807970336,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 10, ""min_samples_leaf"": 4, ""max_features"": 0.7796912034331297}}"
29,13.064997276645846,0.4545011453359127,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 10, ""min_samples_leaf"": 3, ""max_features"": 0.9224573428023899}}"
30,12.689842920863288,0.38834824881056745,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 150, ""max_depth"": 17, ""min_samples_leaf"": 4, ""max_features"": 0.6972566066425376}}"
31,12.72570736584568,0.3911639878205814,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 150, ""max_depth"": 21, ""min_samples_leaf"": 5, ""max_features"": 0.7590426134833621}}"
33,12.715928367069955,0.4035034560804442,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 100, ""max_depth"": 24, ""min_samples_leaf"": 6, ""max_features"": 0.6571366540821155}}"
34,12.54326683900263,0.3768556934644988,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 50, ""max_depth"": 22, ""min_samples_leaf"": 6, ""max_features"": 0.5502433808472529}}"
35,12.57573681633088,0.37887919460098984,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 50, ""max_depth"": 22, ""min_samples_leaf"": 7, ""max_features"": 0.4787298939028911}}"
36,12.656796700442262,0.41826265472332314,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 100, ""max_depth"": 24, ""min_samples_leaf"": 7, ""max_features"": 0.6479328323012915}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/Ridge/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,20.095209482725487,0.6916516309645419,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 0.06026889128682508}}"
1,20.09505866061202,0.6916413460773762,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 34.70266988650411}}"
2,20.095209372157065,0.6916516236872693,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 0.08111941985431921}}"
3,20.095208287767786,0.6916515523052351,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 0.2858549394196191}}"
4,18.059867969850465,0.38705622017315067,15,"{""representation"": ""global"", ""temporal"": true, ""window"": 15, ""hyper"": {""alpha"": 0.0712230583333387}}"
5,20.09520803586304,0.6916515357202747,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 0.3334792728637583}}"
6,20.095194179155364,0.6916506216750673,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 2.991469302130215}}"
7,15.754577423273622,0.3160934196108708,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 297.94544625913585}}"
8,15.476919305166058,0.3308069332954217,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 5.1698794422985355}}"
9,16.180112508544163,0.34230983030864026,30,"{""representation"": ""regime"", ""temporal"": true, ""window"": 30, ""hyper"": {""alpha"": 0.023454342277260523}}"
10,15.479231405512957,0.3323961650188332,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 2.3261440591233864}}"
11,15.479262823698225,0.33241413482179866,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 2.295572830474728}}"
12,15.478054233859933,0.33163244263650987,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 3.6252499263843423}}"
13,14.773805259540731,0.34290683788073206,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 2.7432006787044596}}"
14,14.777694681131845,0.34398304180436,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 0.5882561226051819}}"
15,14.777940771765776,0.3440473688853979,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 0.46427932900777913}}"
17,14.778097382327806,0.3440880907121852,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 0.38607358785868734}}"
21,14.778302025925703,0.34414105196011413,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 0.2846827183520669}}"
22,14.760963553897707,0.337984274545542,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 14.765476615095691}}"
23,15.046012343043424,0.31660171036323903,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""alpha"": 53.861200162061195}}"
24,14.759869526672414,0.3373413086744933,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 16.63820832872317}}"
26,14.757168740549364,0.3350934233650257,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 23.857317635851768}}"
28,14.83785389833363,0.31834208757187077,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 160.17509344014138}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/XGBoost/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,17.319585726871008,0.2793108627834445,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 4, ""learning_rate"": 0.023406777359270506, ""subsample"": 0.9330880728874675, ""colsample_bytree"": 0.8005575058716043, ""min_child_weight"": 8, ""reg_lambda"": 0.011527987128232402}}"
1,16.579086679343455,0.4078964467475169,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 150, ""max_depth"": 5, ""learning_rate"": 0.08283072221077194, ""subsample"": 0.7159725093210578, ""colsample_bytree"": 0.645614570099021, ""min_child_weight"": 7, ""reg_lambda"": 0.02621087878265441}}"
2,14.07103146250859,0.3073854644817386,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.17855789220062088, ""subsample"": 0.6523068845866853, ""colsample_bytree"": 0.5488360570031919, ""min_child_weight"": 7, ""reg_lambda"": 0.2091498132903561}}"
3,13.18528857788849,0.36378657635625183,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.2256465100244266, ""subsample"": 0.7989499894055425, ""colsample_bytree"": 0.9609371175115584, ""min_child_weight"": 1, ""reg_lambda"": 0.03872118032174583}}"
4,16.533640604873877,0.40230258489104964,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 500, ""max_depth"": 5, ""learning_rate"": 0.04279909806035244, ""subsample"": 0.7713480415791243, ""colsample_bytree"": 0.5704621124873813, ""min_child_weight"": 9, ""reg_lambda"": 0.01673601016782578}}"
5,17.34455471224755,0.379513120046534,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 500, ""max_depth"": 8, ""learning_rate"": 0.14401540623155276, ""subsample"": 0.8856351733429728, ""colsample_bytree"": 0.5370223258670452, ""min_child_weight"": 4, ""reg_lambda"": 0.022264204303769682}}"
6,16.934915941245684,0.4372093748829086,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 5, ""learning_rate"": 0.14424921039779712, ""subsample"": 0.8187787356776066, ""colsample_bytree"": 0.9436063712881633, ""min_child_weight"": 5, ""reg_lambda"": 0.022844556850020535}}"
7,12.441586979585068,0.33579684951011196,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 7, ""learning_rate"": 0.2335667365205794, ""subsample"": 0.6246461145744375, ""colsample_bytree"": 0.7051914615178149, ""min_child_weight"": 8, ""reg_lambda"": 0.04857295179217168}}"
8,12.532292987560128,0.3009425059470659,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.04731851459137855, ""subsample"": 0.5550259622638384, ""colsample_bytree"": 0.6139675812709708, ""min_child_weight"": 5, ""reg_lambda"": 2.8447512555118193}}"
10,11.720370435629258,0.30880101896114265,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 150, ""max_depth"": 5, ""learning_rate"": 0.037971224932176945, ""subsample"": 0.5113414878903475, ""colsample_bytree"": 0.7639039667737649, ""min_child_weight"": 9, ""reg_lambda"": 0.040326274350971233}}"
11,11.755653540074114,0.21887143979942514,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 100, ""max_depth"": 6, ""learning_rate"": 0.044744726219686524, ""subsample"": 0.5076106262015234, ""colsample_bytree"": 0.6128456930141007, ""min_child_weight"": 7, ""reg_lambda"": 0.05456165878752718}}"
12,12.225729605763267,0.2662182409197801,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 100, ""max_depth"": 6, ""learning_rate"": 0.030265842358651006, ""subsample"": 0.5186190979792265, ""colsample_bytree"": 0.5525168632521062, ""min_child_weight"": 9, ""reg_lambda"": 0.6337968887155034}}"
17,11.674952192045135,0.30698259283744367,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 7, ""learning_rate"": 0.044268816517276335, ""subsample"": 0.5058375265269975, ""colsample_bytree"": 0.8068276378678519, ""min_child_weight"": 7, ""reg_lambda"": 0.02999893652510109}}"
18,11.83512160271578,0.2911591434164801,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 4, ""learning_rate"": 0.04141710750243501, ""subsample"": 0.6133662288645114, ""colsample_bytree"": 0.8035518048064939, ""min_child_weight"": 10, ""reg_lambda"": 0.15332447846691546}}"
21,11.601988433805207,0.2955760408148963,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 5, ""learning_rate"": 0.04497058730175624, ""subsample"": 0.5322318572547685, ""colsample_bytree"": 0.8135903515086499, ""min_child_weight"": 5, ""reg_lambda"": 0.07887720333722775}}"
22,11.613188723235096,0.31177063061141674,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 7, ""learning_rate"": 0.03463817552736328, ""subsample"": 0.5228416030887572, ""colsample_bytree"": 0.7595732337188618, ""min_child_weight"": 5, ""reg_lambda"": 0.13710729385736264}}"
23,11.52461162349248,0.31716314462990497,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 350, ""max_depth"": 6, ""learning_rate"": 0.02947277257759345, ""subsample"": 0.5018637686000093, ""colsample_bytree"": 0.7724656667183163, ""min_child_weight"": 6, ""reg_lambda"": 0.2224215273464219}}"
28,11.563795112362733,0.2884923379491914,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 5, ""learning_rate"": 0.035442400671479204, ""subsample"": 0.617983202143129, ""colsample_bytree"": 0.7560242923488677, ""min_child_weight"": 8, ""reg_lambda"": 0.07432140847455178}}"
29,11.672015361168995,0.292216811982584,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 3, ""learning_rate"": 0.07832693466719286, ""subsample"": 0.5235590912964805, ""colsample_bytree"": 0.8407076184027162, ""min_child_weight"": 7, ""reg_lambda"": 0.1313073642990264}}"
31,11.75244712349429,0.31907763063047895,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 7, ""learning_rate"": 0.026151383076854853, ""subsample"": 0.5993524009895628, ""colsample_bytree"": 0.8013551486638203, ""min_child_weight"": 6, ""reg_lambda"": 0.6873976991061016}}"
32,11.716218995880896,0.2903583343210934,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 7, ""learning_rate"": 0.04722142198362507, ""subsample"": 0.5236273690794424, ""colsample_bytree"": 0.8186095961465107, ""min_child_weight"": 5, ""reg_lambda"": 0.12431336067461315}}"
34,11.83488502498064,0.326289362395513,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 7, ""learning_rate"": 0.028262188269155216, ""subsample"": 0.5416542741483283, ""colsample_bytree"": 0.8427986293144794, ""min_child_weight"": 6, ""reg_lambda"": 0.06773882791125278}}"
38,11.627342114188759,0.3060399759339254,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 6, ""learning_rate"": 0.05278061495871809, ""subsample"": 0.5970721155002537, ""colsample_bytree"": 0.8762938381168245, ""min_child_weight"": 6, ""reg_lambda"": 0.0942406667473687}}"
41,11.717609539343076,0.3323692950473844,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 7, ""learning_rate"": 0.024432514807537085, ""subsample"": 0.5463980487819587, ""colsample_bytree"": 0.7480576316909402, ""min_child_weight"": 6, ""reg_lambda"": 0.47362388167062486}}"
43,11.778159363712067,0.3373267862882093,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 7, ""learning_rate"": 0.04301407029315687, ""subsample"": 0.5839436891812589, ""colsample_bytree"": 0.8052508181145266, ""min_child_weight"": 4, ""reg_lambda"": 0.07282533697918711}}"
44,11.540056643246468,0.29201920546979726,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 6, ""learning_rate"": 0.02765693078698783, ""subsample"": 0.5645863032686972, ""colsample_bytree"": 0.7783828923824553, ""min_child_weight"": 5, ""reg_lambda"": 0.27281805681976545}}"
45,11.484674611358395,0.2521596528237166,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 5, ""learning_rate"": 0.04657358625702891, ""subsample"": 0.511030101653703, ""colsample_bytree"": 0.7707085002132299, ""min_child_weight"": 4, ""reg_lambda"": 0.09561856410943279}}"
46,11.503430890048255,0.281564200179239,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 5, ""learning_rate"": 0.04056976969708735, ""subsample"": 0.6352602230224578, ""colsample_bytree"": 0.733838070797571, ""min_child_weight"": 5, ""reg_lambda"": 0.05223703681944978}}"
50,11.583754614585509,0.29923427097304606,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 400, ""max_depth"": 6, ""learning_rate"": 0.024857474923458345, ""subsample"": 0.5722879678548346, ""colsample_bytree"": 0.8038443666836932, ""min_child_weight"": 7, ""reg_lambda"": 0.1595045661188249}}"
51,11.51061042803384,0.288283532780785,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 5, ""learning_rate"": 0.05839414417376069, ""subsample"": 0.620661916746592, ""colsample_bytree"": 0.7126497522110885, ""min_child_weight"": 8, ""reg_lambda"": 0.06445853091704849}}"
52,11.444987999476762,0.3220836511678184,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 5, ""learning_rate"": 0.06564878783089668, ""subsample"": 0.5982908706815663, ""colsample_bytree"": 0.7274194062420601, ""min_child_weight"": 9, ""reg_lambda"": 0.08399830609508718}}"
53,11.437375296966904,0.2893139285933722,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 5, ""learning_rate"": 0.05054375608872033, ""subsample"": 0.5985106605947993, ""colsample_bytree"": 0.7155905690443694, ""min_child_weight"": 8, ""reg_lambda"": 0.06630102558709826}}"
54,11.46786346426959,0.2842460378257341,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 5, ""learning_rate"": 0.054119224819667515, ""subsample"": 0.6514795505105953, ""colsample_bytree"": 0.7530270842331994, ""min_child_weight"": 5, ""reg_lambda"": 0.11119578382690866}}"
59,11.640682563230412,0.3253827210064215,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 6, ""learning_rate"": 0.07008702236269544, ""subsample"": 0.6161239469908242, ""colsample_bytree"": 0.7815109345577347, ""min_child_weight"": 8, ""reg_lambda"": 0.04965996046023937}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD001/XGBoost_NASA_Loss/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,27.067070885951427,0.2435897738956679,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 4, ""learning_rate"": 0.023406777359270506, ""subsample"": 0.9330880728874675, ""colsample_bytree"": 0.8005575058716043, ""min_child_weight"": 8, ""reg_lambda"": 0.011527987128232402}}"
1,17.08301683923915,0.708269313698578,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 150, ""max_depth"": 5, ""learning_rate"": 0.08283072221077194, ""subsample"": 0.7159725093210578, ""colsample_bytree"": 0.645614570099021, ""min_child_weight"": 7, ""reg_lambda"": 0.02621087878265441}}"
2,13.505632465035566,0.3278213827373056,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.17855789220062088, ""subsample"": 0.6523068845866853, ""colsample_bytree"": 0.5488360570031919, ""min_child_weight"": 7, ""reg_lambda"": 0.2091498132903561}}"
3,17.507584049894025,0.3371155459948222,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.2256465100244266, ""subsample"": 0.7989499894055425, ""colsample_bytree"": 0.9609371175115584, ""min_child_weight"": 1, ""reg_lambda"": 0.03872118032174583}}"
4,17.059930341228352,0.6847837359836892,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 500, ""max_depth"": 5, ""learning_rate"": 0.04279909806035244, ""subsample"": 0.7713480415791243, ""colsample_bytree"": 0.5704621124873813, ""min_child_weight"": 9, ""reg_lambda"": 0.01673601016782578}}"
5,17.448323669114608,0.463552354574205,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 500, ""max_depth"": 8, ""learning_rate"": 0.14401540623155276, ""subsample"": 0.8856351733429728, ""colsample_bytree"": 0.5370223258670452, ""min_child_weight"": 4, ""reg_lambda"": 0.022264204303769682}}"
6,17.183339340764967,0.6706495965058542,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 5, ""learning_rate"": 0.14424921039779712, ""subsample"": 0.8187787356776066, ""colsample_bytree"": 0.9436063712881633, ""min_child_weight"": 5, ""reg_lambda"": 0.022844556850020535}}"
7,12.415712495651867,0.3137163037341117,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 7, ""learning_rate"": 0.2335667365205794, ""subsample"": 0.6246461145744375, ""colsample_bytree"": 0.7051914615178149, ""min_child_weight"": 8, ""reg_lambda"": 0.04857295179217168}}"
8,12.043081653755701,0.40101786741119416,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.04731851459137855, ""subsample"": 0.5550259622638384, ""colsample_bytree"": 0.6139675812709708, ""min_child_weight"": 5, ""reg_lambda"": 2.8447512555118193}}"
10,12.099255801449578,0.365901610031991,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 9, ""learning_rate"": 0.04398724797286349, ""subsample"": 0.5409438634048439, ""colsample_bytree"": 0.6604144336321783, ""min_child_weight"": 9, ""reg_lambda"": 4.985390561409462}}"
11,11.995014193398626,0.4308181371590776,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 350, ""max_depth"": 9, ""learning_rate"": 0.03840501767644911, ""subsample"": 0.5092866199966892, ""colsample_bytree"": 0.5507187964196311, ""min_child_weight"": 6, ""reg_lambda"": 2.008070155745467}}"
12,12.10821136376363,0.3915053426985752,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 400, ""max_depth"": 10, ""learning_rate"": 0.03529767541238648, ""subsample"": 0.5336416084668992, ""colsample_bytree"": 0.6070610427030017, ""min_child_weight"": 3, ""reg_lambda"": 3.5945922316651338}}"
16,13.843477055838548,0.3370737190851976,30,"{""representation"": ""regime"", ""temporal"": true, ""window"": 30, ""hyper"": {""n_estimators"": 300, ""max_depth"": 9, ""learning_rate"": 0.05622201509767075, ""subsample"": 0.7350284331870716, ""colsample_bytree"": 0.677591115586103, ""min_child_weight"": 8, ""reg_lambda"": 1.7102928943340545}}"
17,12.301025736567933,0.4043917637578761,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 8, ""learning_rate"": 0.06586079408920886, ""subsample"": 0.6504015815281086, ""colsample_bytree"": 0.7262304639754923, ""min_child_weight"": 4, ""reg_lambda"": 3.999123408683371}}"
18,12.1663888582152,0.40292611649112414,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 7, ""learning_rate"": 0.04484191256570486, ""subsample"": 0.564580736941287, ""colsample_bytree"": 0.6284575620137264, ""min_child_weight"": 4, ""reg_lambda"": 9.214025441325587}}"
19,12.09009769109608,0.390250024271245,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 8, ""learning_rate"": 0.02112960392205049, ""subsample"": 0.5483384261671863, ""colsample_bytree"": 0.6465909340714668, ""min_child_weight"": 6, ""reg_lambda"": 0.46167433617984416}}"
21,12.021475862479411,0.44878446223161567,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 550, ""max_depth"": 7, ""learning_rate"": 0.029182827996427532, ""subsample"": 0.5126318086618256, ""colsample_bytree"": 0.7030719175969316, ""min_child_weight"": 4, ""reg_lambda"": 1.2334383157505344}}"
22,12.155857935455732,0.4354799337455033,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 7, ""learning_rate"": 0.03971149718811872, ""subsample"": 0.539104565787164, ""colsample_bytree"": 0.7877230383007054, ""min_child_weight"": 3, ""reg_lambda"": 0.786102064528537}}"
23,12.00706347799144,0.47733244797675045,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.031154788066919258, ""subsample"": 0.5070060730178346, ""colsample_bytree"": 0.6955132196879131, ""min_child_weight"": 4, ""reg_lambda"": 0.4177923132684552}}"
28,12.05254079519279,0.4470139846743411,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.02080866488799911, ""subsample"": 0.5432254964131438, ""colsample_bytree"": 0.6161405048388857, ""min_child_weight"": 3, ""reg_lambda"": 0.26434540873891244}}"
29,11.951092909581487,0.40214994800452497,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 7, ""learning_rate"": 0.021187381038201156, ""subsample"": 0.5395867231199185, ""colsample_bytree"": 0.5678739777423307, ""min_child_weight"": 4, ""reg_lambda"": 0.542536986575932}}"
32,11.96615092884075,0.4663170196723293,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.027402200195933618, ""subsample"": 0.5269599460197056, ""colsample_bytree"": 0.6700199121244322, ""min_child_weight"": 4, ""reg_lambda"": 0.6879571618447723}}"
33,11.99927122705731,0.41402758785506105,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.02990795169127701, ""subsample"": 0.667895791776338, ""colsample_bytree"": 0.6155822422685586, ""min_child_weight"": 5, ""reg_lambda"": 0.511534338363121}}"
38,11.580224319589117,0.36389776760732395,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.02112810826306038, ""subsample"": 0.559016751733112, ""colsample_bytree"": 0.7026187827543747, ""min_child_weight"": 4, ""reg_lambda"": 1.500014191961642}}"
39,11.662397982057616,0.3174613930725547,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.020191494504008443, ""subsample"": 0.615112474312408, ""colsample_bytree"": 0.7442524992164883, ""min_child_weight"": 7, ""reg_lambda"": 1.5668391716029066}}"
40,11.64612119474063,0.3592541635857185,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 9, ""learning_rate"": 0.021296580081793054, ""subsample"": 0.5602276181502265, ""colsample_bytree"": 0.7527733520636145, ""min_child_weight"": 4, ""reg_lambda"": 2.707953796131806}}"
41,11.61334103524965,0.3615950229673556,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.02768367290536487, ""subsample"": 0.5566710677320735, ""colsample_bytree"": 0.7454285747382317, ""min_child_weight"": 4, ""reg_lambda"": 2.1338426469196703}}"
42,11.577588396527634,0.358398926191592,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 9, ""learning_rate"": 0.024573896923082594, ""subsample"": 0.5390676404175042, ""colsample_bytree"": 0.7644649198877074, ""min_child_weight"": 4, ""reg_lambda"": 0.765201647196041}}"
43,11.705128727055468,0.38003516121205905,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 9, ""learning_rate"": 0.02021337446554709, ""subsample"": 0.6245215622602863, ""colsample_bytree"": 0.7939390866401543, ""min_child_weight"": 4, ""reg_lambda"": 1.2198622030997543}}"
44,11.66034976211255,0.36698866904314886,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.028045722696002813, ""subsample"": 0.5259172444855443, ""colsample_bytree"": 0.7572593792704869, ""min_child_weight"": 3, ""reg_lambda"": 1.3204328655800484}}"
45,11.613935185685403,0.3624149484895613,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 8, ""learning_rate"": 0.027879858777867438, ""subsample"": 0.6205595361556525, ""colsample_bytree"": 0.7121030928706278, ""min_child_weight"": 3, ""reg_lambda"": 1.9345910573595333}}"
46,11.632645624986612,0.33907523200802653,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 9, ""learning_rate"": 0.03425969513178954, ""subsample"": 0.5650756212551585, ""colsample_bytree"": 0.7484663249717277, ""min_child_weight"": 4, ""reg_lambda"": 7.295272920146446}}"
47,11.638817969681993,0.35659605965192126,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 9, ""learning_rate"": 0.029042218673767484, ""subsample"": 0.6087703529580166, ""colsample_bytree"": 0.7572119246016675, ""min_child_weight"": 5, ""reg_lambda"": 0.9724756693274336}}"
48,11.65765633547923,0.339267195411278,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.03221387508736324, ""subsample"": 0.5379510060205123, ""colsample_bytree"": 0.745690217403744, ""min_child_weight"": 3, ""reg_lambda"": 6.525071262521687}}"
49,11.52774810411608,0.341918220167333,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 8, ""learning_rate"": 0.025248359663603392, ""subsample"": 0.6117314349071272, ""colsample_bytree"": 0.6287950600543543, ""min_child_weight"": 3, ""reg_lambda"": 3.39506273740267}}"
50,11.62462634890812,0.3588710624164673,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.02156927187071332, ""subsample"": 0.5477181080891692, ""colsample_bytree"": 0.7641146097757467, ""min_child_weight"": 5, ""reg_lambda"": 1.1438301367624633}}"
51,11.461816151469549,0.323339621942952,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.025733854282328838, ""subsample"": 0.6041566364360244, ""colsample_bytree"": 0.6508389274246551, ""min_child_weight"": 4, ""reg_lambda"": 1.6595768761894054}}"
52,11.475916866458224,0.3677024947712732,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 9, ""learning_rate"": 0.03129383300179426, ""subsample"": 0.5840811890486903, ""colsample_bytree"": 0.6661051259759468, ""min_child_weight"": 5, ""reg_lambda"": 2.079076757090939}}"
53,11.430062364135805,0.3652042173011179,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.02102758780367853, ""subsample"": 0.5976708536394459, ""colsample_bytree"": 0.5965350385710313, ""min_child_weight"": 4, ""reg_lambda"": 1.0691220417954215}}"
54,11.504754092322935,0.38044204844668017,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 8, ""learning_rate"": 0.021230941847836596, ""subsample"": 0.6019334574364501, ""colsample_bytree"": 0.6538306465101256, ""min_child_weight"": 4, ""reg_lambda"": 2.237670333276043}}"
55,11.448667468202265,0.3922067249131563,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 8, ""learning_rate"": 0.027779773318989356, ""subsample"": 0.6688096976290449, ""colsample_bytree"": 0.6398400664526029, ""min_child_weight"": 4, ""reg_lambda"": 1.8640807509633102}}"
56,11.344605138905449,0.3284998674280768,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.024751435372469367, ""subsample"": 0.6144549547485179, ""colsample_bytree"": 0.5773599687412635, ""min_child_weight"": 6, ""reg_lambda"": 1.884080422775671}}"
58,11.465713795204428,0.3652779946779951,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 9, ""learning_rate"": 0.022156915863396957, ""subsample"": 0.6020542092773888, ""colsample_bytree"": 0.6460815230441822, ""min_child_weight"": 4, ""reg_lambda"": 0.8435745859523458}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/CatBoost/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,17.214704222911593,0.2714258214152449,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""iterations"": 200, ""depth"": 5, ""learning_rate"": 0.023406777359270506, ""l2_leaf_reg"": 13.39433470675048}}"
1,16.1876521814311,0.3524810835188935,10,"{""representation"": ""regime"", ""temporal"": true, ""window"": 10, ""hyper"": {""iterations"": 300, ""depth"": 8, ""learning_rate"": 0.029180149047462245, ""l2_leaf_reg"": 2.399324290681271}}"
2,16.736169370820043,0.3103244942364071,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""iterations"": 500, ""depth"": 8, ""learning_rate"": 0.022680881003614695, ""l2_leaf_reg"": 6.172115948107071}}"
3,17.597327849610068,0.26171080334055136,10,"{""representation"": ""global"", ""temporal"": true, ""window"": 10, ""hyper"": {""iterations"": 100, ""depth"": 10, ""learning_rate"": 0.04030685101059539, ""l2_leaf_reg"": 7.277150634170934}}"
4,17.536844489855692,0.2824421434982283,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""iterations"": 800, ""depth"": 9, ""learning_rate"": 0.2546633740246392, ""l2_leaf_reg"": 14.594768942966839}}"
5,14.43838490056979,0.252592414487545,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""iterations"": 500, ""depth"": 4, ""learning_rate"": 0.17558477392891916, ""l2_leaf_reg"": 1.2502377950801113}}"
6,16.97477896782962,0.3205772518578806,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""iterations"": 700, ""depth"": 8, ""learning_rate"": 0.14401540623155276, ""l2_leaf_reg"": 10.079659367870727}}"
7,13.68311260663477,0.25262941933800376,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""iterations"": 800, ""depth"": 7, ""learning_rate"": 0.027649274676728182, ""l2_leaf_reg"": 8.471354625326555}}"
9,16.706606328655177,0.2917684723486109,10,"{""representation"": ""regime"", ""temporal"": true, ""window"": 10, ""hyper"": {""iterations"": 200, ""depth"": 10, ""learning_rate"": 0.17842401600508806, ""l2_leaf_reg"": 6.669254439997733}}"
10,13.831555371254499,0.2666442107307704,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""iterations"": 800, ""depth"": 8, ""learning_rate"": 0.07287261571692631, ""l2_leaf_reg"": 18.445338876089934}}"
11,13.731998294119041,0.243529805191415,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""iterations"": 700, ""depth"": 8, ""learning_rate"": 0.05121944772162538, ""l2_leaf_reg"": 17.183790926280032}}"
12,13.733426932500791,0.28257855605600396,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""iterations"": 800, ""depth"": 5, ""learning_rate"": 0.08397983881234862, ""l2_leaf_reg"": 8.335462491675733}}"
13,16.0676451739214,0.34407534466880757,15,"{""representation"": ""regime"", ""temporal"": true, ""window"": 15, ""hyper"": {""iterations"": 600, ""depth"": 6, ""learning_rate"": 0.030483785700664352, ""l2_leaf_reg"": 14.748043776866151}}"
14,13.41501142020183,0.2084024315845796,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 8, ""learning_rate"": 0.11105536840421122, ""l2_leaf_reg"": 6.887720850200306}}"
15,14.029061110912139,0.18164253825627524,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 500, ""depth"": 9, ""learning_rate"": 0.23835467617615874, ""l2_leaf_reg"": 10.384787246209134}}"
17,15.04226916407535,0.2592276000117612,30,"{""representation"": ""regime"", ""temporal"": true, ""window"": 30, ""hyper"": {""iterations"": 600, ""depth"": 7, ""learning_rate"": 0.05529591307302703, ""l2_leaf_reg"": 4.482847313647247}}"
18,13.390570329172258,0.21042070112037606,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 6, ""learning_rate"": 0.1366076873698663, ""l2_leaf_reg"": 10.154674034831803}}"
19,13.340393182454028,0.16675219489834228,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 300, ""depth"": 4, ""learning_rate"": 0.08553139795207124, ""l2_leaf_reg"": 9.081461558834915}}"
20,13.440145318243705,0.20454793697589213,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 500, ""depth"": 5, ""learning_rate"": 0.22393115571852562, ""l2_leaf_reg"": 6.43838569827916}}"
21,13.184762401804884,0.19488603558235296,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 400, ""depth"": 4, ""learning_rate"": 0.14585898254331822, ""l2_leaf_reg"": 10.961419162508166}}"
22,13.245178449438257,0.18353704614283597,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 400, ""depth"": 4, ""learning_rate"": 0.11389681164418043, ""l2_leaf_reg"": 17.087939980398357}}"
23,13.206915966632199,0.17368493856975833,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 300, ""depth"": 4, ""learning_rate"": 0.14436350838076742, ""l2_leaf_reg"": 14.65815948659939}}"
24,13.262144333180007,0.18460462383944767,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 400, ""depth"": 4, ""learning_rate"": 0.08532230141491727, ""l2_leaf_reg"": 18.694025675502782}}"
25,13.323370167424041,0.1998127777397082,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 300, ""depth"": 5, ""learning_rate"": 0.21669431469148043, ""l2_leaf_reg"": 11.95445530129262}}"
29,13.421190682940553,0.19516412619899345,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 400, ""depth"": 4, ""learning_rate"": 0.2575024857935579, ""l2_leaf_reg"": 18.537595087548667}}"
31,13.210794508992581,0.17626931116201053,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 500, ""depth"": 4, ""learning_rate"": 0.07354058338497164, ""l2_leaf_reg"": 15.561897601730811}}"
32,13.285292542715258,0.1597705523246684,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 500, ""depth"": 4, ""learning_rate"": 0.06131689896965561, ""l2_leaf_reg"": 12.812292295064001}}"
34,13.178171974495436,0.18035765601771375,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 4, ""learning_rate"": 0.0791753207005545, ""l2_leaf_reg"": 19.39385722644095}}"
35,13.296149080382424,0.15903877398003533,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 400, ""depth"": 4, ""learning_rate"": 0.1776002542613077, ""l2_leaf_reg"": 17.009552660010375}}"
36,13.163687356788888,0.18070428560744084,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 500, ""depth"": 4, ""learning_rate"": 0.11287044990293908, ""l2_leaf_reg"": 6.375221407140048}}"
37,13.294519265272715,0.17744100143825114,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 300, ""depth"": 4, ""learning_rate"": 0.15989913803652822, ""l2_leaf_reg"": 5.22701229495922}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/LightGBM/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,17.966751303686202,0.22699843514029397,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 4, ""learning_rate"": 0.023406777359270506, ""num_leaves"": 112, ""subsample"": 0.8005575058716043, ""subsample_freq"": 1, ""colsample_bytree"": 0.8540362888980227, ""min_child_samples"": 6, ""reg_lambda"": 8.123245085588687}}"
1,16.814246068819262,0.3064462838411662,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 250, ""max_depth"": 7, ""learning_rate"": 0.16767510257004176, ""num_leaves"": 37, ""subsample"": 0.7571172192068059, ""subsample_freq"": 1, ""colsample_bytree"": 0.7962072844310213, ""min_child_samples"": 9, ""reg_lambda"": 0.6647135865318028}}"
2,17.7921425982017,0.24079863793442569,10,"{""representation"": ""global"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 50, ""max_depth"": 12, ""learning_rate"": 0.04030685101059539, ""num_leaves"": 89, ""subsample"": 0.6558555380447055, ""subsample_freq"": 1, ""colsample_bytree"": 0.7600340105889054, ""min_child_samples"": 57, ""reg_lambda"": 0.035856126103453984}}"
3,16.88995452493488,0.3310082659297466,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 400, ""max_depth"": 12, ""learning_rate"": 0.025415885998427375, ""num_leaves"": 37, ""subsample"": 0.522613644455269, ""subsample_freq"": 1, ""colsample_bytree"": 0.6626651653816322, ""min_child_samples"": 42, ""reg_lambda"": 0.06516990611177176}}"
4,17.248991166480472,0.21923914462145336,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.14401540623155276, ""num_leaves"": 102, ""subsample"": 0.5370223258670452, ""subsample_freq"": 1, ""colsample_bytree"": 0.6792328642721364, ""min_child_samples"": 16, ""reg_lambda"": 3.8842777547031417}}"
5,17.111329977387246,0.28601679406253233,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""n_estimators"": 500, ""max_depth"": 8, ""learning_rate"": 0.16134583428320007, ""num_leaves"": 70, ""subsample"": 0.7613664146909971, ""subsample_freq"": 1, ""colsample_bytree"": 0.7137705091792748, ""min_child_samples"": 7, ""reg_lambda"": 0.021070472806578238}}"
6,16.862669794815496,0.32172915324096907,10,"{""representation"": ""regime"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 100, ""max_depth"": 12, ""learning_rate"": 0.17842401600508806, ""num_leaves"": 86, ""subsample"": 0.9357302950938589, ""subsample_freq"": 1, ""colsample_bytree"": 0.9018360384495572, ""min_child_samples"": 22, ""reg_lambda"": 4.760767751809501}}"
7,16.933263756487158,0.2943597054978521,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 5, ""learning_rate"": 0.06358398347237385, ""num_leaves"": 107, ""subsample"": 0.9303652916281717, ""subsample_freq"": 1, ""colsample_bytree"": 0.5034760652655954, ""min_child_samples"": 54, ""reg_lambda"": 0.1787446325623842}}"
8,16.791858673814723,0.35140883518793237,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 300, ""max_depth"": 6, ""learning_rate"": 0.043254212679768214, ""num_leaves"": 19, ""subsample"": 0.8047821669899484, ""subsample_freq"": 1, ""colsample_bytree"": 0.7513395116144308, ""min_child_samples"": 9, ""reg_lambda"": 0.06853925708853059}}"
9,16.626772053476532,0.335282412086966,10,"{""representation"": ""global"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 400, ""max_depth"": 9, ""learning_rate"": 0.08533946549577034, ""num_leaves"": 25, ""subsample"": 0.917651247794619, ""subsample_freq"": 1, ""colsample_bytree"": 0.6603900324858679, ""min_child_samples"": 22, ""reg_lambda"": 0.013253342600066534}}"
11,16.579792081029915,0.30910638697100046,15,"{""representation"": ""global"", ""temporal"": true, ""window"": 15, ""hyper"": {""n_estimators"": 400, ""max_depth"": 8, ""learning_rate"": 0.03148504058194231, ""num_leaves"": 43, ""subsample"": 0.9738941673960433, ""subsample_freq"": 1, ""colsample_bytree"": 0.5516742555929252, ""min_child_samples"": 7, ""reg_lambda"": 0.05156949409633155}}"
12,16.53132380977042,0.3026215730118502,15,"{""representation"": ""global"", ""temporal"": true, ""window"": 15, ""hyper"": {""n_estimators"": 450, ""max_depth"": 9, ""learning_rate"": 0.028364775501108094, ""num_leaves"": 51, ""subsample"": 0.9341616604754772, ""subsample_freq"": 1, ""colsample_bytree"": 0.5139804668786171, ""min_child_samples"": 12, ""reg_lambda"": 0.07531247593718972}}"
13,16.539842375276102,0.3162368528127796,15,"{""representation"": ""global"", ""temporal"": true, ""window"": 15, ""hyper"": {""n_estimators"": 350, ""max_depth"": 8, ""learning_rate"": 0.022460184320335426, ""num_leaves"": 63, ""subsample"": 0.9561255157819958, ""subsample_freq"": 1, ""colsample_bytree"": 0.5471913136657424, ""min_child_samples"": 19, ""reg_lambda"": 0.11743591519765252}}"
14,16.469051211032586,0.30862737904347354,15,"{""representation"": ""global"", ""temporal"": true, ""window"": 15, ""hyper"": {""n_estimators"": 500, ""max_depth"": 8, ""learning_rate"": 0.026037672571212974, ""num_leaves"": 66, ""subsample"": 0.8611764972486696, ""subsample_freq"": 1, ""colsample_bytree"": 0.6312128112199559, ""min_child_samples"": 45, ""reg_lambda"": 0.020612753872210548}}"
15,16.467980406853673,0.3373150767382074,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 7, ""learning_rate"": 0.02438564018992529, ""num_leaves"": 88, ""subsample"": 0.8717576550617425, ""subsample_freq"": 1, ""colsample_bytree"": 0.5353433628779873, ""min_child_samples"": 48, ""reg_lambda"": 0.012432409232403503}}"
16,16.25096081282431,0.34356766061087085,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 550, ""max_depth"": 7, ""learning_rate"": 0.050015982343829944, ""num_leaves"": 52, ""subsample"": 0.9067028984905618, ""subsample_freq"": 1, ""colsample_bytree"": 0.5357945175371621, ""min_child_samples"": 32, ""reg_lambda"": 0.01398982450554864}}"
17,16.62965421405413,0.34913990168111947,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 5, ""learning_rate"": 0.03803635311333154, ""num_leaves"": 66, ""subsample"": 0.8415612278081931, ""subsample_freq"": 1, ""colsample_bytree"": 0.6491393289068321, ""min_child_samples"": 31, ""reg_lambda"": 0.014235690905501499}}"
18,16.707721962265403,0.3487106603174868,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 5, ""learning_rate"": 0.024261445249416112, ""num_leaves"": 85, ""subsample"": 0.8289480281734284, ""subsample_freq"": 1, ""colsample_bytree"": 0.6281677707479584, ""min_child_samples"": 59, ""reg_lambda"": 0.034990620634422216}}"
19,16.190789484278227,0.3942180631846189,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 8, ""learning_rate"": 0.03479093496713468, ""num_leaves"": 99, ""subsample"": 0.9090201529706396, ""subsample_freq"": 1, ""colsample_bytree"": 0.6005691810889251, ""min_child_samples"": 24, ""reg_lambda"": 0.016476941284687244}}"
20,16.11665054965249,0.4079017401388947,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 9, ""learning_rate"": 0.059664916265293313, ""num_leaves"": 98, ""subsample"": 0.8050776115822735, ""subsample_freq"": 1, ""colsample_bytree"": 0.6352770114378589, ""min_child_samples"": 33, ""reg_lambda"": 0.06540216869046417}}"
21,16.199558526981882,0.40663649777428196,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 9, ""learning_rate"": 0.058231440326209256, ""num_leaves"": 116, ""subsample"": 0.8351567801922987, ""subsample_freq"": 1, ""colsample_bytree"": 0.6129177540574208, ""min_child_samples"": 27, ""reg_lambda"": 0.024584743686069763}}"
22,16.049179816465983,0.41130396641385303,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.036286898229899535, ""num_leaves"": 109, ""subsample"": 0.760964763118192, ""subsample_freq"": 1, ""colsample_bytree"": 0.6119929092783193, ""min_child_samples"": 43, ""reg_lambda"": 0.01090127323211344}}"
23,16.048478003593406,0.38037102632207503,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.026828758548179427, ""num_leaves"": 110, ""subsample"": 0.9395030774043833, ""subsample_freq"": 1, ""colsample_bytree"": 0.5351823942072191, ""min_child_samples"": 53, ""reg_lambda"": 0.02875650339912861}}"
24,16.039881155769486,0.36937776372633163,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.028498211478252045, ""num_leaves"": 116, ""subsample"": 0.83137704375156, ""subsample_freq"": 1, ""colsample_bytree"": 0.5222847057881218, ""min_child_samples"": 57, ""reg_lambda"": 0.01781452150482256}}"
25,16.165145393623188,0.4230362726106751,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.024248167492139566, ""num_leaves"": 112, ""subsample"": 0.8417865831405332, ""subsample_freq"": 1, ""colsample_bytree"": 0.6425428048445602, ""min_child_samples"": 53, ""reg_lambda"": 0.04060064682019708}}"
26,16.13295579740632,0.38469003118114664,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.02247596347735701, ""num_leaves"": 116, ""subsample"": 0.975871402216474, ""subsample_freq"": 1, ""colsample_bytree"": 0.5355794719438106, ""min_child_samples"": 47, ""reg_lambda"": 0.12820338755574887}}"
27,16.023896423479133,0.3935975189667164,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 11, ""learning_rate"": 0.030246787191340944, ""num_leaves"": 124, ""subsample"": 0.8591942026525933, ""subsample_freq"": 1, ""colsample_bytree"": 0.5886733440378593, ""min_child_samples"": 48, ""reg_lambda"": 0.012873097436889353}}"
28,16.11644124696243,0.37762646540094885,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 9, ""learning_rate"": 0.040172360631642534, ""num_leaves"": 127, ""subsample"": 0.8943880337472786, ""subsample_freq"": 1, ""colsample_bytree"": 0.5276717826815841, ""min_child_samples"": 58, ""reg_lambda"": 0.03287881114568645}}"
29,16.441202251866954,0.29982475513143314,20,"{""representation"": ""global"", ""temporal"": true, ""window"": 20, ""hyper"": {""n_estimators"": 550, ""max_depth"": 9, ""learning_rate"": 0.023631814102922918, ""num_leaves"": 117, ""subsample"": 0.8609930948077024, ""subsample_freq"": 1, ""colsample_bytree"": 0.5148716290198373, ""min_child_samples"": 67, ""reg_lambda"": 0.029958002819596208}}"
30,16.01003985123885,0.3775475685478204,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 12, ""learning_rate"": 0.0419183114859604, ""num_leaves"": 105, ""subsample"": 0.8393806622570323, ""subsample_freq"": 1, ""colsample_bytree"": 0.5316075468761997, ""min_child_samples"": 45, ""reg_lambda"": 0.10093047076361904}}"
31,16.13574847157434,0.38753720584608836,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 10, ""learning_rate"": 0.027697150921959488, ""num_leaves"": 112, ""subsample"": 0.8472902858062199, ""subsample_freq"": 1, ""colsample_bytree"": 0.5746311761980271, ""min_child_samples"": 50, ""reg_lambda"": 0.01098804287040915}}"
32,16.03206036017441,0.37990674848636574,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 12, ""learning_rate"": 0.03453627975198074, ""num_leaves"": 122, ""subsample"": 0.8029236144174253, ""subsample_freq"": 1, ""colsample_bytree"": 0.5708525443657003, ""min_child_samples"": 37, ""reg_lambda"": 0.017762494432349563}}"
33,16.305490879696855,0.28104292046577267,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 600, ""max_depth"": 9, ""learning_rate"": 0.03697775381245331, ""num_leaves"": 99, ""subsample"": 0.8372562950297838, ""subsample_freq"": 1, ""colsample_bytree"": 0.538829448665663, ""min_child_samples"": 38, ""reg_lambda"": 0.015764753603386868}}"
34,16.0878434391199,0.4152386063126727,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 11, ""learning_rate"": 0.04477567110029569, ""num_leaves"": 123, ""subsample"": 0.9503481327366597, ""subsample_freq"": 1, ""colsample_bytree"": 0.6106876516950938, ""min_child_samples"": 38, ""reg_lambda"": 0.01498833013815836}}"
35,16.0461584703154,0.390372229104043,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 11, ""learning_rate"": 0.03474324151584908, ""num_leaves"": 119, ""subsample"": 0.8204523122547766, ""subsample_freq"": 1, ""colsample_bytree"": 0.5707323310681476, ""min_child_samples"": 67, ""reg_lambda"": 0.015197643379659656}}"
36,16.030049895189087,0.4024362210444074,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 12, ""learning_rate"": 0.03544174272765035, ""num_leaves"": 114, ""subsample"": 0.8605905785683989, ""subsample_freq"": 1, ""colsample_bytree"": 0.5794761479475958, ""min_child_samples"": 34, ""reg_lambda"": 0.021130912520703208}}"
37,16.40161649397414,0.28047731821551825,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""n_estimators"": 550, ""max_depth"": 12, ""learning_rate"": 0.03645854168074379, ""num_leaves"": 100, ""subsample"": 0.7803788922149746, ""subsample_freq"": 1, ""colsample_bytree"": 0.565861380273627, ""min_child_samples"": 53, ""reg_lambda"": 0.015575470000354317}}"
38,16.026018856345058,0.40473588813253414,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 12, ""learning_rate"": 0.040175999040011484, ""num_leaves"": 118, ""subsample"": 0.8334408284301805, ""subsample_freq"": 1, ""colsample_bytree"": 0.5657196593082496, ""min_child_samples"": 45, ""reg_lambda"": 0.07635309544982892}}"
41,16.0711445397885,0.35407312836882937,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 12, ""learning_rate"": 0.05158353910871441, ""num_leaves"": 112, ""subsample"": 0.8285175158255733, ""subsample_freq"": 1, ""colsample_bytree"": 0.5247049156358992, ""min_child_samples"": 42, ""reg_lambda"": 0.037358932740336054}}"
42,16.071923212834015,0.3769505665433812,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 11, ""learning_rate"": 0.042369969937247655, ""num_leaves"": 112, ""subsample"": 0.8947092346649389, ""subsample_freq"": 1, ""colsample_bytree"": 0.5390620411707889, ""min_child_samples"": 35, ""reg_lambda"": 0.06502302780498423}}"
45,16.011072080957454,0.3971062592600114,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 11, ""learning_rate"": 0.035368776408158886, ""num_leaves"": 118, ""subsample"": 0.8558315431621369, ""subsample_freq"": 1, ""colsample_bytree"": 0.5966684023567044, ""min_child_samples"": 30, ""reg_lambda"": 0.010563863606498263}}"
47,16.071844124849697,0.4047475320234508,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 11, ""learning_rate"": 0.026676599587635505, ""num_leaves"": 96, ""subsample"": 0.8498107030839275, ""subsample_freq"": 1, ""colsample_bytree"": 0.5912085601568511, ""min_child_samples"": 30, ""reg_lambda"": 0.013336396041975349}}"
48,16.05289817889132,0.4089964361687712,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 12, ""learning_rate"": 0.033043509295274706, ""num_leaves"": 123, ""subsample"": 0.8491495266456007, ""subsample_freq"": 1, ""colsample_bytree"": 0.6127042838153893, ""min_child_samples"": 27, ""reg_lambda"": 0.011940750044688593}}"
51,16.130828764229243,0.40342378506804005,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 11, ""learning_rate"": 0.053720449842367776, ""num_leaves"": 119, ""subsample"": 0.8434549419454447, ""subsample_freq"": 1, ""colsample_bytree"": 0.6201155093881032, ""min_child_samples"": 44, ""reg_lambda"": 0.02060629742518724}}"
52,16.01616714724978,0.398580639676367,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 11, ""learning_rate"": 0.03695359611793809, ""num_leaves"": 111, ""subsample"": 0.8091239768654142, ""subsample_freq"": 1, ""colsample_bytree"": 0.5530002594088683, ""min_child_samples"": 44, ""reg_lambda"": 0.026588246514594245}}"
53,16.08530375370313,0.3795365909546484,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.03908470744779721, ""num_leaves"": 109, ""subsample"": 0.8363041252404302, ""subsample_freq"": 1, ""colsample_bytree"": 0.555319944091927, ""min_child_samples"": 30, ""reg_lambda"": 0.012148757928006172}}"
55,16.06211966824016,0.41286492005724884,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 11, ""learning_rate"": 0.03755810882896853, ""num_leaves"": 111, ""subsample"": 0.8884254090365091, ""subsample_freq"": 1, ""colsample_bytree"": 0.5997892414082012, ""min_child_samples"": 44, ""reg_lambda"": 0.030736251165838748}}"
56,16.032246521789236,0.38250419972577115,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 11, ""learning_rate"": 0.04788632362052196, ""num_leaves"": 106, ""subsample"": 0.7710934898145346, ""subsample_freq"": 1, ""colsample_bytree"": 0.5501301785150053, ""min_child_samples"": 40, ""reg_lambda"": 0.01810200304351975}}"
59,16.091047469988936,0.4126050271201927,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 12, ""learning_rate"": 0.04227684333101714, ""num_leaves"": 120, ""subsample"": 0.899786181575189, ""subsample_freq"": 1, ""colsample_bytree"": 0.6106652261670892, ""min_child_samples"": 28, ""reg_lambda"": 0.032000162765704464}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/MLP/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,17.013907245330746,0.35329040811518325,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.002537815508265664, ""learning_rate_init"": 0.0026070247583707684}}"
1,16.927669509193066,0.3398886855706222,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.0005342937261279777, ""learning_rate_init"": 0.0003823475224675188}}"
2,17.65817091273574,0.2854046154783819,15,"{""representation"": ""global"", ""temporal"": true, ""window"": 15, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.01712337597316398, ""learning_rate_init"": 0.0004066563313514797}}"
3,16.922053422287387,0.38189122665416314,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.004467752817973905, ""learning_rate_init"": 0.0004201672054372534}}"
4,17.24716441500912,0.38282830622119274,15,"{""representation"": ""regime"", ""temporal"": true, ""window"": 15, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.02065142557895925, ""learning_rate_init"": 0.0005170191786366995}}"
5,18.548782292956457,0.08899212102042867,15,"{""representation"": ""regime"", ""temporal"": true, ""window"": 15, ""hyper"": {""hidden_layer_sizes"": [128, 64], ""alpha"": 2.9072088906598435e-05, ""learning_rate_init"": 0.005323617594751502}}"
6,17.930151863801616,0.2754819796842558,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.0012329098365270509, ""learning_rate_init"": 0.0007162786999897341}}"
7,17.301564487258297,0.20238055783327466,20,"{""representation"": ""regime"", ""temporal"": true, ""window"": 20, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.01707975034295823, ""learning_rate_init"": 0.0018484491720988621}}"
8,17.399912094947005,0.25156995333526544,20,"{""representation"": ""global"", ""temporal"": true, ""window"": 20, ""hyper"": {""hidden_layer_sizes"": [128, 64], ""alpha"": 0.0004673518999562748, ""learning_rate_init"": 0.00027810936979265544}}"
9,16.765434263854154,0.3547852070548822,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.07076022085822008, ""learning_rate_init"": 0.00031883397351001905}}"
10,16.873189679935756,0.35461181620088383,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 2.5990875064996908e-05, ""learning_rate_init"": 0.00040603622450132007}}"
11,16.894525211728677,0.35223133404167994,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 2.165099942185883e-05, ""learning_rate_init"": 0.00015092395777542054}}"
12,16.83228958978554,0.3522242131668978,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.04491488223084785, ""learning_rate_init"": 0.00014958102201351417}}"
13,16.869844290953885,0.35614495653213507,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.012274627909984785, ""learning_rate_init"": 0.00012946225931831303}}"
14,16.733192759779808,0.35225170542604944,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.09441570932564147, ""learning_rate_init"": 0.00038147208642969516}}"
15,16.74909207654759,0.33176638204167197,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.07781379285996892, ""learning_rate_init"": 0.0010104150426724948}}"
16,16.787257892474393,0.37514648420511615,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.02905304763978254, ""learning_rate_init"": 0.0008709823008230675}}"
20,16.884320299533826,0.36869613065395845,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.0049207738184396145, ""learning_rate_init"": 0.0011478014883273028}}"
22,16.802788496669187,0.35651123370216514,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.04904334441939618, ""learning_rate_init"": 0.0003000304611983544}}"
23,16.787257556655835,0.33824798876392476,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.0952922562373745, ""learning_rate_init"": 0.0009733882573099726}}"
24,16.74267722141553,0.34031485573083803,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.09517527707347101, ""learning_rate_init"": 0.0006236237868716942}}"
25,16.777690999670675,0.3317603224299907,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.08034656039545307, ""learning_rate_init"": 0.0016734874676285702}}"
26,16.750882239103685,0.3214552469395846,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [128, 64], ""alpha"": 0.09855946074091423, ""learning_rate_init"": 0.0005881825934915264}}"
29,16.82204134770382,0.33852258598681473,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.010735895287868676, ""learning_rate_init"": 0.0061917380523240015}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/PyTorch_CNN/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,19.81549362742756,1.136962620252026,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002552951604697378, ""batch_size"": 256, ""dropout"": 0.08493564427131046, ""num_filters"": 128, ""kernel_size"": 3}}"
1,20.395946335083245,1.1451639274382899,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001956070814274847, ""batch_size"": 256, ""dropout"": 0.02602063719411181, ""num_filters"": 64, ""kernel_size"": 3}}"
2,19.92455750749997,1.377562995724743,30,"{""representation"": ""global"", ""window"": 30, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0016559149440361915, ""batch_size"": 512, ""dropout"": 0.31005312934444584, ""num_filters"": 32, ""kernel_size"": 3}}"
3,21.504213737946117,0.7032860235217107,30,"{""representation"": ""global"", ""window"": 30, ""temporal"": false, ""hyper"": {""learning_rate"": 0.000691662498760998, ""batch_size"": 512, ""dropout"": 0.308897907718663, ""num_filters"": 128, ""kernel_size"": 5}}"
4,21.17673662409292,1.9712773081936887,20,"{""representation"": ""global"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0010571906813317184, ""batch_size"": 512, ""dropout"": 0.18888597006477972, ""num_filters"": 128, ""kernel_size"": 5}}"
5,19.65147976743639,0.6446557275276378,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.004041443189081019, ""batch_size"": 512, ""dropout"": 0.09151926619664899, ""num_filters"": 64, ""kernel_size"": 3}}"
6,19.559031163542826,0.4565951861825194,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0006442017924231741, ""batch_size"": 512, ""dropout"": 0.3442922333025374, ""num_filters"": 64, ""kernel_size"": 3}}"
7,19.41437066368742,0.17691717226243497,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0015711531129544835, ""batch_size"": 128, ""dropout"": 0.24382573359195875, ""num_filters"": 32, ""kernel_size"": 3}}"
8,16.58614626839848,0.6698391176056743,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002144252054813578, ""batch_size"": 128, ""dropout"": 0.3341209982356952, ""num_filters"": 32, ""kernel_size"": 5}}"
9,19.151635467881004,0.5006895851504441,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001096569987677643, ""batch_size"": 256, ""dropout"": 0.10317665108606225, ""num_filters"": 64, ""kernel_size"": 3}}"
10,18.719638221203265,0.9605808045431663,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0007605707018582466, ""batch_size"": 128, ""dropout"": 0.09894249765332985, ""num_filters"": 32, ""kernel_size"": 5}}"
11,16.817838861759146,0.9245566274682651,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0005567945772441949, ""batch_size"": 128, ""dropout"": 0.10453762107433612, ""num_filters"": 32, ""kernel_size"": 5}}"
12,16.5139344944436,0.5935870684958385,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.004433118789213871, ""batch_size"": 256, ""dropout"": 0.3531812462139981, ""num_filters"": 32, ""kernel_size"": 5}}"
13,16.809583600156163,0.5674631627814971,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.00424715254713208, ""batch_size"": 128, ""dropout"": 0.3575312644632453, ""num_filters"": 32, ""kernel_size"": 5}}"
14,16.623108726439238,0.4288981746200471,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0038137252394201894, ""batch_size"": 256, ""dropout"": 0.26367121642732727, ""num_filters"": 32, ""kernel_size"": 5}}"
15,17.6605024476958,1.092372812987181,20,"{""representation"": ""regime"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0015269851623182689, ""batch_size"": 128, ""dropout"": 0.37366638431931315, ""num_filters"": 32, ""kernel_size"": 5}}"
16,16.446656721883045,0.5036707985671016,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0012646019561413444, ""batch_size"": 128, ""dropout"": 0.302962690294729, ""num_filters"": 64, ""kernel_size"": 5}}"
17,16.63414338492967,0.6086159603947449,15,"{""representation"": ""regime"", ""window"": 15, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0009958219592545534, ""batch_size"": 128, ""dropout"": 0.24628443955293938, ""num_filters"": 64, ""kernel_size"": 5}}"
18,16.533758431563378,0.3083734508989981,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0008323832948745205, ""batch_size"": 256, ""dropout"": 0.3210189251182718, ""num_filters"": 32, ""kernel_size"": 5}}"
19,18.022694507497757,0.45425474744550826,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0005068691222400101, ""batch_size"": 128, ""dropout"": 0.2839513273471613, ""num_filters"": 64, ""kernel_size"": 5}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/PyTorch_RNN_LSTM/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,12.3957908042847,0.3679133833505343,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002552951604697378, ""batch_size"": 256, ""dropout"": 0.08493564427131046, ""hidden_dim"": 128, ""num_layers"": 2}}"
1,20.027480848591832,1.2410092215438784,50,"{""representation"": ""global"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0016338208828908817, ""batch_size"": 512, ""dropout"": 0.06820964947491662, ""hidden_dim"": 128, ""num_layers"": 3}}"
2,20.316657674596204,1.0712811644109466,50,"{""representation"": ""global"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002298752892366083, ""batch_size"": 512, ""dropout"": 0.07394178221021082, ""hidden_dim"": 32, ""num_layers"": 3}}"
3,12.768661621494928,0.1908850326309471,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0011369027867815917, ""batch_size"": 256, ""dropout"": 0.32087879230161587, ""hidden_dim"": 64, ""num_layers"": 1}}"
4,12.392893205366391,0.47298144612733795,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002100236158351101, ""batch_size"": 128, ""dropout"": 0.13007332881069883, ""hidden_dim"": 128, ""num_layers"": 2}}"
5,16.048413173276803,0.3789147914511112,20,"{""representation"": ""regime"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0006410050209458448, ""batch_size"": 256, ""dropout"": 0.20342827646588113, ""hidden_dim"": 32, ""num_layers"": 3}}"
6,19.403998891945946,0.4153894324025032,20,"{""representation"": ""global"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.000768316328806251, ""batch_size"": 128, ""dropout"": 0.3584365199693973, ""hidden_dim"": 32, ""num_layers"": 2}}"
7,13.015065159198663,0.47479713136290375,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001052380849166306, ""batch_size"": 256, ""dropout"": 0.3887128330883843, ""hidden_dim"": 32, ""num_layers"": 1}}"
10,12.20347124072149,0.44774911958774755,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001166641358902415, ""batch_size"": 128, ""dropout"": 0.182290204565902, ""hidden_dim"": 128, ""num_layers"": 1}}"
11,12.448100817034257,0.6120056539120718,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0010688731090013406, ""batch_size"": 128, ""dropout"": 0.19650816580044883, ""hidden_dim"": 128, ""num_layers"": 1}}"
13,12.6405997875925,0.585176168774527,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002012710449966973, ""batch_size"": 128, ""dropout"": 0.19889643007254898, ""hidden_dim"": 128, ""num_layers"": 3}}"
18,12.515078178218438,0.47084823307444834,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0006138887739449598, ""batch_size"": 128, ""dropout"": 0.29455393951015585, ""hidden_dim"": 128, ""num_layers"": 1}}"
19,12.365886835812878,0.3806251655043801,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0012996584461888612, ""batch_size"": 256, ""dropout"": 0.27015714376442324, ""hidden_dim"": 128, ""num_layers"": 2}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/PyTorch_Transformer/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,15.468714236425521,0.2460317421899503,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002552951604697378, ""batch_size"": 256, ""dropout"": 0.08493564427131046, ""d_model"": 128, ""nhead"": 2, ""num_layers"": 1, ""dim_feedforward"": 64}}"
1,17.4088777732791,0.7231503262517324,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0005807932994623225, ""batch_size"": 256, ""dropout"": 0.12184550766934828, ""d_model"": 64, ""nhead"": 4, ""num_layers"": 1, ""dim_feedforward"": 64}}"
2,18.17038076663503,0.5240791842874105,20,"{""representation"": ""regime"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.004176805377655879, ""batch_size"": 256, ""dropout"": 0.13013213230530574, ""d_model"": 128, ""nhead"": 2, ""num_layers"": 2, ""dim_feedforward"": 128}}"
3,25.40970906621763,1.0757733154801419,20,"{""representation"": ""global"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0011413943879952562, ""batch_size"": 256, ""dropout"": 0.13235920994105968, ""d_model"": 128, ""nhead"": 2, ""num_layers"": 3, ""dim_feedforward"": 256}}"
4,21.123858792042352,0.5498732881558139,10,"{""representation"": ""global"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0021646137637899576, ""batch_size"": 512, ""dropout"": 0.09971689165954999, ""d_model"": 64, ""nhead"": 4, ""num_layers"": 1, ""dim_feedforward"": 64}}"
5,22.299606476344717,0.6016254343491734,40,"{""representation"": ""global"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0008450942890954814, ""batch_size"": 512, ""dropout"": 0.0027808522124762817, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 256}}"
6,17.40811596677135,0.5405743016343983,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001590922513078389, ""batch_size"": 512, ""dropout"": 0.09582475626678898, ""d_model"": 128, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 128}}"
7,17.210105672340493,0.2790886530942752,20,"{""representation"": ""regime"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002379766606935902, ""batch_size"": 256, ""dropout"": 0.25806911616378, ""d_model"": 64, ""nhead"": 2, ""num_layers"": 2, ""dim_feedforward"": 128}}"
8,12.119102810533754,0.3647377003099629,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0021481839210591726, ""batch_size"": 512, ""dropout"": 0.35884410398103084, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 64}}"
10,13.009515772856153,0.4049275955243518,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0008006892867011675, ""batch_size"": 512, ""dropout"": 0.3778945029673634, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 64}}"
11,12.806594341396947,0.7051966843883618,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0018876017913010321, ""batch_size"": 512, ""dropout"": 0.3227018870635695, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 64}}"
12,15.591803066743664,0.543783088823837,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0035214913703556705, ""batch_size"": 512, ""dropout"": 0.22117625442807667, ""d_model"": 128, ""nhead"": 4, ""num_layers"": 2, ""dim_feedforward"": 64}}"
14,13.767886869510177,0.5656633396392301,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002161223933778057, ""batch_size"": 512, ""dropout"": 0.3719608431766212, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 2, ""dim_feedforward"": 64}}"
15,12.719950949224184,0.5829856859403154,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0025775569875316454, ""batch_size"": 512, ""dropout"": 0.3133874763577134, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 64}}"
17,15.957564511683769,0.24230844869644558,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.004848334879564388, ""batch_size"": 128, ""dropout"": 0.3889659194326925, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 64}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/Random_Forest/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,17.386356182283016,0.27157440781326403,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 50, ""max_depth"": 8, ""min_samples_leaf"": 1, ""max_features"": 0.9063233020424546}}"
1,16.51203197411383,0.36600084075973,10,"{""representation"": ""regime"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 100, ""max_depth"": 17, ""min_samples_leaf"": 2, ""max_features"": 0.5045012539746527}}"
2,17.00645917253555,0.2928059846626988,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 17, ""min_samples_leaf"": 1, ""max_features"": 0.7252813963310067}}"
3,16.81531178331936,0.28745720750544246,10,"{""representation"": ""global"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 50, ""max_depth"": 23, ""min_samples_leaf"": 3, ""max_features"": 0.7637655990477874}}"
4,17.00847939434601,0.3002110687439348,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 300, ""max_depth"": 20, ""min_samples_leaf"": 10, ""max_features"": 0.9263791452993542}}"
5,15.632001601417604,0.29103863086601117,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 200, ""max_depth"": 8, ""min_samples_leaf"": 9, ""max_features"": 0.35218545057583955}}"
6,17.059956248366944,0.3164425928205658,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 250, ""max_depth"": 19, ""min_samples_leaf"": 8, ""max_features"": 0.8398892426801621}}"
7,15.078941845552956,0.261279489143944,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 14, ""min_samples_leaf"": 2, ""max_features"": 0.7992713510560965}}"
9,16.606143952195573,0.3738348767884474,10,"{""representation"": ""regime"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 50, ""max_depth"": 23, ""min_samples_leaf"": 9, ""max_features"": 0.7433826295572964}}"
10,15.250006973492546,0.2641828782125759,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 150, ""max_depth"": 19, ""min_samples_leaf"": 5, ""max_features"": 0.9292785077841608}}"
11,15.222716428790928,0.2568063222172775,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 200, ""max_depth"": 19, ""min_samples_leaf"": 6, ""max_features"": 0.9226072784555683}}"
12,15.341058848140879,0.23344895504987867,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 10, ""min_samples_leaf"": 4, ""max_features"": 0.9208176442899881}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/Ridge/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,20.59138731819258,0.21245124273645813,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 0.06026889128682508}}"
1,21.180179933827954,0.16011862528634593,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 34.70266988650411}}"
2,20.59138474395019,0.21243277035186703,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 0.08111941985431921}}"
3,20.591410739725017,0.2123541834015171,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 0.2858549394196191}}"
4,19.21417910794547,0.188387055152248,15,"{""representation"": ""global"", ""temporal"": true, ""window"": 15, ""hyper"": {""alpha"": 0.0712230583333387}}"
5,20.591416766046585,0.21234637721224087,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 0.3334792728637583}}"
6,20.591474841661668,0.21229316017090197,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 2.991469302130215}}"
7,16.44847387156991,0.16386442605939805,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 297.94544625913585}}"
8,16.354236131721713,0.15827251139991674,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 5.1698794422985355}}"
9,17.148999084803894,0.14865122869316286,30,"{""representation"": ""regime"", ""temporal"": true, ""window"": 30, ""hyper"": {""alpha"": 0.023454342277260523}}"
10,16.736372024259133,0.12891061741305684,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 2.3261440591233864}}"
11,16.411803757215473,0.16260512219720702,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 202.98110988398884}}"
12,16.354314937596442,0.1582459021075926,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 3.1873283397227463}}"
13,15.384170130737761,0.16698050007878684,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 2.677888373979133}}"
14,15.384256302400733,0.16688098574441576,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 1.0181659339606826}}"
15,15.384367115107722,0.16681672004289125,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 0.1749666616740491}}"
17,15.384278120982747,0.16686297777265374,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 0.7364270182642626}}"
19,15.384133041309443,0.16747106568880873,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 10.85063770122587}}"
23,15.384306140385112,0.16684409396329483,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 0.46617947954514244}}"
25,15.384232889506222,0.16690339137243693,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 1.383596708757115}}"
26,15.789129677999815,0.16662283450376386,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""alpha"": 17.525344508769724}}"
27,15.606234629149245,0.14261396465746004,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 0.29769833333382206}}"
28,16.084505938901767,0.12459252596202075,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 12.232140867062428}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/XGBoost/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,17.93414168705831,0.22426521266515725,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 4, ""learning_rate"": 0.023406777359270506, ""subsample"": 0.9330880728874675, ""colsample_bytree"": 0.8005575058716043, ""min_child_weight"": 8, ""reg_lambda"": 0.011527987128232402}}"
1,17.170181583462078,0.3128522143882099,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 150, ""max_depth"": 5, ""learning_rate"": 0.08283072221077194, ""subsample"": 0.7159725093210578, ""colsample_bytree"": 0.645614570099021, ""min_child_weight"": 7, ""reg_lambda"": 0.02621087878265441}}"
2,15.405174303853162,0.21073074870772152,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.17855789220062088, ""subsample"": 0.6523068845866853, ""colsample_bytree"": 0.5488360570031919, ""min_child_weight"": 7, ""reg_lambda"": 0.2091498132903561}}"
3,14.754202695626978,0.23652210507182816,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.2256465100244266, ""subsample"": 0.7989499894055425, ""colsample_bytree"": 0.9609371175115584, ""min_child_weight"": 1, ""reg_lambda"": 0.03872118032174583}}"
4,16.798977602149183,0.3135545016956862,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 500, ""max_depth"": 5, ""learning_rate"": 0.04279909806035244, ""subsample"": 0.7713480415791243, ""colsample_bytree"": 0.5704621124873813, ""min_child_weight"": 9, ""reg_lambda"": 0.01673601016782578}}"
5,17.622056012936277,0.320308259240223,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 500, ""max_depth"": 8, ""learning_rate"": 0.14401540623155276, ""subsample"": 0.8856351733429728, ""colsample_bytree"": 0.5370223258670452, ""min_child_weight"": 4, ""reg_lambda"": 0.022264204303769682}}"
6,17.08480980237851,0.34017645941072167,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 5, ""learning_rate"": 0.14424921039779712, ""subsample"": 0.8187787356776066, ""colsample_bytree"": 0.9436063712881633, ""min_child_weight"": 5, ""reg_lambda"": 0.022844556850020535}}"
7,14.335709705087135,0.1858046959739593,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 7, ""learning_rate"": 0.2335667365205794, ""subsample"": 0.6246461145744375, ""colsample_bytree"": 0.7051914615178149, ""min_child_weight"": 8, ""reg_lambda"": 0.04857295179217168}}"
8,13.954203522738112,0.27199240491400734,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.04731851459137855, ""subsample"": 0.5550259622638384, ""colsample_bytree"": 0.6139675812709708, ""min_child_weight"": 5, ""reg_lambda"": 2.8447512555118193}}"
10,13.859936648376934,0.25655945668451263,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 9, ""learning_rate"": 0.04398724797286349, ""subsample"": 0.5409438634048439, ""colsample_bytree"": 0.6604144336321783, ""min_child_weight"": 9, ""reg_lambda"": 4.985390561409462}}"
11,13.791525871181141,0.2400378860355114,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 350, ""max_depth"": 9, ""learning_rate"": 0.03840501767644911, ""subsample"": 0.5092866199966892, ""colsample_bytree"": 0.5507187964196311, ""min_child_weight"": 6, ""reg_lambda"": 2.008070155745467}}"
12,13.737897511834959,0.25447205377678905,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 8, ""learning_rate"": 0.02814414565669774, ""subsample"": 0.5188071861055975, ""colsample_bytree"": 0.5327816228174422, ""min_child_weight"": 8, ""reg_lambda"": 6.380994123134933}}"
17,13.678001912353631,0.23944451640857672,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 8, ""learning_rate"": 0.02286224551814071, ""subsample"": 0.5183883075398142, ""colsample_bytree"": 0.5866601688192898, ""min_child_weight"": 5, ""reg_lambda"": 2.286173715745685}}"
19,14.385222212762452,0.23399378211804328,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 200, ""max_depth"": 5, ""learning_rate"": 0.0214130241899486, ""subsample"": 0.5834580703340053, ""colsample_bytree"": 0.5744118921463852, ""min_child_weight"": 10, ""reg_lambda"": 3.821399492199396}}"
20,13.752732161267891,0.2593909776704542,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 400, ""max_depth"": 8, ""learning_rate"": 0.022135478180183195, ""subsample"": 0.5471035266052687, ""colsample_bytree"": 0.6842111168163153, ""min_child_weight"": 4, ""reg_lambda"": 8.677327095371378}}"
21,13.764262789275552,0.25609257465909463,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 8, ""learning_rate"": 0.03360781684683225, ""subsample"": 0.6639201745113305, ""colsample_bytree"": 0.6182122502007245, ""min_child_weight"": 4, ""reg_lambda"": 8.14199684147819}}"
22,13.683361091941919,0.2311233207383762,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 6, ""learning_rate"": 0.02108385235540145, ""subsample"": 0.5275230850165414, ""colsample_bytree"": 0.5456781905056647, ""min_child_weight"": 4, ""reg_lambda"": 2.128575364149744}}"
23,13.597648530899042,0.23109153937813476,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 550, ""max_depth"": 6, ""learning_rate"": 0.020788746244781636, ""subsample"": 0.5186924978984819, ""colsample_bytree"": 0.5189260490751036, ""min_child_weight"": 4, ""reg_lambda"": 1.6130500225842452}}"
29,13.566848201707884,0.22250934791107957,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 550, ""max_depth"": 5, ""learning_rate"": 0.033391523568980605, ""subsample"": 0.5038836933262888, ""colsample_bytree"": 0.5337316680678507, ""min_child_weight"": 5, ""reg_lambda"": 3.9903183422513004}}"
30,13.792513234790698,0.25149263216934364,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 9, ""learning_rate"": 0.02104131036617149, ""subsample"": 0.5010376204718198, ""colsample_bytree"": 0.643824002135152, ""min_child_weight"": 3, ""reg_lambda"": 2.190127573340943}}"
31,13.807894419563468,0.19808877355470367,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 550, ""max_depth"": 4, ""learning_rate"": 0.024222806368325255, ""subsample"": 0.5114732276615817, ""colsample_bytree"": 0.5628866876816548, ""min_child_weight"": 2, ""reg_lambda"": 4.66376945258238}}"
32,13.60376753260448,0.24398618022155505,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 550, ""max_depth"": 6, ""learning_rate"": 0.02909407841655949, ""subsample"": 0.5978337286879876, ""colsample_bytree"": 0.5903577771836814, ""min_child_weight"": 2, ""reg_lambda"": 1.8277489125121118}}"
33,13.64141182218963,0.22882047574505845,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 400, ""max_depth"": 6, ""learning_rate"": 0.027425313694110318, ""subsample"": 0.5119887690070071, ""colsample_bytree"": 0.5449387246501276, ""min_child_weight"": 6, ""reg_lambda"": 2.617502828775605}}"
34,13.654645837326928,0.22805924064786343,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 550, ""max_depth"": 5, ""learning_rate"": 0.025021247963809213, ""subsample"": 0.5817674881966203, ""colsample_bytree"": 0.5058949029301583, ""min_child_weight"": 6, ""reg_lambda"": 3.31829578642259}}"
35,13.69462864474319,0.23195378631137836,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 350, ""max_depth"": 6, ""learning_rate"": 0.027773643605909035, ""subsample"": 0.5507186698043883, ""colsample_bytree"": 0.5078271002384165, ""min_child_weight"": 7, ""reg_lambda"": 1.5654245314982598}}"
38,13.181820914355791,0.20815399610746443,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 5, ""learning_rate"": 0.026833747728126217, ""subsample"": 0.5795413799773785, ""colsample_bytree"": 0.6738457647424323, ""min_child_weight"": 3, ""reg_lambda"": 5.028942206006406}}"
39,13.12262981551542,0.2008173540319876,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 5, ""learning_rate"": 0.0276728775045774, ""subsample"": 0.6046102198674776, ""colsample_bytree"": 0.642012420620418, ""min_child_weight"": 3, ""reg_lambda"": 9.191153216123288}}"
40,13.077477128486853,0.21988171891695651,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 6, ""learning_rate"": 0.03619032586705671, ""subsample"": 0.5802870857538972, ""colsample_bytree"": 0.6015059909016953, ""min_child_weight"": 4, ""reg_lambda"": 5.495271078384824}}"
41,13.10364421727957,0.21008577660115454,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 5, ""learning_rate"": 0.04245833441065102, ""subsample"": 0.5644914936825967, ""colsample_bytree"": 0.6270223597242331, ""min_child_weight"": 3, ""reg_lambda"": 5.861171725862357}}"
42,13.091031519488231,0.22005914876239718,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 5, ""learning_rate"": 0.04027514419407641, ""subsample"": 0.5970113523736903, ""colsample_bytree"": 0.6443263766752212, ""min_child_weight"": 4, ""reg_lambda"": 3.8536755266584075}}"
43,13.070974120368613,0.2154402099873336,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 5, ""learning_rate"": 0.04043020414903021, ""subsample"": 0.5652254970413297, ""colsample_bytree"": 0.5986889882149711, ""min_child_weight"": 4, ""reg_lambda"": 4.385841341747118}}"
44,13.094228096532168,0.20495649747837807,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 5, ""learning_rate"": 0.0457865480499457, ""subsample"": 0.5999979973024692, ""colsample_bytree"": 0.5840084348447807, ""min_child_weight"": 2, ""reg_lambda"": 6.150024413180936}}"
45,13.062895555425277,0.2146555991496769,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 6, ""learning_rate"": 0.042953784114641666, ""subsample"": 0.5922429787511403, ""colsample_bytree"": 0.6071610915233936, ""min_child_weight"": 4, ""reg_lambda"": 3.706095749986534}}"
46,13.143117555201394,0.15533250913892774,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 4, ""learning_rate"": 0.04696434772705891, ""subsample"": 0.6174694796162313, ""colsample_bytree"": 0.6658882101420901, ""min_child_weight"": 3, ""reg_lambda"": 7.341798062219051}}"
47,13.083168987675691,0.17073401666599444,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 5, ""learning_rate"": 0.040790371468965844, ""subsample"": 0.6219561400203081, ""colsample_bytree"": 0.5392969789056878, ""min_child_weight"": 1, ""reg_lambda"": 9.180293101263382}}"
48,13.197753261094158,0.18190045647405653,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 4, ""learning_rate"": 0.03224508625989752, ""subsample"": 0.6104908142031442, ""colsample_bytree"": 0.637231164155111, ""min_child_weight"": 2, ""reg_lambda"": 4.64782750884627}}"
49,13.073742258663634,0.18833812956238527,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 5, ""learning_rate"": 0.03874207461971653, ""subsample"": 0.6058184273854479, ""colsample_bytree"": 0.6886735187156418, ""min_child_weight"": 3, ""reg_lambda"": 1.8902064386968447}}"
50,13.147911803284217,0.22395047610907,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 6, ""learning_rate"": 0.06742444527689638, ""subsample"": 0.5497941735673031, ""colsample_bytree"": 0.5653472715323152, ""min_child_weight"": 2, ""reg_lambda"": 6.9653567019109115}}"
51,13.08499039523067,0.2030631368037033,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 5, ""learning_rate"": 0.0567505477050205, ""subsample"": 0.5930438865778904, ""colsample_bytree"": 0.6155476291516064, ""min_child_weight"": 5, ""reg_lambda"": 6.25931652679688}}"
52,13.039928084243394,0.1907529171128372,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 5, ""learning_rate"": 0.047749049282444445, ""subsample"": 0.6573873408139774, ""colsample_bytree"": 0.556030675310799, ""min_child_weight"": 3, ""reg_lambda"": 9.65558114284457}}"
53,13.125952169293324,0.19496994414001573,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 4, ""learning_rate"": 0.052780427335235094, ""subsample"": 0.6158469071473505, ""colsample_bytree"": 0.5804674799522989, ""min_child_weight"": 4, ""reg_lambda"": 7.2871608145342295}}"
54,13.097848786361345,0.20071875236049908,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 5, ""learning_rate"": 0.036839955291058935, ""subsample"": 0.6111329816118422, ""colsample_bytree"": 0.5289650522428816, ""min_child_weight"": 4, ""reg_lambda"": 7.6662000102378585}}"
55,13.092195160857134,0.18512337390502034,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 5, ""learning_rate"": 0.046207314346709626, ""subsample"": 0.5700949816681805, ""colsample_bytree"": 0.5008565482747137, ""min_child_weight"": 1, ""reg_lambda"": 8.991152402989186}}"
56,13.096553134821757,0.2018275709024249,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 5, ""learning_rate"": 0.050299221945356166, ""subsample"": 0.731925293379556, ""colsample_bytree"": 0.5733124261088246, ""min_child_weight"": 2, ""reg_lambda"": 8.78296834124212}}"
57,13.22229756777343,0.16344784651318775,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 4, ""learning_rate"": 0.03413777744039364, ""subsample"": 0.5734968872318263, ""colsample_bytree"": 0.5244303888288554, ""min_child_weight"": 3, ""reg_lambda"": 8.575749474419728}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD002/XGBoost_NASA_Loss/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,27.34863615645787,0.10190153891788072,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 4, ""learning_rate"": 0.023406777359270506, ""subsample"": 0.9330880728874675, ""colsample_bytree"": 0.8005575058716043, ""min_child_weight"": 8, ""reg_lambda"": 0.011527987128232402}}"
1,18.56978994537247,0.323068379845622,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 150, ""max_depth"": 5, ""learning_rate"": 0.08283072221077194, ""subsample"": 0.7159725093210578, ""colsample_bytree"": 0.645614570099021, ""min_child_weight"": 7, ""reg_lambda"": 0.02621087878265441}}"
2,15.242531096087609,0.28814909592347066,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.17855789220062088, ""subsample"": 0.6523068845866853, ""colsample_bytree"": 0.5488360570031919, ""min_child_weight"": 7, ""reg_lambda"": 0.2091498132903561}}"
3,17.720585676524347,0.2183821512718393,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.2256465100244266, ""subsample"": 0.7989499894055425, ""colsample_bytree"": 0.9609371175115584, ""min_child_weight"": 1, ""reg_lambda"": 0.03872118032174583}}"
4,17.52699415337984,0.3716199028544506,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 500, ""max_depth"": 5, ""learning_rate"": 0.04279909806035244, ""subsample"": 0.7713480415791243, ""colsample_bytree"": 0.5704621124873813, ""min_child_weight"": 9, ""reg_lambda"": 0.01673601016782578}}"
5,17.500843865961208,0.3686912803761164,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 500, ""max_depth"": 8, ""learning_rate"": 0.14401540623155276, ""subsample"": 0.8856351733429728, ""colsample_bytree"": 0.5370223258670452, ""min_child_weight"": 4, ""reg_lambda"": 0.022264204303769682}}"
6,17.802389075438054,0.39504995122500597,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 5, ""learning_rate"": 0.14424921039779712, ""subsample"": 0.8187787356776066, ""colsample_bytree"": 0.9436063712881633, ""min_child_weight"": 5, ""reg_lambda"": 0.022844556850020535}}"
7,14.00521833615332,0.18181039554444797,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 7, ""learning_rate"": 0.2335667365205794, ""subsample"": 0.6246461145744375, ""colsample_bytree"": 0.7051914615178149, ""min_child_weight"": 8, ""reg_lambda"": 0.04857295179217168}}"
8,13.551775504255877,0.24899101145279673,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.04731851459137855, ""subsample"": 0.5550259622638384, ""colsample_bytree"": 0.6139675812709708, ""min_child_weight"": 5, ""reg_lambda"": 2.8447512555118193}}"
10,13.598250544828563,0.24501539492483165,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 9, ""learning_rate"": 0.04398724797286349, ""subsample"": 0.5409438634048439, ""colsample_bytree"": 0.6604144336321783, ""min_child_weight"": 9, ""reg_lambda"": 4.985390561409462}}"
11,13.640065363998763,0.23487679151045937,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 350, ""max_depth"": 9, ""learning_rate"": 0.03840501767644911, ""subsample"": 0.5092866199966892, ""colsample_bytree"": 0.5507187964196311, ""min_child_weight"": 6, ""reg_lambda"": 2.008070155745467}}"
12,13.720669939956773,0.27322930731180123,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.06122808916784535, ""subsample"": 0.6126067810214022, ""colsample_bytree"": 0.7851268893802819, ""min_child_weight"": 5, ""reg_lambda"": 2.286496369392377}}"
13,15.87950284888002,0.29989270741928603,20,"{""representation"": ""regime"", ""temporal"": true, ""window"": 20, ""hyper"": {""n_estimators"": 500, ""max_depth"": 9, ""learning_rate"": 0.04347087075670261, ""subsample"": 0.7296842368412391, ""colsample_bytree"": 0.6629999569665247, ""min_child_weight"": 10, ""reg_lambda"": 2.62337454001623}}"
17,13.681587613318635,0.23173303609711304,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 9, ""learning_rate"": 0.07740969471730497, ""subsample"": 0.5760587877881572, ""colsample_bytree"": 0.7258969238182924, ""min_child_weight"": 4, ""reg_lambda"": 6.182459342909662}}"
18,15.071458010182377,0.305598939347674,30,"{""representation"": ""regime"", ""temporal"": true, ""window"": 30, ""hyper"": {""n_estimators"": 550, ""max_depth"": 8, ""learning_rate"": 0.02766139031182706, ""subsample"": 0.6820930280614311, ""colsample_bytree"": 0.6428096845991091, ""min_child_weight"": 5, ""reg_lambda"": 0.5002041176044373}}"
19,13.555527670916941,0.26687045904170126,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 350, ""max_depth"": 9, ""learning_rate"": 0.06108302457946817, ""subsample"": 0.5900817597846215, ""colsample_bytree"": 0.6224812912948691, ""min_child_weight"": 8, ""reg_lambda"": 0.2900748863139612}}"
20,13.757322269507124,0.2481774071624901,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.07923967648513949, ""subsample"": 0.5854864754473794, ""colsample_bytree"": 0.648357337500711, ""min_child_weight"": 8, ""reg_lambda"": 0.06131076402245506}}"
21,13.747867144066072,0.28015555204764614,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 9, ""learning_rate"": 0.06640474912229434, ""subsample"": 0.6116571213567341, ""colsample_bytree"": 0.5979900236155912, ""min_child_weight"": 5, ""reg_lambda"": 0.29252761596629157}}"
22,13.745292356536755,0.25328495740213197,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 250, ""max_depth"": 10, ""learning_rate"": 0.05302908664920666, ""subsample"": 0.511687320027325, ""colsample_bytree"": 0.7482576716900646, ""min_child_weight"": 9, ""reg_lambda"": 1.4946222669071723}}"
23,13.708087883784973,0.2568066353322695,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 350, ""max_depth"": 8, ""learning_rate"": 0.04382649815064422, ""subsample"": 0.5310235633989426, ""colsample_bytree"": 0.658647373806752, ""min_child_weight"": 10, ""reg_lambda"": 1.665467358500522}}"
29,13.602419141506644,0.23862790471588233,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 10, ""learning_rate"": 0.03740461665218151, ""subsample"": 0.608268163731713, ""colsample_bytree"": 0.5574101882539557, ""min_child_weight"": 10, ""reg_lambda"": 5.051509696074745}}"
30,13.622813671533866,0.212725560697515,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.08323197405075952, ""subsample"": 0.5758627369468464, ""colsample_bytree"": 0.5451530175936772, ""min_child_weight"": 8, ""reg_lambda"": 9.309595484956883}}"
32,13.653887811493963,0.21817027274525844,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 9, ""learning_rate"": 0.037243412683267516, ""subsample"": 0.5039809163077191, ""colsample_bytree"": 0.5266302788854977, ""min_child_weight"": 10, ""reg_lambda"": 3.1503029511433804}}"
33,13.565954998704186,0.24649448985968395,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.028430157320096232, ""subsample"": 0.5693384058891671, ""colsample_bytree"": 0.5597290207147014, ""min_child_weight"": 10, ""reg_lambda"": 1.0047238862800703}}"
34,13.522544657090828,0.26819978627513913,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 550, ""max_depth"": 9, ""learning_rate"": 0.03771837307994559, ""subsample"": 0.5588102626522843, ""colsample_bytree"": 0.5961590083620958, ""min_child_weight"": 8, ""reg_lambda"": 0.24339629231078663}}"
35,13.4967170622465,0.2546053691690613,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 9, ""learning_rate"": 0.03724656876489904, ""subsample"": 0.5170098998830152, ""colsample_bytree"": 0.6042913493607911, ""min_child_weight"": 7, ""reg_lambda"": 0.5501997527279122}}"
38,13.07799741897669,0.22455947539844537,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 8, ""learning_rate"": 0.03133363196435988, ""subsample"": 0.5573819041477006, ""colsample_bytree"": 0.6787107054186908, ""min_child_weight"": 8, ""reg_lambda"": 0.8955602177144582}}"
39,13.05306771785549,0.20554762339044766,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 8, ""learning_rate"": 0.052746382704325724, ""subsample"": 0.5276780080908249, ""colsample_bytree"": 0.6376542400279662, ""min_child_weight"": 6, ""reg_lambda"": 0.8441747158624584}}"
40,13.064599970006665,0.1997262673035573,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 9, ""learning_rate"": 0.05082817446370039, ""subsample"": 0.5460897747615269, ""colsample_bytree"": 0.6253625731137291, ""min_child_weight"": 7, ""reg_lambda"": 0.769421356514471}}"
41,13.048003540216891,0.21765167224410747,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 8, ""learning_rate"": 0.040915115602247, ""subsample"": 0.5155268374138617, ""colsample_bytree"": 0.690855339910332, ""min_child_weight"": 8, ""reg_lambda"": 0.9257645133590583}}"
42,13.153165806343347,0.2514393118643855,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 8, ""learning_rate"": 0.053875979548412764, ""subsample"": 0.5594118131894348, ""colsample_bytree"": 0.6337015979827928, ""min_child_weight"": 6, ""reg_lambda"": 0.4021039897175455}}"
43,13.131709932225558,0.20210142510177773,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 8, ""learning_rate"": 0.055511230142947746, ""subsample"": 0.5466241962407866, ""colsample_bytree"": 0.6187302139696141, ""min_child_weight"": 7, ""reg_lambda"": 0.8931632118220209}}"
44,12.99316416486632,0.19193247627287768,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 8, ""learning_rate"": 0.04516577566567439, ""subsample"": 0.5678429030410185, ""colsample_bytree"": 0.5997204423063487, ""min_child_weight"": 8, ""reg_lambda"": 2.6920527081058054}}"
45,13.055655890032972,0.22474954702784702,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 8, ""learning_rate"": 0.03805236487894434, ""subsample"": 0.5766909618161595, ""colsample_bytree"": 0.6778604798266754, ""min_child_weight"": 8, ""reg_lambda"": 0.8321339269373286}}"
46,12.999719546959131,0.20548561207720062,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.03022435556936653, ""subsample"": 0.5139304436969788, ""colsample_bytree"": 0.6367580497966927, ""min_child_weight"": 7, ""reg_lambda"": 0.6636290772337823}}"
47,13.200205769954575,0.22958394912662405,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 8, ""learning_rate"": 0.02995539704261915, ""subsample"": 0.5849744435379053, ""colsample_bytree"": 0.6744973955352793, ""min_child_weight"": 8, ""reg_lambda"": 1.3812015238522444}}"
48,13.050367368228782,0.23762759137285167,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 8, ""learning_rate"": 0.04135171365520103, ""subsample"": 0.5800283518647259, ""colsample_bytree"": 0.6691826593443192, ""min_child_weight"": 8, ""reg_lambda"": 0.5139562156377496}}"
49,13.08214282540824,0.2436106438090404,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 8, ""learning_rate"": 0.039292284268402124, ""subsample"": 0.5551556394433161, ""colsample_bytree"": 0.7341447205509989, ""min_child_weight"": 7, ""reg_lambda"": 0.4948849608560094}}"
50,13.012298964945803,0.18837638808403295,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 8, ""learning_rate"": 0.0535274672725716, ""subsample"": 0.6049546306333146, ""colsample_bytree"": 0.5651915570058312, ""min_child_weight"": 9, ""reg_lambda"": 1.141954850858962}}"
51,13.040460595662967,0.21562676533781236,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 8, ""learning_rate"": 0.03614007749646405, ""subsample"": 0.5649211628573166, ""colsample_bytree"": 0.6533698387239794, ""min_child_weight"": 8, ""reg_lambda"": 2.031129612472812}}"
52,12.989360478759574,0.22137616241704877,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 8, ""learning_rate"": 0.03798768845314897, ""subsample"": 0.5919520858241266, ""colsample_bytree"": 0.5564709544435013, ""min_child_weight"": 9, ""reg_lambda"": 1.396822441773842}}"
53,13.065196650168346,0.19610601061125463,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 7, ""learning_rate"": 0.030991457349377662, ""subsample"": 0.614429446241619, ""colsample_bytree"": 0.5925496317482704, ""min_child_weight"": 8, ""reg_lambda"": 0.6539400375809432}}"
54,13.06603103339774,0.22500402966741923,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 8, ""learning_rate"": 0.03955625509484649, ""subsample"": 0.5942640176238924, ""colsample_bytree"": 0.6617935429979499, ""min_child_weight"": 9, ""reg_lambda"": 1.6465979900566285}}"
55,13.094510956328534,0.1953727713895154,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 7, ""learning_rate"": 0.029384883680078386, ""subsample"": 0.6140611651951198, ""colsample_bytree"": 0.554008464897893, ""min_child_weight"": 9, ""reg_lambda"": 1.5419966830297747}}"
56,13.09044422130743,0.2076059866122671,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 8, ""learning_rate"": 0.07403589203854491, ""subsample"": 0.6217332978563009, ""colsample_bytree"": 0.5628481625139728, ""min_child_weight"": 9, ""reg_lambda"": 2.3640241005803477}}"
57,13.08593997720134,0.18713555098777046,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 8, ""learning_rate"": 0.027269735288209224, ""subsample"": 0.540564755730559, ""colsample_bytree"": 0.5576476746572879, ""min_child_weight"": 9, ""reg_lambda"": 2.738195052115353}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/CatBoost/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,15.092856072928978,0.4401893419668698,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""iterations"": 200, ""depth"": 5, ""learning_rate"": 0.023406777359270506, ""l2_leaf_reg"": 13.39433470675048}}"
1,14.331252936417792,0.36043949992384,10,"{""representation"": ""regime"", ""temporal"": true, ""window"": 10, ""hyper"": {""iterations"": 300, ""depth"": 8, ""learning_rate"": 0.029180149047462245, ""l2_leaf_reg"": 2.399324290681271}}"
2,14.525102431238153,0.3945235617222298,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""iterations"": 500, ""depth"": 8, ""learning_rate"": 0.022680881003614695, ""l2_leaf_reg"": 6.172115948107071}}"
3,14.727068009285478,0.43095139702244234,10,"{""representation"": ""global"", ""temporal"": true, ""window"": 10, ""hyper"": {""iterations"": 100, ""depth"": 10, ""learning_rate"": 0.04030685101059539, ""l2_leaf_reg"": 7.277150634170934}}"
4,15.58625427721977,0.3932122176596633,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""iterations"": 800, ""depth"": 9, ""learning_rate"": 0.2546633740246392, ""l2_leaf_reg"": 14.594768942966839}}"
5,11.32347953038027,0.6755600667103231,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""iterations"": 500, ""depth"": 4, ""learning_rate"": 0.17558477392891916, ""l2_leaf_reg"": 1.2502377950801113}}"
6,15.251293964718878,0.413794378716652,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""iterations"": 700, ""depth"": 8, ""learning_rate"": 0.14401540623155276, ""l2_leaf_reg"": 10.079659367870727}}"
7,10.349834115948497,0.7055385390387926,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""iterations"": 800, ""depth"": 7, ""learning_rate"": 0.027649274676728182, ""l2_leaf_reg"": 8.471354625326555}}"
8,14.510688440474294,0.414903823146582,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""iterations"": 500, ""depth"": 6, ""learning_rate"": 0.02142521597283019, ""l2_leaf_reg"": 1.3815607382950852}}"
10,10.518077251203426,0.7506697068621911,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""iterations"": 800, ""depth"": 8, ""learning_rate"": 0.07287261571692631, ""l2_leaf_reg"": 18.445338876089934}}"
11,10.507938009778128,0.7346571516803341,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""iterations"": 700, ""depth"": 8, ""learning_rate"": 0.05121944772162538, ""l2_leaf_reg"": 17.183790926280032}}"
12,10.56394953275176,0.677128741591316,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""iterations"": 800, ""depth"": 5, ""learning_rate"": 0.08397983881234862, ""l2_leaf_reg"": 8.335462491675733}}"
13,13.765031210282208,0.46792172626736805,15,"{""representation"": ""regime"", ""temporal"": true, ""window"": 15, ""hyper"": {""iterations"": 600, ""depth"": 6, ""learning_rate"": 0.030483785700664352, ""l2_leaf_reg"": 14.748043776866151}}"
14,10.30973032954745,0.7413884832458625,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 8, ""learning_rate"": 0.11105536840421122, ""l2_leaf_reg"": 6.887720850200306}}"
15,10.704159082361738,0.7172011670548418,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 500, ""depth"": 9, ""learning_rate"": 0.23835467617615874, ""l2_leaf_reg"": 10.384787246209134}}"
16,13.622658007583002,0.6040748488290506,20,"{""representation"": ""regime"", ""temporal"": true, ""window"": 20, ""hyper"": {""iterations"": 700, ""depth"": 9, ""learning_rate"": 0.14613100980709867, ""l2_leaf_reg"": 5.44937801760488}}"
17,11.803301663612462,0.7803255366811934,30,"{""representation"": ""regime"", ""temporal"": true, ""window"": 30, ""hyper"": {""iterations"": 600, ""depth"": 7, ""learning_rate"": 0.05529591307302703, ""l2_leaf_reg"": 4.482847313647247}}"
18,10.13352770188123,0.7146655497560724,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 6, ""learning_rate"": 0.1366076873698663, ""l2_leaf_reg"": 10.154674034831803}}"
19,10.331680797763326,0.5956664465813357,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 300, ""depth"": 4, ""learning_rate"": 0.08553139795207124, ""l2_leaf_reg"": 9.081461558834915}}"
20,10.353845463205857,0.7698136469093003,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 500, ""depth"": 6, ""learning_rate"": 0.20263357734454543, ""l2_leaf_reg"": 4.741681290507959}}"
21,10.23026747516603,0.6197653059417604,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 400, ""depth"": 4, ""learning_rate"": 0.14585898254331822, ""l2_leaf_reg"": 10.961419162508166}}"
22,10.105145360876865,0.7471061581208931,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 500, ""depth"": 6, ""learning_rate"": 0.15948757995813617, ""l2_leaf_reg"": 16.63347016290314}}"
23,10.336965580222923,0.6669071729714453,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 400, ""depth"": 4, ""learning_rate"": 0.18555823167578875, ""l2_leaf_reg"": 14.52184941055846}}"
24,10.060727541908935,0.7690541863416293,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 500, ""depth"": 6, ""learning_rate"": 0.10662545254959913, ""l2_leaf_reg"": 14.59766645744141}}"
25,10.055861139441955,0.7469321249517242,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 6, ""learning_rate"": 0.1431818880484429, ""l2_leaf_reg"": 13.387196494129286}}"
29,10.412768352393616,0.6326725147366165,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 5, ""learning_rate"": 0.2567956950786304, ""l2_leaf_reg"": 18.32012920289631}}"
30,10.378300045940387,0.6845303058972306,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 500, ""depth"": 5, ""learning_rate"": 0.24997885305000836, ""l2_leaf_reg"": 8.035166804772048}}"
31,10.19102225108296,0.8039715981823001,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 700, ""depth"": 6, ""learning_rate"": 0.10352290558340622, ""l2_leaf_reg"": 12.309535899467628}}"
33,10.467576505708076,0.819661866628835,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 300, ""depth"": 6, ""learning_rate"": 0.25459108709966194, ""l2_leaf_reg"": 10.308980165477651}}"
34,10.354628260141581,0.6205894145542229,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 300, ""depth"": 5, ""learning_rate"": 0.10479095588630112, ""l2_leaf_reg"": 19.967123465345665}}"
35,10.209712342510361,0.721396899636805,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 6, ""learning_rate"": 0.20692114050136273, ""l2_leaf_reg"": 12.266646185266616}}"
36,10.0660050755632,0.7478180485506809,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 700, ""depth"": 5, ""learning_rate"": 0.14823439030711338, ""l2_leaf_reg"": 8.942848139881972}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/LightGBM/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,16.18994467117219,0.43726975138486035,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 4, ""learning_rate"": 0.023406777359270506, ""num_leaves"": 112, ""subsample"": 0.8005575058716043, ""subsample_freq"": 1, ""colsample_bytree"": 0.8540362888980227, ""min_child_samples"": 6, ""reg_lambda"": 8.123245085588687}}"
1,11.530951352208731,0.7460700468126907,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 250, ""max_depth"": 7, ""learning_rate"": 0.16767510257004176, ""num_leaves"": 37, ""subsample"": 0.7571172192068059, ""subsample_freq"": 1, ""colsample_bytree"": 0.7962072844310213, ""min_child_samples"": 9, ""reg_lambda"": 0.6647135865318028}}"
2,15.867820242040281,0.2988805745674977,10,"{""representation"": ""global"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 50, ""max_depth"": 12, ""learning_rate"": 0.04030685101059539, ""num_leaves"": 89, ""subsample"": 0.6558555380447055, ""subsample_freq"": 1, ""colsample_bytree"": 0.7600340105889054, ""min_child_samples"": 57, ""reg_lambda"": 0.035856126103453984}}"
3,14.806520914110507,0.40078823083759163,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 400, ""max_depth"": 12, ""learning_rate"": 0.025415885998427375, ""num_leaves"": 37, ""subsample"": 0.522613644455269, ""subsample_freq"": 1, ""colsample_bytree"": 0.6626651653816322, ""min_child_samples"": 42, ""reg_lambda"": 0.06516990611177176}}"
4,12.45838214588125,0.6486182546002259,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.14401540623155276, ""num_leaves"": 102, ""subsample"": 0.5370223258670452, ""subsample_freq"": 1, ""colsample_bytree"": 0.6792328642721364, ""min_child_samples"": 16, ""reg_lambda"": 3.8842777547031417}}"
5,12.654701847428523,0.5934896372304067,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""n_estimators"": 500, ""max_depth"": 8, ""learning_rate"": 0.16134583428320007, ""num_leaves"": 70, ""subsample"": 0.7613664146909971, ""subsample_freq"": 1, ""colsample_bytree"": 0.7137705091792748, ""min_child_samples"": 7, ""reg_lambda"": 0.021070472806578238}}"
6,15.428702789522498,0.3555021911780653,10,"{""representation"": ""regime"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 100, ""max_depth"": 12, ""learning_rate"": 0.17842401600508806, ""num_leaves"": 86, ""subsample"": 0.9357302950938589, ""subsample_freq"": 1, ""colsample_bytree"": 0.9018360384495572, ""min_child_samples"": 22, ""reg_lambda"": 4.760767751809501}}"
7,14.578817282174208,0.442311416000583,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 5, ""learning_rate"": 0.06358398347237385, ""num_leaves"": 107, ""subsample"": 0.9303652916281717, ""subsample_freq"": 1, ""colsample_bytree"": 0.5034760652655954, ""min_child_samples"": 54, ""reg_lambda"": 0.1787446325623842}}"
8,11.117924929421154,0.7204829498460855,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 300, ""max_depth"": 6, ""learning_rate"": 0.043254212679768214, ""num_leaves"": 19, ""subsample"": 0.8047821669899484, ""subsample_freq"": 1, ""colsample_bytree"": 0.7513395116144308, ""min_child_samples"": 9, ""reg_lambda"": 0.06853925708853059}}"
10,11.451401046849602,0.7375350110150465,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 200, ""max_depth"": 5, ""learning_rate"": 0.03193723880477954, ""num_leaves"": 36, ""subsample"": 0.994615670525111, ""subsample_freq"": 1, ""colsample_bytree"": 0.7768218093259767, ""min_child_samples"": 80, ""reg_lambda"": 0.011734369616928872}}"
11,11.368823050961714,0.6531130232396789,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 250, ""max_depth"": 5, ""learning_rate"": 0.021876520708903467, ""num_leaves"": 50, ""subsample"": 0.9768639858289091, ""subsample_freq"": 1, ""colsample_bytree"": 0.6243740711709365, ""min_child_samples"": 11, ""reg_lambda"": 0.05030322191799626}}"
13,10.365425086953701,0.75960979362961,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 5, ""learning_rate"": 0.0761033678841525, ""num_leaves"": 84, ""subsample"": 0.9936598359179696, ""subsample_freq"": 1, ""colsample_bytree"": 0.6019911645559328, ""min_child_samples"": 7, ""reg_lambda"": 0.12166559293350182}}"
14,10.470140191860033,0.5446070545716114,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 150, ""max_depth"": 7, ""learning_rate"": 0.09338437293709491, ""num_leaves"": 45, ""subsample"": 0.8944680591986923, ""subsample_freq"": 1, ""colsample_bytree"": 0.5242718135271527, ""min_child_samples"": 11, ""reg_lambda"": 0.1688614893817154}}"
15,10.490377784277326,0.6766090536417386,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 150, ""max_depth"": 5, ""learning_rate"": 0.05393281476623622, ""num_leaves"": 24, ""subsample"": 0.9035627670805333, ""subsample_freq"": 1, ""colsample_bytree"": 0.5984550648488456, ""min_child_samples"": 7, ""reg_lambda"": 0.3275372367613778}}"
16,11.095351901201202,0.7176795927523613,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 6, ""learning_rate"": 0.22367040787307724, ""num_leaves"": 56, ""subsample"": 0.9624310103832964, ""subsample_freq"": 1, ""colsample_bytree"": 0.540071955840121, ""min_child_samples"": 6, ""reg_lambda"": 0.7057332775205089}}"
17,10.41364209120253,0.7136683923652928,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 4, ""learning_rate"": 0.05837329159887649, ""num_leaves"": 66, ""subsample"": 0.9727873936064017, ""subsample_freq"": 1, ""colsample_bytree"": 0.6615000393529473, ""min_child_samples"": 17, ""reg_lambda"": 0.09018707323026974}}"
20,11.081052911009035,0.6809487165040253,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 4, ""learning_rate"": 0.13494050200404345, ""num_leaves"": 106, ""subsample"": 0.9615934420701948, ""subsample_freq"": 1, ""colsample_bytree"": 0.6201552285552261, ""min_child_samples"": 12, ""reg_lambda"": 0.08572154942018465}}"
21,10.578372805395626,0.6580735687429371,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 7, ""learning_rate"": 0.08521860450888653, ""num_leaves"": 82, ""subsample"": 0.9725311295500042, ""subsample_freq"": 1, ""colsample_bytree"": 0.5732245457099036, ""min_child_samples"": 13, ""reg_lambda"": 0.04710630575440008}}"
22,10.412191603691367,0.7312412556829115,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 4, ""learning_rate"": 0.061876018822119815, ""num_leaves"": 77, ""subsample"": 0.9627064489835221, ""subsample_freq"": 1, ""colsample_bytree"": 0.6306123018553558, ""min_child_samples"": 11, ""reg_lambda"": 0.22243711161387714}}"
23,10.902699359347766,0.5807352058055925,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 350, ""max_depth"": 3, ""learning_rate"": 0.05930287245739551, ""num_leaves"": 69, ""subsample"": 0.9473555306020222, ""subsample_freq"": 1, ""colsample_bytree"": 0.6938191254772302, ""min_child_samples"": 26, ""reg_lambda"": 0.11961853662868224}}"
24,10.292969072489381,0.6978306347644314,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 5, ""learning_rate"": 0.06054337897900114, ""num_leaves"": 81, ""subsample"": 0.8999230640941187, ""subsample_freq"": 1, ""colsample_bytree"": 0.6158349448027567, ""min_child_samples"": 10, ""reg_lambda"": 0.05752237245506213}}"
25,10.396113804291339,0.6610385034031533,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 350, ""max_depth"": 4, ""learning_rate"": 0.054122474494808434, ""num_leaves"": 56, ""subsample"": 0.9357051069943976, ""subsample_freq"": 1, ""colsample_bytree"": 0.5765358105787541, ""min_child_samples"": 10, ""reg_lambda"": 0.3153273634547883}}"
26,10.711630532659619,0.6850409917925536,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 350, ""max_depth"": 5, ""learning_rate"": 0.08724162191876421, ""num_leaves"": 88, ""subsample"": 0.9585669471809913, ""subsample_freq"": 1, ""colsample_bytree"": 0.513162093995454, ""min_child_samples"": 6, ""reg_lambda"": 0.39482766598875557}}"
29,10.306125922365561,0.771463125031593,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 4, ""learning_rate"": 0.07474525906198135, ""num_leaves"": 69, ""subsample"": 0.9453393902651135, ""subsample_freq"": 1, ""colsample_bytree"": 0.5889833916447005, ""min_child_samples"": 28, ""reg_lambda"": 0.04336485159230636}}"
30,10.258950283763749,0.6752245299752649,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 5, ""learning_rate"": 0.0609536658816263, ""num_leaves"": 84, ""subsample"": 0.8576491285264121, ""subsample_freq"": 1, ""colsample_bytree"": 0.5183997215501566, ""min_child_samples"": 27, ""reg_lambda"": 0.039642718421540844}}"
31,10.34289488686382,0.6783937106532992,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 5, ""learning_rate"": 0.0839603581067004, ""num_leaves"": 83, ""subsample"": 0.8783642079846903, ""subsample_freq"": 1, ""colsample_bytree"": 0.5209243503099781, ""min_child_samples"": 13, ""reg_lambda"": 0.028756636073512913}}"
32,10.264527939828655,0.7723540480616299,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 5, ""learning_rate"": 0.07217591088597407, ""num_leaves"": 96, ""subsample"": 0.8750685740609248, ""subsample_freq"": 1, ""colsample_bytree"": 0.6531179340990905, ""min_child_samples"": 16, ""reg_lambda"": 0.011703823276833145}}"
33,10.356739050714413,0.6658468588513146,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 6, ""learning_rate"": 0.05989831869991252, ""num_leaves"": 80, ""subsample"": 0.8854165876401475, ""subsample_freq"": 1, ""colsample_bytree"": 0.647744092592952, ""min_child_samples"": 10, ""reg_lambda"": 0.013292892429337707}}"
34,10.456604835738338,0.6808358260836799,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 6, ""learning_rate"": 0.09193157791683096, ""num_leaves"": 104, ""subsample"": 0.8862370370474135, ""subsample_freq"": 1, ""colsample_bytree"": 0.6797778004148408, ""min_child_samples"": 5, ""reg_lambda"": 0.01797256022782164}}"
35,10.468675262014504,0.7626479976667275,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 4, ""learning_rate"": 0.0490925723832116, ""num_leaves"": 102, ""subsample"": 0.9240332056711839, ""subsample_freq"": 1, ""colsample_bytree"": 0.6377741033512445, ""min_child_samples"": 19, ""reg_lambda"": 0.05347907822532715}}"
36,10.722252451834814,0.7295381661517978,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 4, ""learning_rate"": 0.04562002778829824, ""num_leaves"": 91, ""subsample"": 0.9366623194917526, ""subsample_freq"": 1, ""colsample_bytree"": 0.6331457965421897, ""min_child_samples"": 22, ""reg_lambda"": 0.015521627812585013}}"
41,10.286188579270306,0.6956801958807517,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 4, ""learning_rate"": 0.07881474991685607, ""num_leaves"": 80, ""subsample"": 0.8959288179703586, ""subsample_freq"": 1, ""colsample_bytree"": 0.5204576777306239, ""min_child_samples"": 35, ""reg_lambda"": 0.12287496610597676}}"
42,10.232671791011846,0.720318441660364,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 5, ""learning_rate"": 0.04930041042345127, ""num_leaves"": 75, ""subsample"": 0.9244404147726857, ""subsample_freq"": 1, ""colsample_bytree"": 0.5557751559298004, ""min_child_samples"": 19, ""reg_lambda"": 0.06216517923902181}}"
45,10.479988440648343,0.7605547586065564,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 400, ""max_depth"": 4, ""learning_rate"": 0.08634743121141958, ""num_leaves"": 91, ""subsample"": 0.9029054255415505, ""subsample_freq"": 1, ""colsample_bytree"": 0.5875300567659851, ""min_child_samples"": 13, ""reg_lambda"": 0.03419118291174982}}"
48,10.304735086858358,0.7485371513053262,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 5, ""learning_rate"": 0.07435456542809882, ""num_leaves"": 87, ""subsample"": 0.9466058511777815, ""subsample_freq"": 1, ""colsample_bytree"": 0.6189368921247252, ""min_child_samples"": 6, ""reg_lambda"": 0.05220596355759103}}"
49,10.20530499302175,0.6668704825974703,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 5, ""learning_rate"": 0.04792050421752722, ""num_leaves"": 94, ""subsample"": 0.8543757536736967, ""subsample_freq"": 1, ""colsample_bytree"": 0.5073270124019141, ""min_child_samples"": 19, ""reg_lambda"": 0.02727508381532699}}"
51,10.283176322890208,0.7367091779062652,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 5, ""learning_rate"": 0.0702464237913117, ""num_leaves"": 93, ""subsample"": 0.9654992532617688, ""subsample_freq"": 1, ""colsample_bytree"": 0.6213432917428424, ""min_child_samples"": 21, ""reg_lambda"": 0.012330596351598881}}"
52,10.313492842278283,0.7019046464168428,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 400, ""max_depth"": 5, ""learning_rate"": 0.053547453563309914, ""num_leaves"": 83, ""subsample"": 0.8938856717829647, ""subsample_freq"": 1, ""colsample_bytree"": 0.5197954810507827, ""min_child_samples"": 28, ""reg_lambda"": 0.052005118548527116}}"
53,10.297436596263028,0.7359092009409773,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 5, ""learning_rate"": 0.06237976288024185, ""num_leaves"": 80, ""subsample"": 0.9290617140624284, ""subsample_freq"": 1, ""colsample_bytree"": 0.64578350647756, ""min_child_samples"": 18, ""reg_lambda"": 0.010680458029643746}}"
54,10.20943507340452,0.6670030875861599,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 5, ""learning_rate"": 0.03832051393756572, ""num_leaves"": 100, ""subsample"": 0.8718638871470049, ""subsample_freq"": 1, ""colsample_bytree"": 0.5074322974544336, ""min_child_samples"": 20, ""reg_lambda"": 0.0713477247166774}}"
56,10.281515791636842,0.6948082211319703,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 5, ""learning_rate"": 0.047208097607386826, ""num_leaves"": 89, ""subsample"": 0.8619231820813962, ""subsample_freq"": 1, ""colsample_bytree"": 0.5296564307868697, ""min_child_samples"": 36, ""reg_lambda"": 0.04106635143719145}}"
57,10.260472960576639,0.6650210531707142,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 5, ""learning_rate"": 0.03316999183876492, ""num_leaves"": 93, ""subsample"": 0.915010244579714, ""subsample_freq"": 1, ""colsample_bytree"": 0.5163882994254264, ""min_child_samples"": 18, ""reg_lambda"": 0.02750779632129632}}"
59,10.36056416811183,0.6832937821470525,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 4, ""learning_rate"": 0.046804765561792666, ""num_leaves"": 92, ""subsample"": 0.8715049780235088, ""subsample_freq"": 1, ""colsample_bytree"": 0.5147028893964444, ""min_child_samples"": 13, ""reg_lambda"": 0.04498262453176404}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/MLP/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,15.064840480298491,0.5760596862839845,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.002537815508265664, ""learning_rate_init"": 0.0026070247583707684}}"
1,14.633408408925604,0.5070365716943345,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.0005342937261279777, ""learning_rate_init"": 0.0003823475224675188}}"
2,15.285228895482362,0.28725086861136256,15,"{""representation"": ""global"", ""temporal"": true, ""window"": 15, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.01712337597316398, ""learning_rate_init"": 0.0004066563313514797}}"
3,15.065708998886246,0.525717608728103,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.004467752817973905, ""learning_rate_init"": 0.0004201672054372534}}"
4,15.068305920142254,0.31464561576148703,15,"{""representation"": ""regime"", ""temporal"": true, ""window"": 15, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.02065142557895925, ""learning_rate_init"": 0.0005170191786366995}}"
5,15.127329898759466,0.3037561517342043,15,"{""representation"": ""regime"", ""temporal"": true, ""window"": 15, ""hyper"": {""hidden_layer_sizes"": [128, 64], ""alpha"": 2.9072088906598435e-05, ""learning_rate_init"": 0.005323617594751502}}"
6,12.537205129160444,0.39222041801062235,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.0012329098365270509, ""learning_rate_init"": 0.0007162786999897341}}"
7,13.893017887004078,0.46955307052543277,20,"{""representation"": ""regime"", ""temporal"": true, ""window"": 20, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.01707975034295823, ""learning_rate_init"": 0.0018484491720988621}}"
8,14.421622961110543,0.39981309885605937,20,"{""representation"": ""global"", ""temporal"": true, ""window"": 20, ""hyper"": {""hidden_layer_sizes"": [128, 64], ""alpha"": 0.0004673518999562748, ""learning_rate_init"": 0.00027810936979265544}}"
9,14.631934162438318,0.5831396817610248,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.07076022085822008, ""learning_rate_init"": 0.00031883397351001905}}"
10,11.954278080263348,0.4740375164808624,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.00023815281563458162, ""learning_rate_init"": 0.0035932255004427057}}"
11,12.087022341004776,0.47477479818302437,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.000636180392728198, ""learning_rate_init"": 0.003007384759347854}}"
13,12.06237642314144,0.4214555277017256,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 8.629713172686319e-05, ""learning_rate_init"": 0.003520007788628053}}"
15,10.218301354183605,0.5865869799547979,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 1.4783198058349258e-05, ""learning_rate_init"": 0.0027219983013791744}}"
16,10.5255994062935,0.5733623570612177,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.0002546996694879112, ""learning_rate_init"": 0.001294886741392901}}"
17,10.574078604075389,0.40283937396771907,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.00013885914677731406, ""learning_rate_init"": 0.0010953432587944697}}"
18,10.4859814129312,0.5793695424328211,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 3.546113095303058e-05, ""learning_rate_init"": 0.0013940410844441302}}"
19,11.171614273100683,0.38230326047439206,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 5.924616049188254e-05, ""learning_rate_init"": 0.0002573831298068418}}"
20,9.958226036972624,0.6477680317977321,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 3.246482642026976e-05, ""learning_rate_init"": 0.006790946119039668}}"
22,9.965754448648257,0.5950936949122799,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 1.0043249528319008e-05, ""learning_rate_init"": 0.006835652658922207}}"
23,10.12907623870814,0.6535131085443999,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 1.0539779188936704e-05, ""learning_rate_init"": 0.0034925508775115973}}"
24,10.0349484000173,0.6307959142340309,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 1.0232196364899775e-05, ""learning_rate_init"": 0.005767289874824627}}"
25,10.115586815421164,0.5671863158617787,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.00013001672506415344, ""learning_rate_init"": 0.009470548174663893}}"
26,10.092977760120387,0.6815697667403369,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 1.5151961687249916e-05, ""learning_rate_init"": 0.00672922628032311}}"
27,11.213476143554137,0.3360925174940447,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 2.556630069147052e-05, ""learning_rate_init"": 0.009486019680555695}}"
28,9.837639991654722,0.542678759918211,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 6.524771183945021e-05, ""learning_rate_init"": 0.00513490116727958}}"
29,9.85744756022601,0.5736037230661494,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 2.3564659850945195e-05, ""learning_rate_init"": 0.004081960568699253}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/PyTorch_CNN/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,19.655841828892445,2.489048942432557,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002552951604697378, ""batch_size"": 256, ""dropout"": 0.08493564427131046, ""num_filters"": 128, ""kernel_size"": 3}}"
1,18.590751730285376,1.8389687834875448,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001956070814274847, ""batch_size"": 256, ""dropout"": 0.02602063719411181, ""num_filters"": 64, ""kernel_size"": 3}}"
2,17.94057441265572,1.0947449501597153,30,"{""representation"": ""global"", ""window"": 30, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0016559149440361915, ""batch_size"": 512, ""dropout"": 0.31005312934444584, ""num_filters"": 32, ""kernel_size"": 3}}"
3,17.339210591469453,1.1845386111391734,30,"{""representation"": ""global"", ""window"": 30, ""temporal"": false, ""hyper"": {""learning_rate"": 0.000691662498760998, ""batch_size"": 512, ""dropout"": 0.308897907718663, ""num_filters"": 128, ""kernel_size"": 5}}"
4,17.52159334759663,1.132210388817119,20,"{""representation"": ""global"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0010571906813317184, ""batch_size"": 512, ""dropout"": 0.18888597006477972, ""num_filters"": 128, ""kernel_size"": 5}}"
5,19.915408892106772,2.29671534191609,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.004041443189081019, ""batch_size"": 512, ""dropout"": 0.09151926619664899, ""num_filters"": 64, ""kernel_size"": 3}}"
6,18.867163901387826,1.17641652082133,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0006442017924231741, ""batch_size"": 512, ""dropout"": 0.3442922333025374, ""num_filters"": 64, ""kernel_size"": 3}}"
7,18.08236980415586,1.1686984847959274,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0015711531129544835, ""batch_size"": 128, ""dropout"": 0.24382573359195875, ""num_filters"": 32, ""kernel_size"": 3}}"
8,15.660354702345339,1.3981726622009696,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002144252054813578, ""batch_size"": 128, ""dropout"": 0.3341209982356952, ""num_filters"": 32, ""kernel_size"": 5}}"
9,18.97065085619755,1.317066783626711,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001096569987677643, ""batch_size"": 256, ""dropout"": 0.10317665108606225, ""num_filters"": 64, ""kernel_size"": 3}}"
11,18.415979423992173,1.3070558133376622,30,"{""representation"": ""global"", ""window"": 30, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0005912129253279409, ""batch_size"": 512, ""dropout"": 0.31401814134288025, ""num_filters"": 128, ""kernel_size"": 5}}"
12,15.492172580631005,1.2875540379059753,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0007973366438244277, ""batch_size"": 256, ""dropout"": 0.29174697772868086, ""num_filters"": 32, ""kernel_size"": 5}}"
13,16.04192778375428,1.48049669891613,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.004310108042627812, ""batch_size"": 128, ""dropout"": 0.3575312644632453, ""num_filters"": 32, ""kernel_size"": 5}}"
14,15.490131870404518,1.0177720121492042,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0005167046168042454, ""batch_size"": 256, ""dropout"": 0.30122064122675135, ""num_filters"": 32, ""kernel_size"": 5}}"
15,15.152928185191556,1.253717556377698,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0008778987571925256, ""batch_size"": 256, ""dropout"": 0.34768971748476474, ""num_filters"": 32, ""kernel_size"": 5}}"
17,15.496064562553101,1.179200529985725,15,"{""representation"": ""regime"", ""window"": 15, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0005073099600493459, ""batch_size"": 256, ""dropout"": 0.28374521916706846, ""num_filters"": 128, ""kernel_size"": 5}}"
18,16.076291945218827,1.4854002101723027,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0012754650291922631, ""batch_size"": 256, ""dropout"": 0.3932461245297806, ""num_filters"": 32, ""kernel_size"": 3}}"
19,16.610358427170897,0.4920466561845163,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0007126202840041217, ""batch_size"": 256, ""dropout"": 0.1635596368446101, ""num_filters"": 32, ""kernel_size"": 3}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/PyTorch_RNN_LSTM/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,11.770472560658163,0.7267552811671526,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002552951604697378, ""batch_size"": 256, ""dropout"": 0.08493564427131046, ""hidden_dim"": 128, ""num_layers"": 2}}"
1,13.813145670314364,1.590388950077616,50,"{""representation"": ""global"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0016338208828908817, ""batch_size"": 512, ""dropout"": 0.06820964947491662, ""hidden_dim"": 128, ""num_layers"": 3}}"
2,13.204304799358752,0.538017946081976,50,"{""representation"": ""global"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002298752892366083, ""batch_size"": 512, ""dropout"": 0.07394178221021082, ""hidden_dim"": 32, ""num_layers"": 3}}"
3,13.612091749999871,0.722855693155017,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0011369027867815917, ""batch_size"": 256, ""dropout"": 0.32087879230161587, ""hidden_dim"": 64, ""num_layers"": 1}}"
4,10.353504941041672,0.7253904265587476,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002100236158351101, ""batch_size"": 128, ""dropout"": 0.13007332881069883, ""hidden_dim"": 128, ""num_layers"": 2}}"
5,15.577605183745396,1.3199726369538793,20,"{""representation"": ""regime"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0006410050209458448, ""batch_size"": 256, ""dropout"": 0.20342827646588113, ""hidden_dim"": 32, ""num_layers"": 3}}"
6,14.864229005193025,1.343179373673003,20,"{""representation"": ""global"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.000768316328806251, ""batch_size"": 128, ""dropout"": 0.3584365199693973, ""hidden_dim"": 32, ""num_layers"": 2}}"
7,13.54057386596145,0.546946948690159,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001052380849166306, ""batch_size"": 256, ""dropout"": 0.3887128330883843, ""hidden_dim"": 32, ""num_layers"": 1}}"
8,12.964832243841222,0.3592224338854169,40,"{""representation"": ""global"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0015432019542643118, ""batch_size"": 128, ""dropout"": 0.30464784613148704, ""hidden_dim"": 64, ""num_layers"": 2}}"
10,12.061954320624794,0.7906556093178898,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001166641358902415, ""batch_size"": 128, ""dropout"": 0.182290204565902, ""hidden_dim"": 128, ""num_layers"": 1}}"
11,10.2333197689152,0.6661953446761654,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0034158985933588444, ""batch_size"": 128, ""dropout"": 0.16783224686740078, ""hidden_dim"": 128, ""num_layers"": 2}}"
12,11.703451501024531,0.5304952146779447,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.004159471535624509, ""batch_size"": 128, ""dropout"": 0.31279740542841283, ""hidden_dim"": 32, ""num_layers"": 2}}"
13,10.835443877251711,1.2272878973431791,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0028119014015017527, ""batch_size"": 128, ""dropout"": 0.1375070688208361, ""hidden_dim"": 128, ""num_layers"": 2}}"
14,10.289112292101583,0.6978093200867174,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0010104441245108186, ""batch_size"": 256, ""dropout"": 0.021484308953968773, ""hidden_dim"": 128, ""num_layers"": 2}}"
15,13.593014419224836,0.9661558001053968,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0006914678883065354, ""batch_size"": 256, ""dropout"": 0.09837822278841621, ""hidden_dim"": 32, ""num_layers"": 1}}"
16,10.369305258226094,0.6740569180523215,60,"{""representation"": ""global"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0005704572829857772, ""batch_size"": 128, ""dropout"": 0.04077282310190339, ""hidden_dim"": 128, ""num_layers"": 2}}"
17,11.798677809905323,1.105486684681184,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.000993839059547503, ""batch_size"": 256, ""dropout"": 0.09105532573160466, ""hidden_dim"": 128, ""num_layers"": 2}}"
18,11.54327210131728,0.5254694702701306,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0013549722576541325, ""batch_size"": 256, ""dropout"": 0.2723234576409911, ""hidden_dim"": 128, ""num_layers"": 3}}"
19,12.17102409000347,1.2546488784448095,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.00289223321193005, ""batch_size"": 256, ""dropout"": 0.1441864440388086, ""hidden_dim"": 128, ""num_layers"": 1}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/PyTorch_Transformer/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,15.010896348780784,0.866688142140949,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002552951604697378, ""batch_size"": 256, ""dropout"": 0.08493564427131046, ""d_model"": 128, ""nhead"": 2, ""num_layers"": 1, ""dim_feedforward"": 64}}"
1,17.506035745961867,0.735835248636232,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0005807932994623225, ""batch_size"": 256, ""dropout"": 0.12184550766934828, ""d_model"": 64, ""nhead"": 4, ""num_layers"": 1, ""dim_feedforward"": 64}}"
2,19.283187031798857,0.4909946196431899,20,"{""representation"": ""regime"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.004176805377655879, ""batch_size"": 256, ""dropout"": 0.13013213230530574, ""d_model"": 128, ""nhead"": 2, ""num_layers"": 2, ""dim_feedforward"": 128}}"
3,17.1600557315252,0.1754154532562181,20,"{""representation"": ""global"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0011413943879952562, ""batch_size"": 256, ""dropout"": 0.13235920994105968, ""d_model"": 128, ""nhead"": 2, ""num_layers"": 3, ""dim_feedforward"": 256}}"
4,16.452214489199672,0.7313772680921843,10,"{""representation"": ""global"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0021646137637899576, ""batch_size"": 512, ""dropout"": 0.09971689165954999, ""d_model"": 64, ""nhead"": 4, ""num_layers"": 1, ""dim_feedforward"": 64}}"
5,13.91419647400746,1.8985766909776989,40,"{""representation"": ""global"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0008450942890954814, ""batch_size"": 512, ""dropout"": 0.0027808522124762817, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 256}}"
6,17.79647740527659,0.03752055478149735,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001590922513078389, ""batch_size"": 512, ""dropout"": 0.09582475626678898, ""d_model"": 128, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 128}}"
7,17.394927553606554,0.9618350418641898,20,"{""representation"": ""regime"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002379766606935902, ""batch_size"": 256, ""dropout"": 0.25806911616378, ""d_model"": 64, ""nhead"": 2, ""num_layers"": 2, ""dim_feedforward"": 128}}"
8,18.959923849544907,1.579466138365404,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0021481839210591726, ""batch_size"": 512, ""dropout"": 0.35884410398103084, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 64}}"
11,16.85509932490133,3.1298298520495256,40,"{""representation"": ""global"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0020694942592001664, ""batch_size"": 512, ""dropout"": 0.06760461674802887, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 256}}"
12,15.611314198719606,1.696187032517846,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0037930973033826775, ""batch_size"": 256, ""dropout"": 0.03251779390596221, ""d_model"": 128, ""nhead"": 2, ""num_layers"": 1, ""dim_feedforward"": 64}}"
13,14.738564091493268,1.746177721002731,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.000522184905912915, ""batch_size"": 512, ""dropout"": 0.0033992167508426867, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 2, ""dim_feedforward"": 256}}"
16,17.09307965686259,0.9836599826690144,10,"{""representation"": ""global"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0005759172906752807, ""batch_size"": 512, ""dropout"": 0.11798596374349804, ""d_model"": 64, ""nhead"": 4, ""num_layers"": 2, ""dim_feedforward"": 256}}"
18,17.009289001067046,1.838504825396485,40,"{""representation"": ""global"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001036944534491735, ""batch_size"": 512, ""dropout"": 0.03607781615609277, ""d_model"": 32, ""nhead"": 2, ""num_layers"": 2, ""dim_feedforward"": 256}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/Random_Forest/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,15.408960749622612,0.5225899170848811,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 50, ""max_depth"": 8, ""min_samples_leaf"": 1, ""max_features"": 0.9063233020424546}}"
1,15.186724760035569,0.4009577256099587,10,"{""representation"": ""regime"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 100, ""max_depth"": 17, ""min_samples_leaf"": 2, ""max_features"": 0.5045012539746527}}"
2,14.920117292526168,0.5305774747311489,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 17, ""min_samples_leaf"": 1, ""max_features"": 0.7252813963310067}}"
3,15.356122704395649,0.46886519495905726,10,"{""representation"": ""global"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 50, ""max_depth"": 23, ""min_samples_leaf"": 3, ""max_features"": 0.7637655990477874}}"
4,15.006627877676598,0.5107054843026334,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 300, ""max_depth"": 20, ""min_samples_leaf"": 10, ""max_features"": 0.9263791452993542}}"
5,11.981758048742815,0.6224814539622662,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 200, ""max_depth"": 8, ""min_samples_leaf"": 9, ""max_features"": 0.35218545057583955}}"
6,14.937691511257267,0.5164823773865561,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 250, ""max_depth"": 19, ""min_samples_leaf"": 8, ""max_features"": 0.8398892426801621}}"
7,11.238441736959729,0.813854668748258,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 14, ""min_samples_leaf"": 2, ""max_features"": 0.7992713510560965}}"
8,14.826956691488585,0.5565033731656848,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 14, ""min_samples_leaf"": 1, ""max_features"": 0.37552399889531307}}"
10,11.420662913432754,0.8275223712768456,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 150, ""max_depth"": 19, ""min_samples_leaf"": 5, ""max_features"": 0.9292785077841608}}"
11,11.401684541180694,0.8204871256772187,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 200, ""max_depth"": 19, ""min_samples_leaf"": 6, ""max_features"": 0.9226072784555683}}"
12,11.564061267298385,0.8559043203137218,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 10, ""min_samples_leaf"": 4, ""max_features"": 0.9208176442899881}}"
13,14.734961081551777,0.41614085193360606,15,"{""representation"": ""regime"", ""temporal"": true, ""window"": 15, ""hyper"": {""n_estimators"": 300, ""max_depth"": 13, ""min_samples_leaf"": 4, ""max_features"": 0.6634774968148336}}"
14,11.503037701895343,0.8277691203889717,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 8, ""min_samples_leaf"": 6, ""max_features"": 0.8404924147315025}}"
15,12.743848245593028,0.7473775284982822,30,"{""representation"": ""regime"", ""temporal"": true, ""window"": 30, ""hyper"": {""n_estimators"": 250, ""max_depth"": 24, ""min_samples_leaf"": 4, ""max_features"": 0.9600404687687509}}"
16,14.07876485348531,0.5625087474888737,20,"{""representation"": ""regime"", ""temporal"": true, ""window"": 20, ""hyper"": {""n_estimators"": 200, ""max_depth"": 17, ""min_samples_leaf"": 10, ""max_features"": 0.8102255754874232}}"
17,11.1640776705175,0.785770308005109,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 18, ""min_samples_leaf"": 8, ""max_features"": 0.7101580228355772}}"
18,11.14069614180546,0.7973638072058247,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 20, ""min_samples_leaf"": 7, ""max_features"": 0.7140568640490587}}"
19,11.061597148944795,0.7745801089243608,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 250, ""max_depth"": 19, ""min_samples_leaf"": 9, ""max_features"": 0.6219749118017004}}"
21,11.080871379069313,0.7772907868829798,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 16, ""min_samples_leaf"": 9, ""max_features"": 0.6271513854058853}}"
22,11.097632328044496,0.782546896620065,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 200, ""max_depth"": 17, ""min_samples_leaf"": 10, ""max_features"": 0.6108165294271596}}"
23,11.011935717512856,0.7609320751745875,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 200, ""max_depth"": 17, ""min_samples_leaf"": 10, ""max_features"": 0.5193823482565607}}"
25,11.504086431898754,0.76346300512942,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 300, ""max_depth"": 19, ""min_samples_leaf"": 9, ""max_features"": 0.5091305840027571}}"
26,10.744404721060723,0.7594310735531161,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 14, ""min_samples_leaf"": 8, ""max_features"": 0.5454059709505314}}"
27,10.752028857031796,0.7490496935028029,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 12, ""min_samples_leaf"": 8, ""max_features"": 0.5401583353444576}}"
28,10.700442341289946,0.7509046990262116,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 16, ""min_samples_leaf"": 9, ""max_features"": 0.4151076436268366}}"
29,10.789854069913428,0.7707752717046946,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 13, ""min_samples_leaf"": 7, ""max_features"": 0.5767747111139642}}"
30,10.666978553571838,0.7004870149543374,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 13, ""min_samples_leaf"": 8, ""max_features"": 0.389588820567743}}"
31,10.735233756158147,0.755311151141885,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 12, ""min_samples_leaf"": 9, ""max_features"": 0.4889766428646118}}"
32,10.742530086700365,0.731245013170397,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 12, ""min_samples_leaf"": 9, ""max_features"": 0.4524763079327579}}"
33,10.67907926994508,0.6982942922831233,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 250, ""max_depth"": 12, ""min_samples_leaf"": 8, ""max_features"": 0.35396733326410657}}"
34,10.749123084877429,0.7138541684715677,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 11, ""min_samples_leaf"": 7, ""max_features"": 0.4147867267687031}}"
35,10.7185152727678,0.700361827022972,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 300, ""max_depth"": 16, ""min_samples_leaf"": 8, ""max_features"": 0.3787121327023321}}"
36,10.688979849368167,0.7318538702525622,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 13, ""min_samples_leaf"": 8, ""max_features"": 0.38995949106427064}}"
37,10.727346017345956,0.7302565237084558,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 13, ""min_samples_leaf"": 10, ""max_features"": 0.3991520459913695}}"
39,10.697367779705194,0.6848602831829994,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 16, ""min_samples_leaf"": 10, ""max_features"": 0.3147960085943589}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/Ridge/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,19.564415905881436,0.3862233342240256,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 0.06026889128682508}}"
1,19.56291386276575,0.38504502478803126,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 34.70266988650411}}"
2,19.564414933556378,0.38622261518712775,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 0.08111941985431921}}"
3,19.564405391130492,0.38621555545680686,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 0.2858549394196191}}"
4,17.769076653858583,0.3291301730212179,15,"{""representation"": ""global"", ""temporal"": true, ""window"": 15, ""hyper"": {""alpha"": 0.0712230583333387}}"
5,19.56440317273626,0.3862139134309796,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 0.3334792728637583}}"
6,19.564280132282484,0.38612236843460285,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 2.991469302130215}}"
7,15.649697120699205,0.32415411655895515,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 297.94544625913585}}"
8,15.25898155779239,0.3102980522474961,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 5.1698794422985355}}"
9,15.986892486992312,0.3136162637983673,30,"{""representation"": ""regime"", ""temporal"": true, ""window"": 30, ""hyper"": {""alpha"": 0.023454342277260523}}"
10,15.261618591699166,0.31098086109246853,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 2.3261440591233864}}"
11,15.26166361376553,0.31099089719609624,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 2.295572830474728}}"
12,15.260117350794255,0.3106221068309525,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 3.6252499263843423}}"
13,14.405307693979768,0.3492757904503293,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 2.7432006787044596}}"
14,14.4125151594436,0.3513346978058546,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 0.5882561226051819}}"
15,14.413031722372557,0.3514720091845039,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 0.46427932900777913}}"
17,14.413364866998691,0.3515599426352813,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 0.38607358785868734}}"
21,14.41380549535393,0.3516755123115157,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 0.2846827183520669}}"
22,14.391146442404064,0.3429014681894168,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 14.765476615095691}}"
23,14.797960739331405,0.33421125702644916,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""alpha"": 53.861200162061195}}"
24,14.390807249448432,0.34228395266831163,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 16.63820832872317}}"
26,14.39191909748082,0.34038298099646774,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 23.857317635851768}}"
28,14.555909597491771,0.33439505421445137,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 160.17509344014138}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/XGBoost/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,16.133934332886554,0.42945630100445953,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 4, ""learning_rate"": 0.023406777359270506, ""subsample"": 0.9330880728874675, ""colsample_bytree"": 0.8005575058716043, ""min_child_weight"": 8, ""reg_lambda"": 0.011527987128232402}}"
1,14.662753098397213,0.3950587840115743,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 150, ""max_depth"": 5, ""learning_rate"": 0.08283072221077194, ""subsample"": 0.7159725093210578, ""colsample_bytree"": 0.645614570099021, ""min_child_weight"": 7, ""reg_lambda"": 0.02621087878265441}}"
2,11.917655670051875,0.6405061895751778,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.17855789220062088, ""subsample"": 0.6523068845866853, ""colsample_bytree"": 0.5488360570031919, ""min_child_weight"": 7, ""reg_lambda"": 0.2091498132903561}}"
3,11.37207347386565,0.7800027348758488,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.2256465100244266, ""subsample"": 0.7989499894055425, ""colsample_bytree"": 0.9609371175115584, ""min_child_weight"": 1, ""reg_lambda"": 0.03872118032174583}}"
4,14.683039847315317,0.4178604091547131,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 500, ""max_depth"": 5, ""learning_rate"": 0.04279909806035244, ""subsample"": 0.7713480415791243, ""colsample_bytree"": 0.5704621124873813, ""min_child_weight"": 9, ""reg_lambda"": 0.01673601016782578}}"
5,15.496372737746452,0.47876823621067555,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 500, ""max_depth"": 8, ""learning_rate"": 0.14401540623155276, ""subsample"": 0.8856351733429728, ""colsample_bytree"": 0.5370223258670452, ""min_child_weight"": 4, ""reg_lambda"": 0.022264204303769682}}"
6,15.1072397476245,0.42065883073279225,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 5, ""learning_rate"": 0.14424921039779712, ""subsample"": 0.8187787356776066, ""colsample_bytree"": 0.9436063712881633, ""min_child_weight"": 5, ""reg_lambda"": 0.022844556850020535}}"
7,10.90046557480321,0.8035152595405546,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 7, ""learning_rate"": 0.2335667365205794, ""subsample"": 0.6246461145744375, ""colsample_bytree"": 0.7051914615178149, ""min_child_weight"": 8, ""reg_lambda"": 0.04857295179217168}}"
8,10.731998018644267,0.716232742978126,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.04731851459137855, ""subsample"": 0.5550259622638384, ""colsample_bytree"": 0.6139675812709708, ""min_child_weight"": 5, ""reg_lambda"": 2.8447512555118193}}"
10,10.699592870644102,0.7223739109867993,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 9, ""learning_rate"": 0.04398724797286349, ""subsample"": 0.5409438634048439, ""colsample_bytree"": 0.6604144336321783, ""min_child_weight"": 9, ""reg_lambda"": 4.985390561409462}}"
11,10.717042832247465,0.6573032122228826,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 350, ""max_depth"": 9, ""learning_rate"": 0.03840501767644911, ""subsample"": 0.5092866199966892, ""colsample_bytree"": 0.5507187964196311, ""min_child_weight"": 6, ""reg_lambda"": 2.008070155745467}}"
12,10.653687599308528,0.6665841983648125,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 8, ""learning_rate"": 0.02814414565669774, ""subsample"": 0.5188071861055975, ""colsample_bytree"": 0.5327816228174422, ""min_child_weight"": 8, ""reg_lambda"": 6.380994123134933}}"
17,10.693172983092643,0.7096409934591589,50,"{""representation"": ""global"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 8, ""learning_rate"": 0.029904959133154822, ""subsample"": 0.6296632745480532, ""colsample_bytree"": 0.6562002260198141, ""min_child_weight"": 8, ""reg_lambda"": 9.454001686241387}}"
19,10.68835555435611,0.7137145527615766,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 250, ""max_depth"": 8, ""learning_rate"": 0.04644248076544047, ""subsample"": 0.638986369765669, ""colsample_bytree"": 0.6196566139087136, ""min_child_weight"": 8, ""reg_lambda"": 0.44892631618981405}}"
20,10.723225237339722,0.7325529186156121,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 350, ""max_depth"": 9, ""learning_rate"": 0.05362301243525418, ""subsample"": 0.5713084593942664, ""colsample_bytree"": 0.6026190511413777, ""min_child_weight"": 9, ""reg_lambda"": 0.05962476691316849}}"
21,10.673268605022447,0.7407840479065996,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 7, ""learning_rate"": 0.03448366354868593, ""subsample"": 0.5340190332385919, ""colsample_bytree"": 0.667621950305763, ""min_child_weight"": 7, ""reg_lambda"": 4.788825527328414}}"
22,10.66391802118326,0.7407548708461233,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 7, ""learning_rate"": 0.054327601913726584, ""subsample"": 0.6367200362100086, ""colsample_bytree"": 0.5965146403438798, ""min_child_weight"": 7, ""reg_lambda"": 1.4099325581447084}}"
23,10.723812786094802,0.7251093513662393,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 200, ""max_depth"": 6, ""learning_rate"": 0.03656866459898126, ""subsample"": 0.529984925475567, ""colsample_bytree"": 0.6648359360003689, ""min_child_weight"": 9, ""reg_lambda"": 1.6236577762850954}}"
29,10.748342242110658,0.7029910495677518,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 200, ""max_depth"": 5, ""learning_rate"": 0.0400171671047017, ""subsample"": 0.5153262042664448, ""colsample_bytree"": 0.6381756183122212, ""min_child_weight"": 8, ""reg_lambda"": 7.014258116617756}}"
31,10.694644097398722,0.7112889109027805,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 7, ""learning_rate"": 0.05425170576656858, ""subsample"": 0.5457615398121933, ""colsample_bytree"": 0.5495731630595446, ""min_child_weight"": 8, ""reg_lambda"": 0.9309655936540289}}"
33,10.660461793674125,0.6788764102479661,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 8, ""learning_rate"": 0.02703351191428962, ""subsample"": 0.5587995624833374, ""colsample_bytree"": 0.5412386981130451, ""min_child_weight"": 7, ""reg_lambda"": 0.8004261122176908}}"
34,10.632321180402133,0.7095420893804237,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 400, ""max_depth"": 8, ""learning_rate"": 0.03673505274300616, ""subsample"": 0.5615613103342025, ""colsample_bytree"": 0.5431409289042077, ""min_child_weight"": 9, ""reg_lambda"": 8.38319925180665}}"
35,10.563874236196057,0.6681012784459822,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 350, ""max_depth"": 8, ""learning_rate"": 0.03324142315048949, ""subsample"": 0.573371796425066, ""colsample_bytree"": 0.5076674195028477, ""min_child_weight"": 9, ""reg_lambda"": 3.303855397785506}}"
38,10.194518506592654,0.7186457812251523,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 7, ""learning_rate"": 0.030760708842692778, ""subsample"": 0.558718697125834, ""colsample_bytree"": 0.6377149521057366, ""min_child_weight"": 9, ""reg_lambda"": 8.113606383504743}}"
39,10.20451765978422,0.7432725180499817,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 400, ""max_depth"": 6, ""learning_rate"": 0.03462334630216477, ""subsample"": 0.5194546492251816, ""colsample_bytree"": 0.5820492090968202, ""min_child_weight"": 9, ""reg_lambda"": 1.5631993508777264}}"
40,10.286832221504007,0.717333044239514,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 8, ""learning_rate"": 0.03982488914580685, ""subsample"": 0.5683058553722705, ""colsample_bytree"": 0.572788503994168, ""min_child_weight"": 9, ""reg_lambda"": 6.409457347939533}}"
41,10.292072713742414,0.7332975603587996,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 350, ""max_depth"": 8, ""learning_rate"": 0.045717617368024356, ""subsample"": 0.5487910889616219, ""colsample_bytree"": 0.5707881663597768, ""min_child_weight"": 10, ""reg_lambda"": 4.555064445975551}}"
42,10.212749569938847,0.7010211041201113,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 7, ""learning_rate"": 0.030105568611956224, ""subsample"": 0.5020556415297969, ""colsample_bytree"": 0.5680654248359672, ""min_child_weight"": 9, ""reg_lambda"": 1.2718258590213138}}"
43,10.24150047102718,0.6879668277876009,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 7, ""learning_rate"": 0.044028004973126006, ""subsample"": 0.5580009793195527, ""colsample_bytree"": 0.5776291012284004, ""min_child_weight"": 9, ""reg_lambda"": 4.731570195818012}}"
44,10.174774653538433,0.7549911115209476,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 400, ""max_depth"": 6, ""learning_rate"": 0.041204125728759804, ""subsample"": 0.5074263485738861, ""colsample_bytree"": 0.5894787026337184, ""min_child_weight"": 10, ""reg_lambda"": 0.9179286276725133}}"
45,10.1677354874825,0.7257225426429935,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 400, ""max_depth"": 7, ""learning_rate"": 0.025768745726190596, ""subsample"": 0.5234782631430063, ""colsample_bytree"": 0.5852448240801282, ""min_child_weight"": 9, ""reg_lambda"": 2.688433695313546}}"
46,10.194872039819648,0.7165874110771593,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 7, ""learning_rate"": 0.03527536250929112, ""subsample"": 0.5811651916563213, ""colsample_bytree"": 0.5838317989124351, ""min_child_weight"": 9, ""reg_lambda"": 2.3025327119136563}}"
47,10.208596552747254,0.6411175563060264,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 400, ""max_depth"": 6, ""learning_rate"": 0.04326359782782875, ""subsample"": 0.5142402840418298, ""colsample_bytree"": 0.5418105479981417, ""min_child_weight"": 9, ""reg_lambda"": 1.4848306630421597}}"
48,10.1292872427973,0.7325341227359758,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 400, ""max_depth"": 5, ""learning_rate"": 0.03455387053874626, ""subsample"": 0.5616996320230385, ""colsample_bytree"": 0.6241368473073444, ""min_child_weight"": 9, ""reg_lambda"": 1.474847015639912}}"
49,10.234175490054596,0.715095700935588,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 7, ""learning_rate"": 0.02872366144366409, ""subsample"": 0.58958927746521, ""colsample_bytree"": 0.6293626164895553, ""min_child_weight"": 9, ""reg_lambda"": 6.0632159572400885}}"
50,10.153263772859608,0.7251905368263079,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 400, ""max_depth"": 6, ""learning_rate"": 0.029242938226427578, ""subsample"": 0.5471407197834982, ""colsample_bytree"": 0.6294036095210158, ""min_child_weight"": 10, ""reg_lambda"": 1.7084270597022886}}"
51,10.232628559638027,0.7331829517816176,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 5, ""learning_rate"": 0.02068189291962307, ""subsample"": 0.5292283587880318, ""colsample_bytree"": 0.6661102059482499, ""min_child_weight"": 10, ""reg_lambda"": 2.1281151852352145}}"
52,10.240872816938097,0.6976338135841247,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 350, ""max_depth"": 6, ""learning_rate"": 0.030323979050422934, ""subsample"": 0.5732904828985872, ""colsample_bytree"": 0.5577007145816678, ""min_child_weight"": 10, ""reg_lambda"": 4.0551126694934965}}"
53,10.081911710451859,0.7444497205841248,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 400, ""max_depth"": 5, ""learning_rate"": 0.03377673182995451, ""subsample"": 0.5342040835005609, ""colsample_bytree"": 0.642234022015107, ""min_child_weight"": 10, ""reg_lambda"": 2.7305054574779684}}"
54,10.26156370993404,0.7600295266316625,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 400, ""max_depth"": 6, ""learning_rate"": 0.04200915173358036, ""subsample"": 0.5471752536977288, ""colsample_bytree"": 0.6831791556553911, ""min_child_weight"": 9, ""reg_lambda"": 2.1582841263857073}}"
55,10.264649241467897,0.7408060664877674,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 400, ""max_depth"": 5, ""learning_rate"": 0.026865835858562743, ""subsample"": 0.5714607791768618, ""colsample_bytree"": 0.6224663340880203, ""min_child_weight"": 10, ""reg_lambda"": 2.6386196125920187}}"
56,10.348752422009259,0.7320812998285857,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 350, ""max_depth"": 5, ""learning_rate"": 0.023712825061366155, ""subsample"": 0.5929414375830202, ""colsample_bytree"": 0.6138530563189942, ""min_child_weight"": 9, ""reg_lambda"": 0.9079874246521865}}"
57,10.323877641570725,0.7056105058495432,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 400, ""max_depth"": 5, ""learning_rate"": 0.021143092816531348, ""subsample"": 0.5203267438520909, ""colsample_bytree"": 0.6036517514450279, ""min_child_weight"": 9, ""reg_lambda"": 3.4586317952608843}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD003/XGBoost_NASA_Loss/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,26.710724085195466,0.2274695258065341,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 4, ""learning_rate"": 0.023406777359270506, ""subsample"": 0.9330880728874675, ""colsample_bytree"": 0.8005575058716043, ""min_child_weight"": 8, ""reg_lambda"": 0.011527987128232402}}"
1,15.198993151541487,0.5434912269122545,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 150, ""max_depth"": 5, ""learning_rate"": 0.08283072221077194, ""subsample"": 0.7159725093210578, ""colsample_bytree"": 0.645614570099021, ""min_child_weight"": 7, ""reg_lambda"": 0.02621087878265441}}"
2,11.835407162251395,0.7541802967756356,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.17855789220062088, ""subsample"": 0.6523068845866853, ""colsample_bytree"": 0.5488360570031919, ""min_child_weight"": 7, ""reg_lambda"": 0.2091498132903561}}"
3,16.002126250949193,0.5853797778070866,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.2256465100244266, ""subsample"": 0.7989499894055425, ""colsample_bytree"": 0.9609371175115584, ""min_child_weight"": 1, ""reg_lambda"": 0.03872118032174583}}"
4,15.049253724561478,0.5214084674134105,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 500, ""max_depth"": 5, ""learning_rate"": 0.04279909806035244, ""subsample"": 0.7713480415791243, ""colsample_bytree"": 0.5704621124873813, ""min_child_weight"": 9, ""reg_lambda"": 0.01673601016782578}}"
5,15.660239987719944,0.4985609671356801,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 500, ""max_depth"": 8, ""learning_rate"": 0.14401540623155276, ""subsample"": 0.8856351733429728, ""colsample_bytree"": 0.5370223258670452, ""min_child_weight"": 4, ""reg_lambda"": 0.022264204303769682}}"
6,15.260269906700637,0.5904425219677039,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 5, ""learning_rate"": 0.14424921039779712, ""subsample"": 0.8187787356776066, ""colsample_bytree"": 0.9436063712881633, ""min_child_weight"": 5, ""reg_lambda"": 0.022844556850020535}}"
7,11.025750340925597,0.7188360337977237,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 7, ""learning_rate"": 0.2335667365205794, ""subsample"": 0.6246461145744375, ""colsample_bytree"": 0.7051914615178149, ""min_child_weight"": 8, ""reg_lambda"": 0.04857295179217168}}"
8,10.649235331681751,0.8812705403883345,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.04731851459137855, ""subsample"": 0.5550259622638384, ""colsample_bytree"": 0.6139675812709708, ""min_child_weight"": 5, ""reg_lambda"": 2.8447512555118193}}"
10,10.778752096344354,0.803027023424025,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 9, ""learning_rate"": 0.04398724797286349, ""subsample"": 0.5409438634048439, ""colsample_bytree"": 0.6604144336321783, ""min_child_weight"": 9, ""reg_lambda"": 4.985390561409462}}"
11,10.737662013802545,0.8323108221927975,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 350, ""max_depth"": 9, ""learning_rate"": 0.03840501767644911, ""subsample"": 0.5092866199966892, ""colsample_bytree"": 0.5507187964196311, ""min_child_weight"": 6, ""reg_lambda"": 2.008070155745467}}"
12,10.758859378059933,0.9145602496559978,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 400, ""max_depth"": 10, ""learning_rate"": 0.03529767541238648, ""subsample"": 0.5336416084668992, ""colsample_bytree"": 0.6070610427030017, ""min_child_weight"": 3, ""reg_lambda"": 3.5945922316651338}}"
13,13.282309206211242,0.6758049524042125,20,"{""representation"": ""regime"", ""temporal"": true, ""window"": 20, ""hyper"": {""n_estimators"": 200, ""max_depth"": 8, ""learning_rate"": 0.05162627515771169, ""subsample"": 0.5148462288738874, ""colsample_bytree"": 0.7488248062015586, ""min_child_weight"": 6, ""reg_lambda"": 0.9471012528603403}}"
16,11.974110378116935,0.8891301717588879,30,"{""representation"": ""regime"", ""temporal"": true, ""window"": 30, ""hyper"": {""n_estimators"": 300, ""max_depth"": 9, ""learning_rate"": 0.05622201509767075, ""subsample"": 0.7350284331870716, ""colsample_bytree"": 0.677591115586103, ""min_child_weight"": 8, ""reg_lambda"": 1.7102928943340545}}"
17,10.803443888544354,0.8892490088385634,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 8, ""learning_rate"": 0.06586079408920886, ""subsample"": 0.6504015815281086, ""colsample_bytree"": 0.7262304639754923, ""min_child_weight"": 4, ""reg_lambda"": 3.999123408683371}}"
18,10.744217866838031,0.8616815734954064,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 7, ""learning_rate"": 0.04484191256570486, ""subsample"": 0.564580736941287, ""colsample_bytree"": 0.6284575620137264, ""min_child_weight"": 4, ""reg_lambda"": 9.214025441325587}}"
19,10.807743743879632,0.8565008260505347,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 8, ""learning_rate"": 0.02112960392205049, ""subsample"": 0.5483384261671863, ""colsample_bytree"": 0.6465909340714668, ""min_child_weight"": 6, ""reg_lambda"": 0.46167433617984416}}"
21,10.667793379202008,0.8634296277713333,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 400, ""max_depth"": 10, ""learning_rate"": 0.04472756870229517, ""subsample"": 0.5806560824488066, ""colsample_bytree"": 0.6311263487918487, ""min_child_weight"": 7, ""reg_lambda"": 0.7292621940233571}}"
22,10.667425633276682,0.8601105305894603,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 9, ""learning_rate"": 0.0550805497210929, ""subsample"": 0.5635267392834582, ""colsample_bytree"": 0.7216545756591068, ""min_child_weight"": 6, ""reg_lambda"": 0.4902520987132648}}"
23,10.655346493716685,0.8774591782499729,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 350, ""max_depth"": 10, ""learning_rate"": 0.04168796324752372, ""subsample"": 0.6248531022069264, ""colsample_bytree"": 0.6378712542888357, ""min_child_weight"": 7, ""reg_lambda"": 0.5260653788996238}}"
25,10.971979716222943,0.8435118844719315,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 550, ""max_depth"": 8, ""learning_rate"": 0.05595699958005902, ""subsample"": 0.5406327017276852, ""colsample_bytree"": 0.5543343713914236, ""min_child_weight"": 8, ""reg_lambda"": 0.5012571977568856}}"
28,10.683877529535973,0.8533135938476766,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 9, ""learning_rate"": 0.036835093344955584, ""subsample"": 0.6317892951648043, ""colsample_bytree"": 0.5836999911961732, ""min_child_weight"": 8, ""reg_lambda"": 1.4449459432840763}}"
29,10.761845399027525,0.7829147673399249,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 7, ""learning_rate"": 0.08205074971470414, ""subsample"": 0.5283973392483321, ""colsample_bytree"": 0.6589521385778313, ""min_child_weight"": 7, ""reg_lambda"": 2.584711630537142}}"
30,10.613009011291927,0.8561125843682129,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 9, ""learning_rate"": 0.04524827998254252, ""subsample"": 0.5078968947144424, ""colsample_bytree"": 0.6279181339316656, ""min_child_weight"": 5, ""reg_lambda"": 0.28694122541805583}}"
31,10.641709099941181,0.7589818082310289,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.06586528980778056, ""subsample"": 0.5683593928161095, ""colsample_bytree"": 0.5500042135404818, ""min_child_weight"": 6, ""reg_lambda"": 0.6020387497157886}}"
32,10.534062972833492,0.8813820202389088,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.035462147377383554, ""subsample"": 0.5271576579555473, ""colsample_bytree"": 0.6119213429195007, ""min_child_weight"": 5, ""reg_lambda"": 0.474977411733127}}"
33,10.67264587536705,0.7985285097156068,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.031553063457050855, ""subsample"": 0.5037034380047961, ""colsample_bytree"": 0.5363487012002138, ""min_child_weight"": 5, ""reg_lambda"": 7.607014241153137}}"
34,10.712234163547127,0.8192143714138297,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.07048510615731526, ""subsample"": 0.5757269581698314, ""colsample_bytree"": 0.6173262365116307, ""min_child_weight"": 7, ""reg_lambda"": 0.7430908140559105}}"
38,10.277183479532983,0.8542378998127188,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.02356306861371202, ""subsample"": 0.5031923926737534, ""colsample_bytree"": 0.7004197356950377, ""min_child_weight"": 4, ""reg_lambda"": 0.30485526871628077}}"
39,10.341872757739088,0.8349987323468412,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.020237089504670375, ""subsample"": 0.5828191480436606, ""colsample_bytree"": 0.7420978866217586, ""min_child_weight"": 6, ""reg_lambda"": 0.3442759898896272}}"
40,10.242971105824292,0.8198842431747763,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 9, ""learning_rate"": 0.02756528404529897, ""subsample"": 0.5035902153626584, ""colsample_bytree"": 0.7017668349975393, ""min_child_weight"": 4, ""reg_lambda"": 0.42547334518781543}}"
41,10.315112895097782,0.8415986101035132,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.029209193125154385, ""subsample"": 0.5338417749164757, ""colsample_bytree"": 0.7439065258690327, ""min_child_weight"": 4, ""reg_lambda"": 0.45481032091774354}}"
42,10.25382218754125,0.8588224317280809,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.024912894349961294, ""subsample"": 0.5325233827423848, ""colsample_bytree"": 0.7163856350282906, ""min_child_weight"": 4, ""reg_lambda"": 0.19037579536521418}}"
43,10.217132714401327,0.8487986166623901,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.03399986681918635, ""subsample"": 0.5731702530955729, ""colsample_bytree"": 0.6823245730037972, ""min_child_weight"": 5, ""reg_lambda"": 0.5393070693407902}}"
44,10.260152662839932,0.7995468091202788,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.026540295575110923, ""subsample"": 0.5194197934705267, ""colsample_bytree"": 0.7283048294068741, ""min_child_weight"": 3, ""reg_lambda"": 0.1246286897734399}}"
45,10.204901291394421,0.8293409522325795,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.024923554918322904, ""subsample"": 0.5351103897075313, ""colsample_bytree"": 0.6879643329305452, ""min_child_weight"": 3, ""reg_lambda"": 0.05372823421376015}}"
46,10.359593103683572,0.8677359543153698,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.023018416835079717, ""subsample"": 0.5598911610224421, ""colsample_bytree"": 0.7645155422302656, ""min_child_weight"": 2, ""reg_lambda"": 0.03957490269907328}}"
47,10.265679252349807,0.8406183933673965,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 9, ""learning_rate"": 0.025845648873252938, ""subsample"": 0.5355074994858496, ""colsample_bytree"": 0.7212600353420534, ""min_child_weight"": 4, ""reg_lambda"": 0.1937892991945048}}"
48,10.364154845943544,0.8240387362400707,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.0261142512186217, ""subsample"": 0.5409151288355977, ""colsample_bytree"": 0.7544856754662687, ""min_child_weight"": 2, ""reg_lambda"": 0.3912811263702172}}"
49,10.226894785100729,0.8736342753706768,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 450, ""max_depth"": 10, ""learning_rate"": 0.02273788895674001, ""subsample"": 0.5109172715899015, ""colsample_bytree"": 0.6802998014204297, ""min_child_weight"": 3, ""reg_lambda"": 0.2366332667577026}}"
50,10.350349068357042,0.8534463304975449,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.020799893957325268, ""subsample"": 0.5676428944811911, ""colsample_bytree"": 0.6634827145262545, ""min_child_weight"": 3, ""reg_lambda"": 0.3386992248809452}}"
51,10.210375541982845,0.8473859359102893,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.02444765071102587, ""subsample"": 0.5871372394408758, ""colsample_bytree"": 0.6219000459198221, ""min_child_weight"": 4, ""reg_lambda"": 0.15311079688856366}}"
52,10.24319196477183,0.8102402613110908,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 9, ""learning_rate"": 0.023325479659678506, ""subsample"": 0.5469996911344228, ""colsample_bytree"": 0.6585934332373055, ""min_child_weight"": 4, ""reg_lambda"": 0.08246822643986688}}"
53,10.17909595890781,0.8211503651293576,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 9, ""learning_rate"": 0.022799779986804426, ""subsample"": 0.5612827883782842, ""colsample_bytree"": 0.6472641755272561, ""min_child_weight"": 3, ""reg_lambda"": 0.06513076584870044}}"
54,10.380716309116517,0.8270980034527459,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.035440392792006045, ""subsample"": 0.5700935416113514, ""colsample_bytree"": 0.7072307656324424, ""min_child_weight"": 3, ""reg_lambda"": 0.11422661941579962}}"
55,10.203988309641499,0.8133877223370977,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 9, ""learning_rate"": 0.028840849086544572, ""subsample"": 0.6330255371982855, ""colsample_bytree"": 0.6334933295803612, ""min_child_weight"": 3, ""reg_lambda"": 0.0575122463490688}}"
56,10.247655707631614,0.8178248582739126,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.032964997987981075, ""subsample"": 0.5034485377415866, ""colsample_bytree"": 0.6573544411171297, ""min_child_weight"": 3, ""reg_lambda"": 0.0678566950465965}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/CatBoost/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,16.887493808888113,0.4560533559535211,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""iterations"": 200, ""depth"": 5, ""learning_rate"": 0.023406777359270506, ""l2_leaf_reg"": 13.39433470675048}}"
1,15.684834132977153,0.6678303847531053,10,"{""representation"": ""regime"", ""temporal"": true, ""window"": 10, ""hyper"": {""iterations"": 300, ""depth"": 8, ""learning_rate"": 0.029180149047462245, ""l2_leaf_reg"": 2.399324290681271}}"
2,16.03114648632399,0.6117431099405163,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""iterations"": 500, ""depth"": 8, ""learning_rate"": 0.022680881003614695, ""l2_leaf_reg"": 6.172115948107071}}"
3,17.773449095995055,0.3812517156069457,10,"{""representation"": ""global"", ""temporal"": true, ""window"": 10, ""hyper"": {""iterations"": 100, ""depth"": 10, ""learning_rate"": 0.04030685101059539, ""l2_leaf_reg"": 7.277150634170934}}"
4,16.634820708679243,0.658995392329748,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""iterations"": 800, ""depth"": 9, ""learning_rate"": 0.2546633740246392, ""l2_leaf_reg"": 14.594768942966839}}"
5,14.07710684354989,0.558173186058412,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""iterations"": 500, ""depth"": 4, ""learning_rate"": 0.17558477392891916, ""l2_leaf_reg"": 1.2502377950801113}}"
6,16.25201278455038,0.6182622723152029,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""iterations"": 700, ""depth"": 8, ""learning_rate"": 0.14401540623155276, ""l2_leaf_reg"": 10.079659367870727}}"
7,13.192229149140662,0.6893232284054017,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""iterations"": 800, ""depth"": 7, ""learning_rate"": 0.027649274676728182, ""l2_leaf_reg"": 8.471354625326555}}"
9,16.205393250192795,0.6962937424420169,10,"{""representation"": ""regime"", ""temporal"": true, ""window"": 10, ""hyper"": {""iterations"": 200, ""depth"": 10, ""learning_rate"": 0.17842401600508806, ""l2_leaf_reg"": 6.669254439997733}}"
10,13.286317298176169,0.6794125151468923,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""iterations"": 800, ""depth"": 8, ""learning_rate"": 0.07287261571692631, ""l2_leaf_reg"": 18.445338876089934}}"
11,13.2412560713062,0.6728373505389308,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""iterations"": 700, ""depth"": 8, ""learning_rate"": 0.05121944772162538, ""l2_leaf_reg"": 17.183790926280032}}"
12,13.18839043377138,0.6495799753393534,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""iterations"": 800, ""depth"": 5, ""learning_rate"": 0.08397983881234862, ""l2_leaf_reg"": 8.335462491675733}}"
14,13.068638967431923,0.6204430936687443,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 5, ""learning_rate"": 0.029052791008846637, ""l2_leaf_reg"": 10.292198623253825}}"
15,13.272179678653782,0.5704821695560635,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 800, ""depth"": 4, ""learning_rate"": 0.02323329766744848, ""l2_leaf_reg"": 19.814916058536728}}"
17,14.612868012860318,0.6340045613319268,30,"{""representation"": ""regime"", ""temporal"": true, ""window"": 30, ""hyper"": {""iterations"": 600, ""depth"": 5, ""learning_rate"": 0.056956104519079426, ""l2_leaf_reg"": 7.326732473983638}}"
18,12.937156697073178,0.6790910016590512,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 6, ""learning_rate"": 0.10488316016872794, ""l2_leaf_reg"": 10.787766443005388}}"
19,13.226876794079184,0.5938438895052939,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 700, ""depth"": 4, ""learning_rate"": 0.02613824687692821, ""l2_leaf_reg"": 3.7098083893670046}}"
20,13.075553054784617,0.6474311370712093,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 700, ""depth"": 6, ""learning_rate"": 0.11854639302234501, ""l2_leaf_reg"": 13.798351948749792}}"
21,13.204367008110339,0.6600721111898012,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 6, ""learning_rate"": 0.15168708875744483, ""l2_leaf_reg"": 8.989838777136004}}"
22,12.908515349812282,0.6309788934974523,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 5, ""learning_rate"": 0.04502694998592672, ""l2_leaf_reg"": 18.958941995144357}}"
23,13.101166040062429,0.6098789571501311,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 500, ""depth"": 5, ""learning_rate"": 0.035794092930212185, ""l2_leaf_reg"": 15.7848535572046}}"
25,12.938879756758052,0.6365638851321541,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 400, ""depth"": 5, ""learning_rate"": 0.08078226637152179, ""l2_leaf_reg"": 6.431803241991869}}"
26,13.07061123038378,0.667902091691219,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 200, ""depth"": 6, ""learning_rate"": 0.08507129965092108, ""l2_leaf_reg"": 6.156557018462499}}"
31,13.019146580357509,0.6642499911236047,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 5, ""learning_rate"": 0.10087281134653615, ""l2_leaf_reg"": 19.645740726493717}}"
32,12.981476978611425,0.6255335388228923,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 500, ""depth"": 5, ""learning_rate"": 0.08727657415707969, ""l2_leaf_reg"": 9.15139438810968}}"
33,12.941552632547419,0.6193921782024453,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 400, ""depth"": 5, ""learning_rate"": 0.10928990642575721, ""l2_leaf_reg"": 12.071538202091395}}"
35,13.010245365790365,0.6739418340815615,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 7, ""learning_rate"": 0.08383086393897915, ""l2_leaf_reg"": 11.483589842784637}}"
36,12.937048328589938,0.6450118499673271,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 500, ""depth"": 6, ""learning_rate"": 0.04580347826146946, ""l2_leaf_reg"": 12.243822579499495}}"
37,12.987946613140005,0.6458144387117222,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 400, ""depth"": 6, ""learning_rate"": 0.04518803127889655, ""l2_leaf_reg"": 10.38255278893675}}"
38,12.95371831189232,0.6847674858845263,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""iterations"": 600, ""depth"": 7, ""learning_rate"": 0.07130924188138557, ""l2_leaf_reg"": 7.874569044351943}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/LightGBM/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,17.67583265008282,0.3791602419947784,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 4, ""learning_rate"": 0.023406777359270506, ""num_leaves"": 112, ""subsample"": 0.8005575058716043, ""subsample_freq"": 1, ""colsample_bytree"": 0.8540362888980227, ""min_child_samples"": 6, ""reg_lambda"": 8.123245085588687}}"
1,16.756513698524696,0.527792621272554,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 250, ""max_depth"": 7, ""learning_rate"": 0.16767510257004176, ""num_leaves"": 37, ""subsample"": 0.7571172192068059, ""subsample_freq"": 1, ""colsample_bytree"": 0.7962072844310213, ""min_child_samples"": 9, ""reg_lambda"": 0.6647135865318028}}"
2,17.69711340915347,0.39365226159817984,10,"{""representation"": ""global"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 50, ""max_depth"": 12, ""learning_rate"": 0.04030685101059539, ""num_leaves"": 89, ""subsample"": 0.6558555380447055, ""subsample_freq"": 1, ""colsample_bytree"": 0.7600340105889054, ""min_child_samples"": 57, ""reg_lambda"": 0.035856126103453984}}"
3,16.265822960732343,0.5796550448334011,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 400, ""max_depth"": 12, ""learning_rate"": 0.025415885998427375, ""num_leaves"": 37, ""subsample"": 0.522613644455269, ""subsample_freq"": 1, ""colsample_bytree"": 0.6626651653816322, ""min_child_samples"": 42, ""reg_lambda"": 0.06516990611177176}}"
4,16.971559989028588,0.5211729275882986,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.14401540623155276, ""num_leaves"": 102, ""subsample"": 0.5370223258670452, ""subsample_freq"": 1, ""colsample_bytree"": 0.6792328642721364, ""min_child_samples"": 16, ""reg_lambda"": 3.8842777547031417}}"
5,16.897177685901916,0.5210187635975643,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""n_estimators"": 500, ""max_depth"": 8, ""learning_rate"": 0.16134583428320007, ""num_leaves"": 70, ""subsample"": 0.7613664146909971, ""subsample_freq"": 1, ""colsample_bytree"": 0.7137705091792748, ""min_child_samples"": 7, ""reg_lambda"": 0.021070472806578238}}"
6,16.290344122641518,0.7398893892751018,10,"{""representation"": ""regime"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 100, ""max_depth"": 12, ""learning_rate"": 0.17842401600508806, ""num_leaves"": 86, ""subsample"": 0.9357302950938589, ""subsample_freq"": 1, ""colsample_bytree"": 0.9018360384495572, ""min_child_samples"": 22, ""reg_lambda"": 4.760767751809501}}"
7,16.23321889822443,0.5591566804830835,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 5, ""learning_rate"": 0.06358398347237385, ""num_leaves"": 107, ""subsample"": 0.9303652916281717, ""subsample_freq"": 1, ""colsample_bytree"": 0.5034760652655954, ""min_child_samples"": 54, ""reg_lambda"": 0.1787446325623842}}"
8,16.701871897157982,0.516934125703729,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 300, ""max_depth"": 6, ""learning_rate"": 0.043254212679768214, ""num_leaves"": 19, ""subsample"": 0.8047821669899484, ""subsample_freq"": 1, ""colsample_bytree"": 0.7513395116144308, ""min_child_samples"": 9, ""reg_lambda"": 0.06853925708853059}}"
9,16.31070983268308,0.5919327305703124,10,"{""representation"": ""global"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 400, ""max_depth"": 9, ""learning_rate"": 0.08533946549577034, ""num_leaves"": 25, ""subsample"": 0.917651247794619, ""subsample_freq"": 1, ""colsample_bytree"": 0.6603900324858679, ""min_child_samples"": 22, ""reg_lambda"": 0.013253342600066534}}"
10,16.162969565554487,0.5658385019694719,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 50, ""max_depth"": 10, ""learning_rate"": 0.07684915506239827, ""num_leaves"": 117, ""subsample"": 0.973717424529964, ""subsample_freq"": 1, ""colsample_bytree"": 0.5023652276879995, ""min_child_samples"": 35, ""reg_lambda"": 0.09115133400327852}}"
11,16.284967543602807,0.5234676886491799,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 7, ""learning_rate"": 0.03655107943076725, ""num_leaves"": 70, ""subsample"": 0.9702648506587636, ""subsample_freq"": 1, ""colsample_bytree"": 0.5042160003975352, ""min_child_samples"": 30, ""reg_lambda"": 0.25746072179892365}}"
12,16.05486889812497,0.6429611406353146,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 12, ""learning_rate"": 0.04917719247954602, ""num_leaves"": 105, ""subsample"": 0.9140598222505272, ""subsample_freq"": 1, ""colsample_bytree"": 0.5130142155336747, ""min_child_samples"": 41, ""reg_lambda"": 0.12833963691532346}}"
13,16.028677250885227,0.6225985294605308,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 12, ""learning_rate"": 0.05085695921084065, ""num_leaves"": 118, ""subsample"": 0.9887092742539697, ""subsample_freq"": 1, ""colsample_bytree"": 0.6614404581365693, ""min_child_samples"": 39, ""reg_lambda"": 0.055357301062444644}}"
14,16.11229450331507,0.6503409196983163,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 11, ""learning_rate"": 0.0839035011814298, ""num_leaves"": 104, ""subsample"": 0.9500888057668408, ""subsample_freq"": 1, ""colsample_bytree"": 0.6838631783149963, ""min_child_samples"": 31, ""reg_lambda"": 0.017156344068490877}}"
15,16.12644379198954,0.662388097896013,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 250, ""max_depth"": 12, ""learning_rate"": 0.06622353587693178, ""num_leaves"": 97, ""subsample"": 0.9316057095240647, ""subsample_freq"": 1, ""colsample_bytree"": 0.648773204915838, ""min_child_samples"": 63, ""reg_lambda"": 0.015413812552615673}}"
16,16.107149156544025,0.6681773881828041,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 150, ""max_depth"": 12, ""learning_rate"": 0.054636223203467955, ""num_leaves"": 110, ""subsample"": 0.9411777516204995, ""subsample_freq"": 1, ""colsample_bytree"": 0.8801629098430304, ""min_child_samples"": 36, ""reg_lambda"": 0.23867979040628204}}"
18,16.0796866524316,0.6398355087348248,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 150, ""max_depth"": 12, ""learning_rate"": 0.06190018687315985, ""num_leaves"": 85, ""subsample"": 0.9581198744806707, ""subsample_freq"": 1, ""colsample_bytree"": 0.5083716589527347, ""min_child_samples"": 48, ""reg_lambda"": 0.08446762510799544}}"
19,16.144620633511682,0.6124675247699589,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 50, ""max_depth"": 11, ""learning_rate"": 0.12415987097615254, ""num_leaves"": 115, ""subsample"": 0.8381964986078193, ""subsample_freq"": 1, ""colsample_bytree"": 0.5114786885706106, ""min_child_samples"": 54, ""reg_lambda"": 0.1822665942044284}}"
20,16.36052329804368,0.6511393515561041,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 350, ""max_depth"": 12, ""learning_rate"": 0.09536063190995918, ""num_leaves"": 108, ""subsample"": 0.9613331398217125, ""subsample_freq"": 1, ""colsample_bytree"": 0.6528095482533981, ""min_child_samples"": 13, ""reg_lambda"": 0.03535105049736246}}"
21,16.07738068930291,0.5805512972915933,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 12, ""learning_rate"": 0.039232284788460414, ""num_leaves"": 117, ""subsample"": 0.8015240463514657, ""subsample_freq"": 1, ""colsample_bytree"": 0.5714266479534555, ""min_child_samples"": 28, ""reg_lambda"": 0.04475352797287101}}"
22,16.041993743268595,0.5982316111911735,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 11, ""learning_rate"": 0.041987417645074195, ""num_leaves"": 120, ""subsample"": 0.9673456534922804, ""subsample_freq"": 1, ""colsample_bytree"": 0.6288796881603438, ""min_child_samples"": 56, ""reg_lambda"": 0.030134449474446993}}"
23,16.01021708312628,0.6362158376811067,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 12, ""learning_rate"": 0.034677659667425344, ""num_leaves"": 119, ""subsample"": 0.9640484582473167, ""subsample_freq"": 1, ""colsample_bytree"": 0.560693702929917, ""min_child_samples"": 42, ""reg_lambda"": 0.01095603721678528}}"
24,16.047567120667644,0.6079500304281659,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 12, ""learning_rate"": 0.0218692391126777, ""num_leaves"": 122, ""subsample"": 0.9956981319829279, ""subsample_freq"": 1, ""colsample_bytree"": 0.66654922125316, ""min_child_samples"": 36, ""reg_lambda"": 0.04227340096828055}}"
26,16.16561616383094,0.6593174007880067,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 11, ""learning_rate"": 0.11264233604217194, ""num_leaves"": 117, ""subsample"": 0.9985537941329566, ""subsample_freq"": 1, ""colsample_bytree"": 0.6128569162977998, ""min_child_samples"": 42, ""reg_lambda"": 0.061580949384931534}}"
27,16.03019181896652,0.6388026815823913,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 10, ""learning_rate"": 0.04215764024025921, ""num_leaves"": 89, ""subsample"": 0.904486314007807, ""subsample_freq"": 1, ""colsample_bytree"": 0.5502638157413137, ""min_child_samples"": 55, ""reg_lambda"": 0.022094690637082092}}"
28,16.037344768645063,0.6446390242689206,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 10, ""learning_rate"": 0.045034281757531686, ""num_leaves"": 104, ""subsample"": 0.9696662009583141, ""subsample_freq"": 1, ""colsample_bytree"": 0.666021340483371, ""min_child_samples"": 22, ""reg_lambda"": 0.015907319234310907}}"
29,16.01205627750395,0.6379916849166293,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 250, ""max_depth"": 11, ""learning_rate"": 0.033987721995336354, ""num_leaves"": 120, ""subsample"": 0.8649707809637792, ""subsample_freq"": 1, ""colsample_bytree"": 0.5070593597549009, ""min_child_samples"": 46, ""reg_lambda"": 0.036910774893209854}}"
30,16.063434256813846,0.6448458328571408,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 10, ""learning_rate"": 0.0638588191583873, ""num_leaves"": 120, ""subsample"": 0.9171303034875519, ""subsample_freq"": 1, ""colsample_bytree"": 0.6934481765458397, ""min_child_samples"": 48, ""reg_lambda"": 0.03689823578054415}}"
33,16.035864367226313,0.6026215674150638,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 12, ""learning_rate"": 0.022066802075497032, ""num_leaves"": 127, ""subsample"": 0.8893688613521341, ""subsample_freq"": 1, ""colsample_bytree"": 0.598326265185935, ""min_child_samples"": 37, ""reg_lambda"": 0.01245345267907764}}"
34,16.055956348437583,0.6510049347357109,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 300, ""max_depth"": 12, ""learning_rate"": 0.03786142277085321, ""num_leaves"": 121, ""subsample"": 0.9209034354316765, ""subsample_freq"": 1, ""colsample_bytree"": 0.5542961935644268, ""min_child_samples"": 34, ""reg_lambda"": 0.03091479910350033}}"
36,16.049866866743592,0.6622051317724799,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 250, ""max_depth"": 11, ""learning_rate"": 0.044843318067678, ""num_leaves"": 107, ""subsample"": 0.842568566191037, ""subsample_freq"": 1, ""colsample_bytree"": 0.5510541282860912, ""min_child_samples"": 44, ""reg_lambda"": 0.022014948836078043}}"
37,15.98808768678629,0.6249132324303478,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 300, ""max_depth"": 10, ""learning_rate"": 0.021754394363921317, ""num_leaves"": 108, ""subsample"": 0.8441880343895716, ""subsample_freq"": 1, ""colsample_bytree"": 0.5274957068320709, ""min_child_samples"": 46, ""reg_lambda"": 0.032279357860576396}}"
38,16.062988866615804,0.6487613007023165,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 150, ""max_depth"": 12, ""learning_rate"": 0.05477846973697912, ""num_leaves"": 116, ""subsample"": 0.9114763642256805, ""subsample_freq"": 1, ""colsample_bytree"": 0.6303096992057566, ""min_child_samples"": 46, ""reg_lambda"": 0.06338336444530064}}"
39,16.025005098587236,0.647084476435127,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 10, ""learning_rate"": 0.04258621385817288, ""num_leaves"": 122, ""subsample"": 0.8612112514999446, ""subsample_freq"": 1, ""colsample_bytree"": 0.578123330531582, ""min_child_samples"": 36, ""reg_lambda"": 0.05959669309139125}}"
40,16.115467865709505,0.6497155035989903,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 12, ""learning_rate"": 0.06280165009629278, ""num_leaves"": 102, ""subsample"": 0.9145257976083557, ""subsample_freq"": 1, ""colsample_bytree"": 0.5486755507839556, ""min_child_samples"": 35, ""reg_lambda"": 0.015641423512551152}}"
41,15.9991137535436,0.6413901748259976,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 250, ""max_depth"": 12, ""learning_rate"": 0.02882544316599006, ""num_leaves"": 111, ""subsample"": 0.9496342927837447, ""subsample_freq"": 1, ""colsample_bytree"": 0.5672654755701203, ""min_child_samples"": 47, ""reg_lambda"": 0.01447428364062025}}"
42,15.999517994219326,0.641209649510477,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 250, ""max_depth"": 11, ""learning_rate"": 0.030064003342366957, ""num_leaves"": 90, ""subsample"": 0.8886543552287902, ""subsample_freq"": 1, ""colsample_bytree"": 0.5561929457313222, ""min_child_samples"": 43, ""reg_lambda"": 0.0177899866701305}}"
43,16.006669541003834,0.6499273116065206,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 350, ""max_depth"": 10, ""learning_rate"": 0.02435966835566729, ""num_leaves"": 108, ""subsample"": 0.8303769405027125, ""subsample_freq"": 1, ""colsample_bytree"": 0.5492657617179162, ""min_child_samples"": 42, ""reg_lambda"": 0.02264999544075302}}"
45,16.00959866324275,0.6422975711354099,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 11, ""learning_rate"": 0.0351521006802588, ""num_leaves"": 115, ""subsample"": 0.8517398298453449, ""subsample_freq"": 1, ""colsample_bytree"": 0.6174479293751848, ""min_child_samples"": 48, ""reg_lambda"": 0.01856481298367706}}"
46,15.99920727790709,0.6459019157593593,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 350, ""max_depth"": 10, ""learning_rate"": 0.025513362549568427, ""num_leaves"": 106, ""subsample"": 0.8907126462041683, ""subsample_freq"": 1, ""colsample_bytree"": 0.5689364362703258, ""min_child_samples"": 59, ""reg_lambda"": 0.03819930105698383}}"
47,16.014949653211126,0.6382382671070604,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 350, ""max_depth"": 12, ""learning_rate"": 0.02684037304563009, ""num_leaves"": 105, ""subsample"": 0.8740670307402343, ""subsample_freq"": 1, ""colsample_bytree"": 0.512721464688423, ""min_child_samples"": 55, ""reg_lambda"": 0.022718879047367867}}"
48,16.012082849978135,0.6557758784851135,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 350, ""max_depth"": 10, ""learning_rate"": 0.02989454714873962, ""num_leaves"": 105, ""subsample"": 0.8203458129511032, ""subsample_freq"": 1, ""colsample_bytree"": 0.5589077065086714, ""min_child_samples"": 46, ""reg_lambda"": 0.03746416121432892}}"
49,16.020649018850673,0.6327400191699056,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 300, ""max_depth"": 12, ""learning_rate"": 0.023912525795895197, ""num_leaves"": 109, ""subsample"": 0.9845974671707729, ""subsample_freq"": 1, ""colsample_bytree"": 0.5339930011077915, ""min_child_samples"": 46, ""reg_lambda"": 0.024220110373061186}}"
50,16.01239418467107,0.6203288591431133,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 300, ""max_depth"": 10, ""learning_rate"": 0.0226111268754126, ""num_leaves"": 95, ""subsample"": 0.9446882484822953, ""subsample_freq"": 1, ""colsample_bytree"": 0.5132577845455102, ""min_child_samples"": 31, ""reg_lambda"": 0.01905728007138073}}"
51,16.016964839336673,0.648840384635864,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 10, ""learning_rate"": 0.04140589317318375, ""num_leaves"": 123, ""subsample"": 0.8995387374872827, ""subsample_freq"": 1, ""colsample_bytree"": 0.6243514374930498, ""min_child_samples"": 55, ""reg_lambda"": 0.0106469261767822}}"
52,15.990242050674732,0.6517436000338325,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 250, ""max_depth"": 11, ""learning_rate"": 0.027818382680539495, ""num_leaves"": 110, ""subsample"": 0.8207300073101977, ""subsample_freq"": 1, ""colsample_bytree"": 0.6176520541426094, ""min_child_samples"": 39, ""reg_lambda"": 0.02267639413785733}}"
54,15.989651093172148,0.6331798034407106,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 300, ""max_depth"": 9, ""learning_rate"": 0.024623198756775582, ""num_leaves"": 123, ""subsample"": 0.8870004330028822, ""subsample_freq"": 1, ""colsample_bytree"": 0.5625204872882205, ""min_child_samples"": 38, ""reg_lambda"": 0.014581048636289909}}"
55,15.987256012843606,0.6442328324532323,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 350, ""max_depth"": 9, ""learning_rate"": 0.02206809196439176, ""num_leaves"": 106, ""subsample"": 0.8570654450689827, ""subsample_freq"": 1, ""colsample_bytree"": 0.600717671541155, ""min_child_samples"": 36, ""reg_lambda"": 0.018279984206522333}}"
56,15.990743044537874,0.6480594372079365,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 250, ""max_depth"": 10, ""learning_rate"": 0.030587909880441477, ""num_leaves"": 109, ""subsample"": 0.8065693669755241, ""subsample_freq"": 1, ""colsample_bytree"": 0.6159039264711151, ""min_child_samples"": 39, ""reg_lambda"": 0.028756136455191133}}"
57,15.979257107521118,0.6377848430864496,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 300, ""max_depth"": 10, ""learning_rate"": 0.024449265853509095, ""num_leaves"": 119, ""subsample"": 0.9023281061722079, ""subsample_freq"": 1, ""colsample_bytree"": 0.6146100590684846, ""min_child_samples"": 53, ""reg_lambda"": 0.023766534262631735}}"
58,15.997798190645778,0.6300884404823925,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 300, ""max_depth"": 10, ""learning_rate"": 0.022597862664933632, ""num_leaves"": 118, ""subsample"": 0.8669014073942173, ""subsample_freq"": 1, ""colsample_bytree"": 0.530880308322363, ""min_child_samples"": 26, ""reg_lambda"": 0.01288565152864745}}"
59,16.00676784817548,0.6495067297517271,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 250, ""max_depth"": 10, ""learning_rate"": 0.029803055764272255, ""num_leaves"": 111, ""subsample"": 0.8711161218921345, ""subsample_freq"": 1, ""colsample_bytree"": 0.5689595674417415, ""min_child_samples"": 38, ""reg_lambda"": 0.02855869038099852}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/MLP/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,16.172237230608935,0.6278876446793975,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.002537815508265664, ""learning_rate_init"": 0.0026070247583707684}}"
1,16.113879988727586,0.7456839539934563,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.0005342937261279777, ""learning_rate_init"": 0.0003823475224675188}}"
2,18.38296792066325,0.4396667021151913,15,"{""representation"": ""global"", ""temporal"": true, ""window"": 15, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.01712337597316398, ""learning_rate_init"": 0.0004066563313514797}}"
3,16.010276128815338,0.6689626125821535,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.004467752817973905, ""learning_rate_init"": 0.0004201672054372534}}"
4,17.10348780063065,0.6496719707433046,15,"{""representation"": ""regime"", ""temporal"": true, ""window"": 15, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.02065142557895925, ""learning_rate_init"": 0.0005170191786366995}}"
5,18.03558637399326,0.5715925790619767,15,"{""representation"": ""regime"", ""temporal"": true, ""window"": 15, ""hyper"": {""hidden_layer_sizes"": [128, 64], ""alpha"": 2.9072088906598435e-05, ""learning_rate_init"": 0.005323617594751502}}"
6,18.85778721715002,0.4838508173776779,30,"{""representation"": ""global"", ""temporal"": true, ""window"": 30, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.0012329098365270509, ""learning_rate_init"": 0.0007162786999897341}}"
7,16.842563683224473,0.547475012604595,20,"{""representation"": ""regime"", ""temporal"": true, ""window"": 20, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.01707975034295823, ""learning_rate_init"": 0.0018484491720988621}}"
9,15.907758078285308,0.678502109138318,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.07076022085822008, ""learning_rate_init"": 0.00031883397351001905}}"
10,15.904652070666668,0.6282821524786887,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 2.5990875064996908e-05, ""learning_rate_init"": 0.00040603622450132007}}"
11,15.941308129563868,0.6376864060935883,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 2.165099942185883e-05, ""learning_rate_init"": 0.00015092395777542054}}"
12,15.872665847745116,0.6280075650832071,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.04491488223084785, ""learning_rate_init"": 0.00014958102201351417}}"
13,15.934782596759936,0.6334836570788707,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.0012580527124235895, ""learning_rate_init"": 0.00012946225931831303}}"
14,15.849992876778368,0.642356385489991,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.024307803800353533, ""learning_rate_init"": 0.0001129446389641891}}"
15,15.931105891803668,0.6393685641018175,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [128, 64], ""alpha"": 0.03042342264144351, ""learning_rate_init"": 0.00013905583448470645}}"
17,15.858797936880276,0.6315236904324353,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.09097662173143836, ""learning_rate_init"": 0.0006193601987732719}}"
18,15.940298510824363,0.6731460272976054,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.05516610254984167, ""learning_rate_init"": 0.001689834358677167}}"
21,15.90693754510786,0.6740344236030078,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.015743720061362248, ""learning_rate_init"": 0.0002978042675785457}}"
23,15.942645331113647,0.6994738835597603,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [256, 128], ""alpha"": 0.0956014081856151, ""learning_rate_init"": 0.0006743738085946358}}"
25,15.92532031632646,0.6956163839503346,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""hidden_layer_sizes"": [128, 64, 32], ""alpha"": 0.08120861723837608, ""learning_rate_init"": 0.00010131113679850053}}"
26,14.781743801833688,0.6245850553108985,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.01679274669129797, ""learning_rate_init"": 0.0001831508128578392}}"
27,14.756609021186822,0.5623064394662096,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.008324807286421161, ""learning_rate_init"": 0.00015030942695843836}}"
28,15.003903257288126,0.5482253307143298,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.004254406369348549, ""learning_rate_init"": 0.00020012057588857996}}"
29,14.844113797032923,0.6318268703281298,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""hidden_layer_sizes"": [64, 32], ""alpha"": 0.0008488768985907142, ""learning_rate_init"": 0.000356059699248016}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/PyTorch_CNN/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,19.314465756830618,0.5550339128646333,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002552951604697378, ""batch_size"": 256, ""dropout"": 0.08493564427131046, ""num_filters"": 128, ""kernel_size"": 3}}"
1,19.968490147747165,1.058784834344505,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001956070814274847, ""batch_size"": 256, ""dropout"": 0.02602063719411181, ""num_filters"": 64, ""kernel_size"": 3}}"
2,21.16703471439263,0.29106883604322487,30,"{""representation"": ""global"", ""window"": 30, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0016559149440361915, ""batch_size"": 512, ""dropout"": 0.31005312934444584, ""num_filters"": 32, ""kernel_size"": 3}}"
3,20.705721833318595,0.5824931539525995,30,"{""representation"": ""global"", ""window"": 30, ""temporal"": false, ""hyper"": {""learning_rate"": 0.000691662498760998, ""batch_size"": 512, ""dropout"": 0.308897907718663, ""num_filters"": 128, ""kernel_size"": 5}}"
4,19.415400316800756,0.7133924011251469,20,"{""representation"": ""global"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0010571906813317184, ""batch_size"": 512, ""dropout"": 0.18888597006477972, ""num_filters"": 128, ""kernel_size"": 5}}"
5,19.340217917266553,0.4493504721126067,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.004041443189081019, ""batch_size"": 512, ""dropout"": 0.09151926619664899, ""num_filters"": 64, ""kernel_size"": 3}}"
6,19.87403094704545,0.6147087105009638,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0006442017924231741, ""batch_size"": 512, ""dropout"": 0.3442922333025374, ""num_filters"": 64, ""kernel_size"": 3}}"
7,19.278446236726822,0.6515711948319709,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0015711531129544835, ""batch_size"": 128, ""dropout"": 0.24382573359195875, ""num_filters"": 32, ""kernel_size"": 3}}"
8,16.395262693907963,0.5089530321385676,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002144252054813578, ""batch_size"": 128, ""dropout"": 0.3341209982356952, ""num_filters"": 32, ""kernel_size"": 5}}"
10,20.228445642811536,0.7484265914412301,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0007605707018582466, ""batch_size"": 128, ""dropout"": 0.09894249765332985, ""num_filters"": 32, ""kernel_size"": 5}}"
11,20.78328100095338,1.4368504081863935,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002041022877906392, ""batch_size"": 128, ""dropout"": 0.2669883775354702, ""num_filters"": 32, ""kernel_size"": 3}}"
12,16.067694775786208,0.49081911579929166,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0011262698856876726, ""batch_size"": 256, ""dropout"": 0.29174697772868086, ""num_filters"": 32, ""kernel_size"": 5}}"
13,16.254071385728153,0.455795358706925,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.004201895670862748, ""batch_size"": 128, ""dropout"": 0.3575312644632453, ""num_filters"": 32, ""kernel_size"": 5}}"
14,16.687850449439516,0.7407982691754471,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0007879277197822641, ""batch_size"": 256, ""dropout"": 0.3584242659851944, ""num_filters"": 32, ""kernel_size"": 5}}"
15,16.744505817480704,0.5412274889599992,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0005963263380020579, ""batch_size"": 256, ""dropout"": 0.20560288336603372, ""num_filters"": 32, ""kernel_size"": 5}}"
16,16.622194594479243,0.47641508539055955,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.003180510342487187, ""batch_size"": 128, ""dropout"": 0.1827709200688888, ""num_filters"": 32, ""kernel_size"": 5}}"
17,16.255801556743204,0.5749782260832169,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0015380390031574304, ""batch_size"": 256, ""dropout"": 0.25165628992302785, ""num_filters"": 64, ""kernel_size"": 5}}"
18,17.101474157399057,0.3605230974902526,15,"{""representation"": ""regime"", ""window"": 15, ""temporal"": false, ""hyper"": {""learning_rate"": 0.000684593241991129, ""batch_size"": 128, ""dropout"": 0.3014551323141699, ""num_filters"": 32, ""kernel_size"": 5}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/PyTorch_RNN_LSTM/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,13.79654466257337,0.6999647365429322,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002552951604697378, ""batch_size"": 256, ""dropout"": 0.08493564427131046, ""hidden_dim"": 128, ""num_layers"": 2}}"
1,20.962482502151406,0.7830007341837072,50,"{""representation"": ""global"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0016338208828908817, ""batch_size"": 512, ""dropout"": 0.06820964947491662, ""hidden_dim"": 128, ""num_layers"": 3}}"
2,20.356099165521655,1.0569928923049057,50,"{""representation"": ""global"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002298752892366083, ""batch_size"": 512, ""dropout"": 0.07394178221021082, ""hidden_dim"": 32, ""num_layers"": 3}}"
3,14.116881362414205,0.5090695984361209,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0011369027867815917, ""batch_size"": 256, ""dropout"": 0.32087879230161587, ""hidden_dim"": 64, ""num_layers"": 1}}"
4,12.373466281033977,0.6892682971139921,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002100236158351101, ""batch_size"": 128, ""dropout"": 0.13007332881069883, ""hidden_dim"": 128, ""num_layers"": 2}}"
5,16.589848906991435,0.45807519183893847,20,"{""representation"": ""regime"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0006410050209458448, ""batch_size"": 256, ""dropout"": 0.20342827646588113, ""hidden_dim"": 32, ""num_layers"": 3}}"
6,20.08253606112105,0.2757583191620094,20,"{""representation"": ""global"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.000768316328806251, ""batch_size"": 128, ""dropout"": 0.3584365199693973, ""hidden_dim"": 32, ""num_layers"": 2}}"
7,14.93875169917029,0.280954826647668,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001052380849166306, ""batch_size"": 256, ""dropout"": 0.3887128330883843, ""hidden_dim"": 32, ""num_layers"": 1}}"
10,13.649295310993551,0.7010614903165632,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001166641358902415, ""batch_size"": 128, ""dropout"": 0.182290204565902, ""hidden_dim"": 128, ""num_layers"": 1}}"
11,14.245456451157585,0.5880241211377527,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0010688731090013406, ""batch_size"": 128, ""dropout"": 0.19650816580044883, ""hidden_dim"": 128, ""num_layers"": 1}}"
13,13.378087702621377,0.31717347974086857,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002012710449966973, ""batch_size"": 128, ""dropout"": 0.19889643007254898, ""hidden_dim"": 128, ""num_layers"": 3}}"
14,12.862248031569765,0.9730956502589635,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0046758233606198326, ""batch_size"": 128, ""dropout"": 0.13790785883622897, ""hidden_dim"": 128, ""num_layers"": 3}}"
17,13.317527260140954,0.11104857926818167,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0015940675683695877, ""batch_size"": 128, ""dropout"": 0.04762942127090697, ""hidden_dim"": 128, ""num_layers"": 2}}"
19,13.844841429534618,0.31811579769152865,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0013227252733052034, ""batch_size"": 128, ""dropout"": 0.27302448871946555, ""hidden_dim"": 128, ""num_layers"": 3}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/PyTorch_Transformer/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,16.91789585264244,0.7093835720279668,50,"{""representation"": ""regime"", ""window"": 50, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002552951604697378, ""batch_size"": 256, ""dropout"": 0.08493564427131046, ""d_model"": 128, ""nhead"": 2, ""num_layers"": 1, ""dim_feedforward"": 64}}"
1,16.933776780857368,0.9292686141703841,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0005807932994623225, ""batch_size"": 256, ""dropout"": 0.12184550766934828, ""d_model"": 64, ""nhead"": 4, ""num_layers"": 1, ""dim_feedforward"": 64}}"
2,18.707161713503833,0.9095934062472094,20,"{""representation"": ""regime"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.004176805377655879, ""batch_size"": 256, ""dropout"": 0.13013213230530574, ""d_model"": 128, ""nhead"": 2, ""num_layers"": 2, ""dim_feedforward"": 128}}"
3,25.327443671317706,3.1501459156032743,20,"{""representation"": ""global"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0011413943879952562, ""batch_size"": 256, ""dropout"": 0.13235920994105968, ""d_model"": 128, ""nhead"": 2, ""num_layers"": 3, ""dim_feedforward"": 256}}"
4,21.896707205466203,1.3146771204968832,10,"{""representation"": ""global"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0021646137637899576, ""batch_size"": 512, ""dropout"": 0.09971689165954999, ""d_model"": 64, ""nhead"": 4, ""num_layers"": 1, ""dim_feedforward"": 64}}"
5,24.597714701983108,0.7847577612870682,40,"{""representation"": ""global"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0008450942890954814, ""batch_size"": 512, ""dropout"": 0.0027808522124762817, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 256}}"
6,17.066915084328084,0.7706543479528746,10,"{""representation"": ""regime"", ""window"": 10, ""temporal"": false, ""hyper"": {""learning_rate"": 0.001590922513078389, ""batch_size"": 512, ""dropout"": 0.09582475626678898, ""d_model"": 128, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 128}}"
7,16.874262706608476,0.6642447078063597,20,"{""representation"": ""regime"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002379766606935902, ""batch_size"": 256, ""dropout"": 0.25806911616378, ""d_model"": 64, ""nhead"": 2, ""num_layers"": 2, ""dim_feedforward"": 128}}"
8,14.049794343348589,0.8537602372805568,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0021481839210591726, ""batch_size"": 512, ""dropout"": 0.35884410398103084, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 64}}"
10,15.502913231306678,0.7452944189379498,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0008006892867011675, ""batch_size"": 512, ""dropout"": 0.3778945029673634, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 64}}"
11,15.234694121992659,0.7521354287870571,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0018876017913010321, ""batch_size"": 512, ""dropout"": 0.3227018870635695, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 64}}"
12,16.807393226627255,1.1730664557480477,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0035214913703556705, ""batch_size"": 512, ""dropout"": 0.22117625442807667, ""d_model"": 128, ""nhead"": 4, ""num_layers"": 2, ""dim_feedforward"": 64}}"
14,15.735563927306059,0.5306421461949447,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.002161223933778057, ""batch_size"": 512, ""dropout"": 0.3719608431766212, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 2, ""dim_feedforward"": 64}}"
15,14.094327606135293,0.6621941786035273,60,"{""representation"": ""regime"", ""window"": 60, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0025775569875316454, ""batch_size"": 512, ""dropout"": 0.3133874763577134, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 64}}"
16,16.932744700184234,0.7231386376131796,20,"{""representation"": ""regime"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.000779134189087929, ""batch_size"": 512, ""dropout"": 0.24746413384746765, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 64}}"
17,16.987645740755905,1.2646280666769487,40,"{""representation"": ""regime"", ""window"": 40, ""temporal"": false, ""hyper"": {""learning_rate"": 0.004848334879564388, ""batch_size"": 128, ""dropout"": 0.3889659194326925, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 64}}"
18,17.01763094495939,0.8846209617530326,20,"{""representation"": ""regime"", ""window"": 20, ""temporal"": false, ""hyper"": {""learning_rate"": 0.0035073224813126505, ""batch_size"": 512, ""dropout"": 0.30494528147000977, ""d_model"": 32, ""nhead"": 4, ""num_layers"": 3, ""dim_feedforward"": 64}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/Random_Forest/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,16.75868281891498,0.5341876760889916,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 50, ""max_depth"": 8, ""min_samples_leaf"": 1, ""max_features"": 0.9063233020424546}}"
1,15.948615237806829,0.6756802014836533,10,"{""representation"": ""regime"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 100, ""max_depth"": 17, ""min_samples_leaf"": 2, ""max_features"": 0.5045012539746527}}"
2,16.215588291930132,0.5853211537608773,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 17, ""min_samples_leaf"": 1, ""max_features"": 0.7252813963310067}}"
3,16.652485924216045,0.5458911198498094,10,"{""representation"": ""global"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 50, ""max_depth"": 23, ""min_samples_leaf"": 3, ""max_features"": 0.7637655990477874}}"
4,16.21769029956302,0.605915854078062,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 300, ""max_depth"": 20, ""min_samples_leaf"": 10, ""max_features"": 0.9263791452993542}}"
5,14.795208108492929,0.5272283321415915,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 200, ""max_depth"": 8, ""min_samples_leaf"": 9, ""max_features"": 0.35218545057583955}}"
6,16.402996045509997,0.5716650353726372,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 250, ""max_depth"": 19, ""min_samples_leaf"": 8, ""max_features"": 0.8398892426801621}}"
7,14.152711411537258,0.5681527624416998,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 14, ""min_samples_leaf"": 2, ""max_features"": 0.7992713510560965}}"
9,16.09083933778969,0.6846826037133533,10,"{""representation"": ""regime"", ""temporal"": true, ""window"": 10, ""hyper"": {""n_estimators"": 50, ""max_depth"": 23, ""min_samples_leaf"": 9, ""max_features"": 0.7433826295572964}}"
10,14.241365332105506,0.5838212858136708,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 150, ""max_depth"": 19, ""min_samples_leaf"": 5, ""max_features"": 0.9292785077841608}}"
11,14.21906741793697,0.5931516287022096,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 200, ""max_depth"": 19, ""min_samples_leaf"": 6, ""max_features"": 0.9226072784555683}}"
12,14.424432270204559,0.5572376242657787,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 10, ""min_samples_leaf"": 4, ""max_features"": 0.9208176442899881}}"
13,16.00800631764638,0.6863808478445289,15,"{""representation"": ""regime"", ""temporal"": true, ""window"": 15, ""hyper"": {""n_estimators"": 300, ""max_depth"": 13, ""min_samples_leaf"": 4, ""max_features"": 0.6634774968148336}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/Ridge/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,20.79700443695906,0.34998969844078276,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 0.06026889128682508}}"
1,22.084530186249584,0.3588625783659535,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 34.70266988650411}}"
2,20.79702280374273,0.35001773801515934,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 0.08111941985431921}}"
3,20.797185456242534,0.3501393755621514,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 0.2858549394196191}}"
4,19.392636987812903,0.40768382749141424,15,"{""representation"": ""global"", ""temporal"": true, ""window"": 15, ""hyper"": {""alpha"": 0.0712230583333387}}"
5,20.797211247412964,0.35015172569494624,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 0.3334792728637583}}"
6,20.79745723608842,0.3502337708529642,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""alpha"": 2.991469302130215}}"
7,17.511931665507,0.2915898587090906,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 297.94544625913585}}"
8,17.429254450147283,0.2736209889265476,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 5.1698794422985355}}"
9,17.98066434115682,0.309611990605137,30,"{""representation"": ""regime"", ""temporal"": true, ""window"": 30, ""hyper"": {""alpha"": 0.023454342277260523}}"
10,18.189433269343915,0.3023182776156415,40,"{""representation"": ""global"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 2.3261440591233864}}"
11,17.47938130584181,0.2868218284764055,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 202.98110988398884}}"
12,17.429816179821398,0.27343327305029175,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""alpha"": 3.1873283397227463}}"
13,16.71002112701147,0.23344905111724026,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 2.677888373979133}}"
14,16.71058673147693,0.23338740722206555,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 1.0181659339606826}}"
15,16.710839243482837,0.2333511277758989,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 0.1749666616740491}}"
17,16.710681746273913,0.23337631394952812,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 0.7364270182642626}}"
19,16.70789458527721,0.23370656630647144,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 10.85063770122587}}"
23,16.71076709426154,0.23336518815740792,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 0.46617947954514244}}"
25,16.71046078187932,0.2334014193275479,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 1.383596708757115}}"
26,17.027951567702203,0.2507862450538643,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""alpha"": 17.525344508769724}}"
27,17.155585094976374,0.2813063461974276,60,"{""representation"": ""global"", ""temporal"": true, ""window"": 60, ""hyper"": {""alpha"": 0.29769833333382206}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/XGBoost/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,17.658587548509114,0.3762845446864468,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 4, ""learning_rate"": 0.023406777359270506, ""subsample"": 0.9330880728874675, ""colsample_bytree"": 0.8005575058716043, ""min_child_weight"": 8, ""reg_lambda"": 0.011527987128232402}}"
1,16.765958531456118,0.5122278708795309,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 150, ""max_depth"": 5, ""learning_rate"": 0.08283072221077194, ""subsample"": 0.7159725093210578, ""colsample_bytree"": 0.645614570099021, ""min_child_weight"": 7, ""reg_lambda"": 0.02621087878265441}}"
2,14.51282320561263,0.5952925163904129,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.17855789220062088, ""subsample"": 0.6523068845866853, ""colsample_bytree"": 0.5488360570031919, ""min_child_weight"": 7, ""reg_lambda"": 0.2091498132903561}}"
3,14.03784997303109,0.6280978833143308,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.2256465100244266, ""subsample"": 0.7989499894055425, ""colsample_bytree"": 0.9609371175115584, ""min_child_weight"": 1, ""reg_lambda"": 0.03872118032174583}}"
4,16.061514232954416,0.6351259154693947,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 500, ""max_depth"": 5, ""learning_rate"": 0.04279909806035244, ""subsample"": 0.7713480415791243, ""colsample_bytree"": 0.5704621124873813, ""min_child_weight"": 9, ""reg_lambda"": 0.01673601016782578}}"
5,16.860883263175317,0.5932681873339147,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 500, ""max_depth"": 8, ""learning_rate"": 0.14401540623155276, ""subsample"": 0.8856351733429728, ""colsample_bytree"": 0.5370223258670452, ""min_child_weight"": 4, ""reg_lambda"": 0.022264204303769682}}"
6,16.531616998750973,0.5507171813641436,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 5, ""learning_rate"": 0.14424921039779712, ""subsample"": 0.8187787356776066, ""colsample_bytree"": 0.9436063712881633, ""min_child_weight"": 5, ""reg_lambda"": 0.022844556850020535}}"
7,13.642248116823183,0.6188047266761567,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 7, ""learning_rate"": 0.2335667365205794, ""subsample"": 0.6246461145744375, ""colsample_bytree"": 0.7051914615178149, ""min_child_weight"": 8, ""reg_lambda"": 0.04857295179217168}}"
8,13.335468662766138,0.6228578122325681,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.04731851459137855, ""subsample"": 0.5550259622638384, ""colsample_bytree"": 0.6139675812709708, ""min_child_weight"": 5, ""reg_lambda"": 2.8447512555118193}}"
10,13.31427635157264,0.6365668883060389,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 9, ""learning_rate"": 0.04398724797286349, ""subsample"": 0.5409438634048439, ""colsample_bytree"": 0.6604144336321783, ""min_child_weight"": 9, ""reg_lambda"": 4.985390561409462}}"
11,13.245855387738288,0.6114463294697008,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 350, ""max_depth"": 9, ""learning_rate"": 0.03840501767644911, ""subsample"": 0.5092866199966892, ""colsample_bytree"": 0.5507187964196311, ""min_child_weight"": 6, ""reg_lambda"": 2.008070155745467}}"
12,13.17669914293017,0.6405770958211371,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 8, ""learning_rate"": 0.02814414565669774, ""subsample"": 0.5188071861055975, ""colsample_bytree"": 0.5327816228174422, ""min_child_weight"": 8, ""reg_lambda"": 6.380994123134933}}"
17,13.151316696768202,0.6490908128515369,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 8, ""learning_rate"": 0.02286224551814071, ""subsample"": 0.5183883075398142, ""colsample_bytree"": 0.5866601688192898, ""min_child_weight"": 5, ""reg_lambda"": 2.286173715745685}}"
19,13.80990952917287,0.5540744605631952,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 200, ""max_depth"": 5, ""learning_rate"": 0.0214130241899486, ""subsample"": 0.5834580703340053, ""colsample_bytree"": 0.5744118921463852, ""min_child_weight"": 10, ""reg_lambda"": 3.821399492199396}}"
20,13.250629738876459,0.6415664888261804,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 400, ""max_depth"": 8, ""learning_rate"": 0.022135478180183195, ""subsample"": 0.5471035266052687, ""colsample_bytree"": 0.6842111168163153, ""min_child_weight"": 4, ""reg_lambda"": 8.677327095371378}}"
21,13.1841274061358,0.6286077986383383,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 550, ""max_depth"": 8, ""learning_rate"": 0.03415649577646418, ""subsample"": 0.6453659732265881, ""colsample_bytree"": 0.5556767052194727, ""min_child_weight"": 4, ""reg_lambda"": 6.156604652161078}}"
22,13.16911042736327,0.6555890992055876,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 6, ""learning_rate"": 0.029224515210639207, ""subsample"": 0.6560261359419673, ""colsample_bytree"": 0.5377066786058777, ""min_child_weight"": 3, ""reg_lambda"": 6.312017973672459}}"
23,13.085661877502414,0.6330602950150462,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 550, ""max_depth"": 6, ""learning_rate"": 0.021377844571975117, ""subsample"": 0.5555576766627507, ""colsample_bytree"": 0.5176803628728603, ""min_child_weight"": 3, ""reg_lambda"": 3.1216635272378905}}"
29,13.15281023509543,0.6550312221498794,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 550, ""max_depth"": 5, ""learning_rate"": 0.03384169832663698, ""subsample"": 0.5054623892513126, ""colsample_bytree"": 0.5334367622354004, ""min_child_weight"": 4, ""reg_lambda"": 5.44998740998891}}"
30,13.149566703236365,0.6548218718337845,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 550, ""max_depth"": 7, ""learning_rate"": 0.03237291952946384, ""subsample"": 0.602135451410545, ""colsample_bytree"": 0.5607822788967811, ""min_child_weight"": 6, ""reg_lambda"": 1.4141276514480536}}"
31,13.128768343703683,0.6423138220674258,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 6, ""learning_rate"": 0.0287445504553845, ""subsample"": 0.5428402778737232, ""colsample_bytree"": 0.5586507756819711, ""min_child_weight"": 3, ""reg_lambda"": 3.476113236025342}}"
32,13.161446547261562,0.663405989312896,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 6, ""learning_rate"": 0.03484128605765407, ""subsample"": 0.6553804748330132, ""colsample_bytree"": 0.5638363909963, ""min_child_weight"": 2, ""reg_lambda"": 2.054637962730561}}"
33,13.144727132933332,0.6592518013180302,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 6, ""learning_rate"": 0.020820497462095237, ""subsample"": 0.553339426975982, ""colsample_bytree"": 0.6334629930733423, ""min_child_weight"": 1, ""reg_lambda"": 1.5797605200722424}}"
34,13.163857701347814,0.6624753237716886,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 7, ""learning_rate"": 0.020786638506367753, ""subsample"": 0.6165073619143829, ""colsample_bytree"": 0.6415234346965397, ""min_child_weight"": 3, ""reg_lambda"": 0.8567011418676481}}"
35,13.157343984596931,0.6706037736302363,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 6, ""learning_rate"": 0.026467154877352145, ""subsample"": 0.5159865858444139, ""colsample_bytree"": 0.6339219134739188, ""min_child_weight"": 1, ""reg_lambda"": 3.0801553235293326}}"
38,12.818884088280026,0.6329642573439073,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 5, ""learning_rate"": 0.026678839993802973, ""subsample"": 0.5502580787869915, ""colsample_bytree"": 0.6488556907652996, ""min_child_weight"": 4, ""reg_lambda"": 6.685416699278224}}"
39,12.771370137084906,0.6358862495553668,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 6, ""learning_rate"": 0.025680587903153983, ""subsample"": 0.6284045005374712, ""colsample_bytree"": 0.6599999705697781, ""min_child_weight"": 2, ""reg_lambda"": 9.079888238484369}}"
40,12.782562000029762,0.6438569801364679,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 6, ""learning_rate"": 0.03605360784280554, ""subsample"": 0.5639707811967875, ""colsample_bytree"": 0.5786344326103258, ""min_child_weight"": 4, ""reg_lambda"": 6.064111162803813}}"
41,12.806781031058481,0.6293848156897235,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 5, ""learning_rate"": 0.042317612652990604, ""subsample"": 0.5549001927681941, ""colsample_bytree"": 0.6085136190440803, ""min_child_weight"": 3, ""reg_lambda"": 6.085228012553192}}"
42,12.786816904443528,0.6427404080107791,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 5, ""learning_rate"": 0.040100156758551594, ""subsample"": 0.5771826806564725, ""colsample_bytree"": 0.6207720234476323, ""min_child_weight"": 5, ""reg_lambda"": 4.4789195068489835}}"
43,12.746848196673776,0.6372885198386837,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 5, ""learning_rate"": 0.04029789571266671, ""subsample"": 0.5555771722151779, ""colsample_bytree"": 0.5816082787454413, ""min_child_weight"": 4, ""reg_lambda"": 4.60837978496}}"
44,12.790945319770525,0.6224680121277473,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 5, ""learning_rate"": 0.04563231137973807, ""subsample"": 0.5883071852893853, ""colsample_bytree"": 0.5679696625131212, ""min_child_weight"": 2, ""reg_lambda"": 6.369681654436413}}"
45,12.781963819155813,0.6524253302139955,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 6, ""learning_rate"": 0.032937857015395644, ""subsample"": 0.6318799869179768, ""colsample_bytree"": 0.6597292215568695, ""min_child_weight"": 2, ""reg_lambda"": 4.744295020208645}}"
46,12.823738375537156,0.6592157755556276,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 5, ""learning_rate"": 0.044921368347310656, ""subsample"": 0.6310518155371093, ""colsample_bytree"": 0.6252938099569746, ""min_child_weight"": 5, ""reg_lambda"": 5.550426597604659}}"
47,12.830817583620151,0.6249147859809143,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 5, ""learning_rate"": 0.042977142265280384, ""subsample"": 0.5471057732574209, ""colsample_bytree"": 0.6173038691080859, ""min_child_weight"": 4, ""reg_lambda"": 2.165504715946532}}"
48,12.846452458170267,0.6132295245579426,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 5, ""learning_rate"": 0.034168012030687436, ""subsample"": 0.611675549846602, ""colsample_bytree"": 0.6368237666588358, ""min_child_weight"": 3, ""reg_lambda"": 9.716408258063797}}"
49,12.801482257280142,0.648455178556897,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 6, ""learning_rate"": 0.025998683110204483, ""subsample"": 0.645805411484444, ""colsample_bytree"": 0.6501895559603463, ""min_child_weight"": 2, ""reg_lambda"": 6.5637880641347275}}"
50,12.87664946062662,0.6259488346767489,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 5, ""learning_rate"": 0.0255315618056073, ""subsample"": 0.6214745772638163, ""colsample_bytree"": 0.6007065582478189, ""min_child_weight"": 4, ""reg_lambda"": 5.080126935075363}}"
51,12.785020292752765,0.6417094255814446,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 6, ""learning_rate"": 0.035848136358946514, ""subsample"": 0.613195898581175, ""colsample_bytree"": 0.5281695959250912, ""min_child_weight"": 1, ""reg_lambda"": 9.626261546552415}}"
52,12.759503294715989,0.6416181446508128,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 6, ""learning_rate"": 0.04206770231856872, ""subsample"": 0.5442559994357067, ""colsample_bytree"": 0.5493310845879895, ""min_child_weight"": 5, ""reg_lambda"": 3.927559252190707}}"
53,12.718174525611202,0.6168391135561053,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 7, ""learning_rate"": 0.029324489533756482, ""subsample"": 0.6291159532911457, ""colsample_bytree"": 0.5114028394798102, ""min_child_weight"": 1, ""reg_lambda"": 6.125276304862555}}"
54,12.705564387795047,0.6058889499876481,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 7, ""learning_rate"": 0.025002340646295813, ""subsample"": 0.5849938629943712, ""colsample_bytree"": 0.5162961614246507, ""min_child_weight"": 2, ""reg_lambda"": 6.7763180941890155}}"
55,12.760752503078052,0.6365602698378704,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 6, ""learning_rate"": 0.04759785307848508, ""subsample"": 0.5248721277689946, ""colsample_bytree"": 0.5010226194383347, ""min_child_weight"": 4, ""reg_lambda"": 7.699605617164919}}"
56,12.74468057620248,0.6163239719832556,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 6, ""learning_rate"": 0.030699670692558925, ""subsample"": 0.5868714274015816, ""colsample_bytree"": 0.5257949578216484, ""min_child_weight"": 3, ""reg_lambda"": 2.6616744122911373}}"
57,12.737034997297274,0.6021734229548746,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 8, ""learning_rate"": 0.029241002797542322, ""subsample"": 0.568940747286034, ""colsample_bytree"": 0.5421379681002494, ""min_child_weight"": 2, ""reg_lambda"": 9.713291069540242}}"

In [ ]:
%%writefile projeto/data/hiperparametros/FD004/XGBoost_NASA_Loss/tentativas.csv
trial,rmse_mean,rmse_se,window,params
0,27.445696851105946,0.19583789079460603,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 100, ""max_depth"": 4, ""learning_rate"": 0.023406777359270506, ""subsample"": 0.9330880728874675, ""colsample_bytree"": 0.8005575058716043, ""min_child_weight"": 8, ""reg_lambda"": 0.011527987128232402}}"
1,18.437156622509928,0.5970615866557903,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 150, ""max_depth"": 5, ""learning_rate"": 0.08283072221077194, ""subsample"": 0.7159725093210578, ""colsample_bytree"": 0.645614570099021, ""min_child_weight"": 7, ""reg_lambda"": 0.02621087878265441}}"
2,14.736126157139813,0.6214083696100364,40,"{""representation"": ""regime"", ""temporal"": true, ""window"": 40, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.17855789220062088, ""subsample"": 0.6523068845866853, ""colsample_bytree"": 0.5488360570031919, ""min_child_weight"": 7, ""reg_lambda"": 0.2091498132903561}}"
3,16.880670419831013,0.463775254840736,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.2256465100244266, ""subsample"": 0.7989499894055425, ""colsample_bytree"": 0.9609371175115584, ""min_child_weight"": 1, ""reg_lambda"": 0.03872118032174583}}"
4,16.57987883342549,0.762333150962591,0,"{""representation"": ""regime"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 500, ""max_depth"": 5, ""learning_rate"": 0.04279909806035244, ""subsample"": 0.7713480415791243, ""colsample_bytree"": 0.5704621124873813, ""min_child_weight"": 9, ""reg_lambda"": 0.01673601016782578}}"
5,16.71842806779303,0.749673140861224,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 500, ""max_depth"": 8, ""learning_rate"": 0.14401540623155276, ""subsample"": 0.8856351733429728, ""colsample_bytree"": 0.5370223258670452, ""min_child_weight"": 4, ""reg_lambda"": 0.022264204303769682}}"
6,17.266390449454228,0.6950445467621551,0,"{""representation"": ""global"", ""temporal"": false, ""window"": 0, ""hyper"": {""n_estimators"": 200, ""max_depth"": 5, ""learning_rate"": 0.14424921039779712, ""subsample"": 0.8187787356776066, ""colsample_bytree"": 0.9436063712881633, ""min_child_weight"": 5, ""reg_lambda"": 0.022844556850020535}}"
7,13.688447472056207,0.5291362367864192,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 200, ""max_depth"": 7, ""learning_rate"": 0.2335667365205794, ""subsample"": 0.6246461145744375, ""colsample_bytree"": 0.7051914615178149, ""min_child_weight"": 8, ""reg_lambda"": 0.04857295179217168}}"
8,13.095599222539168,0.6648216472468684,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.04731851459137855, ""subsample"": 0.5550259622638384, ""colsample_bytree"": 0.6139675812709708, ""min_child_weight"": 5, ""reg_lambda"": 2.8447512555118193}}"
10,13.136307754385385,0.6847424432428808,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 9, ""learning_rate"": 0.04398724797286349, ""subsample"": 0.5409438634048439, ""colsample_bytree"": 0.6604144336321783, ""min_child_weight"": 9, ""reg_lambda"": 4.985390561409462}}"
11,13.10252575202692,0.6699173317569054,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 350, ""max_depth"": 9, ""learning_rate"": 0.03840501767644911, ""subsample"": 0.5092866199966892, ""colsample_bytree"": 0.5507187964196311, ""min_child_weight"": 6, ""reg_lambda"": 2.008070155745467}}"
12,13.109770059233128,0.6977965745007314,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 400, ""max_depth"": 10, ""learning_rate"": 0.03529767541238648, ""subsample"": 0.5336416084668992, ""colsample_bytree"": 0.6070610427030017, ""min_child_weight"": 3, ""reg_lambda"": 3.5945922316651338}}"
13,15.623774088867018,0.7471074004781617,20,"{""representation"": ""regime"", ""temporal"": true, ""window"": 20, ""hyper"": {""n_estimators"": 200, ""max_depth"": 8, ""learning_rate"": 0.05162627515771169, ""subsample"": 0.5148462288738874, ""colsample_bytree"": 0.7488248062015586, ""min_child_weight"": 6, ""reg_lambda"": 0.9471012528603403}}"
16,14.522823198543051,0.7343401143768231,30,"{""representation"": ""regime"", ""temporal"": true, ""window"": 30, ""hyper"": {""n_estimators"": 300, ""max_depth"": 9, ""learning_rate"": 0.05622201509767075, ""subsample"": 0.7350284331870716, ""colsample_bytree"": 0.677591115586103, ""min_child_weight"": 8, ""reg_lambda"": 1.7102928943340545}}"
17,13.285836068639146,0.6855435436822623,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 8, ""learning_rate"": 0.06586079408920886, ""subsample"": 0.6504015815281086, ""colsample_bytree"": 0.7262304639754923, ""min_child_weight"": 4, ""reg_lambda"": 3.999123408683371}}"
18,13.206237439224296,0.6873470934525849,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 7, ""learning_rate"": 0.04484191256570486, ""subsample"": 0.564580736941287, ""colsample_bytree"": 0.6284575620137264, ""min_child_weight"": 4, ""reg_lambda"": 9.214025441325587}}"
19,13.230113983614695,0.6909942004171562,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 450, ""max_depth"": 8, ""learning_rate"": 0.02112960392205049, ""subsample"": 0.5483384261671863, ""colsample_bytree"": 0.6465909340714668, ""min_child_weight"": 6, ""reg_lambda"": 0.46167433617984416}}"
21,13.617659993338947,0.6547353153509939,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 250, ""max_depth"": 9, ""learning_rate"": 0.03393693421764936, ""subsample"": 0.5337735744120936, ""colsample_bytree"": 0.5607691671953405, ""min_child_weight"": 8, ""reg_lambda"": 4.402146771025517}}"
22,13.193058069284723,0.6705214680381627,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 9, ""learning_rate"": 0.04542455929274083, ""subsample"": 0.5451531746245325, ""colsample_bytree"": 0.7012737973533893, ""min_child_weight"": 2, ""reg_lambda"": 1.7877057451575964}}"
23,13.124958405426217,0.6984662617347259,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 250, ""max_depth"": 10, ""learning_rate"": 0.047743032806208424, ""subsample"": 0.6349925919211288, ""colsample_bytree"": 0.5223561860600526, ""min_child_weight"": 5, ""reg_lambda"": 1.261317110330196}}"
28,13.633252218071124,0.6479009377328995,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 300, ""max_depth"": 9, ""learning_rate"": 0.025888089868373584, ""subsample"": 0.553399284942794, ""colsample_bytree"": 0.5242396304041488, ""min_child_weight"": 4, ""reg_lambda"": 2.0041966688677433}}"
29,13.195414441550597,0.7017788975594547,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 350, ""max_depth"": 10, ""learning_rate"": 0.04281142995010173, ""subsample"": 0.7212731250470599, ""colsample_bytree"": 0.6382791197408504, ""min_child_weight"": 3, ""reg_lambda"": 6.423078336599448}}"
30,13.142148400339881,0.6679637498130363,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 600, ""max_depth"": 9, ""learning_rate"": 0.03986624100125785, ""subsample"": 0.5028303406790908, ""colsample_bytree"": 0.6057256509362149, ""min_child_weight"": 2, ""reg_lambda"": 1.6340334799186478}}"
31,13.346816977207652,0.6898250183281126,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 350, ""max_depth"": 10, ""learning_rate"": 0.028838069109573542, ""subsample"": 0.617827904244907, ""colsample_bytree"": 0.6445182112929468, ""min_child_weight"": 3, ""reg_lambda"": 6.155597244509859}}"
32,13.097101962372866,0.6882133754808799,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.02948091729815857, ""subsample"": 0.5338388859994836, ""colsample_bytree"": 0.5958845080902104, ""min_child_weight"": 3, ""reg_lambda"": 4.083495669645308}}"
33,13.167933649757359,0.6843995006061374,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.02754925732504157, ""subsample"": 0.5738023509410479, ""colsample_bytree"": 0.584659260601738, ""min_child_weight"": 4, ""reg_lambda"": 8.652986402599298}}"
34,13.168087142736848,0.6862452441752304,50,"{""representation"": ""regime"", ""temporal"": true, ""window"": 50, ""hyper"": {""n_estimators"": 500, ""max_depth"": 9, ""learning_rate"": 0.033087908735948385, ""subsample"": 0.5687248284541797, ""colsample_bytree"": 0.7358975436754819, ""min_child_weight"": 4, ""reg_lambda"": 4.902597351999091}}"
38,12.734166541435332,0.6481426761655431,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 9, ""learning_rate"": 0.023813558315628165, ""subsample"": 0.5364926222664504, ""colsample_bytree"": 0.5878939192846074, ""min_child_weight"": 3, ""reg_lambda"": 2.4625519244080585}}"
39,12.767315144151606,0.6566735605029917,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.02024235985933185, ""subsample"": 0.6007406162129392, ""colsample_bytree"": 0.6400825527679332, ""min_child_weight"": 5, ""reg_lambda"": 2.334192637642266}}"
40,12.738189559350944,0.6485076145394958,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 9, ""learning_rate"": 0.021619654310033924, ""subsample"": 0.5501582997207786, ""colsample_bytree"": 0.6479905759827774, ""min_child_weight"": 3, ""reg_lambda"": 3.844603243845697}}"
41,12.726994761173367,0.6470013144473569,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.030302907985396553, ""subsample"": 0.6073724627484232, ""colsample_bytree"": 0.6561051750180268, ""min_child_weight"": 4, ""reg_lambda"": 2.7016696062044967}}"
42,12.696727040175741,0.6484108924139923,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.025513753476216008, ""subsample"": 0.5710677773683259, ""colsample_bytree"": 0.6303560206507948, ""min_child_weight"": 4, ""reg_lambda"": 1.059507833075385}}"
43,12.804412848010177,0.6624447406253852,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 9, ""learning_rate"": 0.020220606073823942, ""subsample"": 0.6170700511553763, ""colsample_bytree"": 0.6903318530839238, ""min_child_weight"": 3, ""reg_lambda"": 1.6314742837163458}}"
44,12.727963433140692,0.6432719582666739,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.0296492619229224, ""subsample"": 0.5493830574415, ""colsample_bytree"": 0.6690701276835238, ""min_child_weight"": 3, ""reg_lambda"": 1.6200973789479292}}"
45,12.703365774244315,0.651730528470502,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.02747469442045487, ""subsample"": 0.5757484939651378, ""colsample_bytree"": 0.6293554735007931, ""min_child_weight"": 3, ""reg_lambda"": 0.7174861744869085}}"
46,12.798028159248062,0.6444876776664701,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.023966528900716554, ""subsample"": 0.5759105026065205, ""colsample_bytree"": 0.7058073487358069, ""min_child_weight"": 2, ""reg_lambda"": 0.49310185601461126}}"
47,12.775377294488413,0.638240253528117,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 9, ""learning_rate"": 0.028722960689534813, ""subsample"": 0.5763522130077688, ""colsample_bytree"": 0.6621050449159533, ""min_child_weight"": 4, ""reg_lambda"": 2.4449630906598396}}"
48,12.7013361969486,0.6316151081279614,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.035396945890734076, ""subsample"": 0.6304643505075919, ""colsample_bytree"": 0.6424400359427341, ""min_child_weight"": 5, ""reg_lambda"": 1.7686517212029331}}"
49,12.679571971321845,0.6482749216450153,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.030596642625169482, ""subsample"": 0.5216406959277221, ""colsample_bytree"": 0.634938276650228, ""min_child_weight"": 5, ""reg_lambda"": 1.4329595699665942}}"
50,12.746275400030406,0.6569661693800823,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.021063864413870318, ""subsample"": 0.6384018907521902, ""colsample_bytree"": 0.6195698550010537, ""min_child_weight"": 3, ""reg_lambda"": 1.9526521402658714}}"
51,12.64604293835448,0.6647426220506243,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.024706148380843465, ""subsample"": 0.6154913602657046, ""colsample_bytree"": 0.543514367211477, ""min_child_weight"": 4, ""reg_lambda"": 0.8482007006560248}}"
52,12.645182514894135,0.647453341244166,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.029300062650266905, ""subsample"": 0.5483296767556515, ""colsample_bytree"": 0.5637042395911861, ""min_child_weight"": 4, ""reg_lambda"": 1.1151332022512956}}"
53,12.664777591696799,0.6495884253006167,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.02503133213743937, ""subsample"": 0.5620731166865548, ""colsample_bytree"": 0.5670469771230343, ""min_child_weight"": 3, ""reg_lambda"": 0.8677028163428744}}"
54,12.667321906370914,0.6430174120639737,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.02472248654318285, ""subsample"": 0.517081826869267, ""colsample_bytree"": 0.5796843931385165, ""min_child_weight"": 4, ""reg_lambda"": 1.6975461122256217}}"
55,12.662699267223484,0.6534626070204769,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 550, ""max_depth"": 10, ""learning_rate"": 0.036107646506036005, ""subsample"": 0.5285789197243244, ""colsample_bytree"": 0.5119505938937581, ""min_child_weight"": 4, ""reg_lambda"": 0.9138693254941836}}"
56,12.647877830928772,0.675615320573535,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 600, ""max_depth"": 10, ""learning_rate"": 0.032768398429337625, ""subsample"": 0.5173713098864422, ""colsample_bytree"": 0.5401775198852186, ""min_child_weight"": 4, ""reg_lambda"": 1.0436416287139745}}"
58,12.670764699112548,0.6621201204734097,60,"{""representation"": ""regime"", ""temporal"": true, ""window"": 60, ""hyper"": {""n_estimators"": 500, ""max_depth"": 10, ""learning_rate"": 0.023098456180723485, ""subsample"": 0.5463420493646922, ""colsample_bytree"": 0.5801221747755687, ""min_child_weight"": 4, ""reg_lambda"": 0.6475349111090345}}"

### 3.2 NASA C-MAPSS (dados públicos)

Saxena et al. (2008), NASA Ames Prognostics Data Repository. Os arquivos (`train_FD00x`, `test_FD00x`, `RUL_FD00x`) vão embutidos em base64 (11 células,
15.5 MB); a célula seguinte às partes os reconstrói.

In [ ]:
%%writefile projeto/data/cmaps.tgz.b64.parte00
...

In [ ]:
%%writefile projeto/data/cmaps.tgz.b64.parte01
...

In [ ]:
%%writefile projeto/data/cmaps.tgz.b64.parte02
...

In [ ]:
%%writefile projeto/data/cmaps.tgz.b64.parte03
...

In [ ]:
%%writefile projeto/data/cmaps.tgz.b64.parte04
...

In [ ]:
%%writefile projeto/data/cmaps.tgz.b64.parte05
...

In [ ]:
%%writefile projeto/data/cmaps.tgz.b64.parte06
...

In [ ]:
%%writefile projeto/data/cmaps.tgz.b64.parte07
...

In [ ]:
%%writefile projeto/data/cmaps.tgz.b64.parte08
...

In [ ]:
%%writefile projeto/data/cmaps.tgz.b64.parte09
...

In [ ]:
%%writefile projeto/data/cmaps.tgz.b64.parte10
...

In [ ]:
!cat projeto/data/cmaps.tgz.b64.parte* | base64 -d | tar xz -C projeto
!rm -f projeto/data/cmaps.tgz.b64.parte*
!ls -la projeto/data/CMaps

### 3.3 Instalação do pacote

In [ ]:
!pip install -q -e "projeto[dev,tuning]"

In [ ]:
os.chdir(PASTA)                              # todos os caminhos do projeto são relativos à raiz
!python -c "import pdm_cmaps, torch, xgboost, lightgbm, catboost, sklearn; print('pdm_cmaps ok | torch', torch.__version__)"

## 4. Testes automáticos

Suíte sem os estudos longos (no Colab High-RAM: cerca de 1 minuto).

In [ ]:
!python -m pytest -q -p no:cacheprovider

## 5. Configuração das execuções

`True` liga o bloco. Os tempos são estimativas em uma máquina de 8 núcleos.

| Chave | O que roda | Tempo |
|---|---|---|
| `demo_fd001` | Pipeline completo do FD001 (detecção, RUL, conformal, alertas) | ~20 min |
| `pipeline_4` | Pipeline dos 4 subconjuntos | ~1,5 h |
| `estudos` | Todos os estudos (monitoramento, injeção cega, janela, falhas sorteadas...) e sincronização das tabelas, via `rodar tudo` | muitas horas |
| `rul_oficial` | Teste oficial de RUL: 12 modelos × 5 sementes, redes com regime e janela 30, boosting | horas |
| `rul_ajustado_pronto` | Teste oficial ajustado com os hiperparâmetros de `data/hiperparametros` (sem refazer a busca) | horas |
| `busca_hiperparametros` | Refaz a busca Optuna (40 estudos) e o teste ajustado | **dias** |
| `diagnostico_fd003` | Ganho negativo do Extra Trees no FD003 | ~10 min |
| `estudos_complementares` | Experimentos A a D (conformal ajustado, exogeneidade, baselines, modos de falha) | horas |
| `teste_bloco` | Série saudável reamostrada em blocos (Apêndice A.6) | horas |
| `taxa_recuo` | Taxa de recuo do deslocamento do motor (Apêndice A.5) | segundos |
| `tabelas` | `montar_tabelas.py` | minutos |
| `figuras` | Todos os gráficos | minutos |

In [ ]:
EXECUTAR = {
    "testes": True,
    "demo_fd001": True,
    "pipeline_4": False,
    "estudos": False,
    "rul_oficial": False,
    "rul_ajustado_pronto": False,
    "busca_hiperparametros": False,
    "diagnostico_fd003": False,
    "estudos_complementares": False,
    "teste_bloco": False,
    "taxa_recuo": True,
    "tabelas": False,
    "figuras": False,
}
# Threads por processo: com vários processos em paralelo, 1 ou 2 por processo evita lentidão (ver scripts/README.md)
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("PDM_TORCH_THREADS", "2")

### 5.1 Dividir o trabalho em várias máquinas (opcional)

Reproduzir tudo em uma máquina leva mais de 12 h, mais que uma sessão do Colab. Abra **o mesmo notebook em até 4 sessões** e defina `SESSAO` abaixo; cada uma liga só a sua parte:

| `SESSAO` | O que roda | Tempo aprox. |
|---|---|---|
| `"A"` | pipeline dos 4 subconjuntos + todos os estudos (`rodar tudo`) | ~7 h |
| `"B"` | teste oficial de RUL (`rul_oficial`) | ~4 h |
| `"C"` | teste ajustado (hiperparâmetros prontos), estudos complementares, teste em blocos, diagnóstico do FD003 | ~5 h |
| `"FINAL"` | recebe os resultados das outras (seção 5.2), monta as tabelas e os gráficos | ~15 min |

Ao fim de A, B e C, rode a célula da seção 5.2 (empacota `results/` e baixa). Na sessão FINAL, envie os três arquivos `.tgz` e rode as seções 7 e 8.
Com `SESSAO = None` vale o `EXECUTAR` da seção 5 (uma máquina só).

In [ ]:
SESSAO = None          # "A", "B", "C", "FINAL" ou None
PRESETS = {
    "A": ["testes", "pipeline_4", "estudos"],
    "B": ["rul_oficial"],
    "C": ["rul_ajustado_pronto", "estudos_complementares", "teste_bloco", "diagnostico_fd003", "taxa_recuo"],
    "FINAL": ["tabelas", "figuras"],
}
if SESSAO:
    EXECUTAR = {k: (k in PRESETS[SESSAO]) for k in EXECUTAR}
    print("sessão", SESSAO, "->", [k for k, v in EXECUTAR.items() if v])

### 5.2 Empacotar e juntar os resultados das sessões

In [ ]:
# No fim das sessões A, B e C: empacota results/ e baixa (só os resultados; o código é o mesmo em todas)
if SESSAO in ("A", "B", "C"):
    !cd {PASTA} && tar czf /tmp/results_sessao_{SESSAO}.tgz results tcc-writing/tables && ls -la /tmp/results_sessao_{SESSAO}.tgz
    try:
        from google.colab import files
        files.download(f"/tmp/results_sessao_{SESSAO}.tgz")
    except ImportError:
        print("fora do Colab: arquivo em /tmp")

In [ ]:
# Na sessão FINAL: envie results_sessao_A.tgz, _B.tgz e _C.tgz (uma vez cada) e eles são juntados em results/
if SESSAO == "FINAL":
    from google.colab import files
    for nome in files.upload():
        !tar xzf "{nome}" -C {PASTA}
    !ls {PASTA}/results

## 6. Resultados

### 6.1 Demonstração rápida e pipeline por subconjunto

Saída em `results/maintenance_guide_fd00x/` (tabelas, previsões de RUL, linhas do tempo de alertas).

In [ ]:
if EXECUTAR["demo_fd001"]:
    !python -m pdm_cmaps --subset FD001

In [ ]:
if EXECUTAR["pipeline_4"]:
    for fd in ["FD001", "FD002", "FD003", "FD004"]:
        !python -m pdm_cmaps --subset {fd}

### 6.2 Todos os estudos e sincronização das tabelas

`rodar tudo` é retomável: se a sessão cair, rode de novo e ele continua do último passo concluído. Use `--dry-run` para ver o plano.

In [ ]:
if EXECUTAR["estudos"]:
    !python scripts/rodar.py rodar tudo
else:
    !python scripts/rodar.py lista

### 6.3 Teste oficial de RUL (12 modelos, 5 sementes) e agregação

In [ ]:
if EXECUTAR["rul_oficial"]:
    !python scripts/rodar.py rodar rul_oficial

### 6.4 Teste oficial ajustado

Com os hiperparâmetros e janelas já escolhidos (`data/hiperparametros`). O ajuste final é refeito na CPU, com três sementes, e o teste oficial é lido uma vez por modelo.

In [ ]:
if EXECUTAR["rul_ajustado_pronto"]:
    of = "pdm_cmaps.experiments.benchmark_oficial_completo"
    for fd in ["FD001", "FD002", "FD003", "FD004"]:
        for seed in [42, 7, 123]:
            !python -m {of} --subsets {fd} --part tuned --seeds {seed} --params-dir data/hiperparametros --output-dir results/benchmark_oficial_tuned/{fd}_s{seed}
    !python -m {of} --aggregate-only --output-dir results/benchmark_oficial_tuned

In [ ]:
if EXECUTAR["busca_hiperparametros"]:   # refaz tudo, inclusive a busca (dias)
    !python scripts/rodar.py rodar hiperparametros

### 6.5 Estudos complementares e diagnósticos

In [ ]:
if EXECUTAR["diagnostico_fd003"]:
    !python scripts/rodar.py rodar diagnostico_fd003

In [ ]:
if EXECUTAR["estudos_complementares"]:
    !python -m pdm_cmaps.experiments.estudos_complementares --study all --jobs 4

In [ ]:
if EXECUTAR["teste_bloco"]:
    !python scripts/teste_bloco_injecao.py diag FD004 42
    !python scripts/teste_bloco_injecao.py run bloco FD004 42
    !python scripts/teste_bloco_injecao.py resumir

In [ ]:
if EXECUTAR["taxa_recuo"]:
    !python scripts/taxa_fallback_deslocamento.py

## 7. Tabelas

`rodar tudo` já sincroniza as tabelas dos estudos para `tcc-writing/tables/`. `montar_tabelas.py` acrescenta as derivadas (testes pareados por motor, hiperparâmetros,
métricas estendidas, ajustado contra padrão etc.). O boosting do teste oficial fica em `results/benchmark_oficial/boost_*`; a célula o copia para a pasta que o script lê.

In [ ]:
if EXECUTAR["tabelas"]:
    !python -m pdm_cmaps.cli.update_all --so-sincronizar
    !mkdir -p results/benchmark_oficial_boosting && cp -r results/benchmark_oficial/boost_* results/benchmark_oficial_boosting/ 2>/dev/null; true
    !python tcc-writing/montar_tabelas.py --boost results/benchmark_oficial_boosting
    !ls tcc-writing/tables | wc -l

## 8. Gráficos

Cada script grava em `tcc-writing/figures/` (ou `tcc-writing/diagrams/explicativos/`). Os que leem `results/` precisam dos estudos da seção 6; os que leem `tcc-writing/tables/` precisam da seção 7.

In [ ]:
if EXECUTAR["figuras"]:
    for s in ["make_figuras_novas", "make_figuras_revisao", "make_figuras_mewma", "make_figuras_injecao", "make_figuras_janela",
              "make_figuras_consistencia", "make_figuras_deteccao"]:
        print("==", s)
        !python tcc-writing/{s}.py
    # Brayton e atmosfera; o diagrama do balanço de energia (Chrome headless) não está no notebook
    !cd tcc-writing/diagrams/explicativos && python -c "import make_fisica as m; m.brayton(); m.atmosfera()"
    !python tcc-writing/diagrams/explicativos/make_leitura_curvas.py
    !ls tcc-writing/figures | wc -l

In [ ]:
# Pré-visualização de algumas figuras (se existirem)
from IPython.display import Image, display
for nome in ["F24_bruto_vs_residuo.png", "F23_decomposicao_ganho.png", "F11_curvas_operacionais.png", "F07_trajetorias_intervalo_conformal.png"]:
    p = PASTA / "tcc-writing/figures" / nome
    if p.exists():
        display(Image(filename=str(p), width=700))

## 9. Empacotar para download

In [ ]:
!cd {PASTA} && tar czf /tmp/resultados_tabelas_figuras.tgz tcc-writing/tables tcc-writing/figures tcc-writing/diagrams/explicativos/*.png 2>/dev/null; ls -la /tmp/resultados_tabelas_figuras.tgz
try:
    from google.colab import files
    files.download("/tmp/resultados_tabelas_figuras.tgz")
except ImportError:
    print("fora do Colab: o arquivo está em /tmp/resultados_tabelas_figuras.tgz")